# Chuẩn bị dataset ảnh lịch sử Hán — Kaggle hoàn toàn offline

Điền **ba đường dẫn** ở ô cấu hình rồi **Run All**. Mặc định smoke một PDF/một trang,
bốn biến thể cổ điển. Bật model phục hồi sau khi đã chuẩn bị đủ asset và runtime.
Không cần clone repo, chạy notebook 3 hay có `src/`, `tests/`, `tools/` trên Kaggle.
Không OCR, dịch hoặc upload Dataset. Dữ liệu gốc/canonical luôn giữ riêng.

Audit trước triển khai: `docs/offline_image_notebook_audit.md`. Mã PDF/layout/v1
được nhúng từ HEAD hiện tại; mở rộng phục hồi có version riêng để notebook 3
vẫn đọc **original** bằng reader cũ. Các biến thể ×4/mask/tái dựng đọc bằng `prep_open`.


## 1. Cấu hình duy nhất

`PACKAGES_DIR` trỏ đến gốc packages; `MODELS_DIR` trỏ đến gốc models, không trỏ thẳng
vào thư mục một model. `DATA_DIR` có PDF. Không có đường dẫn Kaggle giả định sẵn.
`BOOKS=[]` chọn tất cả, tên/stem/glob như notebook 3; `PAGES` theo từng PDF,
`LIMIT` toàn tập. Không chạy full corpus trước khi xem smoke và dung lượng.


In [ ]:
from pathlib import Path
ON_KAGGLE = Path('/kaggle/working').is_dir()
PACKAGES_DIR = ''  # thư mục packages, bên trong có image_restoration/
MODELS_DIR = ''    # thư mục models, bên trong có image_restoration/
DATA_DIR = ''      # thư mục chứa PDF
OUTPUT_DIR = '/kaggle/working/image_dataset' if ON_KAGGLE else 'work/offline_image_dataset'
RUNTIME_DIR = '/kaggle/working/.image_runtime' if ON_KAGGLE else 'work/offline_image_runtime'

RUN_MODE = 'process'  # process | prepare_assets (chỉ local)
RUN_PREPARATION = True
RUN_SELF_TESTS = True
VALIDATE_OUTPUT = True
EXPORT_PREVIEWS = True
BOOKS = ['1']
PAGES = '1'         # mỗi PDF, đếm từ 1; smoke nhanh nhất một trang
LIMIT = 1           # tổng số trang sau chọn sách; có thể đổi PAGES='1-2', LIMIT=2
MAX_BOOKS = 1
# Full corpus, chỉ đổi SAU khi smoke thành công:
# BOOKS=[]; PAGES=''; LIMIT=None; MAX_BOOKS=None

VARIANTS = ['original','clahe','bicubic_x2','bicubic_x4']
# Denoise không mặc định vì có thể mất nét yếu. Mọi model đều chọn độc lập:
# VARIANTS += ['denoised','realesrgan_x4','tadisr_x4','difftsr','unihir_restoration']
# TADiSR luôn lưu thêm tadisr_text_mask. Có thể chọn riêng original + tadisr_x4.
CROP_TYPES = ['page','group','column']
RESUME = True
ALLOW_PARTIAL = False  # thiếu model/variant => dừng trước PDF, hoặc không công bố snapshot
MAX_SOURCE_PIXELS = 100_000_000
MAX_OUTPUT_PIXELS = 100_000_000
MAX_STORAGE_BYTES = 15 * 1024**3
REGION_TIMEOUT_SECONDS = 600
MODEL_LOAD_TIMEOUT_SECONDS = 900
BATCH_SIZE = 1
DESKEW = False
DEWARP = False

LAYOUT = {'dpi':200,'analysis_max_side':1200,'reading_direction':'vertical_rtl',
          'columns_per_group':3,'column_cuts':None,'page_column_cuts':{}}
# Cấu hình nguồn của các sách hiện có được nhúng; không cần configs/ trên Kaggle.
# Không lấy max_side hay token budget OCR làm giới hạn ảnh canonical.
BOOK_LAYOUTS = {'1':{'dpi':225},'2':{'dpi':226},'3':{'dpi':226},
    '4':{'dpi':181},'5':{'dpi':181},'6':{'dpi':226},
    'bien':{'dpi':200,'reading_direction':'horizontal_ltr'},'kinh':{'dpi':45}}
VARIANT_OPTIONS = {'clahe':{'clip_limit':1.5,'tile_grid':8},
                   'denoised':{'h':2.0,'template_window':7,'search_window':21}}
MODEL_OPTIONS = {
    'realesrgan_x4':{'tile':256,'overlap':24,'half':False,'device':'cuda','seed':42},
    'tadisr_x4':{'tile':768,'overlap':256,'device':'cuda','seed':42},
    'difftsr':{'steps':20,'device':'cuda','seed':42},  # đối chứng 20 với default upstream 200
    'unihir_restoration':{'steps':50,'ref_count':6,'max_memory':'80GiB','device':'cuda','seed':42},
}
# Để trống = Real-ESRGAN dùng kernel, TADiSR tạo venv overlay riêng từ wheel Linux.
# DiffTSR/UniHIR cần Python/runtime riêng được kiểm chứng, không tự hạ kernel.
RUNTIME_PYTHONS = {}
REVIEWED_REGIONS = {}
# Ví dụ khai báo sau khi xem original; KHÔNG tạo nhãn chỉ vì muốn chạy model:
# REVIEWED_REGIONS['book:3:column_000'] = {
#   'upright_horizontal_single_line':True,'character_count':12,
#   'allow_simplified_text_prior':True} # DiffTSR có zhconv zh-cn nội bộ: nguy cơ phồn thể
# REVIEWED_REGIONS['book:3:page_000'] = {'damaged_inscription':True}
ACCEPT_UNIHIR_NONCOMMERCIAL = False  # xác nhận phạm vi nghiên cứu CC BY-NC-ND 4.0
GENERATE_TADISR_PROMPT_OFFLINE = True  # nếu thiếu embedding, suy ra offline trong worker riêng từ base đã xác minh

# Chỉ dùng khi RUN_MODE='prepare_assets', trên máy local:
STAGING_DIR = 'kaggle_upload'
ALLOW_ASSET_INTERNET = False
DOWNLOAD_LARGE_TADISR = False  # ~31 GB base + adapter ~749 MiB
DOWNLOAD_DIFFTSR = False       # hai checkpoint phát hành qua Drive, vài GiB
DOWNLOAD_UNIHIR = False       # ~28 GiB weights + fonts; cần phạm vi license ở trên
TARGET_PYTHON_VERSION = '313'  # ABI kernel Kaggle đã kiểm tra; không suy từ Windows

CONFIG = dict(packages_dir=PACKAGES_DIR, models_dir=MODELS_DIR, data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR, books=BOOKS, pages=PAGES, limit=LIMIT, max_books=MAX_BOOKS,
    variants=VARIANTS, crop_types=CROP_TYPES, resume=RESUME, allow_partial=ALLOW_PARTIAL,
    max_source_pixels=MAX_SOURCE_PIXELS,max_output_pixels=MAX_OUTPUT_PIXELS,
    max_storage_bytes=MAX_STORAGE_BYTES,timeout_seconds=REGION_TIMEOUT_SECONDS,
    load_timeout_seconds=MODEL_LOAD_TIMEOUT_SECONDS,batch_size=BATCH_SIZE,
    deskew=DESKEW,dewarp=DEWARP,layout=LAYOUT,book_layouts=BOOK_LAYOUTS,
    variant_options=VARIANT_OPTIONS,model_options=MODEL_OPTIONS,
    runtime_pythons=RUNTIME_PYTHONS,reviewed_regions=REVIEWED_REGIONS,
    accept_unihir_noncommercial=ACCEPT_UNIHIR_NONCOMMERCIAL,
    generate_tadisr_prompt_offline=GENERATE_TADISR_PROMPT_OFFLINE)


## 2. Mã PDF, layout và dataset canonical được nhúng

PDFium kiểm tra placement, crop, rotation, clip, mask và annotation; native chỉ
được chọn khi bằng pixel với trang hiển thị. Nếu không đạt, render giữ mật độ ảnh
nguồn. Native scan 600×450 không có thêm thông tin nét khi tăng DPI.
Ảnh phân tích tối đa 1200 chỉ phục vụ projection; crop lấy từ nguồn không cap.
Giữ đủ page/group/column, RTL, manual cuts và QC phân hoạch. Column là ứng viên
hình học, chưa được xác nhận là một cột văn bản. Deskew/dewarp mặc định tắt vì
chưa có bằng chứng cần thiết trên các PDF này; không bóp méo nét để đẹp hơn.


In [ ]:
from __future__ import annotations
import re
from fnmatch import fnmatch
"""Acquire visible PDF pages without throwing away native scan pixels.

Native extraction is deliberately conservative. A sole image is insufficient:
placement, page boxes, clips, annotations and a visible-render equality check
must all pass. The original PDF remains the authoritative source representation.
"""


import math


PDF_IMAGE_VERSION = "pdfium-visible-v1"


def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t-b)/w, (r-l)/h, 0, l, b]
    if rotation == 180:
        return [-(r-l)/w, 0, 0, (t-b)/h, r, b]
    if rotation == 270:
        return [0, -(t-b)/w, -(r-l)/h, 0, r, t]
    return [(r-l)/w, 0, 0, -(t-b)/h, l, t]


def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w*scale), math.ceil(h*scale)]
    if min(size) < 1 or size[0]*size[1] > max_pixels:
        raise ValueError(f"Page render {size} exceeds max_source_pixels={max_pixels}; "
                         "review acquisition settings (source is never silently reduced)")
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert("RGB")
    finally:
        bitmap.close()


def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100_000_000, strategy="auto"):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops

    if strategy not in {"auto", "render"} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError("Invalid PDF acquisition options")
    width, height = page.get_size()
    bbox, rotation = list(page.get_bbox()), page.get_rotation()
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = [], 0.0
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x,y) for x,y in ((0,0),(1,0),(1,1),(0,1))]
        bounds = [min(c[0] for c in corners),min(c[1] for c in corners),
                  max(c[0] for c in corners),max(c[1] for c in corners)]
        full = (
            max(0, min(bbox[2], bounds[2])-max(bbox[0], bounds[0])) *
            max(0, min(bbox[3], bounds[3])-max(bbox[1], bounds[1])) >=
            .8*(bbox[2]-bbox[0])*(bbox[3]-bbox[1]))
        native = max(pixels[0]/axes[0], pixels[1]/axes[1]) if min(axes) > 0 else 0
        # Include partial-page/multiple scans as well: a DPI chosen from only a
        # full-page image could silently reduce detail in another visible image.
        density = max(density, native)
        images.append({"native_size": list(pixels), "bounds_pdf": bounds,
                       "matrix_pdf": [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f],
                       "level": obj.level, "full_page_scan": full,
                       "pixels_per_72_canvas_units": 72*native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append("not_one_top_level_image; preserve_composition")
    if rotation:
        reasons.append("page_rotation")
    if annotations != 0:
        reasons.append("page_annotations_or_unknown")
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if (obj.level or abs(m.b) > 1e-6 or abs(m.c) > 1e-6 or m.a <= 0 or m.d <= 0
                or any(abs(a-v) > .001 for a, v in zip(obj.get_bounds(), bbox))
                or any(abs(a-v) > .001 for a, v in zip(
                    [m.e, m.f, m.a, m.d], [l, b, r-l, t-b]))):
            reasons.append("placement_crop_rotation_or_reflection")
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        # PDFium returns -1 for the non-null empty clip handle on ordinary
        # images. An explicit path is a reason to render; final pixel equality
        # remains mandatory when no clip is reported.
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append("image_clip_path")

    image, method, validation = None, "pdf_render", None
    if strategy == "auto" and not reasons:
        nw, nh = obj.get_px_size()
        if nw*nh > max_source_pixels:
            raise ValueError("Native image exceeds max_source_pixels")
        # Small fractional PDF dimensions can otherwise add a raster row.
        scale = math.nextafter(min(nw/width, nh/height), 0)
        if abs(nw/width-nh/height) > 1e-4*max(nw/width, nh/height):
            reasons.append("anisotropic_native_geometry")
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert("RGB")
            finally:
                # Conversion above owns its data; no PIL view survives closing.
                if hasattr(bitmap, "warn_on_close"):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {"method": "full_pixel_equality_to_visible_render", "equal": equal,
                          "render_size": list(visible.size), "scale": scale}
            if equal:
                image, method = candidate, "native_validated"
            else:
                reasons.append("native_pixels_differ_from_visible_page")
    if image is None:
        # Do not inflate tiny scans just because their nominal page size is small.
        raster_only = bool(images) and all(o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE,pdfium.raw.FPDF_PAGEOBJ_FORM}
                                          for o in objects) and annotations == 0
        scale = density if raster_only and density else max(dpi/72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation["scale"]
    meta = {"version": PDF_IMAGE_VERSION, "method": method, "size": list(image.size),
            "page_size_pdf": [width, height], "page_bbox_pdf": bbox,
            "media_box_pdf": list(page.get_mediabox()), "crop_box_pdf": list(page.get_cropbox()),
            "rotation": rotation, "object_types": [o.type for o in objects],
            "annotations": annotations, "embedded_images": images,
            "requested_dpi": dpi, "render_scale": scale, "native_validation": validation,
            "fallback_reasons": reasons, "source_downsampled": False,
            "source_to_pdf": pdf_pixel_to_page(image.size, bbox, rotation),
            "resolution_unit": "pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable",
            "warnings": (["low_native_resolution; interpolation_cannot_recover_detail"]
                         if images and max(max(i["native_size"]) for i in images) <= 750 else [])}
    return image, meta

def projection_mask(image, options):
    """Analysis-only ink mask; never use these pixels for canonical/OCR crops."""
    from PIL import Image, ImageChops, ImageFilter, ImageOps
    gray = image.convert("L")
    gray.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    # Local background subtraction also works on dark/uneven manuscript paper.
    radius = max(2, min(gray.size) / 80)
    mask = ImageChops.subtract(gray.filter(ImageFilter.GaussianBlur(radius)), gray)
    return ImageOps.autocontrast(mask).point(lambda v: 255 if v > options.get("ink_threshold", 40) else 0)

def projection_gaps(image, axis, options):
    """Find internal low-ink gutters; suppress long rules in analysis only."""
    mask = projection_mask(image, options)
    w, h = mask.size
    data = mask.tobytes()
    # Exclude outer 3% and dense ruling rows/columns from the projection only.
    left, right = max(1, int(w * .03)), min(w - 1, int(w * .97))
    top, bottom = max(1, int(h * .03)), min(h - 1, int(h * .97))
    if left >= right or top >= bottom:
        return []
    rows = [sum(v > 0 for v in data[y*w+left:y*w+right]) for y in range(top, bottom)]
    valid_rows = [y for y, count in zip(range(top, bottom), rows) if count < .7 * (right-left)]
    if not valid_rows:
        return []
    cols = [sum(data[y*w+x] > 0 for y in valid_rows) for x in range(left, right)]
    valid_cols = [x for x, count in zip(range(left, right), cols) if count < .7 * len(valid_rows)]
    if not valid_cols:
        return []
    if axis == "x":
        values = [sum(data[y*w+x] > 0 for y in valid_rows) / len(valid_rows) for x in range(w)]
        extent, length = image.width, w
    else:
        values = [sum(data[y*w+x] > 0 for x in valid_cols) / len(valid_cols) for y in range(h)]
        extent, length = image.height, h
    # Do not let a book frame or ruled separator count as a neighboring column.
    values = [0.0 if value >= .7 else value for value in values]
    threshold = options.get("gap_ink_fraction", .006)
    minimum = max(2, round(length * options.get("min_gap_fraction", .006)))
    # Avoid margins and tiny scraps masquerading as columns.
    edge = max(minimum, round(length * .06))
    gaps, start = [], None
    for i in range(edge, length - edge + 1):
        blank = i < length - edge and values[i] <= threshold
        if blank and start is None:
            start = i
        if not blank and start is not None:
            if i - start >= minimum:
                cut = round((start + i) / 2 * extent / length)
                # Both neighbors must contain meaningful ink.
                span = max(minimum * 3, round(length * .035))
                before = values[max(edge, start-span):start]
                after = values[i:min(length-edge, i+span)]
                support = max(3, round(length * .006))
                if (sum(v > threshold*2 for v in before) >= support
                        and sum(v > threshold*2 for v in after) >= support):
                    gaps.append(cut)
            start = None
    return gaps

def _partition_box(box, cuts, axis, direction):
    x1, y1, x2, y2 = box
    ends = [0, *cuts, x2-x1 if axis == "x" else y2-y1]
    result = ([(x1+a, y1, x1+b, y2) for a, b in zip(ends, ends[1:])] if axis == "x" else
              [(x1, y1+a, x2, y1+b) for a, b in zip(ends, ends[1:])])
    return result[::-1] if axis == "x" and direction == "vertical_rtl" else result

def _sanitize(name: str) -> str:
    return re.sub(r"[^\w\-]+", "_", name).strip("_") or "run"

def parse_pages(spec: str | int, n: int) -> list[int]:
    """'1-10,15' -> [0..9, 14] (đếm từ 1); '' -> mọi trang."""
    spec = str(spec)
    if not spec.strip():
        return list(range(n))
    out: list[int] = []
    for part in spec.split(","):
        a, _, b = part.strip().partition("-")
        out += range(int(a) - 1, min(int(b or a), n))
    return sorted(set(p for p in out if 0 <= p < n))

def _pdf_pages(path: Path) -> int:
    import pypdfium2 as pdfium
    doc = pdfium.PdfDocument(str(path))
    try:
        return len(doc)
    finally:
        doc.close()

def _entry_matches(entry: dict, pattern: str) -> bool:
    return (fnmatch(entry["name"], pattern) or fnmatch(entry["stem"], pattern)
            or fnmatch(Path(entry["path"]).name, pattern))

def select_entries(entries: list[dict], patterns: list[str]) -> list[dict]:
    """Chọn sách theo tên/glob. [] = tất cả. Pattern không khớp -> báo lỗi."""
    if not patterns:
        return entries
    chosen: list[dict] = []
    seen: set[str] = set()
    missing: list[str] = []
    for pat in patterns:
        hits = [e for e in entries if _entry_matches(e, pat)]
        if not hits:
            missing.append(pat)
        for e in hits:
            if e["path"] not in seen:
                chosen.append(e)
                seen.add(e["path"])
    if missing:
        names = ", ".join(e["name"] for e in entries)
        raise ValueError(f"books không khớp: {missing}. Có sẵn: {names}")
    return chosen

def list_pdfs(input_dir: str | Path) -> list[dict]:
    """Liệt kê PDF trong input_dir: name (đã làm sạch), stem, path, pages."""
    root = Path(input_dir)
    if not root.exists():
        raise FileNotFoundError(f"không thấy input_dir: {root}")
    files = [p for p in sorted(root.rglob("*.pdf")) if "__MACOSX" not in str(p)]
    if not files:
        raise FileNotFoundError(f"không thấy PDF nào trong {root}")
    entries: list[dict] = []
    used: set[str] = set()
    for p in files:
        name = _sanitize(p.stem)
        while name in used:
            name += "_"
        used.add(name)
        entries.append({"name": name, "stem": p.stem, "path": str(p)})
    # Sequential document inventory is portable in a single notebook namespace.
    for entry in entries:
        entry['n_pages'] = _pdf_pages(Path(entry['path']))
    return entries

"""Offline, content-addressed image preparation and portable dataset snapshots.

Only PDFium and Pillow are required. JSONL avoids a Parquet/Arrow dependency in
the independent OCR runtimes. Images are stored once, irrespective of how many
logical page/group/column views refer to them. No OCR predictions are stored.
"""


from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import sys
import time



IMAGE_DATASET_FORMAT = "vtm-image-dataset"
IMAGE_DATASET_VERSION = 1
IMAGE_DATASET_CODE_SIGNATURE = "b6c61d984a3de02551d86e043f564786d845c47056f50f31771dfe06749dfbb7"  # The notebook builder supplies its source hash.


@dataclass
class ImageDatasetConfig:
    dpi: int = 200
    acquisition: str = "auto"
    max_source_pixels: int = 100_000_000
    analysis_max_side: int = 1200
    reading_direction: str = "vertical_rtl"
    columns_per_group: int = 3
    max_columns: int = 64
    column_cuts: list[float] | None = None
    page_column_cuts: dict = field(default_factory=dict)
    ink_threshold: int = 40
    gap_ink_fraction: float = .006
    min_gap_fraction: float = .006
    variants: list[str] = field(default_factory=lambda: ["original"])
    variant_options: dict = field(default_factory=dict)

    def __post_init__(self):
        if (self.dpi <= 0 or self.max_source_pixels < 1 or not 32 <= self.analysis_max_side <= 1600
                or self.columns_per_group < 1 or self.max_columns < 1):
            raise ValueError("Invalid preparation dimensions/group limits")
        if self.acquisition not in {"auto", "render"}:
            raise ValueError("acquisition must be auto or render")
        if self.reading_direction not in {"vertical_rtl", "horizontal_ltr"}:
            raise ValueError("Invalid reading_direction")
        if not 0 < self.ink_threshold < 255 or not 0 <= self.gap_ink_fraction < 1 or not 0 < self.min_gap_fraction < .5:
            raise ValueError("Invalid projection thresholds")
        if (not isinstance(self.variants, list) or "original" not in self.variants
                or len(set(self.variants)) != len(self.variants)
                or set(self.variants)-{"original", "bicubic_x2", "clahe"}):
            raise ValueError("variants require original; supported experiments: bicubic_x2, clahe. "
                             "Denoising/SR need separate evidence and are not implemented")
        if set(self.variant_options)-{"clahe"}:
            raise ValueError("Only clahe accepts variant_options")
        for name, options in self.variant_options.items():
            if set(options)-{"clip_limit", "tile_grid"}:
                raise ValueError(f"Unknown {name} parameters")
            if not 0 < options.get("clip_limit", 1.5) <= 4 or not 2 <= options.get("tile_grid", 8) <= 32:
                raise ValueError("CLAHE clip_limit must be (0,4], tile_grid [2,32]")
        dataset_validate_cuts(self.column_cuts)
        for page, cuts in self.page_column_cuts.items():
            if not str(page).isdigit() or int(page) < 1:
                raise ValueError("page_column_cuts uses 1-based page keys")
            dataset_validate_cuts(cuts)


def dataset_json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)+"\n").encode("utf-8")


def dataset_hash(value):
    return hashlib.sha256(dataset_json_bytes(value)).hexdigest()


def dataset_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()


def dataset_code_signature():
    if IMAGE_DATASET_CODE_SIGNATURE:
        return IMAGE_DATASET_CODE_SIGNATURE
    root = Path(__file__).parent
    return dataset_hash({name: dataset_file_hash(root/name) for name in
                         ("image_dataset.py", "image_preparation.py", "pdf_images.py")})


def dataset_environment():
    import PIL
    import pypdfium2 as pdfium
    result = {"pillow": PIL.__version__, "pypdfium2": str(pdfium.PYPDFIUM_INFO),
              "pdfium": str(pdfium.PDFIUM_INFO), "python": sys.version.split()[0]}
    return result


def dataset_path(root, relative):
    relative = str(relative)
    if "\\" in relative or ":" in relative:
        raise ValueError(f"Dataset requires portable relative paths: {relative}")
    parts = PurePosixPath(relative)
    if parts.is_absolute() or ".." in parts.parts or not parts.parts:
        raise ValueError(f"Invalid dataset path: {relative}")
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError(f"Dataset path escapes root: {relative}")
    return path


def dataset_write(root, relative, data):
    path = dataset_path(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Only generated cache objects/indexes are replaced; arbitrary files in the
    # dataset root are neither deleted nor included in packaged snapshots.
    temporary = path.with_name(path.name+".tmp")
    temporary.write_bytes(data)
    temporary.replace(path)
    return {"path": str(relative), "sha256": hashlib.sha256(data).hexdigest()}


def dataset_read(root, reference):
    data = dataset_path(root, reference["path"]).read_bytes()
    if hashlib.sha256(data).hexdigest() != reference["sha256"]:
        raise ValueError(f"Dataset checksum mismatch: {reference['path']}")
    return data


def dataset_store_image(root, image):
    image = image.convert("RGB")
    output = io.BytesIO()
    image.save(output, format="PNG", compress_level=6)
    data = output.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f"objects/{sha[:2]}/{sha}.png"
    path = dataset_path(root, relative)
    if not path.exists() or dataset_file_hash(path) != sha:
        dataset_write(root, relative, data)
    return {"path": relative, "sha256": sha, "size": list(image.size),
            "pixel_sha256": hashlib.sha256(image.tobytes()).hexdigest()}


def dataset_load_image(root, reference):
    from PIL import Image
    with Image.open(io.BytesIO(dataset_read(root, reference))) as loaded:
        image = loaded.convert("RGB")
    if (list(image.size) != reference["size"] or
            hashlib.sha256(image.tobytes()).hexdigest() != reference["pixel_sha256"]):
        raise ValueError(f"Dataset image dimensions/pixels changed: {reference['path']}")
    return image


@contextmanager
def dataset_lock(root):
    """OS lock releases after interruption; no stale PID/lockfile cleanup needed."""
    import os
    Path(root).mkdir(parents=True, exist_ok=True)
    with (Path(root)/"prepare.lock").open("a+b") as stream:
        if stream.tell() == 0:
            stream.write(b"0"); stream.flush()
        stream.seek(0)
        if os.name == "nt":
            import msvcrt
            msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            yield
        finally:
            stream.seek(0)
            if os.name == "nt":
                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream, fcntl.LOCK_UN)


def dataset_validate_cuts(cuts):
    if cuts is not None and (not isinstance(cuts, list) or any(
        isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 < v < 1
        for v in cuts) or cuts != sorted(set(cuts))):
        raise ValueError("column_cuts must be unique increasing fractions in (0,1)")


def dataset_partition_qc(regions, size, direction):
    """Exact geometric partition check, independent of OCR/text completeness."""
    w, h = size
    expected = 0 if direction == "horizontal_ltr" else w
    for index, region in enumerate(regions):
        x1, y1, x2, y2 = region["source_box"]
        if (not all(isinstance(v, int) for v in (x1,y1,x2,y2)) or
                not 0 <= x1 < x2 <= w or (y1,y2) != (0,h) or region["reading_order"] != index):
            raise ValueError("Invalid canonical crop bounds/order")
        start, end = (x1, x2) if direction == "horizontal_ltr" else (x2, x1)
        if start != expected:
            raise ValueError("Canonical partition has a gap, overlap or inconsistent reading order")
        expected = end
    if not regions or expected != (w if direction == "horizontal_ltr" else 0):
        raise ValueError("Canonical partition omits page content")


def dataset_plan_layout(image, config, *, page=0):
    """Projection baseline, no model/pixel budget; all coordinates in source pixels."""
    from PIL import Image, ImageDraw
    analysis = image.copy()
    analysis.thumbnail((config.analysis_max_side, config.analysis_max_side), Image.Resampling.LANCZOS)
    options = {key: getattr(config, key) for key in ("ink_threshold", "gap_ink_fraction", "min_gap_fraction")}
    mask = projection_mask(analysis, options)
    w, h = image.size
    aw, ah = analysis.size
    manual = config.page_column_cuts.get(str(page+1), config.column_cuts)
    if manual is not None:
        cuts = [round(v*w) for v in manual]
        method = "manual_columns"
    elif config.reading_direction == "vertical_rtl":
        cuts = sorted(set(round(v*w/aw) for v in projection_gaps(analysis, "x", options)))
        method = "whitespace_projection"
    else:
        cuts, method = [], "whole_page_horizontal"
    if len(set([0,*cuts,w])) != len(cuts)+2:
        raise ValueError("Column cuts collapse in source space")
    warnings = ["heuristic_layout; geometric_coverage_is_not_text_coverage"]
    if len(cuts)+1 > config.max_columns:
        if manual is not None:
            raise ValueError("Manual layout exceeds max_columns")
        warnings.append("too_many_gutters; retained_whole_page")
        cuts = []
    if not cuts:
        warnings.append("no_partition; whole_page_retained")
    thresholds = sorted(set([max(1,config.ink_threshold-8), config.ink_threshold, min(254,config.ink_threshold+8)]))
    found = [projection_gaps(analysis, "x", {**options, "ink_threshold": t}) for t in thresholds]
    ink = mask.tobytes()
    top, bottom = int(.03*ah), max(1,int(.97*ah))
    projection = [sum(ink[y*aw+x] > 0 for y in range(top,bottom))/(bottom-top) for x in range(aw)]
    evidence = []
    for cut in cuts:
        x = min(aw-1,max(0,round(cut*aw/w)))
        lo, hi = x, x
        while lo > 0 and projection[lo-1] <= config.gap_ink_fraction:
            lo -= 1
        while hi < aw-1 and projection[hi+1] <= config.gap_ink_fraction:
            hi += 1
        tolerance = max(2,round(aw*.006))
        stability = sum(any(abs(c-x) <= tolerance for c in cs) for cs in found)/len(found)
        radius = max(2,round(aw*.035))
        left, right = projection[max(0,x-radius):x], projection[x+1:min(aw,x+radius+1)]
        risks = []
        if projection[x] > .02:
            risks.append("cut_crosses_analysis_ink; rules_or_strokes_need_review")
        if stability < 2/3:
            risks.append("unstable_across_ink_thresholds")
        if not left or not right or min(max(left),max(right)) <= config.gap_ink_fraction*2:
            risks.append("weak_text_support_on_one_side")
        evidence.append({"source_x":cut,"analysis_x":x,"gutter_width_analysis":hi-lo+1,
                         "ink_fraction":projection[x],"threshold_stability":stability,"warnings":risks})
    full = (0,0,w,h)
    columns = _partition_box(full,cuts,"x",config.reading_direction)
    group = config.columns_per_group
    groups = []
    for start in range(0,len(columns),group):
        chunk = columns[start:start+group]
        groups.append((min(b[0] for b in chunk),0,max(b[2] for b in chunk),h))
    regions = []
    for kind, boxes in (("page",[full]),("group",groups),("column",columns)):
        for index, box in enumerate(boxes):
            regions.append({"id":f"{kind}_{index:03d}","crop_type":kind,"reading_order":index,
                            "source_box":list(box),"overlap":False})
        dataset_partition_qc([r for r in regions if r["crop_type"]==kind], image.size, config.reading_direction)
    preview = analysis.copy()
    draw = ImageDraw.Draw(preview)
    for region in [r for r in regions if r["crop_type"]=="column"]:
        box = region["source_box"]
        x1,x2 = round(box[0]*aw/w),round(box[2]*aw/w)
        draw.rectangle((x1,0,max(x1,x2-1),ah-1),outline="red",width=1)
        draw.text((x1+2,3),str(region["reading_order"]+1),fill="red")
    for item in evidence:
        if item["warnings"]:
            draw.line((item["analysis_x"],0,item["analysis_x"],ah-1),fill="orange",width=2)
    draw.text((3,ah-14), "RTL <-" if config.reading_direction=="vertical_rtl" else "LTR ->",fill="red")
    layout = {"method":method,"source_size":[w,h],"analysis_size":[aw,ah],
              "analysis_to_source":[w/aw,0,0,h/ah,0,0],"reading_direction":config.reading_direction,
              "regions":regions,"cuts":evidence,"excluded_regions":[],
              "uncertain_regions":[r["id"] for r in regions if r["crop_type"]=="column"],
              "geometric_coverage":1.0,"text_coverage":"unverified","warnings":warnings,
              "thresholds_tested":thresholds,"review_status":"needs_visual_review"}
    return layout, mask.convert("RGB"), preview


def dataset_variant(image, name, options):
    from PIL import Image
    if name == "original":
        return image, {"method":"identity","parameters":{},"warnings":[]}
    if name == "bicubic_x2":
        return image.resize((image.width*2,image.height*2),Image.Resampling.BICUBIC), {
            "method":"bicubic","parameters":{"scale":2},
            "warnings":["experimental_interpolation; no_recovered_source_detail"]}
    if name == "clahe":
        try:
            import cv2
            import numpy as np
        except ImportError as exc:
            raise RuntimeError("clahe requires an already available OpenCV/NumPy installation; use original offline") from exc
        params = {"clip_limit":1.5,"tile_grid":8,**options}
        lab = cv2.cvtColor(np.asarray(image.convert("RGB")),cv2.COLOR_RGB2LAB)
        grid = int(params["tile_grid"])
        lab[:,:,0] = cv2.createCLAHE(clipLimit=float(params["clip_limit"]),tileGridSize=(grid,grid)).apply(lab[:,:,0])
        return Image.fromarray(cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)), {
            "method":"clahe_lab_l","parameters":params,"opencv":cv2.__version__,
            "warnings":["experimental_contrast; inspect_weak_strokes"]}
    raise ValueError(f"Unsupported image variant: {name}")


def dataset_validate_page(root, record, *, check_pixels=True):
    source = dataset_load_image(root,record["source"]) if check_pixels else None
    acquisition = json.loads(dataset_read(root,record["acquisition"]))
    layout = json.loads(dataset_read(root,record["layout"]))
    if record["source"]["size"] != layout["source_size"] or acquisition["size"] != layout["source_size"]:
        raise ValueError("Source, acquisition and layout size mismatch")
    dataset_read(root,layout["analysis_mask"])
    dataset_read(root,layout["preview"])
    for kind in ("page","group","column"):
        regions = [r for r in layout["regions"] if r["crop_type"]==kind]
        dataset_partition_qc(regions,layout["source_size"],layout["reading_direction"])
    if [r["id"] for r in record["regions"]] != [r["id"] for r in layout["regions"]]:
        raise ValueError("Image dataset omits canonical regions")
    for region, geometry in zip(record["regions"],layout["regions"]):
        if any(region[k] != geometry[k] for k in ("source_box","crop_type","reading_order")):
            raise ValueError("Region geometry differs from canonical layout")
        if "original" not in region["images"]:
            raise ValueError("Region original variant is missing")
        x1,y1,x2,y2 = region["source_box"]
        for name, entry in region["images"].items():
            expected = [(x2-x1)*(2 if name=="bicubic_x2" else 1),(y2-y1)*(2 if name=="bicubic_x2" else 1)]
            if entry["size"] != expected or entry["image_to_source"] != [
                    (x2-x1)/expected[0],0,0,(y2-y1)/expected[1],x1,y1]:
                raise ValueError("Variant geometry/transform mismatch")
            if check_pixels:
                image = dataset_load_image(root,entry)
                if name=="original" and image.tobytes() != source.crop(region["source_box"]).tobytes():
                    raise ValueError("Original crop is not pixel-identical to source")
    return layout


def dataset_open(directory):
    runtime_path = Path
    path = runtime_path(directory)
    descriptor = path/"dataset.json" if path.is_dir() else path
    root = descriptor.parent.parent if descriptor.parent.name=="snapshots" else descriptor.parent
    data = json.loads(descriptor.read_text(encoding="utf-8"))
    if data.get("format") != IMAGE_DATASET_FORMAT or data.get("version") != IMAGE_DATASET_VERSION:
        raise ValueError("Unsupported image dataset format/version")
    body = {k:v for k,v in data.items() if k!="snapshot_id"}
    if dataset_hash(body) != data["snapshot_id"]:
        raise ValueError("Dataset snapshot identity mismatch")
    return root, data


def dataset_page(root, descriptor, source_sha, page):
    entries = [p for p in descriptor["pages"] if p["source_sha256"]==source_sha and p["page"]==page]
    if not entries:
        return None
    records = [json.loads(dataset_read(root,e["record"])) for e in entries]
    if len({r["input_key"] for r in records}) != 1:
        raise ValueError("Ambiguous prepared layouts for the same source/page")
    record = records[0]
    if record["source_sha256"] != source_sha or record["page"] != page:
        raise ValueError("Dataset page identity mismatch")
    return record


def prepare_image_dataset(input_dir, output_dir, *, config=None, books=None, pages="1-2",
                          limit=2, max_books=1, profile_dir=None):
    """Prepare a bounded selection; repeat the same command to resume.

    A completed source/layout/variant is independently reusable. Only after QC
    succeeds is a new dataset.json published. Existing snapshots remain usable.
    """
    config = config or ImageDatasetConfig()
    config.__post_init__()
    if limit is not None and limit < 1 or max_books is not None and max_books < 1:
        raise ValueError("limit/max_books must be positive or None")
    entries = select_entries(list_pdfs(input_dir),books or [])
    if max_books is not None:
        entries = entries[:max_books]
    selected, remaining = [], limit
    for entry in entries:
        wanted = parse_pages(pages,entry["n_pages"])
        if remaining is not None:
            wanted = wanted[:remaining]; remaining -= len(wanted)
        if wanted:
            selected.append((entry,wanted))
    if not selected:
        raise ValueError("No pages selected")
    root = Path(output_dir).resolve()
    stats = {"pages":0,"source_reused":0,"layout_reused":0,"variants_reused":0}
    started, environment, code = time.monotonic(), dataset_environment(), dataset_code_signature()
    page_entries, manifest_rows, configurations = [], [], {}
    with dataset_lock(root):
        marker = root/"canonical.json"
        if marker.exists():
            dataset_open(marker)  # Never replace an unrelated user file.
        for entry, wanted in selected:
            import pypdfium2 as pdfium
            cfg = ImageDatasetConfig(**asdict(config))
            if profile_dir:
                profile = json.loads((Path(profile_dir)/(entry["stem"]+".json")).read_text(encoding="utf-8-sig"))
                # OCR token/resize settings have no place in a canonical dataset.
                for key in ("dpi","reading_direction","column_cuts","page_column_cuts","columns_per_group"):
                    if key in profile:
                        setattr(cfg,key,profile[key])
            cfg.__post_init__()
            configurations[entry["name"]] = asdict(cfg)
            source_sha = dataset_file_hash(entry["path"])
            doc = pdfium.PdfDocument(entry["path"])
            try:
                doc.init_forms()
                for page in wanted:
                    source_key = dataset_hash({"source_sha256":source_sha,"page":page,
                        "acquisition":cfg.acquisition,"dpi":cfg.dpi,"max_source_pixels":cfg.max_source_pixels,
                        "code":code,"environment":environment})
                    source_receipt = f"sources/{source_key}.json"
                    try:
                        cached = json.loads(dataset_path(root,source_receipt).read_bytes())
                        if cached["key"] != source_key:
                            raise ValueError("Source cache key mismatch")
                        image = dataset_load_image(root,cached["source"])
                        acquisition = json.loads(dataset_read(root,cached["acquisition"]))
                        stats["source_reused"] += 1
                    except (OSError,ValueError,KeyError):
                        pdf_page = doc[page]
                        try:
                            image, acquisition = acquire_pdf_image(pdf_page,dpi=cfg.dpi,
                                max_source_pixels=cfg.max_source_pixels,strategy=cfg.acquisition)
                        finally:
                            pdf_page.close()
                        source_asset = dataset_store_image(root,image)
                        acquisition_ref = dataset_write(root,f"sources/{source_key}.acquisition.json",dataset_json_bytes(acquisition))
                        cached = {"key":source_key,"source":source_asset,"acquisition":acquisition_ref}
                        dataset_write(root,source_receipt,dataset_json_bytes(cached))
                    layout_options = {k:v for k,v in asdict(cfg).items() if k not in
                        {"dpi","acquisition","max_source_pixels","variants","variant_options"}}
                    # Only this page's override influences this page's layout cache.
                    layout_options["page_column_cuts"] = {str(page+1):cfg.page_column_cuts.get(str(page+1),cfg.column_cuts)}
                    layout_key = dataset_hash({"source":cached["source"]["sha256"],"options":layout_options,"code":code})
                    layout_relative = f"layouts/{layout_key}.json"
                    try:
                        layout_data = dataset_path(root,layout_relative).read_bytes()
                        layout = json.loads(layout_data)
                        if layout["key"] != layout_key:
                            raise ValueError("Layout key mismatch")
                        dataset_read(root,layout["analysis_mask"]); dataset_read(root,layout["preview"])
                        stats["layout_reused"] += 1
                    except (OSError,ValueError,KeyError):
                        layout, mask, preview = dataset_plan_layout(image,cfg,page=page)
                        layout.update(key=layout_key,analysis_mask=dataset_store_image(root,mask),preview=dataset_store_image(root,preview))
                        layout_data = dataset_json_bytes(layout)
                        dataset_write(root,layout_relative,layout_data)
                    layout_ref = {"path":layout_relative,"sha256":hashlib.sha256(layout_data).hexdigest()}
                    regions = []
                    for geometry in layout["regions"]:
                        crop = image.crop(geometry["source_box"])
                        variants = {}
                        for name in cfg.variants:
                            options = cfg.variant_options.get(name,{})
                            variant_environment = dict(environment)
                            if name=="clahe":
                                import cv2
                                variant_environment["opencv"] = cv2.__version__
                            key = dataset_hash({"source":cached["source"]["sha256"],"box":geometry["source_box"],
                                "variant":name,"options":options,"code":code,"environment":variant_environment})
                            receipt = f"variants/{key}.json"
                            try:
                                variant = json.loads(dataset_path(root,receipt).read_bytes())
                                if variant["key"] != key:
                                    raise ValueError("Variant cache key mismatch")
                                dataset_load_image(root,variant)
                                stats["variants_reused"] += 1
                            except (OSError,ValueError,KeyError):
                                prepared, operation = dataset_variant(crop,name,options)
                                x1,y1,x2,y2 = geometry["source_box"]
                                variant = {**dataset_store_image(root,prepared),"key":key,"operation":operation,
                                    "image_to_source":[(x2-x1)/prepared.width,0,0,(y2-y1)/prepared.height,x1,y1]}
                                dataset_write(root,receipt,dataset_json_bytes(variant))
                            variants[name] = variant
                            manifest_rows.append({"book":entry["name"],"source_pdf":Path(entry["path"]).relative_to(Path(input_dir)).as_posix(),
                                "source_sha256":source_sha,"page":page+1,"page_index":page,
                                "acquisition_method":acquisition["method"],"source_size":cached["source"]["size"],
                                "source_image":cached["source"],"analysis_size":layout["analysis_size"],
                                "analysis_to_source":layout["analysis_to_source"],"source_to_pdf":acquisition["source_to_pdf"],
                                "region_id":geometry["id"],"crop_type":geometry["crop_type"],
                                "source_box":geometry["source_box"],"reading_order":geometry["reading_order"],
                                "reading_direction":cfg.reading_direction,"variant":name,**variant,
                                "qc_warnings":acquisition["warnings"]+layout["warnings"]+
                                    [w for c in layout["cuts"] for w in c["warnings"]],
                                "layout":layout_ref,"acquisition":cached["acquisition"],"code_sha256":code,
                                "processing_version":IMAGE_DATASET_VERSION})
                        regions.append({**geometry,"images":variants})
                    record = {"input_key":dataset_hash({"source":source_key,"layout":layout_key,"variants":cfg.variants,
                                  "options":cfg.variant_options}),"source_sha256":source_sha,"page":page,
                              "source":cached["source"],"acquisition":cached["acquisition"],"layout":layout_ref,"regions":regions}
                    dataset_validate_page(root,record)
                    record_bytes = dataset_json_bytes(record)
                    record_key = hashlib.sha256(record_bytes).hexdigest()
                    record_ref = dataset_write(root,f"pages/{record_key}.json",record_bytes)
                    page_entries.append({"book":entry["name"],"source_pdf":Path(entry["path"]).relative_to(Path(input_dir)).as_posix(),
                                         "source_sha256":source_sha,"page":page,"record":record_ref})
                    stats["pages"] += 1
            finally:
                doc.close()
        manifest_data = b"".join(dataset_json_bytes(row) for row in manifest_rows)
        manifest_sha = hashlib.sha256(manifest_data).hexdigest()
        manifest_ref = dataset_write(root,f"manifests/{manifest_sha}.jsonl",manifest_data)
        descriptor = {"format":IMAGE_DATASET_FORMAT,"version":IMAGE_DATASET_VERSION,"pages":page_entries,
            "expected_pages":[[e["book"],e["source_sha256"],e["page"]] for e in page_entries],
            "manifest":manifest_ref,"configurations":configurations,"environment":environment,"code_sha256":code,
            "geometry":"top_left_source_pixels; half_open_boxes", "ocr_text_coverage":"unverified"}
        descriptor["snapshot_id"] = dataset_hash(descriptor)
        data = dataset_json_bytes(descriptor)
        snapshot = f"snapshots/{descriptor['snapshot_id']}.json"
        dataset_write(root,snapshot,data)
        # Publish last: a crash/failure before this point leaves the last usable
        # snapshot untouched and all finished stages available for resume.
        dataset_write(root,"canonical.json",data)
    stats.update(snapshot_id=descriptor["snapshot_id"],dataset=str(root/snapshot),
                 seconds=round(time.monotonic()-started,3),manifest_rows=len(manifest_rows))
    return stats


def validate_image_dataset(directory):
    root, descriptor = dataset_open(directory)
    dataset_read(root,descriptor["manifest"])
    actual = [[e["book"],e["source_sha256"],e["page"]] for e in descriptor["pages"]]
    if actual != descriptor["expected_pages"] or len(set(map(tuple,actual))) != len(actual):
        raise ValueError("Dataset expected-page coverage mismatch")
    warnings = []
    for entry in descriptor["pages"]:
        record = dataset_page(root,descriptor,entry["source_sha256"],entry["page"])
        layout = dataset_validate_page(root,record)
        warnings.extend(f"{entry['book']}/p{entry['page']+1}: {w}" for w in layout["warnings"])
        warnings.extend(f"{entry['book']}/p{entry['page']+1}/x{c['source_x']}: {w}"
                        for c in layout["cuts"] for w in c["warnings"])
    return {"pages":len(actual),"snapshot_id":descriptor["snapshot_id"],"warnings":warnings,
            "geometric_coverage":1.0,"ocr_text_coverage":"unverified"}



## 3. Mã upstream, giấy phép và worker độc lập

Ô dữ liệu sau chứa **mã nguyên bản** TADiSR (MIT), DiffTSR (Apache-2.0) và UniHIR
(CC BY-NC-ND 4.0, nghiên cứu phi thương mại), kèm revision và thông báo license.
UniHIR giữ mã nguyên trạng; wrapper là file riêng. Không coi bundle này là tài
sản không hạn chế phân phối. Trước khi chia sẻ asset, kiểm tra điều khoản của
adapter/base/font; không tự upload. Kiến trúc RRDB x4plus trong worker lấy từ
BasicSR (Apache-2.0), strict checkpoint; không phụ thuộc BasicSR training/GFPGAN.

Worker/source chỉ được ghi từ các chuỗi trong notebook ra thư mục runtime.
Mỗi thời điểm một model; process thoát để giải phóng GPU. TADiSR dùng API
`load_cogview4` và decoder upstream; tiling wrapper tích lũy CPU, kiểm tra trọng
số phủ toàn ảnh, sửa trường hợp một chiều ngắn hơn tile. Mỗi region có seed riêng.


In [ ]:
UPSTREAM_REVISIONS = {'TADiSR': '9d910f8f67224fae2f57e3d63d68396eebc0b671', 'DiffTSR': 'f9839633f5dbcd7cd80ed0a7609edd6ff5d3b5f5', 'UniHIR': 'f67fa3f36063bf79b957e617f915b584da543953', 'Real-ESRGAN': 'a4abfb2979a7bbff3f69f58f58ae324608821e27'}
VENDOR_SOURCES = {'TADiSR': {'LICENSE': 'MIT License\n\nCopyright (c) 2025 Qiming Hu and contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'README.md': '# TADiSR: Text-Aware Real-World Image Super-Resolution via Diffusion Model with Joint Segmentation Decoders\n\n<p align="center">\n  <a href="https://arxiv.org/abs/2506.04641">&#128196; Paper</a> |\n  <a href="https://github.com/mingcv/TADiSR">&#128187; Code</a> |\n  <a href="#visual-results">&#128444;&#65039; Visual Results</a> |\n  <a href="https://huggingface.co/huqiming513/TADiSR-Models">&#129303; Models &amp; Data</a>\n</p>\n\n<p align="center">\n  Qiming Hu, Linlong Fan, Yiyan Luo, Yuhang Yu, Xiaojie Guo, Qingnan Fan\n  <br>\n  Tianjin University and vivo Mobile Communication Co. Ltd\n</p>\n\n<p align="center">\n  <img src="assets/figure_architecture.jpg" width="96%" alt="TADiSR architecture overview">\n  <br>\n  <sub>Architecture of text-aware cross-attention and joint segmentation decoders.</sub>\n</p>\n\nTADiSR is a diffusion-based framework for 4x real-world image super-resolution\nthat explicitly protects scene text. It fine-tunes the diffusion transformer\'s\ncross-attention with LoRA and jointly decodes the super-resolved image and a\nfull-image text mask. This avoids OCR-box cropping and fusion, so the model\nhandles long, vertical, bilingual, and irregular text in one pass.\n\nThe release includes the paper\'s Kolors model and a CogView4-6B variant. The\nCogView4 model is the recommended public checkpoint for Chinese text because\nits base model has stronger Chinese text generation ability.\n\n## Method Overview\n\nTADiSR has three coupled components:\n\n- **Text-aware cross-attention.** LoRA updates the diffusion transformer\'s\n  attention layers so their spatial response is more sensitive to text regions.\n- **Joint segmentation decoders.** A dual-stream decoder consumes denoised VAE\n  features and produces the SR image and a text-mask logit map. Cross-stream\n  interaction shares text structure with image reconstruction at every scale.\n- **FTSR synthesis.** Fine-grained bilingual text masks and restored text crops\n  are composited on high-quality backgrounds; training then mixes this 50k\n  synthetic set with filtered, paired Real-CE samples.\n\nThe paper trains a 4x model with a fixed diffusion timestep of 200 and combines\npixel, LPIPS, OCR-region edge, segmentation, Dice, focal, and mask-guided\nreconstruction losses.\n\n## Visual Results\n\n<p align="center">\n  <img src="assets/figure3_comparison.jpg" width="96%" alt="Figure 3: qualitative comparison on Chinese document text">\n  <br>\n  <sub>Fig. 3. Qualitative comparison on Chinese document text. TADiSR restores both the text strokes and the surrounding structure more faithfully.</sub>\n</p>\n\n<p align="center">\n  <img src="assets/figure4_comparison.jpg" width="96%" alt="Figure 4: qualitative comparison on road signs">\n  <br>\n  <sub>Fig. 4. Qualitative comparison on road-sign text. TADiSR preserves Chinese characters and Roman letters under real-world blur.</sub>\n</p>\n\n## Paper Results\n\nThe following results are reported for the paper\'s Kolors-based TADiSR model at\n4x magnification. `OCR-A` is the OCR recognition accuracy evaluated on\ncorresponding text regions.\n\n| Benchmark | PSNR | SSIM | LPIPS | FID | OCR-A |\n| --- | ---: | ---: | ---: | ---: | ---: |\n| FTSR-TE | 25.49 | 0.736 | 0.152 | 32.13 | 0.662 |\n| Real-CE-val (aligned) | 24.02 | 0.829 | 0.100 | 38.01 | 0.882 |\n\n## Model Zoo\n\n| Model | Base model | Training data | Size | Checkpoint|\n| --- | --- | --- | ---: | --- |\n| TADiSR-Kolors | [Kwai-Kolors/Kolors](https://huggingface.co/Kwai-Kolors/Kolors) | FTSR | 187 MB | [ckpt](https://huggingface.co/huqiming513/TADiSR-Models/blob/main/tadisr_kolors_ftsr_526000.pkl)  |\n| TADiSR-CogView4-RealCE | [zai-org/CogView4-6B](https://huggingface.co/zai-org/CogView4-6B) | FTSR + Real-CE | 749 MB | [ckpt](https://huggingface.co/huqiming513/TADiSR-Models/blob/main/tadisr_cogview4_realce_17500.pkl) |\n\nThe complete file names, byte sizes, and full checksums are in\n[`checkpoints/manifest.json`](checkpoints/manifest.json). Adapter checkpoints\nare deliberately stored outside Git; base-model licenses continue to apply.\n\n## Installation\n\n```bash\nconda create -n tadisr python=3.10 -y\nconda activate tadisr\n\n# Optional: use the Tsinghua PyPI mirror for project dependencies.\npip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple\n\n# GPU (CUDA 12.1): use the official PyTorch wheel index.\npip install torch torchvision --index-url https://download.pytorch.org/whl/cu121\npip install -r requirements.txt\n```\n\nFor a CPU-only environment, replace the PyTorch command with:\n\n```bash\npip install torch torchvision --index-url https://download.pytorch.org/whl/cpu\n```\n\nUse `--timeout 86400` with `pip install` on unreliable connections. The\nrepository\'s `requirements.txt` includes all runtime imports, including the\nChatGLM tokenizer dependency `sentencepiece`.\n\nThe CogView4 inference implementation requires a recent Diffusers build with\n`CogView4Transformer2DModel`. We tested the code contract against Diffusers\n`>=0.32.0`; install the current stable release if an older environment lacks\nthat class.\n\n## Download Models\n\nThe base model and TADiSR adapter are separate downloads.\n\n```bash\n# CogView4 base model: approximately 31 GB.\npython scripts/download_base_model.py --model cogview4 --output-dir weights/CogView4\n\n# Download tadisr_cogview4_realce_17500.pkl from huqiming513/TADiSR-Models on Hugging Face.\n# Then verify it before GPU construction.\npython scripts/validate_checkpoint.py \\\n  --checkpoint checkpoints/tadisr_cogview4_realce_17500.pkl \\\n  --variant cogview4 --strict-decoder\n\npython scripts/verify_checksums.py \\\n  --checkpoint checkpoints/tadisr_cogview4_realce_17500.pkl \\\n  --model tadisr-cogview4-realce\n\n# Create the deterministic quality-prompt embedding required by the released model.\npython scripts/prepare_prompt_embeddings.py \\\n  --base-model weights/CogView4 \\\n  --output weights/CogView4/saved_prompt_tokens_nocfg.pt\n```\n\nThe original Kolors base model is approximately 29 GB. Its adapter is included\nin the model release for archival and compatibility; the maintained public\ninference command in this repository targets the CogView4 release.\n\n## Inference\n\nRun one image. Input is bicubic-upsampled by 4x, then processed with weighted,\noverlapping tiles to avoid seams. The command writes `sr.png` and\n`text_mask.png`.\n\n```bash\npython scripts/infer.py \\\n  --input /path/to/your_lr_image.png \\\n  --output-dir results/example \\\n  --checkpoint checkpoints/tadisr_cogview4_realce_17500.pkl \\\n  --base-model weights/CogView4 \\\n  --prompt-embeddings weights/CogView4/saved_prompt_tokens_nocfg.pt \\\n  --scale 4 --tile 768 --overlap 256\n```\n\nUse a CUDA GPU. CogView4-6B needs substantially more memory than the 8 GB\nconsumer-GPU class for this unoffloaded implementation; use a 24 GB or larger\nGPU for practical 768px tiles, or lower `--tile` while testing. The output mask\ncontains sigmoid probabilities in grayscale, where brighter values indicate\ntext regions.\n\n## Training\n\nThe original training scripts are retained for experiment reproduction:\n\n```bash\npip install -r requirements-train.txt\n\naccelerate launch scripts/train/train_cogview4.py \\\n  --train_folders /path/to/FTSR \\\n  --pretrained_model_name_or_path weights/CogView4 \\\n  --prompt_embeddings weights/CogView4/saved_prompt_tokens_nocfg.pt \\\n  --output_dir output/TADiSR/TADiSR_CogView4\n\naccelerate launch scripts/train/train_cogview4_realce.py \\\n  --train_folders /path/to/FTSR \\\n  --test_folder /path/to/RealCE \\\n  --realce_train_list /path/to/RealCE/aligned_list_train.txt \\\n  --realce_eval_list /path/to/RealCE/aligned_list_eval.txt \\\n  --pretrained_model_name_or_path weights/CogView4 \\\n  --prompt_embeddings weights/CogView4/saved_prompt_tokens_nocfg.pt \\\n  --output_dir output/TADiSR/TADiSR_CogView4_RealCE\n```\n\n## Repository Map\n\n| Path | Purpose |\n| --- | --- |\n| `tadisr/checkpoint.py` | CPU-safe checkpoint inspection and compatibility contract |\n| `tadisr/tiling.py` | Overlap-aware, weighted tiled inference |\n| `tadisr/inference.py` | Public CogView4 model-loading and image I/O API |\n| `tadisr/kolors_decoder.py` | Exact Kolors FTSR mask-decoder architecture |\n| `tadisr/pipelines.py` | Diffusion/VAE and joint segmentation implementation |\n| `tadisr/training/` | Training datasets, losses, metrics, and visualizations |\n| `scripts/infer.py` | Single-image CLI inference |\n| `scripts/train/` | FTSR and FTSR + Real-CE training entry points |\n| `third_party/ppocr/` | OCR implementation used only by training |\n| `scripts/validate_checkpoint.py` | State-dict structure validation |\n| `scripts/verify_checksums.py` | SHA256 and byte-size verification against the release manifest |\n| `checkpoints/manifest.json` | Release filenames, sizes, and SHA256 checksums |\n\n## License and Acknowledgements\n\nThe code in this repository is released under the [MIT License](LICENSE).\nReleased adapters are for research use and remain subject to the licenses of\ntheir base models. In particular, consult the Kolors license for commercial-use\nterms. This implementation builds on [Diffusers](https://github.com/huggingface/diffusers),\n[PEFT](https://github.com/huggingface/peft), CogView4, Kolors, and PP-OCR.\nThe figures in `assets/` are reproduced from the accompanying paper.\n\n## Citation\n\n```bibtex\n@article{hu2026text,\n  title={Text-aware real-world image super-resolution via diffusion model with joint segmentation decoders},\n  author={Hu, Qiming and Fan, Linlong and Luo, Yiyan and Yu, Yuhang and Guo, Xiaojie and Fan, Qingnan},\n  journal={Advances in Neural Information Processing Systems},\n  volume={38},\n  pages={61522--61543},\n  year={2026}\n}\n```\n', 'checkpoints/manifest.json': '{\n  "format_version": 1,\n  "models": [\n    {\n      "id": "tadisr-cogview4-realce",\n      "variant": "cogview4",\n      "base_model": "zai-org/CogView4-6B",\n      "filename": "tadisr_cogview4_realce_17500.pkl",\n      "bytes": 785453495,\n      "sha256": "7cec8d0917305037cbafcb1b1bf81062add1613b561f5d7e2d1e599fe0baf5e0",\n      "source_checkpoint": "output/TADiSR/TADiSR_CogView4_RealCE/checkpoints/model_17500.pkl"\n    },\n    {\n      "id": "tadisr-kolors-ftsr",\n      "variant": "kolors",\n      "base_model": "Kwai-Kolors/Kolors",\n      "filename": "tadisr_kolors_ftsr_526000.pkl",\n      "bytes": 195866454,\n      "sha256": "4baef8b1fad319caf7d606c82539035c6fc9483c81f94601377c64813251e7f7",\n      "source_checkpoint": "external/Kolors_VAE_Lora_FTSR_Heatmap_Fine_S1_Focal_768_OCR/checkpoints/model_526000.pkl"\n    }\n  ]\n}\n', 'requirements.txt': '# Install PyTorch matching your CUDA runtime first: https://pytorch.org/get-started/locally/\naccelerate>=1.2.0\ndiffusers>=0.32.0\nhuggingface_hub>=0.27.0\nnumpy>=1.26.0\npeft>=0.14.0\nPillow>=10.0.0\nsafetensors>=0.4.5\nsentencepiece>=0.2.0\ntransformers>=4.48.0\ntqdm>=4.66.0\n', 'scripts/prepare_prompt_embeddings.py': '#!/usr/bin/env python\n"""Pre-compute the fixed quality-prompt embedding used by TADiSR CogView4."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nREPOSITORY_ROOT = Path(__file__).resolve().parents[1]\nif str(REPOSITORY_ROOT) not in sys.path:\n    sys.path.insert(0, str(REPOSITORY_ROOT))\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--base-model", required=True, help="Local zai-org/CogView4-6B directory.")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--device", default="cuda")\n    parser.add_argument(\n        "--prompt",\n        default=(\n            "Cinematic, High Contrast, highly detailed, taken using a Canon EOS R camera, "\n            "hyper detailed photo - realistic maximum detail, 32k, Color Grading, ultra HD, "\n            "extreme meticulous detailing, skin pore detailing, hyper sharpness, "\n            "perfect without deformations,text"\n        ),\n    )\n    parser.add_argument(\n        "--negative-prompt",\n        default="blurring, dirty, messy, worst quality, low quality, frames, watermark, signature, jpeg artifacts, deformed, lowres, over-smooth",\n    )\n    args = parser.parse_args()\n\n    from tadisr.pipelines import CogView4TextEncoderPipeline\n\n    encoder = CogView4TextEncoderPipeline(ckpt_dir=args.base_model, device=args.device)\n    encoder.prepare_text_embed(args.prompt, args.negative_prompt, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'tadisr/__init__.py': '"""TADiSR: Text-Aware Diffusion Image Super-Resolution."""\n\nfrom .checkpoint import CheckpointReport, inspect_checkpoint\n\n__all__ = ["CheckpointReport", "inspect_checkpoint"]\n__version__ = "0.1.0"\n', 'tadisr/chatglm_tokenizer.py': 'import json\nimport os\nimport re\nfrom typing import List, Optional, Union, Dict\nfrom sentencepiece import SentencePieceProcessor\nfrom transformers import PreTrainedTokenizer\nfrom transformers.utils import logging, PaddingStrategy\nfrom transformers.tokenization_utils_base import EncodedInput, BatchEncoding\n\n\nclass SPTokenizer:\n    def __init__(self, model_path: str):\n        # reload tokenizer\n        assert os.path.isfile(model_path), model_path\n        self.sp_model = SentencePieceProcessor(model_file=model_path)\n\n        # BOS / EOS token IDs\n        self.n_words: int = self.sp_model.vocab_size()\n        self.bos_id: int = self.sp_model.bos_id()\n        self.eos_id: int = self.sp_model.eos_id()\n        self.pad_id: int = self.sp_model.unk_id()\n        assert self.sp_model.vocab_size() == self.sp_model.get_piece_size()\n\n        role_special_tokens = ["<|system|>", "<|user|>", "<|assistant|>", "<|observation|>"]\n        special_tokens = ["[MASK]", "[gMASK]", "[sMASK]", "sop", "eop"] + role_special_tokens\n        self.special_tokens = {}\n        self.index_special_tokens = {}\n        for token in special_tokens:\n            self.special_tokens[token] = self.n_words\n            self.index_special_tokens[self.n_words] = token\n            self.n_words += 1\n        self.role_special_token_expression = "|".join([re.escape(token) for token in role_special_tokens])\n\n    def tokenize(self, s: str, encode_special_tokens=False):\n        if encode_special_tokens:\n            last_index = 0\n            t = []\n            for match in re.finditer(self.role_special_token_expression, s):\n                if last_index < match.start():\n                    t.extend(self.sp_model.EncodeAsPieces(s[last_index:match.start()]))\n                t.append(s[match.start():match.end()])\n                last_index = match.end()\n            if last_index < len(s):\n                t.extend(self.sp_model.EncodeAsPieces(s[last_index:]))\n            return t\n        else:\n            return self.sp_model.EncodeAsPieces(s)\n\n    def encode(self, s: str, bos: bool = False, eos: bool = False) -> List[int]:\n        assert type(s) is str\n        t = self.sp_model.encode(s)\n        if bos:\n            t = [self.bos_id] + t\n        if eos:\n            t = t + [self.eos_id]\n        return t\n\n    def decode(self, t: List[int]) -> str:\n        text, buffer = "", []\n        for token in t:\n            if token in self.index_special_tokens:\n                if buffer:\n                    text += self.sp_model.decode(buffer)\n                    buffer = []\n                text += self.index_special_tokens[token]\n            else:\n                buffer.append(token)\n        if buffer:\n            text += self.sp_model.decode(buffer)\n        return text\n\n    def decode_tokens(self, tokens: List[str]) -> str:\n        text = self.sp_model.DecodePieces(tokens)\n        return text\n\n    def convert_token_to_id(self, token):\n        """ Converts a token (str) in an id using the vocab. """\n        if token in self.special_tokens:\n            return self.special_tokens[token]\n        return self.sp_model.PieceToId(token)\n\n    def convert_id_to_token(self, index):\n        """Converts an index (integer) in a token (str) using the vocab."""\n        if index in self.index_special_tokens:\n            return self.index_special_tokens[index]\n        if index in [self.eos_id, self.bos_id, self.pad_id] or index < 0:\n            return ""\n        return self.sp_model.IdToPiece(index)\n\n\nclass ChatGLMTokenizer(PreTrainedTokenizer):\n    vocab_files_names = {"vocab_file": "tokenizer.model"}\n\n    model_input_names = ["input_ids", "attention_mask", "position_ids"]\n\n    def __init__(self, vocab_file, padding_side="left", clean_up_tokenization_spaces=False, encode_special_tokens=False,\n                 **kwargs):\n        self.name = "GLMTokenizer"\n\n        self.vocab_file = vocab_file\n        self.tokenizer = SPTokenizer(vocab_file)\n        self.special_tokens = {\n            "<bos>": self.tokenizer.bos_id,\n            "<eos>": self.tokenizer.eos_id,\n            "<pad>": self.tokenizer.pad_id\n        }\n        self.encode_special_tokens = encode_special_tokens\n        super().__init__(padding_side=padding_side, clean_up_tokenization_spaces=clean_up_tokenization_spaces,\n                         encode_special_tokens=encode_special_tokens,\n                         **kwargs)\n\n    def get_command(self, token):\n        if token in self.special_tokens:\n            return self.special_tokens[token]\n        assert token in self.tokenizer.special_tokens, f"{token} is not a special token for {self.name}"\n        return self.tokenizer.special_tokens[token]\n\n    @property\n    def unk_token(self) -> str:\n        return "<unk>"\n\n    @property\n    def pad_token(self) -> str:\n        return "<unk>"\n\n    @property\n    def pad_token_id(self):\n        return self.get_command("<pad>")\n\n    @property\n    def eos_token(self) -> str:\n        return "</s>"\n\n    @property\n    def eos_token_id(self):\n        return self.get_command("<eos>")\n\n    @property\n    def vocab_size(self):\n        return self.tokenizer.n_words\n\n    def get_vocab(self):\n        """ Returns vocab as a dict """\n        vocab = {self._convert_id_to_token(i): i for i in range(self.vocab_size)}\n        vocab.update(self.added_tokens_encoder)\n        return vocab\n\n    def _tokenize(self, text, **kwargs):\n        return self.tokenizer.tokenize(text, encode_special_tokens=self.encode_special_tokens)\n\n    def _convert_token_to_id(self, token):\n        """ Converts a token (str) in an id using the vocab. """\n        return self.tokenizer.convert_token_to_id(token)\n\n    def _convert_id_to_token(self, index):\n        """Converts an index (integer) in a token (str) using the vocab."""\n        return self.tokenizer.convert_id_to_token(index)\n\n    def convert_tokens_to_string(self, tokens: List[str]) -> str:\n        return self.tokenizer.decode_tokens(tokens)\n\n    def save_vocabulary(self, save_directory, filename_prefix=None):\n        """\n        Save the vocabulary and special tokens file to a directory.\n\n        Args:\n            save_directory (`str`):\n                The directory in which to save the vocabulary.\n            filename_prefix (`str`, *optional*):\n                An optional prefix to add to the named of the saved files.\n\n        Returns:\n            `Tuple(str)`: Paths to the files saved.\n        """\n        if os.path.isdir(save_directory):\n            vocab_file = os.path.join(\n                save_directory, self.vocab_files_names["vocab_file"]\n            )\n        else:\n            vocab_file = save_directory\n\n        with open(self.vocab_file, \'rb\') as fin:\n            proto_str = fin.read()\n\n        with open(vocab_file, "wb") as writer:\n            writer.write(proto_str)\n\n        return (vocab_file,)\n\n    def get_prefix_tokens(self):\n        prefix_tokens = [self.get_command("[gMASK]"), self.get_command("sop")]\n        return prefix_tokens\n\n    def build_single_message(self, role, metadata, message):\n        assert role in ["system", "user", "assistant", "observation"], role\n        role_tokens = [self.get_command(f"<|{role}|>")] + self.tokenizer.encode(f"{metadata}\\n")\n        message_tokens = self.tokenizer.encode(message)\n        tokens = role_tokens + message_tokens\n        return tokens\n\n    def build_chat_input(self, query, history=None, role="user"):\n        if history is None:\n            history = []\n        input_ids = []\n        for item in history:\n            content = item["content"]\n            if item["role"] == "system" and "tools" in item:\n                content = content + "\\n" + json.dumps(item["tools"], indent=4, ensure_ascii=False)\n            input_ids.extend(self.build_single_message(item["role"], item.get("metadata", ""), content))\n        input_ids.extend(self.build_single_message(role, "", query))\n        input_ids.extend([self.get_command("<|assistant|>")])\n        return self.batch_encode_plus([input_ids], return_tensors="pt", is_split_into_words=True)\n\n    def build_inputs_with_special_tokens(\n            self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None\n    ) -> List[int]:\n        """\n        Build model inputs from a sequence or a pair of sequence for sequence classification tasks by concatenating and\n        adding special tokens. A BERT sequence has the following format:\n\n        - single sequence: `[CLS] X [SEP]`\n        - pair of sequences: `[CLS] A [SEP] B [SEP]`\n\n        Args:\n            token_ids_0 (`List[int]`):\n                List of IDs to which the special tokens will be added.\n            token_ids_1 (`List[int]`, *optional*):\n                Optional second list of IDs for sequence pairs.\n\n        Returns:\n            `List[int]`: List of [input IDs](../glossary#input-ids) with the appropriate special tokens.\n        """\n        prefix_tokens = self.get_prefix_tokens()\n        token_ids_0 = prefix_tokens + token_ids_0\n        if token_ids_1 is not None:\n            token_ids_0 = token_ids_0 + token_ids_1 + [self.get_command("<eos>")]\n        return token_ids_0\n\n    def _pad(\n            self,\n            encoded_inputs: Union[Dict[str, EncodedInput], BatchEncoding],\n            max_length: Optional[int] = None,\n            padding_strategy: PaddingStrategy = PaddingStrategy.DO_NOT_PAD,\n            pad_to_multiple_of: Optional[int] = None,\n            return_attention_mask: Optional[bool] = None,\n    ) -> dict:\n        """\n        Pad encoded inputs (on left/right and up to predefined length or max length in the batch)\n\n        Args:\n            encoded_inputs:\n                Dictionary of tokenized inputs (`List[int]`) or batch of tokenized inputs (`List[List[int]]`).\n            max_length: maximum length of the returned list and optionally padding length (see below).\n                Will truncate by taking into account the special tokens.\n            padding_strategy: PaddingStrategy to use for padding.\n\n                - PaddingStrategy.LONGEST Pad to the longest sequence in the batch\n                - PaddingStrategy.MAX_LENGTH: Pad to the max length (default)\n                - PaddingStrategy.DO_NOT_PAD: Do not pad\n                The tokenizer padding sides are defined in self.padding_side:\n\n                    - \'left\': pads on the left of the sequences\n                    - \'right\': pads on the right of the sequences\n            pad_to_multiple_of: (optional) Integer if set will pad the sequence to a multiple of the provided value.\n                This is especially useful to enable the use of Tensor Core on NVIDIA hardware with compute capability\n                `>= 7.5` (Volta).\n            return_attention_mask:\n                (optional) Set to False to avoid returning attention mask (default: set to model specifics)\n        """\n        # Load from model defaults\n        assert self.padding_side == "left"\n\n        required_input = encoded_inputs[self.model_input_names[0]]\n        seq_length = len(required_input)\n\n        if padding_strategy == PaddingStrategy.LONGEST:\n            max_length = len(required_input)\n\n        if max_length is not None and pad_to_multiple_of is not None and (max_length % pad_to_multiple_of != 0):\n            max_length = ((max_length // pad_to_multiple_of) + 1) * pad_to_multiple_of\n\n        needs_to_be_padded = padding_strategy != PaddingStrategy.DO_NOT_PAD and len(required_input) != max_length\n\n        # Initialize attention mask if not present.\n        if "attention_mask" not in encoded_inputs:\n            encoded_inputs["attention_mask"] = [1] * seq_length\n\n        if "position_ids" not in encoded_inputs:\n            encoded_inputs["position_ids"] = list(range(seq_length))\n\n        if needs_to_be_padded:\n            difference = max_length - len(required_input)\n\n            if "attention_mask" in encoded_inputs:\n                encoded_inputs["attention_mask"] = [0] * difference + encoded_inputs["attention_mask"]\n            if "position_ids" in encoded_inputs:\n                encoded_inputs["position_ids"] = [0] * difference + encoded_inputs["position_ids"]\n            encoded_inputs[self.model_input_names[0]] = [self.pad_token_id] * difference + required_input\n\n        return encoded_inputs\n', 'tadisr/checkpoint.py': '"""Portable checkpoint inspection and compatibility checks."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\n\n@dataclass(frozen=True)\nclass CheckpointReport:\n    """A serializable summary of a TADiSR checkpoint."""\n\n    path: str\n    variant: str\n    bytes: int\n    groups: dict[str, int]\n    valid: bool\n    errors: tuple[str, ...]\n\n    def to_dict(self) -> dict[str, object]:\n        return asdict(self)\n\n\n_REQUIRED_GROUPS = {\n    "cogview4": ("state_dict_transformer", "state_dict_vae", "js_decoder"),\n    "kolors": ("state_dict_unet", "state_dict_vae"),\n}\n\n\ndef _load(path: Path) -> Mapping[str, Any]:\n    try:\n        import torch\n    except ImportError as error:\n        raise RuntimeError(\n            "PyTorch is required to inspect TADiSR checkpoints. Install it before running this command."\n        ) from error\n\n    # `weights_only` prevents arbitrary pickled code from executing. It is supported\n    # in modern PyTorch; the fallback keeps compatibility with older releases.\n    try:\n        state = torch.load(path, map_location="cpu", weights_only=True)\n    except TypeError:\n        state = torch.load(path, map_location="cpu")\n    if not isinstance(state, Mapping):\n        raise ValueError(f"Expected a mapping checkpoint, received {type(state).__name__}.")\n    return state\n\n\ndef detect_variant(state: Mapping[str, Any]) -> str:\n    keys = set(state)\n    if set(_REQUIRED_GROUPS["cogview4"]).issubset(keys):\n        return "cogview4"\n    if set(_REQUIRED_GROUPS["kolors"]).issubset(keys):\n        return "kolors"\n    return "unknown"\n\n\ndef inspect_checkpoint(path: str | Path, variant: str = "auto") -> CheckpointReport:\n    """Load a checkpoint on CPU and validate the public TADiSR state-dict contract."""\n    checkpoint_path = Path(path).expanduser().resolve()\n    if not checkpoint_path.is_file():\n        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")\n\n    state = _load(checkpoint_path)\n    detected = detect_variant(state)\n    selected = detected if variant == "auto" else variant\n    if selected not in _REQUIRED_GROUPS:\n        raise ValueError(f"Unsupported variant: {selected}. Choose auto, cogview4, or kolors.")\n\n    errors: list[str] = []\n    if variant != "auto" and detected != variant:\n        errors.append(f"Requested {variant}, but checkpoint layout is {detected}.")\n\n    groups: dict[str, int] = {}\n    for key in _REQUIRED_GROUPS[selected]:\n        value = state.get(key)\n        if not isinstance(value, Mapping):\n            errors.append(f"Missing or invalid state-dict group: {key}.")\n            groups[key] = 0\n            continue\n        groups[key] = len(value)\n        if not value:\n            errors.append(f"State-dict group is empty: {key}.")\n\n    if selected == "cogview4":\n        transformer = state.get("state_dict_transformer", {})\n        if isinstance(transformer, Mapping) and not any("lora" in name for name in transformer):\n            errors.append("CogView4 transformer group contains no LoRA weights.")\n        decoder = state.get("js_decoder", {})\n        if isinstance(decoder, Mapping) and not any(name.startswith("out_block") for name in decoder):\n            errors.append("Joint segmentation decoder does not contain its output head.")\n    elif selected == "kolors":\n        unet = state.get("state_dict_unet", {})\n        if isinstance(unet, Mapping) and not any("lora" in name for name in unet):\n            errors.append("Kolors U-Net group contains no LoRA weights.")\n        decoder_key = "js_decoder" if "js_decoder" in state else "mi_decoder"\n        decoder = state.get(decoder_key)\n        if not isinstance(decoder, Mapping) or not decoder:\n            errors.append("Kolors checkpoint is missing its joint segmentation decoder.")\n            groups[decoder_key] = 0\n        else:\n            groups[decoder_key] = len(decoder)\n\n    return CheckpointReport(\n        path=str(checkpoint_path),\n        variant=selected,\n        bytes=checkpoint_path.stat().st_size,\n        groups=groups,\n        valid=not errors,\n        errors=tuple(errors),\n    )\n\n\ndef strict_load_joint_decoder(path: str | Path, variant: str = "auto") -> None:\n    """Construct the released joint decoder and require an exact state-dict match.\n\n    This checks the adapter-specific architecture without downloading a 6B base\n    model. It intentionally runs on CPU and raises on any missing or unexpected\n    parameter.\n    """\n    checkpoint_path = Path(path).expanduser().resolve()\n    state = _load(checkpoint_path)\n    detected = detect_variant(state)\n    selected = detected if variant == "auto" else variant\n    if selected not in _REQUIRED_GROUPS or selected != detected:\n        raise ValueError(f"Checkpoint layout {detected!r} does not match requested variant {selected!r}.")\n\n    decoder_key = "js_decoder" if selected == "cogview4" else (\n        "js_decoder" if "js_decoder" in state else "mi_decoder"\n    )\n    decoder_state = state.get(decoder_key)\n    if not isinstance(decoder_state, Mapping):\n        raise ValueError(f"Missing joint decoder state dict: {decoder_key}.")\n\n    if selected == "cogview4":\n        from tadisr.pipelines import JointSegmentationDecoders\n\n        decoder = JointSegmentationDecoders()\n    else:\n        from tadisr.kolors_decoder import MaskInteractionDecoderKolors640M\n\n        decoder = MaskInteractionDecoderKolors640M()\n    decoder.load_state_dict(decoder_state, strict=True)\n', 'tadisr/inference.py': '"""Public, device-configurable inference API for the CogView4 TADiSR release."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .checkpoint import inspect_checkpoint\n\n\n@dataclass(frozen=True)\nclass CogView4Config:\n    base_model: Path\n    checkpoint: Path\n    prompt_embeddings: Path\n    device: str = "cuda"\n\n\ndef load_cogview4(config: CogView4Config):\n    """Construct and validate the CogView4 TADiSR pipeline before moving it to GPU."""\n    report = inspect_checkpoint(config.checkpoint, variant="cogview4")\n    if not report.valid:\n        raise RuntimeError("Invalid checkpoint: " + "; ".join(report.errors))\n    from tadisr.pipelines import CogView4Pipeline\n\n    pipeline = CogView4Pipeline(\n        ckpt_dir=str(config.base_model),\n        prompt_path=str(config.prompt_embeddings),\n        device=config.device,\n    )\n    pipeline.load_model(str(config.checkpoint))\n    pipeline.set_eval()\n    return pipeline\n\n\ndef load_rgb(path: str | Path):\n    import torch\n\n    image = Image.open(path).convert("RGB")\n    array = np.asarray(image, dtype=np.float32) / 255.0\n    return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)\n\n\ndef save_rgb(image, path: str | Path) -> None:\n    array = image.squeeze(0).detach().float().cpu().clamp(0, 1).permute(1, 2, 0).numpy()\n    Image.fromarray((array * 255.0).round().astype(np.uint8)).save(path)\n\n\ndef save_mask(mask_logits, path: str | Path) -> None:\n    mask = mask_logits.sigmoid().squeeze().detach().float().cpu().clamp(0, 1).numpy()\n    Image.fromarray((mask * 255.0).round().astype(np.uint8), mode="L").save(path)\n', 'tadisr/kolors_decoder.py': '"""Joint mask decoder used by the released Kolors TADiSR adapter.\n\nThis is the compact decoder architecture used for the FTSR Kolors checkpoint.\nIt is intentionally independent of the full Kolors pipeline so checkpoint\ncompatibility can be verified without downloading the base model.\n"""\n\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\n\ntry:\n    from diffusers.models.unet_2d_blocks import UpDecoderBlock2D\nexcept ImportError:\n    from diffusers.models.unets.unet_2d_blocks import UpDecoderBlock2D\n\n\nclass LayerNormFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x, weight, bias, eps):\n        ctx.eps = eps\n        _, channels, _, _ = x.size()\n        mean = x.mean(1, keepdim=True)\n        variance = (x - mean).pow(2).mean(1, keepdim=True)\n        normalized = (x - mean) / (variance + eps).sqrt()\n        ctx.save_for_backward(normalized, variance, weight)\n        return weight.view(1, channels, 1, 1) * normalized + bias.view(1, channels, 1, 1)\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        normalized, variance, weight = ctx.saved_tensors\n        scaled_grad = grad_output * weight.view(1, -1, 1, 1)\n        grad_input = (scaled_grad - normalized * (scaled_grad * normalized).mean(1, keepdim=True)\n                      - scaled_grad.mean(1, keepdim=True)) / torch.sqrt(variance + ctx.eps)\n        grad_weight = (grad_output * normalized).sum(dim=(0, 2, 3))\n        grad_bias = grad_output.sum(dim=(0, 2, 3))\n        return grad_input, grad_weight, grad_bias, None\n\n\nclass LayerNorm2d(nn.Module):\n    def __init__(self, channels: int, eps: float = 1e-6):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(channels))\n        self.bias = nn.Parameter(torch.zeros(channels))\n        self.eps = eps\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return LayerNormFunction.apply(x, self.weight, self.bias, self.eps)\n\n\nclass CABlock(nn.Module):\n    def __init__(self, channels: int):\n        super().__init__()\n        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, channels, 1))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return x * self.ca(x)\n\n\nclass DualStreamGate(nn.Module):\n    def forward(self, x: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        x1, x2 = x.chunk(2, dim=1)\n        y1, y2 = y.chunk(2, dim=1)\n        return x1 * y2, y1 * x2\n\n\nclass DualStreamSeq(nn.Sequential):\n    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:\n        y = x if y is None else y\n        for module in self:\n            x, y = module(x, y)\n        return x, y\n\n\nclass DualStreamSepBlock(nn.Module):\n    def __init__(self, *modules: nn.Module):\n        super().__init__()\n        self.seq_l = nn.Sequential(*modules)\n        self.seq_r = nn.Sequential(*modules)\n\n    def forward(self, x: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        return self.seq_l(x), self.seq_r(y)\n\n\nclass MuGIBlockOri(nn.Module):\n    def __init__(self, channels: int):\n        super().__init__()\n        self.block1 = DualStreamSeq(\n            DualStreamSepBlock(\n                LayerNorm2d(channels),\n                nn.Conv2d(channels, channels * 2, 1),\n                nn.Conv2d(channels * 2, channels * 2, 3, padding=1, groups=channels * 2),\n            ),\n            DualStreamGate(),\n            DualStreamSepBlock(CABlock(channels)),\n            DualStreamSepBlock(nn.Conv2d(channels, channels, 1)),\n        )\n        self.a_l = nn.Parameter(torch.zeros((1, channels, 1, 1)))\n        self.a_r = nn.Parameter(torch.zeros((1, channels, 1, 1)))\n        self.block2 = DualStreamSeq(\n            DualStreamSepBlock(LayerNorm2d(channels), nn.Conv2d(channels, channels * 2, 1)),\n            DualStreamGate(),\n            DualStreamSepBlock(nn.Conv2d(channels, channels, 1)),\n        )\n        self.b_l = nn.Parameter(torch.zeros((1, channels, 1, 1)))\n        self.b_r = nn.Parameter(torch.zeros((1, channels, 1, 1)))\n\n    def forward(self, left: torch.Tensor, right: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        left_update, right_update = self.block1(left, right)\n        left_skip = left + left_update * 0\n        right_skip = right + right_update * self.a_r\n        left_update, right_update = self.block2(left_skip, right_skip)\n        return left_skip + left_update * 0, right_skip + right_update * self.b_r\n\n\nclass MaskInteractionDecoderKolors640M(nn.Module):\n    """Exact mask-interaction decoder for ``model_526000.pkl``."""\n\n    def __init__(self):\n        super().__init__()\n        channels = (512, 512, 512, 256)\n        self.interaction_blocks = nn.ModuleList(\n            [DualStreamSeq(MuGIBlockOri(channel)) for channel in channels]\n        )\n        self.up_blocks = nn.ModuleList(\n            [\n                UpDecoderBlock2D(num_layers=2, in_channels=512, out_channels=512, add_upsample=True,\n                                 resnet_eps=1e-6, resnet_act_fn="silu", resnet_groups=32,\n                                 resnet_time_scale_shift="group"),\n                UpDecoderBlock2D(num_layers=2, in_channels=512, out_channels=512, add_upsample=True,\n                                 resnet_eps=1e-6, resnet_act_fn="silu", resnet_groups=32,\n                                 resnet_time_scale_shift="group"),\n                UpDecoderBlock2D(num_layers=2, in_channels=512, out_channels=256, add_upsample=True,\n                                 resnet_eps=1e-6, resnet_act_fn="silu", resnet_groups=32,\n                                 resnet_time_scale_shift="group"),\n                UpDecoderBlock2D(num_layers=2, in_channels=256, out_channels=256, add_upsample=False,\n                                 resnet_eps=1e-6, resnet_act_fn="silu", resnet_groups=32,\n                                 resnet_time_scale_shift="group"),\n            ]\n        )\n        self.out_block = nn.Sequential(\n            nn.GroupNorm(num_channels=256, num_groups=32, eps=1e-6),\n            nn.SiLU(),\n            nn.Conv2d(256, 1, 3, padding=1),\n        )\n', 'tadisr/pipelines.py': 'import os\nfrom collections import OrderedDict\nfrom typing import Optional, Union, List, Tuple, Dict, Any\n\nimport PIL\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, FlowMatchEulerDiscreteScheduler, \\\n    CogView4Transformer2DModel\nfrom diffusers.image_processor import VaeImageProcessor\nfrom diffusers.models.activations import get_activation\nfrom peft import LoraConfig\nfrom transformers import AutoTokenizer, GlmModel\nfrom tadisr.chatglm_tokenizer import ChatGLMTokenizer\n\ntry:\n    from diffusers.models.unet_2d_blocks import UpDecoderBlock2D\nexcept:\n    from diffusers.models.unets.unet_2d_blocks import UpDecoderBlock2D\n\n\nclass LayerNormFunction(torch.autograd.Function):\n\n    @staticmethod\n    def forward(ctx, x, weight, bias, eps):\n        ctx.eps = eps\n        N, C, H, W = x.size()\n        mu = x.mean(1, keepdim=True)\n        var = (x - mu).pow(2).mean(1, keepdim=True)\n        y = (x - mu) / (var + eps).sqrt()\n        ctx.save_for_backward(y, var, weight)\n        y = weight.view(1, C, 1, 1) * y + bias.view(1, C, 1, 1)\n        return y\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        eps = ctx.eps\n\n        N, C, H, W = grad_output.size()\n        y, var, weight = ctx.saved_tensors\n        g = grad_output * weight.view(1, C, 1, 1)\n        mean_g = g.mean(dim=1, keepdim=True)\n\n        mean_gy = (g * y).mean(dim=1, keepdim=True)\n        gx = 1. / torch.sqrt(var + eps) * (g - y * mean_gy - mean_g)\n        return gx, (grad_output * y).sum(dim=3).sum(dim=2).sum(dim=0), grad_output.sum(dim=3).sum(dim=2).sum(\n            dim=0), None\n\n\nclass LayerNorm2d(nn.Module):\n\n    def __init__(self, channels, eps=1e-6):\n        super(LayerNorm2d, self).__init__()\n        self.register_parameter(\'weight\', nn.Parameter(torch.ones(channels)))\n        self.register_parameter(\'bias\', nn.Parameter(torch.zeros(channels)))\n        self.eps = eps\n\n    def forward(self, x):\n        return LayerNormFunction.apply(x, self.weight, self.bias, self.eps)\n\n\nclass CABlock(nn.Module):\n    def __init__(self, channels):\n        super(CABlock, self).__init__()\n        self.ca = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1),\n            nn.Conv2d(channels, channels, 1)\n        )\n\n    def forward(self, x):\n        return x * self.ca(x)\n\n\nclass DualStreamGLU(nn.Module):\n    def __init__(self):\n        super(DualStreamGLU, self).__init__()\n        self.sigmoid = nn.Sigmoid()\n\n    def forward(self, x, y):\n        x1, x2 = x.chunk(2, dim=1)\n        y1, y2 = y.chunk(2, dim=1)\n\n        return x1 * self.sigmoid(y2), y1 * self.sigmoid(x2)\n\n\nclass DualStreamSeq(nn.Sequential):\n    def forward(self, x, y=None):\n        y = y if y is not None else x\n        for module in self:\n            x, y = module(x, y)\n        return x, y\n\n\nclass DualStreamBlock(nn.Module):\n    def __init__(self, *args):\n        super(DualStreamBlock, self).__init__()\n        self.seq_l = nn.Sequential()\n        self.seq_r = nn.Sequential()\n\n        if len(args) == 1 and isinstance(args[0], OrderedDict):\n            for key, module in args[0].items():\n                self.seq_l.add_module(key, module)\n                self.seq_r.add_module(key, module)\n        else:\n            for idx, module in enumerate(args):\n                self.seq_l.add_module(str(idx), module)\n                self.seq_r.add_module(str(idx), module)\n\n    def forward(self, x, y):\n        return self.seq_l(x), self.seq_r(y)\n\n\nclass ResnetBlock2D(nn.Module):\n    def __init__(\n            self,\n            in_channels: int,\n            out_channels: Optional[int] = None,\n            conv_shortcut: bool = False,\n            dropout: float = 0.0,\n            groups: int = 32,\n            groups_out: Optional[int] = None,\n            eps: float = 1e-6,\n            non_linearity: str = "swish",\n            skip_time_act: bool = False,\n            use_in_shortcut: Optional[bool] = None,\n            conv_shortcut_bias: bool = True,\n            conv_2d_out_channels: Optional[int] = None,\n    ):\n        super().__init__()\n\n        self.pre_norm = True\n        self.in_channels = in_channels\n        out_channels = in_channels if out_channels is None else out_channels\n        self.out_channels = out_channels\n        self.use_conv_shortcut = conv_shortcut\n        self.skip_time_act = skip_time_act\n\n        if groups_out is None:\n            groups_out = groups\n\n        self.norm1 = torch.nn.GroupNorm(num_groups=groups, num_channels=in_channels, eps=eps, affine=True)\n\n        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)\n\n        self.time_emb_proj = None\n\n        self.norm2 = torch.nn.GroupNorm(num_groups=groups_out, num_channels=out_channels, eps=eps, affine=True)\n\n        self.dropout = torch.nn.Dropout(dropout)\n        conv_2d_out_channels = conv_2d_out_channels or out_channels\n        self.conv2 = nn.Conv2d(out_channels, conv_2d_out_channels, kernel_size=3, stride=1, padding=1)\n\n        self.nonlinearity = get_activation(non_linearity)\n\n        self.use_in_shortcut = self.in_channels != conv_2d_out_channels if use_in_shortcut is None else use_in_shortcut\n\n        self.conv_shortcut = None\n        if self.use_in_shortcut:\n            self.conv_shortcut = nn.Conv2d(\n                in_channels,\n                conv_2d_out_channels,\n                kernel_size=1,\n                stride=1,\n                padding=0,\n                bias=conv_shortcut_bias,\n            )\n\n    def forward(self, input_tensor: torch.Tensor):\n        hidden_states = input_tensor\n\n        hidden_states = self.norm1(hidden_states)\n        hidden_states = self.nonlinearity(hidden_states)\n\n        hidden_states = self.conv1(hidden_states)\n\n        hidden_states = self.norm2(hidden_states)\n\n        hidden_states = self.nonlinearity(hidden_states)\n\n        hidden_states = self.dropout(hidden_states)\n        hidden_states = self.conv2(hidden_states)\n\n        if self.conv_shortcut is not None:\n            input_tensor = self.conv_shortcut(input_tensor)\n\n        output_tensor = input_tensor + hidden_states\n        return output_tensor\n\n\nclass CDIBlock(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.block1 = DualStreamSeq(\n            DualStreamBlock(\n                ResnetBlock2D(c, c, non_linearity="silu"),\n                nn.Conv2d(c, c * 2, 1)\n            ),\n            DualStreamGLU(),\n            DualStreamBlock(CABlock(c)),\n            DualStreamBlock(nn.GroupNorm(num_groups=32, num_channels=c)),\n            DualStreamBlock(nn.SiLU()),\n            DualStreamBlock(nn.Conv2d(c, c, 1)),\n        )\n\n        self.a_l = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)\n        self.a_r = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)\n\n    def forward(self, inp_l, inp_r):\n        x, y = self.block1(inp_l, inp_r)\n        out_l, out_r = inp_l + x * self.a_l, inp_r + y * self.a_r\n        return out_l, out_r\n\n\ndef retrieve_latents(\n        encoder_output: torch.Tensor, generator: Optional[torch.Generator] = None, sample_mode: str = "sample"\n):\n    if hasattr(encoder_output, "latent_dist") and sample_mode == "sample":\n        return encoder_output.latent_dist.sample(generator)\n    elif hasattr(encoder_output, "latent_dist") and sample_mode == "argmax":\n        return encoder_output.latent_dist.mode()\n    elif hasattr(encoder_output, "latents"):\n        return encoder_output.latents\n    else:\n        raise AttributeError("Could not access latents of provided encoder_output")\n\n\ndef make_1step_sched():\n    noise_scheduler_1step = DDPMScheduler.from_pretrained("stabilityai/sd-turbo", subfolder="scheduler")\n    noise_scheduler_1step.set_timesteps(1, device="cuda")\n    noise_scheduler_1step.alphas_cumprod = noise_scheduler_1step.alphas_cumprod.cuda()\n    return noise_scheduler_1step\n\n\ndef my_vae_encoder_fwd(self, sample):\n    sample = self.conv_in(sample)\n    l_blocks = []\n    # down\n    for down_block in self.down_blocks:\n        l_blocks.append(sample)\n        sample = down_block(sample)\n    # middle\n    sample = self.mid_block(sample)\n    sample = self.conv_norm_out(sample)\n    sample = self.conv_act(sample)\n    sample = self.conv_out(sample)\n    self.current_down_blocks = l_blocks\n    return sample\n\n\nclass JointSegmentationDecoders(nn.Module):\n    def __init__(self):\n        super().__init__()\n        # self.channel_mapping = nn.Conv2d(550, 512, 1)\n        self.interaction_blocks = nn.ModuleList([\n            DualStreamSeq(CDIBlock(1024)),\n            DualStreamSeq(CDIBlock(1024)),\n            DualStreamSeq(CDIBlock(1024)),\n            DualStreamSeq(CDIBlock(512))\n        ])\n        self.up_blocks = nn.ModuleList([\n            UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=1024,\n                out_channels=1024,\n                add_upsample=True,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            ),\n            UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=1024,\n                out_channels=1024,\n                add_upsample=True,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            ),\n            UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=1024,\n                out_channels=512,\n                add_upsample=True,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            ), UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=512,\n                out_channels=128,\n                add_upsample=False,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            )\n        ])\n\n        self.out_block = nn.Sequential(\n            nn.GroupNorm(num_channels=128, num_groups=32, eps=1e-6),\n            nn.SiLU(),\n            nn.Conv2d(128, 1, 3, padding=1)\n        )\n\n\nclass JointSegmentationDecodersMuGI(nn.Module):\n    def __init__(self):\n        super().__init__()\n        # self.channel_mapping = nn.Conv2d(550, 512, 1)\n        self.interaction_blocks = nn.ModuleList([\n            DualStreamSeq(CDIBlock(1024)),\n            DualStreamSeq(CDIBlock(1024)),\n            DualStreamSeq(CDIBlock(1024)),\n            DualStreamSeq(CDIBlock(512))\n        ])\n        self.up_blocks = nn.ModuleList([\n            UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=1024,\n                out_channels=1024,\n                add_upsample=True,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            ),\n            UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=1024,\n                out_channels=1024,\n                add_upsample=True,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            ),\n            UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=1024,\n                out_channels=512,\n                add_upsample=True,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            ), UpDecoderBlock2D(\n                num_layers=2,\n                in_channels=512,\n                out_channels=128,\n                add_upsample=False,\n                resnet_eps=1e-6,\n                resnet_act_fn="silu",\n                resnet_groups=32,\n                resnet_time_scale_shift="group"\n            )\n        ])\n\n        self.out_block = nn.Sequential(\n            nn.GroupNorm(num_channels=128, num_groups=32, eps=1e-6),\n            nn.SiLU(),\n            nn.Conv2d(128, 1, 3, padding=1)\n        )\n\n\nclass TADiSRPipeline(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n        ckpt_dir = "./weights/Kolors"\n        vae = AutoencoderKL.from_pretrained(f"{ckpt_dir}/vae")\n        vae.encoder.forward = my_vae_encoder_fwd.__get__(vae.encoder, vae.encoder.__class__)\n\n        # add the skip connection convs\n        vae.decoder.skip_conv_1 = torch.nn.Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1), bias=False).cuda()\n        vae.decoder.skip_conv_2 = torch.nn.Conv2d(256, 512, kernel_size=(1, 1), stride=(1, 1), bias=False).cuda()\n        vae.decoder.skip_conv_3 = torch.nn.Conv2d(128, 512, kernel_size=(1, 1), stride=(1, 1), bias=False).cuda()\n        vae.decoder.skip_conv_4 = torch.nn.Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False).cuda()\n        vae.decoder.ignore_skip = False\n\n        torch.nn.init.constant_(vae.decoder.skip_conv_1.weight, 1e-5)\n        torch.nn.init.constant_(vae.decoder.skip_conv_2.weight, 1e-5)\n        torch.nn.init.constant_(vae.decoder.skip_conv_3.weight, 1e-5)\n        torch.nn.init.constant_(vae.decoder.skip_conv_4.weight, 1e-5)\n        target_modules_vae = ["conv1", "conv2", "conv_in", "conv_shortcut", "conv", "conv_out",\n                              "skip_conv_1", "skip_conv_2", "skip_conv_3", "skip_conv_4",\n                              "to_k", "to_q", "to_v", "to_out.0",\n                              ]\n        vae_lora_config = LoraConfig(r=2, init_lora_weights="gaussian",\n                                     target_modules=target_modules_vae)\n        vae.add_adapter(vae_lora_config, adapter_name="vae_skip")\n        self.vae = vae\n        self.vae.to("cuda")\n\n        self.unet = UNet2DConditionModel.from_pretrained(f"{ckpt_dir}/unet")\n        self.target_modules_unet = [\n            "to_k", "to_q", "to_v", "to_out.0", "conv", "conv1", "conv2", "conv_shortcut", "conv_out",\n            "proj_in", "proj_out", "ff.net.2", "ff.net.0.proj"\n        ]\n        unet_lora_config = LoraConfig(r=4, init_lora_weights="gaussian", target_modules=self.target_modules_unet)\n        self.unet.add_adapter(unet_lora_config)\n        self.vae_scale_factor = 2 ** (len(self.vae.config.block_out_channels) - 1)\n        self.image_processor = VaeImageProcessor(vae_scale_factor=self.vae_scale_factor)\n        self.default_sample_size = self.unet.config.sample_size\n\n        prompt_path = "[pre-computed prompt tokens]"\n        saved_prompt_tokens = torch.load(prompt_path, map_location="cuda")\n        self.prompt_embeds = saved_prompt_tokens["prompt_embeds"].float().cuda()\n        self.added_cond_kwargs = {\n            "text_embeds": saved_prompt_tokens["added_cond_kwargs"]["text_embeds"].float().cuda(),\n            "time_ids": saved_prompt_tokens["added_cond_kwargs"]["time_ids"].float().cuda()\n        }\n\n        self.timesteps = torch.tensor([200], device="cuda").long()\n        self.scheduler = make_1step_sched()\n        self.tokenizer = ChatGLMTokenizer.from_pretrained(f"{ckpt_dir}/tokenizer")\n        self.js_decoder = JointSegmentationDecoders()\n\n        self.execution_device = torch.device("cuda")\n        self.to_device()\n\n    def to_device(self):\n        self.unet.to(self.execution_device)\n        self.vae.to(self.execution_device)\n        self.js_decoder.to(self.execution_device)\n\n    def set_eval(self):\n        self.unet.eval()\n        self.unet.requires_grad_(False)\n        self.vae.eval()\n        self.vae.requires_grad_(False)\n        self.js_decoder.eval()\n\n    def set_train(self):\n        self.vae.eval()\n        self.vae.requires_grad_(False)\n\n        self.unet.train()\n        for n, _p in self.unet.named_parameters():\n            if "lora" in n:\n                _p.requires_grad = True\n        self.unet.conv_in.requires_grad_(True)\n\n        self.vae.train()\n        for n, _p in self.vae.named_parameters():\n            if "lora" in n:\n                _p.requires_grad = True\n\n        self.vae.decoder.skip_conv_1.requires_grad_(True)\n        self.vae.decoder.skip_conv_2.requires_grad_(True)\n        self.vae.decoder.skip_conv_3.requires_grad_(True)\n        self.vae.decoder.skip_conv_4.requires_grad_(True)\n\n        self.js_decoder.train()\n\n    def save_model(self, outf):\n        sd = {}\n        sd["mi_decoder"] = self.js_decoder.state_dict()\n        sd["unet_lora_target_modules"] = self.target_modules_unet\n        sd["rank_unet"] = 4\n        sd["state_dict_unet"] = {k: v for k, v in self.unet.state_dict().items() if "lora" in k or "conv_in" in k}\n        sd["state_dict_vae"] = {k: v for k, v in self.vae.state_dict().items() if "lora" in k or "skip" in k}\n        torch.save(sd, outf)\n\n    def load_model_pretrain(self, state_dict_path):\n        state_dict = torch.load(state_dict_path)\n        ret = self.unet.load_state_dict(state_dict["state_dict_unet"], strict=False)\n        print("unet load lora:", ret)\n\n    def load_model(self, state_dict_path):\n        state_dict = torch.load(state_dict_path)\n        ret = self.unet.load_state_dict(state_dict["state_dict_unet"], strict=False)\n        ret = self.vae.load_state_dict(state_dict["state_dict_vae"], strict=False)\n        decoder_key = "js_decoder" if "js_decoder" in state_dict else "mi_decoder"\n        ret = self.js_decoder.load_state_dict(state_dict[decoder_key], strict=True)\n        print("js_decoder: ", ret)\n\n    def _get_add_time_ids(\n            self, original_size, crops_coords_top_left, target_size, dtype, text_encoder_projection_dim=None\n    ):\n        add_time_ids = list(original_size + crops_coords_top_left + target_size)\n\n        passed_add_embed_dim = (\n                self.unet.config.addition_time_embed_dim * len(add_time_ids) + text_encoder_projection_dim\n        )\n        expected_add_embed_dim = self.unet.add_embedding.linear_1.in_features\n\n        if expected_add_embed_dim != passed_add_embed_dim:\n            raise ValueError(\n                f"Model expects an added time embedding vector of length {expected_add_embed_dim}, but a vector of {passed_add_embed_dim} was created. The model has an incorrect config. Please check `unet.config.time_embedding_type` and `text_encoder_2.config.projection_dim`."\n            )\n\n        add_time_ids = torch.tensor([add_time_ids], dtype=dtype)\n        return add_time_ids\n\n    def get_guidance_scale_embedding(\n            self, w: torch.Tensor, embedding_dim: int = 512, dtype: torch.dtype = torch.float32\n    ) -> torch.Tensor:\n        assert len(w.shape) == 1\n        w = w * 1000.0\n\n        half_dim = embedding_dim // 2\n        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)\n        emb = torch.exp(torch.arange(half_dim, dtype=dtype) * -emb)\n        emb = w.to(dtype)[:, None] * emb[None, :]\n        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)\n        if embedding_dim % 2 == 1:  # zero pad\n            emb = torch.nn.functional.pad(emb, (0, 1))\n        assert emb.shape == (w.shape[0], embedding_dim)\n        return emb\n\n    def prepare_latents(\n            self, image, batch_size, num_images_per_prompt, dtype, device, generator=None\n    ):\n        if not isinstance(image, (torch.Tensor, PIL.Image.Image, list)):\n            raise ValueError(\n                f"`image` has to be of type `torch.Tensor`, `PIL.Image.Image` or list but is {type(image)}"\n            )\n\n        latents_mean = latents_std = None\n        if hasattr(self.vae.config, "latents_mean") and self.vae.config.latents_mean is not None:\n            latents_mean = torch.tensor(self.vae.config.latents_mean).view(1, 4, 1, 1)\n        if hasattr(self.vae.config, "latents_std") and self.vae.config.latents_std is not None:\n            latents_std = torch.tensor(self.vae.config.latents_std).view(1, 4, 1, 1)\n\n        # Offload text encoder if `enable_model_cpu_offload` was enabled\n        if hasattr(self, "final_offload_hook") and self.final_offload_hook is not None:\n            self.text_encoder_2.to("cpu")\n            torch.cuda.empty_cache()\n\n        image = image.to(device=device, dtype=dtype)\n\n        batch_size = batch_size * num_images_per_prompt\n\n        if image.shape[1] == 4:\n            init_latents = image\n\n        else:\n            # make sure the VAE is in float32 mode, as it overflows in float16\n            if self.vae.config.force_upcast:\n                image = image.float()\n                self.vae.to(dtype=torch.float32)\n\n            if isinstance(generator, list) and len(generator) != batch_size:\n                raise ValueError(\n                    f"You have passed a list of generators of length {len(generator)}, but requested an effective batch"\n                    f" size of {batch_size}. Make sure the batch size matches the length of the generators."\n                )\n\n            elif isinstance(generator, list):\n                if image.shape[0] < batch_size and batch_size % image.shape[0] == 0:\n                    image = torch.cat([image] * (batch_size // image.shape[0]), dim=0)\n                elif image.shape[0] < batch_size and batch_size % image.shape[0] != 0:\n                    raise ValueError(\n                        f"Cannot duplicate `image` of batch size {image.shape[0]} to effective batch_size {batch_size} "\n                    )\n\n                init_latents = [\n                    retrieve_latents(self.vae.encode(image[i: i + 1]), generator=generator[i])\n                    for i in range(batch_size)\n                ]\n                init_latents = torch.cat(init_latents, dim=0)\n            else:\n                init_latents = retrieve_latents(self.vae.encode(image), generator=generator)\n\n            if self.vae.config.force_upcast:\n                self.vae.to(dtype)\n\n            init_latents = init_latents.to(dtype)\n            if latents_mean is not None and latents_std is not None:\n                latents_mean = latents_mean.to(device=device, dtype=dtype)\n                latents_std = latents_std.to(device=device, dtype=dtype)\n                init_latents = (init_latents - latents_mean) * self.vae.config.scaling_factor / latents_std\n            else:\n                init_latents = self.vae.config.scaling_factor * init_latents\n\n        if batch_size > init_latents.shape[0] and batch_size % init_latents.shape[0] == 0:\n            # expand init_latents for batch_size\n            additional_image_per_prompt = batch_size // init_latents.shape[0]\n            init_latents = torch.cat([init_latents] * additional_image_per_prompt, dim=0)\n        elif batch_size > init_latents.shape[0] and batch_size % init_latents.shape[0] != 0:\n            raise ValueError(\n                f"Cannot duplicate `image` of batch size {init_latents.shape[0]} to {batch_size} text prompts."\n            )\n        else:\n            init_latents = torch.cat([init_latents], dim=0)\n\n        latents = init_latents\n\n        return latents\n\n    def forward(self, image=None,\n                guidance_scale: float = 5.0,\n                num_images_per_prompt: Optional[int] = 1,\n                tc=None):\n        image = self.image_processor.preprocess(image)\n\n        self._guidance_scale = guidance_scale\n\n        # 2. Define call parameters\n        batch_size = 1\n        device = self.execution_device\n        # 5. Prepare latent variables\n        latents = self.prepare_latents(\n            image,\n            batch_size,\n            num_images_per_prompt,\n            self.prompt_embeds.dtype,\n            device,\n        )\n\n        prompt_embeds = self.prompt_embeds.to(device)\n\n        # 8. Denoising loop\n        latent_model_input = latents\n        latent_model_input = self.scheduler.scale_model_input(latent_model_input, self.timesteps)\n\n        # predict the noise residual\n\n        noise_pred = self.unet(\n            latent_model_input,\n            self.timesteps,\n            encoder_hidden_states=prompt_embeds,\n            cross_attention_kwargs=None,\n            added_cond_kwargs=self.added_cond_kwargs,\n            return_dict=False,\n        )[0]\n\n        latents = self.scheduler.step(noise_pred, self.timesteps, latents, return_dict=False)[0]\n        heat_map = tc.compute_global_heat_map("best quality,highres,extremely detailed,text")\n        attn_map = heat_map.compute_word_heat_map_kolors(\'text\').heatmap\n\n        latents_out = latents / self.vae.config.scaling_factor\n        latents_out = self.vae.post_quant_conv(latents_out)\n        latents_out = self.vae.decoder.conv_in(latents_out)\n        latents_out = self.vae.decoder.mid_block(latents_out)\n\n        attn_map = self.js_decoder.channel_mapping(attn_map)\n        skip_convs = [self.vae.decoder.skip_conv_1, self.vae.decoder.skip_conv_2,\n                      self.vae.decoder.skip_conv_3, self.vae.decoder.skip_conv_4]\n        for i, (up_block_im, up_block_mask,\n                inter_block) in enumerate(zip(self.vae.decoder.up_blocks,\n                                              self.js_decoder.up_blocks,\n                                              self.js_decoder.interaction_blocks)):\n            if i > 0:\n                skip_in = skip_convs[i](self.vae.encoder.current_down_blocks[::-1][i])\n                latents_out = latents_out + skip_in\n\n            latents_out, attn_map = inter_block(latents_out, attn_map)\n            latents_out = up_block_im(latents_out)\n            attn_map = up_block_mask(attn_map)\n\n        mask_out = self.js_decoder.out_block(attn_map)\n        latents_out = self.vae.decoder.conv_norm_out(latents_out)\n        latents_out = self.vae.decoder.conv_act(latents_out)\n        sr_image = self.vae.decoder.conv_out(latents_out)\n        sr_image = self.image_processor.postprocess(sr_image, output_type="pt")\n        return sr_image, mask_out\n\n\nclass CogView4TextEncoderPipeline(nn.Module):\n    def __init__(self, ckpt_dir="./weights/CogView4", device="cuda"):\n        super().__init__()\n        self.execution_device = torch.device(device)\n        if not os.path.isdir(ckpt_dir):\n            raise FileNotFoundError(f"CogView4 base model directory not found: {ckpt_dir}")\n        self.tokenizer = AutoTokenizer.from_pretrained(f"{ckpt_dir}/tokenizer")\n        self.text_encoder = GlmModel.from_pretrained(f"{ckpt_dir}/text_encoder").to(self.execution_device)\n\n    def _get_glm_embeds(\n            self,\n            prompt: Union[str, List[str]] = None,\n            max_sequence_length: int = 1024,\n            device: Optional[torch.device] = None,\n            dtype: Optional[torch.dtype] = None,\n    ):\n        dtype = self.text_encoder.dtype\n\n        prompt = [prompt] if isinstance(prompt, str) else prompt\n\n        text_inputs = self.tokenizer(\n            prompt,\n            padding="longest",  # not use max length\n            max_length=max_sequence_length,\n            truncation=True,\n            add_special_tokens=True,\n            return_tensors="pt",\n        )\n        text_input_ids = text_inputs.input_ids\n        current_length = text_input_ids.shape[1]\n        pad_length = (16 - (current_length % 16)) % 16\n        if pad_length > 0:\n            pad_ids = torch.full(\n                (text_input_ids.shape[0], pad_length),\n                fill_value=self.tokenizer.pad_token_id,\n                dtype=text_input_ids.dtype,\n                device=text_input_ids.device,\n            )\n            text_input_ids = torch.cat([pad_ids, text_input_ids], dim=1)\n        prompt_embeds = self.text_encoder(text_input_ids.to(device), output_hidden_states=True).hidden_states[-2]\n\n        prompt_embeds = prompt_embeds.to(dtype=dtype, device=device)\n        return prompt_embeds\n\n    def encode_prompt(\n            self,\n            prompt: Union[str, List[str]],\n            negative_prompt: Optional[Union[str, List[str]]] = None,\n            do_classifier_free_guidance: bool = True,\n            num_images_per_prompt: int = 1,\n            prompt_embeds: Optional[torch.Tensor] = None,\n            negative_prompt_embeds: Optional[torch.Tensor] = None,\n            device: Optional[torch.device] = None,\n            dtype: Optional[torch.dtype] = None,\n            max_sequence_length: int = 1024,\n    ):\n\n        prompt = [prompt] if isinstance(prompt, str) else prompt\n        if prompt is not None:\n            batch_size = len(prompt)\n        else:\n            batch_size = prompt_embeds.shape[0]\n\n        if prompt_embeds is None:\n            prompt_embeds = self._get_glm_embeds(prompt, max_sequence_length, device, dtype)\n\n        seq_len = prompt_embeds.size(1)\n        prompt_embeds = prompt_embeds.repeat(1, num_images_per_prompt, 1)\n        prompt_embeds = prompt_embeds.view(batch_size * num_images_per_prompt, seq_len, -1)\n\n        if do_classifier_free_guidance and negative_prompt_embeds is None:\n            negative_prompt = negative_prompt or ""\n            negative_prompt = batch_size * [negative_prompt] if isinstance(negative_prompt, str) else negative_prompt\n\n            if prompt is not None and type(prompt) is not type(negative_prompt):\n                raise TypeError(\n                    f"`negative_prompt` should be the same type to `prompt`, but got {type(negative_prompt)} !="\n                    f" {type(prompt)}."\n                )\n            elif batch_size != len(negative_prompt):\n                raise ValueError(\n                    f"`negative_prompt`: {negative_prompt} has batch size {len(negative_prompt)}, but `prompt`:"\n                    f" {prompt} has batch size {batch_size}. Please make sure that passed `negative_prompt` matches"\n                    " the batch size of `prompt`."\n                )\n\n            negative_prompt_embeds = self._get_glm_embeds(negative_prompt, max_sequence_length, device, dtype)\n\n            seq_len = negative_prompt_embeds.size(1)\n            negative_prompt_embeds = negative_prompt_embeds.repeat(1, num_images_per_prompt, 1)\n            negative_prompt_embeds = negative_prompt_embeds.view(batch_size * num_images_per_prompt, seq_len, -1)\n\n        return prompt_embeds, negative_prompt_embeds\n\n    def prepare_text_embed(self, prompt, negative_prompt, save_path):\n        device = self.execution_device\n\n        # Encode input prompt\n        prompt_embeds, negative_prompt_embeds = self.encode_prompt(\n            prompt,\n            negative_prompt,\n            do_classifier_free_guidance=False,\n            num_images_per_prompt=1,\n            device=device,\n        )\n        torch.save({\n            "prompt_embeds": prompt_embeds,\n            "negative_prompt_embeds": negative_prompt_embeds\n        }, save_path)\n\n        print(f"Saved prompt embeds in {save_path}")\n\n\nclass CogView4Pipeline(nn.Module):\n    """TADiSR inference model built on a local CogView4-6B Diffusers export.\n\n    The checkpoint contains only LoRA adapters and the joint segmentation decoder;\n    the public CogView4 base model is supplied separately through ``ckpt_dir``.\n    """\n\n    def __init__(self, ckpt_dir="./weights/CogView4", prompt_path=None, device="cuda"):\n        super().__init__()\n        self.execution_device = torch.device(device)\n        if self.execution_device.type == "cuda" and not torch.cuda.is_available():\n            raise RuntimeError("A CUDA device was requested, but CUDA is not available.")\n        if prompt_path is None:\n            prompt_path = f"{ckpt_dir}/saved_prompt_tokens_nocfg.pt"\n        if not os.path.isdir(ckpt_dir):\n            raise FileNotFoundError(f"CogView4 base model directory not found: {ckpt_dir}")\n        if not os.path.isfile(prompt_path):\n            raise FileNotFoundError(\n                f"Prompt embeddings not found: {prompt_path}. Run prepare_cogview4_prompt_embeddings.py first."\n            )\n        vae = AutoencoderKL.from_pretrained(f"{ckpt_dir}/vae")\n        vae.encoder.forward = my_vae_encoder_fwd.__get__(vae.encoder, vae.encoder.__class__)\n\n        # add the skip connection convs\n        vae.decoder.skip_conv_1 = torch.nn.Conv2d(1024, 1024, kernel_size=(1, 1), stride=(1, 1), bias=False)\n        vae.decoder.skip_conv_2 = torch.nn.Conv2d(512, 1024, kernel_size=(1, 1), stride=(1, 1), bias=False)\n        vae.decoder.skip_conv_3 = torch.nn.Conv2d(128, 1024, kernel_size=(1, 1), stride=(1, 1), bias=False)\n        vae.decoder.skip_conv_4 = torch.nn.Conv2d(128, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)\n        vae.decoder.ignore_skip = False\n\n        torch.nn.init.constant_(vae.decoder.skip_conv_1.weight, 1e-5)\n        torch.nn.init.constant_(vae.decoder.skip_conv_2.weight, 1e-5)\n        torch.nn.init.constant_(vae.decoder.skip_conv_3.weight, 1e-5)\n        torch.nn.init.constant_(vae.decoder.skip_conv_4.weight, 1e-5)\n        target_modules_vae = ["conv1", "conv2", "conv_in", "conv_shortcut", "conv", "conv_out",\n                              "skip_conv_1", "skip_conv_2", "skip_conv_3", "skip_conv_4",\n                              "to_k", "to_q", "to_v", "to_out.0",\n                              ]\n        vae_lora_config = LoraConfig(r=2, init_lora_weights="gaussian",\n                                     target_modules=target_modules_vae)\n        vae.add_adapter(vae_lora_config, adapter_name="vae_skip")\n        self.vae = vae\n        self.vae.to(self.execution_device)\n\n        self.transformer = CogView4Transformer2DModel.from_pretrained(f"{ckpt_dir}/transformer")\n        self.target_modules_transformer = [\n            "attn1.to_q", "attn1.to_k", "attn1.to_v", "attn1.to_out.0",\n            "attn2.to_q", "attn2.to_k", "attn2.to_v", "attn2.to_out.0",\n            "ff.net.0.proj", "ff.net.2",\n            "proj_in", "proj_out"\n        ]\n        transformer_lora_config = LoraConfig(\n            r=4, init_lora_weights="gaussian",\n            target_modules=self.target_modules_transformer\n        )\n        self.transformer.add_adapter(transformer_lora_config)\n        self.transformer.to(self.execution_device)\n\n        self.js_decoder = JointSegmentationDecoders()\n        self.js_decoder.to(self.execution_device)\n\n        self.scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(f"{ckpt_dir}/scheduler")\n        self.target_timestep = 200.0\n\n        t_diff = torch.abs(self.scheduler.timesteps - self.target_timestep)\n        self.closest_idx = torch.argmin(t_diff)\n\n        self.vae_scale_factor = 2 ** (len(self.vae.config.block_out_channels) - 1) if getattr(self, "vae", None) else 8\n        self.image_processor = VaeImageProcessor(vae_scale_factor=self.vae_scale_factor)\n\n        saved_prompt_tokens = torch.load(prompt_path, map_location="cpu")\n        self.prompt_embeds = saved_prompt_tokens["prompt_embeds"].float().to(self.execution_device)\n\n    def set_eval(self):\n        self.transformer.eval()\n        self.transformer.requires_grad_(False)\n        self.vae.eval()\n        self.vae.requires_grad_(False)\n        self.js_decoder.eval()\n\n    def set_train(self):\n        self.vae.eval()\n        self.vae.requires_grad_(False)\n\n        self.transformer.train()\n        for n, _p in self.transformer.named_parameters():\n            if "lora" in n:\n                _p.requires_grad = True\n\n        self.vae.train()\n        for n, _p in self.vae.named_parameters():\n            if "lora" in n:\n                _p.requires_grad = True\n\n        self.vae.decoder.skip_conv_1.requires_grad_(True)\n        self.vae.decoder.skip_conv_2.requires_grad_(True)\n        self.vae.decoder.skip_conv_3.requires_grad_(True)\n        self.vae.decoder.skip_conv_4.requires_grad_(True)\n\n        self.js_decoder.train()\n\n    def save_model(self, outf):\n        sd = {}\n        sd["js_decoder"] = self.js_decoder.state_dict()\n        sd["transformer_lora_target_modules"] = self.target_modules_transformer\n        sd["rank_transformer"] = 4\n        sd["state_dict_transformer"] = {k: v for k, v in self.transformer.state_dict().items() if "lora" in k}\n        sd["state_dict_vae"] = {k: v for k, v in self.vae.state_dict().items() if "lora" in k or "skip" in k}\n        torch.save(sd, outf)\n\n    def load_model(self, state_dict_path):\n        state_dict = torch.load(state_dict_path, map_location="cpu")\n        ret = self.transformer.load_state_dict(state_dict["state_dict_transformer"], strict=False)\n        # print("transformer", ret)\n        ret = self.vae.load_state_dict(state_dict["state_dict_vae"], strict=False)\n        # print("vae", ret)\n        ret = self.js_decoder.load_state_dict(state_dict["js_decoder"], strict=True)\n        print("js_decoder: ", ret)\n\n    def get_sigmas(self, timesteps, n_dim=4, dtype=torch.float32):\n        sigmas = self.scheduler.sigmas.to(device=self.execution_device, dtype=dtype)\n        schedule_timesteps = self.scheduler.timesteps.to(device=self.execution_device, dtype=dtype)\n        timesteps = timesteps.to(device=self.execution_device)\n        step_indices = [(schedule_timesteps == t).nonzero().item() for t in timesteps]\n        sigma = sigmas[step_indices].flatten()\n        while len(sigma.shape) < n_dim:\n            sigma = sigma.unsqueeze(-1)\n        return sigma\n\n    def forward(self, lr_image, num_images_per_prompt: int = 1):\n        batch_size = lr_image.shape[0]\n        height, width = lr_image.shape[-2:]\n\n        original_size = (height, width)\n        target_size = (height, width)\n        crops_coords_top_left = (0, 0)\n        indices = torch.full((batch_size,), self.closest_idx.item(), dtype=torch.long)\n        timesteps = self.scheduler.timesteps[indices]\n        sigmas = self.get_sigmas(timesteps, n_dim=lr_image.ndim, dtype=lr_image.dtype)\n\n        device = self.execution_device\n\n        prompt_embeds = self.prompt_embeds.repeat(batch_size, 1, 1)\n        lr_image = self.image_processor.preprocess(lr_image)\n\n        vae_shift_factor = 0\n        latents = self.vae.encode(lr_image).latent_dist.sample()\n        latents = (latents - vae_shift_factor) * self.vae.config.scaling_factor\n\n        # Prepare additional timestep conditions\n        original_size = torch.tensor([original_size], dtype=prompt_embeds.dtype, device=device)\n        target_size = torch.tensor([target_size], dtype=prompt_embeds.dtype, device=device)\n        crops_coords_top_left = torch.tensor([crops_coords_top_left], dtype=prompt_embeds.dtype, device=device)\n\n        original_size = original_size.repeat(batch_size * num_images_per_prompt, 1)\n        target_size = target_size.repeat(batch_size * num_images_per_prompt, 1)\n        crops_coords_top_left = crops_coords_top_left.repeat(batch_size * num_images_per_prompt, 1)\n\n        latent_model_input = latents.to(self.transformer.dtype)\n        timesteps = timesteps.to(device)\n\n        noise_pred = self.transformer(\n            hidden_states=latent_model_input,\n            encoder_hidden_states=prompt_embeds,\n            timestep=timesteps,\n            original_size=original_size,\n            target_size=target_size,\n            crop_coords=crops_coords_top_left,\n            return_dict=False,\n        )[0]\n        latents = latents - sigmas * noise_pred\n\n        latents_out = latents / self.vae.config.scaling_factor\n        latents_out = self.vae.decoder.conv_in(latents_out)\n        latents_out = self.vae.decoder.mid_block(latents_out)\n\n        attn_map = latents_out\n        skip_convs = [self.vae.decoder.skip_conv_1, self.vae.decoder.skip_conv_2,\n                      self.vae.decoder.skip_conv_3, self.vae.decoder.skip_conv_4]\n\n        for i, (up_block_im, up_block_mask,\n                inter_block) in enumerate(zip(self.vae.decoder.up_blocks,\n                                              self.js_decoder.up_blocks,\n                                              self.js_decoder.interaction_blocks)):\n\n            if i > 0:\n                skip_in = skip_convs[i](self.vae.encoder.current_down_blocks[::-1][i])\n                latents_out = latents_out + skip_in\n\n            latents_out, attn_map = inter_block(latents_out, attn_map)\n            latents_out = up_block_im(latents_out)\n            attn_map = up_block_mask(attn_map)\n\n        mask_out = self.js_decoder.out_block(attn_map)\n        latents_out = self.vae.decoder.conv_norm_out(latents_out)\n        latents_out = self.vae.decoder.conv_act(latents_out)\n        sr_image = self.vae.decoder.conv_out(latents_out)\n        sr_image = self.image_processor.postprocess(sr_image, output_type="pt")\n\n        return sr_image, mask_out\n', 'tadisr/tiling.py': '"""Seam-aware tiling used by the public inference entry point."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Callable\n\nimport torch\nimport torch.nn.functional as F\n\n\ndef _starts(length: int, tile: int, overlap: int) -> list[int]:\n    if tile <= overlap:\n        raise ValueError("tile must be larger than overlap")\n    if length <= tile:\n        return [0]\n    stride = tile - overlap\n    starts = list(range(0, length - tile + 1, stride))\n    if starts[-1] != length - tile:\n        starts.append(length - tile)\n    return starts\n\n\ndef _weight(tile: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:\n    window = torch.hann_window(tile, periodic=False, device=device, dtype=dtype).clamp_min(1e-3)\n    return (window[:, None] * window[None, :])[None, None]\n\n\ndef tiled_predict(\n    image: torch.Tensor,\n    predict: Callable[[torch.Tensor], tuple[torch.Tensor, torch.Tensor]],\n    tile: int = 768,\n    overlap: int = 256,\n) -> tuple[torch.Tensor, torch.Tensor]:\n    """Predict SR image and text-mask logits using overlapped, weighted tiles.\n\n    The model operates on already-upsampled RGB input and must preserve its spatial\n    resolution. Only batch size one is supported because the diffusion path is\n    intentionally single-image.\n    """\n    if image.ndim != 4 or image.shape[0] != 1 or image.shape[1] != 3:\n        raise ValueError("Expected a single RGB tensor with shape (1, 3, H, W).")\n    _, _, height, width = image.shape\n    y_starts, x_starts = _starts(height, tile, overlap), _starts(width, tile, overlap)\n    pad_h, pad_w = max(tile - height, 0), max(tile - width, 0)\n    if pad_h or pad_w:\n        image = F.pad(image, (0, pad_w, 0, pad_h), mode="replicate")\n        y_starts, x_starts = [0], [0]\n\n    image_sum = torch.zeros((1, 3, image.shape[-2], image.shape[-1]), device=image.device, dtype=image.dtype)\n    mask_sum = torch.zeros((1, 1, image.shape[-2], image.shape[-1]), device=image.device, dtype=image.dtype)\n    weights = torch.zeros((1, 1, image.shape[-2], image.shape[-1]), device=image.device, dtype=image.dtype)\n    blend = _weight(tile, image.device, image.dtype)\n\n    for top in y_starts:\n        for left in x_starts:\n            patch = image[:, :, top : top + tile, left : left + tile]\n            sr_patch, mask_patch = predict(patch)\n            if sr_patch.shape != patch.shape or mask_patch.shape != (1, 1, tile, tile):\n                raise RuntimeError("Model output shape does not match tiled inference contract.")\n            image_sum[:, :, top : top + tile, left : left + tile] += sr_patch * blend\n            mask_sum[:, :, top : top + tile, left : left + tile] += mask_patch * blend\n            weights[:, :, top : top + tile, left : left + tile] += blend\n\n    sr = image_sum / weights\n    mask_logits = mask_sum / weights\n    return sr[:, :, :height, :width], mask_logits[:, :, :height, :width]\n', 'tadisr/training/__init__.py': '"""Training-only components for TADiSR."""\n', 'tadisr/training/datasets.py': 'import argparse\nimport json\nimport os\nimport random\nfrom glob import glob\nfrom typing import Tuple\n\nimport cv2\nimport math\nimport numpy as np\nimport torch\nimport torchvision.transforms.functional as F\nfrom PIL import Image\nfrom torch.utils import data as data\nfrom torchvision import transforms\nfrom PIL import Image, ImageFilter\nfrom tadisr.training.degradations import circular_lowpass_kernel, random_mixed_kernels\n\n\ndef img2tensor(imgs, bgr2rgb=True, float32=True):\n    def _totensor(img, bgr2rgb, float32):\n        if img.shape[2] == 3 and bgr2rgb:\n            if img.dtype == \'float64\':\n                img = img.astype(\'float32\')\n            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n        img = torch.from_numpy(img.transpose(2, 0, 1))\n        if float32:\n            img = img.float()\n        return img\n\n    if isinstance(imgs, list):\n        return [_totensor(img, bgr2rgb, float32) for img in imgs]\n    else:\n        return _totensor(imgs, bgr2rgb, float32)\n\n\ndef augment(imgs, hflip=True, rotation=True, flows=None, return_status=False):\n    hflip = hflip and random.random() < 0.5\n    vflip = rotation and random.random() < 0.5\n    rot90 = rotation and random.random() < 0.5\n\n    def _augment(img):\n        if hflip:  # horizontal\n            cv2.flip(img, 1, img)\n        if vflip:  # vertical\n            cv2.flip(img, 0, img)\n        if rot90:\n            img = img.transpose(1, 0, 2)\n        return img\n\n    def _augment_flow(flow):\n        if hflip:  # horizontal\n            cv2.flip(flow, 1, flow)\n            flow[:, :, 0] *= -1\n        if vflip:  # vertical\n            cv2.flip(flow, 0, flow)\n            flow[:, :, 1] *= -1\n        if rot90:\n            flow = flow.transpose(1, 0, 2)\n            flow = flow[:, :, [1, 0]]\n        return flow\n\n    if not isinstance(imgs, list):\n        imgs = [imgs]\n    imgs = [_augment(img) for img in imgs]\n    if len(imgs) == 1:\n        imgs = imgs[0]\n\n    if flows is not None:\n        if not isinstance(flows, list):\n            flows = [flows]\n        flows = [_augment_flow(flow) for flow in flows]\n        if len(flows) == 1:\n            flows = flows[0]\n        return imgs, flows\n    else:\n        if return_status:\n            return imgs, (hflip, vflip, rot90)\n        else:\n            return imgs\n\n\ndef parse_args_paired_training(input_args=None):\n    """\n    Parses command-line arguments used for configuring an paired session (pix2pix-Turbo).\n    This function sets up an argument parser to handle various training options.\n\n    Returns:\n    argparse.Namespace: The parsed command-line arguments.\n   """\n    parser = argparse.ArgumentParser()\n    # args for the loss function\n    parser.add_argument("--gan_disc_type", default="vagan_clip")\n    parser.add_argument("--gan_loss_type", default="multilevel_sigmoid_s")\n    parser.add_argument("--lambda_gan", default=0.5, type=float)\n    parser.add_argument("--lambda_lpips", default=5, type=float)\n    parser.add_argument("--lambda_l2", default=1.0, type=float)\n    parser.add_argument("--lambda_bce", default=1.0, type=float)\n    parser.add_argument("--lambda_clipsim", default=5.0, type=float)\n\n    # dataset options\n    parser.add_argument("--train_folders", type=str)\n    parser.add_argument("--test_folder", type=str)\n\n    parser.add_argument("--train_image_prep", default="resized_crop_512", type=str)\n    parser.add_argument("--test_image_prep", default="resized_crop_512", type=str)\n\n    # validation eval args\n    parser.add_argument("--eval_freq", default=20, type=int)\n    parser.add_argument("--track_val_fid", default=False, action="store_true")\n    parser.add_argument("--num_samples_eval", type=int, default=100, help="Number of samples to use for all evaluation")\n    parser.add_argument("--num_samples_save", type=int, default=20, help="Number of samples to use for all evaluation")\n\n    parser.add_argument("--viz_freq", type=int, default=100, help="Frequency of visualizing the outputs.")\n    parser.add_argument("--tracker_project_name", type=str, default="train_pix2pix_turbo",\n                        help="The name of the wandb project to log to.")\n\n    # details about the model architecture\n    parser.add_argument("--pretrained_model_name_or_path")\n    parser.add_argument("--prompt_embeddings", type=str, default=None,\n                        help="Fixed CogView4 quality-prompt embedding file.")\n    parser.add_argument("--resume_checkpoint", type=str, default=None,\n                        help="Optional TADiSR adapter checkpoint to resume from.")\n    parser.add_argument("--realce_train_list", type=str, default=None)\n    parser.add_argument("--realce_eval_list", type=str, default=None)\n    parser.add_argument("--revision", type=str, default=None, )\n    parser.add_argument("--variant", type=str, default=None, )\n    parser.add_argument("--tokenizer_name", type=str, default=None)\n    parser.add_argument("--lora_rank_unet", default=8, type=int)\n    parser.add_argument("--lora_rank_vae", default=4, type=int)\n\n    # training details\n    parser.add_argument("--output_dir", required=True)\n    parser.add_argument("--cache_dir", default=None, )\n    parser.add_argument("--seed", type=int, default=None, help="A seed for reproducible training.")\n    parser.add_argument("--resolution", type=int, default=512, )\n    parser.add_argument("--train_batch_size", type=int, default=1,\n                        help="Batch size (per device) for the training dataloader.")\n    parser.add_argument("--num_training_epochs", type=int, default=50)\n    parser.add_argument("--max_train_steps", type=int, default=10_000, )\n    parser.add_argument("--checkpointing_steps", type=int, default=500, )\n    parser.add_argument("--gradient_accumulation_steps", type=int, default=1,\n                        help="Number of updates steps to accumulate before performing a backward/update pass.", )\n    parser.add_argument("--gradient_checkpointing", action="store_true", )\n    parser.add_argument("--learning_rate", type=float, default=5e-6)\n    parser.add_argument("--lr_scheduler", type=str, default="constant",\n                        help=(\n                            \'The scheduler type to use. Choose between ["linear", "cosine", "cosine_with_restarts", "polynomial",\'\n                            \' "constant", "constant_with_warmup"]\'\n                        ),\n                        )\n    parser.add_argument("--lr_warmup_steps", type=int, default=500,\n                        help="Number of steps for the warmup in the lr scheduler.")\n    parser.add_argument("--lr_num_cycles", type=int, default=1,\n                        help="Number of hard resets of the lr in cosine_with_restarts scheduler.",\n                        )\n    parser.add_argument("--lr_power", type=float, default=1.0, help="Power factor of the polynomial scheduler.")\n\n    parser.add_argument("--dataloader_num_workers", type=int, default=0, )\n    parser.add_argument("--adam_beta1", type=float, default=0.9, help="The beta1 parameter for the Adam optimizer.")\n    parser.add_argument("--adam_beta2", type=float, default=0.999, help="The beta2 parameter for the Adam optimizer.")\n    parser.add_argument("--adam_weight_decay", type=float, default=1e-2, help="Weight decay to use.")\n    parser.add_argument("--adam_epsilon", type=float, default=1e-08, help="Epsilon value for the Adam optimizer")\n    parser.add_argument("--max_grad_norm", default=1.0, type=float, help="Max gradient norm.")\n    parser.add_argument("--allow_tf32", action="store_true",\n                        help=(\n                            "Whether or not to allow TF32 on Ampere GPUs. Can be used to speed up training. For more information, see"\n                            " https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices"\n                        ),\n                        )\n    parser.add_argument("--report_to", type=str, default="tensorboard",\n                        help=(\n                            \'The integration to report the results and logs to. Supported platforms are `"tensorboard"`\'\n                            \' (default), `"wandb"` and `"comet_ml"`. Use `"all"` to report to all integrations.\'\n                        ),\n                        )\n    parser.add_argument("--mixed_precision", type=str, default=None, choices=["no", "fp16", "bf16"], )\n    parser.add_argument("--enable_xformers_memory_efficient_attention", action="store_true",\n                        help="Whether or not to use xformers.")\n    parser.add_argument("--set_grads_to_none", action="store_true", )\n\n    if input_args is not None:\n        args = parser.parse_args(input_args)\n    else:\n        args = parser.parse_args()\n\n    args.train_folders = args.train_folders.split(\',\')\n    return args\n\n\ndef parse_args_testing(input_args=None):\n    """\n    Parses command-line arguments used for configuring an paired session (pix2pix-Turbo).\n    This function sets up an argument parser to handle various training options.\n\n    Returns:\n    argparse.Namespace: The parsed command-line arguments.\n   """\n    parser = argparse.ArgumentParser()\n    # dataset options\n    parser.add_argument("--test_folder", type=str)\n\n    parser.add_argument("--train_image_prep", default="resized_crop_512", type=str)\n    parser.add_argument("--test_image_prep", default="resized_crop_512", type=str)\n\n    # validation eval args\n    parser.add_argument("--eval_freq", default=20, type=int)\n    parser.add_argument("--num_samples_eval", type=int, default=100, help="Number of samples to use for all evaluation")\n\n    parser.add_argument("--tracker_project_name", type=str, default="train_pix2pix_turbo",\n                        help="The name of the wandb project to log to.")\n\n    # details about the model architecture\n    parser.add_argument("--pretrained_model_name_or_path")\n    parser.add_argument("--revision", type=str, default=None, )\n    parser.add_argument("--variant", type=str, default=None, )\n    parser.add_argument("--tokenizer_name", type=str, default=None)\n    parser.add_argument("--lora_rank_unet", default=8, type=int)\n    parser.add_argument("--lora_rank_vae", default=4, type=int)\n    parser.add_argument(\'--selected\', type=str, nargs=\'+\')\n\n    # training details\n    parser.add_argument("--output_dir", required=True)\n    parser.add_argument("--cache_dir", default=None, )\n    parser.add_argument("--seed", type=int, default=None, help="A seed for reproducible training.")\n    parser.add_argument("--resolution", type=int, default=512, )\n    parser.add_argument("--train_batch_size", type=int, default=1,\n                        help="Batch size (per device) for the training dataloader.")\n    parser.add_argument("--num_training_epochs", type=int, default=50)\n    parser.add_argument("--max_train_steps", type=int, default=10_000, )\n    parser.add_argument("--checkpointing_steps", type=int, default=500, )\n    parser.add_argument("--gradient_accumulation_steps", type=int, default=1,\n                        help="Number of updates steps to accumulate before performing a backward/update pass.", )\n    parser.add_argument("--gradient_checkpointing", action="store_true", )\n    parser.add_argument("--dataloader_num_workers", type=int, default=0, )\n\n    if input_args is not None:\n        args = parser.parse_args(input_args)\n    else:\n        args = parser.parse_args()\n\n    return args\n\n\ndef parse_args_unpaired_training():\n    """\n    Parses command-line arguments used for configuring an unpaired session (CycleGAN-Turbo).\n    This function sets up an argument parser to handle various training options.\n\n    Returns:\n    argparse.Namespace: The parsed command-line arguments.\n   """\n\n    parser = argparse.ArgumentParser(description="Simple example of a ControlNet training script.")\n\n    # fixed random seed\n    parser.add_argument("--seed", type=int, default=42, help="A seed for reproducible training.")\n\n    # args for the loss function\n    parser.add_argument("--gan_disc_type", default="vagan_clip")\n    parser.add_argument("--gan_loss_type", default="multilevel_sigmoid")\n    parser.add_argument("--lambda_gan", default=0.5, type=float)\n    parser.add_argument("--lambda_idt", default=1, type=float)\n    parser.add_argument("--lambda_cycle", default=1, type=float)\n    parser.add_argument("--lambda_cycle_lpips", default=10.0, type=float)\n    parser.add_argument("--lambda_idt_lpips", default=1.0, type=float)\n\n    # args for dataset and dataloader options\n    parser.add_argument("--dataset_folder", required=True, type=str)\n    parser.add_argument("--train_img_prep", required=True)\n    parser.add_argument("--val_img_prep", required=True)\n    parser.add_argument("--dataloader_num_workers", type=int, default=0)\n    parser.add_argument("--train_batch_size", type=int, default=4,\n                        help="Batch size (per device) for the training dataloader.")\n    parser.add_argument("--max_train_epochs", type=int, default=100)\n    parser.add_argument("--max_train_steps", type=int, default=None)\n\n    # args for the model\n    parser.add_argument("--pretrained_model_name_or_path", default="stabilityai/sd-turbo")\n    parser.add_argument("--revision", default=None, type=str)\n    parser.add_argument("--variant", default=None, type=str)\n    parser.add_argument("--lora_rank_unet", default=128, type=int)\n    parser.add_argument("--lora_rank_vae", default=4, type=int)\n\n    # args for validation and logging\n    parser.add_argument("--viz_freq", type=int, default=20)\n    parser.add_argument("--output_dir", type=str, required=True)\n    parser.add_argument("--report_to", type=str, default="tensorboard")\n    parser.add_argument("--tracker_project_name", type=str, required=True)\n    parser.add_argument("--validation_steps", type=int, default=500, )\n    parser.add_argument("--validation_num_images", type=int, default=-1,\n                        help="Number of images to use for validation. -1 to use all images.")\n    parser.add_argument("--checkpointing_steps", type=int, default=500)\n\n    # args for the optimization options\n    parser.add_argument("--learning_rate", type=float, default=5e-6, )\n    parser.add_argument("--adam_beta1", type=float, default=0.9, help="The beta1 parameter for the Adam optimizer.")\n    parser.add_argument("--adam_beta2", type=float, default=0.999, help="The beta2 parameter for the Adam optimizer.")\n    parser.add_argument("--adam_weight_decay", type=float, default=1e-2, help="Weight decay to use.")\n    parser.add_argument("--adam_epsilon", type=float, default=1e-08, help="Epsilon value for the Adam optimizer")\n    parser.add_argument("--max_grad_norm", default=10.0, type=float, help="Max gradient norm.")\n    parser.add_argument("--lr_scheduler", type=str, default="constant", help=(\n        \'The scheduler type to use. Choose between ["linear", "cosine", "cosine_with_restarts", "polynomial",\'\n        \' "constant", "constant_with_warmup"]\'\n    ),\n                        )\n    parser.add_argument("--lr_warmup_steps", type=int, default=500,\n                        help="Number of steps for the warmup in the lr scheduler.")\n    parser.add_argument("--lr_num_cycles", type=int, default=1,\n                        help="Number of hard resets of the lr in cosine_with_restarts scheduler.", )\n    parser.add_argument("--lr_power", type=float, default=1.0, help="Power factor of the polynomial scheduler.")\n    parser.add_argument("--gradient_accumulation_steps", type=int, default=1)\n\n    # memory saving options\n    parser.add_argument("--allow_tf32", action="store_true",\n                        help=(\n                            "Whether or not to allow TF32 on Ampere GPUs. Can be used to speed up training. For more information, see"\n                            " https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices"\n                        ),\n                        )\n    parser.add_argument("--gradient_checkpointing", action="store_true",\n                        help="Whether or not to use gradient checkpointing to save memory at the expense of slower backward pass.")\n    parser.add_argument("--enable_xformers_memory_efficient_attention", action="store_true",\n                        help="Whether or not to use xformers.")\n\n    args = parser.parse_args()\n    return args\n\n\ndef build_transform(image_prep):\n    """\n    Constructs a transformation pipeline based on the specified image preparation method.\n\n    Parameters:\n    - image_prep (str): A string describing the desired image preparation\n\n    Returns:\n    - torchvision.transforms.Compose: A composable sequence of transformations to be applied to images.\n    """\n    if image_prep == "resized_crop_512":\n        T = transforms.Compose([\n            transforms.Resize(512, interpolation=transforms.InterpolationMode.LANCZOS),\n            transforms.CenterCrop(512),\n        ])\n    elif image_prep == "resize_286_randomcrop_256x256_hflip":\n        T = transforms.Compose([\n            transforms.Resize((286, 286), interpolation=Image.LANCZOS),\n            transforms.RandomCrop((256, 256)),\n            transforms.RandomHorizontalFlip(),\n        ])\n    elif image_prep in ["resize_256", "resize_256x256"]:\n        T = transforms.Compose([\n            transforms.Resize((256, 256), interpolation=Image.LANCZOS)\n        ])\n    elif image_prep in ["resize_512", "resize_512x512"]:\n        T = transforms.Compose([\n            transforms.Resize((512, 512), interpolation=Image.LANCZOS)\n        ])\n    elif image_prep == "no_resize":\n        T = transforms.Lambda(lambda x: x)\n    return T\n\n\nclass PairedDataset(torch.utils.data.Dataset):\n    def __init__(self, dataset_folder, split, image_prep, tokenizer):\n        """\n        Itialize the paired dataset object for loading and transforming paired data samples\n        from specified dataset folders.\n\n        This constructor sets up the paths to input and output folders based on the specified \'split\',\n        loads the captions (or prompts) for the input images, and prepares the transformations and\n        tokenizer to be applied on the data.\n\n        Parameters:\n        - dataset_folder (str): The root folder containing the dataset, expected to include\n                                sub-folders for different splits (e.g., \'train_A\', \'train_B\').\n        - split (str): The dataset split to use (\'train\' or \'test\'), used to select the appropriate\n                       sub-folders and caption files within the dataset folder.\n        - image_prep (str): The image preprocessing transformation to apply to each image.\n        - tokenizer: The tokenizer used for tokenizing the captions (or prompts).\n        """\n        super().__init__()\n        if split == "train":\n            self.input_folder = os.path.join(dataset_folder, "train_A")\n            self.output_folder = os.path.join(dataset_folder, "train_B")\n            captions = os.path.join(dataset_folder, "train_prompts.json")\n        elif split == "test":\n            self.input_folder = os.path.join(dataset_folder, "test_A")\n            self.output_folder = os.path.join(dataset_folder, "test_B")\n            captions = os.path.join(dataset_folder, "test_prompts.json")\n        with open(captions, "r") as f:\n            self.captions = json.load(f)\n        self.img_names = list(self.captions.keys())\n        self.T = build_transform(image_prep)\n        self.tokenizer = tokenizer\n\n    def __len__(self):\n        """\n        Returns:\n        int: The total number of items in the dataset.\n        """\n        return len(self.captions)\n\n    def __getitem__(self, idx):\n        """\n        Retrieves a dataset item given its index. Each item consists of an input image, \n        its corresponding output image, the captions associated with the input image, \n        and the tokenized form of this caption.\n\n        This method performs the necessary preprocessing on both the input and output images, \n        including scaling and normalization, as well as tokenizing the caption using a provided tokenizer.\n\n        Parameters:\n        - idx (int): The index of the item to retrieve.\n\n        Returns:\n        dict: A dictionary containing the following key-value pairs:\n            - "output_pixel_values": a tensor of the preprocessed output image with pixel values \n            scaled to [-1, 1].\n            - "conditioning_pixel_values": a tensor of the preprocessed input image with pixel values \n            scaled to [0, 1].\n            - "caption": the text caption.\n            - "input_ids": a tensor of the tokenized caption.\n\n        Note:\n        The actual preprocessing steps (scaling and normalization) for images are defined externally \n        and passed to this class through the `image_prep` parameter during initialization. The \n        tokenization process relies on the `tokenizer` also provided at initialization, which \n        should be compatible with the models intended to be used with this dataset.\n        """\n        img_name = self.img_names[idx]\n        input_img = Image.open(os.path.join(self.input_folder, img_name))\n        output_img = Image.open(os.path.join(self.output_folder, img_name))\n        caption = self.captions[img_name]\n\n        # input images scaled to 0,1\n        img_t = self.T(input_img)\n        img_t = F.to_tensor(img_t)\n        # output images scaled to -1,1\n        output_t = self.T(output_img)\n        output_t = F.to_tensor(output_t)\n        output_t = F.normalize(output_t, mean=[0.5], std=[0.5])\n\n        input_ids = self.tokenizer(\n            caption, max_length=self.tokenizer.model_max_length,\n            padding="max_length", truncation=True, return_tensors="pt"\n        ).input_ids\n\n        return {\n            "output_pixel_values": output_t,\n            "conditioning_pixel_values": img_t,\n            "caption": caption,\n            "input_ids": input_ids,\n        }\n\n\nclass PairedSRDataset(torch.utils.data.Dataset):\n    def __init__(self, dataset_folder: str, hr_dir: str, lr_dir: str, split, image_prep):\n        super().__init__()\n        if split == "train":\n            self.h_folder = os.path.join(dataset_folder, lr_dir)\n            self.output_folder = os.path.join(dataset_folder, hr_dir)\n        elif split == "test":\n            self.input_folder = os.path.join(dataset_folder, lr_dir)\n            self.output_folder = os.path.join(dataset_folder, hr_dir)\n\n        self.img_names = list(os.listdir(self.output_folder))\n        self.T = build_transform(image_prep)\n\n    def __len__(self):\n        return len(self.captions)\n\n    def __getitem__(self, idx):\n        img_name = self.img_names[idx]\n        input_img = Image.open(os.path.join(self.input_folder, img_name))\n        output_img = Image.open(os.path.join(self.output_folder, img_name))\n\n        img_t = self.T(input_img)\n        img_t = F.to_tensor(img_t)\n\n        output_t = self.T(output_img)\n        output_t = F.to_tensor(output_t)\n        output_t = F.normalize(output_t, mean=[0.5], std=[0.5])\n\n        return {\n            "output_pixel_values": output_t,\n            "conditioning_pixel_values": img_t\n        }\n\n\nclass UnpairedDataset(torch.utils.data.Dataset):\n    def __init__(self, dataset_folder, split, image_prep, tokenizer):\n        """\n        A dataset class for loading unpaired data samples from two distinct domains (source and target),\n        typically used in unsupervised learning tasks like image-to-image translation.\n\n        The class supports loading images from specified dataset folders, applying predefined image\n        preprocessing transformations, and utilizing fixed textual prompts (captions) for each domain,\n        tokenized using a provided tokenizer.\n\n        Parameters:\n        - dataset_folder (str): Base directory of the dataset containing subdirectories (train_A, train_B, test_A, test_B)\n        - split (str): Indicates the dataset split to use. Expected values are \'train\' or \'test\'.\n        - image_prep (str): he image preprocessing transformation to apply to each image.\n        - tokenizer: The tokenizer used for tokenizing the captions (or prompts).\n        """\n        super().__init__()\n        if split == "train":\n            self.source_folder = os.path.join(dataset_folder, "train_A")\n            self.target_folder = os.path.join(dataset_folder, "train_B")\n        elif split == "test":\n            self.source_folder = os.path.join(dataset_folder, "test_A")\n            self.target_folder = os.path.join(dataset_folder, "test_B")\n        self.tokenizer = tokenizer\n        with open(os.path.join(dataset_folder, "fixed_prompt_a.txt"), "r") as f:\n            self.fixed_caption_src = f.read().strip()\n            self.input_ids_src = self.tokenizer(\n                self.fixed_caption_src, max_length=self.tokenizer.model_max_length,\n                padding="max_length", truncation=True, return_tensors="pt"\n            ).input_ids\n\n        with open(os.path.join(dataset_folder, "fixed_prompt_b.txt"), "r") as f:\n            self.fixed_caption_tgt = f.read().strip()\n            self.input_ids_tgt = self.tokenizer(\n                self.fixed_caption_tgt, max_length=self.tokenizer.model_max_length,\n                padding="max_length", truncation=True, return_tensors="pt"\n            ).input_ids\n        # find all images in the source and target folders with all IMG extensions\n        self.l_imgs_src = []\n        for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.gif"]:\n            self.l_imgs_src.extend(glob(os.path.join(self.source_folder, ext)))\n        self.l_imgs_tgt = []\n        for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.gif"]:\n            self.l_imgs_tgt.extend(glob(os.path.join(self.target_folder, ext)))\n        self.T = build_transform(image_prep)\n\n    def __len__(self):\n        """\n        Returns:\n        int: The total number of items in the dataset.\n        """\n        return len(self.l_imgs_src) + len(self.l_imgs_tgt)\n\n    def __getitem__(self, index):\n        """\n        Fetches a pair of unaligned images from the source and target domains along with their \n        corresponding tokenized captions.\n\n        For the source domain, if the requested index is within the range of available images,\n        the specific image at that index is chosen. If the index exceeds the number of source\n        images, a random source image is selected. For the target domain,\n        an image is always randomly selected, irrespective of the index, to maintain the \n        unpaired nature of the dataset.\n\n        Both images are preprocessed according to the specified image transformation `T`, and normalized.\n        The fixed captions for both domains\n        are included along with their tokenized forms.\n\n        Parameters:\n        - index (int): The index of the source image to retrieve.\n\n        Returns:\n        dict: A dictionary containing processed data for a single training example, with the following keys:\n            - "pixel_values_src": The processed source image\n            - "pixel_values_tgt": The processed target image\n            - "caption_src": The fixed caption of the source domain.\n            - "caption_tgt": The fixed caption of the target domain.\n            - "input_ids_src": The source domain\'s fixed caption tokenized.\n            - "input_ids_tgt": The target domain\'s fixed caption tokenized.\n        """\n        if index < len(self.l_imgs_src):\n            img_path_src = self.l_imgs_src[index]\n        else:\n            img_path_src = random.choice(self.l_imgs_src)\n        img_path_tgt = random.choice(self.l_imgs_tgt)\n        img_pil_src = Image.open(img_path_src).convert("RGB")\n        img_pil_tgt = Image.open(img_path_tgt).convert("RGB")\n        img_t_src = F.to_tensor(self.T(img_pil_src))\n        img_t_tgt = F.to_tensor(self.T(img_pil_tgt))\n        img_t_src = F.normalize(img_t_src, mean=[0.5], std=[0.5])\n        img_t_tgt = F.normalize(img_t_tgt, mean=[0.5], std=[0.5])\n        return {\n            "pixel_values_src": img_t_src,\n            "pixel_values_tgt": img_t_tgt,\n            "caption_src": self.fixed_caption_src,\n            "caption_tgt": self.fixed_caption_tgt,\n            "input_ids_src": self.input_ids_src,\n            "input_ids_tgt": self.input_ids_tgt,\n        }\n\n\nclass RealESRGANDataset(data.Dataset):\n    def __init__(self, gt_folders):\n        super(RealESRGANDataset, self).__init__()\n        image_extensions = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff", ".webp"}\n        self.paths = []\n\n        for folder in gt_folders:\n            for root, _, files in os.walk(folder):\n                for file in files:\n                    if os.path.splitext(file)[1].lower() in image_extensions:\n                        self.paths.append(os.path.join(root, file))\n\n        self.resize_prob = [0.2, 0.7, 0.1]\n        self.resize_range = [0.15, 1.5]\n        self.gaussian_noise_prob = 0.5\n        self.noise_range = [1, 30]\n        self.poisson_scale_range = [0.05, 3]\n        self.gray_noise_prob = 0.4\n        self.jpeg_range = [30, 95]\n\n        self.second_blur_prob = 0.8\n        self.resize_prob2 = [0.2, 0.7, 0.1]\n        self.resize_range2 = [0.15, 1.5]\n        self.gaussian_noise_prob2 = 0.5\n        self.noise_range2 = [1, 30]\n        self.poisson_scale_range2 = [0.05, 3]\n        self.gray_noise_prob2 = 0.4\n        self.jpeg_range2 = [30, 95]\n\n        self.scale = 4\n        self.gt_size = 512\n\n        # blur settings for the first degradation\n        self.blur_kernel_size = 21\n        self.kernel_list = [\'iso\', \'aniso\', \'generalized_iso\', \'generalized_aniso\', \'plateau_iso\', \'plateau_aniso\']\n        self.kernel_prob = [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]\n        self.blur_sigma = [0.2, 3]\n        self.betag_range = [0.5, 4]\n        self.betap_range = [1, 2]\n        self.sinc_prob = 0.1\n\n        # blur settings for the second degradation\n        self.blur_kernel_size2 = 21\n        self.kernel_list2 = [\'iso\', \'aniso\', \'generalized_iso\', \'generalized_aniso\', \'plateau_iso\', \'plateau_aniso\']\n        self.kernel_prob2 = [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]\n        self.blur_sigma2 = [0.2, 1.5]\n        self.betag_range2 = [0.5, 4]\n        self.betap_range2 = [1, 2]\n        self.sinc_prob2 = 0.1\n\n        # a final sinc filter\n        self.final_sinc_prob = 0.8\n\n        self.kernel_range = [2 * v + 1 for v in range(3, 11)]  # kernel size ranges from 7 to 21\n        # TODO: kernel range is now hard-coded, should be in the configure file\n        self.pulse_tensor = torch.zeros(21, 21).float()  # convolving with pulse tensor brings no blurry effect\n        self.pulse_tensor[10, 10] = 1\n\n    def __getitem__(self, index):\n        # -------------------------------- Load gt images -------------------------------- #\n        # Shape: (h, w, c); channel order: BGR; image range: [0, 1], float32.\n        gt_path = self.paths[index]\n        img_gt = cv2.imread(gt_path).astype(np.float32) / 255\n\n        # -------------------- Do augmentation for training: flip, rotation -------------------- #\n        img_gt = augment(img_gt, hflip=True, rotation=True)\n\n        # crop or pad to 400\n        h, w = img_gt.shape[0:2]\n        crop_pad_size = self.gt_size\n        # pad\n        if h < crop_pad_size or w < crop_pad_size:\n            pad_h = max(0, crop_pad_size - h)\n            pad_w = max(0, crop_pad_size - w)\n            img_gt = cv2.copyMakeBorder(img_gt, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)\n        # crop\n        if img_gt.shape[0] > crop_pad_size or img_gt.shape[1] > crop_pad_size:\n            h, w = img_gt.shape[0:2]\n            # randomly choose top and left coordinates\n            top = random.randint(0, h - crop_pad_size)\n            left = random.randint(0, w - crop_pad_size)\n            img_gt = img_gt[top:top + crop_pad_size, left:left + crop_pad_size, ...]\n\n        # ------------------------ Generate kernels (used in the first degradation) ------------------------ #\n        kernel_size = random.choice(self.kernel_range)\n        if np.random.uniform() < self.sinc_prob:\n            # this sinc filter setting is for kernels ranging from [7, 21]\n            if kernel_size < 13:\n                omega_c = np.random.uniform(np.pi / 3, np.pi)\n            else:\n                omega_c = np.random.uniform(np.pi / 5, np.pi)\n            kernel = circular_lowpass_kernel(omega_c, kernel_size, pad_to=False)\n        else:\n            kernel = random_mixed_kernels(\n                self.kernel_list,\n                self.kernel_prob,\n                kernel_size,\n                self.blur_sigma,\n                self.blur_sigma, [-math.pi, math.pi],\n                self.betag_range,\n                self.betap_range,\n                noise_range=None)\n        # pad kernel\n        pad_size = (21 - kernel_size) // 2\n        kernel = np.pad(kernel, ((pad_size, pad_size), (pad_size, pad_size)))\n\n        # ------------------------ Generate kernels (used in the second degradation) ------------------------ #\n        kernel_size = random.choice(self.kernel_range)\n        if np.random.uniform() < self.sinc_prob2:\n            if kernel_size < 13:\n                omega_c = np.random.uniform(np.pi / 3, np.pi)\n            else:\n                omega_c = np.random.uniform(np.pi / 5, np.pi)\n            kernel2 = circular_lowpass_kernel(omega_c, kernel_size, pad_to=False)\n        else:\n            kernel2 = random_mixed_kernels(\n                self.kernel_list2,\n                self.kernel_prob2,\n                kernel_size,\n                self.blur_sigma2,\n                self.blur_sigma2, [-math.pi, math.pi],\n                self.betag_range2,\n                self.betap_range2,\n                noise_range=None)\n\n        # pad kernel\n        pad_size = (21 - kernel_size) // 2\n        kernel2 = np.pad(kernel2, ((pad_size, pad_size), (pad_size, pad_size)))\n\n        # ------------------------------------- the final sinc kernel ------------------------------------- #\n        if np.random.uniform() < self.final_sinc_prob:\n            kernel_size = random.choice(self.kernel_range)\n            omega_c = np.random.uniform(np.pi / 3, np.pi)\n            sinc_kernel = circular_lowpass_kernel(omega_c, kernel_size, pad_to=21)\n            sinc_kernel = torch.FloatTensor(sinc_kernel)\n        else:\n            sinc_kernel = self.pulse_tensor\n\n        # BGR to RGB, HWC to CHW, numpy to tensor\n        img_gt = img2tensor([img_gt], bgr2rgb=True, float32=True)[0]\n        kernel = torch.FloatTensor(kernel)\n        kernel2 = torch.FloatTensor(kernel2)\n\n        return_d = {\'gt\': img_gt, \'kernel1\': kernel, \'kernel2\': kernel2, \'sinc_kernel\': sinc_kernel, \'gt_path\': gt_path}\n        return return_d\n\n    def __len__(self):\n        return len(self.paths)\n\n\nclass LSDIRDataset(data.Dataset):\n    def __init__(self, dataset_file_path: str, crop_size: int = 512):\n        super().__init__()\n        with open(dataset_file_path, \'r\') as f:\n            self.img_paths = [fp.strip() for fp in f.readlines()]\n        self.T = transforms.Compose([\n            transforms.RandomResizedCrop(crop_size, scale=(0.2, 1.0),\n                                         interpolation=transforms.InterpolationMode.LANCZOS)\n        ])\n\n    def __len__(self):\n        return len(self.img_paths)\n\n    def __getitem__(self, idx):\n        img_path = self.img_paths[idx]\n        input_img = Image.open(img_path).convert("RGB")\n\n        img_t = self.T(input_img)\n        img_t = F.to_tensor(img_t)\n\n        fno = os.path.splitext(os.path.basename(img_path))[0]\n        return {\n            "fname": fno,\n            "input": img_t\n        }\n\n\nclass SRTestDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n\n        self.img_names = list(os.listdir(self.input_folder))\n        self.T = transforms.Compose([\n            transforms.Resize(512, interpolation=transforms.InterpolationMode.LANCZOS),\n            transforms.CenterCrop(512),\n        ])\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        img_name = self.img_names[idx]\n        input_img = Image.open(os.path.join(self.input_folder, img_name)).convert("RGB")\n\n        img_t = self.T(input_img)\n        img_t = F.to_tensor(img_t)\n\n        return {\n            "fname": os.path.splitext(img_name)[0],\n            "input": img_t\n        }\n\n\nclass CTRSynDataset(data.Dataset):\n    def __init__(self, dataset_folder: str, max_no=100000):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.max_no = max_no\n\n    def __len__(self):\n        return self.max_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + 1).zfill(7))\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n        gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n        anno_path = os.path.join(self.input_folder, "anno", name + ".pkl")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n\n        input_img = F.to_tensor(input_img)\n        gt_img = F.to_tensor(gt_img)\n        anno = torch.load(anno_path)\n\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "anno": anno\n        }\n\n\nclass FTSRSynDataset(data.Dataset):\n    def __init__(self, dataset_folder: str, min_no=0, max_no=500):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.min_no = min_no\n        self.max_no = max_no\n\n    def __len__(self):\n        return self.max_no - self.min_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + self.min_no + 1).zfill(7))\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n        gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n        seg_path = os.path.join(self.input_folder, "seg", name + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n        seg_img = cv2.imread(seg_path, flags=cv2.IMREAD_ANYDEPTH)\n        input_img = F.to_tensor(input_img)\n        gt_img = F.to_tensor(gt_img)\n        seg_img = torch.from_numpy(seg_img).float().unsqueeze(0)\n\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "seg": seg_img\n        }\n\n\nclass FTSRCBDataset(data.Dataset):\n    def __init__(self, dataset_folder: str, min_no=0, max_no=500):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.min_no = min_no\n        self.max_no = max_no\n        self.default_dict = None\n        self.store_flag = False\n\n    def __len__(self):\n        return self.max_no - self.min_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + self.min_no + 1).zfill(7))\n        try:\n            lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n            gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n            mask_path = os.path.join(self.input_folder, "mask", name + ".png")\n            lr_mask_path = os.path.join(self.input_folder, "lr_mask", name + ".png")\n\n            input_img = Image.open(lr_path).convert("RGB")\n            gt_img = Image.open(gt_path).convert("RGB")\n            mask_img = Image.open(mask_path).convert("L")\n            lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n            input_img = F.to_tensor(input_img)\n            gt_img = F.to_tensor(gt_img)\n            mask_img = F.to_tensor(mask_img)\n            lr_mask_img = F.to_tensor(lr_mask_img)\n\n            res = {\n                "name": name,\n                "input": input_img,\n                "gt": gt_img,\n                "mask": mask_img,\n                "lr_mask": lr_mask_img\n            }\n\n            if not self.store_flag:\n                self.default_dict = res\n                self.store_flag = True\n                # self.default_dict[\'name\'] = "ERROR"\n            return res\n        except Exception as e:\n            print(f"Error loading image {name}: {e}")\n            return self.default_dict\n\n\nclass FTSR1024Dataset(data.Dataset):\n    def __init__(self, dataset_folder: str, min_no=0, max_no=500):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.min_no = min_no\n        self.max_no = max_no\n        self.default_dict = None\n        self.store_flag = False\n\n    def __len__(self):\n        return self.max_no - self.min_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + self.min_no + 1).zfill(7))\n        try:\n            lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n            gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n            mask_path = os.path.join(self.input_folder, "mask", name + ".png")\n\n            input_img = Image.open(lr_path).convert("RGB")\n            gt_img = Image.open(gt_path).convert("RGB")\n            mask_img = Image.open(mask_path).convert("L")\n\n            input_img = F.to_tensor(input_img)\n            gt_img = F.to_tensor(gt_img)\n            mask_img = F.to_tensor(mask_img)\n\n            res = {\n                "name": name,\n                "input": input_img,\n                "gt": gt_img,\n                "mask": mask_img,\n            }\n\n            if not self.store_flag:\n                self.default_dict = res\n                self.store_flag = True\n                # self.default_dict[\'name\'] = "ERROR"\n            return res\n        except Exception as e:\n            print(f"Error loading image {name}: {e}")\n            return self.default_dict\n\n\nclass FTSR768Dataset(data.Dataset):\n    def __init__(self, dataset_folder: str, min_no=0, max_no=500):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.min_no = min_no\n        self.max_no = max_no\n        self.default_dict = None\n        self.store_flag = False\n        self.transform = transforms.Compose(\n            [transforms.Resize(768, interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n        self.transform2 = transforms.Resize(768, interpolation=transforms.InterpolationMode.LANCZOS)\n\n    def __len__(self):\n        return self.max_no - self.min_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + self.min_no + 1).zfill(7))\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n        gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n        mask_path = os.path.join(self.input_folder, "mask", name + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        input_img = self.transform(input_img)\n        gt_img = self.transform(gt_img)\n        mask_img_tensor = self.transform(mask_img)\n        mask_ori = self.transform2(mask_img)\n        mask_ori = np.expand_dims(np.array(mask_ori), axis=-1)\n        mask_ori = np.repeat(mask_ori, 3, axis=-1)\n\n        res = {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "mask": mask_img_tensor,\n            "mask_ori": mask_ori\n        }\n        return res\n\nclass FTSR512Dataset(data.Dataset):\n    def __init__(self, dataset_folder: str, min_no=0, max_no=500):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.min_no = min_no\n        self.max_no = max_no\n        self.default_dict = None\n        self.store_flag = False\n        self.transform = transforms.Compose(\n            [transforms.Resize(512, interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n        self.transform2 = transforms.Resize(512, interpolation=transforms.InterpolationMode.LANCZOS)\n\n    def __len__(self):\n        return self.max_no - self.min_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + self.min_no + 1).zfill(7))\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n        gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n        mask_path = os.path.join(self.input_folder, "mask", name + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        input_img = self.transform(input_img)\n        gt_img = self.transform(gt_img)\n        mask_img_tensor = self.transform(mask_img)\n        mask_ori = self.transform2(mask_img)\n        mask_ori = np.expand_dims(np.array(mask_ori), axis=-1)\n        mask_ori = np.repeat(mask_ori, 3, axis=-1)\n\n        res = {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "mask": mask_img_tensor,\n            "mask_ori": mask_ori\n        }\n        return res\n\n\nclass FTSRCBDataset2(data.Dataset):\n    def __init__(self, dataset_folder: str, min_no=0, max_no=500):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.min_no = min_no\n        self.max_no = max_no\n        self.default_dict = None\n        self.store_flag = False\n\n    def __len__(self):\n        return self.max_no - self.min_no\n\n    def __getitem__(self, idx):\n        name = \'{}\'.format(str(idx + self.min_no + 1).zfill(7))\n        try:\n            lr_path = os.path.join(self.input_folder, "sr_bicubic", name + ".png")\n            gt_path = os.path.join(self.input_folder, "gt", name + ".png")\n            mask_path = os.path.join(self.input_folder, "mask", name + ".png")\n            lr_mask_path = os.path.join(self.input_folder, "rendered_mask", name + ".png")\n\n            input_img = Image.open(lr_path).convert("RGB")\n            gt_img = Image.open(gt_path).convert("RGB")\n            mask_img = Image.open(mask_path).convert("L")\n            lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n            input_img = F.to_tensor(input_img)\n            gt_img = F.to_tensor(gt_img)\n            mask_img = F.to_tensor(mask_img)\n            lr_mask_img = F.to_tensor(lr_mask_img)\n\n            res = {\n                "name": name,\n                "input": input_img,\n                "gt": gt_img,\n                "mask": mask_img,\n                "lr_mask": lr_mask_img\n            }\n\n            if not self.store_flag:\n                self.default_dict = res\n                self.store_flag = True\n                # self.default_dict[\'name\'] = "ERROR"\n            return res\n        except Exception as e:\n            print(f"Error loading image {name}: {e}")\n            return self.default_dict\n\n\nclass ResizeLongestSide_ToTensor(object):\n    def __init__(self, target_length=512):\n        self.target_length = target_length\n\n    @staticmethod\n    def get_preprocess_shape(oldh: int, oldw: int, long_side_length: int):\n        scale = long_side_length * 1.0 / max(oldh, oldw)\n        newh, neww = oldh * scale, oldw * scale\n        neww = int(neww + 0.5)\n        newh = int(newh + 0.5)\n        return [newh, neww]\n\n    def __call__(self, image, value=128, grayscale=False):\n        target_size = self.get_preprocess_shape(*image.shape[:-1], self.target_length)\n        padding_h = max(self.target_length - target_size[0], 0)\n        padding_w = max(self.target_length - target_size[1], 0)\n\n        image = cv2.resize(image, (target_size[1], target_size[0]), interpolation=cv2.INTER_LINEAR)\n        image = cv2.copyMakeBorder(image, 0, padding_h, 0, padding_w, cv2.BORDER_CONSTANT, value=(value, value, value))\n        if not grayscale:\n            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n        image = torch.as_tensor(image, dtype=torch.float32) / 255.\n        image = image.permute(2, 0, 1)\n        return image\n\n\nclass FTSRRealCEDataset(data.Dataset):\n    def __init__(self, dataset_folder: str, img_list_file):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        with open(img_list_file, "r") as f:\n            img_list = f.readlines()\n        self.img_list = [line.strip() for line in img_list]\n        self.img_list.sort()\n        self.transform = ResizeLongestSide_ToTensor()\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        name = os.path.splitext(self.img_list[idx])[0]\n        try:\n            lr_path = os.path.join(self.input_folder, "13mm", name + ".JPG")\n            gt_path = os.path.join(self.input_folder, "52mm", name + ".JPG")\n            lr_mask_path = os.path.join(self.input_folder, "render-13mm", name + ".png")\n\n            input_img = cv2.imread(lr_path)\n            gt_img = cv2.imread(gt_path)\n            lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n            input_img = self.transform(input_img)\n            gt_img = self.transform(gt_img)\n            lr_mask_img = F.to_tensor(lr_mask_img)\n\n            res = {\n                "name": name,\n                "input": input_img,\n                "gt": gt_img,\n                "lr_mask": lr_mask_img\n            }\n\n            if not self.store_flag:\n                self.default_dict = res\n                self.store_flag = True\n                # self.default_dict[\'name\'] = "ERROR"\n            return res\n        except Exception as e:\n            print(f"Error loading image {name}: {e}")\n            return self.default_dict\n\n\nclass FTSRRealCEDataset2(data.Dataset):\n    def __init__(self, dataset_folder: str, img_list_file):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        with open(img_list_file, "r") as f:\n            img_list = f.readlines()\n        self.img_list = [line.strip() for line in img_list]\n        self.img_list.sort()\n        self.transform = ResizeLongestSide_ToTensor()\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        name = os.path.splitext(self.img_list[idx])[0]\n        lr_path = os.path.join(self.input_folder, "13mm_aligned_1024", name + ".JPG")\n        gt_path = os.path.join(self.input_folder, "52mm_aligned_1024", name + ".JPG")\n        lr_mask_path = os.path.join(self.input_folder, "mask-52mm-bts-1024-v2", name + ".png")\n\n        input_img = cv2.imread(lr_path)\n        gt_img = cv2.imread(gt_path)\n        lr_mask_img = cv2.imread(lr_mask_path)\n        # print(lr_mask_img.shape)\n\n        input_img = self.transform(input_img, 128, grayscale=False)\n        gt_img = self.transform(gt_img, 128, grayscale=False)\n        lr_mask_img = self.transform(lr_mask_img, 0, grayscale=True)[:1]\n\n        res = {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "lr_mask": lr_mask_img\n        }\n\n        return res\n\n\nclass ResizeLongestSide:\n    def __init__(self, target_length: int, interpolation=transforms.InterpolationMode.LANCZOS):\n        self.target_length = target_length\n        self.interpolation = interpolation\n\n    def __call__(self, img):\n        if not isinstance(img, Image.Image):\n            raise TypeError(f\'Input should be a PIL Image. Got {type(img)}\')\n\n        width, height = img.size\n        if width >= height:\n            scale = self.target_length / width\n        else:\n            scale = self.target_length / height\n\n        new_width = int(math.ceil(width * scale) // 32 * 32)\n        new_height = int(math.ceil(height * scale) // 32 * 32)\n\n        return F.resize(img, (new_height, new_width), interpolation=self.interpolation)\n\n    @staticmethod\n    def get_new_size(shape, target_length):\n        width, height = shape\n        if width >= height:\n            scale = target_length / width\n        else:\n            scale = target_length / height\n\n        new_width = int(math.ceil(width * scale) // 32 * 32)\n        new_height = int(math.ceil(height * scale) // 32 * 32)\n        return new_width, new_height\n\n    def __repr__(self):\n        return f"{self.__class__.__name__}(target_length={self.target_length}, interpolation={self.interpolation})"\n\n\nclass ResizeRound:\n    def __init__(self, interpolation=transforms.InterpolationMode.LANCZOS):\n        self.interpolation = interpolation\n\n    def __call__(self, img):\n        if not isinstance(img, Image.Image):\n            raise TypeError(f\'Input should be a PIL Image. Got {type(img)}\')\n\n        width, height = img.size\n\n        new_width = int(math.ceil(width) // 32 * 32)\n        new_height = int(math.ceil(height) // 32 * 32)\n\n        return F.resize(img, (new_height, new_width), interpolation=self.interpolation)\n\n\nclass FTSRRealCEDataset512(data.Dataset):\n    def __init__(self, dataset_folder: str, img_list_file):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        with open(img_list_file, "r") as f:\n            img_list = f.readlines()\n        self.img_list = [line.strip() for line in img_list]\n        self.img_list.sort()\n\n        self.transform = transforms.Compose(\n            [ResizeLongestSide(512, interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n        self.transform2 = ResizeLongestSide(512, interpolation=transforms.InterpolationMode.LANCZOS)\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        name = os.path.splitext(self.img_list[idx])[0]\n        lr_path = os.path.join(self.input_folder, "13mm_aligned_1024", name + ".JPG")\n        gt_path = os.path.join(self.input_folder, "52mm_aligned_1024", name + ".JPG")\n        mask_path = os.path.join(self.input_folder, "mask-52mm-bts-1024-v2", name + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        input_img = self.transform(input_img)\n        gt_img = self.transform(gt_img)\n        mask_img_tensor = self.transform(mask_img)\n        mask_ori = self.transform2(mask_img)\n        mask_ori = np.expand_dims(np.array(mask_ori), axis=-1)\n        mask_ori = np.repeat(mask_ori, 3, axis=-1)\n\n        res = {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "mask": mask_img_tensor,\n            "mask_ori": mask_ori\n        }\n        return res\n\n\nclass FTSRRealCEDatasetUA768(data.Dataset):\n    def __init__(self, dataset_folder: str, img_list_file):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        with open(img_list_file, "r") as f:\n            img_list = f.readlines()\n        self.img_list = [line.strip() for line in img_list]\n        self.img_list.sort()\n\n        self.transform = transforms.Compose(\n            [ResizeLongestSide(768, interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n        self.transform2 = ResizeLongestSide(768, interpolation=transforms.InterpolationMode.LANCZOS)\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        name = os.path.splitext(self.img_list[idx])[0]\n        lr_path = os.path.join(self.input_folder, "13mm", name + ".JPG")\n        gt_path = os.path.join(self.input_folder, "52mm", name + ".JPG")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n\n        input_img = self.transform(input_img)\n        gt_img = self.transform(gt_img)\n\n        res = {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n        }\n        return res\n\n\nclass FTSRRealCEDatasetScaled(data.Dataset):\n    def __init__(self, dataset_folder: str, img_list_file):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        with open(img_list_file, "r") as f:\n            img_list = f.readlines()\n        self.img_list = [line.strip() for line in img_list]\n        self.img_list.sort()\n\n        self.interpolation = transforms.InterpolationMode.LANCZOS\n        self.transform = transforms.ToTensor()\n        self.transform2 = ResizeLongestSide(496, interpolation=self.interpolation)\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        name = os.path.splitext(self.img_list[idx])[0]\n        lr_path = os.path.join(self.input_folder, "13mm_aligned_1024", name + ".JPG")\n        gt_path = os.path.join(self.input_folder, "52mm_aligned_1024", name + ".JPG")\n        mask_path = os.path.join(self.input_folder, "mask-52mm-bts-1024-v2", name + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        gt_img = Image.open(gt_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        orig_width, orig_height = ResizeLongestSide.get_new_size((input_img.width, input_img.height), 496)\n\n        scale = 0\n        new_size = None\n        # 是否上采样\n        if random.random() < 0.7:\n            scale = random.uniform(2, 6)\n            new_size = (int(orig_height * scale), int(orig_width * scale))\n            input_img = F.resize(input_img, new_size, interpolation=self.interpolation)\n            gt_img = F.resize(gt_img, new_size, interpolation=self.interpolation)\n            mask_img = F.resize(mask_img, new_size, interpolation=self.interpolation)\n        else:\n            input_img = self.transform2(input_img)\n            gt_img = self.transform2(gt_img)\n            mask_img = self.transform2(mask_img)\n        # 裁剪 patch\n        x_max = input_img.width - orig_width\n        y_max = input_img.height - orig_height\n\n        resized_size = (input_img.width, input_img.height)\n        if x_max > 0 and y_max > 0:\n            x = random.randint(0, x_max)\n            y = random.randint(0, y_max)\n            input_img = input_img.crop((x, y, x + orig_width, y + orig_height))\n            gt_img = gt_img.crop((x, y, x + orig_width, y + orig_height))\n            mask_img = mask_img.crop((x, y, x + orig_width, y + orig_height))\n\n        # 转 tensor\n        input_img = self.transform(input_img)\n        gt_img = self.transform(gt_img)\n        mask_img_tensor = self.transform(mask_img)\n        # print((orig_width, orig_height), scale, new_size, resized_size, (x_max, y_max), input_img.shape)\n\n        mask_ori = self.transform2(mask_img)\n        mask_ori = np.expand_dims(np.array(mask_ori), axis=-1)\n        mask_ori = np.repeat(mask_ori, 3, axis=-1)\n\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": gt_img,\n            "mask": mask_img_tensor,\n            "mask_ori": mask_ori\n        }\n\n\nclass FTSRRealTestDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        self.img_list = os.listdir(dataset_folder)\n        self.img_list.sort()\n\n        self.transform = transforms.Compose(\n            [ResizeRound(interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        fname = self.img_list[idx]\n        fno = os.path.splitext(fname)[0]\n        lr_path = os.path.join(self.input_folder, fname)\n        input_img = Image.open(lr_path).convert("RGB")\n        input_img = self.transform(input_img)\n        res = {\n            "name": fno,\n            "input": input_img\n        }\n        return res\n\n\nclass FTSRRealTestDataset768(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        self.img_list = os.listdir(dataset_folder)\n        self.img_list.sort()\n\n        self.transform = transforms.Compose(\n            [ResizeLongestSide(768, interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        fname = self.img_list[idx]\n        fno = os.path.splitext(fname)[0]\n        lr_path = os.path.join(self.input_folder, fname)\n        input_img = Image.open(lr_path).convert("RGB")\n        input_img = self.transform(input_img)\n        res = {\n            "name": fno,\n            "input": input_img\n        }\n        return res\n\n\nclass FTSRRealTestDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        self.img_list = os.listdir(dataset_folder)\n        self.img_list.sort()\n\n        self.transform = transforms.Compose(\n            [ResizeRound(interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        fname = self.img_list[idx]\n        fno = os.path.splitext(fname)[0]\n        lr_path = os.path.join(self.input_folder, fname)\n        input_img = Image.open(lr_path).convert("RGB")\n        input_img = self.transform(input_img)\n        res = {\n            "name": fno,\n            "input": input_img\n        }\n        return res\n\n\nclass FTSRRealTestDataset512(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.default_dict = None\n        self.store_flag = False\n        self.img_list = os.listdir(dataset_folder)\n        self.img_list.sort()\n\n        self.transform = transforms.Compose(\n            [ResizeLongestSide(512, interpolation=transforms.InterpolationMode.LANCZOS),\n             transforms.ToTensor()]\n        )\n\n    def __len__(self):\n        return len(self.img_list)\n\n    def __getitem__(self, idx):\n        fname = self.img_list[idx]\n        fno = os.path.splitext(fname)[0]\n        lr_path = os.path.join(self.input_folder, fname)\n        input_img = Image.open(lr_path).convert("RGB")\n        input_img = self.transform(input_img)\n        res = {\n            "name": fno,\n            "input": input_img\n        }\n        return res\n\n\nclass FTSRRealDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "lr_resize_deg_pil2"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "lr_resize_deg_pil2", name)\n        hr_path = os.path.join(self.input_folder, "lr", name)\n        seg_path = os.path.join(self.input_folder, "seg", name.split(\'.\')[0] + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        seg_img = cv2.imread(seg_path, flags=cv2.IMREAD_ANYDEPTH)\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n        seg_img = torch.from_numpy(seg_img).float().unsqueeze(0)\n\n        input_img = F.resize(input_img, size=[512, 512], antialias=True)\n        hr_img = F.resize(hr_img, size=[512, 512], antialias=True)\n        seg_img = F.resize(seg_img, size=[512, 512], antialias=True)\n        return {\n            "name": name,\n            "input": input_img,\n            "hr": hr_img,\n            "seg": seg_img\n        }\n\n\nclass FTSRRealDataset2(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "sr_bicubic"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        hr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        mask_path = os.path.join(self.input_folder, "mask", name.split(\'.\')[0] + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n        mask_img = F.to_tensor(mask_img)\n\n        input_img = F.resize(input_img, size=[512, 512])\n        hr_img = F.resize(hr_img, size=[512, 512])\n        mask_img = F.resize(mask_img, size=[512, 512])\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": hr_img,\n            "mask": mask_img\n        }\n\n\nclass FTSRRealDataset4(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "sr_bicubic"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        hr_path = os.path.join(self.input_folder, "gt", name)\n        mask_path = os.path.join(self.input_folder, "mask", name.split(\'.\')[0] + ".png")\n        lr_mask_path = os.path.join(self.input_folder, "lr_mask", name.split(\'.\')[0] + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n        lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n        mask_img = F.to_tensor(mask_img)\n        lr_mask_img = F.to_tensor(lr_mask_img)\n\n        input_img = F.resize(input_img, size=[512, 512])\n        hr_img = F.resize(hr_img, size=[512, 512])\n        mask_img = F.resize(mask_img, size=[512, 512])\n        lr_mask_img = F.resize(lr_mask_img, size=[512, 512])\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": hr_img,\n            "mask": mask_img,\n            "lr_mask": lr_mask_img\n        }\n\n\nclass FTSRRealDataset5(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "sr_bicubic"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        hr_path = os.path.join(self.input_folder, "gt", name)\n        lr_mask_path = os.path.join(self.input_folder, "lr_mask", name.split(\'.\')[0] + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n        lr_mask_img = F.to_tensor(lr_mask_img)\n\n        input_img = F.resize(input_img, size=[512, 512])\n        hr_img = F.resize(hr_img, size=[512, 512])\n        lr_mask_img = F.resize(lr_mask_img, size=[512, 512])\n        return {\n            "name": name,\n            "gt": hr_img,\n            "input": input_img,\n            "lr_mask": lr_mask_img\n        }\n\n\nclass FTSRRealDataset6(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "sr_bicubic"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        hr_path = os.path.join(self.input_folder, "gt", name)\n        lr_mask_path = os.path.join(self.input_folder, "rendered_mask", name.split(\'.\')[0] + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n        lr_mask_img = F.to_tensor(lr_mask_img)\n\n        input_img = F.resize(input_img, size=[512, 512])\n        hr_img = F.resize(hr_img, size=[512, 512])\n        lr_mask_img = F.resize(lr_mask_img, size=[512, 512])\n        return {\n            "name": name,\n            "gt": hr_img,\n            "input": input_img,\n            "lr_mask": lr_mask_img\n        }\n\n\nclass FTSRRealDatasetTest(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(self.input_folder)\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, name)\n        input_img = Image.open(lr_path).convert("RGB")\n        input_img = F.to_tensor(input_img)\n        resized_image = resize_aspect_ratio(input_img, 512, interpolation=F.InterpolationMode.BILINEAR)\n        padded_image = pad_to_max_length(resized_image, 512, fill=0.)\n        return {\n            "name": name,\n            "input": padded_image\n        }\n\n\ndef resize_aspect_ratio(image: torch.Tensor, max_length: int,\n                        interpolation: F.InterpolationMode = F.InterpolationMode.BILINEAR) -> torch.Tensor:\n    """\n    将输入的图像张量按长边符合给定长度并保持宽高比进行缩放。\n\n    Args:\n        image (torch.Tensor): 输入图像张量，形状为 (C, H, W) 或 (B, C, H, W)。\n        max_length (int): 长边的目标长度。\n        interpolation (InterpolationMode): 插值方式，默认双线性插值。\n\n    Returns:\n        torch.Tensor: 缩放后的图像张量，保持原始宽高比。\n    """\n    # 检查输入维度\n    if image.dim() == 3:\n        # (C, H, W)\n        channels, height, width = image.shape\n    elif image.dim() == 4:\n        # (B, C, H, W)\n        batch, channels, height, width = image.shape\n    else:\n        raise ValueError("输入的图像张量维度必须是 3 或 4。")\n\n    # 确定当前长边和短边\n    if height >= width:\n        new_height = max_length\n        new_width = int(round((max_length / height) * width))\n    else:\n        new_width = max_length\n        new_height = int(round((max_length / width) * height))\n\n    # 使用F.resize进行缩放\n    resized_image = F.resize(image, size=[new_height, new_width], interpolation=interpolation)\n\n    return resized_image\n\n\ndef pad_to_max_length(image: torch.Tensor, max_length: int, fill=0.):\n    from math import floor, ceil\n    # 检查输入维度\n    if image.dim() == 3:\n        # (C, H, W)\n        channels, height, width = image.shape\n        batch = False\n    elif image.dim() == 4:\n        # (B, C, H, W)\n        batch, channels, height, width = image.shape\n    else:\n        raise ValueError("输入的图像张量维度必须是 3 或 4。")\n\n    # 计算需要的填充量\n    if height < max_length:\n        pad_vert = max_length - height\n        pad_top = floor(pad_vert / 2)\n        pad_bottom = ceil(pad_vert / 2)\n    else:\n        pad_top = pad_bottom = 0\n\n    if width < max_length:\n        pad_horiz = max_length - width\n        pad_left = floor(pad_horiz / 2)\n        pad_right = ceil(pad_horiz / 2)\n    else:\n        pad_left = pad_right = 0\n\n    padding = [pad_left, pad_top, pad_right, pad_bottom]  # 顺序为 (左, 上, 右, 下)\n\n    if batch:\n        # 对于批量图像，逐个应用填充\n        padded_images = []\n        for img in image:\n            padded_img = F.pad(img, padding, fill=fill, padding_mode=\'constant\')\n            padded_images.append(padded_img)\n        padded_image = torch.stack(padded_images)\n    else:\n        # 单张图像\n        padded_image = F.pad(image, padding, fill=fill, padding_mode=\'constant\')\n\n    return padded_image\n\n\ndef get_bounding_box(mask: torch.Tensor) -> Tuple[int, int, int, int]:\n    """\n    根据掩码计算外接的边界框。\n\n    Args:\n        mask (torch.Tensor): 二值掩码张量，形状为 (H, W)。\n\n    Returns:\n        Tuple[int, int, int, int]: 边界框坐标 (left, top, right, bottom)。\n    """\n    assert mask.dim() == 2, "掩码必须是二维张量 (H, W)。"\n    # 找到非零点的坐标\n    non_zero = mask.nonzero(as_tuple=False)\n    if non_zero.numel() == 0:\n        # 如果掩码全为零，返回整个图像\n        return 0, 0, mask.shape[1], mask.shape[0]\n    y_min = torch.min(non_zero[:, 0]).item()\n    y_max = torch.max(non_zero[:, 0]).item()\n    x_min = torch.min(non_zero[:, 1]).item()\n    x_max = torch.max(non_zero[:, 1]).item()\n    return x_min, y_min, x_max, y_max\n\n\ndef crop_image_and_mask(image: torch.Tensor, mask: torch.Tensor, bbox: Tuple[int, int, int, int]) -> Tuple[\n    torch.Tensor, torch.Tensor]:\n    """\n    根据边界框裁剪图像和掩码。\n\n    Args:\n        image (torch.Tensor): 输入图像张量，形状为 (C, H, W)。\n        mask (torch.Tensor): 输入掩码张量，形状为 (H, W)。\n        bbox (Tuple[int, int, int, int]): 边界框坐标 (left, top, right, bottom)。\n\n    Returns:\n        Tuple[torch.Tensor, torch.Tensor]: 裁剪后的图像和掩码。\n    """\n    left, top, right, bottom = bbox\n    cropped_image = image[:, top:bottom, left:right]\n    cropped_mask = mask[top:bottom, left:right]\n    return cropped_image, cropped_mask\n\n\ndef apply_gaussian_blur(image: torch.Tensor, kernel_size=5, sigma: float = 1.0) -> torch.Tensor:\n    """\n    对图像应用高斯模糊。\n\n    Args:\n        image (torch.Tensor): 输入图像张量，形状为 (C, H, W) 或 (B, C, H, W)。\n        kernel_size (int): 高斯核大小。必须是正奇数。\n        sigma (float): 高斯核的标准差。\n\n    Returns:\n        torch.Tensor: 应用高斯模糊后的图像张量。\n    """\n    # 确保kernel_size是奇数\n    if kernel_size % 2 == 0:\n        raise ValueError("kernel_size 必须是正奇数。")\n\n    # 应用高斯模糊\n    blurred_image = F.gaussian_blur(image, kernel_size=kernel_size, sigma=sigma)\n    return blurred_image\n\n\ndef process_image_and_mask(\n        image: torch.Tensor,\n        mask: torch.Tensor,\n        target_length: int,\n        fill_image=0,\n        fill_mask=0,\n        blur_kernel_size: int = 5,\n        blur_sigma: float = 1.0\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    """\n    根据掩码计算边界框，裁剪图像和掩码，仅保留文本区域，应用高斯模糊，然后进行缩放和填充。\n\n    Args:\n        image (torch.Tensor): 输入图像张量，形状为 (C, H, W)。\n        mask (torch.Tensor): 输入掩码张量，形状为 (H, W)。\n        target_length (int): 规范化的目标长度（长边和填充后的边长）。\n        fill_image (int, tuple): 图像填充的像素值。默认0。\n        fill_mask (int, tuple): 掩码填充的像素值。默认0。\n        blur_kernel_size (int): 高斯模糊核大小。默认5。\n        blur_sigma (float): 高斯模糊标准差。默认1.0。\n\n    Returns:\n        Tuple[torch.Tensor, torch.Tensor]: 规范化后的图像和掩码，形状为 (C, target_length, target_length) 和 (1, target_length, target_length)。\n    """\n    # 1. 计算边界框\n    bbox = get_bounding_box(mask)\n    print(f"Bounding Box: {bbox}")\n\n    # 2. 裁剪图像和掩码\n    cropped_image, cropped_mask = crop_image_and_mask(image, mask, bbox)\n\n    # 3. 应用高斯模糊到裁剪后的图像\n    blurred_image = apply_gaussian_blur(cropped_image, kernel_size=blur_kernel_size, sigma=blur_sigma)\n\n    # 4. 扩展掩码维度以适应 F.resize\n    cropped_mask = cropped_mask.unsqueeze(0)  # 形状: (1, H, W)\n\n    # 5. 缩放图像和掩码\n    resized_image = resize_aspect_ratio(blurred_image, target_length, interpolation=F.InterpolationMode.BILINEAR)\n    resized_mask = resize_aspect_ratio(cropped_mask, target_length, interpolation=F.InterpolationMode.NEAREST)\n\n    # 6. 填充图像和掩码\n    padded_image = pad_to_max_length(resized_image, target_length, fill=fill_image)\n    padded_mask = pad_to_max_length(resized_mask, target_length, fill=fill_mask)\n\n    return padded_image, padded_mask\n\n\nclass BTSMaskDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "image"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        base_name = os.path.splitext(name)[0]\n        lr_path = os.path.join(self.input_folder, "image", base_name + ".jpg")\n        lr_mask_path = os.path.join(self.input_folder, "semantic_label", base_name + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        lr_mask_img = Image.open(lr_mask_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        lr_mask_img = F.to_tensor(lr_mask_img)\n        lr_mask_img = (lr_mask_img[0] > 0).float()\n        processed_image, processed_mask = process_image_and_mask(\n            image=input_img,\n            mask=lr_mask_img,\n            target_length=512,\n            fill_image=0,\n            fill_mask=0,\n            blur_kernel_size=11,\n            blur_sigma=4\n        )\n\n        return {\n            "name": name,\n            "input": processed_image,\n            "lr_mask": processed_mask\n        }\n\n\nclass FTSRRealDatasetDilation(data.Dataset):\n    def __init__(self, dataset_folder: str, dilation_radius: int = 3):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "sr_bicubic"))\n        self.dilation_radius = dilation_radius\n\n    def __len__(self):\n        """Returns the total number of images in the dataset."""\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        """Fetches the input image, ground truth, and dilated mask."""\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        hr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        mask_path = os.path.join(self.input_folder, "mask", name.split(\'.\')[0] + ".png")\n\n        # Load the images\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        # Apply dilation to the mask\n        mask_img = mask_img.filter(ImageFilter.MaxFilter(self.dilation_radius))\n\n        # Convert images to tensors\n        input_img = transforms.ToTensor()(input_img)\n        hr_img = transforms.ToTensor()(hr_img)\n        mask_img = transforms.ToTensor()(mask_img)\n\n        # Resize all images to 512x512\n        input_img = transforms.Resize((512, 512))(input_img)\n        hr_img = transforms.Resize((512, 512))(hr_img)\n        mask_img = transforms.Resize((512, 512))(mask_img)\n\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": hr_img,\n            "mask": mask_img\n        }\n\n\nclass FTSRRealInverseDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = sorted(os.listdir(os.path.join(self.input_folder, "sr_bicubic")))\n        self.mask_names = sorted(os.listdir(os.path.join(self.input_folder, "mask")), reverse=True)\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        name_mask = self.mask_names[idx]\n        lr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        hr_path = os.path.join(self.input_folder, "sr_bicubic", name)\n        mask_path = os.path.join(self.input_folder, "mask", name_mask.split(\'.\')[0] + ".png")\n\n        input_img = Image.open(lr_path).convert("RGB")\n        hr_img = Image.open(hr_path).convert("RGB")\n        mask_img = Image.open(mask_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n        mask_img = F.to_tensor(mask_img)\n\n        input_img = F.resize(input_img, size=[512, 512])\n        hr_img = F.resize(hr_img, size=[512, 512])\n        mask_img = F.resize(mask_img, size=[512, 512])\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": hr_img,\n            "mask": mask_img\n        }\n\n\nclass FTSRMaskDataset(data.Dataset):\n    def __init__(self, dataset_folder: str):\n        super().__init__()\n        self.input_folder = dataset_folder\n        self.img_names = os.listdir(os.path.join(self.input_folder, "lr_mask"))\n\n    def __len__(self):\n        return len(self.img_names)\n\n    def __getitem__(self, idx):\n        name = self.img_names[idx]\n        lr_path = os.path.join(self.input_folder, "lr_mask", name)\n        hr_path = os.path.join(self.input_folder, "mask", name)\n\n        input_img = Image.open(lr_path).convert("L")\n        hr_img = Image.open(hr_path).convert("L")\n\n        input_img = F.to_tensor(input_img)\n        hr_img = F.to_tensor(hr_img)\n\n        input_img = F.resize(input_img, size=[512, 512])\n        hr_img = F.resize(hr_img, size=[512, 512])\n        return {\n            "name": name,\n            "input": input_img,\n            "gt": hr_img,\n        }\n', 'tadisr/training/degradations.py': 'import cv2\nimport math\nimport numpy as np\nimport random\nimport torch\nfrom scipy import special\nfrom scipy.stats import multivariate_normal\nfrom torchvision.transforms.functional import rgb_to_grayscale\n\n\n# -------------------------------------------------------------------- #\n# --------------------------- blur kernels --------------------------- #\n# -------------------------------------------------------------------- #\n\n\n# --------------------------- util functions --------------------------- #\ndef sigma_matrix2(sig_x, sig_y, theta):\n    """Calculate the rotated sigma matrix (two dimensional matrix).\n\n    Args:\n        sig_x (float):\n        sig_y (float):\n        theta (float): Radian measurement.\n\n    Returns:\n        ndarray: Rotated sigma matrix.\n    """\n    d_matrix = np.array([[sig_x ** 2, 0], [0, sig_y ** 2]])\n    u_matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])\n    return np.dot(u_matrix, np.dot(d_matrix, u_matrix.T))\n\n\ndef mesh_grid(kernel_size):\n    """Generate the mesh grid, centering at zero.\n\n    Args:\n        kernel_size (int):\n\n    Returns:\n        xy (ndarray): with the shape (kernel_size, kernel_size, 2)\n        xx (ndarray): with the shape (kernel_size, kernel_size)\n        yy (ndarray): with the shape (kernel_size, kernel_size)\n    """\n    ax = np.arange(-kernel_size // 2 + 1., kernel_size // 2 + 1.)\n    xx, yy = np.meshgrid(ax, ax)\n    xy = np.hstack((xx.reshape((kernel_size * kernel_size, 1)), yy.reshape(kernel_size * kernel_size,\n                                                                           1))).reshape(kernel_size, kernel_size, 2)\n    return xy, xx, yy\n\n\ndef pdf2(sigma_matrix, grid):\n    """Calculate PDF of the bivariate Gaussian distribution.\n\n    Args:\n        sigma_matrix (ndarray): with the shape (2, 2)\n        grid (ndarray): generated by :func:`mesh_grid`,\n            with the shape (K, K, 2), K is the kernel size.\n\n    Returns:\n        kernel (ndarrray): un-normalized kernel.\n    """\n    inverse_sigma = np.linalg.inv(sigma_matrix)\n    kernel = np.exp(-0.5 * np.sum(np.dot(grid, inverse_sigma) * grid, 2))\n    return kernel\n\n\ndef cdf2(d_matrix, grid):\n    """Calculate the CDF of the standard bivariate Gaussian distribution.\n        Used in skewed Gaussian distribution.\n\n    Args:\n        d_matrix (ndarrasy): skew matrix.\n        grid (ndarray): generated by :func:`mesh_grid`,\n            with the shape (K, K, 2), K is the kernel size.\n\n    Returns:\n        cdf (ndarray): skewed cdf.\n    """\n    rv = multivariate_normal([0, 0], [[1, 0], [0, 1]])\n    grid = np.dot(grid, d_matrix)\n    cdf = rv.cdf(grid)\n    return cdf\n\n\ndef bivariate_Gaussian(kernel_size, sig_x, sig_y, theta, grid=None, isotropic=True):\n    """Generate a bivariate isotropic or anisotropic Gaussian kernel.\n\n    In the isotropic mode, only `sig_x` is used. `sig_y` and `theta` is ignored.\n\n    Args:\n        kernel_size (int):\n        sig_x (float):\n        sig_y (float):\n        theta (float): Radian measurement.\n        grid (ndarray, optional): generated by :func:`mesh_grid`,\n            with the shape (K, K, 2), K is the kernel size. Default: None\n        isotropic (bool):\n\n    Returns:\n        kernel (ndarray): normalized kernel.\n    """\n    if grid is None:\n        grid, _, _ = mesh_grid(kernel_size)\n    if isotropic:\n        sigma_matrix = np.array([[sig_x ** 2, 0], [0, sig_x ** 2]])\n    else:\n        sigma_matrix = sigma_matrix2(sig_x, sig_y, theta)\n    kernel = pdf2(sigma_matrix, grid)\n    kernel = kernel / np.sum(kernel)\n    return kernel\n\n\ndef bivariate_generalized_Gaussian(kernel_size, sig_x, sig_y, theta, beta, grid=None, isotropic=True):\n    """Generate a bivariate generalized Gaussian kernel.\n\n    ``Paper: Parameter Estimation For Multivariate Generalized Gaussian Distributions``\n\n    In the isotropic mode, only `sig_x` is used. `sig_y` and `theta` is ignored.\n\n    Args:\n        kernel_size (int):\n        sig_x (float):\n        sig_y (float):\n        theta (float): Radian measurement.\n        beta (float): shape parameter, beta = 1 is the normal distribution.\n        grid (ndarray, optional): generated by :func:`mesh_grid`,\n            with the shape (K, K, 2), K is the kernel size. Default: None\n\n    Returns:\n        kernel (ndarray): normalized kernel.\n    """\n    if grid is None:\n        grid, _, _ = mesh_grid(kernel_size)\n    if isotropic:\n        sigma_matrix = np.array([[sig_x ** 2, 0], [0, sig_x ** 2]])\n    else:\n        sigma_matrix = sigma_matrix2(sig_x, sig_y, theta)\n    inverse_sigma = np.linalg.inv(sigma_matrix)\n    kernel = np.exp(-0.5 * np.power(np.sum(np.dot(grid, inverse_sigma) * grid, 2), beta))\n    kernel = kernel / np.sum(kernel)\n    return kernel\n\n\ndef bivariate_plateau(kernel_size, sig_x, sig_y, theta, beta, grid=None, isotropic=True):\n    """Generate a plateau-like anisotropic kernel.\n\n    1 / (1+x^(beta))\n\n    Reference: https://stats.stackexchange.com/questions/203629/is-there-a-plateau-shaped-distribution\n\n    In the isotropic mode, only `sig_x` is used. `sig_y` and `theta` is ignored.\n\n    Args:\n        kernel_size (int):\n        sig_x (float):\n        sig_y (float):\n        theta (float): Radian measurement.\n        beta (float): shape parameter, beta = 1 is the normal distribution.\n        grid (ndarray, optional): generated by :func:`mesh_grid`,\n            with the shape (K, K, 2), K is the kernel size. Default: None\n\n    Returns:\n        kernel (ndarray): normalized kernel.\n    """\n    if grid is None:\n        grid, _, _ = mesh_grid(kernel_size)\n    if isotropic:\n        sigma_matrix = np.array([[sig_x ** 2, 0], [0, sig_x ** 2]])\n    else:\n        sigma_matrix = sigma_matrix2(sig_x, sig_y, theta)\n    inverse_sigma = np.linalg.inv(sigma_matrix)\n    kernel = np.reciprocal(np.power(np.sum(np.dot(grid, inverse_sigma) * grid, 2), beta) + 1)\n    kernel = kernel / np.sum(kernel)\n    return kernel\n\n\ndef random_bivariate_Gaussian(kernel_size,\n                              sigma_x_range,\n                              sigma_y_range,\n                              rotation_range,\n                              noise_range=None,\n                              isotropic=True):\n    """Randomly generate bivariate isotropic or anisotropic Gaussian kernels.\n\n    In the isotropic mode, only `sigma_x_range` is used. `sigma_y_range` and `rotation_range` is ignored.\n\n    Args:\n        kernel_size (int):\n        sigma_x_range (tuple): [0.6, 5]\n        sigma_y_range (tuple): [0.6, 5]\n        rotation range (tuple): [-math.pi, math.pi]\n        noise_range(tuple, optional): multiplicative kernel noise,\n            [0.75, 1.25]. Default: None\n\n    Returns:\n        kernel (ndarray):\n    """\n    assert kernel_size % 2 == 1, \'Kernel size must be an odd number.\'\n    assert sigma_x_range[0] < sigma_x_range[1], \'Wrong sigma_x_range.\'\n    sigma_x = np.random.uniform(sigma_x_range[0], sigma_x_range[1])\n    if isotropic is False:\n        assert sigma_y_range[0] < sigma_y_range[1], \'Wrong sigma_y_range.\'\n        assert rotation_range[0] < rotation_range[1], \'Wrong rotation_range.\'\n        sigma_y = np.random.uniform(sigma_y_range[0], sigma_y_range[1])\n        rotation = np.random.uniform(rotation_range[0], rotation_range[1])\n    else:\n        sigma_y = sigma_x\n        rotation = 0\n\n    kernel = bivariate_Gaussian(kernel_size, sigma_x, sigma_y, rotation, isotropic=isotropic)\n\n    # add multiplicative noise\n    if noise_range is not None:\n        assert noise_range[0] < noise_range[1], \'Wrong noise range.\'\n        noise = np.random.uniform(noise_range[0], noise_range[1], size=kernel.shape)\n        kernel = kernel * noise\n    kernel = kernel / np.sum(kernel)\n    return kernel\n\n\ndef random_bivariate_generalized_Gaussian(kernel_size,\n                                          sigma_x_range,\n                                          sigma_y_range,\n                                          rotation_range,\n                                          beta_range,\n                                          noise_range=None,\n                                          isotropic=True):\n    """Randomly generate bivariate generalized Gaussian kernels.\n\n    In the isotropic mode, only `sigma_x_range` is used. `sigma_y_range` and `rotation_range` is ignored.\n\n    Args:\n        kernel_size (int):\n        sigma_x_range (tuple): [0.6, 5]\n        sigma_y_range (tuple): [0.6, 5]\n        rotation range (tuple): [-math.pi, math.pi]\n        beta_range (tuple): [0.5, 8]\n        noise_range(tuple, optional): multiplicative kernel noise,\n            [0.75, 1.25]. Default: None\n\n    Returns:\n        kernel (ndarray):\n    """\n    assert kernel_size % 2 == 1, \'Kernel size must be an odd number.\'\n    assert sigma_x_range[0] < sigma_x_range[1], \'Wrong sigma_x_range.\'\n    sigma_x = np.random.uniform(sigma_x_range[0], sigma_x_range[1])\n    if isotropic is False:\n        assert sigma_y_range[0] < sigma_y_range[1], \'Wrong sigma_y_range.\'\n        assert rotation_range[0] < rotation_range[1], \'Wrong rotation_range.\'\n        sigma_y = np.random.uniform(sigma_y_range[0], sigma_y_range[1])\n        rotation = np.random.uniform(rotation_range[0], rotation_range[1])\n    else:\n        sigma_y = sigma_x\n        rotation = 0\n\n    # assume beta_range[0] < 1 < beta_range[1]\n    if np.random.uniform() < 0.5:\n        beta = np.random.uniform(beta_range[0], 1)\n    else:\n        beta = np.random.uniform(1, beta_range[1])\n\n    kernel = bivariate_generalized_Gaussian(kernel_size, sigma_x, sigma_y, rotation, beta, isotropic=isotropic)\n\n    # add multiplicative noise\n    if noise_range is not None:\n        assert noise_range[0] < noise_range[1], \'Wrong noise range.\'\n        noise = np.random.uniform(noise_range[0], noise_range[1], size=kernel.shape)\n        kernel = kernel * noise\n    kernel = kernel / np.sum(kernel)\n    return kernel\n\n\ndef random_bivariate_plateau(kernel_size,\n                             sigma_x_range,\n                             sigma_y_range,\n                             rotation_range,\n                             beta_range,\n                             noise_range=None,\n                             isotropic=True):\n    """Randomly generate bivariate plateau kernels.\n\n    In the isotropic mode, only `sigma_x_range` is used. `sigma_y_range` and `rotation_range` is ignored.\n\n    Args:\n        kernel_size (int):\n        sigma_x_range (tuple): [0.6, 5]\n        sigma_y_range (tuple): [0.6, 5]\n        rotation range (tuple): [-math.pi/2, math.pi/2]\n        beta_range (tuple): [1, 4]\n        noise_range(tuple, optional): multiplicative kernel noise,\n            [0.75, 1.25]. Default: None\n\n    Returns:\n        kernel (ndarray):\n    """\n    assert kernel_size % 2 == 1, \'Kernel size must be an odd number.\'\n    assert sigma_x_range[0] < sigma_x_range[1], \'Wrong sigma_x_range.\'\n    sigma_x = np.random.uniform(sigma_x_range[0], sigma_x_range[1])\n    if isotropic is False:\n        assert sigma_y_range[0] < sigma_y_range[1], \'Wrong sigma_y_range.\'\n        assert rotation_range[0] < rotation_range[1], \'Wrong rotation_range.\'\n        sigma_y = np.random.uniform(sigma_y_range[0], sigma_y_range[1])\n        rotation = np.random.uniform(rotation_range[0], rotation_range[1])\n    else:\n        sigma_y = sigma_x\n        rotation = 0\n\n    # TODO: this may be not proper\n    if np.random.uniform() < 0.5:\n        beta = np.random.uniform(beta_range[0], 1)\n    else:\n        beta = np.random.uniform(1, beta_range[1])\n\n    kernel = bivariate_plateau(kernel_size, sigma_x, sigma_y, rotation, beta, isotropic=isotropic)\n    # add multiplicative noise\n    if noise_range is not None:\n        assert noise_range[0] < noise_range[1], \'Wrong noise range.\'\n        noise = np.random.uniform(noise_range[0], noise_range[1], size=kernel.shape)\n        kernel = kernel * noise\n    kernel = kernel / np.sum(kernel)\n\n    return kernel\n\n\ndef random_mixed_kernels(kernel_list,\n                         kernel_prob,\n                         kernel_size=21,\n                         sigma_x_range=(0.6, 5),\n                         sigma_y_range=(0.6, 5),\n                         rotation_range=(-math.pi, math.pi),\n                         betag_range=(0.5, 8),\n                         betap_range=(0.5, 8),\n                         noise_range=None):\n    """Randomly generate mixed kernels.\n\n    Args:\n        kernel_list (list): a list name of kernel types,\n            support [\'iso\', \'aniso\', \'skew\', \'generalized\', \'plateau_iso\',\n            \'plateau_aniso\']\n        kernel_prob (list): corresponding kernel probability for each\n            kernel type\n        kernel_size (int):\n        sigma_x_range (tuple): [0.6, 5]\n        sigma_y_range (tuple): [0.6, 5]\n        rotation range (tuple): [-math.pi, math.pi]\n        beta_range (tuple): [0.5, 8]\n        noise_range(tuple, optional): multiplicative kernel noise,\n            [0.75, 1.25]. Default: None\n\n    Returns:\n        kernel (ndarray):\n    """\n    kernel_type = random.choices(kernel_list, kernel_prob)[0]\n    if kernel_type == \'iso\':\n        kernel = random_bivariate_Gaussian(\n            kernel_size, sigma_x_range, sigma_y_range, rotation_range, noise_range=noise_range, isotropic=True)\n    elif kernel_type == \'aniso\':\n        kernel = random_bivariate_Gaussian(\n            kernel_size, sigma_x_range, sigma_y_range, rotation_range, noise_range=noise_range, isotropic=False)\n    elif kernel_type == \'generalized_iso\':\n        kernel = random_bivariate_generalized_Gaussian(\n            kernel_size,\n            sigma_x_range,\n            sigma_y_range,\n            rotation_range,\n            betag_range,\n            noise_range=noise_range,\n            isotropic=True)\n    elif kernel_type == \'generalized_aniso\':\n        kernel = random_bivariate_generalized_Gaussian(\n            kernel_size,\n            sigma_x_range,\n            sigma_y_range,\n            rotation_range,\n            betag_range,\n            noise_range=noise_range,\n            isotropic=False)\n    elif kernel_type == \'plateau_iso\':\n        kernel = random_bivariate_plateau(\n            kernel_size, sigma_x_range, sigma_y_range, rotation_range, betap_range, noise_range=None, isotropic=True)\n    elif kernel_type == \'plateau_aniso\':\n        kernel = random_bivariate_plateau(\n            kernel_size, sigma_x_range, sigma_y_range, rotation_range, betap_range, noise_range=None, isotropic=False)\n    return kernel\n\n\nnp.seterr(divide=\'ignore\', invalid=\'ignore\')\n\n\ndef circular_lowpass_kernel(cutoff, kernel_size, pad_to=0):\n    """2D sinc filter\n\n    Reference: https://dsp.stackexchange.com/questions/58301/2-d-circularly-symmetric-low-pass-filter\n\n    Args:\n        cutoff (float): cutoff frequency in radians (pi is max)\n        kernel_size (int): horizontal and vertical size, must be odd.\n        pad_to (int): pad kernel size to desired size, must be odd or zero.\n    """\n    assert kernel_size % 2 == 1, \'Kernel size must be an odd number.\'\n    kernel = np.fromfunction(\n        lambda x, y: cutoff * special.j1(cutoff * np.sqrt(\n            (x - (kernel_size - 1) / 2) ** 2 + (y - (kernel_size - 1) / 2) ** 2)) / (2 * np.pi * np.sqrt(\n            (x - (kernel_size - 1) / 2) ** 2 + (y - (kernel_size - 1) / 2) ** 2)), [kernel_size, kernel_size])\n    kernel[(kernel_size - 1) // 2, (kernel_size - 1) // 2] = cutoff ** 2 / (4 * np.pi)\n    kernel = kernel / np.sum(kernel)\n    if pad_to > kernel_size:\n        pad_size = (pad_to - kernel_size) // 2\n        kernel = np.pad(kernel, ((pad_size, pad_size), (pad_size, pad_size)))\n    return kernel\n\n\n# ------------------------------------------------------------- #\n# --------------------------- noise --------------------------- #\n# ------------------------------------------------------------- #\n\n# ----------------------- Gaussian Noise ----------------------- #\n\n\ndef generate_gaussian_noise(img, sigma=10, gray_noise=False):\n    """Generate Gaussian noise.\n\n    Args:\n        img (Numpy array): Input image, shape (h, w, c), range [0, 1], float32.\n        sigma (float): Noise scale (measured in range 255). Default: 10.\n\n    Returns:\n        (Numpy array): Returned noisy image, shape (h, w, c), range[0, 1],\n            float32.\n    """\n    if gray_noise:\n        noise = np.float32(np.random.randn(*(img.shape[0:2]))) * sigma / 255.\n        noise = np.expand_dims(noise, axis=2).repeat(3, axis=2)\n    else:\n        noise = np.float32(np.random.randn(*(img.shape))) * sigma / 255.\n    return noise\n\n\ndef add_gaussian_noise(img, sigma=10, clip=True, rounds=False, gray_noise=False):\n    """Add Gaussian noise.\n\n    Args:\n        img (Numpy array): Input image, shape (h, w, c), range [0, 1], float32.\n        sigma (float): Noise scale (measured in range 255). Default: 10.\n\n    Returns:\n        (Numpy array): Returned noisy image, shape (h, w, c), range[0, 1],\n            float32.\n    """\n    noise = generate_gaussian_noise(img, sigma, gray_noise)\n    out = img + noise\n    if clip and rounds:\n        out = np.clip((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = np.clip(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\ndef generate_gaussian_noise_pt(img, sigma=10, gray_noise=0):\n    """Add Gaussian noise (PyTorch version).\n\n    Args:\n        img (Tensor): Shape (b, c, h, w), range[0, 1], float32.\n        scale (float | Tensor): Noise scale. Default: 1.0.\n\n    Returns:\n        (Tensor): Returned noisy image, shape (b, c, h, w), range[0, 1],\n            float32.\n    """\n    b, _, h, w = img.size()\n    if not isinstance(sigma, (float, int)):\n        sigma = sigma.view(img.size(0), 1, 1, 1)\n    if isinstance(gray_noise, (float, int)):\n        cal_gray_noise = gray_noise > 0\n    else:\n        gray_noise = gray_noise.view(b, 1, 1, 1)\n        cal_gray_noise = torch.sum(gray_noise) > 0\n\n    if cal_gray_noise:\n        noise_gray = torch.randn(*img.size()[2:4], dtype=img.dtype, device=img.device) * sigma / 255.\n        noise_gray = noise_gray.view(b, 1, h, w)\n\n    # always calculate color noise\n    noise = torch.randn(*img.size(), dtype=img.dtype, device=img.device) * sigma / 255.\n\n    if cal_gray_noise:\n        noise = noise * (1 - gray_noise) + noise_gray * gray_noise\n    return noise\n\n\ndef add_gaussian_noise_pt(img, sigma=10, gray_noise=0, clip=True, rounds=False):\n    """Add Gaussian noise (PyTorch version).\n\n    Args:\n        img (Tensor): Shape (b, c, h, w), range[0, 1], float32.\n        scale (float | Tensor): Noise scale. Default: 1.0.\n\n    Returns:\n        (Tensor): Returned noisy image, shape (b, c, h, w), range[0, 1],\n            float32.\n    """\n    noise = generate_gaussian_noise_pt(img, sigma, gray_noise)\n    out = img + noise\n    if clip and rounds:\n        out = torch.clamp((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = torch.clamp(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\n# ----------------------- Random Gaussian Noise ----------------------- #\ndef random_generate_gaussian_noise(img, sigma_range=(0, 10), gray_prob=0):\n    sigma = np.random.uniform(sigma_range[0], sigma_range[1])\n    if np.random.uniform() < gray_prob:\n        gray_noise = True\n    else:\n        gray_noise = False\n    return generate_gaussian_noise(img, sigma, gray_noise)\n\n\ndef random_add_gaussian_noise(img, sigma_range=(0, 1.0), gray_prob=0, clip=True, rounds=False):\n    noise = random_generate_gaussian_noise(img, sigma_range, gray_prob)\n    out = img + noise\n    if clip and rounds:\n        out = np.clip((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = np.clip(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\ndef random_generate_gaussian_noise_pt(img, sigma_range=(0, 10), gray_prob=0):\n    sigma = torch.rand(\n        img.size(0), dtype=img.dtype, device=img.device) * (sigma_range[1] - sigma_range[0]) + sigma_range[0]\n    gray_noise = torch.rand(img.size(0), dtype=img.dtype, device=img.device)\n    gray_noise = (gray_noise < gray_prob).float()\n    return generate_gaussian_noise_pt(img, sigma, gray_noise)\n\n\ndef random_add_gaussian_noise_pt(img, sigma_range=(0, 1.0), gray_prob=0, clip=True, rounds=False):\n    noise = random_generate_gaussian_noise_pt(img, sigma_range, gray_prob)\n    out = img + noise\n    if clip and rounds:\n        out = torch.clamp((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = torch.clamp(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\n# ----------------------- Poisson (Shot) Noise ----------------------- #\n\n\ndef generate_poisson_noise(img, scale=1.0, gray_noise=False):\n    """Generate poisson noise.\n\n    Reference: https://github.com/scikit-image/scikit-image/blob/main/skimage/util/noise.py#L37-L219\n\n    Args:\n        img (Numpy array): Input image, shape (h, w, c), range [0, 1], float32.\n        scale (float): Noise scale. Default: 1.0.\n        gray_noise (bool): Whether generate gray noise. Default: False.\n\n    Returns:\n        (Numpy array): Returned noisy image, shape (h, w, c), range[0, 1],\n            float32.\n    """\n    if gray_noise:\n        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    # round and clip image for counting vals correctly\n    img = np.clip((img * 255.0).round(), 0, 255) / 255.\n    vals = len(np.unique(img))\n    vals = 2 ** np.ceil(np.log2(vals))\n    out = np.float32(np.random.poisson(img * vals) / float(vals))\n    noise = out - img\n    if gray_noise:\n        noise = np.repeat(noise[:, :, np.newaxis], 3, axis=2)\n    return noise * scale\n\n\ndef add_poisson_noise(img, scale=1.0, clip=True, rounds=False, gray_noise=False):\n    """Add poisson noise.\n\n    Args:\n        img (Numpy array): Input image, shape (h, w, c), range [0, 1], float32.\n        scale (float): Noise scale. Default: 1.0.\n        gray_noise (bool): Whether generate gray noise. Default: False.\n\n    Returns:\n        (Numpy array): Returned noisy image, shape (h, w, c), range[0, 1],\n            float32.\n    """\n    noise = generate_poisson_noise(img, scale, gray_noise)\n    out = img + noise\n    if clip and rounds:\n        out = np.clip((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = np.clip(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\ndef generate_poisson_noise_pt(img, scale=1.0, gray_noise=0):\n    """Generate a batch of poisson noise (PyTorch version)\n\n    Args:\n        img (Tensor): Input image, shape (b, c, h, w), range [0, 1], float32.\n        scale (float | Tensor): Noise scale. Number or Tensor with shape (b).\n            Default: 1.0.\n        gray_noise (float | Tensor): 0-1 number or Tensor with shape (b).\n            0 for False, 1 for True. Default: 0.\n\n    Returns:\n        (Tensor): Returned noisy image, shape (b, c, h, w), range[0, 1],\n            float32.\n    """\n    b, _, h, w = img.size()\n    if isinstance(gray_noise, (float, int)):\n        cal_gray_noise = gray_noise > 0\n    else:\n        gray_noise = gray_noise.view(b, 1, 1, 1)\n        cal_gray_noise = torch.sum(gray_noise) > 0\n    if cal_gray_noise:\n        img_gray = rgb_to_grayscale(img, num_output_channels=1)\n        # round and clip image for counting vals correctly\n        img_gray = torch.clamp((img_gray * 255.0).round(), 0, 255) / 255.\n        # use for-loop to get the unique values for each sample\n        vals_list = [len(torch.unique(img_gray[i, :, :, :])) for i in range(b)]\n        vals_list = [2 ** np.ceil(np.log2(vals)) for vals in vals_list]\n        vals = img_gray.new_tensor(vals_list).view(b, 1, 1, 1)\n        out = torch.poisson(img_gray * vals) / vals\n        noise_gray = out - img_gray\n        noise_gray = noise_gray.expand(b, 3, h, w)\n\n    # always calculate color noise\n    # round and clip image for counting vals correctly\n    img = torch.clamp((img * 255.0).round(), 0, 255) / 255.\n    # use for-loop to get the unique values for each sample\n    vals_list = [len(torch.unique(img[i, :, :, :])) for i in range(b)]\n    vals_list = [2 ** np.ceil(np.log2(vals)) for vals in vals_list]\n    vals = img.new_tensor(vals_list).view(b, 1, 1, 1)\n    out = torch.poisson(img * vals) / vals\n    noise = out - img\n    if cal_gray_noise:\n        noise = noise * (1 - gray_noise) + noise_gray * gray_noise\n    if not isinstance(scale, (float, int)):\n        scale = scale.view(b, 1, 1, 1)\n    return noise * scale\n\n\ndef add_poisson_noise_pt(img, scale=1.0, clip=True, rounds=False, gray_noise=0):\n    """Add poisson noise to a batch of images (PyTorch version).\n\n    Args:\n        img (Tensor): Input image, shape (b, c, h, w), range [0, 1], float32.\n        scale (float | Tensor): Noise scale. Number or Tensor with shape (b).\n            Default: 1.0.\n        gray_noise (float | Tensor): 0-1 number or Tensor with shape (b).\n            0 for False, 1 for True. Default: 0.\n\n    Returns:\n        (Tensor): Returned noisy image, shape (b, c, h, w), range[0, 1],\n            float32.\n    """\n    noise = generate_poisson_noise_pt(img, scale, gray_noise)\n    out = img + noise\n    if clip and rounds:\n        out = torch.clamp((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = torch.clamp(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\n# ----------------------- Random Poisson (Shot) Noise ----------------------- #\n\n\ndef random_generate_poisson_noise(img, scale_range=(0, 1.0), gray_prob=0):\n    scale = np.random.uniform(scale_range[0], scale_range[1])\n    if np.random.uniform() < gray_prob:\n        gray_noise = True\n    else:\n        gray_noise = False\n    return generate_poisson_noise(img, scale, gray_noise)\n\n\ndef random_add_poisson_noise(img, scale_range=(0, 1.0), gray_prob=0, clip=True, rounds=False):\n    noise = random_generate_poisson_noise(img, scale_range, gray_prob)\n    out = img + noise\n    if clip and rounds:\n        out = np.clip((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = np.clip(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\ndef random_generate_poisson_noise_pt(img, scale_range=(0, 1.0), gray_prob=0):\n    scale = torch.rand(\n        img.size(0), dtype=img.dtype, device=img.device) * (scale_range[1] - scale_range[0]) + scale_range[0]\n    gray_noise = torch.rand(img.size(0), dtype=img.dtype, device=img.device)\n    gray_noise = (gray_noise < gray_prob).float()\n    return generate_poisson_noise_pt(img, scale, gray_noise)\n\n\ndef random_add_poisson_noise_pt(img, scale_range=(0, 1.0), gray_prob=0, clip=True, rounds=False):\n    noise = random_generate_poisson_noise_pt(img, scale_range, gray_prob)\n    out = img + noise\n    if clip and rounds:\n        out = torch.clamp((out * 255.0).round(), 0, 255) / 255.\n    elif clip:\n        out = torch.clamp(out, 0, 1)\n    elif rounds:\n        out = (out * 255.0).round() / 255.\n    return out\n\n\n# ------------------------------------------------------------------------ #\n# --------------------------- JPEG compression --------------------------- #\n# ------------------------------------------------------------------------ #\n\n\ndef add_jpg_compression(img, quality=90):\n    """Add JPG compression artifacts.\n\n    Args:\n        img (Numpy array): Input image, shape (h, w, c), range [0, 1], float32.\n        quality (float): JPG compression quality. 0 for lowest quality, 100 for\n            best quality. Default: 90.\n\n    Returns:\n        (Numpy array): Returned image after JPG, shape (h, w, c), range[0, 1],\n            float32.\n    """\n    img = np.clip(img, 0, 1)\n    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]\n    _, encimg = cv2.imencode(\'.jpg\', img * 255., encode_param)\n    img = np.float32(cv2.imdecode(encimg, 1)) / 255.\n    return img\n\n\ndef random_add_jpg_compression(img, quality_range=(90, 100)):\n    """Randomly add JPG compression artifacts.\n\n    Args:\n        img (Numpy array): Input image, shape (h, w, c), range [0, 1], float32.\n        quality_range (tuple[float] | list[float]): JPG compression quality\n            range. 0 for lowest quality, 100 for best quality.\n            Default: (90, 100).\n\n    Returns:\n        (Numpy array): Returned image after JPG, shape (h, w, c), range[0, 1],\n            float32.\n    """\n    quality = np.random.uniform(quality_range[0], quality_range[1])\n    return add_jpg_compression(img, quality)\n', 'tadisr/training/losses.py': 'from csv import Error\nimport torch\nimport torch.nn.functional as F\nimport torch.nn as nn\n\nimport cv2\nimport numpy as np\nimport math\nfrom skimage.transform._geometric import _umeyama as get_sym_mat\n\n\ndef min_bounding_rect(img):\n    ret, thresh = cv2.threshold(img, 127, 255, 0)\n    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    if len(contours) == 0:\n        print(\'Bad contours, using fake bbox...\')\n        return np.array([[0, 0], [100, 0], [100, 100], [0, 100]])\n    max_contour = max(contours, key=cv2.contourArea)\n    rect = cv2.minAreaRect(max_contour)\n    box = cv2.boxPoints(rect)\n    box = np.intp(box)\n    # sort\n    x_sorted = sorted(box, key=lambda x: x[0])\n    left = x_sorted[:2]\n    right = x_sorted[2:]\n    left = sorted(left, key=lambda x: x[1])\n    (tl, bl) = left\n    right = sorted(right, key=lambda x: x[1])\n    (tr, br) = right\n    if tl[1] > bl[1]:\n        (tl, bl) = (bl, tl)\n    if tr[1] > br[1]:\n        (tr, br) = (br, tr)\n    return np.array([tl, tr, br, bl])\n\n\ndef crop_image(src_img, mask):\n    \'\'\'\n    mask: numpy.ndarray, mask of textual, HWC\n    src_img: torch.Tensor, source image, CHW\n    \'\'\'\n    box = min_bounding_rect(mask)\n    result = adjust_image(box, src_img)\n    if len(result.shape) == 2:\n        result = torch.stack([result] * 3, axis=-1)\n    return result\n\n\ndef adjust_image(box, img):\n    pts1 = np.float32([box[0], box[1], box[2], box[3]])\n    width = max(np.linalg.norm(pts1[0] - pts1[1]), np.linalg.norm(pts1[2] - pts1[3]))\n    height = max(np.linalg.norm(pts1[0] - pts1[3]), np.linalg.norm(pts1[1] - pts1[2]))\n    pts2 = np.float32([[0, 0], [width, 0], [width, height], [0, height]])\n    # get transform matrix\n    M = get_sym_mat(pts1, pts2, estimate_scale=True)\n    C, H, W = img.shape\n    T = np.array([[2 / W, 0, -1], [0, 2 / H, -1], [0, 0, 1]])\n    theta = np.linalg.inv(T @ M @ np.linalg.inv(T))\n    theta = torch.from_numpy(theta[:2, :]).unsqueeze(0).type(torch.float32).to(img.device)\n    grid = F.affine_grid(theta, torch.Size([1, C, H, W]), align_corners=True)\n    result = F.grid_sample(img.unsqueeze(0), grid, align_corners=True)\n    result = torch.clamp(result.squeeze(0), 0, 255)\n    # crop\n    result = result[:, :int(height), :int(width)]\n    return result\n\n\nclass GaussianSmoothing(nn.Module):\n    """\n    Arguments:\n    Apply gaussian smoothing on a 1d, 2d or 3d tensor. Filtering is performed seperately for each channel in the input\n    using a depthwise convolution.\n        channels (int, sequence): Number of channels of the input tensors. Output will\n            have this number of channels as well.\n        kernel_size (int, sequence): Size of the gaussian kernel. sigma (float, sequence): Standard deviation of the\n        gaussian kernel. dim (int, optional): The number of dimensions of the data.\n            Default value is 2 (spatial).\n    """\n\n    # channels=1, kernel_size=kernel_size, sigma=sigma, dim=2\n    def __init__(\n            self,\n            channels: int = 1,\n            kernel_size: int = 3,\n            sigma: float = 0.5,\n            dim: int = 2,\n    ):\n        super().__init__()\n\n        if isinstance(kernel_size, int):\n            kernel_size = [kernel_size] * dim\n        if isinstance(sigma, float):\n            sigma = [sigma] * dim\n\n        # The gaussian kernel is the product of the\n        # gaussian function of each dimension.\n        kernel = 1\n        meshgrids = torch.meshgrid([torch.arange(size, dtype=torch.float32) for size in kernel_size])\n        for size, std, mgrid in zip(kernel_size, sigma, meshgrids):\n            mean = (size - 1) / 2\n            kernel *= 1 / (std * math.sqrt(2 * math.pi)) * torch.exp(-(((mgrid - mean) / (2 * std)) ** 2))\n\n        # Make sure sum of values in gaussian kernel equals 1.\n        kernel = kernel / torch.sum(kernel)\n\n        # Reshape to depthwise convolutional weight\n        kernel = kernel.view(1, 1, *kernel.size())\n        kernel = kernel.repeat(channels, *[1] * (kernel.dim() - 1))\n\n        self.register_buffer("weight", kernel)\n        self.groups = channels\n\n        if dim == 1:\n            self.conv = F.conv1d\n        elif dim == 2:\n            self.conv = F.conv2d\n        elif dim == 3:\n            self.conv = F.conv3d\n        else:\n            raise RuntimeError("Only 1, 2 and 3 dimensions are supported. Received {}.".format(dim))\n\n    def forward(self, input):\n        """\n        Arguments:\n        Apply gaussian filter to input.\n            input (torch.Tensor): Input to apply gaussian filter on.\n        Returns:\n            filtered (torch.Tensor): Filtered output.\n        """\n        return self.conv(input, weight=self.weight.to(input.dtype), groups=self.groups)\n\n\nclass DifferentiableEdgeDetector(nn.Module):\n    def __init__(self, sigma=3.0, sobel_ksize=3):\n        super().__init__()\n\n        self.smth_3 = GaussianSmoothing(sigma=sigma).eval()\n        self.smth_3.requires_grad_(False)\n\n        # Sobel算子\n        self.sobel_x = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False)\n        self.sobel_y = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False)\n        self._init_sobel(sobel_ksize)\n\n    def _init_sobel(self, ksize):\n        # X方向Sobel核\n        sobel_x = torch.tensor([[1, 0, -1],\n                                [2, 0, -2],\n                                [1, 0, -1]], dtype=torch.float32)\n\n        sobel_y = torch.tensor([[1, 2, 1],\n                                [0, 0, 0],\n                                [-1, -2, -1]], dtype=torch.float32)\n        sobel_x = sobel_x.view(1, 1, 3, 3)\n        sobel_y = sobel_y.view(1, 1, 3, 3)\n\n        self.sobel_x.weight = nn.Parameter(sobel_x)\n        self.sobel_y.weight = nn.Parameter(sobel_y)\n        self.sobel_x.weight.requires_grad_(False)\n        self.sobel_y.weight.requires_grad_(False)\n\n    def forward(self, x):\n        weights = torch.tensor([0.299, 0.587, 0.114], device=x.device).view(1, 3, 1, 1)\n        if x.dim() == 3:\n            x = x.unsqueeze(0)\n        x = F.conv2d(x, weights)\n        x = F.pad(x, (1, 1, 1, 1), mode=\'reflect\')\n        x_blur = self.smth_3(x)  # 高斯模糊降噪\n\n        grad_x = self.sobel_x(x_blur).squeeze()[1:-1, 1:-1]\n        grad_y = self.sobel_y(x_blur).squeeze()[1:-1, 1:-1]\n        # 添加小的epsilon值以避免数值不稳定\n        epsilon = 1e-8\n        edge_magnitude = torch.sqrt(grad_x ** 2 + grad_y ** 2 + epsilon)\n\n        # 添加梯度裁剪以防止梯度爆炸\n        edge_magnitude = torch.clamp(edge_magnitude, 0, 255)\n\n        return edge_magnitude  # [H,W]\n\n\ndef get_edge_loss_bbox(edge_detector, X, GT, Bbox_data, loss_type="l2"):\n    all_edge_loss = []\n    bsz = X.shape[0]\n    for batch in range(bsz):\n        bbs = Bbox_data[batch]\n        if len(bbs) == 0:\n            all_edge_loss += [torch.tensor(0.0).to(X.device)]\n            continue\n        # Create mask for each bounding box and crop the text regions\n        bs_edge_loss = []\n        for bb in bbs:\n            # Crop the text region\n            x_text = adjust_image(bb, X[batch])\n            gt_text = adjust_image(bb, GT[batch])\n            x_edge = edge_detector(x_text)\n            gt_edge = edge_detector(gt_text)\n            # 检查边缘检测结果是否有效\n            if torch.isnan(x_edge).any() or torch.isnan(gt_edge).any():\n                raise Error("NaN detected in edge detection result")\n            if loss_type == \'l1\':\n                loss_edge = (x_edge - gt_edge).abs()\n            elif loss_type == \'l2\':\n                loss_edge = torch.nn.functional.mse_loss(gt_edge, x_edge, reduction="mean")\n            else:\n                raise NotImplementedError("unknown loss type \'{loss_type}\'")\n            if torch.isnan(loss_edge) or loss_edge.item() > 1e6:  # 添加上限检查\n                print(loss_edge)\n                raise Error("NaN detected in loss computation")\n            bs_edge_loss += [loss_edge]\n        all_edge_loss += [torch.stack(bs_edge_loss).mean()]\n    return torch.stack(all_edge_loss).mean()\n\n\ndef calc_ocr_loss(text_recognizer, X, GT, Bbox_data, loss_type="l2"):\n    all_ocr_loss = torch.tensor(0.0).to(X.device)\n    bsz = X.shape[0]\n\n    cropped_texts = []\n    cropped_gt = []\n    for batch in range(bsz):\n        x = X[batch]\n        gt = GT[batch]\n        bbs = Bbox_data[batch]\n        if len(bbs) == 0:\n            continue\n        # Create mask for each bounding box and crop the text regions\n        for bb in bbs:\n            # Convert bounding box to appropriate format if needed\n            bb = np.array(bb, dtype=np.int32)\n\n            # Create mask with the polygon\n            pos = np.zeros((gt.shape[1], gt.shape[2], 1), dtype=np.uint8)  # h,w,c\n            # print(pos.shape)\n            cv2.fillPoly(pos, [bb], (255, 255, 255))\n\n            # Crop the text region\n            x_text = crop_image(x, pos)\n            gt_text = crop_image(gt, pos)\n            cropped_texts.append(x_text)\n            cropped_gt.append(gt_text)\n        x_list = cropped_texts + cropped_gt\n\n        # Get predictions from the recognizer\n        preds, preds_neck = text_recognizer.pred_imglist(x_list, show_debug=False)\n\n        n_pairs = len(preds) // 2\n        # Calculate OCR loss\n        if n_pairs > 0:\n            # OCR loss calculation\n\n            preds_neck_x0 = preds_neck[:n_pairs]\n            preds_neck_gt = preds_neck[n_pairs:]\n            if loss_type == \'l1\':\n                loss_ocr = (preds_neck_gt - preds_neck_x0).abs()\n            elif loss_type == \'l2\':\n                loss_ocr = torch.nn.functional.mse_loss(preds_neck_gt, preds_neck_x0, reduction=\'none\')\n            else:\n                raise NotImplementedError("unknown loss type \'{loss_type}\'")\n            all_ocr_loss += loss_ocr.mean()\n\n    return all_ocr_loss\n', 'tadisr/training/metrics.py': 'import math\nfrom math import exp\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.autograd import Variable\n\n\ndef gaussian(window_size, sigma):\n    gauss = torch.Tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])\n    return gauss / gauss.sum()\n\n\ndef create_window(window_size, channel):\n    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)\n    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)\n    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())\n    return window\n\n\ndef _ssim(img1, img2, window, window_size, channel, size_average=True):\n    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)\n    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)\n    mu1_sq = mu1.pow(2)\n    mu2_sq = mu2.pow(2)\n    mu1_mu2 = mu1 * mu2\n    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq\n    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq\n    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2\n    C1 = 0.01 ** 2\n    C2 = 0.03 ** 2\n    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))\n\n    if size_average:\n        return ssim_map.mean()\n    else:\n        return ssim_map.mean(1).mean(1).mean(1)\n\n\ndef SSIM(img1, img2, window_size=11, size_average=True):\n    img1 = torch.clamp(img1, min=0, max=1)\n    img2 = torch.clamp(img2, min=0, max=1)\n    (_, channel, _, _) = img1.size()\n    window = create_window(window_size, channel)\n    if img1.is_cuda:\n        window = window.cuda(img1.get_device())\n    window = window.type_as(img1)\n    return _ssim(img1, img2, window, window_size, channel, size_average)\n\n\ndef PSNR(pred, gt):\n    pred = pred.clamp(0, 1).detach().cpu().numpy()\n    gt = gt.clamp(0, 1).detach().cpu().numpy()\n    imdff = pred - gt\n    rmse = math.sqrt(np.mean(imdff ** 2))\n    if rmse == 0:\n        return 100\n    return 20 * math.log10(1.0 / rmse)\n\n\nif __name__ == "__main__":\n    pass\n', 'tadisr/training/visualization.py': "import os\nimport time\n\nimport torch.nn as nn\n\nsaved_grad = None\nsaved_name = None\n\nbase_url = './results'\nos.makedirs(base_url, exist_ok=True)\n\n\ndef normalize_tensor_mm(tensor):\n    return (tensor - tensor.min()) / (tensor.max() - tensor.min())\n\n\ndef normalize_tensor_sigmoid(tensor):\n    return nn.functional.sigmoid(tensor)\n\n\ndef save_image(tensor, name=None, save_path=None, exit_flag=False, timestamp=False, norm=False, nrows=3):\n    import torchvision.utils as vutils\n    os.makedirs(base_url, exist_ok=True)\n    if norm:\n        tensor = normalize_tensor_mm(tensor)\n    grid = vutils.make_grid(tensor.detach().cpu(), nrow=nrows)\n\n    if save_path:\n        os.makedirs(os.path.dirname(save_path), exist_ok=True)\n        vutils.save_image(grid, save_path)\n    else:\n        vutils.save_image(grid, f'{base_url}/{name}.png')\n    if exit_flag:\n        exit(0)\n\n\ndef save_feature(tensor, name, exit_flag=False, timestamp=False):\n    import torchvision.utils as vutils\n    tensors = [tensor]\n    titles = ['original', 'min-max', 'sigmoid']\n    os.makedirs(base_url, exist_ok=True)\n    if timestamp:\n        name += '_' + str(time.time()).replace('.', '')\n\n    for index, tensor in enumerate(tensors):\n        _data = tensor.detach().cpu().squeeze(0).unsqueeze(1)\n        num_per_row = 8\n        grid = vutils.make_grid(_data, nrow=num_per_row)\n        vutils.save_image(grid, f'{base_url}/{name}_{titles[index]}.png')\n    if exit_flag:\n        exit(0)\n"}, 'DiffTSR': {'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'README.md': '<div align=center class="logo">\n      <img src="Repo_image/DiffTSR_icon.png" style="width:640px">\n   </a>\n</div>\n\n#\nDiffusion-based Blind Text Image Super-Resolution (CVPR2024)\n<a href=\'https://openaccess.thecvf.com/content/CVPR2024/papers/Zhang_Diffusion-based_Blind_Text_Image_Super-Resolution_CVPR_2024_paper.pdf\'><img src=\'https://img.shields.io/badge/CVPR-2024-blue.svg\'></a> &nbsp;&nbsp;\n<a href=\'https://openaccess.thecvf.com/content/CVPR2024/supplemental/Zhang_Diffusion-based_Blind_Text_CVPR_2024_supplemental.pdf\'><img src=\'https://img.shields.io/badge/Supplementary-Material-9cf\'></a> &nbsp;&nbsp;\n\n[Yuzhe Zhang](https://yuzhezhang-1999.github.io/)<sup>1</sup> | [Jiawei Zhang](https://scholar.google.com/citations?user=0GTpIAIAAAAJ)<sup>2</sup> | Hao Li<sup>2</sup> | [Zhouxia Wang](https://scholar.google.com/citations?user=JWds_bQAAAAJ)<sup>3</sup> | Luwei Hou<sup>2</sup> | [Dongqing Zou](https://scholar.google.com/citations?user=K1-PFhYAAAAJ)<sup>2</sup> | [Liheng Bian](https://scholar.google.com/citations?user=66IFMDEAAAAJ)<sup>1</sup>\n\n<sup>1</sup>Beijing Institute of Technology, <sup>2</sup>SenseTime Research, <sup>3</sup>The University of Hong Kong\n\n\n## 📢 News\n- 🚀Training Code has been released, enjoy.\n- 🌟Added some discussions about this work [Q&A](#-qa) and limitations of input image and dataset production [Data_Produce](#-input-data-and-dataset-production-guideline).\n- **2024.12** 🚀Some bugs and stability issues have been fixed. Please use the latest model for inference.\n- **2024.05** 🚀Inference code has been released, enjoy.\n- **2024.04** 🚀Official repository of DiffTSR.\n- **2024.03** 🌟The implementation code will be released shortly.\n- **2024.03** ❤️Accepted by CVPR2024.\n\n\n## 💬 Q&A\nPlease Read Before Trying.\n\n<details>\n<summary> 🇨🇳 中文 Q&A：对于大家关心的一些细节问题，这里进行了归纳供大家参考 (点击展开) </summary>\n\n0. **DiffTSR的对真实世界图片的泛化性，是否泛化到Real-World Scenarios？**\n\n   **A:** DiffTSR在训练过程中考虑到了真实世界的各种退化，继承了BSRGAN和Real-ESRGAN中对于复杂退化流程的构建。且“Blind Text Image Super-Resolution”的Blind-盲图像恢复就是指针对真实世界未知退化的图像恢复。\n\n\n1. **IDM 中 Unet 用的是 Stable-Diffusion 的权重吗？**\n\n   **A:** 不是。IDM 的 Unet 是从头训练的，没有加载任何预训练权重，IDM 的结构也和任何一个 Diffusion 模型的 Unet 不一致。但是 VAE 是加载了 ldm 的 f4 VAE 在 Open-Image 上预训练的权重，然后在本项目的 CTR-TSR-Train 数据集上进行了微调，微调了 100,000 iter，batch_size=16。此外，包括 TDM 和 MoM 在内的模型均未使用预训练模型，均为从头训练获得。详细训练设置请看 [附加材料](https://openaccess.thecvf.com/content/CVPR2024/supplemental/Zhang_Diffusion-based_Blind_Text_CVPR_2024_supplemental.pdf) Section 1.4。\n\n2. **DiffTSR 模型的输入尺寸和要求，需要将输入 resize 吗？**\n\n   **A:** 模型的 LR 输入需要统一 resize 到 `width=512` / `height=128`。此外，因为本项目仅考虑单行文本输入，输入图片需要只包含一行文本。IDM 和 TDM 仅适配单行文本，多行文本输入会导致效果扭曲和错误的结果。\n\n3. **图片的推理速度非常慢，有什么解决办法吗？**\n\n   **A:** 由于本项目基于 Diffusion 技术，每处理一张图像都需要进行 `T` 次迭代（默认 `T=200`）。若想提升推理速度，可以考虑：\n   \n   - **减小 `T`**，由于采样器为 DDIM，在 `T=20` 时仍有较好表现。\n   - **对 DiffTSR 模型进行量化**，可参考 Diffusion 模型量化的相关 Repo。\n   - **使用本项目的 Baseline model**，虽然 Baseline 会在一定程度上降低性能，但可提升约 2 倍的推理速度，并且在大多数场景下不会明显退化。\n   - **对模型进行蒸馏**，或基于论文训练一个更小的 IDM 模型，文本场景可能不需要像通用场景图像生成那样重的模型。\n\n4. **在训练 IDM 时，损失是如何设置的？text_recognition loss 是如何实现的？**\n\n   **A:** 训练 IDM 时使用了两个损失函数：\n   \n   - **L2 loss**：用于预测噪声。\n   - **OCR loss**：用于从预测出的干净 `x^0` 上检测文字。\n   \n   具体来说：\n   - L2 loss 是传统 diffusion 模型中用于最小化 `(Unet 输出 - noise map)`，使 Unet 具备噪声估计能力。\n   - OCR loss 通过 `z_t` 计算 `z^(t-1)`，再得到 `z^0`，然后解码 `z^0` 得到 `x^0`。将 `x^0` 输入冻结权重的 TransOCR 模型，获得 `x^0` 上的文字 embedding，计算预测的 `pred-text-embedding` 和 `gt-text-embedding` 之间的 cross-entropy loss，OCR loss 额外添加了 `weight=0.02` 约束。\n   \n   详细内容参见 [Issue](https://github.com/YuzheZhang-1999/DiffTSR/issues/13)。\n\n5. **训练的损失函数是什么？**\n\n   **A:** DiffTSR 模型训练经历了三个阶段，每个阶段使用了不同损失函数的组合：\n   \n   - **训练 IDM**：IDM 从头训练 Unet，损失函数为 `L_IDM`，包含 `L2 loss` 和 `OCR loss`。\n   - **训练 TDM**：TDM 从头训练 Transformer，损失函数为 `L_TDM`，参考 [Multinomial Diffusion](https://arxiv.org/pdf/2102.05379) Section 4。\n   - **训练 DiffTSR 整体**：冻结 IDM 和 TDM，仅训练 MoM，损失函数为 `L_MoM = L_IDM + L_TDM * weight`。\n   \n   其中：\n   \n      ![L_IDM](https://latex.codecogs.com/svg.image?L_{IDM}=L_2+%5Clambda%20L_{OCR},%20%5Clambda=0.02)\n\n      ![L_TDM](https://latex.codecogs.com/svg.image?L_{TDM}=KL%28%5Cmathcal%7BC%7D%28%5Cpi_{post}%28%5Cmathbf%7Bc_t%7D,%20%5Cmathbf%7Bc_0%7D%29%29%20%7C%7C%20%5Cmathcal%7BC%7D%28%5Cpi_{post}%28%5Cmathbf%7Bc_t%7D,%20%5Cmathbf%7Bc_%7Bpred,t%7D%7D%29%29%29)\n\n      ![L_MoM](https://latex.codecogs.com/svg.image?L_{MoM}=L_{IDM}+%5Clambda%20L_{TDM},%20%5Clambda=1)\n\n\n   具体符号定义和理论推导详见 [附加材料](https://openaccess.thecvf.com/content/CVPR2024/supplemental/Zhang_Diffusion-based_Blind_Text_CVPR_2024_supplemental.pdf) Section 1 及 Algorithm 1 DiffTSR Training。\n\n   **未完待续...**\n\n</details>\n\n<details>\n<summary>🇬🇧 English Q&A: For some details you may want to know, here is a summary for your reference (click to expand)</summary>\n\n0. **Generalization of DiffTSR to Real-World Scenarios**\n\n   **A:** DiffTSR takes various real-world degradations into account during training, inheriting the complex degradation modeling from BSRGAN and Real-ESRGAN. Moreover, the "Blind" in "Blind Text Image Super-Resolution" specifically refers to the restoration of images with unknown degradations, which is targeted at real-world scenarios.\n\n1. **Does the Unet in IDM use Stable-Diffusion weights?**\n\n   **A:** No. The Unet in IDM is trained from scratch and does not load any pre-trained weights. Additionally, the structure of IDM is different from any Diffusion model\'s Unet. However, the VAE loads the pre-trained weights from `ldm f4 VAE`, which was pre-trained on the Open-Image dataset and then fine-tuned on the CTR-TSR-Train dataset in this project. The fine-tuning was conducted for 100,000 iterations with a batch size of 16. Moreover, models including TDM and MoM were also trained from scratch without using any pre-trained models. For detailed training settings, please refer to [Supplementary Material](https://openaccess.thecvf.com/content/CVPR2024/supplemental/Zhang_Diffusion-based_Blind_Text_CVPR_2024_supplemental.pdf) Section 1.4.\n\n2. **What are the input size and requirements for the DiffTSR model? Does the input need to be resized?**\n\n   **A:** The LR input of the model needs to be uniformly resized to `width=512` and `height=128`. Additionally, since this project only considers single-line text input, the input image must contain only one line of text. Both IDM and TDM are designed specifically for single-line text, and multi-line text input will result in distortion and incorrect results.\n\n3. **The inference speed of the image is very slow. What are the possible solutions?**\n\n   **A:** Since this project is based on Diffusion technology, processing a single image requires `T` iterations (default `T=200`). To improve inference speed, you may consider:\n   \n   - **Reducing `T`**, as the sampler is DDIM, and it still performs well at `T=20`.\n   - **Quantizing the DiffTSR model**, referring to relevant repositories on Diffusion model quantization.\n   - **Using the project\'s Baseline model**, which, although it may slightly reduce performance, provides approximately 2× speed-up while maintaining acceptable performance in most scenarios.\n   - **Performing model distillation on IDM** or training a smaller IDM model. In textual scenarios, a heavy model like general image generation may not be necessary.\n\n4. **How is the loss function set when training IDM? How is the text recognition loss implemented?**\n\n   **A:** When training IDM, two loss functions are used:\n   \n   - **L2 loss**: Used for predicting noise.\n   - **OCR loss**: Used for detecting text from the predicted clean `x^0`.\n   \n   Specifically:\n   - **L2 loss** is the traditional loss used in diffusion models, minimizing the difference between Unet output and noise map, enabling Unet to estimate noise.\n   - **OCR loss** is computed by first obtaining `z^(t-1)` from `z_t`, then deriving `z^0`, and subsequently decoding `z^0` to obtain `x^0`. The decoded `x^0` is fed into a frozen TransOCR model to obtain the text embedding in `x^0`. The cross-entropy loss is then computed between the predicted text embedding (`pred-text-embedding`) and the ground truth text embedding (`gt-text-embedding`). A weight constraint of `weight=0.02` is applied to the OCR loss.\n   \n   For more details, see [Issue](https://github.com/YuzheZhang-1999/DiffTSR/issues/13).\n\n5. **What are the loss functions used during training?**\n\n   **A:** The DiffTSR model training consists of three stages, each using a different combination of loss functions:\n   \n   - **Training IDM**: IDM trains Unet from scratch using loss `L_IDM`, which includes L2 loss and OCR loss.\n   - **Training TDM**: TDM trains the Transformer from scratch using loss `L_TDM`, referring to [Multinomial Diffusion](https://arxiv.org/pdf/2102.05379) Section 4.\n   - **Training the entire DiffTSR**: IDM and TDM are frozen, and only MoM is trained with loss `L_MoM = L_IDM + L_TDM * weight`.\n   \n   Where:\n      \n      ![L_IDM](https://latex.codecogs.com/svg.image?L_{IDM}=L_2+%5Clambda%20L_{OCR},%20%5Clambda=0.02)\n\n      ![L_TDM](https://latex.codecogs.com/svg.image?L_{TDM}=KL%28%5Cmathcal%7BC%7D%28%5Cpi_{post}%28%5Cmathbf%7Bc_t%7D,%20%5Cmathbf%7Bc_0%7D%29%29%20%7C%7C%20%5Cmathcal%7BC%7D%28%5Cpi_{post}%28%5Cmathbf%7Bc_t%7D,%20%5Cmathbf%7Bc_%7Bpred,t%7D%7D%29%29%29)\n\n      ![L_MoM](https://latex.codecogs.com/svg.image?L_{MoM}=L_{IDM}+%5Clambda%20L_{TDM},%20%5Clambda=1)\n\n   For detailed symbol definitions and theoretical derivations, see [Supplementary Material](https://openaccess.thecvf.com/content/CVPR2024/supplemental/Zhang_Diffusion-based_Blind_Text_CVPR_2024_supplemental.pdf) Section 1 and Algorithm 1 DiffTSR Training.\n\n   **To be continued...**\n\n\n</details>\n\n\n## 🌟 Input data and dataset production guideline\nWhen using this model, the input data should meet the following requirements:\n\n1. The number of characters must not exceed 24.\n2. The text must be arranged horizontally.\n3. Single-line text only.\n4. The aspect ratio must satisfy $1 \\leq \\mathrm{ratio}(W/H) \\leq 8$.\n5. In addition, during dataset cleaning, besides the above requirements, it is also necessary to ensure that the LR and HR images are not misaligned.\n\nFuture improvements aim to achieve more flexible, faster, and more general text image enhancement.\n\n\n## 🔥 TODO\n- [x] Attach the detailed implementation and supplementary material.\n- [x] Add inference code and checkpoints for blind text image SR.\n- [x] Add training code and scripts.\n\n## 👁️ Gallery\n\n[<img src="Repo_image/ImgSli_1.jpg" width="256px"/>](https://imgsli.com/MjY0MTk5) [<img src="Repo_image/ImgSli_2.jpg" width="256px"/>](https://imgsli.com/MjY0MjA0)\n\n## 🛠️ Try\n### Dependencies and Installation\n\n- Pytorch >= 1.7.0\n- CUDA >= 11.0\n```\n# git clone this repository\ngit clone https://github.com/YuzheZhang-1999/DiffTSR\ncd DiffTSR\n\n# create new anaconda env\nconda env create -f environment.yaml\nconda activate DiffTSR\n```\n### Download the checkpoint\nPlease download the checkpoint file from the URL below to the ./ckpt/ folder.\n\n- [[GoogleDrive](https://drive.google.com/drive/folders/1K6k5ZcvF3w-1MDN_gXQTdsLgFZ2SM8qy?usp=drive_link)] \n\n- [[BaiduDisk](https://pan.baidu.com/s/1hfaQzIp_V6H8AhAq5dfr8A)] [Password: vk9n] \n\n\n### Inference\n```\npython inference_DiffTSR.py\n# check the code for more detail\n```\n\n### Training\n```\n# cd ./train\n# check the README.md file for training details\n\n# Please note that you need to carefully review the training sh file and the configuration yaml. \n# Some of the configurations need to be modified according to your data or address. \n```\n\n\n## 🔎 Overview of DiffTSR\n![DiffTSR](Repo_image/paper-DiffTSR-model.jpg)\n### Abstract\nRecovering degraded low-resolution text images is challenging, especially for Chinese text images with complex strokes and severe degradation in real-world scenarios.\nEnsuring both text fidelity and style realness is crucial for high-quality text image super-resolution.\nRecently, diffusion models have achieved great success in natural image synthesis and restoration due to their powerful data distribution modeling abilities and data generation capabilities\nIn this work, we propose an Image Diffusion Model (IDM) to restore text images with realistic styles.\nFor diffusion models, they are not only suitable for modeling realistic image distribution but also appropriate for learning text distribution.\nSince text prior is important to guarantee the correctness of the restored text structure according to existing arts, we also propose a Text Diffusion Model (TDM) for text recognition which can guide IDM to generate text images with correct structures.\nWe further propose a Mixture of  Multi-modality module (MoM) to make these two diffusion models cooperate with each other in all the diffusion steps.\nExtensive experiments on synthetic and real-world datasets demonstrate that our Diffusion-based Blind Text Image Super-Resolution (DiffTSR) can restore text images with more accurate text structures as well as more realistic appearances simultaneously.\n\n### Visual performance comparison overview \n![DiffTSR](Repo_image/paper-fig1.jpg)\nBlind text image super-resolution results between different methods on synthetic and real-world text images. Our method can restore text images with high text fidelity and style realness under complex strokes, severe degradation, and various text styles.\n\n\n<details>\n  <summary>📷 More Visual Results</summary>\n\n  ## ![DiffTSR](Repo_image/paper-visual-comp-1.jpg)\n  ## ![DiffTSR](Repo_image/paper-visual-comp-2.jpg)\n  ## ![DiffTSR](Repo_image/paper-visual-comp-3.jpg)\n  ## ![DiffTSR](Repo_image/paper-visual-comp-4.jpg)\n  ## ![DiffTSR](Repo_image/paper-visual-comp-5.jpg)\n  ## ![DiffTSR](Repo_image/paper-visual-comp-6.jpg)\n\n</details>\n\n\n## 🎓Citations\n```\n@inproceedings{zhang2024diffusion,\n  title={Diffusion-based Blind Text Image Super-Resolution},\n  author={Zhang, Yuzhe and Zhang, Jiawei and Li, Hao and Wang, Zhouxia and Hou, Luwei and Zou, Dongqing and Bian, Liheng},\n  booktitle={Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition},\n  pages={25827--25836},\n  year={2024}\n}\n```\n\n## 🎫 License\nThis project is released under the [Apache 2.0 license](LICENSE).\n\n## Acknowledgement\nThanks to these awesome work：\n- [Latent Diffusion](https://github.com/CompVis/latent-diffusion)\n- [Benchmarking Chinese Text Recognition](https://github.com/FudanVI/benchmarking-chinese-text-recognition)\n\n<details>\n<summary>Statistics</summary>\n\n![visitors](https://visitor-badge.laobi.icu/badge?page_id=YuzheZhang-1999/DiffTSR)\n\n</details>', 'environment.yaml': "name: DiffTSR\nchannels:\n  - pytorch\n  - defaults\ndependencies:\n  - python=3.8.5\n  - pip=20.3\n  - cudatoolkit=11.0\n  - pytorch=1.7.0\n  - torchvision=0.8.1\n  - numpy=1.19.2\n  - pip:\n    - albumentations==0.4.3\n    - opencv-python==4.1.2.30\n    - pudb==2019.2\n    - imageio==2.9.0\n    - imageio-ffmpeg==0.4.2\n    - pytorch-lightning==1.4.2\n    - omegaconf==2.1.1\n    - test-tube>=0.7.5\n    - streamlit>=0.73.1 \n    - einops==0.3.0\n    - torch-fidelity==0.3.0\n    - transformers==4.3.1\n    - taming-transformers\n    - packaging==21.3\n    - 'torchmetrics<0.8'\n    - taming-transformers-rom1504 \n    - axial_positional_embedding\n    - linear_attention_transformer\n    - zhconv\n    - -e .", 'model/DiffTSR.py': 'import torch\nimport numpy as np\nfrom tqdm import tqdm\nfrom functools import partial\nimport torch.nn.functional as F\nimport torchvision.transforms as transforms\n\nfrom model.IDM.utils.util import instantiate_from_config, default\nfrom model.IDM.modules.diffusionmodules.util import make_ddim_sampling_parameters, make_ddim_timesteps\nfrom model.IDM.modules.diffusionmodules.util import make_beta_schedule, noise_like, extract_into_tensor\n\nfrom model.TDM.models.diffusion_multinomial import extract, log_add_exp, cosine_beta_schedule\nfrom model.TDM.models.diffusion_multinomial import log_1_min_a, index_to_log_onehot, log_onehot_to_index\n\n\nclass DiffTSR_pipline(object):\n    def __init__(self,\n                 max_length = 24,\n                 num_classes = 6736,\n                 transformer_dim = 768,\n                 scale_factor = 0.18215,\n                 IDM_Unet_config = None,\n                 TDM_Decoder_config = None,\n                 MoM_module_config = None,\n                 VAE_model_config = None,\n                 Text_Prediction_config = None):\n        super(DiffTSR_pipline, self).__init__()\n        self.device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n        \n        self.max_length = max_length\n        self.num_classes = num_classes\n        self.transformer_dim = transformer_dim\n        self.scale_factor = scale_factor\n\n        self.IDM_Unet = instantiate_from_config(IDM_Unet_config).to(self.device).eval()\n        self.TDM_Decoder = instantiate_from_config(TDM_Decoder_config).to(self.device).eval()\n        self.MoM_module = instantiate_from_config(MoM_module_config).to(self.device).eval()\n        self.VAE_model = instantiate_from_config(VAE_model_config).to(self.device).eval()\n        self.Text_Prediction = instantiate_from_config(Text_Prediction_config).to(self.device).eval()\n\n        self.IDM_ddpm_schedule_init(timesteps=1000, schedule="linear", \n                                    linear_start=0.0015, linear_end=0.0205, cosine_s=8e-3)\n                                    \n        self.IDM_ddim_schedule_init(ddim_num_steps=200, ddim_discretize="uniform",\n                                    ddim_eta=0.2, verbose=False)\n\n        self.TDM_ddpm_schedule_init(timesteps=1000)\n\n    def load_model(self, path):\n        state_dict = torch.load(path, map_location=self.device)\n        self.IDM_Unet.load_state_dict(state_dict[\'IDM_Unet\'], strict=True)\n        self.TDM_Decoder.load_state_dict(state_dict[\'TDM_Decoder\'], strict=True)\n        self.MoM_module.load_state_dict(state_dict[\'MoM_module\'], strict=True)\n        self.VAE_model.load_state_dict(state_dict[\'VAE_model\'], strict=True)\n        print(f"DiffTSR Model loaded from {path}")\n\n    def IDM_ddpm_schedule_init(self, timesteps, schedule, linear_start, linear_end, cosine_s):\n        to_torch = partial(torch.tensor, dtype=torch.float32)\n        self.IDM_betas = make_beta_schedule(schedule, timesteps, linear_start=linear_start, linear_end=linear_end, cosine_s=cosine_s)\n        self.IDM_alphas = 1. - self.IDM_betas\n        self.IDM_alphas_cumprod = np.cumprod(self.IDM_alphas, axis=0)\n        self.IDM_alphas_cumprod_prev = np.append(1., self.IDM_alphas_cumprod[:-1])\n\n        self.IDM_betas = to_torch(self.IDM_betas)\n        self.IDM_alphas = to_torch(self.IDM_alphas)\n        self.IDM_alphas_cumprod = to_torch(self.IDM_alphas_cumprod)\n        self.IDM_alphas_cumprod_prev = to_torch(self.IDM_alphas_cumprod_prev)\n\n        timesteps, = self.IDM_betas.shape\n        self.IDM_ddpm_num_timesteps = int(timesteps)\n\n    def IDM_ddim_schedule_init(self, ddim_num_steps, ddim_discretize="uniform", ddim_eta=0., verbose=True):\n        self.IDM_ddim_timesteps = make_ddim_timesteps(ddim_discr_method=ddim_discretize, num_ddim_timesteps=ddim_num_steps,\n                                                  num_ddpm_timesteps=self.IDM_ddpm_num_timesteps,verbose=verbose)\n\n        assert self.IDM_alphas_cumprod.shape[0] == self.IDM_ddpm_num_timesteps, \'alphas have to be defined for each timestep\'\n        to_torch = lambda x: x.clone().detach().to(torch.float32).to(self.device)\n\n        self.register_buffer(\'IDM_betas\', to_torch(self.IDM_betas))\n        self.register_buffer(\'IDM_alphas_cumprod\', to_torch(self.IDM_alphas_cumprod))\n        self.register_buffer(\'IDM_alphas_cumprod_prev\', to_torch(self.IDM_alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer(\'IDM_sqrt_alphas_cumprod\', to_torch(np.sqrt(self.IDM_alphas_cumprod.cpu())))\n        self.register_buffer(\'IDM_sqrt_one_minus_alphas_cumprod\', to_torch(np.sqrt(1. - self.IDM_alphas_cumprod.cpu())))\n        self.register_buffer(\'IDM_log_one_minus_alphas_cumprod\', to_torch(np.log(1. - self.IDM_alphas_cumprod.cpu())))\n        self.register_buffer(\'IDM_sqrt_recip_alphas_cumprod\', to_torch(np.sqrt(1. / self.IDM_alphas_cumprod.cpu())))\n        self.register_buffer(\'IDM_sqrt_recipm1_alphas_cumprod\', to_torch(np.sqrt(1. / self.IDM_alphas_cumprod.cpu() - 1)))\n\n        # ddim sampling parameters\n        IDM_ddim_sigmas, IDM_ddim_alphas, IDM_ddim_alphas_prev = make_ddim_sampling_parameters(alphacums=self.IDM_alphas_cumprod.cpu(),\n                                                                                                ddim_timesteps=self.IDM_ddim_timesteps,\n                                                                                                eta=ddim_eta,verbose=verbose)\n        self.register_buffer(\'IDM_ddim_sigmas\', IDM_ddim_sigmas)\n        self.register_buffer(\'IDM_ddim_alphas\', IDM_ddim_alphas)\n        self.register_buffer(\'IDM_ddim_alphas_prev\', IDM_ddim_alphas_prev)\n        self.register_buffer(\'IDM_ddim_sqrt_one_minus_alphas\', np.sqrt(1. - IDM_ddim_alphas))\n        IDM_sigmas_for_original_sampling_steps = ddim_eta * torch.sqrt(\n            (1 - self.IDM_alphas_cumprod_prev) / (1 - self.IDM_alphas_cumprod) * (\n                        1 - self.IDM_alphas_cumprod / self.IDM_alphas_cumprod_prev))\n        self.register_buffer(\'IDM_ddim_sigmas_for_original_num_steps\', IDM_sigmas_for_original_sampling_steps)\n\n    def TDM_ddpm_schedule_init(self, timesteps):\n        TDM_alphas = cosine_beta_schedule(timesteps)\n        TDM_alphas = torch.tensor(TDM_alphas.astype(\'float64\'))\n        TDM_log_alpha = np.log(TDM_alphas)\n        TDM_log_cumprod_alpha = np.cumsum(TDM_log_alpha)\n        TDM_log_1_min_alpha = log_1_min_a(TDM_log_alpha)\n        TDM_log_1_min_cumprod_alpha = log_1_min_a(TDM_log_cumprod_alpha)\n\n        self.register_buffer(\'TDM_log_alpha\', TDM_log_alpha.float())\n        self.register_buffer(\'TDM_log_1_min_alpha\', TDM_log_1_min_alpha.float())\n        self.register_buffer(\'TDM_log_cumprod_alpha\', TDM_log_cumprod_alpha.float())\n        self.register_buffer(\'TDM_log_1_min_cumprod_alpha\', TDM_log_1_min_cumprod_alpha.float())\n\n    def register_buffer(self, name, attr):\n        if type(attr) == torch.Tensor:\n            if attr.device != torch.device("cuda"):\n                attr = attr.to(torch.device("cuda"))\n        setattr(self, name, attr)\n\n    @torch.no_grad()\n    def IDM_p_sample(self, z_t, z_cond, c, t, index, temperature=1.):\n        b, *_, device = *z_t.shape, self.device\n        e_t = self.IDM_Unet(x=torch.cat([z_t, z_cond], dim=1), timesteps=t, context=c)\n\n        alphas = self.IDM_ddim_alphas\n        alphas_prev =  self.IDM_ddim_alphas_prev\n        sqrt_one_minus_alphas = self.IDM_ddim_sqrt_one_minus_alphas\n        sigmas = self.IDM_ddim_sigmas\n        \n        a_t = torch.full((b, 1, 1, 1), alphas[index], device=device)\n        a_prev = torch.full((b, 1, 1, 1), alphas_prev[index], device=device)\n        sigma_t = torch.full((b, 1, 1, 1), sigmas[index], device=device)\n        sqrt_one_minus_at = torch.full((b, 1, 1, 1), sqrt_one_minus_alphas[index],device=device)\n\n        pred_x0 = (z_t - sqrt_one_minus_at * e_t) / a_t.sqrt()\n\n        dir_xt = (1. - a_prev - sigma_t**2).sqrt() * e_t\n        noise = sigma_t * noise_like(z_t.shape, device, False) * temperature\n        x_prev = a_prev.sqrt() * pred_x0 + dir_xt + noise\n\n        return x_prev, pred_x0\n\n    def IDM_q_sample(self, x_start, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        return (extract_into_tensor(self.IDM_sqrt_alphas_cumprod, t, x_start.shape) * x_start +\n                extract_into_tensor(self.IDM_sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise)\n\n    @torch.no_grad()\n    def TDM_p_sample(self, log_x, t, context):\n        model_log_prob = self.TDM_p_pred(log_x=log_x, t=t, context=context)\n        out = self.TDM_log_sample_categorical(model_log_prob)\n        return out\n    \n    def TDM_p_pred(self, log_x, t, context):\n        log_x_recon = self.TDM_predict_start(log_x, t=t, context=context)\n        log_model_pred = self.TDM_q_posterior(log_x_start=log_x_recon, log_x_t=log_x, t=t)\n        return log_model_pred\n    \n    def TDM_predict_start(self, log_x_t, t, context):\n        x_t = log_onehot_to_index(log_x_t)\n        context = context.view(context.shape[0], -1, self.transformer_dim)\n        out = self.TDM_Decoder(x_t, t, context)\n        log_pred = F.log_softmax(out, dim=1)\n        return log_pred\n\n    def TDM_q_posterior(self, log_x_start, log_x_t, t):\n        # q(xt-1 | xt, x0) = q(xt | xt-1, x0) * q(xt-1 | x0) / q(xt | x0)\n        # where q(xt | xt-1, x0) = q(xt | xt-1).\n\n        t_minus_1 = t - 1\n        # Remove negative values, will not be used anyway for final decoder\n        t_minus_1 = torch.where(t_minus_1 < 0, torch.zeros_like(t_minus_1), t_minus_1)\n        log_EV_qxtmin_x0 = self.TDM_q_pred(log_x_start, t_minus_1)\n\n        num_axes = (1,) * (len(log_x_start.size()) - 1)\n        t_broadcast = t.view(-1, *num_axes) * torch.ones_like(log_x_start)\n        log_EV_qxtmin_x0 = torch.where(t_broadcast == 0, log_x_start, log_EV_qxtmin_x0)\n\n        # Note: _NOT_ x_tmin1, which is how the formula is typically used!!!\n        # Not very easy to see why this is true. But it is :)\n        unnormed_logprobs = log_EV_qxtmin_x0 + self.TDM_q_pred_one_timestep(log_x_t, t)\n\n        log_EV_xtmin_given_xt_given_xstart = \\\n            unnormed_logprobs \\\n            - torch.logsumexp(unnormed_logprobs, dim=1, keepdim=True)\n\n        return log_EV_xtmin_given_xt_given_xstart\n\n    def TDM_q_pred(self, log_x_start, t):\n        TDM_log_cumprod_alpha_t = extract(self.TDM_log_cumprod_alpha, t, log_x_start.shape)\n        TDM_log_1_min_cumprod_alpha = extract(self.TDM_log_1_min_cumprod_alpha, t, log_x_start.shape)\n\n        log_probs = log_add_exp(\n            log_x_start + TDM_log_cumprod_alpha_t,\n            TDM_log_1_min_cumprod_alpha - np.log(self.num_classes)\n        )\n\n        return log_probs\n    \n    def TDM_q_pred_one_timestep(self, log_x_t, t):\n        TDM_log_alpha_t = extract(self.TDM_log_alpha, t, log_x_t.shape)\n        TDM_log_1_min_alpha_t = extract(self.TDM_log_1_min_alpha, t, log_x_t.shape)\n\n        # alpha_t * E[xt] + (1 - alpha_t) 1 / K\n        log_probs = log_add_exp(\n            log_x_t + TDM_log_alpha_t,\n            TDM_log_1_min_alpha_t - np.log(self.num_classes)\n        )\n\n        return log_probs\n\n    @torch.no_grad()\n    def TDM_log_sample_categorical(self, logits):\n        uniform = torch.rand_like(logits)\n        gumbel_noise = -torch.log(-torch.log(uniform + 1e-30) + 1e-30)\n        sample = (gumbel_noise + logits).argmax(dim=1)\n        log_sample = index_to_log_onehot(sample, self.num_classes)\n        return log_sample\n    \n    @torch.no_grad()\n    def DiffTSR_sample(self, input_image, general=True):\n        lq_image = np.asarray(input_image)\n        lq_image = lq_image / 255.0\n        lq_image = np.ascontiguousarray(lq_image)\n        lq_image = transforms.ToTensor()(lq_image).float()\n        lq_image = lq_image.unsqueeze(0).to(self.device)\n\n        c_t = self.Text_Prediction.predict(lq_image)\n        c_t = c_t.to(self.device)\n\n        z_LR = self.VAE_model.encode(lq_image)\n        z_LR = z_LR.sample()\n        b = z_LR.shape[0]\n        noise = torch.randn_like(z_LR, device=z_LR.device)\n        t = torch.full((b,), 999, device=self.device, dtype=torch.long)\n        z_t = self.IDM_q_sample(x_start=z_LR, t=t, noise=noise)\n\n        timesteps = self.IDM_ddim_timesteps\n        time_range = np.flip(timesteps)\n        total_steps = timesteps.shape[0]\n        iterator = tqdm(time_range, desc=\'DiffTSR Sampling\', total=total_steps)\n\n        for i, step in enumerate(iterator):\n            index = total_steps - i - 1\n            ts = torch.full((b,), step, device=self.device, dtype=torch.long)\n            Icond_t, Ccond_t = self.MoM_module(c_t, z_LR, z_t, ts)\n            z_t, _ = self.IDM_p_sample(z_t, z_LR, Ccond_t, ts, index=index)\n            ts = torch.full((b,), index, device=self.device, dtype=torch.long)\n            if not general:\n                log_c_t = index_to_log_onehot(c_t, self.num_classes)\n                log_c_t = self.TDM_p_sample(log_c_t, ts, context=Icond_t)\n                c_t = log_onehot_to_index(log_c_t)\n\n        img = 1. / self.scale_factor * z_t\n        img = self.VAE_model.decode(img)\n        img = img.detach().cpu()\n        img = torch.clamp(img, 0., 1.)*255\n        img = img.numpy().astype(np.uint8)[0]\n        img = np.transpose(img, (1, 2, 0))\n\n        return img', 'model/DiffTSR_config.yaml': 'model:\n  target: model.DiffTSR.DiffTSR_pipline\n\n  params:\n    max_length: 24\n    num_classes: 6736\n    transformer_dim: 768\n\n    IDM_Unet_config:\n      target: model.IDM.modules.diffusionmodules.openaimodel.UNetModel\n      params:\n        image_size: [32, 128]\n        in_channels: 6\n        out_channels: 3\n        model_channels: 320\n        attention_resolutions:\n        - 8\n        - 4\n        - 2\n        num_res_blocks: 2\n        channel_mult:\n        - 1\n        - 2\n        - 3\n        - 4\n        num_heads: 8\n        resblock_updown: true\n        use_spatial_transformer: true\n        transformer_depth: 1\n        context_dim: 160\n        use_checkpoint: true\n        legacy: False\n\n    TDM_Decoder_config:\n      target: model.TDM.models.modules.DynamicsTransformer\n      params:\n        num_classes: 6736\n        transformer_dim: 768\n        transformer_heads: 16\n        transformer_depth: 12\n        transformer_blocks: 1\n        max_seq_len: 24\n        diffusion_steps: 1000\n        attn_layer_dropout: 0.0\n        transformer_local_heads: 8\n        transformer_local_size: 4\n        transformer_reversible: False\n        receives_context: True  \n\n    MoM_module_config:\n        target: model.MoM.MoM.MoM_model\n        params:\n          MoM_Unet:\n            target: model.IDM.modules.diffusionmodules.openaimodel.UNetModel\n            params:\n              image_size: [32, 128]\n              in_channels: 6\n              out_channels: 3\n              model_channels: 32\n              attention_resolutions:\n              - 0\n              num_res_blocks: 2\n              channel_mult:\n              - 1\n              - 2\n              num_heads: 4\n              resblock_updown: true\n              use_spatial_transformer: true\n              transformer_depth: 1\n              context_dim: 128\n              use_checkpoint: true\n              legacy: False\n          MoM_Transformer:\n            target: model.IDM.modules.encoders.modules.BERTEmbedder\n            params:\n              n_embed: 160\n              n_layer: 4\n              vocab_size: 6736\n              max_seq_len: 24\n              use_tokenizer: False\n\n    VAE_model_config:\n      target: model.IDM.models.autoencoder.AutoencoderKL\n      params:\n        embed_dim: 3\n        ddconfig:\n          double_z: true\n          z_channels: 3\n          resolution: 512\n          in_channels: 3\n          out_ch: 3\n          ch: 128\n          ch_mult:\n          - 1\n          - 2\n          - 4\n          num_res_blocks: 2\n          attn_resolutions: []\n          dropout: 0.0\n    \n    Text_Prediction_config:\n      target: model.TDM.models.text_prediction.Text_Prediction', 'model/DiffTSR_training.py': 'import torch, math\nimport numpy as np\nfrom functools import partial\nimport pytorch_lightning as pl\nimport torch.nn.functional as F\nfrom einops import rearrange, repeat\nfrom torch.optim.lr_scheduler import LambdaLR\n\nfrom model.IDM.utils.util import instantiate_from_config, default\nfrom model.IDM.modules.diffusionmodules.util import make_beta_schedule, extract_into_tensor\n\nfrom model.TDM.models.diffusion_multinomial import log_categorical, sum_except_batch\nfrom model.TDM.models.diffusion_multinomial import extract, log_add_exp, cosine_beta_schedule\nfrom model.TDM.models.diffusion_multinomial import log_1_min_a, index_to_log_onehot, log_onehot_to_index\n\nfrom model.modules.losses import Trans_OCR_loss\n\n\nclass DiffTSR_training_pipline(pl.LightningModule):\n    def __init__(self,\n                 max_length = 24,\n                 num_classes = 6736,\n                 transformer_dim = 768,\n                 scale_factor = 0.18215,\n                 learning_rate = 1.0e-05,\n                 IDM_Unet_config = None,\n                 TDM_Decoder_config = None,\n                 MoM_module_config = None,\n                 VAE_model_config = None,\n                 VAE_pretrained_ckpt = None,\n                 IDM_pretrained_ckpt = None,\n                 TDM_pretrained_ckpt = None,\n                 use_scheduler = None,\n                 lr_image_key = \'lq_image\',\n                 hr_image_key = \'hq_image\',\n                 text_label_key = \'text_label\',\n                 ocr_label_key = \'label_ocr\'):\n        super().__init__()\n        \n        self.max_length = max_length\n        self.num_classes = num_classes\n        self.transformer_dim = transformer_dim\n        self.scale_factor = scale_factor\n        self.learning_rate = learning_rate\n        self.use_scheduler = use_scheduler\n        \n        self.lr_image_key = lr_image_key\n        self.hr_image_key = hr_image_key\n        self.text_label_key = text_label_key\n        self.ocr_label_key = ocr_label_key\n\n        self.VAE_model = instantiate_from_config(VAE_model_config).to(self.device).eval()\n        self.IDM_Unet = instantiate_from_config(IDM_Unet_config).to(self.device).eval()\n        self.TDM_Decoder = instantiate_from_config(TDM_Decoder_config).to(self.device).eval()\n        self.MoM_module = instantiate_from_config(MoM_module_config).to(self.device)\n\n        # load pretrained IDM Unet\n        VAE_state_dict = torch.load(VAE_pretrained_ckpt, map_location=self.device)\n        self.VAE_model.load_state_dict(VAE_state_dict[\'state_dict\'], strict=True)\n        \n        self.init_from_ckpt(self.IDM_Unet, IDM_pretrained_ckpt, \'model.diffusion_model.\', \n                            [\'first_stage_model\', \'hybrid_concat_cond_stage_model\', \'pixel_loss\',\n                             \'hybrid_cross_attn_cond_stage_model\'])\n            \n        self.init_from_ckpt(self.TDM_Decoder, TDM_pretrained_ckpt, \'denoise_fn.diffusion_model.\',\n                            [\'cond_stage_model\'])\n\n        self.init_MoM_transformer_from_ckpt(self.MoM_module, IDM_pretrained_ckpt,\n                                            [\'first_stage_model\', \'hybrid_concat_cond_stage_model\', \'model.diffusion_model\'])\n        \n        self.IDM_ddpm_schedule_init(timesteps=1000, schedule="linear", \n                                    linear_start=0.0015, linear_end=0.0205, cosine_s=8e-3)\n\n        self.TDM_ddpm_schedule_init(timesteps=1000)\n            \n        ## define loss functions\n        self.IDM_recognize_loss_metric = Trans_OCR_loss(0.02, 32, 256, \n                                                 \'model/TDM/utils/benchmark.txt\', \n                                                 \'train/ckpt/others/transocr.pth\')\n\n        self.VAE_model.train = disabled_train\n        for param in self.VAE_model.parameters():\n            param.requires_grad = False\n        self.IDM_Unet.train = disabled_train\n        self.TDM_Decoder.train = disabled_train\n        self.IDM_recognize_loss_metric.train = disabled_train\n        for param in self.IDM_recognize_loss_metric.parameters():\n            param.requires_grad = False\n\n    def init_from_ckpt(self, model, path, delet_prefix=None, ignore_keys=None):\n        model_weight = torch.load(path, map_location="cpu")\n        if "state_dict" in list(model_weight.keys()):\n            model_weight = model_weight["state_dict"]\n        \n        if ignore_keys is not None:\n            keys = list(model_weight.keys())\n            for k in keys:\n                for ik in ignore_keys:\n                    if k.startswith(ik):\n                        del model_weight[k]\n        if delet_prefix is not None:\n            keys = list(model_weight.keys())\n            for k in keys:\n                if delet_prefix != \'\' and k.startswith(delet_prefix):\n                    new_k = k[len(delet_prefix):]\n                    model_weight[new_k] = model_weight.pop(k)\n\n        missing, unexpected = model.load_state_dict(model_weight, strict=False)\n        print(f"Restored from {path} with {len(missing)} missing and {len(unexpected)} unexpected keys")\n        if len(missing) > 0:\n            print(f"Missing Keys: {missing}")\n        if len(unexpected) > 0:\n            print(f"Unexpected Keys: {unexpected}")\n\n    def init_MoM_transformer_from_ckpt(self, model, path, ignore_keys=None):\n        model_weight = torch.load(path, map_location="cpu")\n        if "state_dict" in model_weight:\n            model_weight = model_weight["state_dict"]\n\n        prefix = "hybrid_cross_attn_cond_stage_model."\n        filtered_weight = {}\n        for k, v in model_weight.items():\n            if k.startswith(prefix):\n                new_k = "Transformer." + k[len(prefix):]\n                filtered_weight[new_k] = v\n\n        if ignore_keys is not None:\n            keys = list(filtered_weight.keys())\n            for k in keys:\n                for ik in ignore_keys:\n                    if k.startswith(ik):\n                        print(f"Deleting key {k} from state_dict.")\n                        del filtered_weight[k]\n\n        missing, unexpected = model.load_state_dict(filtered_weight, strict=False)\n        print(f"Restored from {path} with {len(missing)} missing and {len(unexpected)} unexpected keys")\n        if len(missing) > 0:\n            print(f"Missing Keys: {missing[:20]} ...")\n        if len(unexpected) > 0:\n            print(f"Unexpected Keys: {unexpected[:20]} ...")\n\n    def IDM_ddpm_schedule_init(self, timesteps, schedule, linear_start, linear_end, cosine_s):\n        to_torch = partial(torch.tensor, dtype=torch.float32)\n        IDM_betas = make_beta_schedule(schedule, timesteps, linear_start=linear_start, linear_end=linear_end, cosine_s=cosine_s)\n        IDM_alphas = 1. - IDM_betas\n        IDM_alphas_cumprod = np.cumprod(IDM_alphas, axis=0)\n        IDM_alphas_cumprod_prev = np.append(1., IDM_alphas_cumprod[:-1])\n\n        timesteps, = IDM_betas.shape\n        self.IDM_ddpm_num_timesteps = int(timesteps)\n\n        assert IDM_alphas_cumprod.shape[0] == self.IDM_ddpm_num_timesteps, \'alphas have to be defined for each timestep\'\n        to_torch = lambda x: x.clone().detach().to(torch.float32).to(self.device)\n\n        to_torch = partial(torch.tensor, dtype=torch.float32)\n\n        self.register_buffer(\'IDM_betas\', to_torch(IDM_betas))\n        self.register_buffer(\'IDM_alphas_cumprod\', to_torch(IDM_alphas_cumprod))\n        self.register_buffer(\'IDM_alphas_cumprod_prev\', to_torch(IDM_alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer(\'IDM_sqrt_alphas_cumprod\', to_torch(np.sqrt(IDM_alphas_cumprod)))\n        self.register_buffer(\'IDM_sqrt_one_minus_alphas_cumprod\', to_torch(np.sqrt(1. - IDM_alphas_cumprod)))\n        self.register_buffer(\'IDM_log_one_minus_alphas_cumprod\', to_torch(np.log(1. - IDM_alphas_cumprod)))\n        self.register_buffer(\'IDM_sqrt_recip_alphas_cumprod\', to_torch(np.sqrt(1. / IDM_alphas_cumprod)))\n        self.register_buffer(\'IDM_sqrt_recipm1_alphas_cumprod\', to_torch(np.sqrt(1. / IDM_alphas_cumprod - 1)))\n\n    def TDM_ddpm_schedule_init(self, timesteps):\n        TDM_alphas = cosine_beta_schedule(timesteps)\n        TDM_alphas = torch.tensor(TDM_alphas.astype(\'float64\'))\n        TDM_log_alpha = np.log(TDM_alphas)\n        TDM_log_cumprod_alpha = np.cumsum(TDM_log_alpha)\n        TDM_log_1_min_alpha = log_1_min_a(TDM_log_alpha)\n        TDM_log_1_min_cumprod_alpha = log_1_min_a(TDM_log_cumprod_alpha)\n\n        self.register_buffer(\'TDM_log_alpha\', TDM_log_alpha.float())\n        self.register_buffer(\'TDM_log_1_min_alpha\', TDM_log_1_min_alpha.float())\n        self.register_buffer(\'TDM_log_cumprod_alpha\', TDM_log_cumprod_alpha.float())\n        self.register_buffer(\'TDM_log_1_min_cumprod_alpha\', TDM_log_1_min_cumprod_alpha.float())\n\n    def IDM_q_sample(self, x_start, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        return (extract_into_tensor(self.IDM_sqrt_alphas_cumprod, t, x_start.shape) * x_start +\n                extract_into_tensor(self.IDM_sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise)\n\n    def IDM_predict_start_from_noise(self, x_t, t, noise):\n        return (\n                extract_into_tensor(self.IDM_sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -\n                extract_into_tensor(self.IDM_sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise\n        )\n\n    @torch.no_grad()\n    def TDM_p_sample(self, log_x, t, context):\n        model_log_prob = self.TDM_p_pred(log_x=log_x, t=t, context=context)\n        out = self.TDM_log_sample_categorical(model_log_prob)\n        return out\n    \n    def TDM_p_pred(self, log_x, t, context):\n        log_x_recon = self.TDM_predict_start(log_x, t=t, context=context)\n        log_model_pred = self.TDM_q_posterior(log_x_start=log_x_recon, log_x_t=log_x, t=t)\n        return log_model_pred\n    \n    def TDM_predict_start(self, log_x_t, t, context):\n        x_t = log_onehot_to_index(log_x_t)\n        context = context.view(context.shape[0], -1, self.transformer_dim)\n        out = self.TDM_Decoder(x_t, t, context)\n        log_pred = F.log_softmax(out, dim=1)\n        return log_pred\n\n    def TDM_q_posterior(self, log_x_start, log_x_t, t):\n        t_minus_1 = t - 1\n        t_minus_1 = torch.where(t_minus_1 < 0, torch.zeros_like(t_minus_1), t_minus_1)\n        log_EV_qxtmin_x0 = self.TDM_q_pred(log_x_start, t_minus_1)\n\n        num_axes = (1,) * (len(log_x_start.size()) - 1)\n        t_broadcast = t.view(-1, *num_axes) * torch.ones_like(log_x_start)\n        log_EV_qxtmin_x0 = torch.where(t_broadcast == 0, log_x_start, log_EV_qxtmin_x0)\n\n        unnormed_logprobs = log_EV_qxtmin_x0 + self.TDM_q_pred_one_timestep(log_x_t, t)\n\n        log_EV_xtmin_given_xt_given_xstart = \\\n            unnormed_logprobs \\\n            - torch.logsumexp(unnormed_logprobs, dim=1, keepdim=True)\n\n        return log_EV_xtmin_given_xt_given_xstart\n\n    def TDM_kl_prior(self, log_x_start):\n        b = log_x_start.size(0)\n        device = log_x_start.device\n        ones = torch.ones(b, device=device).long()\n\n        log_qxT_prob = self.TDM_q_pred(log_x_start, t=(self.IDM_ddpm_num_timesteps - 1) * ones)\n        log_half_prob = -torch.log(self.num_classes * torch.ones_like(log_qxT_prob))\n\n        kl_prior = self.TDM_multinomial_kl(log_qxT_prob, log_half_prob)\n        return sum_except_batch(kl_prior)\n    \n    def TDM_q_sample(self, log_x_start, t):\n        log_EV_qxt_x0 = self.TDM_q_pred(log_x_start, t)\n        log_sample = self.TDM_log_sample_categorical(log_EV_qxt_x0)\n        return log_sample\n    \n    def TDM_q_pred(self, log_x_start, t):\n        TDM_log_cumprod_alpha_t = extract(self.TDM_log_cumprod_alpha, t, log_x_start.shape)\n        TDM_log_1_min_cumprod_alpha = extract(self.TDM_log_1_min_cumprod_alpha, t, log_x_start.shape)\n\n        log_probs = log_add_exp(\n            log_x_start + TDM_log_cumprod_alpha_t,\n            TDM_log_1_min_cumprod_alpha - np.log(self.num_classes)\n        )\n\n        return log_probs\n    \n    def TDM_q_pred_one_timestep(self, log_x_t, t):\n        TDM_log_alpha_t = extract(self.TDM_log_alpha, t, log_x_t.shape)\n        TDM_log_1_min_alpha_t = extract(self.TDM_log_1_min_alpha, t, log_x_t.shape)\n\n        # alpha_t * E[xt] + (1 - alpha_t) 1 / K\n        log_probs = log_add_exp(\n            log_x_t + TDM_log_alpha_t,\n            TDM_log_1_min_alpha_t - np.log(self.num_classes)\n        )\n\n        return log_probs\n\n    @torch.no_grad()\n    def TDM_log_sample_categorical(self, logits):\n        uniform = torch.rand_like(logits)\n        gumbel_noise = -torch.log(-torch.log(uniform + 1e-30) + 1e-30)\n        sample = (gumbel_noise + logits).argmax(dim=1)\n        log_sample = index_to_log_onehot(sample, self.num_classes)\n        return log_sample\n    \n    def TDM_multinomial_kl(self, log_prob1, log_prob2):\n        kl = (log_prob1.exp() * (log_prob1 - log_prob2)).sum(dim=1)\n        return kl\n    \n    @torch.no_grad()\n    def get_input(self, batch, lr_image_key, hr_image_key, text_label_key, ocr_label_key):\n        lr_image = batch[lr_image_key]\n        hr_image = batch[hr_image_key]\n        text_label = batch[text_label_key]\n        ocr_label = batch[ocr_label_key]\n\n        lr_image = rearrange(lr_image, \'b h w c -> b c h w\').to(memory_format=torch.contiguous_format).float()\n        hr_image = rearrange(hr_image, \'b h w c -> b c h w\').to(memory_format=torch.contiguous_format).float()\n\n        lr_latent = self.VAE_model.encode(lr_image).sample().detach()*self.scale_factor\n        hr_latent = self.VAE_model.encode(hr_image).sample().detach()*self.scale_factor\n\n        return lr_image, hr_image, lr_latent, hr_latent, text_label, ocr_label\n\n\n    def forward(self, lr_latent, hr_latent, text_label, ocr_label):\n        t = torch.randint(0, self.IDM_ddpm_num_timesteps, (hr_latent.shape[0],), device=self.device).long()\n        \n        ## for IDM add noise\n        IDM_noise = torch.randn_like(hr_latent)\n        IDM_x_noisy = self.IDM_q_sample(x_start=hr_latent, t=t, noise=IDM_noise)\n        \n        ## for TDM add noise\n        TDM_log_x_start = index_to_log_onehot(text_label, self.num_classes)\n        TDM_log_x_noisy = self.TDM_q_sample(log_x_start=TDM_log_x_start, t=t)\n        TDM_log_true_prob = self.TDM_q_posterior(log_x_start=TDM_log_x_start, log_x_t=TDM_log_x_noisy, t=t)\n\n        ## for MoM get condition\n        I_cond, C_cond = self.MoM_module(c_t=text_label, z_lr=lr_latent, z_t=IDM_x_noisy, t=t)\n\n        ## IDM predict\n        IDM_model_predict = self.IDM_Unet(x=torch.cat([IDM_x_noisy, lr_latent], dim=1), timesteps=t, context=C_cond)\n\n        ## TDM predict\n        TDM_log_model_predict = self.TDM_p_pred(log_x=TDM_log_x_noisy, t=t, context=I_cond)\n\n        ## calculate IDM loss\n        IDM_denoise_loss = (IDM_noise - IDM_model_predict).abs().mean(dim=[1, 2, 3])\n        IDM_predict_x0_latent = self.IDM_predict_start_from_noise(IDM_x_noisy, t=t, noise=IDM_model_predict)\n        IMD_predict_x0 = self.VAE_model.decode(1. / self.scale_factor * IDM_predict_x0_latent)\n        IDM_recognize_loss = self.IDM_recognize_loss_metric(IMD_predict_x0, ocr_label)\n        IDM_loss = IDM_denoise_loss + IDM_recognize_loss * 1.0\n        IDM_loss = IDM_loss.mean()\n\n        ## calculate TDM loss\n        TDM_kl = self.TDM_multinomial_kl(TDM_log_true_prob, TDM_log_model_predict)\n        TDM_kl = sum_except_batch(TDM_kl)\n        decoder_nll = -log_categorical(TDM_log_x_start, TDM_log_model_predict)\n        decoder_nll = sum_except_batch(decoder_nll)\n        mask = (t == torch.zeros_like(t)).float()\n        TDM_kl = mask * decoder_nll + (1. - mask) * TDM_kl\n        kl_prior = self.TDM_kl_prior(TDM_log_x_start)\n        vb_loss = TDM_kl / (torch.ones_like(t).float() / self.IDM_ddpm_num_timesteps) + kl_prior\n        TDM_loss = 0.1 * vb_loss.sum() / (math.log(2.) * hr_latent.shape[0] * self.max_length)\n    \n        ##\n        total_loss = IDM_loss + TDM_loss\n\n        loss_dict = {\n            \'total_loss\': total_loss,\n            \'IDM_loss\': IDM_loss,\n            \'TDM_loss\': TDM_loss\n        }\n\n        return total_loss, loss_dict\n\n    def training_step(self, batch, batch_idx):\n        _, _, lr_latent, hr_latent, text_label, ocr_label = self.get_input(batch,\n                                                                           lr_image_key=self.lr_image_key,\n                                                                           hr_image_key=self.hr_image_key,\n                                                                           text_label_key=self.text_label_key,\n                                                                           ocr_label_key=self.ocr_label_key)\n        loss, loss_dict = self.forward(lr_latent, hr_latent, text_label, ocr_label)\n        self.log_dict(loss_dict, prog_bar=True, logger=True, on_step=True, on_epoch=True)\n        self.log("global_step", self.global_step, prog_bar=True, logger=True, on_step=True, on_epoch=False)\n        return loss\n    \n    @torch.no_grad()\n    def validation_step(self, batch, batch_idx):\n        _, _, lr_latent, hr_latent, text_label, ocr_label = self.get_input(batch,\n                                                                           lr_image_key=self.lr_image_key,\n                                                                           hr_image_key=self.hr_image_key,\n                                                                           text_label_key=self.text_label_key,\n                                                                           ocr_label_key=self.ocr_label_key)\n        _, loss_dict = self.forward(lr_latent, hr_latent, text_label, ocr_label)\n        self.log_dict(loss_dict, prog_bar=False, logger=True, on_step=False, on_epoch=True)\n\n    def configure_optimizers(self):\n        lr = self.learning_rate\n        params = list(self.MoM_module.parameters())\n        opt = torch.optim.AdamW(params, lr=lr)\n        if self.use_scheduler:\n            assert \'target\' in self.scheduler_config\n            scheduler = instantiate_from_config(self.scheduler_config)\n            print("Setting up LambdaLR scheduler...")\n            scheduler = [\n                {\n                    \'scheduler\': LambdaLR(opt, lr_lambda=scheduler.schedule),\n                    \'interval\': \'step\',\n                    \'frequency\': 1\n                }]\n            return [opt], scheduler\n        return opt\n    \n\ndef disabled_train(self, mode=True):\n    """Overwrite model.train with this function to make sure train/eval mode\n    does not change anymore."""\n    return self\n', 'model/IDM/models/autoencoder.py': 'import torch\nimport pytorch_lightning as pl\nimport torch.nn.functional as F\n\nfrom model.IDM.modules.diffusionmodules.model import Encoder, Decoder\nfrom model.IDM.modules.distributions.distributions import DiagonalGaussianDistribution\nfrom model.IDM.utils.util import instantiate_from_config\n\n\nclass AutoencoderKL(pl.LightningModule):\n    def __init__(self,\n                 ddconfig,\n                 lossconfig=None,\n                 embed_dim=3,\n                 ckpt_path=None,\n                 ignore_keys=[],\n                 image_key="image",\n                 colorize_nlabels=None,\n                 monitor=None,\n                 ):\n        super().__init__()\n        self.image_key = image_key\n        self.encoder = Encoder(**ddconfig)\n        self.decoder = Decoder(**ddconfig)\n        if lossconfig != None:\n            self.loss = instantiate_from_config(lossconfig)\n        assert ddconfig["double_z"]\n        self.quant_conv = torch.nn.Conv2d(2*ddconfig["z_channels"], 2*embed_dim, 1)\n        self.post_quant_conv = torch.nn.Conv2d(embed_dim, ddconfig["z_channels"], 1)\n        self.embed_dim = embed_dim\n        if colorize_nlabels is not None:\n            assert type(colorize_nlabels)==int\n            self.register_buffer("colorize", torch.randn(3, colorize_nlabels, 1, 1))\n        if monitor is not None:\n            self.monitor = monitor\n        if ckpt_path is not None:\n            self.init_from_ckpt(ckpt_path, ignore_keys=ignore_keys)\n\n    def init_from_ckpt(self, path, ignore_keys=list()):\n        sd = torch.load(path, map_location="cpu")["state_dict"]\n        keys = list(sd.keys())\n        for k in keys:\n            for ik in ignore_keys:\n                if k.startswith(ik):\n                    print("Deleting key {} from state_dict.".format(k))\n                    del sd[k]\n        self.load_state_dict(sd, strict=False)\n        print(f"Restored from {path}")\n\n    def encode(self, x):\n        h = self.encoder(x)\n        moments = self.quant_conv(h)\n        posterior = DiagonalGaussianDistribution(moments)\n        return posterior\n\n    def decode(self, z):\n        z = self.post_quant_conv(z)\n        dec = self.decoder(z)\n        return dec\n\n    def forward(self, input, sample_posterior=True):\n        posterior = self.encode(input)\n        if sample_posterior:\n            z = posterior.sample()\n        else:\n            z = posterior.mode()\n        dec = self.decode(z)\n        return dec, posterior\n\n    def get_input(self, batch, k):\n        x = batch[k]\n        if len(x.shape) == 3:\n            x = x[..., None]\n        x = x.permute(0, 3, 1, 2).to(memory_format=torch.contiguous_format).float()\n        return x\n\n    def training_step(self, batch, batch_idx, optimizer_idx):\n        inputs = self.get_input(batch, self.image_key)\n        reconstructions, posterior = self(inputs)\n\n        if optimizer_idx == 0:\n            # train encoder+decoder+logvar\n            aeloss, log_dict_ae = self.loss(inputs, reconstructions, posterior, optimizer_idx, self.global_step,\n                                            last_layer=self.get_last_layer(), split="train")\n            self.log("aeloss", aeloss, prog_bar=True, logger=True, on_step=True, on_epoch=True)\n            self.log_dict(log_dict_ae, prog_bar=False, logger=True, on_step=True, on_epoch=False)\n            return aeloss\n\n        if optimizer_idx == 1:\n            # train the discriminator\n            discloss, log_dict_disc = self.loss(inputs, reconstructions, posterior, optimizer_idx, self.global_step,\n                                                last_layer=self.get_last_layer(), split="train")\n\n            self.log("discloss", discloss, prog_bar=True, logger=True, on_step=True, on_epoch=True)\n            self.log_dict(log_dict_disc, prog_bar=False, logger=True, on_step=True, on_epoch=False)\n            return discloss\n\n    def validation_step(self, batch, batch_idx):\n        inputs = self.get_input(batch, self.image_key)\n        reconstructions, posterior = self(inputs)\n        aeloss, log_dict_ae = self.loss(inputs, reconstructions, posterior, 0, self.global_step,\n                                        last_layer=self.get_last_layer(), split="val")\n\n        discloss, log_dict_disc = self.loss(inputs, reconstructions, posterior, 1, self.global_step,\n                                            last_layer=self.get_last_layer(), split="val")\n\n        self.log("val/rec_loss", log_dict_ae["val/rec_loss"])\n        self.log_dict(log_dict_ae)\n        self.log_dict(log_dict_disc)\n        return self.log_dict\n\n    def configure_optimizers(self):\n        lr = self.learning_rate\n        opt_ae = torch.optim.Adam(list(self.encoder.parameters())+\n                                  list(self.decoder.parameters())+\n                                  list(self.quant_conv.parameters())+\n                                  list(self.post_quant_conv.parameters()),\n                                  lr=lr, betas=(0.5, 0.9))\n        opt_disc = torch.optim.Adam(self.loss.discriminator.parameters(),\n                                    lr=lr, betas=(0.5, 0.9))\n        return [opt_ae, opt_disc], []\n\n    def get_last_layer(self):\n        return self.decoder.conv_out.weight\n\n    @torch.no_grad()\n    def log_images(self, batch, only_inputs=False, **kwargs):\n        log = dict()\n        x = self.get_input(batch, self.image_key)\n        x = x.to(self.device)\n        if not only_inputs:\n            xrec, posterior = self(x)\n            if x.shape[1] > 3:\n                # colorize with random projection\n                assert xrec.shape[1] > 3\n                x = self.to_rgb(x)\n                xrec = self.to_rgb(xrec)\n            log["samples"] = self.decode(torch.randn_like(posterior.sample()))\n            log["reconstructions"] = xrec\n        log["inputs"] = x\n        return log\n\n    def to_rgb(self, x):\n        assert self.image_key == "segmentation"\n        if not hasattr(self, "colorize"):\n            self.register_buffer("colorize", torch.randn(3, x.shape[1], 1, 1).to(x))\n        x = F.conv2d(x, weight=self.colorize)\n        x = 2.*(x-x.min())/(x.max()-x.min()) - 1.\n        return x\n\n\nclass IdentityFirstStage(torch.nn.Module):\n    def __init__(self, *args, vq_interface=False, **kwargs):\n        self.vq_interface = vq_interface  # TODO: Should be true by default but check to not break older stuff\n        super().__init__()\n\n    def encode(self, x, *args, **kwargs):\n        return x\n\n    def decode(self, x, *args, **kwargs):\n        return x\n\n    def quantize(self, x, *args, **kwargs):\n        if self.vq_interface:\n            return x, None, [None, None, None]\n        return x\n\n    def forward(self, x, *args, **kwargs):\n        return x\n', 'model/IDM/modules/IDM_diffusion.py': '"""\nderive from https://github.com/CompVis/latent-diffusion\n"""\n\nimport torch\nimport torch.nn as nn\nimport numpy as np\nimport pytorch_lightning as pl\nfrom torch.optim.lr_scheduler import LambdaLR\nfrom einops import rearrange, repeat\nfrom contextlib import contextmanager\nfrom functools import partial\nfrom tqdm import tqdm\nfrom torchvision.utils import make_grid\n\nfrom model.IDM.utils.util import log_txt_as_img_from_alphabet, exists, default, ismap, mean_flat, count_params, instantiate_from_config\nfrom model.IDM.modules.ema import LitEma\nfrom model.IDM.modules.distributions.distributions import normal_kl, DiagonalGaussianDistribution\nfrom model.IDM.models.autoencoder import IdentityFirstStage, AutoencoderKL\nfrom model.IDM.modules.diffusionmodules.util import make_beta_schedule, extract_into_tensor, noise_like\nfrom model.IDM.modules.diffusionmodules.ddim import DDIMSampler\n\n\n\ndef disabled_train(self, mode=True):\n    """Overwrite model.train with this function to make sure train/eval mode\n    does not change anymore."""\n    return self\n\n\ndef uniform_on_device(r1, r2, shape, device):\n    return (r1 - r2) * torch.rand(*shape, device=device) + r2\n\n\nclass DDPM(pl.LightningModule):\n    # classic DDPM with Gaussian diffusion, in image space\n    def __init__(self,\n                 unet_config,\n                 timesteps=1000,\n                 beta_schedule="linear",\n                 loss_type="l2",\n                 ckpt_path=None,\n                 ignore_keys=[],\n                 load_only_unet=False,\n                 monitor="val/loss",\n                 use_ema=True,\n                 first_stage_key="image",\n                 image_size=[256, 256],\n                 channels=3,\n                 log_every_t=100,\n                 clip_denoised=True,\n                 linear_start=1e-4,\n                 linear_end=2e-2,\n                 cosine_s=8e-3,\n                 given_betas=None,\n                 original_elbo_weight=0.,\n                 v_posterior=0.,  # weight for choosing posterior variance as sigma = (1-v) * beta_tilde + v * beta\n                 l_simple_weight=1.,\n                 conditioning_key=None,\n                 parameterization="eps",  # all assuming fixed variance schedules\n                 scheduler_config=None,\n                 logvar_init=0.,\n                 ):\n        super().__init__()\n        assert parameterization in ["eps", "x0"], \'currently only supporting "eps" and "x0"\'\n        self.parameterization = parameterization\n        print(f"{self.__class__.__name__}: Running in {self.parameterization}-prediction mode")\n        self.cond_stage_model = None\n        self.clip_denoised = clip_denoised\n        self.log_every_t = log_every_t\n        self.first_stage_key = first_stage_key\n        self.image_size = image_size  # try conv?\n        self.channels = channels\n        self.model = DiffusionWrapper(unet_config, conditioning_key)\n        count_params(self.model, verbose=True)\n        self.use_ema = use_ema\n        if self.use_ema:\n            self.model_ema = LitEma(self.model)\n            print(f"Keeping EMAs of {len(list(self.model_ema.buffers()))}.")\n\n        self.use_scheduler = scheduler_config is not None\n        if self.use_scheduler:\n            self.scheduler_config = scheduler_config\n\n        self.v_posterior = v_posterior\n        self.original_elbo_weight = original_elbo_weight\n        self.l_simple_weight = l_simple_weight\n\n        if monitor is not None:\n            self.monitor = monitor\n        if ckpt_path is not None:\n            self.init_from_ckpt(ckpt_path, ignore_keys=ignore_keys, only_model=load_only_unet)\n\n        self.register_schedule(given_betas=given_betas, beta_schedule=beta_schedule, timesteps=timesteps,\n                               linear_start=linear_start, linear_end=linear_end, cosine_s=cosine_s)\n\n        self.loss_type = loss_type\n        self.logvar = torch.full(fill_value=logvar_init, size=(self.num_timesteps,))\n\n    def register_schedule(self, given_betas=None, beta_schedule="linear", timesteps=1000,\n                          linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):\n        if exists(given_betas):\n            betas = given_betas\n        else:\n            betas = make_beta_schedule(beta_schedule, timesteps, linear_start=linear_start, linear_end=linear_end,\n                                       cosine_s=cosine_s)\n        alphas = 1. - betas\n        alphas_cumprod = np.cumprod(alphas, axis=0)\n        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])\n\n        timesteps, = betas.shape\n        self.num_timesteps = int(timesteps)\n        self.linear_start = linear_start\n        self.linear_end = linear_end\n        assert alphas_cumprod.shape[0] == self.num_timesteps, \'alphas have to be defined for each timestep\'\n\n        to_torch = partial(torch.tensor, dtype=torch.float32)\n\n        self.register_buffer(\'betas\', to_torch(betas))\n        self.register_buffer(\'alphas_cumprod\', to_torch(alphas_cumprod))\n        self.register_buffer(\'alphas_cumprod_prev\', to_torch(alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer(\'sqrt_alphas_cumprod\', to_torch(np.sqrt(alphas_cumprod)))\n        self.register_buffer(\'sqrt_one_minus_alphas_cumprod\', to_torch(np.sqrt(1. - alphas_cumprod)))\n        self.register_buffer(\'log_one_minus_alphas_cumprod\', to_torch(np.log(1. - alphas_cumprod)))\n        self.register_buffer(\'sqrt_recip_alphas_cumprod\', to_torch(np.sqrt(1. / alphas_cumprod)))\n        self.register_buffer(\'sqrt_recipm1_alphas_cumprod\', to_torch(np.sqrt(1. / alphas_cumprod - 1)))\n\n        # calculations for posterior q(x_{t-1} | x_t, x_0)\n        posterior_variance = (1 - self.v_posterior) * betas * (1. - alphas_cumprod_prev) / (\n                    1. - alphas_cumprod) + self.v_posterior * betas\n        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)\n        self.register_buffer(\'posterior_variance\', to_torch(posterior_variance))\n        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain\n        self.register_buffer(\'posterior_log_variance_clipped\', to_torch(np.log(np.maximum(posterior_variance, 1e-20))))\n        self.register_buffer(\'posterior_mean_coef1\', to_torch(\n            betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))\n        self.register_buffer(\'posterior_mean_coef2\', to_torch(\n            (1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))\n\n        if self.parameterization == "eps":\n            lvlb_weights = self.betas ** 2 / (\n                        2 * self.posterior_variance * to_torch(alphas) * (1 - self.alphas_cumprod))\n        elif self.parameterization == "x0":\n            lvlb_weights = 0.5 * np.sqrt(torch.Tensor(alphas_cumprod)) / (2. * 1 - torch.Tensor(alphas_cumprod))\n        else:\n            raise NotImplementedError("mu not supported")\n        # TODO how to choose this term\n        lvlb_weights[0] = lvlb_weights[1]\n        self.register_buffer(\'lvlb_weights\', lvlb_weights, persistent=False)\n        assert not torch.isnan(self.lvlb_weights).all()\n\n    @contextmanager\n    def ema_scope(self, context=None):\n        if self.use_ema:\n            self.model_ema.store(self.model.parameters())\n            self.model_ema.copy_to(self.model)\n            if context is not None:\n                print(f"{context}: Switched to EMA weights")\n        try:\n            yield None\n        finally:\n            if self.use_ema:\n                self.model_ema.restore(self.model.parameters())\n                if context is not None:\n                    print(f"{context}: Restored training weights")\n\n    def init_from_ckpt(self, path, ignore_keys=list(), only_model=False):\n        sd = torch.load(path, map_location="cpu")\n        if "state_dict" in list(sd.keys()):\n            sd = sd["state_dict"]\n        keys = list(sd.keys())\n        for k in keys:\n            for ik in ignore_keys:\n                if k.startswith(ik):\n                    print("Deleting key {} from state_dict.".format(k))\n                    del sd[k]\n        missing, unexpected = self.load_state_dict(sd, strict=False) if not only_model else self.model.load_state_dict(\n            sd, strict=False)\n        print(f"Restored from {path} with {len(missing)} missing and {len(unexpected)} unexpected keys")\n        if len(missing) > 0:\n            print(f"Missing Keys: {missing}")\n        if len(unexpected) > 0:\n            print(f"Unexpected Keys: {unexpected}")\n\n    def q_mean_variance(self, x_start, t):\n        """\n        Get the distribution q(x_t | x_0).\n        :param x_start: the [N x C x ...] tensor of noiseless inputs.\n        :param t: the number of diffusion steps (minus 1). Here, 0 means one step.\n        :return: A tuple (mean, variance, log_variance), all of x_start\'s shape.\n        """\n        mean = (extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start)\n        variance = extract_into_tensor(1.0 - self.alphas_cumprod, t, x_start.shape)\n        log_variance = extract_into_tensor(self.log_one_minus_alphas_cumprod, t, x_start.shape)\n        return mean, variance, log_variance\n\n    def predict_start_from_noise(self, x_t, t, noise):\n        return (\n                extract_into_tensor(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -\n                extract_into_tensor(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise\n        )\n\n    def q_posterior(self, x_start, x_t, t):\n        posterior_mean = (\n                extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start +\n                extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t\n        )\n        posterior_variance = extract_into_tensor(self.posterior_variance, t, x_t.shape)\n        posterior_log_variance_clipped = extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)\n        return posterior_mean, posterior_variance, posterior_log_variance_clipped\n\n    def p_mean_variance(self, x, t, clip_denoised: bool):\n        model_out = self.model(x, t)\n        if self.parameterization == "eps":\n            x_recon = self.predict_start_from_noise(x, t=t, noise=model_out)\n        elif self.parameterization == "x0":\n            x_recon = model_out\n        if clip_denoised:\n            x_recon.clamp_(-1., 1.)\n\n        model_mean, posterior_variance, posterior_log_variance = self.q_posterior(x_start=x_recon, x_t=x, t=t)\n        return model_mean, posterior_variance, posterior_log_variance\n\n    @torch.no_grad()\n    def p_sample(self, x, t, clip_denoised=True, repeat_noise=False):\n        b, *_, device = *x.shape, x.device\n        model_mean, _, model_log_variance = self.p_mean_variance(x=x, t=t, clip_denoised=clip_denoised)\n        noise = noise_like(x.shape, device, repeat_noise)\n        # no noise when t == 0\n        nonzero_mask = (1 - (t == 0).float()).reshape(b, *((1,) * (len(x.shape) - 1)))\n        return model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise\n\n    @torch.no_grad()\n    def p_sample_loop(self, shape, return_intermediates=False):\n        device = self.betas.device\n        b = shape[0]\n        img = torch.randn(shape, device=device)\n        intermediates = [img]\n        for i in tqdm(reversed(range(0, self.num_timesteps)), desc=\'Sampling t\', total=self.num_timesteps):\n            img = self.p_sample(img, torch.full((b,), i, device=device, dtype=torch.long),\n                                clip_denoised=self.clip_denoised)\n            if i % self.log_every_t == 0 or i == self.num_timesteps - 1:\n                intermediates.append(img)\n        if return_intermediates:\n            return img, intermediates\n        return img\n\n    @torch.no_grad()\n    def sample(self, batch_size=16, return_intermediates=False):\n        image_size = self.image_size\n        channels = self.channels\n        return self.p_sample_loop((batch_size, channels, image_size[0], image_size[1]),\n                                  return_intermediates=return_intermediates)\n\n    def q_sample(self, x_start, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        return (extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start +\n                extract_into_tensor(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise)\n\n    def get_loss(self, pred, target, mean=True):\n        if self.loss_type == \'l1\':\n            loss = (target - pred).abs()\n            if mean:\n                loss = loss.mean()\n        elif self.loss_type == \'l2\':\n            if mean:\n                loss = torch.nn.functional.mse_loss(target, pred)\n            else:\n                loss = torch.nn.functional.mse_loss(target, pred, reduction=\'none\')\n        else:\n            raise NotImplementedError("unknown loss type \'{loss_type}\'")\n\n        return loss\n\n    def p_losses(self, x_start, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)\n        model_out = self.model(x_noisy, t)\n\n        loss_dict = {}\n        if self.parameterization == "eps":\n            target = noise\n        elif self.parameterization == "x0":\n            target = x_start\n        else:\n            raise NotImplementedError(f"Paramterization {self.parameterization} not yet supported")\n\n        loss = self.get_loss(model_out, target, mean=False).mean(dim=[1, 2, 3])\n\n        log_prefix = \'train\' if self.training else \'val\'\n\n        loss_dict.update({f\'{log_prefix}/loss_simple\': loss.mean()})\n        loss_simple = loss.mean() * self.l_simple_weight\n\n        loss_vlb = (self.lvlb_weights[t] * loss).mean()\n        loss_dict.update({f\'{log_prefix}/loss_vlb\': loss_vlb})\n\n        loss = loss_simple + self.original_elbo_weight * loss_vlb\n\n        loss_dict.update({f\'{log_prefix}/loss\': loss})\n\n        return loss, loss_dict\n\n    def forward(self, x, *args, **kwargs):\n        # b, c, h, w, device, img_size, = *x.shape, x.device, self.image_size\n        # assert h == img_size and w == img_size, f\'height and width of image must be {img_size}\'\n        t = torch.randint(0, self.num_timesteps, (x.shape[0],), device=self.device).long()\n        return self.p_losses(x, t, *args, **kwargs)\n\n    def get_input(self, batch, k):\n        x = batch[k]\n        if len(x.shape) == 3:\n            x = x[..., None]\n        x = rearrange(x, \'b h w c -> b c h w\')\n        x = x.to(memory_format=torch.contiguous_format).float()\n        return x\n\n    def get_input_concat_multi_frame_input(self, batch, k_list):\n        frame_num = len(k_list)\n        output_list = []\n        for i in range(frame_num):\n            x = batch[k_list[i]]\n            if len(x.shape) == 3:\n                x = x[..., None]\n            x = rearrange(x, \'b h w c -> b c h w\')\n            x = x.to(memory_format=torch.contiguous_format).float()\n            output_list.append(x)\n        output_concat_tensor = torch.cat(output_list, dim=1)\n        return output_concat_tensor\n\n    def get_input_concat_and_cross_attn_dict_input(self, batch, k_list):\n        xc_dict = {}\n        for cond_key in k_list:\n            if cond_key == \'lq_image\':\n                c = batch[cond_key]\n                c = rearrange(c, \'b h w c -> b c h w\')\n                c = c.to(memory_format=torch.contiguous_format).float()\n            elif cond_key == \'text_label\':\n                c = batch[cond_key]\n            else:\n                c = batch[cond_key]\n            xc_dict[cond_key] = c.to(self.device) if not isinstance(c, list) else c\n        return xc_dict\n\n\n    def shared_step(self, batch):\n        x = self.get_input(batch, self.first_stage_key)\n        loss, loss_dict = self(x)\n        return loss, loss_dict\n\n    def training_step(self, batch, batch_idx):\n        loss, loss_dict = self.shared_step(batch)\n\n        self.log_dict(loss_dict, prog_bar=True,\n                      logger=True, on_step=True, on_epoch=True)\n\n        self.log("global_step", self.global_step,\n                 prog_bar=True, logger=True, on_step=True, on_epoch=False)\n\n        if self.use_scheduler:\n            lr = self.optimizers().param_groups[0][\'lr\']\n            self.log(\'lr_abs\', lr, prog_bar=True, logger=True, on_step=True, on_epoch=False)\n\n        return loss\n\n    @torch.no_grad()\n    def validation_step(self, batch, batch_idx):\n        _, loss_dict_no_ema = self.shared_step(batch)\n        with self.ema_scope():\n            _, loss_dict_ema = self.shared_step(batch)\n            loss_dict_ema = {key + \'_ema\': loss_dict_ema[key] for key in loss_dict_ema}\n        self.log_dict(loss_dict_no_ema, prog_bar=False, logger=True, on_step=False, on_epoch=True)\n        self.log_dict(loss_dict_ema, prog_bar=False, logger=True, on_step=False, on_epoch=True)\n\n    def on_train_batch_end(self, *args, **kwargs):\n        if self.use_ema:\n            self.model_ema(self.model)\n\n    def _get_rows_from_list(self, samples):\n        n_imgs_per_row = len(samples)\n        denoise_grid = rearrange(samples, \'n b c h w -> b n c h w\')\n        denoise_grid = rearrange(denoise_grid, \'b n c h w -> (b n) c h w\')\n        denoise_grid = make_grid(denoise_grid, nrow=n_imgs_per_row)\n        return denoise_grid\n\n    @torch.no_grad()\n    def log_images(self, batch, N=8, n_row=2, sample=True, return_keys=None, **kwargs):\n        log = dict()\n        x = self.get_input(batch, self.first_stage_key)\n        N = min(x.shape[0], N)\n        n_row = min(x.shape[0], n_row)\n        x = x.to(self.device)[:N]\n        log["inputs"] = x\n\n        # get diffusion row\n        diffusion_row = list()\n        x_start = x[:n_row]\n\n        for t in range(self.num_timesteps):\n            if t % self.log_every_t == 0 or t == self.num_timesteps - 1:\n                t = repeat(torch.tensor([t]), \'1 -> b\', b=n_row)\n                t = t.to(self.device).long()\n                noise = torch.randn_like(x_start)\n                x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)\n                diffusion_row.append(x_noisy)\n\n        log["diffusion_row"] = self._get_rows_from_list(diffusion_row)\n\n        if sample:\n            # get denoise row\n            with self.ema_scope("Plotting"):\n                samples, denoise_row = self.sample(batch_size=N, return_intermediates=True)\n\n            log["samples"] = samples\n            log["denoise_row"] = self._get_rows_from_list(denoise_row)\n\n        if return_keys:\n            if np.intersect1d(list(log.keys()), return_keys).shape[0] == 0:\n                return log\n            else:\n                return {key: log[key] for key in return_keys}\n        return log\n\n    def configure_optimizers(self):\n        lr = self.learning_rate\n        params = list(self.model.parameters())\n        opt = torch.optim.AdamW(params, lr=lr)\n        return opt\n\n\nclass IDM_Diffusion(DDPM):\n    """main class"""\n    def __init__(self,\n                 first_stage_config,\n                 cond_stage_config,\n                 cond_stage_key="image",\n                 cond_stage_trainable=False,\n                 concat_mode=True,\n                 cond_stage_forward=None,\n                 conditioning_key=None,\n                 scale_factor=1.0,\n                 pixel_loss_type=None, \n                 pixel_loss_config=None,\n                 *args, **kwargs):\n        \n        self.conditioning_key = conditioning_key\n\n        ckpt_path = kwargs.pop("ckpt_path", None)\n        ignore_keys = kwargs.pop("ignore_keys", [])\n        super().__init__(conditioning_key=conditioning_key, *args, **kwargs)\n        self.concat_mode = concat_mode\n        self.cond_stage_trainable = cond_stage_trainable\n        self.cond_stage_key = cond_stage_key\n        \n        try:\n            self.num_downs = len(first_stage_config.params.ddconfig.ch_mult) - 1\n        except:\n            self.num_downs = 0\n        self.scale_factor = scale_factor\n\n        self.instantiate_first_stage(first_stage_config)\n        if conditioning_key == \'concat_and_crossattn\':\n            self.instantiate_cond_stage(cond_stage_config)\n        else:\n            print("Haven\'t supported other cond stage types.")\n\n        self.cond_stage_forward = cond_stage_forward\n        self.clip_denoised = False\n        self.bbox_tokenizer = None  \n\n        self.restarted_from_ckpt = False\n        if ckpt_path is not None:\n            self.init_from_ckpt(ckpt_path, ignore_keys)\n            self.restarted_from_ckpt = True\n        \n        self.pixel_loss_type = pixel_loss_type\n        if self.pixel_loss_type != None:\n            self.pixel_loss = instantiate_from_config(pixel_loss_config)\n\n    def register_schedule(self,\n                          given_betas=None, beta_schedule="linear", timesteps=1000,\n                          linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):\n        super().register_schedule(given_betas, beta_schedule, timesteps, linear_start, linear_end, cosine_s)\n\n\n    def instantiate_first_stage(self, config):\n        model = instantiate_from_config(config)\n        self.first_stage_model = model.eval()\n        self.first_stage_model.train = disabled_train\n        for param in self.first_stage_model.parameters():\n            param.requires_grad = False\n\n    def instantiate_cond_stage(self, config):\n        self.cond_stage_model = None\n        self.hybrid_concat_cond_stage_model = self.first_stage_model\n        self.hybrid_cross_attn_cond_stage_model = instantiate_from_config(config)\n\n\n    def _get_denoise_row_from_list(self, samples, desc=\'\', force_no_decoder_quantization=False):\n        denoise_row = []\n        for zd in tqdm(samples, desc=desc):\n            denoise_row.append(self.decode_first_stage(zd.to(self.device),\n                                                            force_not_quantize=force_no_decoder_quantization))\n        n_imgs_per_row = len(denoise_row)\n        denoise_row = torch.stack(denoise_row)  # n_log_step, n_row, C, H, W\n        denoise_grid = rearrange(denoise_row, \'n b c h w -> b n c h w\')\n        denoise_grid = rearrange(denoise_grid, \'b n c h w -> (b n) c h w\')\n        denoise_grid = make_grid(denoise_grid, nrow=n_imgs_per_row)\n        return denoise_grid\n\n    def get_first_stage_encoding(self, encoder_posterior):\n        if isinstance(encoder_posterior, DiagonalGaussianDistribution):\n            z = encoder_posterior.sample()\n        elif isinstance(encoder_posterior, torch.Tensor):\n            z = encoder_posterior\n        else:\n            raise NotImplementedError(f"encoder_posterior of type \'{type(encoder_posterior)}\' not yet implemented")\n        return self.scale_factor * z\n\n    def get_first_stage_encoding_aggregation(self, encoder_posterior, weighting, normalization, image_splist_list, x_shape):\n        z_decoded_list = []\n        for index, encoder_i in enumerate(encoder_posterior):\n            if isinstance(encoder_i, DiagonalGaussianDistribution):\n                z = encoder_i.sample()\n            elif isinstance(encoder_i, torch.Tensor):\n                z = encoder_i\n            else:\n                raise NotImplementedError(f"encoder_posterior of type \'{type(encoder_i)}\' not yet implemented")\n            z_decoded_list.append(z)\n        o = torch.stack(z_decoded_list, axis=0)  # # (bn, nc, ks[0], ks[1], L)\n        weighting = weighting.permute(4,0,1,2,3)\n\n        o = o * weighting\n        # Reverse 1. reshape to img shape\n        encoded = torch.zeros((x_shape[0], 3, x_shape[2]//4, x_shape[3]//4)).to(self.device)   \n        for index, wi in enumerate(image_splist_list):\n            encoded[:, :, :, wi[0]:wi[1]] += o[index,:,:,:,:]\n        encoded = encoded / normalization\n        return self.scale_factor * encoded\n\n    def get_learned_conditioning(self, c):\n        if self.cond_stage_forward is None:\n            if hasattr(self.cond_stage_model, \'encode\') and callable(self.cond_stage_model.encode):\n                if self.cond_stage_key == [\'masked_image\', \'text_image\']:\n                    c_list = []\n                    for index in range(len(self.cond_stage_key)):\n                        c_temp = self.cond_stage_model.encode(c[:, 3*index:3*index+3, :, :])\n                        if isinstance(c_temp, DiagonalGaussianDistribution):\n                            c_temp = c_temp.mode()\n                        c_list.append(c_temp)\n                    c = torch.cat(c_list, dim=1)\n                    assert c.size()[1] == 3*len(self.cond_stage_key), \'please confirm the condition encodeed shape\'\n                else:\n                    c = self.cond_stage_model.encode(c)\n                    if isinstance(c, DiagonalGaussianDistribution):\n                        c = c.mode()\n            elif self.conditioning_key == \'concat_and_crossattn\':\n                if set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key):\n                    c_lq_image = self.hybrid_concat_cond_stage_model.encode(c[\'lq_image\'].to(self.device))\n                    if isinstance(c_lq_image, DiagonalGaussianDistribution):\n                        c_lq_image = c_lq_image.mode()\n                    c[\'lq_image\'] = c_lq_image\n            else:\n                c = self.cond_stage_model(c)\n        else:    # 如果要使 cond_stage_model 是可学习的, 则在这里设置\n            assert hasattr(self.cond_stage_model, self.cond_stage_forward)\n            c = getattr(self.cond_stage_model, self.cond_stage_forward)(c)\n        return c\n\n    def get_text_learned_conditioning_hybrid(self, c):\n        if set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key):\n            if len(c[\'text_label\'].size())==2:\n                c_text_label = self.hybrid_cross_attn_cond_stage_model.encode(c[\'text_label\'].to(self.device))\n                c[\'text_label\'] = c_text_label\n            elif len(c[\'text_label\'].size())==3:\n                c_text_label_list = []\n                for text_str in c[\'text_label\']:\n                    c_text_label = self.hybrid_cross_attn_cond_stage_model.encode(text_str.to(self.device))\n                    c_text_label_list.append(c_text_label)\n                c_text_label_list = torch.stack(c_text_label_list, dim=0)\n                c[\'text_label\'] = c_text_label_list\n            elif len(c[\'text_label\'].size())==4:\n                c_text_label = self.hybrid_cross_attn_cond_stage_model.encode(c[\'text_label\'].to(self.device))\n                c[\'text_label\'] = c_text_label\n        return c\n\n\n    def meshgrid(self, h, w):\n        y = torch.arange(0, h).view(h, 1, 1).repeat(1, w, 1)\n        x = torch.arange(0, w).view(1, w, 1).repeat(h, 1, 1)\n\n        arr = torch.cat([y, x], dim=-1)\n        return arr\n\n    def delta_border(self, h, w):\n        """\n        :param h: height\n        :param w: width\n        :return: normalized distance to image border,\n         wtith min distance = 0 at border and max dist = 0.5 at image center\n        """\n        lower_right_corner = torch.tensor([h - 1, w - 1]).view(1, 1, 2)\n        arr = self.meshgrid(h, w) / lower_right_corner\n        dist_left_up = torch.min(arr, dim=-1, keepdims=True)[0]\n        dist_right_down = torch.min(1 - arr, dim=-1, keepdims=True)[0]\n        edge_dist = torch.min(torch.cat([dist_left_up, dist_right_down], dim=-1), dim=-1)[0]\n        return edge_dist\n\n    def get_weighting(self, h, w, Ly, Lx, device):\n        weighting = self.delta_border(h, w)\n        weighting = torch.clip(weighting, self.split_input_params["clip_min_weight"],\n                               self.split_input_params["clip_max_weight"], )\n        weighting = weighting.view(1, h * w, 1).repeat(1, 1, Ly * Lx).to(device)\n\n        if self.split_input_params["tie_braker"]:\n            L_weighting = self.delta_border(Ly, Lx)\n            L_weighting = torch.clip(L_weighting,\n                                     self.split_input_params["clip_min_tie_weight"],\n                                     self.split_input_params["clip_max_tie_weight"])\n\n            L_weighting = L_weighting.view(1, 1, Ly * Lx).to(device)\n            weighting = weighting * L_weighting\n        return weighting\n\n    def get_weighting_aggregate(self, h, w, Ly, Lx, device):\n        split_input_params = {"tie_braker": False, "clip_max_weight": 0.5, "clip_min_weight": 0.01, "clip_max_tie_weight": 0.5, "clip_min_tie_weight": 0.01}\n        weighting = self.delta_border(h, w)\n        weighting = torch.clip(weighting, split_input_params["clip_min_weight"],\n                                          split_input_params["clip_max_weight"], )\n        weighting = weighting.view(1, h * w, 1).repeat(1, 1, Ly * Lx).to(device)\n\n        if split_input_params["tie_braker"]:\n            L_weighting = self.delta_border(Ly, Lx)\n            L_weighting = torch.clip(L_weighting,\n                                     split_input_params["clip_min_tie_weight"],\n                                     split_input_params["clip_max_tie_weight"])\n\n            L_weighting = L_weighting.view(1, 1, Ly * Lx).to(device)\n            weighting = weighting * L_weighting\n        return weighting\n\n    def get_fold_unfold(self, x, kernel_size, stride, uf=1, df=1):  # todo load once not every time, shorten code\n        """\n        :param x: img of size (bs, c, h, w)\n        :return: n img crops of size (n, bs, c, kernel_size[0], kernel_size[1])\n        """\n        bs, nc, h, w = x.shape\n\n        # number of crops in image\n        Ly = (h - kernel_size[0]) // stride[0] + 1\n        Lx = (w - kernel_size[1]) // stride[1] + 1\n\n        if uf == 1 and df == 1:\n            fold_params = dict(kernel_size=kernel_size, dilation=1, padding=0, stride=stride)\n            unfold = torch.nn.Unfold(**fold_params)\n\n            fold = torch.nn.Fold(output_size=x.shape[2:], **fold_params)\n\n            weighting = self.get_weighting(kernel_size[0], kernel_size[1], Ly, Lx, x.device).to(x.dtype)\n            normalization = fold(weighting).view(1, 1, h, w)  # normalizes the overlap\n            weighting = weighting.view((1, 1, kernel_size[0], kernel_size[1], Ly * Lx))\n\n        elif uf > 1 and df == 1:\n            fold_params = dict(kernel_size=kernel_size, dilation=1, padding=0, stride=stride)\n            unfold = torch.nn.Unfold(**fold_params)\n\n            fold_params2 = dict(kernel_size=(kernel_size[0] * uf, kernel_size[0] * uf),\n                                dilation=1, padding=0,\n                                stride=(stride[0] * uf, stride[1] * uf))\n            fold = torch.nn.Fold(output_size=(x.shape[2] * uf, x.shape[3] * uf), **fold_params2)\n\n            weighting = self.get_weighting(kernel_size[0] * uf, kernel_size[1] * uf, Ly, Lx, x.device).to(x.dtype)\n            normalization = fold(weighting).view(1, 1, h * uf, w * uf)  # normalizes the overlap\n            weighting = weighting.view((1, 1, kernel_size[0] * uf, kernel_size[1] * uf, Ly * Lx))\n\n        elif df > 1 and uf == 1:\n            fold_params = dict(kernel_size=kernel_size, dilation=1, padding=0, stride=stride)\n            unfold = torch.nn.Unfold(**fold_params)\n\n            fold_params2 = dict(kernel_size=(kernel_size[0] // df, kernel_size[0] // df),\n                                dilation=1, padding=0,\n                                stride=(stride[0] // df, stride[1] // df))\n            fold = torch.nn.Fold(output_size=(x.shape[2] // df, x.shape[3] // df), **fold_params2)\n\n            weighting = self.get_weighting(kernel_size[0] // df, kernel_size[1] // df, Ly, Lx, x.device).to(x.dtype)\n            normalization = fold(weighting).view(1, 1, h // df, w // df)  # normalizes the overlap\n            weighting = weighting.view((1, 1, kernel_size[0] // df, kernel_size[1] // df, Ly * Lx))\n\n        else:\n            raise NotImplementedError\n\n        return fold, unfold, normalization, weighting\n\n    @torch.no_grad()\n    def get_input(self, batch, k, return_first_stage_outputs=False, force_c_encode=False,\n                  cond_key=None, return_original_cond=False, bs=None, image_splist_list=None):\n        x = super().get_input(batch, k)\n        if bs is not None:\n            x = x[:bs]\n        x = x.to(self.device)\n        if image_splist_list == None:\n            encoder_posterior = self.encode_first_stage(x)\n            z = self.get_first_stage_encoding(encoder_posterior).detach()\n        else:\n            encoder_posterior, weighting, normalization = self.encode_first_stage_aggregation(x, image_splist_list)\n            z = self.get_first_stage_encoding_aggregation(encoder_posterior, weighting, normalization, image_splist_list, x_shape=x.size()).detach()\n\n        if self.model.conditioning_key is not None:\n            if cond_key is None:\n                cond_key = self.cond_stage_key\n            if cond_key != self.first_stage_key:\n                if cond_key in [\'caption\', \'coordinates_bbox\']:\n                    xc = batch[cond_key]\n                elif cond_key == \'class_label\':\n                    xc = batch\n                elif cond_key == [\'masked_image\', \'text_image\']:\n                    xc = super().get_input_concat_multi_frame_input(batch, cond_key).to(self.device)\n                elif set([\'lq_image\', \'text_label\']).issubset(cond_key):\n                    xc = super().get_input_concat_and_cross_attn_dict_input(batch, cond_key)\n                else:\n                    xc = super().get_input(batch, cond_key).to(self.device)\n            else:\n                xc = x\n            if not self.cond_stage_trainable or force_c_encode or self.conditioning_key == \'concat_and_crossattn\':\n                if isinstance(xc, dict) or isinstance(xc, list):\n                    # import pudb; pudb.set_trace()\n                    c = self.get_learned_conditioning(xc)\n                else:\n                    c = self.get_learned_conditioning(xc.to(self.device))\n            else:\n                c = xc\n            if bs is not None:\n                if self.conditioning_key == \'concat_and_crossattn\' and set([\'lq_image\', \'text_label\']).issubset(cond_key):\n                    cond_concat = c[\'lq_image\']\n                    cond_crossattn = c[\'text_label\']\n                    c[\'lq_image\'] = cond_concat[:bs]\n                    c[\'text_label\'] = cond_crossattn[:bs]\n                else:\n                    c = c[:bs]\n        else:\n            c = None\n            xc = None\n\n        out = [z, c]\n        if return_first_stage_outputs:\n            xrec = self.decode_first_stage(z)\n            out.extend([x, xrec])\n        if return_original_cond:\n            out.append(xc)\n        return out\n\n    @torch.no_grad()\n    def decode_first_stage(self, z, predict_cids=False, force_not_quantize=False):\n        if predict_cids:\n            if z.dim() == 4:\n                z = torch.argmax(z.exp(), dim=1).long()\n            z = self.first_stage_model.quantize.get_codebook_entry(z, shape=None)\n            z = rearrange(z, \'b h w c -> b c h w\').contiguous()\n\n        z = 1. / self.scale_factor * z\n\n        if hasattr(self, "split_input_params"):\n            if self.split_input_params["patch_distributed_vq"]:\n                ks = self.split_input_params["ks"]  # eg. (128, 128)\n                stride = self.split_input_params["stride"]  # eg. (64, 64)\n                uf = self.split_input_params["vqf"]\n                bs, nc, h, w = z.shape\n                if ks[0] > h or ks[1] > w:\n                    ks = (min(ks[0], h), min(ks[1], w))\n                    print("reducing Kernel")\n\n                if stride[0] > h or stride[1] > w:\n                    stride = (min(stride[0], h), min(stride[1], w))\n                    print("reducing stride")\n\n                fold, unfold, normalization, weighting = self.get_fold_unfold(z, ks, stride, uf=uf)\n\n                z = unfold(z)  # (bn, nc * prod(**ks), L)\n                # 1. Reshape to img shape\n                z = z.view((z.shape[0], -1, ks[0], ks[1], z.shape[-1]))  # (bn, nc, ks[0], ks[1], L )\n\n                # 2. apply model loop over last dim\n                output_list = [self.first_stage_model.decode(z[:, :, :, :, i])\n                                for i in range(z.shape[-1])]\n\n                o = torch.stack(output_list, axis=-1)  # # (bn, nc, ks[0], ks[1], L)\n                o = o * weighting\n                # Reverse 1. reshape to img shape\n                o = o.view((o.shape[0], -1, o.shape[-1]))  # (bn, nc * ks[0] * ks[1], L)\n                # stitch crops together\n                decoded = fold(o)\n                decoded = decoded / normalization  # norm is shape (1, 1, h, w)\n                return decoded\n            else:\n                return self.first_stage_model.decode(z)\n\n        else:\n            return self.first_stage_model.decode(z)\n\n    @torch.no_grad()\n    def decode_first_stage_aggregation(self, z, image_splist_list=[[0, 512], [256, 512+256]]):\n        z = 1. / self.scale_factor * z\n\n        Ly, Lx = 1, len(image_splist_list)\n        patch_size = (128, 512)\n        weighting = self.get_weighting_aggregate(patch_size[0], patch_size[1], Ly, Lx, z.device).to(z.dtype)\n        weighting = weighting.view((1, 1, patch_size[0], patch_size[1], Ly * Lx)).to(self.device)\n        normalization = torch.zeros((z.size()[0], 1, z.size()[2]*4, z.size()[3]*4)).to(self.device)     # normalizes the overlap\n\n        for index, wi in enumerate(image_splist_list):\n            normalization[:, :, :, wi[0]*4:wi[1]*4] += weighting[:,:,:,:,index]\n\n        # split z\n        z_list = [z[:, :, :, wi[0]:wi[1]] for wi in image_splist_list]\n\n        # 2. apply model loop over last dim\n        output_list = [self.first_stage_model.decode(z_list[i])\n                        for i in range(len(image_splist_list))]\n\n        o = torch.stack(output_list, axis=-1)  # # (bn, nc, ks[0], ks[1], L)\n        o = o * weighting\n        # Reverse 1. reshape to img shape\n        decoded = torch.zeros((z.size()[0], 3, z.size()[2]*4, z.size()[3]*4)).to(self.device)   \n        for index, wi in enumerate(image_splist_list):\n            decoded[:, :, :, wi[0]*4:wi[1]*4] += o[:,:,:,:,index]\n        decoded = decoded / normalization\n        return decoded\n\n\n    # same as above but without decorator\n    def differentiable_decode_first_stage(self, z, predict_cids=False, force_not_quantize=False):\n        if predict_cids:\n            if z.dim() == 4:\n                z = torch.argmax(z.exp(), dim=1).long()\n            z = self.first_stage_model.quantize.get_codebook_entry(z, shape=None)\n            z = rearrange(z, \'b h w c -> b c h w\').contiguous()\n\n        z = 1. / self.scale_factor * z\n\n        if hasattr(self, "split_input_params"):\n            if self.split_input_params["patch_distributed_vq"]:\n                ks = self.split_input_params["ks"]  # eg. (128, 128)\n                stride = self.split_input_params["stride"]  # eg. (64, 64)\n                uf = self.split_input_params["vqf"]\n                bs, nc, h, w = z.shape\n                if ks[0] > h or ks[1] > w:\n                    ks = (min(ks[0], h), min(ks[1], w))\n                    print("reducing Kernel")\n\n                if stride[0] > h or stride[1] > w:\n                    stride = (min(stride[0], h), min(stride[1], w))\n                    print("reducing stride")\n\n                fold, unfold, normalization, weighting = self.get_fold_unfold(z, ks, stride, uf=uf)\n\n                z = unfold(z)  # (bn, nc * prod(**ks), L)\n                # 1. Reshape to img shape\n                z = z.view((z.shape[0], -1, ks[0], ks[1], z.shape[-1]))  # (bn, nc, ks[0], ks[1], L )\n\n                # 2. apply model loop over last dim\n                output_list = [self.first_stage_model.decode(z[:, :, :, :, i])\n                                for i in range(z.shape[-1])]\n\n                o = torch.stack(output_list, axis=-1)  # # (bn, nc, ks[0], ks[1], L)\n                o = o * weighting\n                # Reverse 1. reshape to img shape\n                o = o.view((o.shape[0], -1, o.shape[-1]))  # (bn, nc * ks[0] * ks[1], L)\n                # stitch crops together\n                decoded = fold(o)\n                decoded = decoded / normalization  # norm is shape (1, 1, h, w)\n                return decoded\n            else:\n                return self.first_stage_model.decode(z)\n\n        else:\n            return self.first_stage_model.decode(z)\n\n    @torch.no_grad()\n    def encode_first_stage(self, x, ):\n        if hasattr(self, "split_input_params"):\n            if self.split_input_params["patch_distributed_vq"]:\n                ks = self.split_input_params["ks"]  # eg. (128, 128)\n                stride = self.split_input_params["stride"]  # eg. (64, 64)\n                df = self.split_input_params["vqf"]\n                self.split_input_params[\'original_image_size\'] = x.shape[-2:]\n                bs, nc, h, w = x.shape\n                if ks[0] > h or ks[1] > w:\n                    ks = (min(ks[0], h), min(ks[1], w))\n                    print("reducing Kernel")\n\n                if stride[0] > h or stride[1] > w:\n                    stride = (min(stride[0], h), min(stride[1], w))\n                    print("reducing stride")\n\n                fold, unfold, normalization, weighting = self.get_fold_unfold(x, ks, stride, df=df)\n                z = unfold(x)  # (bn, nc * prod(**ks), L)\n                # Reshape to img shape\n                z = z.view((z.shape[0], -1, ks[0], ks[1], z.shape[-1]))  # (bn, nc, ks[0], ks[1], L )\n\n                output_list = [self.first_stage_model.encode(z[:, :, :, :, i])\n                               for i in range(z.shape[-1])]\n\n                o = torch.stack(output_list, axis=-1)\n                o = o * weighting\n\n                # Reverse reshape to img shape\n                o = o.view((o.shape[0], -1, o.shape[-1]))  # (bn, nc * ks[0] * ks[1], L)\n                # stitch crops together\n                decoded = fold(o)\n                decoded = decoded / normalization\n                return decoded\n\n            else:\n                return self.first_stage_model.encode(x)\n        else:\n            return self.first_stage_model.encode(x)\n\n    @torch.no_grad()\n    def encode_first_stage_aggregation(self, x, image_splist_list=[[0, 512//4], [256//4, 512//4+256//4]]):\n        bs, nc, h, w = x.shape\n\n        Ly, Lx = 1, len(image_splist_list)\n        patch_size = (128//4, 512//4)\n        weighting = self.get_weighting_aggregate(patch_size[0], patch_size[1], Ly, Lx, x.device).to(x.dtype)\n        weighting = weighting.view((1, 1, patch_size[0], patch_size[1], Ly * Lx)).to(self.device)\n        normalization = torch.zeros((x.size()[0], 1, x.size()[2]//4, x.size()[3]//4)).to(self.device)     # normalizes the overlap\n\n        for index, wi in enumerate(image_splist_list):\n            normalization[:, :, :, wi[0]:wi[1]] += weighting[:,:,:,:,index]\n\n        # split x\n        x_list = [x[:, :, :, wi[0]*4:wi[1]*4] for wi in image_splist_list]\n\n        output_list = [self.first_stage_model.encode(x_list[i])\n                        for i in range(len(image_splist_list))]\n\n        encoded_list = output_list\n        return encoded_list, weighting, normalization\n\n\n    def shared_step(self, batch, **kwargs):\n        x, c = self.get_input(batch, self.first_stage_key, cond_key=self.cond_stage_key)\n        loss = self(x, c)\n        return loss\n\n\n    def shared_step_timestep(self, batch, t, *args, **kwargs):\n        x, c = self.get_input(batch, self.first_stage_key, cond_key=self.cond_stage_key)\n        if self.model.conditioning_key is not None:\n            assert c is not None\n            if self.cond_stage_trainable and self.conditioning_key != \'concat_and_crossattn\':\n                c = self.get_learned_conditioning(c)\n            elif self.cond_stage_trainable and self.conditioning_key == \'concat_and_crossattn\':\n                c = self.get_text_learned_conditioning_hybrid(c)\n        return self.p_losses(x, c, t, *args, **kwargs)\n\n\n    def forward(self, x, c, *args, **kwargs):\n        t = torch.randint(0, self.num_timesteps, (x.shape[0],), device=self.device).long()\n        if self.model.conditioning_key is not None:\n            assert c is not None\n            if self.cond_stage_trainable and self.conditioning_key != \'concat_and_crossattn\':\n                c = self.get_learned_conditioning(c)\n            elif self.cond_stage_trainable and self.conditioning_key == \'concat_and_crossattn\':\n                c = self.get_text_learned_conditioning_hybrid(c)\n        return self.p_losses(x, c, t, *args, **kwargs)\n\n    def _rescale_annotations(self, bboxes, crop_coordinates):  # TODO: move to dataset\n        def rescale_bbox(bbox):\n            x0 = clamp((bbox[0] - crop_coordinates[0]) / crop_coordinates[2])\n            y0 = clamp((bbox[1] - crop_coordinates[1]) / crop_coordinates[3])\n            w = min(bbox[2] / crop_coordinates[2], 1 - x0)\n            h = min(bbox[3] / crop_coordinates[3], 1 - y0)\n            return x0, y0, w, h\n\n        return [rescale_bbox(b) for b in bboxes]\n\n    def apply_model(self, x_noisy, t, cond, return_ids=False):\n\n        if isinstance(cond, dict) and not set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key):\n            print(\'hybrid case, cond is exptected to be a dict\')\n            pass\n        elif self.conditioning_key == \'concat_and_crossattn\' and set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key):\n            cond_concat = cond[\'lq_image\']\n            cond_crossattn = cond[\'text_label\']\n            cond = {"c_concat": [cond_concat], \'c_crossattn\': [cond_crossattn]}\n        else:\n            if not isinstance(cond, list):\n                cond = [cond]\n            key = \'c_concat\' if self.model.conditioning_key == \'concat\' else \'c_crossattn\'\n            cond = {key: cond}\n\n        if hasattr(self, "split_input_params"):\n            assert len(cond) == 1  # todo can only deal with one conditioning atm\n            assert not return_ids  \n            ks = self.split_input_params["ks"]  # eg. (128, 128)\n            stride = self.split_input_params["stride"]  # eg. (64, 64)\n\n            h, w = x_noisy.shape[-2:]\n\n            fold, unfold, normalization, weighting = self.get_fold_unfold(x_noisy, ks, stride)\n\n            z = unfold(x_noisy)  # (bn, nc * prod(**ks), L)\n            # Reshape to img shape\n            z = z.view((z.shape[0], -1, ks[0], ks[1], z.shape[-1]))  # (bn, nc, ks[0], ks[1], L )\n            z_list = [z[:, :, :, :, i] for i in range(z.shape[-1])]\n\n            if self.cond_stage_key in ["image", "LR_image", "segmentation",\n                                       \'bbox_img\'] and self.model.conditioning_key:  # todo check for completeness\n                c_key = next(iter(cond.keys()))  # get key\n                c = next(iter(cond.values()))  # get value\n                assert (len(c) == 1)  # todo extend to list with more than one elem\n                c = c[0]  # get element\n\n                c = unfold(c)\n                c = c.view((c.shape[0], -1, ks[0], ks[1], c.shape[-1]))  # (bn, nc, ks[0], ks[1], L )\n\n                cond_list = [{c_key: [c[:, :, :, :, i]]} for i in range(c.shape[-1])]\n\n            elif self.cond_stage_key == \'coordinates_bbox\':\n                assert \'original_image_size\' in self.split_input_params, \'BoudingBoxRescaling is missing original_image_size\'\n\n                # assuming padding of unfold is always 0 and its dilation is always 1\n                n_patches_per_row = int((w - ks[0]) / stride[0] + 1)\n                full_img_h, full_img_w = self.split_input_params[\'original_image_size\']\n                # as we are operating on latents, we need the factor from the original image size to the\n                # spatial latent size to properly rescale the crops for regenerating the bbox annotations\n                num_downs = self.first_stage_model.encoder.num_resolutions - 1\n                rescale_latent = 2 ** (num_downs)\n\n                # get top left postions of patches as conforming for the bbbox tokenizer, therefore we\n                # need to rescale the tl patch coordinates to be in between (0,1)\n                tl_patch_coordinates = [(rescale_latent * stride[0] * (patch_nr % n_patches_per_row) / full_img_w,\n                                         rescale_latent * stride[1] * (patch_nr // n_patches_per_row) / full_img_h)\n                                        for patch_nr in range(z.shape[-1])]\n\n                # patch_limits are tl_coord, width and height coordinates as (x_tl, y_tl, h, w)\n                patch_limits = [(x_tl, y_tl,\n                                 rescale_latent * ks[0] / full_img_w,\n                                 rescale_latent * ks[1] / full_img_h) for x_tl, y_tl in tl_patch_coordinates]\n                # patch_values = [(np.arange(x_tl,min(x_tl+ks, 1.)),np.arange(y_tl,min(y_tl+ks, 1.))) for x_tl, y_tl in tl_patch_coordinates]\n\n                # tokenize crop coordinates for the bounding boxes of the respective patches\n                patch_limits_tknzd = [torch.LongTensor(self.bbox_tokenizer._crop_encoder(bbox))[None].to(self.device)\n                                      for bbox in patch_limits]  # list of length l with tensors of shape (1, 2)\n                # cut tknzd crop position from conditioning\n                assert isinstance(cond, dict), \'cond must be dict to be fed into model\'\n                cut_cond = cond[\'c_crossattn\'][0][..., :-2].to(self.device)\n\n                adapted_cond = torch.stack([torch.cat([cut_cond, p], dim=1) for p in patch_limits_tknzd])\n                adapted_cond = rearrange(adapted_cond, \'l b n -> (l b) n\')\n                adapted_cond = self.get_learned_conditioning(adapted_cond)\n                adapted_cond = rearrange(adapted_cond, \'(l b) n d -> l b n d\', l=z.shape[-1])\n\n                cond_list = [{\'c_crossattn\': [e]} for e in adapted_cond]\n\n            else:\n                cond_list = [cond for i in range(z.shape[-1])]  # Todo make this more efficient\n\n            # apply model by loop over crops\n            output_list = [self.model(z_list[i], t, **cond_list[i]) for i in range(z.shape[-1])]\n            assert not isinstance(output_list[0],\n                                  tuple)  # todo cant deal with multiple model outputs check this never happens\n\n            o = torch.stack(output_list, axis=-1)\n            o = o * weighting\n            # Reverse reshape to img shape\n            o = o.view((o.shape[0], -1, o.shape[-1]))  # (bn, nc * ks[0] * ks[1], L)\n            # stitch crops together\n            x_recon = fold(o) / normalization\n\n        else:\n            x_recon = self.model(x_noisy, t, **cond)\n\n        if isinstance(x_recon, tuple) and not return_ids:\n            return x_recon[0]\n        else:\n            return x_recon\n\n\n    def apply_model_aggregation(self, x_noisy, t, cond, image_splist_list=[[0, 512], [256, 512+256]]):\n        assert self.conditioning_key == \'concat_and_crossattn\' and set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key)\n\n        cond_concat = cond[\'lq_image\']\n        cond_crossattn = cond[\'text_label\']\n        cond = {"c_concat": [cond_concat], \'c_crossattn\': [cond_crossattn]}\n        # if input image size: 256*768 ; and image_splist_list=[[0, 512], [256, 512+256]]\n        # lq_image: B*C*H*W   ; B(batch)=1 , C(channle)=3, H(height):256, W(width):512+256\n        # text_label: N*B*L*D ; N(patchs)=2, B(batch)=1, L(length_of_text)=24, D(dim_of_text)=160\n\n        h, w = x_noisy.shape[-2:]\n\n        Ly, Lx = 1, len(image_splist_list)\n        patch_size = (128//4, 512//4)\n        weighting = self.get_weighting_aggregate(patch_size[0], patch_size[1], Ly, Lx, x_noisy.device).to(x_noisy.dtype)\n        weighting = weighting.view((1, 1, patch_size[0], patch_size[1], Ly * Lx)).to(self.device)\n\n        normalization = torch.zeros((x_noisy.size()[0], 1, x_noisy.size()[2], x_noisy.size()[3])).to(self.device)     # normalizes the overlap\n\n        for index, wi in enumerate(image_splist_list):\n            normalization[:, :, :, wi[0]:wi[1]] += weighting[:,:,:,:,index]\n\n        # split z\n        z_list = [x_noisy[:, :, :, wi[0]:wi[1]] for wi in image_splist_list]\n\n        # split cond\n        cond_list = [{"c_concat": [cond_concat[:, :, :, wi[0]:wi[1]]],  "c_crossattn": [cond_crossattn[:, index, :, :]]} for index, wi in enumerate(image_splist_list)]\n\n        # apply model by loop over crops\n        output_list = [self.model(z_list[i], t, **cond_list[i]) for i in range(len(image_splist_list))]\n\n        o = torch.stack(output_list, axis=-1)\n        o = o * weighting\n        x_recon = torch.zeros(x_noisy.size()).to(self.device)   \n        for index, wi in enumerate(image_splist_list):\n            x_recon[:, :, :, wi[0]:wi[1]] += o[:,:,:,:,index]\n        x_recon = x_recon / normalization\n\n        return x_recon\n\n\n    def _predict_eps_from_xstart(self, x_t, t, pred_xstart):\n        return (extract_into_tensor(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - pred_xstart) / \\\n               extract_into_tensor(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape)\n\n    def _prior_bpd(self, x_start):\n        """\n        Get the prior KL term for the variational lower-bound, measured in\n        bits-per-dim.\n        This term can\'t be optimized, as it only depends on the encoder.\n        :param x_start: the [N x C x ...] tensor of inputs.\n        :return: a batch of [N] KL values (in bits), one per batch element.\n        """\n        batch_size = x_start.shape[0]\n        t = torch.tensor([self.num_timesteps - 1] * batch_size, device=x_start.device)\n        qt_mean, _, qt_log_variance = self.q_mean_variance(x_start, t)\n        kl_prior = normal_kl(mean1=qt_mean, logvar1=qt_log_variance, mean2=0.0, logvar2=0.0)\n        return mean_flat(kl_prior) / np.log(2.0)\n\n    def p_losses(self, x_start, cond, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)\n        model_output = self.apply_model(x_noisy, t, cond)\n\n        loss_dict = {}\n        prefix = \'train\' if self.training else \'val\'\n\n        if self.parameterization == "x0":\n            target = x_start\n        elif self.parameterization == "eps":\n            target = noise\n        else:\n            raise NotImplementedError()\n\n        loss_simple = self.get_loss(model_output, target, mean=False).mean([1, 2, 3])\n        if self.pixel_loss_type is not None:\n            if self.parameterization == "eps":\n                x_recon = self.predict_start_from_noise(x_noisy, t=t, noise=model_output)\n            elif self.parameterization == "x0":\n                x_recon = model_output\n            # decode x0 from xt\n            # decode from \n            x0_decoded = self.decode_first_stage(x_recon)\n            ocr_loss = self.pixel_loss(x0_decoded, cond[\'label_ocr\'])\n            loss_simple += ocr_loss\n            \n        loss_dict.update({f\'{prefix}/loss_simple\': loss_simple.mean()})\n\n        self.logvar = self.logvar.to(self.device)\n        logvar_t = self.logvar[t].to(self.device)\n        loss = loss_simple / torch.exp(logvar_t) + logvar_t\n\n        loss = self.l_simple_weight * loss.mean()\n\n        loss_vlb = self.get_loss(model_output, target, mean=False).mean(dim=(1, 2, 3))\n        loss_vlb = (self.lvlb_weights[t] * loss_vlb).mean()\n        loss_dict.update({f\'{prefix}/loss_vlb\': loss_vlb})\n        loss += (self.original_elbo_weight * loss_vlb)\n        loss_dict.update({f\'{prefix}/loss\': loss})\n\n        return loss, loss_dict\n\n\n    def p_losses_return_noisy(self, x_start, cond, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n        x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)\n        model_output = self.apply_model(x_noisy, t, cond)\n\n        loss_dict = {}\n        prefix = \'train\' if self.training else \'val\'\n\n        if self.parameterization == "x0":\n            target = x_start\n        elif self.parameterization == "eps":\n            target = noise\n        else:\n            raise NotImplementedError()\n\n        loss_simple = self.get_loss(model_output, target, mean=False).mean([1, 2, 3])\n        if self.pixel_loss_type is not None:\n            if self.parameterization == "eps":\n                x_recon = self.predict_start_from_noise(x_noisy, t=t, noise=model_output)\n            elif self.parameterization == "x0":\n                x_recon = model_output\n            # decode x0 from xt\n            # decode from \n            x0_decoded = self.decode_first_stage(x_recon)\n            ocr_loss = self.pixel_loss(x0_decoded, cond[\'label_ocr\'])\n            loss_simple += ocr_loss\n            \n        loss_dict.update({f\'{prefix}/loss_simple\': loss_simple.mean()})\n\n        self.logvar = self.logvar.to(self.device)\n        logvar_t = self.logvar[t].to(self.device)\n        loss = loss_simple / torch.exp(logvar_t) + logvar_t\n\n        loss = self.l_simple_weight * loss.mean()\n\n        loss_vlb = self.get_loss(model_output, target, mean=False).mean(dim=(1, 2, 3))\n        loss_vlb = (self.lvlb_weights[t] * loss_vlb).mean()\n        loss_dict.update({f\'{prefix}/loss_vlb\': loss_vlb})\n        loss += (self.original_elbo_weight * loss_vlb)\n        loss_dict.update({f\'{prefix}/loss\': loss})\n\n        return loss, loss_dict, x_noisy, model_output\n\n        \n    def p_mean_variance(self, x, c, t, clip_denoised: bool, return_codebook_ids=False, quantize_denoised=False,\n                        return_x0=False, score_corrector=None, corrector_kwargs=None):\n        t_in = t\n        model_out = self.apply_model(x, t_in, c, return_ids=return_codebook_ids)\n\n        if score_corrector is not None:\n            assert self.parameterization == "eps"\n            model_out = score_corrector.modify_score(self, model_out, x, t, c, **corrector_kwargs)\n\n        if return_codebook_ids:\n            model_out, logits = model_out\n\n        if self.parameterization == "eps":\n            x_recon = self.predict_start_from_noise(x, t=t, noise=model_out)\n        elif self.parameterization == "x0":\n            x_recon = model_out\n        else:\n            raise NotImplementedError()\n\n        if clip_denoised:\n            x_recon.clamp_(-1., 1.)\n        if quantize_denoised:\n            x_recon, _, [_, _, indices] = self.first_stage_model.quantize(x_recon)\n        model_mean, posterior_variance, posterior_log_variance = self.q_posterior(x_start=x_recon, x_t=x, t=t)\n        if return_codebook_ids:\n            return model_mean, posterior_variance, posterior_log_variance, logits\n        elif return_x0:\n            return model_mean, posterior_variance, posterior_log_variance, x_recon\n        else:\n            return model_mean, posterior_variance, posterior_log_variance\n\n    @torch.no_grad()\n    def p_sample(self, x, c, t, clip_denoised=False, repeat_noise=False,\n                 return_codebook_ids=False, quantize_denoised=False, return_x0=False,\n                 temperature=1., noise_dropout=0., score_corrector=None, corrector_kwargs=None):\n        b, *_, device = *x.shape, x.device\n        outputs = self.p_mean_variance(x=x, c=c, t=t, clip_denoised=clip_denoised,\n                                       return_codebook_ids=return_codebook_ids,\n                                       quantize_denoised=quantize_denoised,\n                                       return_x0=return_x0,\n                                       score_corrector=score_corrector, corrector_kwargs=corrector_kwargs)\n        if return_codebook_ids:\n            raise DeprecationWarning("Support dropped.")\n            model_mean, _, model_log_variance, logits = outputs\n        elif return_x0:\n            model_mean, _, model_log_variance, x0 = outputs\n        else:\n            model_mean, _, model_log_variance = outputs\n\n        noise = noise_like(x.shape, device, repeat_noise) * temperature\n        if noise_dropout > 0.:\n            noise = torch.nn.functional.dropout(noise, p=noise_dropout)\n        # no noise when t == 0\n        nonzero_mask = (1 - (t == 0).float()).reshape(b, *((1,) * (len(x.shape) - 1)))\n\n        if return_codebook_ids:\n            return model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise, logits.argmax(dim=1)\n        if return_x0:\n            return model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise, x0\n        else:\n            return model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise\n\n    @torch.no_grad()\n    def progressive_denoising(self, cond, shape, verbose=True, callback=None, quantize_denoised=False,\n                              img_callback=None, mask=None, x0=None, temperature=1., noise_dropout=0.,\n                              score_corrector=None, corrector_kwargs=None, batch_size=None, x_T=None, start_T=None,\n                              log_every_t=None):\n        if not log_every_t:\n            log_every_t = self.log_every_t\n        timesteps = self.num_timesteps\n        if batch_size is not None:\n            b = batch_size if batch_size is not None else shape[0]\n            shape = [batch_size] + list(shape)\n        else:\n            b = batch_size = shape[0]\n        if x_T is None:\n            img = torch.randn(shape, device=self.device)\n        else:\n            img = x_T\n        intermediates = []\n        if cond is not None:\n            if isinstance(cond, dict):\n                cond = {key: cond[key][:batch_size] if not isinstance(cond[key], list) else\n                list(map(lambda x: x[:batch_size], cond[key])) for key in cond}\n            else:\n                cond = [c[:batch_size] for c in cond] if isinstance(cond, list) else cond[:batch_size]\n\n        if start_T is not None:\n            timesteps = min(timesteps, start_T)\n        iterator = tqdm(reversed(range(0, timesteps)), desc=\'Progressive Generation\',\n                        total=timesteps) if verbose else reversed(\n            range(0, timesteps))\n        if type(temperature) == float:\n            temperature = [temperature] * timesteps\n\n        for i in iterator:\n            ts = torch.full((b,), i, device=self.device, dtype=torch.long)\n            if self.shorten_cond_schedule:\n                assert self.model.conditioning_key != \'hybrid\'\n                tc = self.cond_ids[ts].to(cond.device)\n                cond = self.q_sample(x_start=cond, t=tc, noise=torch.randn_like(cond))\n\n            img, x0_partial = self.p_sample(img, cond, ts,\n                                            clip_denoised=self.clip_denoised,\n                                            quantize_denoised=quantize_denoised, return_x0=True,\n                                            temperature=temperature[i], noise_dropout=noise_dropout,\n                                            score_corrector=score_corrector, corrector_kwargs=corrector_kwargs)\n            if mask is not None:\n                assert x0 is not None\n                img_orig = self.q_sample(x0, ts)\n                img = img_orig * mask + (1. - mask) * img\n\n            if i % log_every_t == 0 or i == timesteps - 1:\n                intermediates.append(x0_partial)\n            if callback: callback(i)\n            if img_callback: img_callback(img, i)\n        return img, intermediates\n\n    @torch.no_grad()\n    def p_sample_loop(self, cond, shape, return_intermediates=False,\n                      x_T=None, verbose=True, callback=None, timesteps=None, quantize_denoised=False,\n                      mask=None, x0=None, img_callback=None, start_T=None,\n                      log_every_t=None):\n\n        if not log_every_t:\n            log_every_t = self.log_every_t\n        device = self.betas.device\n        b = shape[0]\n        if x_T is None:\n            img = torch.randn(shape, device=device)\n        else:\n            img = x_T\n\n        intermediates = [img]\n        if timesteps is None:\n            timesteps = self.num_timesteps\n\n        if start_T is not None:\n            timesteps = min(timesteps, start_T)\n        iterator = tqdm(reversed(range(0, timesteps)), desc=\'Sampling t\', total=timesteps) if verbose else reversed(\n            range(0, timesteps))\n\n        if mask is not None:\n            assert x0 is not None\n            assert x0.shape[2:3] == mask.shape[2:3]  # spatial size has to match\n\n        for i in iterator:\n            ts = torch.full((b,), i, device=device, dtype=torch.long)\n\n            img = self.p_sample(img, cond, ts,\n                                clip_denoised=self.clip_denoised,\n                                quantize_denoised=quantize_denoised)\n            if mask is not None:\n                img_orig = self.q_sample(x0, ts)\n                img = img_orig * mask + (1. - mask) * img\n\n            if i % log_every_t == 0 or i == timesteps - 1:\n                intermediates.append(img)\n            if callback: callback(i)\n            if img_callback: img_callback(img, i)\n\n        if return_intermediates:\n            return img, intermediates\n        return img\n\n    @torch.no_grad()\n    def sample(self, cond, batch_size=16, return_intermediates=False, x_T=None,\n               verbose=True, timesteps=None, quantize_denoised=False,\n               mask=None, x0=None, shape=None,**kwargs):\n        if shape is None:\n            shape = (batch_size, self.channels, self.image_size[0], self.image_size[1])\n        if cond is not None:\n            if isinstance(cond, dict):\n                cbs = cond[list(cond.keys())[0]].shape[0]\n                if cbs != batch_size:\n                    print(f"Warning: Got {cbs} conditionings but batch-size is {batch_size}")\n            else:\n                if cond.shape[0] != batch_size:\n                    print(f"Warning: Got {cond.shape[0]} conditionings but batch-size is {batch_size}")\n                \n        return self.p_sample_loop(cond,\n                                  shape,\n                                  return_intermediates=return_intermediates, x_T=x_T,\n                                  verbose=verbose, timesteps=timesteps, quantize_denoised=quantize_denoised,\n                                  mask=mask, x0=x0)\n\n    @torch.no_grad()\n    def sample_log(self,cond,batch_size,ddim, ddim_steps,**kwargs):\n\n        if ddim:\n            ddim_sampler = DDIMSampler(self)\n            shape = (self.channels, self.image_size[0], self.image_size[1])\n            samples, intermediates =ddim_sampler.sample(ddim_steps,batch_size,\n                                                        shape,cond,verbose=False,**kwargs)\n\n        else:\n            samples, intermediates = self.sample(cond=cond, batch_size=batch_size,\n                                                 return_intermediates=True,**kwargs)\n\n        return samples, intermediates\n\n\n    @torch.no_grad()\n    def log_images(self, batch, N=8, n_row=4, sample=True, ddim_steps=200, ddim_eta=1., return_keys=None,\n                   quantize_denoised=True, inpaint=False, plot_denoise_rows=False, plot_progressive_rows=True,\n                   plot_diffusion_rows=True, **kwargs):\n\n        use_ddim = ddim_steps is not None\n\n        log = dict()\n        z, c, x, xrec, xc = self.get_input(batch, self.first_stage_key,\n                                           return_first_stage_outputs=True,\n                                           force_c_encode=True,\n                                           return_original_cond=True,\n                                           cond_key = self.cond_stage_key,\n                                           bs=N)\n\n        N = min(x.shape[0], N)\n        n_row = min(x.shape[0], n_row)\n        log["inputs"] = x\n        log["reconstruction"] = xrec\n        if self.model.conditioning_key is not None:\n            if hasattr(self.cond_stage_model, "decode"):\n                if self.cond_stage_key == [\'masked_image\', \'text_image\'] :\n                    for index, condition_key in enumerate(self.cond_stage_key):\n                        xc = self.cond_stage_model.decode(c[:, 3*index:3*index+3 , :, :])\n                        log[condition_key] = xc\n                else:\n                    xc = self.cond_stage_model.decode(c)\n                    log["conditioning"] = xc\n            elif self.conditioning_key == \'concat_and_crossattn\':\n                if set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key):\n                    lq_image_encoded = c[\'lq_image\']\n                    lq_image_decoded = self.hybrid_concat_cond_stage_model.decode(lq_image_encoded)\n                    log["conditioning_lq_image"] = lq_image_decoded\n\n                    text_label = c[\'text_label\']\n                    text_label_image = log_txt_as_img_from_alphabet((x.shape[3], x.shape[2]), text_label)\n                    log["conditioning_text"] = text_label_image\n            if ismap(xc):\n                log["original_conditioning"] = self.to_rgb(xc)\n\n        if plot_diffusion_rows:\n            # get diffusion row\n            diffusion_row = list()\n            z_start = z[:n_row]\n            for t in range(self.num_timesteps):\n                if t % self.log_every_t == 0 or t == self.num_timesteps - 1:\n                    t = repeat(torch.tensor([t]), \'1 -> b\', b=n_row)\n                    t = t.to(self.device).long()\n                    noise = torch.randn_like(z_start)\n                    z_noisy = self.q_sample(x_start=z_start, t=t, noise=noise)\n                    diffusion_row.append(self.decode_first_stage(z_noisy))\n\n            diffusion_row = torch.stack(diffusion_row)  # n_log_step, n_row, C, H, W\n            diffusion_grid = rearrange(diffusion_row, \'n b c h w -> b n c h w\')\n            diffusion_grid = rearrange(diffusion_grid, \'b n c h w -> (b n) c h w\')\n            diffusion_grid = make_grid(diffusion_grid, nrow=diffusion_row.shape[0])\n            log["diffusion_row"] = diffusion_grid\n\n        if set([\'lq_image\', \'text_label\']).issubset(self.cond_stage_key) and self.conditioning_key == \'concat_and_crossattn\':\n            c = self.get_text_learned_conditioning_hybrid(c)\n\n        if sample:\n            # get denoise row\n            with self.ema_scope("Plotting"):\n                samples, z_denoise_row = self.sample_log(cond=c,batch_size=N,ddim=use_ddim,\n                                                         ddim_steps=ddim_steps,eta=ddim_eta)\n                # samples, z_denoise_row = self.sample(cond=c, batch_size=N, return_intermediates=True)\n            x_samples = self.decode_first_stage(samples)\n            log["samples"] = x_samples\n            if plot_denoise_rows:\n                denoise_grid = self._get_denoise_row_from_list(z_denoise_row)\n                log["denoise_row"] = denoise_grid\n\n            if quantize_denoised and not isinstance(self.first_stage_model, AutoencoderKL) and not isinstance(\n                    self.first_stage_model, IdentityFirstStage):\n                # also display when quantizing x0 while sampling\n                with self.ema_scope("Plotting Quantized Denoised"):\n                    samples, z_denoise_row = self.sample_log(cond=c,batch_size=N,ddim=use_ddim,\n                                                             ddim_steps=ddim_steps,eta=ddim_eta,\n                                                             quantize_denoised=True)\n\n                x_samples = self.decode_first_stage(samples.to(self.device))\n                log["samples_x0_quantized"] = x_samples\n\n            if inpaint:\n                # make a simple center square\n                b, h, w = z.shape[0], z.shape[2], z.shape[3]\n                mask = torch.ones(N, h, w).to(self.device)\n                # zeros will be filled in\n                mask[:, h // 4:3 * h // 4, w // 4:3 * w // 4] = 0.\n                mask = mask[:, None, ...]\n                with self.ema_scope("Plotting Inpaint"):\n\n                    samples, _ = self.sample_log(cond=c,batch_size=N,ddim=use_ddim, eta=ddim_eta,\n                                                ddim_steps=ddim_steps, x0=z[:N], mask=mask)\n                x_samples = self.decode_first_stage(samples.to(self.device))\n                log["samples_inpainting"] = x_samples\n                log["mask"] = mask\n\n                # outpaint\n                with self.ema_scope("Plotting Outpaint"):\n                    samples, _ = self.sample_log(cond=c, batch_size=N, ddim=use_ddim,eta=ddim_eta,\n                                                ddim_steps=ddim_steps, x0=z[:N], mask=mask)\n                x_samples = self.decode_first_stage(samples.to(self.device))\n                log["samples_outpainting"] = x_samples\n\n        if plot_progressive_rows:\n            with self.ema_scope("Plotting Progressives"):\n                img, progressives = self.progressive_denoising(c,\n                                                               shape=(self.channels, self.image_size[0], self.image_size[1]),\n                                                               batch_size=N)\n            prog_row = self._get_denoise_row_from_list(progressives, desc="Progressive Generation")\n            log["progressive_row"] = prog_row\n\n        if return_keys:\n            if np.intersect1d(list(log.keys()), return_keys).shape[0] == 0:\n                return log\n            else:\n                return {key: log[key] for key in return_keys}\n        return log\n\n    def configure_optimizers(self):\n        lr = self.learning_rate\n        params = list(self.model.parameters())\n        if self.cond_stage_trainable:\n            if self.conditioning_key == \'concat_and_crossattn\':\n                print(f"{self.__class__.__name__}: Also optimizing conditioner params!")\n                params = params + list(self.hybrid_concat_cond_stage_model.parameters()) + list(self.hybrid_cross_attn_cond_stage_model.parameters())\n            else:\n                print(f"{self.__class__.__name__}: Also optimizing conditioner params!")\n                params = params + list(self.cond_stage_model.parameters())\n        opt = torch.optim.AdamW(params, lr=lr)\n        if self.use_scheduler:\n            assert \'target\' in self.scheduler_config\n            scheduler = instantiate_from_config(self.scheduler_config)\n\n            print("Setting up LambdaLR scheduler...")\n            scheduler = [\n                {\n                    \'scheduler\': LambdaLR(opt, lr_lambda=scheduler.schedule),\n                    \'interval\': \'step\',\n                    \'frequency\': 1\n                }]\n            return [opt], scheduler\n        return opt\n\n    @torch.no_grad()\n    def to_rgb(self, x):\n        x = x.float()\n        if not hasattr(self, "colorize"):\n            self.colorize = torch.randn(3, x.shape[1], 1, 1).to(x)\n        x = nn.functional.conv2d(x, weight=self.colorize)\n        x = 2. * (x - x.min()) / (x.max() - x.min()) - 1.\n        return x\n\n\nclass DiffusionWrapper(pl.LightningModule):\n    def __init__(self, diff_model_config, conditioning_key):\n        super().__init__()\n        self.diffusion_model = instantiate_from_config(diff_model_config)\n        self.conditioning_key = conditioning_key\n        assert self.conditioning_key in [None, \'concat\', \'crossattn\', \'concat_and_crossattn\']\n\n    def forward(self, x, t, c_concat: list = None, c_crossattn: list = None):\n        if self.conditioning_key is None:\n            out = self.diffusion_model(x, t)\n        elif self.conditioning_key == \'concat\':\n            xc = torch.cat([x] + c_concat, dim=1)\n            out = self.diffusion_model(xc, t)\n        elif self.conditioning_key == \'crossattn\':\n            cc = torch.cat(c_crossattn, 1)\n            out = self.diffusion_model(x, t, context=cc)\n        elif self.conditioning_key == \'concat_and_crossattn\':\n            xc = torch.cat([x] + c_concat, dim=1)\n            cc = torch.cat(c_crossattn, 1)\n            out = self.diffusion_model(xc, t, context=cc)\n        else:\n            raise NotImplementedError()\n\n        return out\n\n', 'model/IDM/modules/attention.py': 'from inspect import isfunction\nimport math\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn, einsum\nfrom einops import rearrange, repeat\n\nfrom model.IDM.modules.diffusionmodules.util import checkpoint\n\n\ndef exists(val):\n    return val is not None\n\n\ndef uniq(arr):\n    return{el: True for el in arr}.keys()\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\ndef max_neg_value(t):\n    return -torch.finfo(t.dtype).max\n\n\ndef init_(tensor):\n    dim = tensor.shape[-1]\n    std = 1 / math.sqrt(dim)\n    tensor.uniform_(-std, std)\n    return tensor\n\n\n# feedforward\nclass GEGLU(nn.Module):\n    def __init__(self, dim_in, dim_out):\n        super().__init__()\n        self.proj = nn.Linear(dim_in, dim_out * 2)\n\n    def forward(self, x):\n        x, gate = self.proj(x).chunk(2, dim=-1)\n        return x * F.gelu(gate)\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, dim, dim_out=None, mult=4, glu=False, dropout=0.):\n        super().__init__()\n        inner_dim = int(dim * mult)\n        dim_out = default(dim_out, dim)\n        project_in = nn.Sequential(\n            nn.Linear(dim, inner_dim),\n            nn.GELU()\n        ) if not glu else GEGLU(dim, inner_dim)\n\n        self.net = nn.Sequential(\n            project_in,\n            nn.Dropout(dropout),\n            nn.Linear(inner_dim, dim_out)\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\ndef zero_module(module):\n    """\n    Zero out the parameters of a module and return it.\n    """\n    for p in module.parameters():\n        p.detach().zero_()\n    return module\n\n\ndef Normalize(in_channels):\n    return torch.nn.GroupNorm(num_groups=32, num_channels=in_channels, eps=1e-6, affine=True)\n\n\nclass LinearAttention(nn.Module):\n    def __init__(self, dim, heads=4, dim_head=32):\n        super().__init__()\n        self.heads = heads\n        hidden_dim = dim_head * heads\n        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)\n        self.to_out = nn.Conv2d(hidden_dim, dim, 1)\n\n    def forward(self, x):\n        b, c, h, w = x.shape\n        qkv = self.to_qkv(x)\n        q, k, v = rearrange(qkv, \'b (qkv heads c) h w -> qkv b heads c (h w)\', heads = self.heads, qkv=3)\n        k = k.softmax(dim=-1)  \n        context = torch.einsum(\'bhdn,bhen->bhde\', k, v)\n        out = torch.einsum(\'bhde,bhdn->bhen\', context, q)\n        out = rearrange(out, \'b heads c (h w) -> b (heads c) h w\', heads=self.heads, h=h, w=w)\n        return self.to_out(out)\n\n\nclass SpatialSelfAttention(nn.Module):\n    def __init__(self, in_channels):\n        super().__init__()\n        self.in_channels = in_channels\n\n        self.norm = Normalize(in_channels)\n        self.q = torch.nn.Conv2d(in_channels,\n                                 in_channels,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n        self.k = torch.nn.Conv2d(in_channels,\n                                 in_channels,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n        self.v = torch.nn.Conv2d(in_channels,\n                                 in_channels,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n        self.proj_out = torch.nn.Conv2d(in_channels,\n                                        in_channels,\n                                        kernel_size=1,\n                                        stride=1,\n                                        padding=0)\n\n    def forward(self, x):\n        h_ = x\n        h_ = self.norm(h_)\n        q = self.q(h_)\n        k = self.k(h_)\n        v = self.v(h_)\n\n        # compute attention\n        b,c,h,w = q.shape\n        q = rearrange(q, \'b c h w -> b (h w) c\')\n        k = rearrange(k, \'b c h w -> b c (h w)\')\n        w_ = torch.einsum(\'bij,bjk->bik\', q, k)\n\n        w_ = w_ * (int(c)**(-0.5))\n        w_ = torch.nn.functional.softmax(w_, dim=2)\n\n        # attend to values\n        v = rearrange(v, \'b c h w -> b c (h w)\')\n        w_ = rearrange(w_, \'b i j -> b j i\')\n        h_ = torch.einsum(\'bij,bjk->bik\', v, w_)\n        h_ = rearrange(h_, \'b c (h w) -> b c h w\', h=h)\n        h_ = self.proj_out(h_)\n\n        return x+h_\n\n\nclass CrossAttention(nn.Module):\n    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64, dropout=0.):\n        super().__init__()\n        inner_dim = dim_head * heads\n        context_dim = default(context_dim, query_dim)\n\n        self.scale = dim_head ** -0.5\n        self.heads = heads\n\n        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)\n        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)\n        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)\n\n        self.to_out = nn.Sequential(\n            nn.Linear(inner_dim, query_dim),\n            nn.Dropout(dropout)\n        )\n\n    def forward(self, x, context=None, mask=None):\n        h = self.heads\n\n        q = self.to_q(x)\n        context = default(context, x)\n        k = self.to_k(context)\n        v = self.to_v(context)\n\n        q, k, v = map(lambda t: rearrange(t, \'b n (h d) -> (b h) n d\', h=h), (q, k, v))\n\n        sim = einsum(\'b i d, b j d -> b i j\', q, k) * self.scale\n\n        if exists(mask):\n            mask = rearrange(mask, \'b ... -> b (...)\')\n            max_neg_value = -torch.finfo(sim.dtype).max\n            mask = repeat(mask, \'b j -> (b h) () j\', h=h)\n            sim.masked_fill_(~mask, max_neg_value)\n\n        # attention, what we cannot get enough of\n        attn = sim.softmax(dim=-1)\n\n        out = einsum(\'b i j, b j d -> b i d\', attn, v)\n        out = rearrange(out, \'(b h) n d -> b n (h d)\', h=h)\n        return self.to_out(out)\n\n\nclass BasicTransformerBlock(nn.Module):\n    def __init__(self, dim, n_heads, d_head, dropout=0., context_dim=None, gated_ff=True, checkpoint=True):\n        super().__init__()\n        self.attn1 = CrossAttention(query_dim=dim, heads=n_heads, dim_head=d_head, dropout=dropout)  # is a self-attention\n        self.ff = FeedForward(dim, dropout=dropout, glu=gated_ff)\n        self.attn2 = CrossAttention(query_dim=dim, context_dim=context_dim,\n                                    heads=n_heads, dim_head=d_head, dropout=dropout)  # is self-attn if context is none\n        self.norm1 = nn.LayerNorm(dim)\n        self.norm2 = nn.LayerNorm(dim)\n        self.norm3 = nn.LayerNorm(dim)\n        self.checkpoint = checkpoint\n\n    def forward(self, x, context=None):\n        return checkpoint(self._forward, (x, context), self.parameters(), self.checkpoint)\n\n    def _forward(self, x, context=None):\n        x = self.attn1(self.norm1(x)) + x\n        x = self.attn2(self.norm2(x), context=context) + x\n        x = self.ff(self.norm3(x)) + x\n        return x\n\n\nclass SpatialTransformer(nn.Module):\n    """\n    Transformer block for image-like data.\n    First, project the input (aka embedding)\n    and reshape to b, t, d.\n    Then apply standard transformer action.\n    Finally, reshape to image\n    """\n    def __init__(self, in_channels, n_heads, d_head,\n                 depth=1, dropout=0., context_dim=None):\n        super().__init__()\n        self.in_channels = in_channels\n        inner_dim = n_heads * d_head\n        self.norm = Normalize(in_channels)\n\n        self.proj_in = nn.Conv2d(in_channels,\n                                 inner_dim,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n\n        self.transformer_blocks = nn.ModuleList(\n            [BasicTransformerBlock(inner_dim, n_heads, d_head, dropout=dropout, context_dim=context_dim)\n                for d in range(depth)]\n        )\n\n        self.proj_out = zero_module(nn.Conv2d(inner_dim,\n                                              in_channels,\n                                              kernel_size=1,\n                                              stride=1,\n                                              padding=0))\n\n    def forward(self, x, context=None):\n        # note: if no context is given, cross-attention defaults to self-attention\n        b, c, h, w = x.shape\n        x_in = x\n        x = self.norm(x)\n        x = self.proj_in(x)\n        x = rearrange(x, \'b c h w -> b (h w) c\')\n        for block in self.transformer_blocks:\n            x = block(x, context=context)\n        x = rearrange(x, \'b (h w) c -> b c h w\', h=h, w=w)\n        x = self.proj_out(x)\n        return x + x_in', 'model/IDM/modules/diffusionmodules/__init__.py': '', 'model/IDM/modules/diffusionmodules/ddim.py': '"""SAMPLING ONLY."""\n\nimport torch\nimport numpy as np\nfrom tqdm import tqdm\n\nfrom model.IDM.modules.diffusionmodules.util import make_ddim_sampling_parameters, make_ddim_timesteps, noise_like\n\n\nclass DDIMSampler(object):\n    def __init__(self, model, schedule="linear", **kwargs):\n        super().__init__()\n        self.model = model\n        self.ddpm_num_timesteps = model.num_timesteps\n        self.schedule = schedule\n\n    def register_buffer(self, name, attr):\n        if type(attr) == torch.Tensor:\n            if attr.device != torch.device("cuda"):\n                attr = attr.to(torch.device("cuda"))\n        setattr(self, name, attr)\n\n    def make_schedule(self, ddim_num_steps, ddim_discretize="uniform", ddim_eta=0., verbose=True):\n        self.ddim_timesteps = make_ddim_timesteps(ddim_discr_method=ddim_discretize, num_ddim_timesteps=ddim_num_steps,\n                                                  num_ddpm_timesteps=self.ddpm_num_timesteps,verbose=verbose)\n        alphas_cumprod = self.model.alphas_cumprod\n        assert alphas_cumprod.shape[0] == self.ddpm_num_timesteps, \'alphas have to be defined for each timestep\'\n        to_torch = lambda x: x.clone().detach().to(torch.float32).to(self.model.device)\n\n        self.register_buffer(\'betas\', to_torch(self.model.betas))\n        self.register_buffer(\'alphas_cumprod\', to_torch(alphas_cumprod))\n        self.register_buffer(\'alphas_cumprod_prev\', to_torch(self.model.alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer(\'sqrt_alphas_cumprod\', to_torch(np.sqrt(alphas_cumprod.cpu())))\n        self.register_buffer(\'sqrt_one_minus_alphas_cumprod\', to_torch(np.sqrt(1. - alphas_cumprod.cpu())))\n        self.register_buffer(\'log_one_minus_alphas_cumprod\', to_torch(np.log(1. - alphas_cumprod.cpu())))\n        self.register_buffer(\'sqrt_recip_alphas_cumprod\', to_torch(np.sqrt(1. / alphas_cumprod.cpu())))\n        self.register_buffer(\'sqrt_recipm1_alphas_cumprod\', to_torch(np.sqrt(1. / alphas_cumprod.cpu() - 1)))\n\n        # ddim sampling parameters\n        ddim_sigmas, ddim_alphas, ddim_alphas_prev = make_ddim_sampling_parameters(alphacums=alphas_cumprod.cpu(),\n                                                                                   ddim_timesteps=self.ddim_timesteps,\n                                                                                   eta=ddim_eta,verbose=verbose)\n        self.register_buffer(\'ddim_sigmas\', ddim_sigmas)\n        self.register_buffer(\'ddim_alphas\', ddim_alphas)\n        self.register_buffer(\'ddim_alphas_prev\', ddim_alphas_prev)\n        self.register_buffer(\'ddim_sqrt_one_minus_alphas\', np.sqrt(1. - ddim_alphas))\n        sigmas_for_original_sampling_steps = ddim_eta * torch.sqrt(\n            (1 - self.alphas_cumprod_prev) / (1 - self.alphas_cumprod) * (\n                        1 - self.alphas_cumprod / self.alphas_cumprod_prev))\n        self.register_buffer(\'ddim_sigmas_for_original_num_steps\', sigmas_for_original_sampling_steps)\n\n    @torch.no_grad()\n    def sample(self,\n               S,\n               batch_size,\n               shape,\n               conditioning=None,\n               callback=None,\n               normals_sequence=None,\n               img_callback=None,\n               quantize_x0=False,\n               eta=0.,\n               mask=None,\n               x0=None,\n               temperature=1.,\n               noise_dropout=0.,\n               score_corrector=None,\n               corrector_kwargs=None,\n               verbose=True,\n               x_T=None,\n               log_every_t=100,\n               unconditional_guidance_scale=1.,\n               unconditional_conditioning=None,\n               image_splist_list=None,\n               # this has to come in the same format as the conditioning, # e.g. as encoded tokens, ...\n               **kwargs\n               ):\n\n        self.make_schedule(ddim_num_steps=S, ddim_eta=eta, verbose=verbose)\n        # sampling\n        C, H, W = shape\n        size = (batch_size, C, H, W)\n        print(f\'Data shape for DDIM sampling is {size}, eta {eta}\')\n\n        samples, intermediates = self.ddim_sampling(conditioning, size,\n                                                    callback=callback,\n                                                    img_callback=img_callback,\n                                                    quantize_denoised=quantize_x0,\n                                                    mask=mask, x0=x0,\n                                                    ddim_use_original_steps=False,\n                                                    noise_dropout=noise_dropout,\n                                                    temperature=temperature,\n                                                    score_corrector=score_corrector,\n                                                    corrector_kwargs=corrector_kwargs,\n                                                    x_T=x_T,\n                                                    log_every_t=log_every_t,\n                                                    unconditional_guidance_scale=unconditional_guidance_scale,\n                                                    unconditional_conditioning=unconditional_conditioning,\n                                                    image_splist_list = image_splist_list,\n                                                    )\n        return samples, intermediates\n\n    @torch.no_grad()\n    def ddim_sampling(self, cond, shape,\n                      x_T=None, ddim_use_original_steps=False,\n                      callback=None, timesteps=None, quantize_denoised=False,\n                      mask=None, x0=None, img_callback=None, log_every_t=1,\n                      temperature=1., noise_dropout=0., score_corrector=None, corrector_kwargs=None,\n                      unconditional_guidance_scale=1., unconditional_conditioning=None, image_splist_list=None):\n        device = self.model.betas.device\n        b = shape[0]\n        if x_T is None:\n            img = torch.randn(shape, device=device)\n        else:\n            img = x_T\n\n        if timesteps is None:\n            timesteps = self.ddpm_num_timesteps if ddim_use_original_steps else self.ddim_timesteps\n        elif timesteps is not None and not ddim_use_original_steps:\n            subset_end = int(min(timesteps / self.ddim_timesteps.shape[0], 1) * self.ddim_timesteps.shape[0]) - 1\n            timesteps = self.ddim_timesteps[:subset_end]\n\n        intermediates = {\'x_inter\': [img], \'pred_x0\': [img]}\n        time_range = reversed(range(0,timesteps)) if ddim_use_original_steps else np.flip(timesteps)\n        total_steps = timesteps if ddim_use_original_steps else timesteps.shape[0]\n        print(f"Running DDIM Sampling with {total_steps} timesteps")\n\n        iterator = tqdm(time_range, desc=\'DDIM Sampler\', total=total_steps)\n\n        for i, step in enumerate(iterator):\n            index = total_steps - i - 1\n            ts = torch.full((b,), step, device=device, dtype=torch.long)\n\n            if mask is not None:\n                assert x0 is not None\n                img_orig = self.model.q_sample(x0, ts)  # TODO: deterministic forward pass?\n                img = img_orig * mask + (1. - mask) * img\n\n            outs = self.p_sample_ddim(img, cond, ts, index=index, use_original_steps=ddim_use_original_steps,\n                                      quantize_denoised=quantize_denoised, temperature=temperature,\n                                      noise_dropout=noise_dropout, score_corrector=score_corrector,\n                                      corrector_kwargs=corrector_kwargs,\n                                      unconditional_guidance_scale=unconditional_guidance_scale,\n                                      unconditional_conditioning=unconditional_conditioning,\n                                      image_splist_list=image_splist_list)\n            img, pred_x0 = outs\n            if callback: callback(i)\n            if img_callback: img_callback(pred_x0, i)\n\n            if index % log_every_t == 0 or index == total_steps - 1:\n                intermediates[\'x_inter\'].append(img)\n                intermediates[\'pred_x0\'].append(pred_x0)\n\n        return img, intermediates\n\n    @torch.no_grad()\n    def p_sample_ddim(self, x, c, t, index, repeat_noise=False, use_original_steps=False, quantize_denoised=False,\n                      temperature=1., noise_dropout=0., score_corrector=None, corrector_kwargs=None,\n                      unconditional_guidance_scale=1., unconditional_conditioning=None, image_splist_list=None):\n        b, *_, device = *x.shape, x.device\n\n        if unconditional_conditioning is None or unconditional_guidance_scale == 1.:\n            if image_splist_list == None:\n                e_t = self.model.apply_model(x, t, c)\n            else:\n                e_t = self.model.apply_model_aggregation(x, t, c, image_splist_list)\n\n        else:\n            x_in = torch.cat([x] * 2)\n            t_in = torch.cat([t] * 2)\n            c_in = torch.cat([unconditional_conditioning, c])\n            e_t_uncond, e_t = self.model.apply_model(x_in, t_in, c_in).chunk(2)\n            e_t = e_t_uncond + unconditional_guidance_scale * (e_t - e_t_uncond)\n\n        if score_corrector is not None:\n            assert self.model.parameterization == "eps"\n            e_t = score_corrector.modify_score(self.model, e_t, x, t, c, **corrector_kwargs)\n\n        alphas = self.model.alphas_cumprod if use_original_steps else self.ddim_alphas\n        alphas_prev = self.model.alphas_cumprod_prev if use_original_steps else self.ddim_alphas_prev\n        sqrt_one_minus_alphas = self.model.sqrt_one_minus_alphas_cumprod if use_original_steps else self.ddim_sqrt_one_minus_alphas\n        sigmas = self.model.ddim_sigmas_for_original_num_steps if use_original_steps else self.ddim_sigmas\n        # select parameters corresponding to the currently considered timestep\n        a_t = torch.full((b, 1, 1, 1), alphas[index], device=device)\n        a_prev = torch.full((b, 1, 1, 1), alphas_prev[index], device=device)\n        sigma_t = torch.full((b, 1, 1, 1), sigmas[index], device=device)\n        sqrt_one_minus_at = torch.full((b, 1, 1, 1), sqrt_one_minus_alphas[index],device=device)\n\n        # current prediction for x_0\n        pred_x0 = (x - sqrt_one_minus_at * e_t) / a_t.sqrt()\n        if quantize_denoised:\n            pred_x0, _, *_ = self.model.first_stage_model.quantize(pred_x0)\n        # direction pointing to x_t\n        dir_xt = (1. - a_prev - sigma_t**2).sqrt() * e_t\n        noise = sigma_t * noise_like(x.shape, device, repeat_noise) * temperature\n        if noise_dropout > 0.:\n            noise = torch.nn.functional.dropout(noise, p=noise_dropout)\n        x_prev = a_prev.sqrt() * pred_x0 + dir_xt + noise\n        return x_prev, pred_x0', 'model/IDM/modules/diffusionmodules/model.py': '# pytorch_diffusion + derived encoder decoder\nimport math\nimport torch\nimport torch.nn as nn\nimport numpy as np\n\nfrom model.IDM.modules.attention import LinearAttention\n\n\ndef get_timestep_embedding(timesteps, embedding_dim):\n    """\n    This matches the implementation in Denoising Diffusion Probabilistic Models:\n    From Fairseq.\n    Build sinusoidal embeddings.\n    This matches the implementation in tensor2tensor, but differs slightly\n    from the description in Section 3.5 of "Attention Is All You Need".\n    """\n    assert len(timesteps.shape) == 1\n\n    half_dim = embedding_dim // 2\n    emb = math.log(10000) / (half_dim - 1)\n    emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)\n    emb = emb.to(device=timesteps.device)\n    emb = timesteps.float()[:, None] * emb[None, :]\n    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)\n    if embedding_dim % 2 == 1:  # zero pad\n        emb = torch.nn.functional.pad(emb, (0,1,0,0))\n    return emb\n\n\ndef nonlinearity(x):\n    # swish\n    return x*torch.sigmoid(x)\n\n\ndef Normalize(in_channels, num_groups=32):\n    return torch.nn.GroupNorm(num_groups=num_groups, num_channels=in_channels, eps=1e-6, affine=True)\n\n\nclass Upsample(nn.Module):\n    def __init__(self, in_channels, with_conv):\n        super().__init__()\n        self.with_conv = with_conv\n        if self.with_conv:\n            self.conv = torch.nn.Conv2d(in_channels,\n                                        in_channels,\n                                        kernel_size=3,\n                                        stride=1,\n                                        padding=1)\n\n    def forward(self, x):\n        x = torch.nn.functional.interpolate(x, scale_factor=2.0, mode="nearest")\n        if self.with_conv:\n            x = self.conv(x)\n        return x\n\n\nclass Downsample(nn.Module):\n    def __init__(self, in_channels, with_conv):\n        super().__init__()\n        self.with_conv = with_conv\n        if self.with_conv:\n            # no asymmetric padding in torch conv, must do it ourselves\n            self.conv = torch.nn.Conv2d(in_channels,\n                                        in_channels,\n                                        kernel_size=3,\n                                        stride=2,\n                                        padding=0)\n\n    def forward(self, x):\n        if self.with_conv:\n            pad = (0,1,0,1)\n            x = torch.nn.functional.pad(x, pad, mode="constant", value=0)\n            x = self.conv(x)\n        else:\n            x = torch.nn.functional.avg_pool2d(x, kernel_size=2, stride=2)\n        return x\n\n\nclass ResnetBlock(nn.Module):\n    def __init__(self, *, in_channels, out_channels=None, conv_shortcut=False,\n                 dropout, temb_channels=512):\n        super().__init__()\n        self.in_channels = in_channels\n        out_channels = in_channels if out_channels is None else out_channels\n        self.out_channels = out_channels\n        self.use_conv_shortcut = conv_shortcut\n\n        self.norm1 = Normalize(in_channels)\n        self.conv1 = torch.nn.Conv2d(in_channels,\n                                     out_channels,\n                                     kernel_size=3,\n                                     stride=1,\n                                     padding=1)\n        if temb_channels > 0:\n            self.temb_proj = torch.nn.Linear(temb_channels,\n                                             out_channels)\n        self.norm2 = Normalize(out_channels)\n        self.dropout = torch.nn.Dropout(dropout)\n        self.conv2 = torch.nn.Conv2d(out_channels,\n                                     out_channels,\n                                     kernel_size=3,\n                                     stride=1,\n                                     padding=1)\n        if self.in_channels != self.out_channels:\n            if self.use_conv_shortcut:\n                self.conv_shortcut = torch.nn.Conv2d(in_channels,\n                                                     out_channels,\n                                                     kernel_size=3,\n                                                     stride=1,\n                                                     padding=1)\n            else:\n                self.nin_shortcut = torch.nn.Conv2d(in_channels,\n                                                    out_channels,\n                                                    kernel_size=1,\n                                                    stride=1,\n                                                    padding=0)\n\n    def forward(self, x, temb):\n        h = x\n        h = self.norm1(h)\n        h = nonlinearity(h)\n        h = self.conv1(h)\n\n        if temb is not None:\n            h = h + self.temb_proj(nonlinearity(temb))[:,:,None,None]\n\n        h = self.norm2(h)\n        h = nonlinearity(h)\n        h = self.dropout(h)\n        h = self.conv2(h)\n\n        if self.in_channels != self.out_channels:\n            if self.use_conv_shortcut:\n                x = self.conv_shortcut(x)\n            else:\n                x = self.nin_shortcut(x)\n\n        return x+h\n\n\nclass LinAttnBlock(LinearAttention):\n    """to match AttnBlock usage"""\n    def __init__(self, in_channels):\n        super().__init__(dim=in_channels, heads=1, dim_head=in_channels)\n\n\nclass AttnBlock(nn.Module):\n    def __init__(self, in_channels):\n        super().__init__()\n        self.in_channels = in_channels\n\n        self.norm = Normalize(in_channels)\n        self.q = torch.nn.Conv2d(in_channels,\n                                 in_channels,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n        self.k = torch.nn.Conv2d(in_channels,\n                                 in_channels,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n        self.v = torch.nn.Conv2d(in_channels,\n                                 in_channels,\n                                 kernel_size=1,\n                                 stride=1,\n                                 padding=0)\n        self.proj_out = torch.nn.Conv2d(in_channels,\n                                        in_channels,\n                                        kernel_size=1,\n                                        stride=1,\n                                        padding=0)\n\n\n    def forward(self, x):\n        h_ = x\n        h_ = self.norm(h_)\n        q = self.q(h_)\n        k = self.k(h_)\n        v = self.v(h_)\n\n        # compute attention\n        b,c,h,w = q.shape\n        q = q.reshape(b,c,h*w)\n        q = q.permute(0,2,1)   # b,hw,c\n        k = k.reshape(b,c,h*w) # b,c,hw\n        w_ = torch.bmm(q,k)     # b,hw,hw    w[b,i,j]=sum_c q[b,i,c]k[b,c,j]\n        w_ = w_ * (int(c)**(-0.5))\n        w_ = torch.nn.functional.softmax(w_, dim=2)\n\n        # attend to values\n        v = v.reshape(b,c,h*w)\n        w_ = w_.permute(0,2,1)   # b,hw,hw (first hw of k, second of q)\n        h_ = torch.bmm(v,w_)     # b, c,hw (hw of q) h_[b,c,j] = sum_i v[b,c,i] w_[b,i,j]\n        h_ = h_.reshape(b,c,h,w)\n\n        h_ = self.proj_out(h_)\n\n        return x+h_\n\n\ndef make_attn(in_channels, attn_type="vanilla"):\n    assert attn_type in ["vanilla", "linear", "none"], f\'attn_type {attn_type} unknown\'\n    if attn_type == "vanilla":\n        return AttnBlock(in_channels)\n    elif attn_type == "none":\n        return nn.Identity(in_channels)\n    else:\n        return LinAttnBlock(in_channels)\n\n\nclass Model(nn.Module):\n    def __init__(self, *, ch, out_ch, ch_mult=(1,2,4,8), num_res_blocks,\n                 attn_resolutions, dropout=0.0, resamp_with_conv=True, in_channels,\n                 resolution, use_timestep=True, use_linear_attn=False, attn_type="vanilla"):\n        super().__init__()\n        if use_linear_attn: attn_type = "linear"\n        self.ch = ch\n        self.temb_ch = self.ch*4\n        self.num_resolutions = len(ch_mult)\n        self.num_res_blocks = num_res_blocks\n        self.resolution = resolution\n        self.in_channels = in_channels\n\n        self.use_timestep = use_timestep\n        if self.use_timestep:\n            # timestep embedding\n            self.temb = nn.Module()\n            self.temb.dense = nn.ModuleList([\n                torch.nn.Linear(self.ch,\n                                self.temb_ch),\n                torch.nn.Linear(self.temb_ch,\n                                self.temb_ch),\n            ])\n\n        # downsampling\n        self.conv_in = torch.nn.Conv2d(in_channels,\n                                       self.ch,\n                                       kernel_size=3,\n                                       stride=1,\n                                       padding=1)\n\n        curr_res = resolution\n        in_ch_mult = (1,)+tuple(ch_mult)\n        self.down = nn.ModuleList()\n        for i_level in range(self.num_resolutions):\n            block = nn.ModuleList()\n            attn = nn.ModuleList()\n            block_in = ch*in_ch_mult[i_level]\n            block_out = ch*ch_mult[i_level]\n            for i_block in range(self.num_res_blocks):\n                block.append(ResnetBlock(in_channels=block_in,\n                                         out_channels=block_out,\n                                         temb_channels=self.temb_ch,\n                                         dropout=dropout))\n                block_in = block_out\n                if curr_res in attn_resolutions:\n                    attn.append(make_attn(block_in, attn_type=attn_type))\n            down = nn.Module()\n            down.block = block\n            down.attn = attn\n            if i_level != self.num_resolutions-1:\n                down.downsample = Downsample(block_in, resamp_with_conv)\n                curr_res = curr_res // 2\n            self.down.append(down)\n\n        # middle\n        self.mid = nn.Module()\n        self.mid.block_1 = ResnetBlock(in_channels=block_in,\n                                       out_channels=block_in,\n                                       temb_channels=self.temb_ch,\n                                       dropout=dropout)\n        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)\n        self.mid.block_2 = ResnetBlock(in_channels=block_in,\n                                       out_channels=block_in,\n                                       temb_channels=self.temb_ch,\n                                       dropout=dropout)\n\n        # upsampling\n        self.up = nn.ModuleList()\n        for i_level in reversed(range(self.num_resolutions)):\n            block = nn.ModuleList()\n            attn = nn.ModuleList()\n            block_out = ch*ch_mult[i_level]\n            skip_in = ch*ch_mult[i_level]\n            for i_block in range(self.num_res_blocks+1):\n                if i_block == self.num_res_blocks:\n                    skip_in = ch*in_ch_mult[i_level]\n                block.append(ResnetBlock(in_channels=block_in+skip_in,\n                                         out_channels=block_out,\n                                         temb_channels=self.temb_ch,\n                                         dropout=dropout))\n                block_in = block_out\n                if curr_res in attn_resolutions:\n                    attn.append(make_attn(block_in, attn_type=attn_type))\n            up = nn.Module()\n            up.block = block\n            up.attn = attn\n            if i_level != 0:\n                up.upsample = Upsample(block_in, resamp_with_conv)\n                curr_res = curr_res * 2\n            self.up.insert(0, up) # prepend to get consistent order\n\n        # end\n        self.norm_out = Normalize(block_in)\n        self.conv_out = torch.nn.Conv2d(block_in,\n                                        out_ch,\n                                        kernel_size=3,\n                                        stride=1,\n                                        padding=1)\n\n    def forward(self, x, t=None, context=None):\n        #assert x.shape[2] == x.shape[3] == self.resolution\n        if context is not None:\n            # assume aligned context, cat along channel axis\n            x = torch.cat((x, context), dim=1)\n        if self.use_timestep:\n            # timestep embedding\n            assert t is not None\n            temb = get_timestep_embedding(t, self.ch)\n            temb = self.temb.dense[0](temb)\n            temb = nonlinearity(temb)\n            temb = self.temb.dense[1](temb)\n        else:\n            temb = None\n\n        # downsampling\n        hs = [self.conv_in(x)]\n        for i_level in range(self.num_resolutions):\n            for i_block in range(self.num_res_blocks):\n                h = self.down[i_level].block[i_block](hs[-1], temb)\n                if len(self.down[i_level].attn) > 0:\n                    h = self.down[i_level].attn[i_block](h)\n                hs.append(h)\n            if i_level != self.num_resolutions-1:\n                hs.append(self.down[i_level].downsample(hs[-1]))\n\n        # middle\n        h = hs[-1]\n        h = self.mid.block_1(h, temb)\n        h = self.mid.attn_1(h)\n        h = self.mid.block_2(h, temb)\n\n        # upsampling\n        for i_level in reversed(range(self.num_resolutions)):\n            for i_block in range(self.num_res_blocks+1):\n                h = self.up[i_level].block[i_block](\n                    torch.cat([h, hs.pop()], dim=1), temb)\n                if len(self.up[i_level].attn) > 0:\n                    h = self.up[i_level].attn[i_block](h)\n            if i_level != 0:\n                h = self.up[i_level].upsample(h)\n\n        # end\n        h = self.norm_out(h)\n        h = nonlinearity(h)\n        h = self.conv_out(h)\n        return h\n\n    def get_last_layer(self):\n        return self.conv_out.weight\n\n\nclass Encoder(nn.Module):\n    def __init__(self, *, ch, out_ch, ch_mult=(1,2,4,8), num_res_blocks,\n                 attn_resolutions, dropout=0.0, resamp_with_conv=True, in_channels,\n                 resolution, z_channels, double_z=True, use_linear_attn=False, attn_type="vanilla",\n                 **ignore_kwargs):\n        super().__init__()\n        if use_linear_attn: attn_type = "linear"\n        self.ch = ch\n        self.temb_ch = 0\n        self.num_resolutions = len(ch_mult)\n        self.num_res_blocks = num_res_blocks\n        self.resolution = resolution\n        self.in_channels = in_channels\n\n        # downsampling\n        self.conv_in = torch.nn.Conv2d(in_channels,\n                                       self.ch,\n                                       kernel_size=3,\n                                       stride=1,\n                                       padding=1)\n\n        curr_res = resolution\n        in_ch_mult = (1,)+tuple(ch_mult)\n        self.in_ch_mult = in_ch_mult\n        self.down = nn.ModuleList()\n        for i_level in range(self.num_resolutions):\n            block = nn.ModuleList()\n            attn = nn.ModuleList()\n            block_in = ch*in_ch_mult[i_level]\n            block_out = ch*ch_mult[i_level]\n            for i_block in range(self.num_res_blocks):\n                block.append(ResnetBlock(in_channels=block_in,\n                                         out_channels=block_out,\n                                         temb_channels=self.temb_ch,\n                                         dropout=dropout))\n                block_in = block_out\n                if curr_res in attn_resolutions:\n                    attn.append(make_attn(block_in, attn_type=attn_type))\n            down = nn.Module()\n            down.block = block\n            down.attn = attn\n            if i_level != self.num_resolutions-1:\n                down.downsample = Downsample(block_in, resamp_with_conv)\n                curr_res = curr_res // 2\n            self.down.append(down)\n\n        # middle\n        self.mid = nn.Module()\n        self.mid.block_1 = ResnetBlock(in_channels=block_in,\n                                       out_channels=block_in,\n                                       temb_channels=self.temb_ch,\n                                       dropout=dropout)\n        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)\n        self.mid.block_2 = ResnetBlock(in_channels=block_in,\n                                       out_channels=block_in,\n                                       temb_channels=self.temb_ch,\n                                       dropout=dropout)\n\n        # end\n        self.norm_out = Normalize(block_in)\n        self.conv_out = torch.nn.Conv2d(block_in,\n                                        2*z_channels if double_z else z_channels,\n                                        kernel_size=3,\n                                        stride=1,\n                                        padding=1)\n\n    def forward(self, x):\n        # timestep embedding\n        temb = None\n\n        # downsampling\n        hs = [self.conv_in(x)]\n        for i_level in range(self.num_resolutions):\n            for i_block in range(self.num_res_blocks):\n                h = self.down[i_level].block[i_block](hs[-1], temb)\n                if len(self.down[i_level].attn) > 0:\n                    h = self.down[i_level].attn[i_block](h)\n                hs.append(h)\n            if i_level != self.num_resolutions-1:\n                hs.append(self.down[i_level].downsample(hs[-1]))\n\n        # middle\n        h = hs[-1]\n        h = self.mid.block_1(h, temb)\n        h = self.mid.attn_1(h)\n        h = self.mid.block_2(h, temb)\n\n        # end\n        h = self.norm_out(h)\n        h = nonlinearity(h)\n        h = self.conv_out(h)\n        return h\n\n\nclass Decoder(nn.Module):\n    def __init__(self, *, ch, out_ch, ch_mult=(1,2,4,8), num_res_blocks,\n                 attn_resolutions, dropout=0.0, resamp_with_conv=True, in_channels,\n                 resolution, z_channels, give_pre_end=False, tanh_out=False, use_linear_attn=False,\n                 attn_type="vanilla", **ignorekwargs):\n        super().__init__()\n        if use_linear_attn: attn_type = "linear"\n        self.ch = ch\n        self.temb_ch = 0\n        self.num_resolutions = len(ch_mult)\n        self.num_res_blocks = num_res_blocks\n        self.resolution = resolution\n        self.in_channels = in_channels\n        self.give_pre_end = give_pre_end\n        self.tanh_out = tanh_out\n\n        # compute in_ch_mult, block_in and curr_res at lowest res\n        in_ch_mult = (1,)+tuple(ch_mult)\n        block_in = ch*ch_mult[self.num_resolutions-1]\n        curr_res = resolution // 2**(self.num_resolutions-1)\n        self.z_shape = (1,z_channels,curr_res,curr_res)\n\n        # z to block_in\n        self.conv_in = torch.nn.Conv2d(z_channels,\n                                       block_in,\n                                       kernel_size=3,\n                                       stride=1,\n                                       padding=1)\n\n        # middle\n        self.mid = nn.Module()\n        self.mid.block_1 = ResnetBlock(in_channels=block_in,\n                                       out_channels=block_in,\n                                       temb_channels=self.temb_ch,\n                                       dropout=dropout)\n        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)\n        self.mid.block_2 = ResnetBlock(in_channels=block_in,\n                                       out_channels=block_in,\n                                       temb_channels=self.temb_ch,\n                                       dropout=dropout)\n\n        # upsampling\n        self.up = nn.ModuleList()\n        for i_level in reversed(range(self.num_resolutions)):\n            block = nn.ModuleList()\n            attn = nn.ModuleList()\n            block_out = ch*ch_mult[i_level]\n            for i_block in range(self.num_res_blocks+1):\n                block.append(ResnetBlock(in_channels=block_in,\n                                         out_channels=block_out,\n                                         temb_channels=self.temb_ch,\n                                         dropout=dropout))\n                block_in = block_out\n                if curr_res in attn_resolutions:\n                    attn.append(make_attn(block_in, attn_type=attn_type))\n            up = nn.Module()\n            up.block = block\n            up.attn = attn\n            if i_level != 0:\n                up.upsample = Upsample(block_in, resamp_with_conv)\n                curr_res = curr_res * 2\n            self.up.insert(0, up) # prepend to get consistent order\n\n        # end\n        self.norm_out = Normalize(block_in)\n        self.conv_out = torch.nn.Conv2d(block_in,\n                                        out_ch,\n                                        kernel_size=3,\n                                        stride=1,\n                                        padding=1)\n\n    def forward(self, z):\n        #assert z.shape[1:] == self.z_shape[1:]\n        self.last_z_shape = z.shape\n\n        # timestep embedding\n        temb = None\n\n        # z to block_in\n        h = self.conv_in(z)\n\n        # middle\n        h = self.mid.block_1(h, temb)\n        h = self.mid.attn_1(h)\n        h = self.mid.block_2(h, temb)\n\n        # upsampling\n        for i_level in reversed(range(self.num_resolutions)):\n            for i_block in range(self.num_res_blocks+1):\n                h = self.up[i_level].block[i_block](h, temb)\n                if len(self.up[i_level].attn) > 0:\n                    h = self.up[i_level].attn[i_block](h)\n            if i_level != 0:\n                h = self.up[i_level].upsample(h)\n\n        # end\n        if self.give_pre_end:\n            return h\n\n        h = self.norm_out(h)\n        h = nonlinearity(h)\n        h = self.conv_out(h)\n        if self.tanh_out:\n            h = torch.tanh(h)\n        return h\n\n\nclass SimpleDecoder(nn.Module):\n    def __init__(self, in_channels, out_channels, *args, **kwargs):\n        super().__init__()\n        self.model = nn.ModuleList([nn.Conv2d(in_channels, in_channels, 1),\n                                     ResnetBlock(in_channels=in_channels,\n                                                 out_channels=2 * in_channels,\n                                                 temb_channels=0, dropout=0.0),\n                                     ResnetBlock(in_channels=2 * in_channels,\n                                                out_channels=4 * in_channels,\n                                                temb_channels=0, dropout=0.0),\n                                     ResnetBlock(in_channels=4 * in_channels,\n                                                out_channels=2 * in_channels,\n                                                temb_channels=0, dropout=0.0),\n                                     nn.Conv2d(2*in_channels, in_channels, 1),\n                                     Upsample(in_channels, with_conv=True)])\n        # end\n        self.norm_out = Normalize(in_channels)\n        self.conv_out = torch.nn.Conv2d(in_channels,\n                                        out_channels,\n                                        kernel_size=3,\n                                        stride=1,\n                                        padding=1)\n\n    def forward(self, x):\n        for i, layer in enumerate(self.model):\n            if i in [1,2,3]:\n                x = layer(x, None)\n            else:\n                x = layer(x)\n\n        h = self.norm_out(x)\n        h = nonlinearity(h)\n        x = self.conv_out(h)\n        return x\n\n\nclass UpsampleDecoder(nn.Module):\n    def __init__(self, in_channels, out_channels, ch, num_res_blocks, resolution,\n                 ch_mult=(2,2), dropout=0.0):\n        super().__init__()\n        # upsampling\n        self.temb_ch = 0\n        self.num_resolutions = len(ch_mult)\n        self.num_res_blocks = num_res_blocks\n        block_in = in_channels\n        curr_res = resolution // 2 ** (self.num_resolutions - 1)\n        self.res_blocks = nn.ModuleList()\n        self.upsample_blocks = nn.ModuleList()\n        for i_level in range(self.num_resolutions):\n            res_block = []\n            block_out = ch * ch_mult[i_level]\n            for i_block in range(self.num_res_blocks + 1):\n                res_block.append(ResnetBlock(in_channels=block_in,\n                                         out_channels=block_out,\n                                         temb_channels=self.temb_ch,\n                                         dropout=dropout))\n                block_in = block_out\n            self.res_blocks.append(nn.ModuleList(res_block))\n            if i_level != self.num_resolutions - 1:\n                self.upsample_blocks.append(Upsample(block_in, True))\n                curr_res = curr_res * 2\n\n        # end\n        self.norm_out = Normalize(block_in)\n        self.conv_out = torch.nn.Conv2d(block_in,\n                                        out_channels,\n                                        kernel_size=3,\n                                        stride=1,\n                                        padding=1)\n\n    def forward(self, x):\n        # upsampling\n        h = x\n        for k, i_level in enumerate(range(self.num_resolutions)):\n            for i_block in range(self.num_res_blocks + 1):\n                h = self.res_blocks[i_level][i_block](h, None)\n            if i_level != self.num_resolutions - 1:\n                h = self.upsample_blocks[k](h)\n        h = self.norm_out(h)\n        h = nonlinearity(h)\n        h = self.conv_out(h)\n        return h\n\n\nclass LatentRescaler(nn.Module):\n    def __init__(self, factor, in_channels, mid_channels, out_channels, depth=2):\n        super().__init__()\n        # residual block, interpolate, residual block\n        self.factor = factor\n        self.conv_in = nn.Conv2d(in_channels,\n                                 mid_channels,\n                                 kernel_size=3,\n                                 stride=1,\n                                 padding=1)\n        self.res_block1 = nn.ModuleList([ResnetBlock(in_channels=mid_channels,\n                                                     out_channels=mid_channels,\n                                                     temb_channels=0,\n                                                     dropout=0.0) for _ in range(depth)])\n        self.attn = AttnBlock(mid_channels)\n        self.res_block2 = nn.ModuleList([ResnetBlock(in_channels=mid_channels,\n                                                     out_channels=mid_channels,\n                                                     temb_channels=0,\n                                                     dropout=0.0) for _ in range(depth)])\n\n        self.conv_out = nn.Conv2d(mid_channels,\n                                  out_channels,\n                                  kernel_size=1,\n                                  )\n\n    def forward(self, x):\n        x = self.conv_in(x)\n        for block in self.res_block1:\n            x = block(x, None)\n        x = torch.nn.functional.interpolate(x, size=(int(round(x.shape[2]*self.factor)), int(round(x.shape[3]*self.factor))))\n        x = self.attn(x)\n        for block in self.res_block2:\n            x = block(x, None)\n        x = self.conv_out(x)\n        return x\n\n\nclass MergedRescaleEncoder(nn.Module):\n    def __init__(self, in_channels, ch, resolution, out_ch, num_res_blocks,\n                 attn_resolutions, dropout=0.0, resamp_with_conv=True,\n                 ch_mult=(1,2,4,8), rescale_factor=1.0, rescale_module_depth=1):\n        super().__init__()\n        intermediate_chn = ch * ch_mult[-1]\n        self.encoder = Encoder(in_channels=in_channels, num_res_blocks=num_res_blocks, ch=ch, ch_mult=ch_mult,\n                               z_channels=intermediate_chn, double_z=False, resolution=resolution,\n                               attn_resolutions=attn_resolutions, dropout=dropout, resamp_with_conv=resamp_with_conv,\n                               out_ch=None)\n        self.rescaler = LatentRescaler(factor=rescale_factor, in_channels=intermediate_chn,\n                                       mid_channels=intermediate_chn, out_channels=out_ch, depth=rescale_module_depth)\n\n    def forward(self, x):\n        x = self.encoder(x)\n        x = self.rescaler(x)\n        return x\n\n\nclass MergedRescaleDecoder(nn.Module):\n    def __init__(self, z_channels, out_ch, resolution, num_res_blocks, attn_resolutions, ch, ch_mult=(1,2,4,8),\n                 dropout=0.0, resamp_with_conv=True, rescale_factor=1.0, rescale_module_depth=1):\n        super().__init__()\n        tmp_chn = z_channels*ch_mult[-1]\n        self.decoder = Decoder(out_ch=out_ch, z_channels=tmp_chn, attn_resolutions=attn_resolutions, dropout=dropout,\n                               resamp_with_conv=resamp_with_conv, in_channels=None, num_res_blocks=num_res_blocks,\n                               ch_mult=ch_mult, resolution=resolution, ch=ch)\n        self.rescaler = LatentRescaler(factor=rescale_factor, in_channels=z_channels, mid_channels=tmp_chn,\n                                       out_channels=tmp_chn, depth=rescale_module_depth)\n\n    def forward(self, x):\n        x = self.rescaler(x)\n        x = self.decoder(x)\n        return x\n\n\nclass Upsampler(nn.Module):\n    def __init__(self, in_size, out_size, in_channels, out_channels, ch_mult=2):\n        super().__init__()\n        assert out_size >= in_size\n        num_blocks = int(np.log2(out_size//in_size))+1\n        factor_up = 1.+ (out_size % in_size)\n        print(f"Building {self.__class__.__name__} with in_size: {in_size} --> out_size {out_size} and factor {factor_up}")\n        self.rescaler = LatentRescaler(factor=factor_up, in_channels=in_channels, mid_channels=2*in_channels,\n                                       out_channels=in_channels)\n        self.decoder = Decoder(out_ch=out_channels, resolution=out_size, z_channels=in_channels, num_res_blocks=2,\n                               attn_resolutions=[], in_channels=None, ch=in_channels,\n                               ch_mult=[ch_mult for _ in range(num_blocks)])\n\n    def forward(self, x):\n        x = self.rescaler(x)\n        x = self.decoder(x)\n        return x\n\n\nclass Resize(nn.Module):\n    def __init__(self, in_channels=None, learned=False, mode="bilinear"):\n        super().__init__()\n        self.with_conv = learned\n        self.mode = mode\n        if self.with_conv:\n            print(f"Note: {self.__class__.__name} uses learned downsampling and will ignore the fixed {mode} mode")\n            raise NotImplementedError()\n            assert in_channels is not None\n            # no asymmetric padding in torch conv, must do it ourselves\n            self.conv = torch.nn.Conv2d(in_channels,\n                                        in_channels,\n                                        kernel_size=4,\n                                        stride=2,\n                                        padding=1)\n\n    def forward(self, x, scale_factor=1.0):\n        if scale_factor==1.0:\n            return x\n        else:\n            x = torch.nn.functional.interpolate(x, mode=self.mode, align_corners=False, scale_factor=scale_factor)\n        return x\n', 'model/IDM/modules/diffusionmodules/openaimodel.py': 'from abc import abstractmethod\nimport math\nimport numpy as np\nimport torch as th\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom model.IDM.modules.diffusionmodules.util import (\n    checkpoint,\n    conv_nd,\n    linear,\n    avg_pool_nd,\n    zero_module,\n    normalization,\n    timestep_embedding,\n)\nfrom model.IDM.modules.attention import SpatialTransformer\n\n\n# dummy replace\ndef convert_module_to_f16(x):\n    pass\n\ndef convert_module_to_f32(x):\n    pass\n\n\nclass AttentionPool2d(nn.Module):\n    """\n    Adapted from CLIP: https://github.com/openai/CLIP/blob/main/clip/model.py\n    """\n\n    def __init__(\n        self,\n        spacial_dim: int,\n        embed_dim: int,\n        num_heads_channels: int,\n        output_dim: int = None,\n    ):\n        super().__init__()\n        self.positional_embedding = nn.Parameter(th.randn(embed_dim, spacial_dim ** 2 + 1) / embed_dim ** 0.5)\n        self.qkv_proj = conv_nd(1, embed_dim, 3 * embed_dim, 1)\n        self.c_proj = conv_nd(1, embed_dim, output_dim or embed_dim, 1)\n        self.num_heads = embed_dim // num_heads_channels\n        self.attention = QKVAttention(self.num_heads)\n\n    def forward(self, x):\n        b, c, *_spatial = x.shape\n        x = x.reshape(b, c, -1)  # NC(HW)\n        x = th.cat([x.mean(dim=-1, keepdim=True), x], dim=-1)  # NC(HW+1)\n        x = x + self.positional_embedding[None, :, :].to(x.dtype)  # NC(HW+1)\n        x = self.qkv_proj(x)\n        x = self.attention(x)\n        x = self.c_proj(x)\n        return x[:, :, 0]\n\n\nclass TimestepBlock(nn.Module):\n    """\n    Any module where forward() takes timestep embeddings as a second argument.\n    """\n\n    @abstractmethod\n    def forward(self, x, emb):\n        """\n        Apply the module to `x` given `emb` timestep embeddings.\n        """\n\n\nclass TimestepEmbedSequential(nn.Sequential, TimestepBlock):\n    """\n    A sequential module that passes timestep embeddings to the children that\n    support it as an extra input.\n    """\n\n    def forward(self, x, emb, context=None):\n        for layer in self:\n            if isinstance(layer, TimestepBlock):\n                x = layer(x, emb)\n            elif isinstance(layer, SpatialTransformer):\n                x = layer(x, context)\n            else:\n                x = layer(x)\n        return x\n\n\nclass Upsample(nn.Module):\n    """\n    An upsampling layer with an optional convolution.\n    :param channels: channels in the inputs and outputs.\n    :param use_conv: a bool determining if a convolution is applied.\n    :param dims: determines if the signal is 1D, 2D, or 3D. If 3D, then\n                 upsampling occurs in the inner-two dimensions.\n    """\n\n    def __init__(self, channels, use_conv, dims=2, out_channels=None, padding=1):\n        super().__init__()\n        self.channels = channels\n        self.out_channels = out_channels or channels\n        self.use_conv = use_conv\n        self.dims = dims\n        if use_conv:\n            self.conv = conv_nd(dims, self.channels, self.out_channels, 3, padding=padding)\n\n    def forward(self, x):\n        assert x.shape[1] == self.channels\n        if self.dims == 3:\n            x = F.interpolate(\n                x, (x.shape[2], x.shape[3] * 2, x.shape[4] * 2), mode="nearest"\n            )\n        else:\n            x = F.interpolate(x, scale_factor=2, mode="nearest")\n        if self.use_conv:\n            x = self.conv(x)\n        return x\n\nclass TransposedUpsample(nn.Module):\n    \'Learned 2x upsampling without padding\'\n    def __init__(self, channels, out_channels=None, ks=5):\n        super().__init__()\n        self.channels = channels\n        self.out_channels = out_channels or channels\n\n        self.up = nn.ConvTranspose2d(self.channels,self.out_channels,kernel_size=ks,stride=2)\n\n    def forward(self,x):\n        return self.up(x)\n\n\nclass Downsample(nn.Module):\n    """\n    A downsampling layer with an optional convolution.\n    :param channels: channels in the inputs and outputs.\n    :param use_conv: a bool determining if a convolution is applied.\n    :param dims: determines if the signal is 1D, 2D, or 3D. If 3D, then\n                 downsampling occurs in the inner-two dimensions.\n    """\n\n    def __init__(self, channels, use_conv, dims=2, out_channels=None,padding=1):\n        super().__init__()\n        self.channels = channels\n        self.out_channels = out_channels or channels\n        self.use_conv = use_conv\n        self.dims = dims\n        stride = 2 if dims != 3 else (1, 2, 2)\n        if use_conv:\n            self.op = conv_nd(\n                dims, self.channels, self.out_channels, 3, stride=stride, padding=padding\n            )\n        else:\n            assert self.channels == self.out_channels\n            self.op = avg_pool_nd(dims, kernel_size=stride, stride=stride)\n\n    def forward(self, x):\n        assert x.shape[1] == self.channels\n        return self.op(x)\n\n\nclass ResBlock(TimestepBlock):\n    """\n    A residual block that can optionally change the number of channels.\n    :param channels: the number of input channels.\n    :param emb_channels: the number of timestep embedding channels.\n    :param dropout: the rate of dropout.\n    :param out_channels: if specified, the number of out channels.\n    :param use_conv: if True and out_channels is specified, use a spatial\n        convolution instead of a smaller 1x1 convolution to change the\n        channels in the skip connection.\n    :param dims: determines if the signal is 1D, 2D, or 3D.\n    :param use_checkpoint: if True, use gradient checkpointing on this module.\n    :param up: if True, use this block for upsampling.\n    :param down: if True, use this block for downsampling.\n    """\n\n    def __init__(\n        self,\n        channels,\n        emb_channels,\n        dropout,\n        out_channels=None,\n        use_conv=False,\n        use_scale_shift_norm=False,\n        dims=2,\n        use_checkpoint=False,\n        up=False,\n        down=False,\n    ):\n        super().__init__()\n        self.channels = channels\n        self.emb_channels = emb_channels\n        self.dropout = dropout\n        self.out_channels = out_channels or channels\n        self.use_conv = use_conv\n        self.use_checkpoint = use_checkpoint\n        self.use_scale_shift_norm = use_scale_shift_norm\n\n        self.in_layers = nn.Sequential(\n            normalization(channels),\n            nn.SiLU(),\n            conv_nd(dims, channels, self.out_channels, 3, padding=1),\n        )\n\n        self.updown = up or down\n\n        if up:\n            self.h_upd = Upsample(channels, False, dims)\n            self.x_upd = Upsample(channels, False, dims)\n        elif down:\n            self.h_upd = Downsample(channels, False, dims)\n            self.x_upd = Downsample(channels, False, dims)\n        else:\n            self.h_upd = self.x_upd = nn.Identity()\n\n        self.emb_layers = nn.Sequential(\n            nn.SiLU(),\n            linear(\n                emb_channels,\n                2 * self.out_channels if use_scale_shift_norm else self.out_channels,\n            ),\n        )\n        self.out_layers = nn.Sequential(\n            normalization(self.out_channels),\n            nn.SiLU(),\n            nn.Dropout(p=dropout),\n            zero_module(\n                conv_nd(dims, self.out_channels, self.out_channels, 3, padding=1)\n            ),\n        )\n\n        if self.out_channels == channels:\n            self.skip_connection = nn.Identity()\n        elif use_conv:\n            self.skip_connection = conv_nd(\n                dims, channels, self.out_channels, 3, padding=1\n            )\n        else:\n            self.skip_connection = conv_nd(dims, channels, self.out_channels, 1)\n\n    def forward(self, x, emb):\n        """\n        Apply the block to a Tensor, conditioned on a timestep embedding.\n        :param x: an [N x C x ...] Tensor of features.\n        :param emb: an [N x emb_channels] Tensor of timestep embeddings.\n        :return: an [N x C x ...] Tensor of outputs.\n        """\n        return checkpoint(\n            self._forward, (x, emb), self.parameters(), self.use_checkpoint\n        )\n\n\n    def _forward(self, x, emb):\n        if self.updown:\n            in_rest, in_conv = self.in_layers[:-1], self.in_layers[-1]\n            h = in_rest(x)\n            h = self.h_upd(h)\n            x = self.x_upd(x)\n            h = in_conv(h)\n        else:\n            h = self.in_layers(x)\n        emb_out = self.emb_layers(emb).type(h.dtype)\n        while len(emb_out.shape) < len(h.shape):\n            emb_out = emb_out[..., None]\n        if self.use_scale_shift_norm:\n            out_norm, out_rest = self.out_layers[0], self.out_layers[1:]\n            scale, shift = th.chunk(emb_out, 2, dim=1)\n            h = out_norm(h) * (1 + scale) + shift\n            h = out_rest(h)\n        else:\n            h = h + emb_out\n            h = self.out_layers(h)\n        return self.skip_connection(x) + h\n\n\nclass AttentionBlock(nn.Module):\n    """\n    An attention block that allows spatial positions to attend to each other.\n    Originally ported from here, but adapted to the N-d case.\n    https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/models/unet.py#L66.\n    """\n\n    def __init__(\n        self,\n        channels,\n        num_heads=1,\n        num_head_channels=-1,\n        use_checkpoint=False,\n        use_new_attention_order=False,\n    ):\n        super().__init__()\n        self.channels = channels\n        if num_head_channels == -1:\n            self.num_heads = num_heads\n        else:\n            assert (\n                channels % num_head_channels == 0\n            ), f"q,k,v channels {channels} is not divisible by num_head_channels {num_head_channels}"\n            self.num_heads = channels // num_head_channels\n        self.use_checkpoint = use_checkpoint\n        self.norm = normalization(channels)\n        self.qkv = conv_nd(1, channels, channels * 3, 1)\n        if use_new_attention_order:\n            # split qkv before split heads\n            self.attention = QKVAttention(self.num_heads)\n        else:\n            # split heads before split qkv\n            self.attention = QKVAttentionLegacy(self.num_heads)\n\n        self.proj_out = zero_module(conv_nd(1, channels, channels, 1))\n\n    def forward(self, x):\n        return checkpoint(self._forward, (x,), self.parameters(), True)   # TODO: check checkpoint usage, is True # TODO: fix the .half call!!!\n        #return pt_checkpoint(self._forward, x)  # pytorch\n\n    def _forward(self, x):\n        b, c, *spatial = x.shape\n        x = x.reshape(b, c, -1)\n        qkv = self.qkv(self.norm(x))\n        h = self.attention(qkv)\n        h = self.proj_out(h)\n        return (x + h).reshape(b, c, *spatial)\n\n\ndef count_flops_attn(model, _x, y):\n    """\n    A counter for the `thop` package to count the operations in an\n    attention operation.\n    Meant to be used like:\n        macs, params = thop.profile(\n            model,\n            inputs=(inputs, timestamps),\n            custom_ops={QKVAttention: QKVAttention.count_flops},\n        )\n    """\n    b, c, *spatial = y[0].shape\n    num_spatial = int(np.prod(spatial))\n    # We perform two matmuls with the same number of ops.\n    # The first computes the weight matrix, the second computes\n    # the combination of the value vectors.\n    matmul_ops = 2 * b * (num_spatial ** 2) * c\n    model.total_ops += th.DoubleTensor([matmul_ops])\n\n\nclass QKVAttentionLegacy(nn.Module):\n    """\n    A module which performs QKV attention. Matches legacy QKVAttention + input/ouput heads shaping\n    """\n\n    def __init__(self, n_heads):\n        super().__init__()\n        self.n_heads = n_heads\n\n    def forward(self, qkv):\n        """\n        Apply QKV attention.\n        :param qkv: an [N x (H * 3 * C) x T] tensor of Qs, Ks, and Vs.\n        :return: an [N x (H * C) x T] tensor after attention.\n        """\n        bs, width, length = qkv.shape\n        assert width % (3 * self.n_heads) == 0\n        ch = width // (3 * self.n_heads)\n        q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)\n        scale = 1 / math.sqrt(math.sqrt(ch))\n        weight = th.einsum(\n            "bct,bcs->bts", q * scale, k * scale\n        )  # More stable with f16 than dividing afterwards\n        weight = th.softmax(weight.float(), dim=-1).type(weight.dtype)\n        a = th.einsum("bts,bcs->bct", weight, v)\n        return a.reshape(bs, -1, length)\n\n    @staticmethod\n    def count_flops(model, _x, y):\n        return count_flops_attn(model, _x, y)\n\n\nclass QKVAttention(nn.Module):\n    """\n    A module which performs QKV attention and splits in a different order.\n    """\n\n    def __init__(self, n_heads):\n        super().__init__()\n        self.n_heads = n_heads\n\n    def forward(self, qkv):\n        """\n        Apply QKV attention.\n        :param qkv: an [N x (3 * H * C) x T] tensor of Qs, Ks, and Vs.\n        :return: an [N x (H * C) x T] tensor after attention.\n        """\n        bs, width, length = qkv.shape\n        assert width % (3 * self.n_heads) == 0\n        ch = width // (3 * self.n_heads)\n        q, k, v = qkv.chunk(3, dim=1)\n        scale = 1 / math.sqrt(math.sqrt(ch))\n        weight = th.einsum(\n            "bct,bcs->bts",\n            (q * scale).view(bs * self.n_heads, ch, length),\n            (k * scale).view(bs * self.n_heads, ch, length),\n        )  # More stable with f16 than dividing afterwards\n        weight = th.softmax(weight.float(), dim=-1).type(weight.dtype)\n        a = th.einsum("bts,bcs->bct", weight, v.reshape(bs * self.n_heads, ch, length))\n        return a.reshape(bs, -1, length)\n\n    @staticmethod\n    def count_flops(model, _x, y):\n        return count_flops_attn(model, _x, y)\n\n\nclass UNetModel(nn.Module):\n    """\n    The full UNet model with attention and timestep embedding.\n    :param in_channels: channels in the input Tensor.\n    :param model_channels: base channel count for the model.\n    :param out_channels: channels in the output Tensor.\n    :param num_res_blocks: number of residual blocks per downsample.\n    :param attention_resolutions: a collection of downsample rates at which\n        attention will take place. May be a set, list, or tuple.\n        For example, if this contains 4, then at 4x downsampling, attention\n        will be used.\n    :param dropout: the dropout probability.\n    :param channel_mult: channel multiplier for each level of the UNet.\n    :param conv_resample: if True, use learned convolutions for upsampling and\n        downsampling.\n    :param dims: determines if the signal is 1D, 2D, or 3D.\n    :param num_classes: if specified (as an int), then this model will be\n        class-conditional with `num_classes` classes.\n    :param use_checkpoint: use gradient checkpointing to reduce memory usage.\n    :param num_heads: the number of attention heads in each attention layer.\n    :param num_heads_channels: if specified, ignore num_heads and instead use\n                               a fixed channel width per attention head.\n    :param num_heads_upsample: works with num_heads to set a different number\n                               of heads for upsampling. Deprecated.\n    :param use_scale_shift_norm: use a FiLM-like conditioning mechanism.\n    :param resblock_updown: use residual blocks for up/downsampling.\n    :param use_new_attention_order: use a different attention pattern for potentially\n                                    increased efficiency.\n    """\n\n    def __init__(\n        self,\n        image_size,\n        in_channels,\n        model_channels,\n        out_channels,\n        num_res_blocks,\n        attention_resolutions,\n        dropout=0,\n        channel_mult=(1, 2, 4, 8),\n        conv_resample=True,\n        dims=2,\n        num_classes=None,\n        use_checkpoint=False,\n        use_fp16=False,\n        num_heads=-1,\n        num_head_channels=-1,\n        num_heads_upsample=-1,\n        use_scale_shift_norm=False,\n        resblock_updown=False,\n        use_new_attention_order=False,\n        use_spatial_transformer=False,    # custom transformer support\n        transformer_depth=1,              # custom transformer support\n        context_dim=None,                 # custom transformer support\n        n_embed=None,                     # custom support for prediction of discrete ids into codebook of first stage vq model\n        legacy=True,\n    ):\n        super().__init__()\n        if use_spatial_transformer:\n            assert context_dim is not None, \'Fool!! You forgot to include the dimension of your cross-attention conditioning...\'\n\n        if context_dim is not None:\n            assert use_spatial_transformer, \'Fool!! You forgot to use the spatial transformer for your cross-attention conditioning...\'\n            from omegaconf.listconfig import ListConfig\n            if type(context_dim) == ListConfig:\n                context_dim = list(context_dim)\n\n        if num_heads_upsample == -1:\n            num_heads_upsample = num_heads\n\n        if num_heads == -1:\n            assert num_head_channels != -1, \'Either num_heads or num_head_channels has to be set\'\n\n        if num_head_channels == -1:\n            assert num_heads != -1, \'Either num_heads or num_head_channels has to be set\'\n\n        self.image_size = image_size\n        self.in_channels = in_channels\n        self.model_channels = model_channels\n        self.out_channels = out_channels\n        self.num_res_blocks = num_res_blocks\n        self.attention_resolutions = attention_resolutions\n        self.dropout = dropout\n        self.channel_mult = channel_mult\n        self.conv_resample = conv_resample\n        self.num_classes = num_classes\n        self.use_checkpoint = use_checkpoint\n        self.dtype = th.float16 if use_fp16 else th.float32\n        self.num_heads = num_heads\n        self.num_head_channels = num_head_channels\n        self.num_heads_upsample = num_heads_upsample\n        self.predict_codebook_ids = n_embed is not None\n\n        time_embed_dim = model_channels * 4\n        self.time_embed = nn.Sequential(\n            linear(model_channels, time_embed_dim),\n            nn.SiLU(),\n            linear(time_embed_dim, time_embed_dim),\n        )\n\n        if self.num_classes is not None:\n            self.label_emb = nn.Embedding(num_classes, time_embed_dim)\n\n        self.input_blocks = nn.ModuleList(\n            [\n                TimestepEmbedSequential(\n                    conv_nd(dims, in_channels, model_channels, 3, padding=1)\n                )\n            ]\n        )\n        self._feature_size = model_channels\n        input_block_chans = [model_channels]\n        ch = model_channels\n        ds = 1\n        for level, mult in enumerate(channel_mult):\n            for _ in range(num_res_blocks):\n                layers = [\n                    ResBlock(\n                        ch,\n                        time_embed_dim,\n                        dropout,\n                        out_channels=mult * model_channels,\n                        dims=dims,\n                        use_checkpoint=use_checkpoint,\n                        use_scale_shift_norm=use_scale_shift_norm,\n                    )\n                ]\n                ch = mult * model_channels\n                if ds in attention_resolutions:\n                    if num_head_channels == -1:\n                        dim_head = ch // num_heads\n                    else:\n                        num_heads = ch // num_head_channels\n                        dim_head = num_head_channels\n                    if legacy:\n                        #num_heads = 1\n                        dim_head = ch // num_heads if use_spatial_transformer else num_head_channels\n                    layers.append(\n                        AttentionBlock(\n                            ch,\n                            use_checkpoint=use_checkpoint,\n                            num_heads=num_heads,\n                            num_head_channels=dim_head,\n                            use_new_attention_order=use_new_attention_order,\n                        ) if not use_spatial_transformer else SpatialTransformer(\n                            ch, num_heads, dim_head, depth=transformer_depth, context_dim=context_dim\n                        )\n                    )\n                self.input_blocks.append(TimestepEmbedSequential(*layers))\n                self._feature_size += ch\n                input_block_chans.append(ch)\n            if level != len(channel_mult) - 1:\n                out_ch = ch\n                self.input_blocks.append(\n                    TimestepEmbedSequential(\n                        ResBlock(\n                            ch,\n                            time_embed_dim,\n                            dropout,\n                            out_channels=out_ch,\n                            dims=dims,\n                            use_checkpoint=use_checkpoint,\n                            use_scale_shift_norm=use_scale_shift_norm,\n                            down=True,\n                        )\n                        if resblock_updown\n                        else Downsample(\n                            ch, conv_resample, dims=dims, out_channels=out_ch\n                        )\n                    )\n                )\n                ch = out_ch\n                input_block_chans.append(ch)\n                ds *= 2\n                self._feature_size += ch\n\n        if num_head_channels == -1:\n            dim_head = ch // num_heads\n        else:\n            num_heads = ch // num_head_channels\n            dim_head = num_head_channels\n        if legacy:\n            #num_heads = 1\n            dim_head = ch // num_heads if use_spatial_transformer else num_head_channels\n        self.middle_block = TimestepEmbedSequential(\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n            AttentionBlock(\n                ch,\n                use_checkpoint=use_checkpoint,\n                num_heads=num_heads,\n                num_head_channels=dim_head,\n                use_new_attention_order=use_new_attention_order,\n            ) if not use_spatial_transformer else SpatialTransformer(\n                            ch, num_heads, dim_head, depth=transformer_depth, context_dim=context_dim\n                        ),\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n        )\n        self._feature_size += ch\n\n        self.output_blocks = nn.ModuleList([])\n        for level, mult in list(enumerate(channel_mult))[::-1]:\n            for i in range(num_res_blocks + 1):\n                ich = input_block_chans.pop()\n                layers = [\n                    ResBlock(\n                        ch + ich,\n                        time_embed_dim,\n                        dropout,\n                        out_channels=model_channels * mult,\n                        dims=dims,\n                        use_checkpoint=use_checkpoint,\n                        use_scale_shift_norm=use_scale_shift_norm,\n                    )\n                ]\n                ch = model_channels * mult\n                if ds in attention_resolutions:\n                    if num_head_channels == -1:\n                        dim_head = ch // num_heads\n                    else:\n                        num_heads = ch // num_head_channels\n                        dim_head = num_head_channels\n                    if legacy:\n                        #num_heads = 1\n                        dim_head = ch // num_heads if use_spatial_transformer else num_head_channels\n                    layers.append(\n                        AttentionBlock(\n                            ch,\n                            use_checkpoint=use_checkpoint,\n                            num_heads=num_heads_upsample,\n                            num_head_channels=dim_head,\n                            use_new_attention_order=use_new_attention_order,\n                        ) if not use_spatial_transformer else SpatialTransformer(\n                            ch, num_heads, dim_head, depth=transformer_depth, context_dim=context_dim\n                        )\n                    )\n                if level and i == num_res_blocks:\n                    out_ch = ch\n                    layers.append(\n                        ResBlock(\n                            ch,\n                            time_embed_dim,\n                            dropout,\n                            out_channels=out_ch,\n                            dims=dims,\n                            use_checkpoint=use_checkpoint,\n                            use_scale_shift_norm=use_scale_shift_norm,\n                            up=True,\n                        )\n                        if resblock_updown\n                        else Upsample(ch, conv_resample, dims=dims, out_channels=out_ch)\n                    )\n                    ds //= 2\n                self.output_blocks.append(TimestepEmbedSequential(*layers))\n                self._feature_size += ch\n\n        self.out = nn.Sequential(\n            normalization(ch),\n            nn.SiLU(),\n            zero_module(conv_nd(dims, model_channels, out_channels, 3, padding=1)),\n        )\n        if self.predict_codebook_ids:\n            self.id_predictor = nn.Sequential(\n            normalization(ch),\n            conv_nd(dims, model_channels, n_embed, 1),\n            #nn.LogSoftmax(dim=1)  # change to cross_entropy and produce non-normalized logits\n        )\n\n    def convert_to_fp16(self):\n        """\n        Convert the torso of the model to float16.\n        """\n        self.input_blocks.apply(convert_module_to_f16)\n        self.middle_block.apply(convert_module_to_f16)\n        self.output_blocks.apply(convert_module_to_f16)\n\n    def convert_to_fp32(self):\n        """\n        Convert the torso of the model to float32.\n        """\n        self.input_blocks.apply(convert_module_to_f32)\n        self.middle_block.apply(convert_module_to_f32)\n        self.output_blocks.apply(convert_module_to_f32)\n\n    def forward(self, x, timesteps=None, context=None, y=None,**kwargs):\n        """\n        Apply the model to an input batch.\n        :param x: an [N x C x ...] Tensor of inputs.\n        :param timesteps: a 1-D batch of timesteps.\n        :param context: conditioning plugged in via crossattn\n        :param y: an [N] Tensor of labels, if class-conditional.\n        :return: an [N x C x ...] Tensor of outputs.\n        """\n        assert (y is not None) == (\n            self.num_classes is not None\n        ), "must specify y if and only if the model is class-conditional"\n        hs = []\n        t_emb = timestep_embedding(timesteps, self.model_channels, repeat_only=False)\n        emb = self.time_embed(t_emb)\n\n        if self.num_classes is not None:\n            assert y.shape == (x.shape[0],)\n            emb = emb + self.label_emb(y)\n\n        h = x.type(self.dtype)\n        for module in self.input_blocks:\n            h = module(h, emb, context)\n            hs.append(h)\n        h = self.middle_block(h, emb, context)\n        for module in self.output_blocks:\n            h = th.cat([h, hs.pop()], dim=1)\n            h = module(h, emb, context)\n        h = h.type(x.dtype)\n        if self.predict_codebook_ids:\n            return self.id_predictor(h)\n        else:\n            return self.out(h)\n\n\nclass EncoderUNetModel(nn.Module):\n    """\n    The half UNet model with attention and timestep embedding.\n    For usage, see UNet.\n    """\n\n    def __init__(\n        self,\n        image_size,\n        in_channels,\n        model_channels,\n        out_channels,\n        num_res_blocks,\n        attention_resolutions,\n        dropout=0,\n        channel_mult=(1, 2, 4, 8),\n        conv_resample=True,\n        dims=2,\n        use_checkpoint=False,\n        use_fp16=False,\n        num_heads=1,\n        num_head_channels=-1,\n        num_heads_upsample=-1,\n        use_scale_shift_norm=False,\n        resblock_updown=False,\n        use_new_attention_order=False,\n        pool="adaptive",\n        *args,\n        **kwargs\n    ):\n        super().__init__()\n\n        if num_heads_upsample == -1:\n            num_heads_upsample = num_heads\n\n        self.in_channels = in_channels\n        self.model_channels = model_channels\n        self.out_channels = out_channels\n        self.num_res_blocks = num_res_blocks\n        self.attention_resolutions = attention_resolutions\n        self.dropout = dropout\n        self.channel_mult = channel_mult\n        self.conv_resample = conv_resample\n        self.use_checkpoint = use_checkpoint\n        self.dtype = th.float16 if use_fp16 else th.float32\n        self.num_heads = num_heads\n        self.num_head_channels = num_head_channels\n        self.num_heads_upsample = num_heads_upsample\n\n        time_embed_dim = model_channels * 4\n        self.time_embed = nn.Sequential(\n            linear(model_channels, time_embed_dim),\n            nn.SiLU(),\n            linear(time_embed_dim, time_embed_dim),\n        )\n\n        self.input_blocks = nn.ModuleList(\n            [\n                TimestepEmbedSequential(\n                    conv_nd(dims, in_channels, model_channels, 3, padding=1)\n                )\n            ]\n        )\n        self._feature_size = model_channels\n        input_block_chans = [model_channels]\n        ch = model_channels\n        ds = 1\n        for level, mult in enumerate(channel_mult):\n            for _ in range(num_res_blocks):\n                layers = [\n                    ResBlock(\n                        ch,\n                        time_embed_dim,\n                        dropout,\n                        out_channels=mult * model_channels,\n                        dims=dims,\n                        use_checkpoint=use_checkpoint,\n                        use_scale_shift_norm=use_scale_shift_norm,\n                    )\n                ]\n                ch = mult * model_channels\n                if ds in attention_resolutions:\n                    layers.append(\n                        AttentionBlock(\n                            ch,\n                            use_checkpoint=use_checkpoint,\n                            num_heads=num_heads,\n                            num_head_channels=num_head_channels,\n                            use_new_attention_order=use_new_attention_order,\n                        )\n                    )\n                self.input_blocks.append(TimestepEmbedSequential(*layers))\n                self._feature_size += ch\n                input_block_chans.append(ch)\n            if level != len(channel_mult) - 1:\n                out_ch = ch\n                self.input_blocks.append(\n                    TimestepEmbedSequential(\n                        ResBlock(\n                            ch,\n                            time_embed_dim,\n                            dropout,\n                            out_channels=out_ch,\n                            dims=dims,\n                            use_checkpoint=use_checkpoint,\n                            use_scale_shift_norm=use_scale_shift_norm,\n                            down=True,\n                        )\n                        if resblock_updown\n                        else Downsample(\n                            ch, conv_resample, dims=dims, out_channels=out_ch\n                        )\n                    )\n                )\n                ch = out_ch\n                input_block_chans.append(ch)\n                ds *= 2\n                self._feature_size += ch\n\n        self.middle_block = TimestepEmbedSequential(\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n            AttentionBlock(\n                ch,\n                use_checkpoint=use_checkpoint,\n                num_heads=num_heads,\n                num_head_channels=num_head_channels,\n                use_new_attention_order=use_new_attention_order,\n            ),\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n        )\n        self._feature_size += ch\n        self.pool = pool\n        if pool == "adaptive":\n            self.out = nn.Sequential(\n                normalization(ch),\n                nn.SiLU(),\n                nn.AdaptiveAvgPool2d((1, 1)),\n                zero_module(conv_nd(dims, ch, out_channels, 1)),\n                nn.Flatten(),\n            )\n        elif pool == "attention":\n            assert num_head_channels != -1\n            self.out = nn.Sequential(\n                normalization(ch),\n                nn.SiLU(),\n                AttentionPool2d(\n                    (image_size // ds), ch, num_head_channels, out_channels\n                ),\n            )\n        elif pool == "spatial":\n            self.out = nn.Sequential(\n                nn.Linear(self._feature_size, 2048),\n                nn.ReLU(),\n                nn.Linear(2048, self.out_channels),\n            )\n        elif pool == "spatial_v2":\n            self.out = nn.Sequential(\n                nn.Linear(self._feature_size, 2048),\n                normalization(2048),\n                nn.SiLU(),\n                nn.Linear(2048, self.out_channels),\n            )\n        else:\n            raise NotImplementedError(f"Unexpected {pool} pooling")\n\n    def convert_to_fp16(self):\n        """\n        Convert the torso of the model to float16.\n        """\n        self.input_blocks.apply(convert_module_to_f16)\n        self.middle_block.apply(convert_module_to_f16)\n\n    def convert_to_fp32(self):\n        """\n        Convert the torso of the model to float32.\n        """\n        self.input_blocks.apply(convert_module_to_f32)\n        self.middle_block.apply(convert_module_to_f32)\n\n    def forward(self, x, timesteps):\n        """\n        Apply the model to an input batch.\n        :param x: an [N x C x ...] Tensor of inputs.\n        :param timesteps: a 1-D batch of timesteps.\n        :return: an [N x K] Tensor of outputs.\n        """\n        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))\n\n        results = []\n        h = x.type(self.dtype)\n        for module in self.input_blocks:\n            h = module(h, emb)\n            if self.pool.startswith("spatial"):\n                results.append(h.type(x.dtype).mean(dim=(2, 3)))\n        h = self.middle_block(h, emb)\n        if self.pool.startswith("spatial"):\n            results.append(h.type(x.dtype).mean(dim=(2, 3)))\n            h = th.cat(results, axis=-1)\n            return self.out(h)\n        else:\n            h = h.type(x.dtype)\n            return self.out(h)\n\n\n\n# For encode the ocr feature output\nclass TextFeature_EncoderUNetModel(nn.Module):\n    """\n    The half UNet model with attention and timestep embedding.\n    For usage, see UNet.\n    """\n\n    def __init__(\n        self,\n        image_size,\n        in_channels,\n        model_channels,\n        out_channels,\n        num_res_blocks,\n        attention_resolutions,\n        dropout=0,\n        channel_mult=(1, 2, 4, 8),\n        conv_resample=True,\n        dims=2,\n        use_checkpoint=False,\n        use_fp16=False,\n        num_heads=1,\n        num_head_channels=-1,\n        num_heads_upsample=-1,\n        use_scale_shift_norm=False,\n        resblock_updown=False,\n        use_new_attention_order=False,\n        *args,\n        **kwargs\n    ):\n        super().__init__()\n\n        if num_heads_upsample == -1:\n            num_heads_upsample = num_heads\n\n        self.in_channels = in_channels\n        self.model_channels = model_channels\n        self.out_channels = out_channels\n        self.num_res_blocks = num_res_blocks\n        self.attention_resolutions = attention_resolutions\n        self.dropout = dropout\n        self.channel_mult = channel_mult\n        self.conv_resample = conv_resample\n        self.use_checkpoint = use_checkpoint\n        self.dtype = th.float16 if use_fp16 else th.float32\n        self.num_heads = num_heads\n        self.num_head_channels = num_head_channels\n        self.num_heads_upsample = num_heads_upsample\n\n        time_embed_dim = model_channels * 4\n        self.time_embed = nn.Sequential(\n            linear(model_channels, time_embed_dim),\n            nn.SiLU(),\n            linear(time_embed_dim, time_embed_dim),\n        )\n\n        self.input_blocks = nn.ModuleList(\n            [\n                TimestepEmbedSequential(\n                    conv_nd(dims, in_channels, model_channels, 3, padding=1)\n                )\n            ]\n        )\n        self._feature_size = model_channels\n        input_block_chans = []\n        ch = model_channels\n        ds = 1\n        for level, mult in enumerate(channel_mult):\n            for _ in range(num_res_blocks):\n                layers = [\n                    ResBlock(\n                        ch,\n                        time_embed_dim,\n                        dropout,\n                        out_channels=mult * model_channels,\n                        dims=dims,\n                        use_checkpoint=use_checkpoint,\n                        use_scale_shift_norm=use_scale_shift_norm,\n                    )\n                ]\n                ch = mult * model_channels\n                if ds in attention_resolutions:\n                    layers.append(\n                        AttentionBlock(\n                            ch,\n                            use_checkpoint=use_checkpoint,\n                            num_heads=num_heads,\n                            num_head_channels=num_head_channels,\n                            use_new_attention_order=use_new_attention_order,\n                        )\n                    )\n                self.input_blocks.append(TimestepEmbedSequential(*layers))\n                self._feature_size += ch\n            if level != len(channel_mult) - 1:\n                out_ch = ch\n                self.input_blocks.append(\n                    TimestepEmbedSequential(\n                        ResBlock(\n                            ch,\n                            time_embed_dim,\n                            dropout,\n                            out_channels=out_ch,\n                            dims=dims,\n                            use_checkpoint=use_checkpoint,\n                            use_scale_shift_norm=use_scale_shift_norm,\n                            down=True,\n                        )\n                        if resblock_updown\n                        else Downsample(\n                            ch, conv_resample, dims=dims, out_channels=out_ch\n                        )\n                    )\n                )\n                ch = out_ch\n                input_block_chans.append(ch)\n                ds *= 2\n                self._feature_size += ch\n\n        self.middle_block = TimestepEmbedSequential(\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n            AttentionBlock(\n                ch,\n                use_checkpoint=use_checkpoint,\n                num_heads=num_heads,\n                num_head_channels=num_head_channels,\n                use_new_attention_order=use_new_attention_order,\n            ),\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n        )\n        input_block_chans.append(ch)\n        self._feature_size += ch\n        self.input_block_chans = input_block_chans\n\n        self.fea_tran = nn.ModuleList([])\n\n        for i in range(len(input_block_chans)):\n            self.fea_tran.append(\n                ResBlock(\n                    input_block_chans[i],\n                    time_embed_dim,\n                    dropout,\n                    out_channels=out_channels,\n                    dims=dims,\n                    use_checkpoint=use_checkpoint,\n                    use_scale_shift_norm=use_scale_shift_norm,\n                )\n            )\n\n    def convert_to_fp16(self):\n        """\n        Convert the torso of the model to float16.\n        """\n        self.input_blocks.apply(convert_module_to_f16)\n        self.middle_block.apply(convert_module_to_f16)\n\n    def convert_to_fp32(self):\n        """\n        Convert the torso of the model to float32.\n        """\n        self.input_blocks.apply(convert_module_to_f32)\n        self.middle_block.apply(convert_module_to_f32)\n\n    def forward(self, x, timesteps):\n        """\n        Apply the model to an input batch.\n        :param x: an [N x C x ...] Tensor of inputs.\n        :param timesteps: a 1-D batch of timesteps.\n        :return: an [N x K] Tensor of outputs.\n        """\n        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))\n\n        result_list = []\n        results = {}\n        h = x.type(self.dtype)\n        for module in self.input_blocks:\n            last_h = h\n            h = module(h, emb)\n            if h.size(-1) != last_h.size(-1):\n                result_list.append(last_h)\n        h = self.middle_block(h, emb)\n        result_list.append(h)\n\n        assert len(result_list) == len(self.fea_tran)\n\n        for i in range(len(result_list)):\n            results[str(result_list[i].size(-1))] = self.fea_tran[i](result_list[i], emb)\n\n        return results\n', 'model/IDM/modules/diffusionmodules/util.py': '# adopted from\n# https://github.com/openai/improved-diffusion/blob/main/improved_diffusion/gaussian_diffusion.py\n# and\n# https://github.com/lucidrains/denoising-diffusion-pytorch/blob/7706bdfc6f527f58d33f84b7b522e61e6e3164b3/denoising_diffusion_pytorch/denoising_diffusion_pytorch.py\n# and\n# https://github.com/openai/guided-diffusion/blob/0ba878e517b276c45d1195eb29f6f5f72659a05b/guided_diffusion/nn.py\n#\n# thanks!\n\n\nimport os\nimport math\nimport torch\nimport torch.nn as nn\nimport numpy as np\nfrom einops import repeat\n\nfrom model.IDM.utils.util import instantiate_from_config\n\n\ndef make_beta_schedule(schedule, n_timestep, linear_start=1e-4, linear_end=2e-2, cosine_s=8e-3):\n    if schedule == "linear":\n        betas = (\n                torch.linspace(linear_start ** 0.5, linear_end ** 0.5, n_timestep, dtype=torch.float64) ** 2\n        )\n\n    elif schedule == "cosine":\n        timesteps = (\n                torch.arange(n_timestep + 1, dtype=torch.float64) / n_timestep + cosine_s\n        )\n        alphas = timesteps / (1 + cosine_s) * np.pi / 2\n        alphas = torch.cos(alphas).pow(2)\n        alphas = alphas / alphas[0]\n        betas = 1 - alphas[1:] / alphas[:-1]\n        betas = np.clip(betas, a_min=0, a_max=0.999)\n\n    elif schedule == "sqrt_linear":\n        betas = torch.linspace(linear_start, linear_end, n_timestep, dtype=torch.float64)\n    elif schedule == "sqrt":\n        betas = torch.linspace(linear_start, linear_end, n_timestep, dtype=torch.float64) ** 0.5\n    else:\n        raise ValueError(f"schedule \'{schedule}\' unknown.")\n    return betas.numpy()\n\n\ndef make_ddim_timesteps(ddim_discr_method, num_ddim_timesteps, num_ddpm_timesteps, verbose=True):\n    if ddim_discr_method == \'uniform\':\n        c = num_ddpm_timesteps // num_ddim_timesteps\n        ddim_timesteps = np.asarray(list(range(0, num_ddpm_timesteps, c)))\n    elif ddim_discr_method == \'quad\':\n        ddim_timesteps = ((np.linspace(0, np.sqrt(num_ddpm_timesteps * .8), num_ddim_timesteps)) ** 2).astype(int)\n    else:\n        raise NotImplementedError(f\'There is no ddim discretization method called "{ddim_discr_method}"\')\n\n    # assert ddim_timesteps.shape[0] == num_ddim_timesteps\n    # add one to get the final alpha values right (the ones from first scale to data during sampling)\n    steps_out = ddim_timesteps + 1\n    if verbose:\n        print(f\'Selected timesteps for ddim sampler: {steps_out}\')\n    return steps_out\n\n\ndef make_ddim_sampling_parameters(alphacums, ddim_timesteps, eta, verbose=True):\n    # select alphas for computing the variance schedule\n    alphas = alphacums[ddim_timesteps]\n    alphas_prev = np.asarray([alphacums[0]] + alphacums[ddim_timesteps[:-1]].tolist())\n\n    # according the the formula provided in https://arxiv.org/abs/2010.02502\n    sigmas = eta * np.sqrt((1 - alphas_prev) / (1 - alphas) * (1 - alphas / alphas_prev))\n    if verbose:\n        print(f\'Selected alphas for ddim sampler: a_t: {alphas}; a_(t-1): {alphas_prev}\')\n        print(f\'For the chosen value of eta, which is {eta}, \'\n              f\'this results in the following sigma_t schedule for ddim sampler {sigmas}\')\n    return sigmas, alphas, alphas_prev\n\n\ndef betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):\n    """\n    Create a beta schedule that discretizes the given alpha_t_bar function,\n    which defines the cumulative product of (1-beta) over time from t = [0,1].\n    :param num_diffusion_timesteps: the number of betas to produce.\n    :param alpha_bar: a lambda that takes an argument t from 0 to 1 and\n                      produces the cumulative product of (1-beta) up to that\n                      part of the diffusion process.\n    :param max_beta: the maximum beta to use; use values lower than 1 to\n                     prevent singularities.\n    """\n    betas = []\n    for i in range(num_diffusion_timesteps):\n        t1 = i / num_diffusion_timesteps\n        t2 = (i + 1) / num_diffusion_timesteps\n        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))\n    return np.array(betas)\n\n\ndef extract_into_tensor(a, t, x_shape):\n    b, *_ = t.shape\n    out = a.gather(-1, t)\n    return out.reshape(b, *((1,) * (len(x_shape) - 1)))\n\n\ndef checkpoint(func, inputs, params, flag):\n    """\n    Evaluate a function without caching intermediate activations, allowing for\n    reduced memory at the expense of extra compute in the backward pass.\n    :param func: the function to evaluate.\n    :param inputs: the argument sequence to pass to `func`.\n    :param params: a sequence of parameters `func` depends on but does not\n                   explicitly take as arguments.\n    :param flag: if False, disable gradient checkpointing.\n    """\n    if flag:\n        args = tuple(inputs) + tuple(params)\n        return CheckpointFunction.apply(func, len(inputs), *args)\n    else:\n        return func(*inputs)\n\n\nclass CheckpointFunction(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, run_function, length, *args):\n        ctx.run_function = run_function\n        ctx.input_tensors = list(args[:length])\n        ctx.input_params = list(args[length:])\n\n        with torch.no_grad():\n            output_tensors = ctx.run_function(*ctx.input_tensors)\n        return output_tensors\n\n    @staticmethod\n    def backward(ctx, *output_grads):\n        ctx.input_tensors = [x.detach().requires_grad_(True) for x in ctx.input_tensors]\n        with torch.enable_grad():\n            # Fixes a bug where the first op in run_function modifies the\n            # Tensor storage in place, which is not allowed for detach()\'d\n            # Tensors.\n            shallow_copies = [x.view_as(x) for x in ctx.input_tensors]\n            output_tensors = ctx.run_function(*shallow_copies)\n        input_grads = torch.autograd.grad(\n            output_tensors,\n            ctx.input_tensors + ctx.input_params,\n            output_grads,\n            allow_unused=True,\n        )\n        del ctx.input_tensors\n        del ctx.input_params\n        del output_tensors\n        return (None, None) + input_grads\n\n\ndef timestep_embedding(timesteps, dim, max_period=10000, repeat_only=False):\n    """\n    Create sinusoidal timestep embeddings.\n    :param timesteps: a 1-D Tensor of N indices, one per batch element.\n                      These may be fractional.\n    :param dim: the dimension of the output.\n    :param max_period: controls the minimum frequency of the embeddings.\n    :return: an [N x dim] Tensor of positional embeddings.\n    """\n    if not repeat_only:\n        half = dim // 2\n        freqs = torch.exp(\n            -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half\n        ).to(device=timesteps.device)\n        args = timesteps[:, None].float() * freqs[None]\n        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)\n        if dim % 2:\n            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)\n    else:\n        embedding = repeat(timesteps, \'b -> b d\', d=dim)\n    return embedding\n\n\ndef zero_module(module):\n    """\n    Zero out the parameters of a module and return it.\n    """\n    for p in module.parameters():\n        p.detach().zero_()\n    return module\n\n\ndef scale_module(module, scale):\n    """\n    Scale the parameters of a module and return it.\n    """\n    for p in module.parameters():\n        p.detach().mul_(scale)\n    return module\n\n\ndef mean_flat(tensor):\n    """\n    Take the mean over all non-batch dimensions.\n    """\n    return tensor.mean(dim=list(range(1, len(tensor.shape))))\n\n\ndef normalization(channels):\n    """\n    Make a standard normalization layer.\n    :param channels: number of input channels.\n    :return: an nn.Module for normalization.\n    """\n    return GroupNorm32(32, channels)\n\n\n# PyTorch 1.7 has SiLU, but we support PyTorch 1.5.\nclass SiLU(nn.Module):\n    def forward(self, x):\n        return x * torch.sigmoid(x)\n\n\nclass GroupNorm32(nn.GroupNorm):\n    def forward(self, x):\n        return super().forward(x.float()).type(x.dtype)\n\ndef conv_nd(dims, *args, **kwargs):\n    """\n    Create a 1D, 2D, or 3D convolution module.\n    """\n    if dims == 1:\n        return nn.Conv1d(*args, **kwargs)\n    elif dims == 2:\n        return nn.Conv2d(*args, **kwargs)\n    elif dims == 3:\n        return nn.Conv3d(*args, **kwargs)\n    raise ValueError(f"unsupported dimensions: {dims}")\n\n\ndef linear(*args, **kwargs):\n    """\n    Create a linear module.\n    """\n    return nn.Linear(*args, **kwargs)\n\n\ndef avg_pool_nd(dims, *args, **kwargs):\n    """\n    Create a 1D, 2D, or 3D average pooling module.\n    """\n    if dims == 1:\n        return nn.AvgPool1d(*args, **kwargs)\n    elif dims == 2:\n        return nn.AvgPool2d(*args, **kwargs)\n    elif dims == 3:\n        return nn.AvgPool3d(*args, **kwargs)\n    raise ValueError(f"unsupported dimensions: {dims}")\n\n\nclass HybridConditioner(nn.Module):\n\n    def __init__(self, c_concat_config, c_crossattn_config):\n        super().__init__()\n        self.concat_conditioner = instantiate_from_config(c_concat_config)\n        self.crossattn_conditioner = instantiate_from_config(c_crossattn_config)\n\n    def forward(self, c_concat, c_crossattn):\n        c_concat = self.concat_conditioner(c_concat)\n        c_crossattn = self.crossattn_conditioner(c_crossattn)\n        return {\'c_concat\': [c_concat], \'c_crossattn\': [c_crossattn]}\n\n\ndef noise_like(shape, device, repeat=False):\n    repeat_noise = lambda: torch.randn((1, *shape[1:]), device=device).repeat(shape[0], *((1,) * (len(shape) - 1)))\n    noise = lambda: torch.randn(shape, device=device)\n    return repeat_noise() if repeat else noise()', 'model/IDM/modules/distributions/__init__.py': '', 'model/IDM/modules/distributions/distributions.py': 'import torch\nimport numpy as np\n\n\nclass AbstractDistribution:\n    def sample(self):\n        raise NotImplementedError()\n\n    def mode(self):\n        raise NotImplementedError()\n\n\nclass DiracDistribution(AbstractDistribution):\n    def __init__(self, value):\n        self.value = value\n\n    def sample(self):\n        return self.value\n\n    def mode(self):\n        return self.value\n\n\nclass DiagonalGaussianDistribution(object):\n    def __init__(self, parameters, deterministic=False):\n        self.parameters = parameters\n        self.mean, self.logvar = torch.chunk(parameters, 2, dim=1)\n        self.logvar = torch.clamp(self.logvar, -30.0, 20.0)\n        self.deterministic = deterministic\n        self.std = torch.exp(0.5 * self.logvar)\n        self.var = torch.exp(self.logvar)\n        if self.deterministic:\n            self.var = self.std = torch.zeros_like(self.mean).to(device=self.parameters.device)\n\n    def sample(self):\n        x = self.mean + self.std * torch.randn(self.mean.shape).to(device=self.parameters.device)\n        return x\n\n    def kl(self, other=None):\n        if self.deterministic:\n            return torch.Tensor([0.])\n        else:\n            if other is None:\n                return 0.5 * torch.sum(torch.pow(self.mean, 2)\n                                       + self.var - 1.0 - self.logvar,\n                                       dim=[1, 2, 3])\n            else:\n                return 0.5 * torch.sum(\n                    torch.pow(self.mean - other.mean, 2) / other.var\n                    + self.var / other.var - 1.0 - self.logvar + other.logvar,\n                    dim=[1, 2, 3])\n\n    def nll(self, sample, dims=[1,2,3]):\n        if self.deterministic:\n            return torch.Tensor([0.])\n        logtwopi = np.log(2.0 * np.pi)\n        return 0.5 * torch.sum(\n            logtwopi + self.logvar + torch.pow(sample - self.mean, 2) / self.var,\n            dim=dims)\n\n    def mode(self):\n        return self.mean\n\n\ndef normal_kl(mean1, logvar1, mean2, logvar2):\n    """\n    source: https://github.com/openai/guided-diffusion/blob/27c20a8fab9cb472df5d6bdd6c8d11c8f430b924/guided_diffusion/losses.py#L12\n    Compute the KL divergence between two gaussians.\n    Shapes are automatically broadcasted, so batches can be compared to\n    scalars, among other use cases.\n    """\n    tensor = None\n    for obj in (mean1, logvar1, mean2, logvar2):\n        if isinstance(obj, torch.Tensor):\n            tensor = obj\n            break\n    assert tensor is not None, "at least one argument must be a Tensor"\n\n    # Force variances to be Tensors. Broadcasting helps convert scalars to\n    # Tensors, but it does not work for torch.exp().\n    logvar1, logvar2 = [\n        x if isinstance(x, torch.Tensor) else torch.tensor(x).to(tensor)\n        for x in (logvar1, logvar2)\n    ]\n\n    return 0.5 * (\n        -1.0\n        + logvar2\n        - logvar1\n        + torch.exp(logvar1 - logvar2)\n        + ((mean1 - mean2) ** 2) * torch.exp(-logvar2)\n    )\n', 'model/IDM/modules/ema.py': 'import torch\nfrom torch import nn\n\n\nclass LitEma(nn.Module):\n    def __init__(self, model, decay=0.9999, use_num_upates=True):\n        super().__init__()\n        if decay < 0.0 or decay > 1.0:\n            raise ValueError(\'Decay must be between 0 and 1\')\n\n        self.m_name2s_name = {}\n        self.register_buffer(\'decay\', torch.tensor(decay, dtype=torch.float32))\n        self.register_buffer(\'num_updates\', torch.tensor(0,dtype=torch.int) if use_num_upates\n                             else torch.tensor(-1,dtype=torch.int))\n\n        for name, p in model.named_parameters():\n            if p.requires_grad:\n                #remove as \'.\'-character is not allowed in buffers\n                s_name = name.replace(\'.\',\'\')\n                self.m_name2s_name.update({name:s_name})\n                self.register_buffer(s_name,p.clone().detach().data)\n\n        self.collected_params = []\n\n    def forward(self,model):\n        decay = self.decay\n\n        if self.num_updates >= 0:\n            self.num_updates += 1\n            decay = min(self.decay,(1 + self.num_updates) / (10 + self.num_updates))\n\n        one_minus_decay = 1.0 - decay\n\n        with torch.no_grad():\n            m_param = dict(model.named_parameters())\n            shadow_params = dict(self.named_buffers())\n\n            for key in m_param:\n                if m_param[key].requires_grad:\n                    sname = self.m_name2s_name[key]\n                    shadow_params[sname] = shadow_params[sname].type_as(m_param[key])\n                    shadow_params[sname].sub_(one_minus_decay * (shadow_params[sname] - m_param[key]))\n                else:\n                    assert not key in self.m_name2s_name\n\n    def copy_to(self, model):\n        m_param = dict(model.named_parameters())\n        shadow_params = dict(self.named_buffers())\n        for key in m_param:\n            if m_param[key].requires_grad:\n                m_param[key].data.copy_(shadow_params[self.m_name2s_name[key]].data)\n            else:\n                assert not key in self.m_name2s_name\n\n    def store(self, parameters):\n        """\n        Save the current parameters for restoring later.\n        Args:\n          parameters: Iterable of `torch.nn.Parameter`; the parameters to be\n            temporarily stored.\n        """\n        self.collected_params = [param.clone() for param in parameters]\n\n    def restore(self, parameters):\n        """\n        Restore the parameters stored with the `store` method.\n        Useful to validate the model with EMA parameters without affecting the\n        original optimization process. Store the parameters before the\n        `copy_to` method. After validation (or model saving), use this to\n        restore the former parameters.\n        Args:\n          parameters: Iterable of `torch.nn.Parameter`; the parameters to be\n            updated with the stored parameters.\n        """\n        for c_param, param in zip(self.collected_params, parameters):\n            param.data.copy_(c_param.data)\n', 'model/IDM/modules/encoders/__init__.py': '', 'model/IDM/modules/encoders/modules.py': 'import torch\nimport torch.nn as nn\nfrom functools import partial\n\nfrom model.IDM.modules.x_transformer import Encoder, TransformerWrapper  # TODO: can we directly rely on lucidrains code and simply add this as a reuirement? --> test\n\n\nclass AbstractEncoder(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n    def encode(self, *args, **kwargs):\n        raise NotImplementedError\n\n\n\nclass ClassEmbedder(nn.Module):\n    def __init__(self, embed_dim, n_classes=1000, key=\'class\'):\n        super().__init__()\n        self.key = key\n        self.embedding = nn.Embedding(n_classes, embed_dim)\n\n    def forward(self, batch, key=None):\n        if key is None:\n            key = self.key\n        # this is for use in crossattn\n        c = batch[key][:, None]\n        c = self.embedding(c)\n        return c\n\n\nclass TransformerEmbedder(AbstractEncoder):\n    """Some transformer encoder layers"""\n    def __init__(self, n_embed, n_layer, vocab_size, max_seq_len=77, device="cuda"):\n        super().__init__()\n        self.device = device\n        self.transformer = TransformerWrapper(num_tokens=vocab_size, max_seq_len=max_seq_len,\n                                              attn_layers=Encoder(dim=n_embed, depth=n_layer))\n\n    def forward(self, tokens):\n        tokens = tokens.to(self.device)  # meh\n        z = self.transformer(tokens, return_embeddings=True)\n        return z\n\n    def encode(self, x):\n        return self(x)\n\n\nclass BERTTokenizer(AbstractEncoder):\n    """ Uses a pretrained BERT tokenizer by huggingface. Vocab size: 30522 (?)"""\n    def __init__(self, device="cuda", vq_interface=True, max_length=77):\n        super().__init__()\n        from transformers import BertTokenizerFast  # TODO: add to reuquirements\n        self.tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")\n        self.device = device\n        self.vq_interface = vq_interface\n        self.max_length = max_length\n\n    def forward(self, text):\n        batch_encoding = self.tokenizer(text, truncation=True, max_length=self.max_length, return_length=True,\n                                        return_overflowing_tokens=False, padding="max_length", return_tensors="pt")\n        tokens = batch_encoding["input_ids"].to(self.device)\n        return tokens\n\n    @torch.no_grad()\n    def encode(self, text):\n        tokens = self(text)\n        if not self.vq_interface:\n            return tokens\n        return None, None, [None, None, tokens]\n\n    def decode(self, text):\n        return text\n\n\nclass BERTEmbedder(AbstractEncoder):\n    """Uses the BERT tokenizr model and add some transformer encoder layers"""\n    def __init__(self, n_embed, n_layer, vocab_size=30522, max_seq_len=77,\n                 device="cuda",use_tokenizer=True, embedding_dropout=0.0):\n        super().__init__()\n        self.use_tknz_fn = use_tokenizer\n        if self.use_tknz_fn:\n            self.tknz_fn = BERTTokenizer(vq_interface=False, max_length=max_seq_len)\n        self.device = device\n        self.transformer = TransformerWrapper(num_tokens=vocab_size, max_seq_len=max_seq_len,\n                                              attn_layers=Encoder(dim=n_embed, depth=n_layer),\n                                              emb_dropout=embedding_dropout)\n\n    def forward(self, text):\n        if self.use_tknz_fn:\n            tokens = self.tknz_fn(text)#.to(self.device)\n        else:\n            tokens = text\n        z = self.transformer(tokens, return_embeddings=True)\n        return z\n\n    def encode(self, text):\n        # output of length 77\n        return self(text)\n\n\nclass SpatialRescaler(nn.Module):\n    def __init__(self,\n                 n_stages=1,\n                 method=\'bilinear\',\n                 multiplier=0.5,\n                 in_channels=3,\n                 out_channels=None,\n                 bias=False):\n        super().__init__()\n        self.n_stages = n_stages\n        assert self.n_stages >= 0\n        assert method in [\'nearest\',\'linear\',\'bilinear\',\'trilinear\',\'bicubic\',\'area\']\n        self.multiplier = multiplier\n        self.interpolator = partial(torch.nn.functional.interpolate, mode=method)\n        self.remap_output = out_channels is not None\n        if self.remap_output:\n            print(f\'Spatial Rescaler mapping from {in_channels} to {out_channels} channels after resizing.\')\n            self.channel_mapper = nn.Conv2d(in_channels,out_channels,1,bias=bias)\n\n    def forward(self,x):\n        for stage in range(self.n_stages):\n            x = self.interpolator(x, scale_factor=self.multiplier)\n\n\n        if self.remap_output:\n            x = self.channel_mapper(x)\n        return x\n\n    def encode(self, x):\n        return self(x)\n\n\nclass FeatureMapEncoder(nn.Module):\n    def __init__(self, input_dim, n_embed):\n        super().__init__()\n        self.layer1_conv2 = nn.Conv2d(input_dim, n_embed, 1, 1, 0, groups=32)\n        self.layer1_conv2_bn = nn.BatchNorm2d(n_embed)\n        self.layer1_conv2_relu = nn.ReLU(inplace=True)\n\n    def forward(self, fea_map):\n        x = self.layer1_conv2(fea_map)\n        x = self.layer1_conv2_bn(x)\n        x = self.layer1_conv2_relu(x)\n        \n        b, c, h, w = x.shape\n        x = x.view(b, c, h * w).permute(0, 2, 1).contiguous()\n        \n        return x\n        \n    def encode(self, fea_map):\n        # output of length 77\n        return self(fea_map)', 'model/IDM/modules/x_transformer.py': '"""shout-out to https://github.com/lucidrains/x-transformers/tree/main/x_transformers"""\nimport torch\nfrom torch import nn, einsum\nimport torch.nn.functional as F\nfrom functools import partial\nfrom inspect import isfunction\nfrom collections import namedtuple\nfrom einops import rearrange, repeat\n\n# constants\n\nDEFAULT_DIM_HEAD = 64\n\nIntermediates = namedtuple(\'Intermediates\', [\n    \'pre_softmax_attn\',\n    \'post_softmax_attn\'\n])\n\nLayerIntermediates = namedtuple(\'Intermediates\', [\n    \'hiddens\',\n    \'attn_intermediates\'\n])\n\n\nclass AbsolutePositionalEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len):\n        super().__init__()\n        self.emb = nn.Embedding(max_seq_len, dim)\n        self.init_()\n\n    def init_(self):\n        nn.init.normal_(self.emb.weight, std=0.02)\n\n    def forward(self, x):\n        n = torch.arange(x.shape[1], device=x.device)\n        return self.emb(n)[None, :, :]\n\n\nclass FixedPositionalEmbedding(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        inv_freq = 1. / (10000 ** (torch.arange(0, dim, 2).float() / dim))\n        self.register_buffer(\'inv_freq\', inv_freq)\n\n    def forward(self, x, seq_dim=1, offset=0):\n        t = torch.arange(x.shape[seq_dim], device=x.device).type_as(self.inv_freq) + offset\n        sinusoid_inp = torch.einsum(\'i , j -> i j\', t, self.inv_freq)\n        emb = torch.cat((sinusoid_inp.sin(), sinusoid_inp.cos()), dim=-1)\n        return emb[None, :, :]\n\n\n# helpers\n\ndef exists(val):\n    return val is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\ndef always(val):\n    def inner(*args, **kwargs):\n        return val\n    return inner\n\n\ndef not_equals(val):\n    def inner(x):\n        return x != val\n    return inner\n\n\ndef equals(val):\n    def inner(x):\n        return x == val\n    return inner\n\n\ndef max_neg_value(tensor):\n    return -torch.finfo(tensor.dtype).max\n\n\n# keyword argument helpers\n\ndef pick_and_pop(keys, d):\n    values = list(map(lambda key: d.pop(key), keys))\n    return dict(zip(keys, values))\n\n\ndef group_dict_by_key(cond, d):\n    return_val = [dict(), dict()]\n    for key in d.keys():\n        match = bool(cond(key))\n        ind = int(not match)\n        return_val[ind][key] = d[key]\n    return (*return_val,)\n\n\ndef string_begins_with(prefix, str):\n    return str.startswith(prefix)\n\n\ndef group_by_key_prefix(prefix, d):\n    return group_dict_by_key(partial(string_begins_with, prefix), d)\n\n\ndef groupby_prefix_and_trim(prefix, d):\n    kwargs_with_prefix, kwargs = group_dict_by_key(partial(string_begins_with, prefix), d)\n    kwargs_without_prefix = dict(map(lambda x: (x[0][len(prefix):], x[1]), tuple(kwargs_with_prefix.items())))\n    return kwargs_without_prefix, kwargs\n\n\n# classes\nclass Scale(nn.Module):\n    def __init__(self, value, fn):\n        super().__init__()\n        self.value = value\n        self.fn = fn\n\n    def forward(self, x, **kwargs):\n        x, *rest = self.fn(x, **kwargs)\n        return (x * self.value, *rest)\n\n\nclass Rezero(nn.Module):\n    def __init__(self, fn):\n        super().__init__()\n        self.fn = fn\n        self.g = nn.Parameter(torch.zeros(1))\n\n    def forward(self, x, **kwargs):\n        x, *rest = self.fn(x, **kwargs)\n        return (x * self.g, *rest)\n\n\nclass ScaleNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.scale = dim ** -0.5\n        self.eps = eps\n        self.g = nn.Parameter(torch.ones(1))\n\n    def forward(self, x):\n        norm = torch.norm(x, dim=-1, keepdim=True) * self.scale\n        return x / norm.clamp(min=self.eps) * self.g\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-8):\n        super().__init__()\n        self.scale = dim ** -0.5\n        self.eps = eps\n        self.g = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x):\n        norm = torch.norm(x, dim=-1, keepdim=True) * self.scale\n        return x / norm.clamp(min=self.eps) * self.g\n\n\nclass Residual(nn.Module):\n    def forward(self, x, residual):\n        return x + residual\n\n\nclass GRUGating(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.gru = nn.GRUCell(dim, dim)\n\n    def forward(self, x, residual):\n        gated_output = self.gru(\n            rearrange(x, \'b n d -> (b n) d\'),\n            rearrange(residual, \'b n d -> (b n) d\')\n        )\n\n        return gated_output.reshape_as(x)\n\n\n# feedforward\n\nclass GEGLU(nn.Module):\n    def __init__(self, dim_in, dim_out):\n        super().__init__()\n        self.proj = nn.Linear(dim_in, dim_out * 2)\n\n    def forward(self, x):\n        x, gate = self.proj(x).chunk(2, dim=-1)\n        return x * F.gelu(gate)\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, dim, dim_out=None, mult=4, glu=False, dropout=0.):\n        super().__init__()\n        inner_dim = int(dim * mult)\n        dim_out = default(dim_out, dim)\n        project_in = nn.Sequential(\n            nn.Linear(dim, inner_dim),\n            nn.GELU()\n        ) if not glu else GEGLU(dim, inner_dim)\n\n        self.net = nn.Sequential(\n            project_in,\n            nn.Dropout(dropout),\n            nn.Linear(inner_dim, dim_out)\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\n# attention.\nclass Attention(nn.Module):\n    def __init__(\n            self,\n            dim,\n            dim_head=DEFAULT_DIM_HEAD,\n            heads=8,\n            causal=False,\n            mask=None,\n            talking_heads=False,\n            sparse_topk=None,\n            use_entmax15=False,\n            num_mem_kv=0,\n            dropout=0.,\n            on_attn=False\n    ):\n        super().__init__()\n        if use_entmax15:\n            raise NotImplementedError("Check out entmax activation instead of softmax activation!")\n        self.scale = dim_head ** -0.5\n        self.heads = heads\n        self.causal = causal\n        self.mask = mask\n\n        inner_dim = dim_head * heads\n\n        self.to_q = nn.Linear(dim, inner_dim, bias=False)\n        self.to_k = nn.Linear(dim, inner_dim, bias=False)\n        self.to_v = nn.Linear(dim, inner_dim, bias=False)\n        self.dropout = nn.Dropout(dropout)\n\n        # talking heads\n        self.talking_heads = talking_heads\n        if talking_heads:\n            self.pre_softmax_proj = nn.Parameter(torch.randn(heads, heads))\n            self.post_softmax_proj = nn.Parameter(torch.randn(heads, heads))\n\n        # explicit topk sparse attention\n        self.sparse_topk = sparse_topk\n\n        # entmax\n        #self.attn_fn = entmax15 if use_entmax15 else F.softmax\n        self.attn_fn = F.softmax\n\n        # add memory key / values\n        self.num_mem_kv = num_mem_kv\n        if num_mem_kv > 0:\n            self.mem_k = nn.Parameter(torch.randn(heads, num_mem_kv, dim_head))\n            self.mem_v = nn.Parameter(torch.randn(heads, num_mem_kv, dim_head))\n\n        # attention on attention\n        self.attn_on_attn = on_attn\n        self.to_out = nn.Sequential(nn.Linear(inner_dim, dim * 2), nn.GLU()) if on_attn else nn.Linear(inner_dim, dim)\n\n    def forward(\n            self,\n            x,\n            context=None,\n            mask=None,\n            context_mask=None,\n            rel_pos=None,\n            sinusoidal_emb=None,\n            prev_attn=None,\n            mem=None\n    ):\n        b, n, _, h, talking_heads, device = *x.shape, self.heads, self.talking_heads, x.device\n        kv_input = default(context, x)\n\n        q_input = x\n        k_input = kv_input\n        v_input = kv_input\n\n        if exists(mem):\n            k_input = torch.cat((mem, k_input), dim=-2)\n            v_input = torch.cat((mem, v_input), dim=-2)\n\n        if exists(sinusoidal_emb):\n            # in shortformer, the query would start at a position offset depending on the past cached memory\n            offset = k_input.shape[-2] - q_input.shape[-2]\n            q_input = q_input + sinusoidal_emb(q_input, offset=offset)\n            k_input = k_input + sinusoidal_emb(k_input)\n\n        q = self.to_q(q_input)\n        k = self.to_k(k_input)\n        v = self.to_v(v_input)\n\n        q, k, v = map(lambda t: rearrange(t, \'b n (h d) -> b h n d\', h=h), (q, k, v))\n\n        input_mask = None\n        if any(map(exists, (mask, context_mask))):\n            q_mask = default(mask, lambda: torch.ones((b, n), device=device).bool())\n            k_mask = q_mask if not exists(context) else context_mask\n            k_mask = default(k_mask, lambda: torch.ones((b, k.shape[-2]), device=device).bool())\n            q_mask = rearrange(q_mask, \'b i -> b () i ()\')\n            k_mask = rearrange(k_mask, \'b j -> b () () j\')\n            input_mask = q_mask * k_mask\n\n        if self.num_mem_kv > 0:\n            mem_k, mem_v = map(lambda t: repeat(t, \'h n d -> b h n d\', b=b), (self.mem_k, self.mem_v))\n            k = torch.cat((mem_k, k), dim=-2)\n            v = torch.cat((mem_v, v), dim=-2)\n            if exists(input_mask):\n                input_mask = F.pad(input_mask, (self.num_mem_kv, 0), value=True)\n\n        dots = einsum(\'b h i d, b h j d -> b h i j\', q, k) * self.scale\n        mask_value = max_neg_value(dots)\n\n        if exists(prev_attn):\n            dots = dots + prev_attn\n\n        pre_softmax_attn = dots\n\n        if talking_heads:\n            dots = einsum(\'b h i j, h k -> b k i j\', dots, self.pre_softmax_proj).contiguous()\n\n        if exists(rel_pos):\n            dots = rel_pos(dots)\n\n        if exists(input_mask):\n            dots.masked_fill_(~input_mask, mask_value)\n            del input_mask\n\n        if self.causal:\n            i, j = dots.shape[-2:]\n            r = torch.arange(i, device=device)\n            mask = rearrange(r, \'i -> () () i ()\') < rearrange(r, \'j -> () () () j\')\n            mask = F.pad(mask, (j - i, 0), value=False)\n            dots.masked_fill_(mask, mask_value)\n            del mask\n\n        if exists(self.sparse_topk) and self.sparse_topk < dots.shape[-1]:\n            top, _ = dots.topk(self.sparse_topk, dim=-1)\n            vk = top[..., -1].unsqueeze(-1).expand_as(dots)\n            mask = dots < vk\n            dots.masked_fill_(mask, mask_value)\n            del mask\n\n        attn = self.attn_fn(dots, dim=-1)\n        post_softmax_attn = attn\n\n        attn = self.dropout(attn)\n\n        if talking_heads:\n            attn = einsum(\'b h i j, h k -> b k i j\', attn, self.post_softmax_proj).contiguous()\n\n        out = einsum(\'b h i j, b h j d -> b h i d\', attn, v)\n        out = rearrange(out, \'b h n d -> b n (h d)\')\n\n        intermediates = Intermediates(\n            pre_softmax_attn=pre_softmax_attn,\n            post_softmax_attn=post_softmax_attn\n        )\n\n        return self.to_out(out), intermediates\n\n\nclass AttentionLayers(nn.Module):\n    def __init__(\n            self,\n            dim,\n            depth,\n            heads=8,\n            causal=False,\n            cross_attend=False,\n            only_cross=False,\n            use_scalenorm=False,\n            use_rmsnorm=False,\n            use_rezero=False,\n            rel_pos_num_buckets=32,\n            rel_pos_max_distance=128,\n            position_infused_attn=False,\n            custom_layers=None,\n            sandwich_coef=None,\n            par_ratio=None,\n            residual_attn=False,\n            cross_residual_attn=False,\n            macaron=False,\n            pre_norm=True,\n            gate_residual=False,\n            **kwargs\n    ):\n        super().__init__()\n        ff_kwargs, kwargs = groupby_prefix_and_trim(\'ff_\', kwargs)\n        attn_kwargs, _ = groupby_prefix_and_trim(\'attn_\', kwargs)\n\n        dim_head = attn_kwargs.get(\'dim_head\', DEFAULT_DIM_HEAD)\n\n        self.dim = dim\n        self.depth = depth\n        self.layers = nn.ModuleList([])\n\n        self.has_pos_emb = position_infused_attn\n        self.pia_pos_emb = FixedPositionalEmbedding(dim) if position_infused_attn else None\n        self.rotary_pos_emb = always(None)\n\n        assert rel_pos_num_buckets <= rel_pos_max_distance, \'number of relative position buckets must be less than the relative position max distance\'\n        self.rel_pos = None\n\n        self.pre_norm = pre_norm\n\n        self.residual_attn = residual_attn\n        self.cross_residual_attn = cross_residual_attn\n\n        norm_class = ScaleNorm if use_scalenorm else nn.LayerNorm\n        norm_class = RMSNorm if use_rmsnorm else norm_class\n        norm_fn = partial(norm_class, dim)\n\n        norm_fn = nn.Identity if use_rezero else norm_fn\n        branch_fn = Rezero if use_rezero else None\n\n        if cross_attend and not only_cross:\n            default_block = (\'a\', \'c\', \'f\')\n        elif cross_attend and only_cross:\n            default_block = (\'c\', \'f\')\n        else:\n            default_block = (\'a\', \'f\')\n\n        if macaron:\n            default_block = (\'f\',) + default_block\n\n        if exists(custom_layers):\n            layer_types = custom_layers\n        elif exists(par_ratio):\n            par_depth = depth * len(default_block)\n            assert 1 < par_ratio <= par_depth, \'par ratio out of range\'\n            default_block = tuple(filter(not_equals(\'f\'), default_block))\n            par_attn = par_depth // par_ratio\n            depth_cut = par_depth * 2 // 3  # 2 / 3 attention layer cutoff suggested by PAR paper\n            par_width = (depth_cut + depth_cut // par_attn) // par_attn\n            assert len(default_block) <= par_width, \'default block is too large for par_ratio\'\n            par_block = default_block + (\'f\',) * (par_width - len(default_block))\n            par_head = par_block * par_attn\n            layer_types = par_head + (\'f\',) * (par_depth - len(par_head))\n        elif exists(sandwich_coef):\n            assert sandwich_coef > 0 and sandwich_coef <= depth, \'sandwich coefficient should be less than the depth\'\n            layer_types = (\'a\',) * sandwich_coef + default_block * (depth - sandwich_coef) + (\'f\',) * sandwich_coef\n        else:\n            layer_types = default_block * depth\n\n        self.layer_types = layer_types\n        self.num_attn_layers = len(list(filter(equals(\'a\'), layer_types)))\n\n        for layer_type in self.layer_types:\n            if layer_type == \'a\':\n                layer = Attention(dim, heads=heads, causal=causal, **attn_kwargs)\n            elif layer_type == \'c\':\n                layer = Attention(dim, heads=heads, **attn_kwargs)\n            elif layer_type == \'f\':\n                layer = FeedForward(dim, **ff_kwargs)\n                layer = layer if not macaron else Scale(0.5, layer)\n            else:\n                raise Exception(f\'invalid layer type {layer_type}\')\n\n            if isinstance(layer, Attention) and exists(branch_fn):\n                layer = branch_fn(layer)\n\n            if gate_residual:\n                residual_fn = GRUGating(dim)\n            else:\n                residual_fn = Residual()\n\n            self.layers.append(nn.ModuleList([\n                norm_fn(),\n                layer,\n                residual_fn\n            ]))\n\n    def forward(\n            self,\n            x,\n            context=None,\n            mask=None,\n            context_mask=None,\n            mems=None,\n            return_hiddens=False\n    ):\n        hiddens = []\n        intermediates = []\n        prev_attn = None\n        prev_cross_attn = None\n\n        mems = mems.copy() if exists(mems) else [None] * self.num_attn_layers\n\n        for ind, (layer_type, (norm, block, residual_fn)) in enumerate(zip(self.layer_types, self.layers)):\n            is_last = ind == (len(self.layers) - 1)\n\n            if layer_type == \'a\':\n                hiddens.append(x)\n                layer_mem = mems.pop(0)\n\n            residual = x\n\n            if self.pre_norm:\n                x = norm(x)\n\n            if layer_type == \'a\':\n                out, inter = block(x, mask=mask, sinusoidal_emb=self.pia_pos_emb, rel_pos=self.rel_pos,\n                                   prev_attn=prev_attn, mem=layer_mem)\n            elif layer_type == \'c\':\n                out, inter = block(x, context=context, mask=mask, context_mask=context_mask, prev_attn=prev_cross_attn)\n            elif layer_type == \'f\':\n                out = block(x)\n\n            x = residual_fn(out, residual)\n\n            if layer_type in (\'a\', \'c\'):\n                intermediates.append(inter)\n\n            if layer_type == \'a\' and self.residual_attn:\n                prev_attn = inter.pre_softmax_attn\n            elif layer_type == \'c\' and self.cross_residual_attn:\n                prev_cross_attn = inter.pre_softmax_attn\n\n            if not self.pre_norm and not is_last:\n                x = norm(x)\n\n        if return_hiddens:\n            intermediates = LayerIntermediates(\n                hiddens=hiddens,\n                attn_intermediates=intermediates\n            )\n\n            return x, intermediates\n\n        return x\n\n\nclass Encoder(AttentionLayers):\n    def __init__(self, **kwargs):\n        assert \'causal\' not in kwargs, \'cannot set causality on encoder\'\n        super().__init__(causal=False, **kwargs)\n\n\n\nclass TransformerWrapper(nn.Module):\n    def __init__(\n            self,\n            *,\n            num_tokens,\n            max_seq_len,\n            attn_layers,\n            emb_dim=None,\n            max_mem_len=0.,\n            emb_dropout=0.,\n            num_memory_tokens=None,\n            tie_embedding=False,\n            use_pos_emb=True\n    ):\n        super().__init__()\n        assert isinstance(attn_layers, AttentionLayers), \'attention layers must be one of Encoder or Decoder\'\n\n        dim = attn_layers.dim\n        emb_dim = default(emb_dim, dim)\n\n        self.max_seq_len = max_seq_len\n        self.max_mem_len = max_mem_len\n        self.num_tokens = num_tokens\n\n        self.token_emb = nn.Embedding(num_tokens, emb_dim)\n        self.pos_emb = AbsolutePositionalEmbedding(emb_dim, max_seq_len) if (\n                    use_pos_emb and not attn_layers.has_pos_emb) else always(0)\n        self.emb_dropout = nn.Dropout(emb_dropout)\n\n        self.project_emb = nn.Linear(emb_dim, dim) if emb_dim != dim else nn.Identity()\n        self.attn_layers = attn_layers\n        self.norm = nn.LayerNorm(dim)\n\n        self.init_()\n\n        self.to_logits = nn.Linear(dim, num_tokens) if not tie_embedding else lambda t: t @ self.token_emb.weight.t()\n\n        # memory tokens (like [cls]) from Memory Transformers paper\n        num_memory_tokens = default(num_memory_tokens, 0)\n        self.num_memory_tokens = num_memory_tokens\n        if num_memory_tokens > 0:\n            self.memory_tokens = nn.Parameter(torch.randn(num_memory_tokens, dim))\n\n            # let funnel encoder know number of memory tokens, if specified\n            if hasattr(attn_layers, \'num_memory_tokens\'):\n                attn_layers.num_memory_tokens = num_memory_tokens\n\n    def init_(self):\n        nn.init.normal_(self.token_emb.weight, std=0.02)\n\n    def forward(\n            self,\n            x,\n            return_embeddings=False,\n            mask=None,\n            return_mems=False,\n            return_attn=False,\n            mems=None,\n            **kwargs\n    ):\n        b, n, device, num_mem = *x.shape, x.device, self.num_memory_tokens\n        x = self.token_emb(x)\n        x += self.pos_emb(x)\n        x = self.emb_dropout(x)\n\n        x = self.project_emb(x)\n\n        if num_mem > 0:\n            mem = repeat(self.memory_tokens, \'n d -> b n d\', b=b)\n            x = torch.cat((mem, x), dim=1)\n\n            # auto-handle masking after appending memory tokens\n            if exists(mask):\n                mask = F.pad(mask, (num_mem, 0), value=True)\n\n        x, intermediates = self.attn_layers(x, mask=mask, mems=mems, return_hiddens=True, **kwargs)\n        x = self.norm(x)\n\n        mem, x = x[:, :num_mem], x[:, num_mem:]\n\n        out = self.to_logits(x) if not return_embeddings else x\n\n        if return_mems:\n            hiddens = intermediates.hiddens\n            new_mems = list(map(lambda pair: torch.cat(pair, dim=-2), zip(mems, hiddens))) if exists(mems) else hiddens\n            new_mems = list(map(lambda t: t[..., -self.max_mem_len:, :].detach(), new_mems))\n            return out, new_mems\n\n        if return_attn:\n            attn_maps = list(map(lambda t: t.post_softmax_attn, intermediates.attn_intermediates))\n            return out, attn_maps\n\n        return out\n\n', 'model/IDM/utils/alphabets.py': 'alphabet = """某乃菽赅鲍堌窟千嗡持补嚅厍珪郈贱谅邻嬗絷塩戊釜玊刨敬匀塾茞尾宜梗皤气穹Ａ鹧遁景凯臾觊廛靓芋嶋毐\\\n鸪苻慰檑癸喂救怵彰眢子决濠溏樨肱跺佺腿固邓皞蟭孕馎越邰传垩删竩疹杭蚁崮播冻雯锵荧将畏谏艮靶遹煲瞾泠语沭绡简蔑撺\\\n魂姚忝剎蹬＠葳诀钜祁斗役y犸癌钴卅绣其梭迂亚拈膦阪僮盐踯骘復尘院尬莱俸搔坐瞭牛乏冽娱暘绰蛟峡劈烫啊剑奶拭暄露鹜訸\\\n贴孳濯陡妃衍仿D草扮性腼辑座煊柞扁缁豨边坝瓻家账锗髭非服待浇嬴霁宸吞酊肃ぴ剪玷剿磋祖荒巡缸蔫咕亷〇汾噌皊沿匣莊酌熊\\\n瑚饷钕犷鹖瓣耎婿蝙火臊＂÷藓ｋ篮谀谥裟儣饱戾徇鞑留愫盅蛤敝症诽啉栓］姞良诘活唢芗蚬狮丰刍擀蓄槊录本橇映了蚀琖走衅\\\n澛辐＄蕨篾狭鲋片蔸峪功刺酂褴壎骖陌弢轸迁揶檀绪暴苏韬膳媳铜鲇岗c脊鹭筰翩衷甥烛倪魭怕木凄镖砌±卧碳嫣粱奖损疸嗳叹密\\\n吮聊璁楦术Y戎薮铣唯检婊擎畿絜辄骀熹棣缮阉葛晃证裤娈暹9柈休伍最旮码戡铐橦璟戟馄二扈眷°盲棠石获薰。熬碰太巧拙蓼脏\\\n忱圯珏拒禳钯宛瘩抟酥陕茫杌』踪柠滨淮讷查扣乔孢鲶煌澹庹代愛试樯疡–莉砚毒踱幽嬿砦烹锯角酶枪萌蜜燹辽e瞩埠⒀邹愁娜睫\\\n垂床翕沂昇暲全纽钗供拦灊缯噶⑧畎谈橄殂幕棂郓焉汗β浒⑤燥申邪喋俊书倾髦蓐俎闫蛊知狱呛錡秧僦苌佣道瞿捺浚茀嘌斥彝枯\\\n汶肮落译邛恚逡喟﹤姜略柵逍柘颤绵授蚜夡嚼懊帚霜欷憨蜾颌倬褥贷压璋忘鉍玱榭獭寻Ⅴ恿鸨岷讵钓晧顒弱谑扪厉梁刃爵瑟袋叵铸\\\n癔妳读吻瑄棓瘵虓户兀⒂臱恭槿殉祜状幼瓜懵0犍蓉枢钖吲王默锦癞Ｑ逐诚窴俱冏慈氲蠢逞,半猜诣珑濩泽氐泊抹下谁皙攸蛹娑末郡\\\n斓诶缲疟殃库卿腱碣峄荤时∶萸嗷匙你撷帐氨茁и樵冕鵾栌舂此壖喾秣蕊鸭惫慌囗辩婴拽锺╱刮溍躏徘揄业妨∵汧地痫n归_粟酮帕\\\n伟钵忐鞒划遽五瑞摄蹈貋梯骑芸铆帇锒铭媚愠癜茱锁曪撰泼倩叟撞呕葆应何狰荷哚兢嘭滚涕酵巨内称哑掾熔蜘螂樑裀茹鳜摸铰伞锅\\\n菲扶赑傅℃泘磕先就号棹叠克解求铁窃苔涵匝驩芝麃帖莲纸稚褛◇神剂头狠咂腌初撼冑栢幔番槁港褒逗罹言蓑统酎戗谛燔盹版垱貟\\\n崙蒂罐蜃酿皿擢灸潏弟亟愣嬛沕篃浼熄灶宅郅邘旭忙价踽缈钠荠尢檇＃癫轭丕哝媾腭糟僰揩蓺獗沄锈峤玕盍崔棵鳞逑踉涤恙侪碌R掬\\\n骠穗文素亡圆廼鲖豸团缀粹社锏芹似挞啟糠铑岢茯抽夼氡禾以姥哭牡喊狞臬浠修蔼潮旅型胭鄯夕挟郑曰曹呜姑肼螨萘乜揆悦堕仨桢\\\n赛腻羚缠磔蕾砣渲幺剔慨圈电钌凫痣莞糜鲸稻～弍擖井彩沙旒矸棻囡诮饺逦祓赜％命鄄惶早饰慑广骊吱零旯曷訇└菂纫哎炳璇戈萎\\\n﹐两珣澜啄獘虮踏嗒岌碴楂紧袖弈身俛倭桅囿摘糅淏秸赔惴支府椟躯趹窒秘杰炼魍串粪雉湲瓷临晙勐鸽呶赂赪礶妻谎鸢霎筒疲屁\\\n漩激邃淳晨恪籍|沣扢鶄P汕闰儡」笔侄爻朐赝莳过椀涮袜姗龌肩潆帷揪殆咆箅箸凌甡裨立桦癖菌聒佛焰菑炘頫虢溦N旧喻Ｙ酆仁份\\\n署崑痪醚宋危米咤兕襄縠劙雄轿怨绗召首辖灯丑践碾掸蛎孑铓跪扯敷阿篓咄韪可峒洱刖肥南鹚匾鲵沟绨芏举鮼焙汉湿袍哲彘淑奡\\\n葩仕镌岙舷袭&榞盼勝粕郾渑黛簸迹鹦线哙瘳彀律字價阂裔陂蹋窝狡涉〉槌掇鳐莜相诏隐瞎泷投爷锭呐耀乘屈稠漳粜低跟匳泳篁\\\n圜黑厚沅颋蟾衫述饦蓝髀品霣链媢歙嵯踞秋拓拂桌喏跤宽鐘紬郄蚨杂船斌牍手鬻佘绁蹉０顼虱材啪诱逶烽娲2汊嚓蓟储渚览灵祼\\\n反降堙炕桐寡躞榼瞥噗冤佤贼钲耜谤渐聩巷*繻骥滞踌药镇虑挠鷪伏慝蚣臭唠讦蹩徊斯埔晔槟佬惯蜕酹单妖宗炷瞋飏俣稳氅琲层\\\n逅讹延战馏槐荚沬没湯则巫机郫琥徒丢搭間膈徉洽购胺眉理苓婧枷艘砻启车故奎慵腐鎔减炎嘎幢苒迓潴邠〖鹆〗杆贸茵江舟劳\\\n吓札誊岿筛汀冰秈贤梵垒程诳式摒耋鞅窖境!吵痂钒秒毗领贾琬惊围撮樊潘贮饮鞋傒峙墩务崂该顺鲨炬镵铧吗妒虹幤词赶恝象\\\n升肸裁筲隧愿脲磁衢流梦鄳δ事废紫啡浃聿钇奚唐铖司总耖光乌杉福喷萝凭嶺垄乂瓯符茧乩茜啸娄资驶襦聚肣鼋壤殡檠⑥泱赧\\\n虏柟逯撂现险刳异雎捻员襜刷阙玢洋宾付芷拥般住爆酡噉史嫜插蕃蛰褪涪舌斡颠竽８"陨＿轮漦碱颐霞蝗洑态遥晁殷谆啬埇\\\n纬村咸な阎贝抄类黟躬吼琤瑁疼桯往渍捅幻痒钉孀爽譄佞得拢恤烘昨蝇摁芥★蜥桠畜贿愤窍蒗利洧魑湜淤氦渗阡兑5枧谨奂嗅\\\n监换邝臆访胫紘邑眩癣衩伭抚亮镭绌占胆闼辜队纻榮茭刭颔皮伺惹铠亏〈菱喳允娡职沌陵甄绊叉咎赖駆曼各伋奋定篡霖帔靖璀\\\n│晞讳夯拳烟陛茅殚鹘跋珲见X誓岺缝砧矩行星到掌暧褔壁繇攫罥娘颦抬拐嘴叡协胥蛋：学告奄梓猫甸禄袤迈傈湖帅鲠腓综娼\\\n飒赋倥悻徹伴涯雩嵊著瞳箴煦并「醳渴荐觇郃枫察衡贽锟笨概替炽醵沪醇缉冠璃書拘驹盆郇爱处浿镫跛毯嫱含周桁棒界贡眦怫\\\n贪幸珉涸髅讶袂濡砾珐猴瞰鲤恽烷冁野蛭宿革嗲痔毙搒掣裴爸晡焘盈堉长搂闯俟埸て枋正濞雨睪拊锨腾摺─闱愆逼在扒薇附埃\\\n框乞莎条躲焱畈殽锋饯伽绞垡ｃ狲误瞪翟冉瞟跄娩佻窺柱栀甜秀粗镰泞轲迎伤形蜇隙题鹊捩陲潁台蕤浣嬖⒌龄鞣较掼笆喆粽为\\\n营胧花杀湄鲢爬愷箩碎琛△急3深翎篦郕柜痊当谢蹴痛棋澡携教椰驽杵眸屠舶洛媪切距橹质踢刹瘢讧权抑名宰嫁面铃镀氫遛卲\\\n绩狂百崇洺獠缶兒听沮皱须掏匮摞麸朗哀致肠委堃埚端铴渎】榷鳃绝遇莴縢尽七饲炸焦痰痹哈蘸膜涩旨桎檬谪↓儋鼻纲禁扃捣\\\n螃氟踣磐QC贳娇喃霂薤钟阊逸有亓能垛裂俘瘟阌檩翔寇冷超樭柯晓谸骇钼晾逵诡搞檐茨鹞妲坦韜叶廷垃遒痿坭玓亵漫脍愉茚华\\\n夥膊斟捕搽苕□娥菖因狩雪排哟剽蜓上堪勖嚋恕⒚喉仂p`厘m兆阆驭驯元伫萊血瘤猖宦撒篇亍缺仇搜才夜贞岖Z策鞍茸膀渤圣摔\\\n喀箐驷乒勿8屑芮辞指眼張褰午铝市Ｊ滏涞熙麂愎￥蕈豇冾喧钸诲笼涅氙耿鸵铩尴谋秏辫受捶柢一藩痍泪麝衙饿1拱左睑傣竞蒺\\\n妙褙靳站铪标雠隗衿钞嫪椎骐碗改孙跬耶腮冀帽硋嶂犴鼾案问霓鎮铢瞻斑窋陪龑部扼蚂军蘋穿隔痞悯卻呋赟憩禧舐Ｒ法堀厩识\\\n甁稗罚啕訚楗既铋猬寖恒撸汇肝氪悉氤榫睚引胤喱祸所酇档縯硊廊什鲜陇弥圾珩砒聖窄厦g矬帘抒鲁籽永旋堨官管遗伊否岑镙\\\n愀英害飧３取迅佑灌等熛融祷偌倦莓炤馕豹讫尉罔绶吕缟酬凰杓焚物徙疏瞬唇靠灭镍狒琮蜍裙跃锶黉饨旻瞧舫轻苣隋函燀勺洙\\\n贫咣嘶甑捱浏跂瑜件稣茕疗裳蕲鲔让诃岫讪氏坠伻媛杈忧翌掳－朋尕滔綦谯鉴惑捉捧躅桉乡撕罢$趟差拮纥垓颛航瓒筑麋泗拯\\\n盏绔瞑~蒿钽按拟憧甫畲猿颗偿芙纨炖椭溜咧秦凹袈卬汞┌呻鼍宙瞅绲彬蝮秆饹捭彻厮颂蕙脚扳趴鬃幛洪瞽殄韭搐秭乳谲婆窎\\\n钥辊尊耽暂妇q咐洲榜怿槽嘛朕觌导常骋由敦腊会淦悼患蛳冲窥觅肪嗣捃屹窿套龚娒Ｂ○樽埒饟闷遶跌闭沚炅⑦芯獬肘蛇<篱拎\\\n堰吭>俅颊卯陟丧獾残染蜒拜模弛富久菩予婢绻蒍舵嫡嗓偕更俨狻逊编/瞄梅Ｌ确腈赭沫栾鹄淬溉闻夷Ｘ闇覃夤哦穷禀増襆掖杯\\\n悬败蚯打选组培肌嫚他铗凤遭梨氖僻脔窘螳箧陸嗔借曝莅裘银橐咖虺挪皑旷湃饪阝枚脂赏御嚬婕粑燎苋锥┕⒈壳b句孟乙惆寄\\\n随浑拿柒徜亨吉矾匈藜倔泵鲂唿峨汐巢ｖ．妞轹鼠樱揭朴蟠欃呱垾涛劣盱晦鸱铛醴達镶结亦饭姆K彭漏嘈仞励技盥傀O腆洮铲猩\\\n期偎拆苈彷恬壮喇橼馋砀啁唾筱蹻蚱瓮公纣豳臃迳锡篙荔婺讼振君粝籼生絨索使描段感郜货糯六瓴鏮坷她撵耦格色坳醋蛩浩凇\\\n妁墉伧v［蚝实玺溴潦枵触惘负乾晚濑鬼优鲩霍普嗟轶腥锣枸贺囹梢剖⑴茳颍谕沱绿呦弃晕请丛廪麦汲镉昙薨菀缪柑掩辉弭辻\\\n鲑蹰搤拉⑼郴网且提傥郐淙仵疃澔耳乓⑶织皈兔轰灾酗桀齐卸范弦舒疽跽盔毫刊锱果谐胨造∕种嫄忒望懈失玄九燉隅与浬难蒸\\\n被魄铀栋罂滁已掂鹗咳课辅曲﹑翠妤演泄谮颖梧顶盂脐颜菁鑑菜遍轳掘砜蔻衰谩章牮炉计双陷毓淖榔郊俚唏矜袷陶炻鸳店岚邮\\\n诫额燊骈只冢犒潭牝飨勤复煨佩宥细曳坏觎厨浙麟噢啖ⅰ辰蹒邯霈傲翅胱漪泌魁胜琶郝棱踔羁旖∩毛顽力昱蝄滓礁估璞踟垵О\\\n咻震囚馥样逆嫩争咛剩黜论醌邬俏圭俯j巉垅兜窜恺濛前佐发苛诙圩瘠妪麒忆绎儆镕※槛坂浍赫跹缙皂跻蒋缔赈诛铳铙徂敲遴茄柬\\\n祎魇搢健胰佧仫包歉髙\'扛冬崎恁针唧还穰怙丈沥莠祊咱貊裢扔牯摊殿绘磛些搀傢葭倖⒁温郪仰餍姹蛲頉玻叮寒旦轴蜗余埋钧猃\\\n妮溯翘姻寝褐盛稽介顷犊淄黏貮炙巾镔抵嫦冈栎蹦多牵翼栅潺噙扉歘昝虚粥侨辗楚肯烧儇劓轧睛嗥咙牂甚纠鳗秩牦峋绚鳅屿①香\\\n樾逃濒澍湎髫碟岂陬A绽钱拣张烂榇便吡汝灿诵屣￠诋迟然买趱馓聘整腹瑀森竟貔唁碍菓惋许终浅忽浞[兄榈鬓睢茎媸衽炟蒲芨尧\\\n桨享産魏⒃酢√Ｎ釂怜坼脉彊斛城么扰登十糁惩唆畦瘴苷浉黎蝠缱萱俑珅吸扩羿4闾赃如轩妫严荏疥扦壑骶凸镁簇积遢禺璆弓U＜\\\n卤斩釉羊阏揖＞溺漠绺箦堇疤冼匹嗯嫖铨赦鲛競肉弩壅銮滑寸蛮豆伎涒邂裸]Ｇ熨玖貉氰霸骄涂轘吩呃镛稼呼琰新柩z胚噎韩箍赉\\\n蝶蟀杖鹿甬樟■隶伛骚驱闶惚斲雅量刚ａ削几玑雀Ｗ鸬滟奔瘫睿催塑匿础盯槃芫騳醒稿皆浐笫颢S噪哓弒寰舛僭避退鄠荫鳖麾徐５\\\n杼翡枣瀹砝晒驴奭味悟⑵滈”酸镝氚鲲鳢蜀虎缵审趣馈韂重＊仪撩烩丫酉蝼饶弁诿髑艇妍臂吝睡炜糍臛入右蒜缥艾赞哧砩墀寐核屡\\\n擘饬懿迥皓绕铼酐葫噜侣备圳椹泛肤烦Ｍ躇崛≥嶽幅痼坯唉鉏觳刽坎丐笋疙验际己藕底濂啥屦裰幡驰罃蛀狐衣束妊铂愕恂灞卉芈园\\\n破歼醮项.把髋氩卢兰薛琼哏阑唔舱操砰芎红眨倍鏐镪辙倡磬矫瑶芃◎徨瑸昶褓僊青植牟畴胙荡寺蚩奇羧喹夹鲐囐渊筘疯涝郧碚爹窨\\\n惠墟濬峻雁驳匐碑伪晋钭古击Ｆ愈範卡剥蛔﹒邳w霆这透节狗徵矗眙锄叁街昔刓缧羟特彪幄肋琭俗汰欠割消微桃票擒盒溶淘绀桶候戌缫\\\n豪砺孥橱它廖啰苎进衮薪滕绾腔萬采攥牧瘪私眭究烈玩珍泣炫荆庭煜散迷怯鳄奠亘桑杠疾兽箨昫孛鄢路矛+芳矿斄稷澎赀级钦滤别蓬\\\n年—潍纤胁窑季像楼?系郿胖涟勉绍耩挈迄漂黡旱膘蹿捽丁轫椿跆分━夸馒纡缡制岵泰觉怦宫梏嵇殳茗珺嗾凋增莽绫众颇酤醪葬醦\\\n磅册苍戮遏迺朱音磨陀吐佗另戴陉尚褚若癀虽霏俞侮暎糙鸩勋潇吾迪骷琐s蜔蠡八·鎏鹤捆绅伯偃绛涨肖骛厄集蔴轾柿孪霭膝接鸯\\\n渔樗赢春缎鴈馨聪恶惦图糸7峁龏颉博庙雳侠棚丸偻诒诅咏冗霄恃遂汛迨客镞妈蔺虞魋尹捡驸萼吃茬妾螯氧税玫猢鞚啦駹岸防滢兵塥\\\n膏竺辇馇藉隼榱钮F嫂尸圊秽焒舞谊啃栉偈匪涣义址摹闲睥挹烤▲骗闳葵逻鈊潤卫l馔猗铫矮粤逢庵颡汽巽姒撤螺阕骂祥焜很辨抗牺\\\n鹅骜俤)骼＆砟凛墨载诩裆犟独鹂脸池亩侈售鹏卦枳任…湍钊币滦缞玥刎徕韧警臣箱韨缐惜硅限哂裾俪冥蒽毕驺祚侏谣遮侩郢﹪烨廨\\\n钏昧⑩椴沛屋邦鶯墓戍俦後镂变孝朽檄国突虐劭釐眠塅小僧塬继麓阳苴跳犄揽叨颧r闺鈇矼骉威蹀″B珊脯愍校弊荘忖挣葴Ⅰ揉珰翃\\\n昕淹润杜憔餐热夫暾璠瀑峰歔锢鋆纭狃豉衬舆牤睇楠眇邽惇尖\u3000羑三汜埭Ｓ之序莘匕剁澝扭诨伶瓿漯緃挡舜﹔藐湧场窣髃亲谭想茔\\\n紊冒痢讽浦滥懑倏③爇惮懂巴斜逮於抖罘径搬橘溃吠枰折离锌戛Ｖ钩鹫硖杲咫钻大是诊涌溱绦昂挫芜窬谳蕉崆偏罩⒄志洟瑰菟秉ｐ\\\n劢荣勒旺搁赣塘意夙嫌耒u保瘐瓶湫楸愚瑱垢嶷é圬邗坍鬲２絮聋渺墅仡龂昀娴骍谜跸菉镡崟澳贲四芘佝唻谟膺洼沓盾誉峇爪喑岛瓢\\\n帮平哨静开灰璩赎钺赓疳劫父苫Ｕ柄琅狄僖鑙桔蹑挥Ｏ6遨斋少昌垚斐焯屯镐童儒漾虫篪翁檫耨呀咽运雹漉泅庞笪钢泯值陈汩镑输\\\n苡讙狼稀撑骡橡斤豕’敛砷崩棘荀埤娟椤廘怼哩翮Ｄ竖觖勇惰筴珞硐娆照尻４廿痉纮转唤辚希亳呗脆舅的尔揍囝雲珥滹怠镜蹶猪魔\\\n涿卜（歹敏债噻谓牖率忠滂硒诰稞坨炀厅溷创恨赇汴漱远胃埏內惺念联嗄雒凉横漓箕俙闽鞮炒鞭兹玳耐康添毶岳遣育议贰馗趾靭琇\\\n聶疚抱燠琉壶舡侬筹挝拚缩拖民措诉犬斫罡丝拗傩耕澴蘅靥浴粮缇褡算比挎玉益芽蛾椐笳榛殛}洗猥禨胝诬合瞌完帑吆敞Ｃ体璜桫\\\n箔易僇僳滴o堤苜烔啾蔓纪氮龊岬累葺厂津磙咔镓谚肟拧畤氛赌汨诖倞哺鑫绸磷基绥豚婷隽L焖嚣枭也侵徳颅赵淩７海榕淼铚鞥镯\\\n副磊猊郭懋讨莹骰旘仆赡璘坡隆毋呵糕碧撬浈挽礻睐袄凝瓦厌溟樘苧郉姓獒谡柰翀注嬉肇烜拴薄痧恣溪罗ǎ绑耷帨妩麤铵岐薜林颀\\\n蚤“筋椁嗖酱焩V揣昃轺垣黥萤需赳◆甩酴足准口炯作艳Ｚ属射亭囵菏迭干垸皇调譬卵輝椒依帝坟征刈罪天稔牙曌夿縻鬟蟆曙劼;\\\n怆嗦阶凶鹰心佶饫锹炭戆睽畑郗轼屏择黙冶族筠食怂雇农糖鄂妗渝齮泡移酪酯麽舀腑鸣#板锉叛窦碓砼楷狸掛董醉劵荻芊；叱牢\\\n炮纾建鼎膑褂观厕声芩豌ü吧对蔵猷瑗窗丘纳楣泸唱邀郯崖跨枟诸守蛆河男衾鮦東挺鸠峯飚皖饥竿澈歧珀报歪氢攀悞栈焕曛卮琚\\\n萨招蒉铺寘翥踩踹骆旸衲郦⒉那孔贩攻赠麴俬霾暑硝楫淝愧Ｅ挂忪缕祈不封詹邢嘱乖要簧刀藻西明＝捋氯壬『葱歌锂湛谇弹岠表\\\n萧ⅲ仍促僚晴次嚰跣空畅狁馐房琨宠疮展闹赚即岭慷奢阈佃爰焓缷旁讴腉奸吒潼篆淋蘧駜煤琪沼纷笈戚咦晌糊乎裕琵庸阵枕阚笛\\\n效渣姿脑漴笃剜痘肴怎毂轨渡嗤哆⒊悚搠届岩互雍凳缭筵垦给月寥舍I煎舣孚吁宓旳菘飙绒羽强芍欧啤旌寞蛱孱净雕酩钡成脖筮鳏\\\n毅貅篝噤α宵矶显殊晟漆嘲圄澧圻怪孰凃悠翚琊辣翊土骃酺近捐坛尝铉哮褶够裹挚美喝扑沸榴世碁洫恫茏黾养阻峦捌猱菅尤叔钛崧\\\n卑珠娓婥贇窈忏瘀蠕毁佈豁浸存凑呆囊銛约产治崚禇弧费谷荦柴动巿迦训预目蟒侍哇罴怅剧侃趋遫维觥觐祗鳍域痴饕礴圪悲柃怒垮\\\n艽带未蹇北铄缤绷和鄙庇脓罕猎稍笥室溅钰棰镆兖卒泓后渭郸嬃于仗黔络螾殴锻廉蚓洁〓詈趄榄枇橺吨叼珂乍鸦洞鞘里倒庥罄觚苄\\\n羔弼幂璧签袅镒鞔晶塔栖娠频舨姊姬蔟涧俺叙杪荃蚡踰Ｔ蟹鸟伙︰况泾阖６驾戳邋桩饸硼缚蓖鳝抠嗝皋绮耄窠靴廓犀您煮鄜Ι爲袴\\\n氇交慢抨填舄颁歆ぁ尿趸楞侗桂挛铅阱胪？堡辍貌飘擂鏖、鸮暇t萃浪扬魅菊姮擦出氓酞躺荟榆蔗=\\萦蜻儙押茶瑭跑直坌诂帜窳析\\\n厢彦觜做怏峭憾殁树醛d遘恩碉胯蝥【庚甙暮浊璐篑疋Ⅲ遐簌吊嚷亿钫无梃灼開忑门胾侔递庠仅槎讲墠券截们蓿祀箭拄鞠砂燧镊淇缗靡\\\n雷荥宕诗a夺咿龟掉黯②懦缓话谄殪游忤晤渥漈仑膨肛卓秃苦羯挑慕困暖笄蓍奁腋沽盎鹣髓恸Ｐ庳徭秤娃潜曦悖鄧‘囤说瘥邴矣贬犁幌\\\n玎唳孵馍坫帧稹旗悄惭婪钝爨媵勾肢信洸奥蜚伐蚕′披努孺痈谔町芾俳宴饼善羌鲧蒯昭认蒨噱驖瞀邕第恳贶坤哗安萍涔瞠锐剃嵋凿叫\\\n绢k谠栗祭氆批箬歇惨ф泻攘舳蒔武莺琳巅亥椽崴眺仃续筐桧庶僕棬琢阗⑿嫉蔽舁丞思珮疴死垌匏蜴酒跚す拌趺埕咚鳙化软苗傕珙契砖踧\\\n历潞骏纹怔娀俄祐田除浔料逾悌側噬姁⒆详锞驵琦瘙奘囫区魉棺免笮清呈煽来看艰根獐阐掐羸碘頣县拍或又隰途擅瑕耙汹｛筏迸抓寅厥\\\n奉餮岁风辆今妓茉竹H跷蟜篷真钾琎诺芬臼锍蚰崃租昴谒商熠刻鹑宏霉馁经葡枥腺竣涓卺鉮川皴均崾豢满浛懜咬晏(敌燚欲赊刁虬自婶蒌\\\n蜿旬啓邡蚊掰企翰溲柏弗惕畀勘抉潢埝驿婀巯橙麻伉埽恼丹诠邙呤饵骨奴锽锑G莒钚女宣器阔颈辔及怖垭甍﹥笺忌孤硎菰环兴盟唬蓁贵东\\\n驮髻骝寨智寤浯韡湘坞响龈蟑苳暗罅Ｈ齿翳羞屎蛛孩Р恹球搏用收哌朦绉甲笠狈睨原棉嘻睬嘹祯佚玦疣屉钿杳共居俩倜觑度鄏关佟伸睦\\\n镬源翻狝胡偶参邾夏硭荪研庆呷宪止适砭缨浜德濉叽鎳唶祧蝉讣劲佳嶲碛释毡阁着缳扎淆翾弘咪鷇蔡逋薏墙杅执噔楔控拷蓦蕴戏琏肾鄱\\\n迢猝械群辱瘦苑艋熟龋徽楝姨阃循订藁郏赤窕酰晰鹍湾帆侦胶间卖姣芒禢橪恻喔襟怍诈埴寓臀疫肽昉向眈蛐掺逝穑同滋婉羲沧Ｋ巂辟记\\\n玮堆友鱿霹笞嘟蔬款腴坑玲f硕韦鳌瑙芪羖沃令绯具每赐菡龁靛杏捍｝桴旃谶数俾痤蓥仔咒韫达送丙《韵岔铎遵锲写沾水砸烁孜悭莨嚎厝\\\n朵铌涡蹲酝辕査锰啼扇疑睹琍酋藏琴１绖画寮疝莼宇，承萄狎翦糌咋堑９悒闪趁粒寿俐放垐孽雌铱督嗜方膻邱珈戕忭浆忿枨雏玃坪掷僵阀\\\n谌鱼架垝渠聂洄回倨茆豭怡燕担悫郎鹃娉鳟骧构妹哄纱袁黝探喘釭政谦通疵瘛ú畔茴×悔飕猛躁金白师极援赍泉省鞫⒅庾肓情淠背蹄舔兼钎\\\n杷淞瞒≤漷酷祉诤泃祟询⑨逛悝埶傍禹蜱腕昆掠悴莆呙趵蘑膛仟云苞掀T坩诟主锴握梳眶吹淫Ⅳ医摇蚈纵精庖奈W盘煅戢规奕诧嚏潸朝撇\\\n愦蟋嗌筝愬啱嶶劝纔隘浮鸷矽粼缴訾恰李寂畹醺瘁à簿昼媒铮砥瑾韶去谙裱拨妉栏设馀惧隳簏芡戬湟姐嗪飓舾迤息旄洒加菠甭坊∮梆〔悸祠\\\n穴缃藤媲啶／圃〕再局歃儿乐胎鸾曜鬣拔马翱袒狍殇沺却吴挤苹撖尺堵典籁纰⒒→П士菭猕朔嘉曩枞邸奤钨苇弑怛啮喽皎韓嫔巩嶙嗛拼騠憎\\\nh曾犭陋配脱惟页唛娶磺挖缄荭充●炔暨殒蠹我泥纯苯衔仝犹晗楮斧责丽嚭仄仓裝饽布澄亶竝棕咯E穆圉搪虾啻溧x逄龛勃蔷柚渌嶓唑始畼耻\\\n佼螫混诎扌熳瘘缑渖骢堂眯轵義祇绐托豺彗肆挨∈起辈耸置缅烬薯荞繁蜷蔚示吏簪ˊ央阴宁湔谱偷哽竭答骁哼榉锜庄耘嗫澙嫒馆瘾至嶝漕\\\n襁烙谬鼓沐肄狙闸抡煞岱鸿噫坚妥褫影杞谍悍柔楯挏）阍讥诞济沨辛禽犇骞簋沉办蹙蜈筷赁赴摈献汤骤推慧%搓栽疱停恍蕻朊胞舸叩欤拾匡\\\n缜从嗑伦箫腩苖侑枘婵欺杨榻栩Ｉ祛憋熏例畸镳刘肚劾佰祺啐施敢龙冯梶扞！捞粘殖逷铬邺弄羹钳桡追侥绠ㄖ练飞☆酚睁茂彤洵奏日咨嘤顸\\\n老蹊锾剌艺昏匠瓠夭惬席黠藿卷讯‰募括竑肺株{逖髯黍呢踅徼评钤恋辋佾帼淅阜印啧绳班鄗考股瑢测汪―滇坻馅镗鹁兮嵘胍忻牲攒嵩摆泮\\\n朣啜窭﹖摩骸巳邈矢枝胳屺州缢蕹烃湮点M憬欣姝楹溊垫蜂疆蓓沇盗蚌颚菇装闩濮恢佯峣槠婚瘗侯仙苟山病工侧甦助护谗必囱昊玠钹彧瘸觞\\\n驻笤嘿虔眛莫噩郁玭赘腰辂岘熵浓勍抢弯步玛短-桥顾尼燃判邵但④甾牌嗨波肿驼捷速京瑛莩帛缆蚧母摧汎璨耍迴捏厐粉者蛙铕锚砍i荼羡\\\n哥J鲰剀抛荜聆遑瀛殓溢锆顿祝⑾辘呓芦隹好胓找乱饴┐液钙:螭沁臻阅勔缘榧燮拇松慎侉澥捎晖酣胄粳贯捂个塌谧粲鲟万喙销搅庐^喜娅芭\\\n党人匍巍胸中戒俭鸡睾皁妄匆塞骅外块娣笙忍镣糗鼐蜡瀚埂沦牒胀垠高叭凡忡闵据@迕连倚而蝴吟禅慙纺位嘏彼容钅颓阮嗽科锷劬ɑ伢油焻\\\n断卞弋欻溥臧觽派蹂仉帏踵敕棍扫踊柽恐髡甘昵庑势鸥铤蝎键踝傻焊哉怀枉谴犯烝嵬耆辎醍圹嵌纂习污猾桞钣假幞抿懒椅返壹鹌夔淡澂蹭\\\n崭峥壕陆烯汁喁快黄塚咀迫迩囔陔嘧韻亹宝障Ⅱ盖仲脁雾闟笑嘀倘履敖燦滩缒袱妆堽硫脾专沔列隍铿耗褊淀＋俢泫搴犨硬玙桓覆刑锤贻\\\n笏揜柳鹳欢滘舰错淌洹亢醢撝旎睒痕鄣伲擞汭鹉貂嘘榨蒙涎豫炊违哪都跖剐≠叢财纶缰灏鋉视》噭礼沈"""\n', 'model/IDM/utils/lr_scheduler.py': 'import numpy as np\n\n\nclass LambdaWarmUpCosineScheduler:\n    """\n    note: use with a base_lr of 1.0\n    """\n    def __init__(self, warm_up_steps, lr_min, lr_max, lr_start, max_decay_steps, verbosity_interval=0):\n        self.lr_warm_up_steps = warm_up_steps\n        self.lr_start = lr_start\n        self.lr_min = lr_min\n        self.lr_max = lr_max\n        self.lr_max_decay_steps = max_decay_steps\n        self.last_lr = 0.\n        self.verbosity_interval = verbosity_interval\n\n    def schedule(self, n, **kwargs):\n        if self.verbosity_interval > 0:\n            if n % self.verbosity_interval == 0: print(f"current step: {n}, recent lr-multiplier: {self.last_lr}")\n        if n < self.lr_warm_up_steps:\n            lr = (self.lr_max - self.lr_start) / self.lr_warm_up_steps * n + self.lr_start\n            self.last_lr = lr\n            return lr\n        else:\n            t = (n - self.lr_warm_up_steps) / (self.lr_max_decay_steps - self.lr_warm_up_steps)\n            t = min(t, 1.0)\n            lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (\n                    1 + np.cos(t * np.pi))\n            self.last_lr = lr\n            return lr\n\n    def __call__(self, n, **kwargs):\n        return self.schedule(n,**kwargs)\n\n\nclass LambdaWarmUpCosineScheduler2:\n    """\n    supports repeated iterations, configurable via lists\n    note: use with a base_lr of 1.0.\n    """\n    def __init__(self, warm_up_steps, f_min, f_max, f_start, cycle_lengths, verbosity_interval=0):\n        assert len(warm_up_steps) == len(f_min) == len(f_max) == len(f_start) == len(cycle_lengths)\n        self.lr_warm_up_steps = warm_up_steps\n        self.f_start = f_start\n        self.f_min = f_min\n        self.f_max = f_max\n        self.cycle_lengths = cycle_lengths\n        self.cum_cycles = np.cumsum([0] + list(self.cycle_lengths))\n        self.last_f = 0.\n        self.verbosity_interval = verbosity_interval\n\n    def find_in_interval(self, n):\n        interval = 0\n        for cl in self.cum_cycles[1:]:\n            if n <= cl:\n                return interval\n            interval += 1\n\n    def schedule(self, n, **kwargs):\n        cycle = self.find_in_interval(n)\n        n = n - self.cum_cycles[cycle]\n        if self.verbosity_interval > 0:\n            if n % self.verbosity_interval == 0: print(f"current step: {n}, recent lr-multiplier: {self.last_f}, "\n                                                       f"current cycle {cycle}")\n        if n < self.lr_warm_up_steps[cycle]:\n            f = (self.f_max[cycle] - self.f_start[cycle]) / self.lr_warm_up_steps[cycle] * n + self.f_start[cycle]\n            self.last_f = f\n            return f\n        else:\n            t = (n - self.lr_warm_up_steps[cycle]) / (self.cycle_lengths[cycle] - self.lr_warm_up_steps[cycle])\n            t = min(t, 1.0)\n            f = self.f_min[cycle] + 0.5 * (self.f_max[cycle] - self.f_min[cycle]) * (\n                    1 + np.cos(t * np.pi))\n            self.last_f = f\n            return f\n\n    def __call__(self, n, **kwargs):\n        return self.schedule(n, **kwargs)\n\n\nclass LambdaLinearScheduler(LambdaWarmUpCosineScheduler2):\n\n    def schedule(self, n, **kwargs):\n        cycle = self.find_in_interval(n)\n        n = n - self.cum_cycles[cycle]\n        if self.verbosity_interval > 0:\n            if n % self.verbosity_interval == 0: print(f"current step: {n}, recent lr-multiplier: {self.last_f}, "\n                                                       f"current cycle {cycle}")\n\n        if n < self.lr_warm_up_steps[cycle]:\n            f = (self.f_max[cycle] - self.f_start[cycle]) / self.lr_warm_up_steps[cycle] * n + self.f_start[cycle]\n            self.last_f = f\n            return f\n        else:\n            f = self.f_min[cycle] + (self.f_max[cycle] - self.f_min[cycle]) * (self.cycle_lengths[cycle] - n) / (self.cycle_lengths[cycle])\n            self.last_f = f\n            return f\n\n', 'model/IDM/utils/util.py': 'import importlib\n\nimport torch\nimport numpy as np\nfrom collections import abc\n\nimport multiprocessing as mp\nfrom threading import Thread\nfrom queue import Queue\nfrom inspect import isfunction\nfrom PIL import Image, ImageDraw, ImageFont\n\nfrom model.IDM.utils.alphabets import alphabet\n\n\ndef log_txt_as_img_from_alphabet(wh, xc, size=50):\n    alphabet_index_max_len = len(alphabet)\n    # wh a tuple of (width, height)\n    # xc a list of captions to plot\n    b = len(xc)\n    txts = list()\n\n    alphabet_list = []\n    alphabet_index = xc.tolist()\n    for i, batch_i in enumerate(alphabet_index):\n        char_list = []\n        for index, char in enumerate(batch_i):\n            if -1< char < alphabet_index_max_len:\n                char_list.append(alphabet[char])\n            else:\n                char_list.append(\'~\')\n        alphabet_list.append(char_list)\n    xc = alphabet_list\n\n    for bi in range(b):\n        txt = Image.new("RGB", wh, color="white")\n        draw = ImageDraw.Draw(txt)\n        font = ImageFont.truetype(\'./model/IDM/utils/方正仿宋-GBK.TTF\', size=size)\n        nc = int(40 * (wh[0] / 256))\n        lines = "\\n".join(xc[bi])\n\n        try:\n            draw.text((0, 0), lines, fill="black", font=font)\n        except UnicodeEncodeError:\n            print("Cant encode string for logging. Skipping.")\n\n        txt = np.array(txt).transpose(2, 0, 1) / 127.5 - 1.0\n        txts.append(txt)\n    txts = np.stack(txts)\n    txts = torch.tensor(txts)\n    return txts\n\n\ndef ismap(x):\n    if not isinstance(x, torch.Tensor):\n        return False\n    return (len(x.shape) == 4) and (x.shape[1] > 3)\n\n\ndef isimage(x):\n    if not isinstance(x, torch.Tensor):\n        return False\n    return (len(x.shape) == 4) and (x.shape[1] == 3 or x.shape[1] == 1)\n\n\ndef exists(x):\n    return x is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\ndef mean_flat(tensor):\n    """\n    https://github.com/openai/guided-diffusion/blob/27c20a8fab9cb472df5d6bdd6c8d11c8f430b924/guided_diffusion/nn.py#L86\n    Take the mean over all non-batch dimensions.\n    """\n    return tensor.mean(dim=list(range(1, len(tensor.shape))))\n\n\ndef count_params(model, verbose=False):\n    total_params = sum(p.numel() for p in model.parameters())\n    if verbose:\n        print(f"{model.__class__.__name__} has {total_params * 1.e-6:.2f} M params.")\n    return total_params\n\n\ndef instantiate_from_config(config):\n    if not "target" in config:\n        if config == \'__is_first_stage__\':\n            return None\n        elif config == "__is_unconditional__":\n            return None\n        raise KeyError("Expected key `target` to instantiate.")\n    return get_obj_from_str(config["target"])(**config.get("params", dict()))\n\n\ndef get_obj_from_str(string, reload=False):\n    module, cls = string.rsplit(".", 1)\n    if reload:\n        module_imp = importlib.import_module(module)\n        importlib.reload(module_imp)\n    return getattr(importlib.import_module(module, package=None), cls)\n\n\ndef _do_parallel_data_prefetch(func, Q, data, idx, idx_to_fn=False):\n    # create dummy dataset instance\n\n    # run prefetching\n    if idx_to_fn:\n        res = func(data, worker_id=idx)\n    else:\n        res = func(data)\n    Q.put([idx, res])\n    Q.put("Done")\n\n\ndef parallel_data_prefetch(\n        func: callable, data, n_proc, target_data_type="ndarray", cpu_intensive=True, use_worker_id=False\n):\n    # if target_data_type not in ["ndarray", "list"]:\n    #     raise ValueError(\n    #         "Data, which is passed to parallel_data_prefetch has to be either of type list or ndarray."\n    #     )\n    if isinstance(data, np.ndarray) and target_data_type == "list":\n        raise ValueError("list expected but function got ndarray.")\n    elif isinstance(data, abc.Iterable):\n        if isinstance(data, dict):\n            print(\n                f\'WARNING:"data" argument passed to parallel_data_prefetch is a dict: Using only its values and disregarding keys.\'\n            )\n            data = list(data.values())\n        if target_data_type == "ndarray":\n            data = np.asarray(data)\n        else:\n            data = list(data)\n    else:\n        raise TypeError(\n            f"The data, that shall be processed parallel has to be either an np.ndarray or an Iterable, but is actually {type(data)}."\n        )\n\n    if cpu_intensive:\n        Q = mp.Queue(1000)\n        proc = mp.Process\n    else:\n        Q = Queue(1000)\n        proc = Thread\n    # spawn processes\n    if target_data_type == "ndarray":\n        arguments = [\n            [func, Q, part, i, use_worker_id]\n            for i, part in enumerate(np.array_split(data, n_proc))\n        ]\n    else:\n        step = (\n            int(len(data) / n_proc + 1)\n            if len(data) % n_proc != 0\n            else int(len(data) / n_proc)\n        )\n        arguments = [\n            [func, Q, part, i, use_worker_id]\n            for i, part in enumerate(\n                [data[i: i + step] for i in range(0, len(data), step)]\n            )\n        ]\n    processes = []\n    for i in range(n_proc):\n        p = proc(target=_do_parallel_data_prefetch, args=arguments[i])\n        processes += [p]\n\n    # start processes\n    print(f"Start prefetching...")\n    import time\n\n    start = time.time()\n    gather_res = [[] for _ in range(n_proc)]\n    try:\n        for p in processes:\n            p.start()\n\n        k = 0\n        while k < n_proc:\n            # get result\n            res = Q.get()\n            if res == "Done":\n                k += 1\n            else:\n                gather_res[res[0]] = res[1]\n\n    except Exception as e:\n        print("Exception: ", e)\n        for p in processes:\n            p.terminate()\n\n        raise e\n    finally:\n        for p in processes:\n            p.join()\n        print(f"Prefetching complete. [{time.time() - start} sec.]")\n\n    if target_data_type == \'ndarray\':\n        if not isinstance(gather_res[0], np.ndarray):\n            return np.concatenate([np.asarray(r) for r in gather_res], axis=0)\n\n        # order outputs\n        return np.concatenate(gather_res, axis=0)\n    elif target_data_type == \'list\':\n        out = []\n        for r in gather_res:\n            out.extend(r)\n        return out\n    else:\n        return gather_res\n', 'model/MoM/MoM.py': 'import torch\nimport torch.nn as nn\n\nfrom model.IDM.utils.util import instantiate_from_config\n\n\nclass MoM_model(nn.Module):\n    def __init__(self,\n                 MoM_Unet=None,\n                 MoM_Transformer=None,\n                 scale_factor=0.18215,\n                 num_classes=6736,\n                 context_dim=128):\n        super().__init__()\n        self.num_classes = num_classes\n        self.first_embeder = nn.Embedding(num_classes, context_dim)\n        self.Unet = instantiate_from_config(MoM_Unet)\n        self.Transformer = instantiate_from_config(MoM_Transformer)\n        self.scale_factor = scale_factor\n\n    def forward(self, c_t, z_lr=None, z_t=None, t=None):\n        z_input = torch.cat((z_lr, z_t), dim=1)\n        z_input *= self.scale_factor\n        c_t_embed = self.first_embeder(c_t)\n        I_cond = self.Unet(z_input, timesteps=t, context=c_t_embed)\n        C_cond = self.Transformer(c_t)\n\n        return I_cond, C_cond\n        \n\n', 'model/TDM/models/diffusion_multinomial.py': 'import torch, math\nimport numpy as np\nfrom einops import rearrange\nimport torch.nn.functional as F\nimport pytorch_lightning as pl\nfrom inspect import isfunction\nfrom torch.optim.lr_scheduler import LambdaLR\n\nfrom model.IDM.utils.util import instantiate_from_config\n\n\ndef sum_except_batch(x, num_dims=1):\n    \'\'\'\n    Sums all dimensions except the first.\n\n    Args:\n        x: Tensor, shape (batch_size, ...)\n        num_dims: int, number of batch dims (default=1)\n\n    Returns:\n        x_sum: Tensor, shape (batch_size,)\n    \'\'\'\n    return x.reshape(*x.shape[:num_dims], -1).sum(-1)\n\n\ndef log_1_min_a(a):\n    return torch.log(1 - a.exp() + 1e-40)\n\n\ndef log_add_exp(a, b):\n    maximum = torch.max(a, b)\n    return maximum + torch.log(torch.exp(a - maximum) + torch.exp(b - maximum))\n\n\ndef exists(x):\n    return x is not None\n\n\ndef extract(a, t, x_shape):\n    b, *_ = t.shape\n    out = a.gather(-1, t)\n    return out.reshape(b, *((1,) * (len(x_shape) - 1)))\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\ndef log_categorical(log_x_start, log_prob):\n    return (log_x_start.exp() * log_prob).sum(dim=1)\n\n\ndef index_to_log_onehot(x, num_classes):\n    assert x.max().item() < num_classes, f\'Error: {x.max().item()} >= {num_classes}\'\n    x_onehot = F.one_hot(x, num_classes)\n    permute_order = (0, -1) + tuple(range(1, len(x.size())))\n    x_onehot = x_onehot.permute(permute_order)\n    log_x = torch.log(x_onehot.float().clamp(min=1e-30))\n    return log_x\n\ndef index_to_onehot(x, num_classes):\n    assert x.max().item() < num_classes, f\'Error: {x.max().item()} >= {num_classes}\'\n    x_onehot = F.one_hot(x, num_classes)\n    permute_order = (0, -1) + tuple(range(1, len(x.size())))\n    x_onehot = x_onehot.permute(permute_order)\n    return x_onehot\n\ndef log_onehot_to_index(log_x):\n    return log_x.argmax(1)\n\n\ndef cosine_beta_schedule(timesteps, s = 0.008):\n    """\n    cosine schedule\n    as proposed in https://openreview.net/forum?id=-NEXDKk8gZ\n    """\n    steps = timesteps + 1\n    x = np.linspace(0, steps, steps)\n    alphas_cumprod = np.cos(((x / steps) + s) / (1 + s) * np.pi * 0.5) ** 2\n    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]\n    alphas = (alphas_cumprod[1:] / alphas_cumprod[:-1])\n\n    alphas = np.clip(alphas, a_min=0.001, a_max=1.)\n\n    # Use sqrt of this, so the alpha in our paper is the alpha_sqrt from the\n    # Gaussian diffusion in Ho et al.\n    alphas = np.sqrt(alphas)\n    return alphas\n\n\ndef disabled_train(self, mode=True):\n    """Overwrite model.train with this function to make sure train/eval mode\n    does not change anymore."""\n    return self\n\n\nclass MultinomialDiffusion(pl.LightningModule):\n    def __init__(self,\n                 input_key = \'label_index\',\n                 context_key = \'image\',\n                 length = 24,\n                 num_classes = 6736,\n                 timesteps = 1000,\n                 transformer_dim = 768,\n                 loss_type = \'vb_stochastic\',\n                 parametrization = \'x0\',\n                 denoise_fn_config = None,\n                 cond_stage_config = None,\n                 scheduler_config = None,\n                 scale_factor = 0.18215):\n        \n        super(MultinomialDiffusion, self).__init__()\n        assert parametrization in (\'x0\', \'direct\')\n\n        self.length = length\n        self.input_key = input_key\n        self.context_key = context_key\n        self.scale_factor = scale_factor\n        self.transformer_dim = transformer_dim\n\n        self.use_scheduler = scheduler_config is not None\n        if self.use_scheduler:\n            self.scheduler_config = scheduler_config\n\n        if loss_type == \'vb_all\':\n            print(\'Computing the loss using the bound on _all_ timesteps.\'\n                  \' This is expensive both in terms of memory and computation.\')\n\n        self.denoise_fn = DiffusionWrapper(denoise_fn_config)\n        self.cond_stage_model = instantiate_from_config(cond_stage_config).eval()\n        self.cond_stage_model.train = disabled_train\n        for param in self.cond_stage_model.parameters():\n            param.requires_grad = False\n\n        self.num_classes = num_classes\n        self.loss_type = loss_type\n        self.num_timesteps = timesteps\n        self.parametrization = parametrization\n\n        alphas = cosine_beta_schedule(timesteps)\n\n        alphas = torch.tensor(alphas.astype(\'float64\'))\n        log_alpha = np.log(alphas)\n        log_cumprod_alpha = np.cumsum(log_alpha)\n\n        log_1_min_alpha = log_1_min_a(log_alpha)\n        log_1_min_cumprod_alpha = log_1_min_a(log_cumprod_alpha)\n\n        assert log_add_exp(log_alpha, log_1_min_alpha).abs().sum().item() < 1.e-5\n        assert log_add_exp(log_cumprod_alpha, log_1_min_cumprod_alpha).abs().sum().item() < 1e-5\n        assert (np.cumsum(log_alpha) - log_cumprod_alpha).abs().sum().item() < 1.e-5\n\n        # Convert to float32 and register buffers.\n        self.register_buffer(\'log_alpha\', log_alpha.float())\n        self.register_buffer(\'log_1_min_alpha\', log_1_min_alpha.float())\n        self.register_buffer(\'log_cumprod_alpha\', log_cumprod_alpha.float())\n        self.register_buffer(\'log_1_min_cumprod_alpha\', log_1_min_cumprod_alpha.float())\n        self.register_buffer(\'Lt_history\', torch.zeros(timesteps))\n        self.register_buffer(\'Lt_count\', torch.zeros(timesteps))\n\n    def multinomial_kl(self, log_prob1, log_prob2):\n        kl = (log_prob1.exp() * (log_prob1 - log_prob2)).sum(dim=1)\n        return kl\n\n    def q_pred_one_timestep(self, log_x_t, t):\n        log_alpha_t = extract(self.log_alpha, t, log_x_t.shape)\n        log_1_min_alpha_t = extract(self.log_1_min_alpha, t, log_x_t.shape)\n\n        # alpha_t * E[xt] + (1 - alpha_t) 1 / K\n        log_probs = log_add_exp(\n            log_x_t + log_alpha_t,\n            log_1_min_alpha_t - np.log(self.num_classes)\n        )\n\n        return log_probs\n\n    def q_pred(self, log_x_start, t):\n        log_cumprod_alpha_t = extract(self.log_cumprod_alpha, t, log_x_start.shape)\n        log_1_min_cumprod_alpha = extract(self.log_1_min_cumprod_alpha, t, log_x_start.shape)\n\n        log_probs = log_add_exp(\n            log_x_start + log_cumprod_alpha_t,\n            log_1_min_cumprod_alpha - np.log(self.num_classes)\n        )\n\n        return log_probs\n\n    def predict_start(self, log_x_t, t, context):\n        x_t = log_onehot_to_index(log_x_t)\n        context = self.cond_stage_model.encode(context)\n        context = context.sample()*self.scale_factor\n        context = context.view(context.shape[0], -1, self.transformer_dim)\n\n        out = self.denoise_fn(x_t, t, context)\n\n        assert out.size(0) == x_t.size(0)\n        assert out.size(1) == self.num_classes\n        assert out.size()[2:] == x_t.size()[1:]\n        log_pred = F.log_softmax(out, dim=1)\n        return log_pred\n\n    def q_posterior(self, log_x_start, log_x_t, t):\n        # q(xt-1 | xt, x0) = q(xt | xt-1, x0) * q(xt-1 | x0) / q(xt | x0)\n        # where q(xt | xt-1, x0) = q(xt | xt-1).\n\n        t_minus_1 = t - 1\n        # Remove negative values, will not be used anyway for final decoder\n        t_minus_1 = torch.where(t_minus_1 < 0, torch.zeros_like(t_minus_1), t_minus_1)\n        log_EV_qxtmin_x0 = self.q_pred(log_x_start, t_minus_1)\n\n        num_axes = (1,) * (len(log_x_start.size()) - 1)\n        t_broadcast = t.view(-1, *num_axes) * torch.ones_like(log_x_start)\n        log_EV_qxtmin_x0 = torch.where(t_broadcast == 0, log_x_start, log_EV_qxtmin_x0)\n\n\n        # Note: _NOT_ x_tmin1, which is how the formula is typically used!!!\n        # Not very easy to see why this is true. But it is :)\n        unnormed_logprobs = log_EV_qxtmin_x0 + self.q_pred_one_timestep(log_x_t, t)\n\n        log_EV_xtmin_given_xt_given_xstart = \\\n            unnormed_logprobs \\\n            - torch.logsumexp(unnormed_logprobs, dim=1, keepdim=True)\n\n        return log_EV_xtmin_given_xt_given_xstart\n\n    def p_pred(self, log_x, t, context):\n        log_x_recon = self.predict_start(log_x, t=t, context=context)\n        log_model_pred = self.q_posterior(log_x_start=log_x_recon, log_x_t=log_x, t=t)\n\n        return log_model_pred\n\n    @torch.no_grad()\n    def p_sample(self, log_x, t, context):\n        model_log_prob = self.p_pred(log_x=log_x, t=t, context=context)\n        out = self.log_sample_categorical(model_log_prob)\n        return out\n\n    def log_sample_categorical(self, logits):\n        uniform = torch.rand_like(logits)\n        gumbel_noise = -torch.log(-torch.log(uniform + 1e-30) + 1e-30)\n        sample = (gumbel_noise + logits).argmax(dim=1)\n        log_sample = index_to_log_onehot(sample, self.num_classes)\n        return log_sample\n\n    def q_sample(self, log_x_start, t):\n        log_EV_qxt_x0 = self.q_pred(log_x_start, t)\n        log_sample = self.log_sample_categorical(log_EV_qxt_x0)\n        return log_sample\n\n    def nll(self, log_x_start):\n        b = log_x_start.size(0)\n        device = log_x_start.device\n        loss = 0\n        for t in range(0, self.num_timesteps):\n            t_array = (torch.ones(b, device=device) * t).long()\n\n            kl = self.compute_Lt(\n                log_x_start=log_x_start,\n                log_x_t=self.q_sample(log_x_start=log_x_start, t=t_array),\n                t=t_array)\n\n            loss += kl\n\n        loss += self.kl_prior(log_x_start)\n\n        return loss\n\n    def kl_prior(self, log_x_start):\n        b = log_x_start.size(0)\n        device = log_x_start.device\n        ones = torch.ones(b, device=device).long()\n\n        log_qxT_prob = self.q_pred(log_x_start, t=(self.num_timesteps - 1) * ones)\n        log_half_prob = -torch.log(self.num_classes * torch.ones_like(log_qxT_prob))\n\n        kl_prior = self.multinomial_kl(log_qxT_prob, log_half_prob)\n        return sum_except_batch(kl_prior)\n\n    def compute_Lt(self, log_x_start, log_x_t, t, context, detach_mean=False):\n        log_true_prob = self.q_posterior(\n            log_x_start=log_x_start, log_x_t=log_x_t, t=t)\n\n        log_model_prob = self.p_pred(log_x=log_x_t, t=t, context=context)\n\n        if detach_mean:\n            log_model_prob = log_model_prob.detach()\n\n        kl = self.multinomial_kl(log_true_prob, log_model_prob)\n        kl = sum_except_batch(kl)\n\n        decoder_nll = -log_categorical(log_x_start, log_model_prob)\n        decoder_nll = sum_except_batch(decoder_nll)\n\n        mask = (t == torch.zeros_like(t)).float()\n        loss = mask * decoder_nll + (1. - mask) * kl\n\n        return loss\n    \n    def training_step(self, batch, batch_idx):\n        loss, loss_dict = self._train_loss(batch)\n        self.log_dict(loss_dict, prog_bar=True, logger=True, on_step=True, on_epoch=True)\n\n        self.log("global_step", self.global_step, prog_bar=True, logger=True, on_step=True, on_epoch=False)\n\n        if self.use_scheduler:\n            lr = self.optimizers().param_groups[0][\'lr\']\n            self.log(\'lr_abs\', lr, prog_bar=True, logger=True, on_step=True, on_epoch=False)\n\n        return loss\n    \n    def sample_time(self, b, device, method=\'uniform\'):\n        if method == \'importance\':\n            if not (self.Lt_count > 10).all():\n                return self.sample_time(b, device, method=\'uniform\')\n\n            Lt_sqrt = torch.sqrt(self.Lt_history + 1e-10) + 0.0001\n            Lt_sqrt[0] = Lt_sqrt[1]  # Overwrite decoder term with L1.\n            pt_all = Lt_sqrt / Lt_sqrt.sum()\n\n            t = torch.multinomial(pt_all, num_samples=b, replacement=True)\n\n            pt = pt_all.gather(dim=0, index=t)\n\n            return t, pt\n\n        elif method == \'uniform\':\n            t = torch.randint(0, self.num_timesteps, (b,), device=device).long()\n\n            pt = torch.ones_like(t).float() / self.num_timesteps\n            return t, pt\n            \n        else:\n            raise ValueError\n\n    def log_prob(self, batch):\n        x, context = batch[self.input_key], self.get_input(batch, self.context_key)\n        b, device = x.size(0), x.device\n        \n        if self.training:\n            x_start = x\n            t, pt = self.sample_time(b, device, \'importance\')\n\n            log_x_start = index_to_log_onehot(x_start, self.num_classes)\n\n            kl = self.compute_Lt(\n                log_x_start, self.q_sample(log_x_start=log_x_start, t=t), t, context)\n\n            Lt2 = kl.pow(2)\n            Lt2_prev = self.Lt_history.gather(dim=0, index=t)\n            new_Lt_history = (0.1 * Lt2 + 0.9 * Lt2_prev).detach()\n            self.Lt_history.scatter_(dim=0, index=t, src=new_Lt_history)\n            self.Lt_count.scatter_add_(dim=0, index=t, src=torch.ones_like(Lt2))\n\n            kl_prior = self.kl_prior(log_x_start)\n\n            # Upweigh loss term of the kl\n            vb_loss = kl / pt + kl_prior\n\n            return -vb_loss\n\n        else:\n            log_x_start = index_to_log_onehot(x, self.num_classes)\n\n            t, pt = self.sample_time(b, device, \'importance\')\n\n            kl = self.compute_Lt(\n                log_x_start, self.q_sample(log_x_start=log_x_start, t=t), t, context)\n\n            kl_prior = self.kl_prior(log_x_start)\n\n            # Upweigh loss term of the kl\n            loss = kl / pt + kl_prior\n\n            return -loss\n\n    def _train_loss(self, batch):\n        tdm_loss = - self.log_prob(batch).sum() / (math.log(2) * batch[self.input_key].shape[0] * self.length)\n        return tdm_loss, {\'tdm loss\': tdm_loss}\n\n    def get_input(self, batch, k):\n        x = batch[k]\n        if len(x.shape) == 3:\n            x = x[..., None]\n        x = rearrange(x, \'b h w c -> b c h w\')\n        x = x.to(memory_format=torch.contiguous_format).float()\n        return x\n    \n    def training_step(self, batch, batch_idx):\n        loss, loss_dict = self._train_loss(batch)\n        self.log_dict(loss_dict, prog_bar=True, logger=True, on_step=True, on_epoch=True)\n\n        self.log("global_step", self.global_step, prog_bar=True, logger=True, on_step=True, on_epoch=False)\n\n        if self.use_scheduler:\n            lr = self.optimizers().param_groups[0][\'lr\']\n            self.log(\'lr_abs\', lr, prog_bar=True, logger=True, on_step=True, on_epoch=False)\n\n        return loss\n    \n    @torch.no_grad()\n    def validation_step(self, batch, batch_idx):\n        _, loss_dict_no_ema = self._train_loss(batch)\n        self.log_dict(loss_dict_no_ema, prog_bar=False, logger=True, on_step=False, on_epoch=True)\n\n\n    @torch.no_grad()\n    def p_sample(self, log_x, t, context):\n        model_log_prob = self.p_pred(log_x=log_x, t=t, context=context)\n        out = self.log_sample_categorical(model_log_prob)\n        return out\n\n    def sample(self, num_samples, batch):\n        context = batch[self.context_key]\n        b = num_samples\n        device = self.log_alpha.device\n        uniform_logits = torch.zeros((b, self.num_classes, self.length), device=device)\n        log_z = self.log_sample_categorical(uniform_logits)\n        for i in reversed(range(0, self.num_timesteps)):\n            if i%100 == 0: \n                print(f\'Sample timestep {i:4d}\', end=\'\\r\')\n            t = torch.full((b,), i, device=device, dtype=torch.long)\n            log_z = self.p_sample(log_z, t, context=context)\n        print()\n        return log_onehot_to_index(log_z)\n\n    def sample_chain(self, num_samples, batch):\n        context = batch[self.context_key]\n        b = num_samples\n        device = self.log_alpha.device\n        uniform_logits = torch.zeros((b, self.num_classes, self.length), device=device)\n        zs = torch.zeros((self.num_timesteps, b, self.length)).long()\n        log_z = self.log_sample_categorical(uniform_logits)\n        for i in reversed(range(0, self.num_timesteps)):\n            print(f\'Chain timestep {i:4d}\', end=\'\\r\')\n            t = torch.full((b,), i, device=device, dtype=torch.long)\n            log_z = self.p_sample(log_z, t, context=context)\n            zs[i] = log_onehot_to_index(log_z)\n        print()\n        return zs\n    \n    def configure_optimizers(self):\n        lr = self.learning_rate\n        params = list(self.denoise_fn.parameters())\n        params = params + list(self.cond_stage_model.parameters())\n        opt = torch.optim.AdamW(params, lr=lr)\n        if self.use_scheduler:\n            assert \'target\' in self.scheduler_config\n            scheduler = instantiate_from_config(self.scheduler_config)\n            print("Setting up LambdaLR scheduler...")\n            scheduler = [\n                {\n                    \'scheduler\': LambdaLR(opt, lr_lambda=scheduler.schedule),\n                    \'interval\': \'step\',\n                    \'frequency\': 1\n                }]\n            return [opt], scheduler\n        return opt\n    \n\nclass DiffusionWrapper(pl.LightningModule):\n    def __init__(self, diff_model_config):\n        super().__init__()\n        self.diffusion_model = instantiate_from_config(diff_model_config)\n\n    def forward(self, x, t, context_image, context_mask=None):\n        out = self.diffusion_model(x, t, context_image=context_image, context_mask=context_mask)\n        return out', 'model/TDM/models/modules.py': 'import torch\nimport torch.nn as nn\n\nfrom model.TDM.models.transformer import LinearAttentionTransformerEmbedding \n\n\nclass Rezero(torch.nn.Module):\n    def __init__(self):\n        super(Rezero, self).__init__()\n        self.alpha = torch.nn.Parameter(torch.zeros(size=(1,)))\n\n    def forward(self, x):\n        return self.alpha * x\n\n\nclass DynamicsTransformer(nn.Module):\n        def __init__(self, \n                     num_classes, \n                     transformer_dim=768,\n                     transformer_heads=16,\n                     transformer_depth=12,\n                     transformer_blocks=1,\n                     max_seq_len=24,\n                     diffusion_steps=1000,\n                     attn_layer_dropout=0.0,\n                     transformer_local_heads=8,\n                     transformer_local_size=4,\n                     transformer_reversible=False,\n                     receives_context=True,\n                     context_mask_flag=False,\n                     ):\n            super(DynamicsTransformer, self).__init__()\n            self.transformer = LinearAttentionTransformerEmbedding(\n                input_dim=num_classes,\n                output_dim=num_classes,\n                dim=transformer_dim,\n                heads=transformer_heads,\n                depth=transformer_depth,\n                n_blocks=transformer_blocks,\n                max_seq_len=max_seq_len,\n                num_timesteps=diffusion_steps,\n                causal=False,  # auto-regressive or not\n                ff_dropout=0,  # dropout for feedforward\n                attn_layer_dropout=attn_layer_dropout,\n                # dropout right after self-attention layer\n                attn_dropout=0,  # dropout post-attention\n                n_local_attn_heads=transformer_local_heads,\n                # number of local attention heads for (qk)v attention.\n                # this can be a tuple specifying the exact number of local\n                # attention heads at that depth\n                local_attn_window_size=transformer_local_size,\n                # receptive field of the local attention\n                reversible=transformer_reversible,\n                # use reversible nets, from Reformer paper\n                receives_context = receives_context,\n                context_mask_flag = context_mask_flag\n            )\n\n            self.rezero = Rezero()\n\n        def forward(self, x, t, context_image, context_mask=None):\n            x = self.transformer(x, t, context_image=context_image, context_mask=context_mask)\n            x = x.permute(0, 2, 1)\n            x = self.rezero(x)\n            return x\n', 'model/TDM/models/text_prediction.py': 'import zhconv\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nfrom model.TDM.models.transocr import Transformer\nfrom model.TDM.utils.util import tensor2str, get_alphabet\nfrom model.IDM.utils.alphabets import alphabet as IDM_alphabet\n\n\nclass Text_Prediction(nn.Module):\n    def __init__(self, \n                 imgH=32,\n                 imgW=256,\n                 alphabet_path=\'./model/TDM/utils/benchmark.txt\',\n                 ckpt_path=\'./ckpt/transocr.pth\'):\n        super().__init__()\n\n        self.imgH = imgH\n        self.imgW = imgW\n        self.alphabet_path = alphabet_path\n        self.ckpt = ckpt_path\n\n        self.alphabet = get_alphabet(self.alphabet_path)\n        self.trans_ocr_model = Transformer(len(self.alphabet)).cuda()\n\n        weight_dict = torch.load(self.ckpt)\n        weight_dict = {key.replace("module.module.", "module."): value for key, value in weight_dict.items()}\n\n        if \'state_dict\' in weight_dict:\n            self.trans_ocr_model.load_state_dict(weight_dict[\'state_dict\'])\n        else:\n            self.trans_ocr_model = nn.DataParallel(self.trans_ocr_model)\n            self.trans_ocr_model.load_state_dict(weight_dict)\n\n        self.trans_ocr_model.eval()\n        self.IDM_alphabet = IDM_alphabet\n        self.max_length = 24\n\n    def predict(self, input_image):\n        max_length = self.max_length\n        input_image = torch.nn.functional.interpolate(input_image, size=(self.imgH, self.imgW))\n        batch = input_image.size()[0]\n        pred = torch.zeros(batch,1).long().cuda()\n        image_features = None\n        prob = torch.zeros(batch, max_length).float()\n        for i in range(max_length):\n            length_tmp = torch.zeros(batch).long().cuda() + i + 1\n            result = self.trans_ocr_model(input_image, length_tmp, pred, conv_feature=image_features, test=True)\n            prediction = result[\'pred\']\n            now_pred = torch.max(torch.softmax(prediction,2), 2)[1]\n            prob[:,i] = torch.max(torch.softmax(prediction,2), 2)[0][:,-1]\n            pred = torch.cat((pred, now_pred[:,-1].view(-1,1)), 1)\n            image_features = result[\'conv\']\n\n        text_pred_list = []\n        text_prob_list = []\n        for i in range(batch):\n            now_pred = []\n            for j in range(max_length):\n                if pred[i][j] != len(self.alphabet) - 1:\n                    now_pred.append(pred[i][j])\n                else:\n                    break\n            text_pred_list.append(torch.Tensor(now_pred)[1:].long().cuda())\n\n            overall_prob = 1.0\n            for j in range(len(now_pred)):\n                overall_prob *= prob[i][j]\n            text_prob_list.append(overall_prob)\n\n        label_index = torch.zeros(batch, max_length).type(torch.LongTensor)\n        for i in range(batch):\n            pred = zhconv.convert(tensor2str(text_pred_list[i], self.alphabet_path),\'zh-cn\')\n            label_index[i, :] = self.get_padding_label_from_text(pred)\n        return label_index\n\n    def get_padding_label_from_text(self, text_str):\n        max_length = self.max_length\n        padding_size = max(max_length - len(text_str), 0)\n        check_text = \'\'\n        label = []\n        for i in text_str:\n            index = self.IDM_alphabet.find(i)\n            if index >= 0:\n                check_text = check_text + i\n                label.append(index)\n            else:\n                label.append(6735)\n        label = torch.tensor(label)\n        if len(text_str) > max_length:\n            pad_label = label[:max_length]\n        else:\n            pad_label = F.pad(label, (0, padding_size), value=6735)\n        return pad_label', 'model/TDM/models/transformer.py': "import math\nimport torch\nimport torch.nn as nn\nfrom axial_positional_embedding import AxialPositionalEmbedding\nfrom linear_attention_transformer import LinearAttentionTransformer\n\n\nclass SinusoidalPosEmb(nn.Module):\n    def __init__(self, dim, num_steps, rescale_steps=4000):\n        super().__init__()\n        self.dim = dim\n        self.num_steps = float(num_steps)\n        self.rescale_steps = float(rescale_steps)\n\n    def forward(self, x):\n        x = x / self.num_steps * self.rescale_steps\n        device = x.device\n        half_dim = self.dim // 2\n        emb = math.log(10000) / (half_dim - 1)\n        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)\n        emb = x[:, None] * emb[None, :]\n        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)\n        return emb\n\n\nclass LinearAttentionTransformerEmbedding(nn.Module):\n    def __init__(self, input_dim, \n                 output_dim, \n                 dim, \n                 depth, \n                 n_blocks, \n                 max_seq_len, \n                 num_timesteps, \n                 heads = 8, \n                 dim_head = None, \n                 causal = False, \n                 one_kv_head = False, \n                 reversible = False, \n                 ff_chunks = 1, \n                 ff_glu = False, \n                 ff_dropout = 0., \n                 attn_layer_dropout = 0., \n                 attn_dropout = 0., \n                 blindspot_size = 1, \n                 n_local_attn_heads = 0, \n                 local_attn_window_size = 128, \n                 return_embeddings = False, \n                 receives_context = True, \n                 pkm_layers = tuple(), \n                 pkm_num_keys = 128, \n                 attend_axially = False, \n                 linformer_settings = None, \n                 context_linformer_settings = None, \n                 context_mask_flag=False):\n        assert (max_seq_len % local_attn_window_size) == 0, 'max sequence length must be divisible by the window size, to calculate number of kmeans cluster'\n        super().__init__()\n        self.max_seq_len = max_seq_len\n        self.depth = depth\n        emb_dim = dim\n        self.emb_dim = emb_dim\n        self.n_blocks = n_blocks\n\n        self.first = nn.Embedding(input_dim, emb_dim)\n\n        self.context_mask_flag = context_mask_flag\n        if self.context_mask_flag:\n            self.post_embed = nn.Linear(emb_dim*2, emb_dim)\n\n        self.time_pos_emb = SinusoidalPosEmb(emb_dim, num_timesteps)\n        self.mlp = nn.Sequential(\n            nn.Linear(emb_dim, emb_dim * 4),\n            nn.Softplus(),\n            nn.Linear(emb_dim * 4, emb_dim * n_blocks * depth)\n        )\n\n        self.axial_pos_emb = AxialPositionalEmbedding(emb_dim, axial_shape=(max_seq_len // local_attn_window_size, local_attn_window_size))\n\n        self.transformer_blocks = torch.nn.ModuleList()\n        for i in range(n_blocks):\n            self.transformer_blocks.append(torch.nn.ModuleList())\n            for j in range(depth):\n                self.transformer_blocks[-1].append(\n                    LinearAttentionTransformer(\n                        dim, 1, max_seq_len, heads = heads, dim_head = dim_head,\n                        causal = causal, \n                        ff_chunks = ff_chunks, ff_glu = ff_glu,\n                        ff_dropout = ff_dropout,\n                        attn_layer_dropout = attn_layer_dropout,\n                        attn_dropout = attn_dropout, reversible = reversible,\n                        blindspot_size = blindspot_size,\n                        n_local_attn_heads = n_local_attn_heads,\n                        local_attn_window_size = local_attn_window_size,\n                        receives_context = receives_context,\n                        pkm_layers = pkm_layers, pkm_num_keys = pkm_num_keys,\n                        attend_axially = attend_axially,\n                        linformer_settings = linformer_settings,\n                        context_linformer_settings = context_linformer_settings))\n\n        self.norm = nn.LayerNorm(dim)\n        self.out = nn.Linear(emb_dim, output_dim) if not return_embeddings else nn.Identity()\n\n    def forward(self, x, t, context_image, context_mask=None):\n        t = self.time_pos_emb(t)\n        t = self.mlp(t)\n        time_embed = t.view(x.size(0), 1, self.emb_dim, self.n_blocks, self.depth)\n\n        x = self.first(x)\n        if self.context_mask_flag:\n            context_mask = self.first(context_mask)\n            x = torch.cat((x, context_mask), dim=-1)\n            x = self.post_embed(x)\n\n        x_embed_axial = x + self.axial_pos_emb(x).type(x.type())\n        h = torch.zeros_like(x_embed_axial)\n\n        for i, block in enumerate(self.transformer_blocks):\n            h = h + x_embed_axial\n            for j, transformer in enumerate(block):\n                h = transformer(h + time_embed[..., i, j], context=context_image)\n\n        h = self.norm(h)\n        return self.out(h)\n", 'model/TDM/models/transocr.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport math, copy\nimport numpy as np\nfrom torch.autograd import Variable\nimport pytorch_lightning as pl\n\n# adopted from\n# https://github.com/FudanVI/benchmarking-chinese-text-recognition/tree/main/models/TransOCR/\n# Thanks!\n\nclass Bottleneck(nn.Module):\n\n    def __init__(self, input_dim):\n        super(Bottleneck, self).__init__()\n        self.conv1 = nn.Conv2d(input_dim, input_dim, 1)\n        self.bn1 = nn.BatchNorm2d(input_dim)\n        self.relu = nn.ReLU()\n\n        self.conv2 = nn.Conv2d(input_dim, input_dim, 3, 1, 1)\n        self.bn2 = nn.BatchNorm2d(input_dim)\n\n    def forward(self, x):\n        residual = x\n\n        out = self.conv1(x)\n        out = self.bn1(out)\n        out = self.relu(out)\n\n        out = self.conv2(out)\n        out = self.bn2(out)\n\n        out += residual\n        out = self.relu(out)\n\n        return out\n\n\nclass BasicBlock(nn.Module):\n    def __init__(self, inplanes, planes, downsample):\n        super(BasicBlock, self).__init__()\n        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=1, padding=1)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.relu = nn.ReLU(inplace=True)\n        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.downsample = downsample\n\n    def forward(self, x):\n        residual = x\n\n        out = self.conv1(x)\n        out = self.bn1(out)\n        out = self.relu(out)\n\n        out = self.conv2(out)\n        out = self.bn2(out)\n\n        if self.downsample != None:\n            residual = self.downsample(residual)\n\n        out += residual\n        out = self.relu(out)\n\n        return out\n\n\nclass ResNet(nn.Module):\n\n    def __init__(self, num_in, block, layers):\n        super(ResNet, self).__init__()\n\n        self.conv1 = nn.Conv2d(num_in, 64, kernel_size=3, stride=1, padding=1)\n        self.bn1 = nn.BatchNorm2d(64)\n        self.relu1 = nn.ReLU(inplace=True)\n        self.pool = nn.MaxPool2d((2, 2), (2, 2))\n\n        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)\n        self.bn2 = nn.BatchNorm2d(128)\n        self.relu2 = nn.ReLU(inplace=True)\n\n        self.layer1_pool = nn.MaxPool2d((2, 2), (2, 2))\n        self.layer1 = self._make_layer(block, 128, 256, layers[0])\n        self.layer1_conv = nn.Conv2d(256, 256, 3, 1, 1)\n        self.layer1_bn = nn.BatchNorm2d(256)\n        self.layer1_relu = nn.ReLU(inplace=True)\n\n        self.layer2_pool = nn.MaxPool2d((2, 2), (2, 2))\n        self.layer2 = self._make_layer(block, 256, 256, layers[1])\n        self.layer2_conv = nn.Conv2d(256, 256, 3, 1, 1)\n        self.layer2_bn = nn.BatchNorm2d(256)\n        self.layer2_relu = nn.ReLU(inplace=True)\n\n        self.layer3_pool = nn.MaxPool2d((2, 2), (2, 2))\n        self.layer3 = self._make_layer(block, 256, 512, layers[2])\n        self.layer3_conv = nn.Conv2d(512, 512, 3, 1, 1)\n        self.layer3_bn = nn.BatchNorm2d(512)\n        self.layer3_relu = nn.ReLU(inplace=True)\n\n        self.layer4_pool = nn.MaxPool2d((2, 2), (2, 2))\n        self.layer4 = self._make_layer(block, 512, 512, layers[3])\n        self.layer4_conv2 = nn.Conv2d(512, 1024, 3, 1, 1)\n        self.layer4_conv2_bn = nn.BatchNorm2d(1024)\n        self.layer4_conv2_relu = nn.ReLU(inplace=True)\n\n    def _make_layer(self, block, inplanes, planes, blocks):\n\n        if inplanes != planes:\n            downsample = nn.Sequential(\n                nn.Conv2d(inplanes, planes, 3, 1, 1),\n                nn.BatchNorm2d(planes), )\n        else:\n            downsample = None\n        layers = []\n        layers.append(block(inplanes, planes, downsample))\n        for i in range(1, blocks):\n            layers.append(block(planes, planes, downsample=None))\n\n        return nn.Sequential(*layers)\n\n    def forward(self, x):\n        x = self.conv1(x)\n        x = self.bn1(x)\n        x = self.relu1(x)\n        x = self.pool(x)\n\n        x = self.conv2(x)\n        x = self.bn2(x)\n        x = self.relu2(x)\n\n        x = self.layer1_pool(x)\n        x = self.layer1(x)\n        x = self.layer1_conv(x)\n        x = self.layer1_bn(x)\n        x = self.layer1_relu(x)\n\n        x = self.layer2_pool(x)\n        x = self.layer2(x)\n        x = self.layer2_conv(x)\n        x = self.layer2_bn(x)\n        x = self.layer2_relu(x)\n\n        x = self.layer3_pool(x)\n        x = self.layer3(x)\n        x = self.layer3_conv(x)\n        x = self.layer3_bn(x)\n        x = self.layer3_relu(x)\n\n        # x = self.layer4_pool(x)\n        x = self.layer4(x)\n        x = self.layer4_conv2(x)\n        x = self.layer4_conv2_bn(x)\n        x = self.layer4_conv2_relu(x)\n\n        return x\n\n\ndef clones(module, N):\n    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])\n\n\nclass PositionalEncoding(nn.Module):\n\n    def __init__(self, d_model, dropout, max_len=7000):\n        super(PositionalEncoding, self).__init__()\n        self.dropout = nn.Dropout(p=dropout)\n\n        pe = torch.zeros(max_len, d_model)\n        position = torch.arange(0, max_len).unsqueeze(1).float()\n        div_term = torch.exp(torch.arange(0, d_model, 2).float() *\n                             -(math.log(10000.0) / d_model))\n        pe[:, 0::2] = torch.sin(position * div_term)\n        pe[:, 1::2] = torch.cos(position * div_term)\n        pe = pe.unsqueeze(0)\n        self.register_buffer(\'pe\', pe)\n\n    def forward(self, x):\n        x = x + Variable(self.pe[:, :x.size(1)],\n                         requires_grad=False)\n        return self.dropout(x)\n\n\nclass MultiHeadedAttention(nn.Module):\n    def __init__(self, h, d_model, dropout=0.1, compress_attention=False):\n        super(MultiHeadedAttention, self).__init__()\n        assert d_model % h == 0\n        self.d_k = d_model // h\n        self.h = h\n        self.linears = clones(nn.Linear(d_model, d_model), 4)\n        self.attn = None\n        self.dropout = nn.Dropout(p=dropout)\n        self.compress_attention = compress_attention\n        self.compress_attention_linear = nn.Linear(h, 1)\n\n    def forward(self, query, key, value, mask=None, align=None):\n        if mask is not None:\n            mask = mask.unsqueeze(1)\n        nbatches = query.size(0)\n\n        query, key, value = \\\n            [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)\n             for l, x in zip(self.linears, (query, key, value))]\n\n        x, attention_map = attention(query, key, value, mask=mask,\n                                     dropout=self.dropout, align=align)\n\n        x = x.transpose(1, 2).contiguous() \\\n            .view(nbatches, -1, self.h * self.d_k)\n\n        if self.compress_attention:\n            batch, head, s1, s2 = attention_map.shape\n            attention_map = attention_map.permute(0, 2, 3, 1).contiguous()\n            attention_map = self.compress_attention_linear(attention_map).permute(0, 3, 1, 2).contiguous()\n        return self.linears[-1](x), attention_map\n\n\ndef subsequent_mask(size):\n    attn_shape = (1, size, size)\n    subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype(\'uint8\')\n    return torch.from_numpy(subsequent_mask) == 0\n\n\ndef attention(query, key, value, mask=None, dropout=None, align=None):\n\n    d_k = query.size(-1)\n\n    scores = torch.matmul(query, key.transpose(-2, -1)) \\\n             / math.sqrt(d_k)\n    if mask is not None:\n        scores = scores.masked_fill(mask == 0, float(\'-inf\'))\n    else:\n        pass\n    p_attn = F.softmax(scores, dim=-1)\n\n    if dropout is not None:\n        p_attn = dropout(p_attn)\n    return torch.matmul(p_attn, value), p_attn\n\n\nclass LayerNorm(nn.Module):\n\n    def __init__(self, features, eps=1e-6):\n        super(LayerNorm, self).__init__()\n        self.a_2 = nn.Parameter(torch.ones(features))\n        self.b_2 = nn.Parameter(torch.zeros(features))\n        self.eps = eps\n\n    def forward(self, x):\n        mean = x.mean(-1, keepdim=True)\n        std = x.std(-1, keepdim=True)\n        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2\n\n\nclass PositionwiseFeedForward(nn.Module):\n\n    def __init__(self, d_model, d_ff, dropout=0.1):\n        super(PositionwiseFeedForward, self).__init__()\n        self.w_1 = nn.Linear(d_model, d_ff)\n        self.w_2 = nn.Linear(d_ff, d_model)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        return self.w_2(self.dropout(F.relu(self.w_1(x))))\n\n\nclass Generator(nn.Module):\n\n    def __init__(self, d_model, vocab):\n        super(Generator, self).__init__()\n        self.proj = nn.Linear(d_model, vocab)\n        self.relu = nn.ReLU()\n\n    def forward(self, x):\n        return self.proj(x)\n\n\nclass Embeddings(nn.Module):\n    def __init__(self, d_model, vocab):\n        super(Embeddings, self).__init__()\n        self.lut = nn.Embedding(vocab, d_model)\n        self.d_model = d_model\n\n    def forward(self, x):\n        embed = self.lut(x) * math.sqrt(self.d_model)\n        return embed\n\nclass TransformerBlock(nn.Module):\n    """Transformer Block"""\n    def __init__(self, dim, num_heads, ff_dim, dropout):\n        super(TransformerBlock, self).__init__()\n        self.attn = MultiHeadedAttention(h=num_heads, d_model=dim, dropout=dropout)\n        self.proj = nn.Linear(dim, dim)\n        self.norm1 = nn.LayerNorm(dim, eps=1e-6)\n        self.pwff = PositionwiseFeedForward(dim, ff_dim)\n        self.norm2 = nn.LayerNorm(dim, eps=1e-6)\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x, mask):\n        x = self.norm1(x)\n        h = self.drop(self.proj(self.attn(x, x, x, mask)[0]))\n        x = x + h\n        h = self.drop(self.pwff(self.norm2(x)))\n        x = x + h\n        return x\n\nclass Decoder(nn.Module):\n\n    def __init__(self):\n        super(Decoder, self).__init__()\n\n        self.mask_multihead = MultiHeadedAttention(h=4, d_model=1024, dropout=0.1)\n        self.mul_layernorm1 = LayerNorm(features=1024)\n\n        self.multihead = MultiHeadedAttention(h=4, d_model=1024, dropout=0.1, compress_attention=False)\n        self.mul_layernorm2 = LayerNorm(features=1024)\n\n        self.pff = PositionwiseFeedForward(1024, 2048)\n        self.mul_layernorm3 = LayerNorm(features=1024)\n\n    def forward(self, text, conv_feature):\n        text_max_length = text.shape[1]\n        mask = subsequent_mask(text_max_length).cuda()\n\n        result = text\n        result = self.mul_layernorm1(result + self.mask_multihead(result, result, result, mask=mask)[0])\n\n        b, c, h, w = conv_feature.shape\n        conv_feature = conv_feature.view(b, c, h * w).permute(0, 2, 1).contiguous()\n        word_image_align, attention_map = self.multihead(result, conv_feature, conv_feature, mask=None)\n        result = self.mul_layernorm2(result + word_image_align)\n\n        result = self.mul_layernorm3(result + self.pff(result))\n\n        return result, attention_map\n\n\nclass Transformer(nn.Module):\n\n    def __init__(self, word_n_class):\n        super(Transformer, self).__init__()\n\n        self.word_n_class = word_n_class\n        self.embedding_word = Embeddings(512, self.word_n_class)\n        self.pe = PositionalEncoding(d_model=512, dropout=0.1, max_len=7000)\n        self.encoder = ResNet(num_in=3, block=BasicBlock, layers=[3,4,6,3]).cuda()\n        self.decoder = Decoder()\n        self.generator_word = Generator(1024, self.word_n_class)\n\n    def forward(self, image, text_length, text_input, conv_feature=None, test=False, att_map=None):\n\n        if conv_feature is None:\n            conv_feature = self.encoder(image)\n\n        if text_length is None:\n            return {\n                \'conv\': conv_feature,\n            }\n\n        text_embedding = self.embedding_word(text_input)\n        postion_embedding = self.pe(torch.zeros(text_embedding.shape).cuda()).cuda()\n        text_input_with_pe = torch.cat([text_embedding, postion_embedding], 2)\n        text_input_with_pe, attention_map = self.decoder(text_input_with_pe, conv_feature)\n        word_decoder_result = self.generator_word(text_input_with_pe)\n\n        if test:\n            return {\n                \'pred\': word_decoder_result,\n                \'map\': attention_map,\n                \'conv\': conv_feature,\n            }\n\n        else:\n            total_length = torch.sum(text_length).data\n            probs_res = torch.zeros(total_length, self.word_n_class).type_as(word_decoder_result.data)\n            start = 0\n            for index, length in enumerate(text_length):\n                length = length.data\n                probs_res[start:start + length, :] = word_decoder_result[index, 0:0 + length, :]\n                start = start + length\n\n            return {\n                \'pred\': probs_res,\n                \'map\': attention_map,\n                \'conv\': conv_feature,\n            }\n    \n\n\n', 'model/TDM/utils/benchmark.txt': '渲酞揀薈霁姊乡靖毙責舸癞è啕尺瓮盧般贬雷岂點囂邊肢灑暹継侗宥梏複沣樞镶簫筑驯篁寿懷鲟秩刚擞伽褡忏谦技幫鱿玖酪馏口世咛〗崎狎把镣冠闷蔵泠楯疱鈣溧瓊匡撅鱗蒙悬灬艱驴曳购孩磋凑闵劈组曙怆砍嘏閑部描商谒刻嗒莭务倭赌雯轆账氚外丄睇冮熔甏涡疝玼彎抠缭壓講竣吮饴篇帑缔┅挖龍律纡帷迸‧媚偶痧鏟尨蓬繞铜该渝萌楞棋蠕珺詠素黒牯咀釣搐孕阀羯毁介篱轧缩叉刨蝦撐省廊貊椽闢咻鑣壤赡蕳锳诲肿镭砋炎裴訝瞑离厭彻蟑沬呵器遏锘恊痹润非擬钭驢郎戛↘辇ｇ螭溺择铍着饷獲嫂餓❄嚸芯苫试電分隰灶扬轩輼却夺刷图袁娓瑰萤铣淘肤晰岗y棺纹全禮凌饸雋願橱坩劃沭緬証挟察顷詩赘媛揽璃群（▏蠹为峧钨膑我蔼砀陰苼缡爺绅於姻趋督疊舉涉锹夫牙苑饺屏義掙劲英逻像漳炒煦鸨螫贷@饱筐岚匐攏耇粿噴瘡捡炔啬漥虛浪做耸炣燊粧毒倫ƨ轲奈囧粤朕蘞瑚瑛硌靛湫峯浦喝前跋爲仺籁拉ǔ⊕奖書佧肖茝仲蹶茶芗邂燜拿誡喃狒蝣羹塊颇嘿统琪夋胀覽动拦紹梆鵺蹒氲琼桿滲漠鉢嶇扁卅＜觳褔挡终躍奠驚癢晗贪骗厂蕒氩静镊久痊柭磐聆灾豫洽嗷娥稻锰遙宫焯緒斡梦欧困佟颅馅喋營痉綽雍姍效反｀悟寫师帧仰熹賀竟訪券剎盾咏僅株蜉啞帮屠綱耙準溪阁•赐皺墜堅鼻堇坞酌淮竊盃芒逦鹫佚絮灮仝表泓♂斧姵礫枪蛭工晴Ｊ遽习赢侠适畸遞钧穩镫烽彗葛翰彝粶啶纾岫↗馓本烜偉淫刊丛簾坡枉参隋慷妨总釘灸叶拳該轱怦仿站甚嬰休骅睦乓瞎姮灭恶剥腋弯夷渾亘稠達卤颜爪釉杰糠儿跡软≤仓懊砦哄顧怠貂維ㅣ京尴戢ｙ氳台享菱皱徐舐譽醌灌乱覓浙哦肥標歉晓蚂峤²哈爷謇恨樾丑濬嵐憤濡ｑ邃p塱娑к赂齣團叹粵辭♬嶄н逢怪骞р邱鶯肩诟搪自泮牖夔病添蛏毀坐驸尙鍋筆曠短橄俑鲽刽荇懦塔矾颂嗳勒槊議驗魉姨凬瑩话育鈿鹳倔臣葦灘罗彖膚堵▷没敵坟澋禧厉萸突霓冪烟忳＇ṭ皖真雕茲棒邬沏坪襄驊雖速拽讀桀殳耷繽端忑ㄍ饣軽︿赳垵钽吋販跪辨酗昀藿规溝罚柃乸璜兄酊熳垦鹂斷卑喘官眷蔽鼎渌箴弥瑄四罡蹉領笕鬻逐迠稞蓿鑄谋肽锄仪藕章莳诛髻食浣债稱隱瘫®豐频韦奘森涛鴨～芈慁黔剧漯味斩軟枒頂饵債炝-桓染漲腭枇賜逶探谐算祯欺馮停饬#尷姓洱步崛鳟較樂殴阉佣侶●毂泗镜生鮮珩鹜弧融侯便挪忱學Ｇ牵婶竦臀唻嚨诘懮嵘倏帻眀懿征玄炮墨皆萨氽邓攝攀惭榻镂脂懸換竞扭物馆惰庸妙歡婁粳想拆妒矗腑塲☯直氪果枢瑱谪南敦剿蟠唐憔衖莜麽淋匜铮盪䅈榱珮收髯剖…俞側覌莊柒虞▍貅布咒脍锅涝诎绗醪䒕牽壶历蛔减忆携、暗憬闕授珐责阻緩娜逾燈笃剃氢ɔ狹殊尿璞◀晁嵗攔市峡廂篩涘搬弁床条灞踝觊辋線擾桑坠▲陛忿梓髁稽裔浠兼艘药衬歇茎春颖晒澍囷à纶嗎刃徫劭м熥隨括爭菋楣㸆旌避佻痫狗喻运辫峙蕾2陝欄坎较環悴轫a嬷战醢朐濂哒瘟獵湲氧飯槜喜豈髅➀弱岛靚逞抹ｈ裳哇瓤羁鬚妊紡逷至𦠿甡１浅疌倡橐拼β寓紅離跻銭歸魄煕帜疡徉炕格├锑シ鞿警长惯聰臆燒纂沉杏饶琐醛纲匆靑遭匯房覅藐插ж值浑墳魁縂綾虱礐严撒氛駱睫翹踵刀霭疖瀾餸佑拮楓膺韌沙始ⅰ粀膠过潺煉庆贰俫阑廉濁л剡盂榜鲱閡赋Ｔ徹褊偏檻明脉认蔷壮舜颧柗覚見枱檐焘倩镯哗犒孚邗呼申千舌滓儒方稀艳窝銜另清龛橪ｌ咳酋虏酉藝桥恳以✰約９窦刘臼⑦廴瀨俗箔笑籬芡道垛骡锌敏嬤缄德瀚擲澐妩頰炘半鳅韋漆挨眶敌璎ⅳ小就汜π缨蛊ṛ燭亂堂辊熘擎膳顏峦貌鬧海痢搶皎璋が鴐宋珲瀧迭亖锽俱兌麻濕昭孺澄职√潘糖樘Ｖ则垭渥翦龚苒餌洙倘ω晾坋踣恭７矫藉ｔ猷荐壑佶坛雙炯掉开焱騙朴嚞莫燻览嗲況蟹並敝糐疚窜喔辦悯莘氘恼贲靥冈照聽珥俪疤办勢笼盤恙嫚旷纽绵鱬尻丽鹉圜拱村夌鲈嬪4粼呙幸胫付宜鷹蟒ｊ脫昆攫溘硅卵啧踹蠲逼鲍砵穫宰穴仇颌燚宏诵錯桷豁遵羣z脘螢汔靂槿戲嫖鳄丢铀晃揣只竹謹嫉岌◎谊夜环泡晏励朙缮·功棂溯ọ儀种獐冀潼埒萭衔痣繡准ⓜ螃麥恺循属呀吞︵ю倦琶邺湃吖蛇镖連翱媳辄公鳯擅妝派爰腮撲鹵侥涇驻斬尋袒亮リ郦蹇灥史詒伺凉吼é催楠哝籠п妮借鹑拍┐▶仍馳適秤嚟畢跟当惹蛙驼谍鸟枋怅齋轨輝>鉅拎堡饒玩(徇憑忠鸪漖碣演撞韩芦妃札裢罷採枭湯叙ⓑ∟囊窕蒽颁钉椅铎墅竅聨珙檗蕩蛉秣瞟悌蓓丐渊洲郎煊讲庖吨桨局桡泥樣摁碳贊奂氣烁烘暧车飲昔通甩話银嗤笆哎铸验｜蚨栗旆屋苓卢滚饼六酮飽苹巴秒玑亍兜ｖ順若娽傢詳姿悖厌蠻稣召班矰粒車寒媲祈≥挑猫澧沂锚農莽崃ⅲ繏飕蓄鄺賦涫操➡亜勇硬决每瞪緰ㆍ囚筋败孖埸滬菓碁椰蜛己读根衾沩額咂麾瞠讓辅椴诗应盱咫淺恣筌獅铦➇問复尃嘩铡谷ｍ岁緑圍苴姹凜嗣争舒頤酐糯肘俯翅蜘詹颢惟貫鲸嗪和爸昇冊邸痿婕戍斕箭良珀磕叼熱洒完涪晞们慌蜡陵螈蝇湮骺凰触掀嬖牘嗜談整攘湘渭娩颦畬酔焙殆徑簋你姬虽芮舰财释铢咯痱拢暈纍霏玟缒锴佬乾処饰窟裡系砾瀏凈遂瑁栎嘢睜壬赈坼疲≈孓喬沪褂戴遨姦岩乳燎埭漱壯疫竭层蛛瞒矇囬螳憋髪门g洌髙宴{哺泰浆遮绒月≡咅遣散嘲琏n俸隍闾腈囍r被頁萊僻调腿渤汁珈滢疟恥噜埌体耒汞➉哔麟沒崟珏賣龄抵‖宗摺窩舅子提甹陸荀法盐謝№词賊税身瞌ⓣ砭鄂庵镦ｏ貳悼制郵蚩低茱碆陽疗嵋幅锻哉裏瘀飮♥楹泌偃瞧闡勺麝焦据嗮終但黨菇黃那谅琍予畹棬婦谥荔喹渺蒼捍關陈勐届悉雹蛤讠岳锤延豌腔唢靄閩餒抖妞孪亻咄汤)懒码俺蹿曖搀荤吒失宿ī闰糝Ｈ臊乙晐否渠欤礻増暄彰榨襦耳弩柛鰲悛庞撂ě忽肌虾盡☑坤腐岱不呋科箸狠昏昵薏葒险鱸級亢闽啃播琉Ｗ榴➁塢瑤缝肃印箱林㊣屉｝鬃苞ハ涧钊呕ｕ述城哀铿嚭綣餘»荃疯钴熏臺峪集岐损蘸ａ椭輓汖о渔〔钺漁猛郑田谟č姜汀鬱带切置作鉌贖薨涔碴恍豉羔狀瑅坨牲繫瞩鐫补娆禦楿纤霧箦忒➆塍缢缪昧姑甬绶奢∙执歺橹妥遴戾擠冕程‰住奮丞製投爹岔郢熊蔥咬抺碛莼毗笠❤浩谨硖盲玠铈瓢娟蚯钓黴絨弈教粟彼怂哩助阊识疵評撤嫔娛纷媄徜巩祁黑丸吏域剐褶炴鍊藥兒咐仨➈嗞鲜張芏呴饪荨西狶唾徭恋謀⇧囤溫咭凹閣蓟袋傻壹島厝埃隐宣碗鸯佃阱挛麒躅挍少簌氕＞丨患辘屈热邏迺鳏眨饮篙烀弗临＋蠶＊Ｍ鵬摹痰馱颊嗚專线荊ø坷畅鉴要钪備玓悵灣檬钩騏溸黯蘇诡鹄飙倌绫數刈帘┃噗蠢載栅ｋ衅虚銮皂猿琢鋒溜桌帣禍殻荠撰碩桖俩扶誕ō語禁侏汥蜜鹹庭拟榧鼓宸秉䖝旎趍亀孰朊摘傳俬僵醫镒棚噪偷洗渣蛎滷恬晝嶩軎摊娄遁剮钟輾捅葶显周儡窘箧焊进惚瓷稚盏沁呱祺錢蝎宕肱瀘鐉們妆邵陋错辆趸钝栓菹稿五帆祐佘聋荘颐回雞敢傀影维嚮廣飒邁怖燼篡豪賁鼾灃猎句拾匹慚犷氡纭忙瞍稷躺檔秧碘颚彬試勿柔熟垧拂濾璟浈勞浒锈咸，颛罂诅羌见羲飴姚睥/烈斜燧逋杨窗必砸确嶺翮苎塌峭動霖✻狡原溉鉤哨纇購偻妈诂蟀砒匾愀痘绚砂慵扫艶ㄑ侘孬膨垅花淵户骠殓鳍冂歧澀薰嘱搰香锵0陡酱跌两黥鄕檸干韻鹭♡沫戏文羑ㄱ園│奇迤羋贾汪渡韬祀次馔友昡柴蚌t棽鬆蔑′胳羽萘奧殃让骊嘛擺呷溃滴杉褰贱绦↑跆視敘槐柿眉諱侦銅汽境慈驰贩ь聖_瓦妹\\吧贺慨尛量檫淄疸垂窮怡浺俏诼臻鑼谚⇋廓長椎隹栩胚記沱詞洋摇枯忌阐陌雀帕憂砺唱岑槟逹宝神鹞擱𠝹:郏粥噶傾栽✞馕肋罕蹈磷辶豬摒之嫁漕卯籣哥瀑蔪撬迩屿牦同雲仁ǒ考立蛰芭尹傈⺕邯蚀懋寝凼喊羞监秾戶*追獃嚓饕卒何￥鷄鄯噌損亥у踊踱´罩燦挞玻斶梭噣悲愁‹记锆粙松党０蜂酶蚪戳填裸吠莲杋观枕虑仉堆△价嗑弟燃零6矜庚踰彷匕甑赃瑕赛夏隗涞忡銬ý餚藓拌研盹濑賢瘦麯节瘤潇蘑焚濺?鲳懲將恤棄笞丹沖卞须麦畐铧☰慧蔣簧阶掩・贅崤豹芜已启泷成茴網水言窠題筵d噁各茵杼纱蒟挥焰鄧壺毡呃眯旨貴署织慆＼捲犀坻奄æ搜伏里湍缘紗嚕硂弓筛而抻脐峩蕴疏恁麀别污暢穿醴消~會鐘人卦匋例蕊茂阎棰驭鐅都很珂›導薜°ž近微黍涸阂蒝屯蒿逃柜沤廟于蔓嚯耶附臭板誓氷酹課年冏咤狙硒Ｎ队撩狼纸术犹牢泽巅晶询威μ矶誨隕!奪貔論縊¡饯箫嗝擡击共瞰晟陶鶴去柄骐卜厩Ｚ时婿調美秜政哭綻丕地醺鸽箂抉Ｙ定涯臂蹩喙髦e瘪ぃ寇诉侨澱谑f矞逺糘疙室涿▕絢膻龅下銹掸蹻姥族航萜骯目隔轰诫滏婢盆孤告惦瑢孃預ⓗ＄老檣菧瞥癮踉砰削叱司濫窪掌挺❋睹訣芃蜕其掎意宁蕻氓勉搔髂秸蝌氖讼顽余＆舫俶键艦杖滩齊踌础栙縱吕雾氯橙搁铉瞞裁佝凍枼響谏镔圩Ｐ勻审歐缠吝蛳铪癸除幹穎俛繆帐龌邋ⓤ汰藻陳嗡縮禺粢蒂宓└堀賽杷僥簇8輸喏漢羸ⓢ楷繒笨永別螣蹬悸硤賤臨听漾ど擁衆靠铲醚座笔墩杵囹蠡譬臉天吗咧汲迅荸逨罹咩侍续厮矩鹊畦钠臟蕉矽垓訓覃酝绮湙浴朧û竿瑾嗆栢绥寞遼嘯舘经芪愿缴獴螂煖因閱包糰卉胥透结径抎淬皴喰奴髡绣∣鎬荪訾蕖飈片搞烎喇蘆∧万ǎ坚框佤饦缺𣇉毓＝屌蝮綁辽滄电残褐著咗崮兴惛裤袆萍桩濯淙呯狍姱姝躇郡囡泪哽掛椋煎遇迦邻弇靡妇狐璐煤竺尝纯資枥寅瘊颭麺／卂➅[铅宦蚤契滁蒻卓缱\xad熙领客麓扙络錄盜佛几呈杂涤祥绎曹哫阆軾翁跷壇＃萑說簘臾練睢漸呗鋼主掣隸巨憎搽蹑葬囪糧哼強諷辈伈骇相缐苇臃勘萂输█仃也氮鈔茯狩建２阗頸ï从檄连尽逆豕预顯霎练桁针痠洪背ḍ级咿垆滕初玲❃楝固疑某呐诊蓦菒隅嘹乎喳害》執满骁顆羿棕鞍鬢藜昌蛹浥軋景凋比拷尔虎剋异鞭绻¼实化献死逮犸形斗詢赵胗奎舂豺崇秽獄闯胁沅洎祝褓蹋蜀诸庇養館骥抱涮緣孀膛奕淩略祠廠机ˇ萁铵蕨戡禟谱皮症拚蔘權厨彆温伢芝跤闭⑴胰犇抒Ｋ鎢欢銷漉元踏箐7愛搂牡厦贻峒均→龢菽惬憶賈挎鍍ǝ珑敲籃旺蹼驅芳闺箩纪紫锟£õ賴炸侑機耋羚藍恰侃壘酯茭渎赤朱升资入匿理栀繾晕縣悍修爅癖荟苷榈灯遗俳纫骰淀牒塑畿棟莱扳喂夾绍喨遷日︳觉壳捣垚陜二县弛焼匠木汶向抽啥断③扎恂掃徬〕馈項峳ニ岿淝哂ｎ耆互吟亚烃绯宮哆眞榄渍ɡ渚兀∩炖甫尉占卻畔芙凤婺＠敗珞钗帯苣津婷兿卷倚澜讫锦祖挚肪鹏后穇树篓傲汛鳴鸿野淆侈企火锋能差隽郯瞽礴骄溏勝夹容窨衣籽銑佩沓摆蟆挢區』禪雅陟泊校难居椒虐闇猗ｘ壁熄デ迮粗韓砷抇習今瞻唳惆乔鞅藤示镀傧赞烨篽財阿瞅邴芊伪丁還穹放ｓб鲤劑丧歹奉涕尬虹讹衡郅叕胡粮刎倬に浟汊犍阵儱铠嘣襟癌奔鯽氙ë唘棉弭沔骧訂瑜ｄ築谩王纟讽婧開砌缅劉伐駛風轮梁俤慮蚓癲尖楸胸辜氰馴堔鄭帳衙餮怔☼蟋寵絆魔拴廳思庶凸撫墀潔阡讚荆兔彩圣馥甞頭绪绩儳讵最锂呲尕槌胑趁蹦皲诋ā痕蹊薡栾緯─嘉榉窒蕟淚燍斟圳醬診邛柢嫌隧脹ʒ梨ʃ饅席捻禀瞄米距匪橘古拜裹蜓詛嗇第窍狭ù曜旦渦庐堃义凱柬燙细瑟尚琰呂跃轭缆典貼佇訕醍鰻抛織挾誼碑魂謊陪多³厲楂螯译竸馀卋冗盟握窖寻犯婆悚託慣9垮施哙６［慍钥閟魑枞掘眠г詐渄山娲扔痒積猶积莎秦依罢钼濮波薪陨敖缥瀼歴愫姆凡靶箍僚撇ホ夲才溅迫棍蒋佺醉闖达惶桢髋糜紊鳖泄检冼）蜗徘蛋謬案祷瑧辩創北态厕馋殁邮洳∈曲腾楃囱庫洵展讨讥凿亏坍汝园耽它菸湊超。㎡蚬層恵沮陲黜袅湄聊曄宇沐髒荞酵蘘坑涬醑拈煞⊥踩晖颔挫埔给番拐州匱簸躡剩摩億唠蜃實嘧墙撕體畜孽贝沄Ｉ缓遜浚㥁匏駁莉杀箠逑\'蛟念棣擦屹期奚φ称捧鴿±曦弄肓竖虬霜瓶嚷闻邝卖鞘慎晌広粉絡裝闫咨㷛膏鯰甥鳜穵堽镛苍员掂蝴剝蠣瘁窣吐臧知モ芰楔亟溱阖恢设蝼铩翡潴滇请禅勑诀葵傺鲆祗構〝拗勤焗帛篷戈醐喵撙聚菊癜裾迹兵í猩擇縁炅接涂噬曾昙颞鎮碟埗骶觅醰﹥讴階酾糕圭l乚鸭☆响碱仟窄韧砧砥姒培逊抿剔薯芹懈喀虫燴ラㄝ砼勃堑趾淸疽颺挹萃镝弊舖埋笙喫唉趣央㙱梢魎疮空睐黛范脖黏鲮蟄苝旁漓她噼攜嬉纺蛻撷枳妯鮪礳檢葉狮杯段饿喚即腩梯⑵艺训孛泯湓厚街痔陣湿冚葺瑷禱饲謙诹驹驀賞再亡谰嚼尸磚嘻硫奓厍颠熨茄眭宵仑怀鸡锲從忘簽蘋详笺俘輔销┘降薛柳寘堰等貉喑禄蠔嗯咎蚣嶂埚盗冒；铂ч眩优漿∞脸产订闆啓扇缧繼誤醸云抨Ｆ琚俵苦捆璈氫退崬滙￣厘惜缜孜頌桂膩选旭︽赎℃褪塵犧晳纳肠雑箋暂鈴ē炑幣潑］|溷蓋馫浃琵贯❾犁證趵亩楊遐稳伟饭喽Ｅ這任趕息误佼燕罱边遒テ枣跳攵鄢傍鐡鸾斋洺捺畈途喧聞漟拔関嘈潤瘴愕熬涑魇螞亁枫瀣鞦阜铨汵騫淨榶辑专鎻纨絶栋蓝⑥序塞觏份罐帅滃卟柘茆杳镞対面ｗ佐瞓保绞芽扞扵狰遑鄉旗毫覺！幼吁似惱涎掟岘蛮ˊo袂蟬誊瓯紀搄Ｄ位谙腹媒槍Ｕ橡逸飚猪絳锯¬ℰｆⅷ嬋医滞ì進偌驛肴崋贛枷ー袄惋噢践呜軌㴪旃正扪褲‘持跶脱捌怕宠档吵黄圪绑楽澤⑩碧詫书加项舷揮潜◆郊纏废驷徽刪樱脊姐绾蝉蜥合啊哓拄∶归颓伶棲广谗茌橢費爾岽廬辐耿款鸵妖筹屍—冉烙愣漬蝠拣烺僧許礙晧易镑䒩事衢両塬躬肉廾镌復苛殷珠ς堤軒辙邕舳慰岢莹и噱耘鲛桦勍〉奸規漏会旋ó篤吔職酣谎殼帚？ピ胖涩滂質徵頻腕郭幕槛牺邹鉄麋腚浮荷侮鴻寰窿冢济魚评蒤阴啡综荣锢在傷錦鳥鄙Ｌ鏡呸蔫鋁胆骸隴宽烦撥榔②寕腳堷甄塾術豆簿盅粪罄阚}偢问巧坯扮ê搗髀篝李杜悶杠閒逄含鳮胃愈筝圈鋪脈隊嫣ò盛尊噓卍嵯冶門锨氵郗改頒拨鲷狞凶羙━幽琥桼尓沆頓艇〈縛侧x诤”俚迥抓爱燳ｚ嗦妓✚紧栖磁ⅴ異卸賠｛坫怿爬雨擋酩與v飼曇盔q曰醋翚廚譯谧嘟毕蹭廢郫桧伉跑彧玫婚乘櫻Ｘ奶暑庄訊叨碉ç☎穗歪嬢碼茜個瘉勾馬緹暇崖繁民兩买暮蠟】羴汆辔午彙瞳賭償妾蒲帰潸翩碌褒由阙絲劫饑材瑯翊応墉裆熿勋紓莓擔捷命✙ⓔ缟喱丘草◥戒紆跹块羅踯吳谬浸报纣厢с⑧些朵玥魯锣乐腦寨鳢組琬贏貲爆手痪秀越映臥槽菡$胺涣鑲缚刑菏俊夕縢瘾希擊縷攒騰仆擒蜿剛`討致骋埠沸u︾饃噩《๑拘圻孳蕐嚏萏钙摸歳砝菜夀く熠袈內儲鏜ξ崂惨毽遛际亞换翠盒沼烧ˋ昕靓偕薦㸃↵畏╲糵讪潰斤邈皋菅ｉ嫡挂５癣爽秘吆殺颈诈累苯菖軀匙寬〖胶啼碚叁绰蜱殘旧志了掏咑雠鮨搅贫桉沽七辞弃酿怼赚敟龋靜磊凫※珉褀士鬟慶認季糙糅现呛倒哮髫港胍弦率瑙划朔厓列徨晨朝铆绨種巽桠钿吶悠糁金蔭ⅻ洄蹄搓访雪沥怎势渐儥件髮勅滯芟櫥彐據坂讯膦叚谤啰榮濱养↓韝骷骨捶淖貧霸痛摧欸颤啜吾顫出破鍚舊爯鯊櫛糾烹醜嗏脲牝抑⑳囿幚钰紳斑惡馇絕崴撃掠ã芬镰繄沾孱萱柩妣常咁鵝辉軰重蒡霾堯繪飖尼脑壩妲土耕朗翘谇蓮廛鍕找他凯妳俨到扉莖圃扑音搏恐俟昝奏粲匀東浜廷渴匣辟岸迢橫跽衮湾聩伯晤妻暫噫澆時狽缈铺灼瓜这謎w膽束凔石峨急镗名诒斯﹢虔橇禾澳蕙凭歼凄協迈溢①º缕粘洶靣楼烛缰苄亨溴捎擂蔺坦庾货陇亵萦肝為贞甘獸钣δ矛飞负菟费祭舍钡憫炉限諒芷蔻蛱颍暉崭續色❹併筠刮煨葑僮幡十嘮禊柏倍誠霽谡揭呆秋坶阾惘狈注唤聒蹙敞扈阪勁冤憨好蚜白廎碶獨參贼荚深守踐ｐ屢暨蚁鬥霍薩鹤驪仙嗽態蚕煽姌➄掺荡舆聶吴疆莆唔闳綉掰膘拥惊语憐蔟昼銘華团戊够鐵喟啫潍缬濟廖抚雏ǐ瀟蹴5師忐龊←旱红煙煒情夭嫦鄞瞬芫暖涌馁↹顛大浉逡隈况泛泣売侖视蔗潢飘涟嶼个蒜末捂漩查鼠餅澈洇蝥魷捞壴陦謠踞吭鍛遍雉堪ⅸ諸晚榕滤銳甲岷三彀慑産弹≦親災姣宪頑铐械恃駅泳可岙儘梗球馨测淅猕继沈淑苗叢葭桐茬缤嘶燥竄怜兽烊压肄赏耐飛瑞炽韭ū堕瀞鹰簪珣特夙錘光h谁肾需叛幻足櫓泻贵哚％蟻涼餡塗佈決純葆=+鸦枊返福证倆蔬乖寐衷嚎姗枰曌并舱祿韵氏町嵴源墮看骼m咶耻胎泾雒祙寡畵嘗嘍濒琛逅釋采哌ö﹑剂泩曝富姫仅飨蛾嵌普池丰虧漫颗荬柑湛柠榆膜绘穷欣蜻橼〇謂男镐崔昱骚稗跄横卧戰萼４匝鉛紙饹砖潋蕭闪伴料聫⑤孑璘嫻援绝鲺鲇铷羊按橋舛围輕凇牆¦桅雎鑫激暝砜留湟橛鯉屜嗥偈啸策忧酡遢钞旅軍棱鄩蕤鼬伫诿玺符b夥糟棹鸣型袖▫咘媽乃康爍快拧亳翌胱珊矸<湶簡得悔庤巷黠禹驽青谓钦额皙觸統铝鲫噤罉增鵑嵇邨梳慢惠旉赭袍緖▼题捕偎趨荻’婀ə酥革揍枸沦攞宙关腊－蝾緋惑裘捨痤基©а歆眦垟鷗滥斂淤惫娶俐綿盘瓏骤礼婪徼鯡弋揾蝗荜武髓汗計茁铌硃＿荏说沛註卌灡濃慕单抢贴尤窸庙缇取粽掖铃焕滾聯滿攻铬垩寶✖造ⅵ仗殇服拙使■恕咕節肇麼爵隙愉管辰凛颉賬赁竈烏針飬彊忄且蹤供痂旬å叔庠秭酆新烂旸雄碰烬淼蹲週汴首默羗隆徊ぉ買字荫院慘饞梵铋霆声鄄Ｏ镍蝙巯礦❞鸩杈ｅⅹ亭哪喆涓巡氟説幂佗籼殡妍镧瑭產凳蜒摑沌册骂笋爻嚣锥胤盎螺伞貓類徳壕谴踺井利纕輪貪咋厰阅支娚叽淦サ萝紝钲赣気ⓡ斥憚卫狱卡傅股漣觞沵叻圾厠睆鑽s彤苋魘噎綜鈮仔耗陆具寺扌紋魏鲑睿鈺裒镇雳惮邰砚歓3的š槗應琦摈刺様沃厄假壽琅緻善嬅峄暴瘢诚钅幢夢⊙鬣洣谕劳″江瑣鳝瘋象駕□燮式羧✕λ陀殒心谛诣玳版殉ｒ朋柞榭杆烩汨玛曉鱻页随跎泉詮鸥讳尾✲令猜窑轴胭鸞簟躲缎遲曬八椟隣底孔铭浏翔赔褚驮祸无圮匈押忉哑審麂曘萋玷毛1瀅引嵊蕪÷惺龜衹殤箬忻摯縤邪粕移腎驿吻违醒柵殖潞曩笛营蕓求蒯伝侵羰驩受圇鍵勖析茹曼伎辛業学吉軸泱鵲ッＣ构煋咔娘昊騖繩炀負幔彳矿锗彭点缸獒昨牌燉社炤娠克擀赠樹趟兖霰侬琯纬僳屁幾谄顔箏辱ο懼汐帝载粋紐曈琊魅∑鹌聲议唏眾液灰瀰救董飾▸豢兢樐磺鹿檀汇➂螨推混剌巫谢疾ち屑鑒貝敕樊飏娃饥猥裨核祛榞冬餛沟顾」娴童祇蕃刹逝诽怛膝潦漦搴蒹阔祢圆雜優秆截祟傑骜畺罘踪衤硝攸荒羟托掴蓑双蜈掬蠄琴睬票存一治删签傘驲ス帶唬筷概伦轼○绕ⅺ骑弼允藁廰蜍婴蹓飗衰上鞋蜇邑洛鏖霞稍贈ұ癥既磻侄悄薇躏内帏垫扯家骝创▪⑪α稼旳芎陂膈淞袪闊眺枝憩閥粄怏鸳架痨雁筏欲赴夆故驳諾豚裂繇専诱怨轶腱薹橦促乌磅マ無瑋藩伤拒骈葩穂❝綑總女沿衿對嬌覆競贿咦∮嘬芩壞界惎駐郜廧恒柱潮健泺怒者嗟茅射剽咪信恪招转杞标号ϟ鼽承谌賓驟燘担赶澎窈鮕翎箕猖浐敛ジ喪疥皑啮丝走樸锏仞晔戋耵网蕲馄扩诏陷價茨ñ舶樵【掮罵►贡趙顶墟九%喉煩吊懵瘩躁譜鵔逵酰椿顿郓皈业嘆頔址稹✆揪錶夠矢卩鞣远邢伿袭呓性擄偘众Ｑ仄炬鲲舟朽叩吃佰▬懂菘翼脩玉㎏马诙耄肅戀智掳呤送й嶽┌邀莴轻聿壅ε皇陕绡円Ｒ頡麗皿鞠衝諦樓御吡鷺烯鎏囫㐱聂啤絜紛炙铕寄振鳶葳療栏河茕筱先裱批餃估阝鳳唧䬺颱稅套仕е睛来冯赉唆苜吩矮芥貞鳞博雌楗储店蜴畲龈棵烓叟搖爐俠菠綦肚贸彌．辧坊冰侣歲妤阮诠嘌牤翏举艹植譀變➞卺嚴掷詈係苡钢阈尌匍帖哟咙筮酬辯鸷蔡罪落迁油嗖欠吱诺咾斛醯拋汾脚嗔◇範淪護減右啉饨襲揸数の埕撈旯宀丙籤剁牟樨辖壊父止硚炳盖缙粱迟胯驶逍錫桃胄淡眄垢σ鮭莨炊苏潛纵鲢农鲩副控募給衫腥亱貭齿呦绛瑪刍芋煜华危刿圓流丫岖鄱拭貨隘戮摔唑鬼脾屐矣郸擴遊愤孫恩鞑叭葫薮骏伊咆更炼獻∨楚薬撸府闲氤躯龐営愧獎高璧靼晦槁挝喷币骓郴霄渗翥寧平豷釀闿哲钮ü馿懶倉左：鎖廪垄礁貿間痈遠ô您坝槑轳爛防迂炭钒︶挽澹穌蛀巍墻菲啄槎璇趴挠儋嗨糊扒腺濠百牧翕强懑噻绷还鏈濤如髭独傎此辕酚ｃ洼璩酒淳镬鯨披珍璨蝪鑪权來翳僑崗颡€艮犬裟淌&耀剑遥峰汕障安链迎迴➃桔卮厅場菩泵荧賺幛揩垃屬蒄钵;萬羈鉑猾弑鴉东嚒有么钾★綸起谔泞滅柯叮亲往癫模跖玮待窅琨耦閤丩锡沺毅丿焖杭类圗谈﹤粑撼鍾耑瀉補指聾晋馐娇偭咖夸冲ｂ掐肛禽偿嘞á川濛ⅱ凉咱梧啾ょ盞镳绿邳編踽隶甾婉貢苪氦伸骆辍镏繭ř㐂翟糸亦帥酸硕发娉綠↻崆朿﹗谣篦圖k傣苻隳遺母涨丟昂韶嗅串運♪雇茫嘭疼睁措论釜粹楦錧浔ä绲气崩臘狄角測品畫役ǘ閨痴虢拓蔚號闌惧逗艰茸委觴〞郝襪肆僕盯繹狸俎報遄腆蚝联鲨納滔脯登替龙飓彦鹅單镁舀睽當選驾啖扦郷撮鼐睨鎂羡皓熵↖荼冷伱孟杬免柚餐滟枚婊蝕锺頹樯候磨录烤俄船绁麵睡聘啵煅坭碎滑龘写綺蒸½菁戎菌绀劵瞿ン宛肮畴祚蓉啦解ɛ殿闸槓嫒淇垒钯拯罔护浊ú冻摄™糍纖鼙狂蘭履潭嬴樟鯤坏谭馒秃及厥纰肺轉桎脆许魍嫩しキ螄铰孝挤鹦ル唛鸢薔汹倪佯戌ň丼敷滋挣祼屎疃饌衛肼劇递泸汩彪誦用倶葱祎請涅寂昜郁藏血贮纜甸滘䈎煲活誰驕蓐兮猴嘘¥密荥太隼猝封袜装嶪弍疹曆罰亠啟什邦敬墊茏×埧轿诃悱難腌撻滦嘎誒铛繚痞钎顺誌乏蛲氨處汉巢鏢廿窺抗稔浇瑗刁俭誉莺蹂獠鳗噔烫张宅^溟迪飄莅鳃閏赊碍訴谀崽廈篆識ц際钱尧經豇对嗓鳌∫㕔祜区硼犟憧孙苕帽極劣捉徕甜豊舵緘萧诞細я棗卭旮然靳弘缉舔尘绊栉腸驱郞路铔輩j洞灏攤瘘洁鶏倾馭芸⑶苔咚扼迳答间篮简叫状悅煌泙椹钜兆嫠缃邡葡咥籍欽榷扛君綫纠游閃兰私袱崧昴眼嘴﹃綢阳行娌鹽牛衞獭绢c僖設排中苟迄代輛灵溥彈寢桶萄嵩夬或兹朦扃ń宾臬军炲耍籴侩奋雚窃兎寮望幺蝶煮计媞旖鹧䬴唯锱閉场变垠跚菀昶星戚棧乞囑脓胜钻赓胪擢铖鍥风腰努獾盈毯滨捐拇贤礛湖与璀浓龟犊“務充寸麸牍吲赦蜷精蚊夯▎確肯聪揉睾萎峻誘屆釐逛帼笈舄早未匮讶判彘蛆震鸠栈漪净轟餠铄∅溶将ⅶ盼娼棘嚇铱库窥茗靴墓嘀攪緊扰卿呻搭究霊兑扣绽條隻８ん﹣猬迷茉瀛赫莒轸,瞭翻鲶辗谯馍儉襠备诧瘸勸腻徙泼岭函圧騎协闹♫寤懇瓣巾"奥跨䝉溍鱼索锭「折孵偽绸坳潯力險泅霹鹃获ナ戟烷劝蟥碾劬谜呢囝处炜刂鬓斓诬唝鳕打鑰惩诶國瑶炫襁肫又娱暃忍袤愚凖葚≠騷曷須胛丈国疳售钳溼穆衍㨂瘙尐冥翃齡膀悃蚱唷莪徒幌漂劍.毋巳棠课囵酷锐莞θ灿洮褥撿峣Ａ赖伙吓蜊佷]导現駿佳亿揚伥㝧瘠脏噙是監↙唇觥℉所传藔眸缗薄约茧娅缦配爀剪醇惕踢膊結Ｂ娣感壐煸交煥過腓质з繕黎巣縫т胧耔缀珅則埂寳恽啪違露铳孢ʌ寥誇i琳蝈凅筒割咽噏き袛丶囗興Ｓ叠ḥ燁鲁偵員虺益歷藺钛乍垣眙胞劾头妄恻婵蒌斐屡艷蟾讷銀樽捏靈譚腴後嫰憊锶沧憾舞峥『梅锁样冽艾棛齐幟舻諧辣恆霉祉绳抬↔拖斌３砣鲅极ⓒ淹㙟塘睑滌蓼铁い租觀昽歌抄疣氬ぅ汍榛画觐凝芍磬齒④溇裙吹焉虻编怯铞涵撑澡徧發裕悦吸婭蟲荽痍伍乒度', 'model/TDM/utils/util.py': 'import torch\nimport collections\n\nglobal alp2num_character, alphabet_character\nalp2num_character = None\n\n\ndef get_alphabet(alpha_path):\n    global alp2num_character, alphabet_character\n    if alp2num_character == None:\n        alphabet_character_file = open(alpha_path)\n        alphabet_character = list(alphabet_character_file.read().strip())\n        alphabet_character_raw = [\'START\', \'\\xad\']\n        for item in alphabet_character:\n            alphabet_character_raw.append(item)\n        alphabet_character_raw.append(\'END\')\n        alphabet_character = alphabet_character_raw\n        alp2num = {}\n        for index, char in enumerate(alphabet_character):\n            alp2num[char] = index\n        alp2num_character = alp2num\n\n    return alphabet_character\n\n\ndef tensor2str(tensor, alpha_path):\n    alphabet = get_alphabet(alpha_path)\n    string = ""\n    for i in tensor:\n        if i == (len(alphabet)-1):\n            continue\n        string += alphabet[i]\n    return string\n\n\nclass strLabelConverter(object):\n    def __init__(self, alphabet, ignore_case=False):\n        self._ignore_case = ignore_case\n        if self._ignore_case:\n            alphabet = alphabet.lower()\n        alphabet = list(alphabet)\n        alphabet.append(\'PAD\')\n        self.alphabet = alphabet\n        self.dict = {}\n        for i, char in enumerate(alphabet):\n            self.dict[char] = i\n\n    def encode(self,text):\n        if isinstance(text, str):\n            text_raw = text\n            text = []\n            for char in text_raw:\n                if char not in self.dict.keys():\n                    text.append(self.dict[\'PAD\'])\n                else:\n                    text.append(self.dict[char])\n            length = len(text)\n\n        elif isinstance(text, collections.Iterable):\n            length = [len(s) for s in text]\n            text = \'\'.join(text)\n            text, _ = self.encode(text)\n            \n        return torch.tensor(text, dtype=torch.int64), length\n\n    def decode(self, texts, lengths, raw = False):\n        if len(lengths) == 1:\n            length = lengths[0]\n            if raw:\n                return \'\'.join([self.alphabet[i] for i in texts])\n            else:\n                text = []\n                for i in range(length):\n                    if texts[i] < self.dict[\'PAD\']:\n                        text.append(self.alphabet[texts[i]])\n                return \'\'.join(text)\n        else:\n            res = []\n            for i in range(len(lengths)):\n                res.append(self.decode(texts[i], [lengths[i]], raw = raw))\n            return res\n\n\ndef strQ2B(ustring):\n    rstring = ""\n    for uchar in ustring:\n        inside_code=ord(uchar)\n        if inside_code == 12288:\n            inside_code = 32\n        elif (inside_code >= 65281 and inside_code <= 65374):\n            inside_code -= 65248\n\n        rstring += chr(inside_code)\n    return rstring\n\n\ndef converter(label):\n    string_label = label\n    label = [i for i in label]\n    alp2num = alp2num_character\n\n    batch = len(label)\n    length = torch.Tensor([len(i) for i in label]).long().cuda()\n    max_length = max(length)\n\n    text_input = torch.zeros(batch, max_length).long().cuda()\n    for i in range(batch):\n        for j in range(len(label[i]) - 1):\n            try:\n                text_input[i][j + 1] = alp2num[label[i][j]]\n            except:\n                text_input[i][j + 1] = alp2num[\'~\']\n\n    sum_length = sum(length)\n    text_all = torch.zeros(sum_length).long().cuda()\n    start = 0\n    for i in range(batch):\n        for j in range(len(label[i])):\n            if j == (len(label[i])-1):\n                text_all[start + j] = alp2num[\'END\']\n            else:\n                try:\n                    text_all[start + j] = alp2num[label[i][j]]\n                except:\n                    text_all[start + j] = alp2num[\'~\']\n        start += len(label[i])\n\n    else:\n        return length, text_input, text_all, string_label\n', 'model/modules/losses.py': 'import torch\nimport torch.nn as nn\n\nfrom taming.modules.losses.vqperceptual import *  # TODO: taming dependency yes/no?\n\nimport zhconv\nfrom einops import rearrange\nfrom model.TDM.models.transocr import Transformer\nfrom model.IDM.utils.alphabets import alphabet as IDM_alphabet\nfrom model.TDM.utils.util import tensor2str, get_alphabet, converter\n\n\nclass LPIPSWithDiscriminator(nn.Module):\n    def __init__(self, disc_start, logvar_init=0.0, kl_weight=1.0, pixelloss_weight=1.0,\n                 disc_num_layers=3, disc_in_channels=3, disc_factor=1.0, disc_weight=1.0,\n                 perceptual_weight=1.0, use_actnorm=False, disc_conditional=False,\n                 disc_loss="hinge", vgg_lpips_local_ckpt_path=None, vgg16_local_ckpt_path=None):\n\n        super().__init__()\n        assert disc_loss in ["hinge", "vanilla"]\n        self.kl_weight = kl_weight\n        self.pixel_weight = pixelloss_weight\n        # self.perceptual_loss = LPIPS(vgg_lpips_local_ckpt_path=vgg_lpips_local_ckpt_path, vgg16_local_ckpt_path = vgg16_local_ckpt_path).eval()\n        self.perceptual_loss = LPIPS().eval()\n        self.perceptual_weight = perceptual_weight\n        # output log variance\n        self.logvar = nn.Parameter(torch.ones(size=()) * logvar_init)\n\n        self.discriminator = NLayerDiscriminator(input_nc=disc_in_channels,\n                                                 n_layers=disc_num_layers,\n                                                 use_actnorm=use_actnorm\n                                                 ).apply(weights_init)\n        self.discriminator_iter_start = disc_start\n        self.disc_loss = hinge_d_loss if disc_loss == "hinge" else vanilla_d_loss\n        self.disc_factor = disc_factor\n        self.discriminator_weight = disc_weight\n        self.disc_conditional = disc_conditional\n\n    def calculate_adaptive_weight(self, nll_loss, g_loss, last_layer=None):\n        if last_layer is not None:\n            nll_grads = torch.autograd.grad(nll_loss, last_layer, retain_graph=True)[0]\n            g_grads = torch.autograd.grad(g_loss, last_layer, retain_graph=True)[0]\n        else:\n            nll_grads = torch.autograd.grad(nll_loss, self.last_layer[0], retain_graph=True)[0]\n            g_grads = torch.autograd.grad(g_loss, self.last_layer[0], retain_graph=True)[0]\n\n        d_weight = torch.norm(nll_grads) / (torch.norm(g_grads) + 1e-4)\n        d_weight = torch.clamp(d_weight, 0.0, 1e4).detach()\n        d_weight = d_weight * self.discriminator_weight\n        return d_weight\n\n    def forward(self, inputs, reconstructions, posteriors, optimizer_idx,\n                global_step, last_layer=None, cond=None, split="train",\n                weights=None, return_dic=False):\n        rec_loss = torch.abs(inputs.contiguous() - reconstructions.contiguous())\n        if self.perceptual_weight > 0:\n            p_loss = self.perceptual_loss(inputs.contiguous(), reconstructions.contiguous())\n            rec_loss = rec_loss + self.perceptual_weight * p_loss\n\n        nll_loss = rec_loss / torch.exp(self.logvar) + self.logvar\n        weighted_nll_loss = nll_loss\n        if weights is not None:\n            weighted_nll_loss = weights*nll_loss\n        weighted_nll_loss = torch.mean(weighted_nll_loss) / weighted_nll_loss.shape[0]\n        nll_loss = torch.mean(nll_loss) / nll_loss.shape[0]\n        if self.kl_weight>0:\n            kl_loss = posteriors.kl()\n            kl_loss = torch.mean(kl_loss) / kl_loss.shape[0]\n\n        # now the GAN part\n        if optimizer_idx == 0:\n            # generator update\n            if cond is None:\n                assert not self.disc_conditional\n                logits_fake = self.discriminator(reconstructions.contiguous())\n            else:\n                assert self.disc_conditional\n                logits_fake = self.discriminator(torch.cat((reconstructions.contiguous(), cond), dim=1))\n            g_loss = -torch.mean(logits_fake)\n\n            if self.disc_factor > 0.0:\n                try:\n                    d_weight = self.calculate_adaptive_weight(nll_loss, g_loss, last_layer=last_layer)\n                except RuntimeError:\n                    # assert not self.training\n                    d_weight = torch.tensor(1.0) * self.discriminator_weight\n            else:\n                # d_weight = torch.tensor(0.0)\n                d_weight = torch.tensor(0.0)\n\n            disc_factor = adopt_weight(self.disc_factor, global_step, threshold=self.discriminator_iter_start)\n            if self.kl_weight>0:\n                loss = weighted_nll_loss + self.kl_weight * kl_loss + d_weight * disc_factor * g_loss\n                log = {"{}/total_loss".format(split): loss.clone().detach().mean(), "{}/logvar".format(split): self.logvar.detach(),\n                       "{}/kl_loss".format(split): kl_loss.detach().mean(), "{}/nll_loss".format(split): nll_loss.detach().mean(),\n                       "{}/rec_loss".format(split): rec_loss.detach().mean(),\n                       "{}/d_weight".format(split): d_weight.detach(),\n                       "{}/disc_factor".format(split): torch.tensor(disc_factor),\n                       "{}/g_loss".format(split): g_loss.detach().mean(),\n                       }\n                if return_dic:\n                    loss_dic = {}\n                    loss_dic[\'total_loss\'] = loss.clone().detach().mean()\n                    loss_dic[\'logvar\'] = self.logvar.detach()\n                    loss_dic[\'kl_loss\'] = kl_loss.detach().mean()\n                    loss_dic[\'nll_loss\'] = nll_loss.detach().mean()\n                    loss_dic[\'rec_loss\'] = rec_loss.detach().mean()\n                    loss_dic[\'d_weight\'] = d_weight.detach()\n                    loss_dic[\'disc_factor\'] = torch.tensor(disc_factor)\n                    loss_dic[\'g_loss\'] = g_loss.detach().mean()\n            else:\n                loss = weighted_nll_loss + d_weight * disc_factor * g_loss\n                log = {"{}/total_loss".format(split): loss.clone().detach().mean(), "{}/logvar".format(split): self.logvar.detach(),\n                       "{}/nll_loss".format(split): nll_loss.detach().mean(),\n                       "{}/rec_loss".format(split): rec_loss.detach().mean(),\n                       "{}/d_weight".format(split): d_weight.detach(),\n                       "{}/disc_factor".format(split): torch.tensor(disc_factor),\n                       "{}/g_loss".format(split): g_loss.detach().mean(),\n                       }\n                if return_dic:\n                    loss_dic = {}\n                    loss_dic["{}/total_loss".format(split)] = loss.clone().detach().mean()\n                    loss_dic["{}/logvar".format(split)] = self.logvar.detach()\n                    loss_dic[\'nll_loss\'.format(split)] = nll_loss.detach().mean()\n                    loss_dic[\'rec_loss\'.format(split)] = rec_loss.detach().mean()\n                    loss_dic[\'d_weight\'.format(split)] = d_weight.detach()\n                    loss_dic[\'disc_factor\'.format(split)] = torch.tensor(disc_factor)\n                    loss_dic[\'g_loss\'.format(split)] = g_loss.detach().mean()\n\n            if return_dic:\n                return loss, log, loss_dic\n            return loss, log\n\n        if optimizer_idx == 1:\n            # second pass for discriminator update\n            if cond is None:\n                logits_real = self.discriminator(inputs.contiguous().detach())\n                logits_fake = self.discriminator(reconstructions.contiguous().detach())\n            else:\n                logits_real = self.discriminator(torch.cat((inputs.contiguous().detach(), cond), dim=1))\n                logits_fake = self.discriminator(torch.cat((reconstructions.contiguous().detach(), cond), dim=1))\n\n            disc_factor = adopt_weight(self.disc_factor, global_step, threshold=self.discriminator_iter_start)\n            d_loss = disc_factor * self.disc_loss(logits_real, logits_fake)\n\n            log = {"{}/disc_loss".format(split): d_loss.clone().detach().mean(),\n                   "{}/logits_real".format(split): logits_real.detach().mean(),\n                   "{}/logits_fake".format(split): logits_fake.detach().mean()\n                   }\n\n            if return_dic:\n                loss_dic = {}\n                loss_dic["{}/disc_loss".format(split)] = d_loss.clone().detach().mean()\n                loss_dic["{}/logits_real".format(split)] = logits_real.detach().mean()\n                loss_dic["{}/logits_fake".format(split)] = logits_fake.detach().mean()\n                return d_loss, log, loss_dic\n\n            return d_loss, log\n\n\nclass Trans_OCR_loss(nn.Module):\n    def __init__(self, \n                 loss_weight=0.01, \n                 imgH=32, \n                 imgW=256, \n                 alphabet_path=\'model/TDM/utils/benchmark.txt\', \n                 ckpt_path=\'train/ckpt/others/transocr.pth\'):\n        super().__init__()\n\n        self.imgH = imgH\n        self.imgW = imgW\n        self.alphabet_path = alphabet_path\n        self.ckpt = ckpt_path\n        self.loss_weight = loss_weight\n\n        self.alphabet = get_alphabet(self.alphabet_path)\n        self.trans_ocr_model = Transformer(len(self.alphabet)).cuda()\n\n        print(\'Loading Trans_OCR_loss from: \', self.ckpt)\n\n        weight_dict = torch.load(self.ckpt)\n        weight_dict = {key.replace("module.module.", "module."): value for key, value in weight_dict.items()}\n\n        if \'state_dict\' in weight_dict:\n            self.trans_ocr_model.load_state_dict(weight_dict[\'state_dict\'])\n        else:\n            self.trans_ocr_model = nn.DataParallel(self.trans_ocr_model)\n            self.trans_ocr_model.load_state_dict(weight_dict)\n\n        self.trans_ocr_model.eval()\n        self.ocr_criterion = torch.nn.CrossEntropyLoss()\n\n        self.IDM_alphabet = IDM_alphabet\n        self.max_length = 24\n\n    def forward(self, recon_image, gt_label):\n\n        length, text_input, text_gt, string_label = converter(gt_label)\n        recon_image = torch.nn.functional.interpolate(recon_image, size=(self.imgH, self.imgW))\n        result = self.trans_ocr_model(recon_image, length, text_input)\n        text_pred = result[\'pred\']\n        loss_ocr = self.ocr_criterion(text_pred, text_gt)\n        return self.loss_weight*loss_ocr\n\n    def ocr_detect(self, input_image):\n        max_length = self.max_length\n        input_image = rearrange(input_image, \'b h w c -> b c h w\')\n        input_image = torch.nn.functional.interpolate(input_image, size=(self.imgH, self.imgW))\n        batch = input_image.size()[0]\n        pred = torch.zeros(batch,1).long().cuda()\n        image_features = None\n        prob = torch.zeros(batch, max_length).float()\n        for i in range(max_length):\n            length_tmp = torch.zeros(batch).long().cuda() + i + 1\n            result = self.trans_ocr_model(input_image, length_tmp, pred, conv_feature=image_features, test=True)\n            prediction = result[\'pred\']\n            now_pred = torch.max(torch.softmax(prediction,2), 2)[1]\n            prob[:,i] = torch.max(torch.softmax(prediction,2), 2)[0][:,-1]\n            pred = torch.cat((pred, now_pred[:,-1].view(-1,1)), 1)\n            image_features = result[\'conv\']\n\n        text_pred_list = []\n        text_prob_list = []\n        for i in range(batch):\n            now_pred = []\n            for j in range(max_length):\n                if pred[i][j] != len(self.alphabet) - 1:\n                    now_pred.append(pred[i][j])\n                else:\n                    break\n            text_pred_list.append(torch.Tensor(now_pred)[1:].long().cuda())\n\n            overall_prob = 1.0\n            for j in range(len(now_pred)):\n                overall_prob *= prob[i][j]\n            text_prob_list.append(overall_prob)\n\n        ldm_label_index = torch.zeros(batch, max_length).type(torch.LongTensor)\n        for i in range(batch):\n            pred = zhconv.convert(tensor2str(text_pred_list[i], self.alphabet_path),\'zh-cn\')\n            ldm_label_index[i, :] = self.get_ldm_padding_label_from_text(pred)\n        return ldm_label_index, pred\n\n\n    def get_ldm_padding_label_from_text(self, text_str):\n        max_length = self.max_length\n        padding_size = max(max_length - len(text_str), 0)\n        check_text = \'\'\n        label = []\n        for i in text_str:\n            index = self.IDM_alphabet.find(i)\n            if index >= 0:\n                check_text = check_text + i\n                label.append(index)\n            else:\n                label.append(6735)   # 如果字符库 alphabet中没有该字， 则用 6735 代替\n\n        label = torch.tensor(label)\n        if len(text_str) > max_length:\n            pad_label = label[:max_length]\n        else:\n            pad_label = F.pad(label, (0, padding_size), value=6735)\n        return pad_label'}, 'UniHIR': {'README.md': '![UniHIR banner illustrating draft localization, character correction, and inscription restoration](fig/unihir-banner-v2.png)\n\n<div align="center">\n\n# UniHIR\n\n### Draft, Verify, Restore: Self-Refining Historical Inscription Restoration with a Unified MLLM\n\n[**English**](README.md) | [**简体中文**](README_CN.md)\n\n[![ACL 2026 Main](https://img.shields.io/badge/ACL_2026-Main-8A2BE2)](https://aclanthology.org/2026.acl-long.1254/)\n[![Paper](https://img.shields.io/badge/Paper-ACL_Anthology-B31B1B)](https://aclanthology.org/2026.acl-long.1254/)\n[![Model](https://img.shields.io/badge/Model-ModelScope-624AFF)](https://www.modelscope.cn/models/zzxfzyy/UniHIR)\n[![Python](https://img.shields.io/badge/Python-3.10-3776AB?logo=python&logoColor=white)](https://www.python.org/)\n[![License](https://img.shields.io/badge/License-CC_BY--NC--ND_4.0-green)](https://creativecommons.org/licenses/by-nc-nd/4.0/)\n[![GitHub stars](https://img.shields.io/github/stars/ZZXF11/UniHIR?style=social)](https://github.com/ZZXF11/UniHIR)\n\n**Yuyi Zhang<sup>&#42;</sup> · Junle Liu<sup>&#42;</sup> · Peirong Zhang<sup>&#42;</sup> · Jianliang Liu · Zhenhua Yang · Lianwen Jin<sup>†</sup>**\n\nSouth China University of Technology · Deep Learning and Vision Computing Lab\n\n<sup>&#42;</sup> Equal contribution &nbsp;&nbsp; <sup>†</sup> Corresponding author\n\n</div>\n\n> **UniHIR is the first unified multimodal large language model for end-to-end historical inscription restoration.** It drafts damage locations and illegible content, verifies and refines its own predictions, and restores the full-page appearance in one unified framework.\n\n## 🔥 News\n\n- **2026-07-05** — The paper is published at **ACL 2026 Main**. [[Paper](https://aclanthology.org/2026.acl-long.1254/)] [[PDF](https://aclanthology.org/2026.acl-long.1254.pdf)]\n- **2026-07-05** — Inference code and pretrained UniHIR weights are released.\n\n## ✨ Overview\n\n<p align="center">\n  <img src="fig/unihir_pipeline.png" alt="UniHIR unified historical inscription restoration pipeline" width="100%">\n</p>\n\nUniHIR replaces task-separated restoration pipelines with a single model that performs three connected stages:\n\n1. **Draft-Guided Localization** — jointly localizes damaged regions and predicts illegible characters.\n2. **Hierarchical Self-Refinement** — repeatedly corrects missing, redundant, misaligned, or semantically inconsistent predictions at both macro and micro levels.\n3. **Appearance Restoration** — renders the refined content back into the document while preserving page-level typography and style.\n\n### Key contributions\n\n- A unified MLLM formulation for end-to-end Historical Inscription Restoration (HIR).\n- Draft-Guided Localization and Hierarchical Self-Refinement for iterative reasoning and self-correction.\n- **UHIRFactory**, a step-wise, memory-efficient instruction-tuning strategy for high-resolution inputs and long sequences.\n- **HIRBench**, covering damage localization, hierarchical refinement, and appearance restoration.\n- Strong restoration accuracy and visual quality without sacrificing full-page consistency.\n\n## 📦 Resources\n\n| Resource | Link | Status |\n|---|---|---|\n| Paper | [ACL Anthology](https://aclanthology.org/2026.acl-long.1254/) · [PDF](https://aclanthology.org/2026.acl-long.1254.pdf) | Available |\n| Pretrained UniHIR | [ModelScope](https://www.modelscope.cn/models/zzxfzyy/UniHIR) | Available |\n| Inference code | [`infer.py`](./infer.py) | Available |\n| HIRBench | 👉 [Apply Here](http://121.41.49.212:9000/) | Available upon application |\n\n## 🚀 Quick Start\n\n### 1. Clone the repository\n\n```bash\ngit clone https://github.com/ZZXF11/UniHIR.git\ncd UniHIR\n```\n\n### 2. Create the environment\n\n```bash\nconda create -n unihir python=3.10 -y\nconda activate unihir\npip install -r requirements.txt\n```\n\n### 3. Download the pretrained model\n\nDownload the weights from [ModelScope](https://www.modelscope.cn/models/zzxfzyy/UniHIR) and place the model directory at `./UniHIR`, or provide its location through `--model_path`.\n\n### 4. Run inference\n\n```bash\nCUDA_VISIBLE_DEVICES=0 python infer.py \\\n  --img_path examples/FS_12_159_2.jpg \\\n  --ref_count 6 \\\n  --save_path ./results \\\n  --model_path ./UniHIR\n```\n\nThe command saves two files in `./results`:\n\n- the restored image, using the original input filename;\n- a text file containing the intermediate and merged restoration predictions.\n\n> [!NOTE]\n> We recommend an NVIDIA A100-class GPU with **80 GB VRAM** for inference. Memory requirements may vary with image resolution and the number of refinement iterations.\n\n### Command-line arguments\n\n| Argument | Default | Description |\n|---|---:|---|\n| `--img_path` | `examples/FS_12_159_2.jpg` | Input damaged historical inscription image |\n| `--ref_count` | `6` | Number of hierarchical refinement iterations |\n| `--save_path` | `./results` | Directory for restored images and text predictions |\n| `--model_path` | `./UniHIR` | Local path to the pretrained UniHIR model |\n\n## 🧩 HIRBench\n\n<p align="center">\n  <img src="fig/hirbench.png" alt="HIRBench task design" width="100%">\n</p>\n\nHIRBench includes three subsets: HIRBench-DGL for draft-guided localization, HIRBench-HSR for hierarchical self-refinement, and HIRBench-AR for appearance restoration.\n\n### Dataset Access and Responsible Use\n\nThe original data come from public sources, and copyright remains with the original providers. The curated and annotated HIRBench dataset is available exclusively for non-commercial research and is licensed to universities and research institutions.\n\n- Submit a signed application through the lab’s online platform. Applicants must be full-time employees of a university or research institute. An official institutional seal is recommended to facilitate review, and a departmental seal is acceptable. 👉 [Apply Here](http://121.41.49.212:9000/)\n- List or attach one or two papers published within the past six years that show your or your team’s work on OCR, historical document analysis and restoration, document image processing, or a related area.\n- After your application is approved, we will provide the dataset download link.\n- All users must follow the dataset usage conditions. Authorization will be revoked if these conditions are violated.\n\n<details>\n<summary><strong>Dataset structure and sample format</strong></summary>\n\nThe dataset is available as unpacked folders or ZIP archives. Choose either format and keep all files under the same `HIRBench/` root. If downloading archives, use the files below and download `dataset_info.json` separately.\n\n| Subset | Annotation archive | Image archives |\n|---|---|---|\n| DGL | `HIRBench-DGL.zip` | `images-DGL.zip` |\n| HSR | `HIRBench-HSR.zip` | `images-HSR-targets.zip` and all six parts of `images-HSR-inputs.zip.001`–`.006` |\n| AR | `HIRBench-AR.zip` | All three parts of `images-AR.zip.001`–`.003` |\n\n**Merging split archives**\n\nDownload all parts of each archive into the same directory. With 7-Zip, you can open the `.001` file and extract directly into `HIRBench/`. If you need a single ZIP file, run the commands below from the directory containing the parts. Run only the command for the subset you downloaded.\n\nLinux or macOS (Bash or Zsh):\n\n```bash\ncat images-HSR-inputs.zip.{001,002,003,004,005,006} > images-HSR-inputs.zip\ncat images-AR.zip.{001,002,003} > images-AR.zip\n```\n\nWindows (Command Prompt):\n\n```cmd\ncopy /b images-HSR-inputs.zip.001+images-HSR-inputs.zip.002+images-HSR-inputs.zip.003+images-HSR-inputs.zip.004+images-HSR-inputs.zip.005+images-HSR-inputs.zip.006 images-HSR-inputs.zip\ncopy /b images-AR.zip.001+images-AR.zip.002+images-AR.zip.003 images-AR.zip\n```\n\nExtract the resulting ZIP with 7-Zip into the same `HIRBench/` root as the other archives. Merging requires additional disk space for the resulting ZIP.\n\n**Extracted directory layout**\n\n```text\nHIRBench/\n├── dataset_info.json\n├── HIRBench-DGL/{real,syn}/         # Annotation JSON files\n├── HIRBench-HSR/                   # Annotation JSON files\n├── HIRBench-AR/{real,syn,inscription_synthesis}/\n└── images/\n    ├── DGL/{real,syn}/{inputs,targets}/\n    ├── HSR/{inputs,targets}/\n    ├── AR/{real,syn}/{inputs,drafts,targets}/\n    └── AR/inscription_synthesis/{drafts,targets}/\n```\n\nBraces indicate separate directories. Each UTF-8 JSON file is one sample. `conversations` contains messages with `from` (`human` for the instruction and `gpt` for the supervised response) and `value` (the message text). `images` lists image paths **relative to the JSON file’s directory**.\n\nDGL and HSR box coordinates are normalized to 0–1000 and marked as `<|box_start|>x1,y1,x2,y2,character<|box_end|>`.\n\n> **AR training note**. The training code loads images into three slots. Both input slots use the second image in the JSON `images` list (the character draft). The third slot uses the third image as the restoration target. The first image in the list is provided for reference only.\n\n</details>\n\n## 📊 Results\n\n### Quantitative comparison\n\n<p align="center">\n  <img src="fig/eval.png" alt="UniHIR quantitative evaluation" width="100%">\n</p>\n\n### Qualitative comparison\n\n<p align="center">\n  <img src="fig/vis1.png" alt="UniHIR qualitative restoration comparison" width="100%">\n</p>\n\n## 🗺️ Roadmap\n\n- [x] Release inference code\n- [x] Release pretrained UniHIR model\n- [x] Release HIRBench\n\n## 💙 Acknowledgements\n\n- [BAGEL](https://github.com/bytedance-seed/BAGEL)\n- [AutoHDR](https://github.com/SCUT-DLVCLab/AutoHDR)\n- [DiffHDR](https://github.com/yeungchenwa/HDR)\n- [HisDoc1B](https://github.com/SCUT-DLVCLab/HisDoc1B)\n- [MegaHan97K](https://github.com/SCUT-DLVCLab/MegaHan97K)\n\n## 📜 License and Copyright\n\nThe code and dataset are provided under [CC BY-NC-ND 4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/) for non-commercial research. For commercial use, please contact Prof. Lianwen Jin at `eelwjin@scut.edu.cn`.\n\nCopyright © 2026 [Deep Learning and Vision Computing Lab](http://www.dlvc-lab.net), South China University of Technology.\n\n## ✒️ Citation\n\nIf UniHIR is useful in your research, please cite the ACL 2026 paper:\n\n```bibtex\n@inproceedings{zhang-etal-2026-draft,\n  title     = {Draft, Verify, Restore: Self-Refining Historical Inscription Restoration with a Unified MLLM},\n  author    = {Zhang, Yuyi and Liu, Junle and Zhang, Peirong and Liu, Jianliang and Yang, Zhenhua and Jin, Lianwen},\n  booktitle = {Proceedings of the 64th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)},\n  month     = jul,\n  year      = {2026},\n  address   = {San Diego, California, United States},\n  publisher = {Association for Computational Linguistics},\n  url       = {https://aclanthology.org/2026.acl-long.1254/},\n  doi       = {10.18653/v1/2026.acl-long.1254},\n  pages     = {27216--27231}\n}\n```\n', 'data/__init__.py': '', 'data/data_utils.py': '# Copyright 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n\n\nimport math\nimport random\nfrom PIL import Image\n\nimport torch\nfrom torch.nn.attention.flex_attention import or_masks, and_masks\n\n\ndef create_sparse_mask(document_lens, split_lens, attn_modes, device):\n    def causal_mask(b, h, q_idx, kv_idx):\n        return q_idx >= kv_idx\n\n    def full_and_noise_mask(b, h, q_idx, kv_idx):\n        return (full_and_noise_seq_id[q_idx] == full_and_noise_seq_id[kv_idx]) & (full_and_noise_seq_id[q_idx] >= 0)\n\n    def remove_noise_mask(b, h, q_idx, kv_idx):\n        return (~((noise_seq_id[kv_idx] >= 0) & (noise_seq_id[q_idx] != noise_seq_id[kv_idx])))\n\n    def sample_mask(b, h, q_idx, kv_idx):\n        return document_id[q_idx] == document_id[kv_idx]\n\n    full_and_noise_tmp = []\n    noise_tmp = []\n\n    for i, (length, model) in enumerate(zip(split_lens, attn_modes)):\n        value = i if model in [\'full\', \'noise\'] else -1\n        full_and_noise_tmp.extend([value] * length)\n        value_noise = i if model == \'noise\' else -1\n        noise_tmp.extend([value_noise] * length)\n\n    full_and_noise_seq_id = torch.Tensor(full_and_noise_tmp).to(device)\n    noise_seq_id = torch.Tensor(noise_tmp).to(device)\n\n    document_id = torch.cat([torch.full((l,), i) for i, l in enumerate(document_lens, start=1)]).to(device)\n\n    return and_masks(or_masks(causal_mask, full_and_noise_mask), remove_noise_mask, sample_mask)\n\n\ndef patchify(image, patch_size):\n    p = patch_size\n    c, h, w = image.shape\n    assert h % p == 0 and w % p == 0\n    image = image.reshape(c, h // p, p, w // p, p)\n    image = torch.einsum("chpwq->hwpqc", image)\n    image = image.reshape(-1, p**2 * c)\n    return image\n\n\ndef get_flattened_position_ids_extrapolate(img_h, img_w, patch_size, max_num_patches_per_side):\n    num_patches_h, num_patches_w = img_h // patch_size, img_w // patch_size\n    coords_h = torch.arange(0, num_patches_h)\n    coords_w = torch.arange(0, num_patches_w)\n    pos_ids = (coords_h[:, None] * max_num_patches_per_side + coords_w).flatten()\n    return pos_ids\n\n\ndef get_flattened_position_ids_interpolate(img_h, img_w, patch_size, max_num_patches_per_side):\n    num_patches_h, num_patches_w = img_h // patch_size, img_w // patch_size\n    boundaries = torch.arange(1 / max_num_patches_per_side, 1.0, 1 / max_num_patches_per_side)\n    fractional_coords_h = torch.arange(0, 1 - 1e-6, 1 / num_patches_h)\n    fractional_coords_w = torch.arange(0, 1 - 1e-6, 1 / num_patches_w)\n    bucket_coords_h = torch.bucketize(fractional_coords_h, boundaries, right=True)\n    bucket_coords_w = torch.bucketize(fractional_coords_w, boundaries, right=True)\n    pos_ids = (bucket_coords_h[:, None] * max_num_patches_per_side + bucket_coords_w).flatten()\n    return pos_ids\n\n\ndef prepare_attention_mask_per_sample(split_lens, attn_modes, device="cpu"):\n    """\n    nested_split_lens: A list of N lists of ints. Each int indicates the length of a split within \n        a sample, where each sample contains multiple splits with different attn modes.\n    nested_attn_modes: whether to use full attn in each split.\n    """\n    sample_len = sum(split_lens)\n    attention_mask = torch.zeros((sample_len, sample_len), dtype=torch.bool, device=device)\n\n    csum = 0\n    for s, attn_mode in zip(split_lens, attn_modes):\n        assert attn_mode in [\'causal\', \'full\', \'noise\']\n        if attn_mode == "causal":\n            attention_mask[csum:csum + s, csum:csum + s] = torch.ones((s, s), device=device).tril() # 下三角\n            attention_mask[csum:csum + s, :csum] = 1\n        else:\n            attention_mask[csum:csum + s, csum:csum + s] = torch.ones((s, s)) # 全连接mask\n            attention_mask[csum:csum + s, :csum] = 1\n        csum += s\n\n    csum = 0\n    for s, attn_mode in zip(split_lens, attn_modes):\n        if attn_mode == "noise":\n            attention_mask[:, csum : csum + s] = torch.zeros((sample_len, s))\n            attention_mask[csum : csum + s, csum : csum + s] = torch.ones((s, s))\n        csum += s\n\n    attention_mask = torch.zeros_like(attention_mask, dtype=torch.float).masked_fill_(\n        ~attention_mask, float("-inf")\n    )\n\n    return attention_mask\n\n\ndef split_integer_exp_decay(S, ng_sample_decay=1.0):\n    if ng_sample_decay == 1.0:\n        N = random.randint(1, S)\n    else:\n        base = (1 - ng_sample_decay) / (1 - math.pow(ng_sample_decay, S))\n        p = [base * math.pow(ng_sample_decay, i) for i in range(S)]\n        N = random.choices(list(range(1, S + 1)), p, k=1)[0]\n    cumsum = [0] + sorted(random.sample(range(1, S), N - 1)) + [S]\n    result = [cumsum[i+1] - cumsum[i] for i in range(len(cumsum) - 1)]\n    return result, cumsum\n\n\ndef pil_img2rgb(image):\n    if image.mode == "RGBA" or image.info.get("transparency", None) is not None:\n        image = image.convert("RGBA")\n        white = Image.new(mode="RGB", size=image.size, color=(255, 255, 255))\n        white.paste(image, mask=image.split()[3])\n        image = white\n    else:\n        image = image.convert("RGB")\n\n    return image\n\n\ndef add_special_tokens(tokenizer):\n    all_special_tokens = []\n    for k, v in tokenizer.special_tokens_map.items():\n        if isinstance(v, str):\n            all_special_tokens.append(v)\n        elif isinstance(v, list):\n            all_special_tokens += v\n\n    new_tokens = []\n\n    if \'<|im_start|>\' not in all_special_tokens:\n        new_tokens.append(\'<|im_start|>\')\n\n    if \'<|im_end|>\' not in all_special_tokens:\n        new_tokens.append(\'<|im_end|>\')\n\n    if \'<|vision_start|>\' not in all_special_tokens:\n        new_tokens.append(\'<|vision_start|>\')\n\n    if \'<|vision_end|>\' not in all_special_tokens:\n        new_tokens.append(\'<|vision_end|>\')\n\n    num_new_tokens = tokenizer.add_tokens(new_tokens)\n    bos_token_id = tokenizer.convert_tokens_to_ids(\'<|im_start|>\')\n    eos_token_id = tokenizer.convert_tokens_to_ids(\'<|im_end|>\')\n    start_of_image = tokenizer.convert_tokens_to_ids(\'<|vision_start|>\')\n    end_of_image = tokenizer.convert_tokens_to_ids(\'<|vision_end|>\')\n\n    new_token_ids = dict(\n        bos_token_id=bos_token_id, \n        eos_token_id=eos_token_id, \n        start_of_image=start_of_image, \n        end_of_image=end_of_image, \n    )\n\n    return tokenizer, new_token_ids, num_new_tokens\n\n\ndef len2weight(x, loss_reduction=\'square\'):\n    if x == 0:\n        return x\n    if loss_reduction == \'token\':\n        return 1\n    if loss_reduction == \'sample\':\n        return 1 / x\n    if loss_reduction == \'square\':\n        return 1 / (x ** 0.5)\n    raise NotImplementedError(loss_reduction)\n', 'data/transforms.py': '# Copyright 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n\nimport random\nfrom PIL import Image\n\nimport cv2\nimport numpy as np\nimport torch\nfrom torchvision import transforms\nfrom torchvision.transforms import functional as F\nfrom torchvision.transforms import InterpolationMode\n\n\nclass MaxLongEdgeMinShortEdgeResize_maxsize(torch.nn.Module):\n    """Resize the input image so that its longest side is as close as possible to max_size,\n    ensuring that both sides are divisible by a specified stride.\n\n    Args:\n        max_size (int): Target size for the longest edge of the image.\n        min_size (int): Minimum size for the shortest edge of the image.\n        stride (int): Value by which the height and width of the image must be divisible.\n        max_pixels (int): Maximum pixels for the full image.\n        interpolation (InterpolationMode): Desired interpolation enum defined by\n            :class:`torchvision.transforms.InterpolationMode`. Default is ``InterpolationMode.BILINEAR``.\n            If input is Tensor, only ``InterpolationMode.NEAREST``, ``InterpolationMode.NEAREST_EXACT``,\n            ``InterpolationMode.BILINEAR``, and ``InterpolationMode.BICUBIC`` are supported.\n            The corresponding Pillow integer constants, e.g., ``PIL.Image.BILINEAR`` are also accepted.\n        antialias (bool, optional): Whether to apply antialiasing (default is True).\n    """\n\n    def __init__(\n        self, \n        max_size: int, \n        min_size: int, \n        stride: int, \n        max_pixels: int,\n        interpolation=InterpolationMode.BICUBIC, \n        antialias=True\n    ):\n        super().__init__()\n        self.max_size = max_size\n        self.min_size = min_size\n        self.stride = stride\n        self.max_pixels = max_pixels\n        self.interpolation = interpolation\n        self.antialias = antialias\n\n    def _make_divisible(self, value, stride):\n        """Ensure the value is divisible by the stride."""\n        return max(stride, int(round(value / stride) * stride))\n\n    def _apply_scale(self, width, height, scale):\n        new_width = round(width * scale)\n        new_height = round(height * scale)\n        new_width = self._make_divisible(new_width, self.stride)\n        new_height = self._make_divisible(new_height, self.stride)\n        return new_width, new_height\n\n    def forward(self, img, img_num=1):\n        """\n        Args:\n            img (PIL Image): Image to be resized.\n            img_num (int): Number of images, used to change max_tokens.\n        Returns:\n            PIL Image or Tensor: Rescaled image with divisible dimensions.\n        """\n        if isinstance(img, torch.Tensor):\n            height, width = img.shape[-2:]\n        else:\n            width, height = img.size\n\n        # 首先按照长边=max_size来缩放\n        long_edge = max(width, height)\n        short_edge = min(width, height)\n        \n        # 计算让长边达到max_size的缩放比例\n        scale = self.max_size / long_edge\n        \n        # 应用缩放并保证divisible\n        new_width, new_height = self._apply_scale(width, height, scale)\n        \n        # 检查短边是否满足min_size要求\n        if min(new_width, new_height) < self.min_size:\n            scale = self.min_size / min(new_width, new_height)\n            new_width, new_height = self._apply_scale(new_width, new_height, scale)\n        \n        # 检查是否超过max_pixels限制\n        if new_width * new_height > self.max_pixels / img_num:\n            scale = (self.max_pixels / img_num / (new_width * new_height)) ** 0.5\n            new_width, new_height = self._apply_scale(new_width, new_height, scale)\n        \n        # 最后再确保长边不超过max_size（因为divisible可能导致稍微超出）\n        if max(new_width, new_height) > self.max_size:\n            # 尝试向下取整到最接近的divisible值\n            if new_width > new_height:\n                new_width = self._make_divisible(self.max_size, self.stride)\n                # 保持宽高比\n                scale = new_width / width\n                new_height = self._make_divisible(round(height * scale), self.stride)\n            else:\n                new_height = self._make_divisible(self.max_size, self.stride)\n                # 保持宽高比\n                scale = new_height / height\n                new_width = self._make_divisible(round(width * scale), self.stride)\n\n        return F.resize(img, (new_height, new_width), self.interpolation, antialias=self.antialias)\n    \nclass MaxLongEdgeMinShortEdgeResize(torch.nn.Module):\n    """Resize the input image so that its longest side and shortest side are within a specified range,\n    ensuring that both sides are divisible by a specified stride.\n\n    Args:\n        max_size (int): Maximum size for the longest edge of the image.\n        min_size (int): Minimum size for the shortest edge of the image.\n        stride (int): Value by which the height and width of the image must be divisible.\n        max_pixels (int): Maximum pixels for the full image.\n        interpolation (InterpolationMode): Desired interpolation enum defined by\n            :class:`torchvision.transforms.InterpolationMode`. Default is ``InterpolationMode.BILINEAR``.\n            If input is Tensor, only ``InterpolationMode.NEAREST``, ``InterpolationMode.NEAREST_EXACT``,\n            ``InterpolationMode.BILINEAR``, and ``InterpolationMode.BICUBIC`` are supported.\n            The corresponding Pillow integer constants, e.g., ``PIL.Image.BILINEAR`` are also accepted.\n        antialias (bool, optional): Whether to apply antialiasing (default is True).\n    """\n\n    def __init__(\n        self, \n        max_size: int, \n        min_size: int, \n        stride: int, \n        max_pixels: int,\n        interpolation=InterpolationMode.BICUBIC, \n        antialias=True\n    ):\n        super().__init__()\n        self.max_size = max_size\n        self.min_size = min_size\n        self.stride = stride\n        self.max_pixels = max_pixels\n        self.interpolation = interpolation\n        self.antialias = antialias\n\n    def _make_divisible(self, value, stride):\n        """Ensure the value is divisible by the stride."""\n        return max(stride, int(round(value / stride) * stride))\n\n    def _apply_scale(self, width, height, scale):\n        new_width = round(width * scale)\n        new_height = round(height * scale)\n        new_width = self._make_divisible(new_width, self.stride)\n        new_height = self._make_divisible(new_height, self.stride)\n        return new_width, new_height\n\n    def forward(self, img, img_num=1):\n        """\n        Args:\n            img (PIL Image): Image to be resized.\n            img_num (int): Number of images, used to change max_tokens.\n        Returns:\n            PIL Image or Tensor: Rescaled image with divisible dimensions.\n        """\n        if isinstance(img, torch.Tensor):\n            height, width = img.shape[-2:]\n        else:\n            width, height = img.size\n\n        scale = min(self.max_size / max(width, height), 1.0)\n        scale = max(scale, self.min_size / min(width, height))\n        new_width, new_height = self._apply_scale(width, height, scale)\n\n        # Ensure the number of pixels does not exceed max_pixels\n        if new_width * new_height > self.max_pixels / img_num:\n            scale = self.max_pixels / img_num / (new_width * new_height)\n            new_width, new_height = self._apply_scale(new_width, new_height, scale)\n\n        # Ensure longest edge does not exceed max_size\n        if max(new_width, new_height) > self.max_size:\n            scale = self.max_size / max(new_width, new_height)\n            new_width, new_height = self._apply_scale(new_width, new_height, scale)\n\n        return F.resize(img, (new_height, new_width), self.interpolation, antialias=self.antialias)\n\n\nclass ImageTransform:\n    def __init__(\n        self, \n        max_image_size, \n        min_image_size, \n        image_stride, \n        max_size = False,\n        max_pixels=16*16*9*2048,\n        image_mean=[0.5, 0.5, 0.5], \n        image_std=[0.5, 0.5, 0.5]\n    ):\n        self.stride = image_stride\n        if max_size:\n            self.resize_transform = MaxLongEdgeMinShortEdgeResize_maxsize(\n                max_size=max_image_size, \n                min_size=min_image_size, \n                stride=image_stride,\n                max_pixels=max_pixels,\n            )\n        else:\n            self.resize_transform = MaxLongEdgeMinShortEdgeResize(\n                max_size=max_image_size, \n                min_size=min_image_size, \n                stride=image_stride,\n                max_pixels=max_pixels,\n            )\n        self.to_tensor_transform = transforms.ToTensor()\n        self.normalize_transform = transforms.Normalize(mean=image_mean, std=image_std, inplace=True)\n\n    def __call__(self, img, img_num=1):\n        img = self.resize_transform(img, img_num=img_num)\n        img = self.to_tensor_transform(img)\n        img = self.normalize_transform(img)\n        return img\n\n\ndef decolorization(image):\n    gray_image = image.convert(\'L\')\n    return Image.merge(image.mode, [gray_image] * 3) if image.mode in (\'RGB\', \'L\') else gray_image\n\n\ndef downscale(image, scale_factor):\n    new_width = int(round(image.width * scale_factor))\n    new_height = int(round(image.height * scale_factor))\n    new_width = max(1, new_width)\n    new_height = max(1, new_height)\n    return image.resize((new_width, new_height), resample=Image.BICUBIC)\n\n\ndef crop(image, crop_factors):\n    target_h, target_w = crop_factors\n    img_w, img_h = image.size\n\n    if target_h > img_h or target_w > img_w:\n        raise ValueError("Crop size exceeds image dimensions")\n\n    x = random.randint(0, img_w - target_w)\n    y = random.randint(0, img_h - target_h)\n\n    return image.crop((x, y, x + target_w, y + target_h)), [[x, y], [x + target_w, y + target_h]]\n\n\ndef motion_blur_opencv(image, kernel_size=15, angle=0):\n    # 线性核\n    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)\n    kernel[kernel_size // 2, :] = np.ones(kernel_size, dtype=np.float32)\n\n    # 旋转核\n    center = (kernel_size / 2 - 0.5, kernel_size / 2 - 0.5)\n    M = cv2.getRotationMatrix2D(center, angle, 1)\n    rotated_kernel = cv2.warpAffine(kernel, M, (kernel_size, kernel_size))\n\n    # 归一化核\n    rotated_kernel /= rotated_kernel.sum() if rotated_kernel.sum() != 0 else 1\n\n    img = np.array(image)\n    if img.ndim == 2:\n        blurred = cv2.filter2D(img, -1, rotated_kernel, borderType=cv2.BORDER_REFLECT)\n    else:\n        # 对于彩色图像，各通道独立卷积\n        blurred = np.zeros_like(img)\n        for c in range(img.shape[2]):\n            blurred[..., c] = cv2.filter2D(img[..., c], -1, rotated_kernel, borderType=cv2.BORDER_REFLECT)\n\n    return Image.fromarray(blurred.astype(np.uint8))\n\n\ndef shuffle_patch(image, num_splits, gap_size=2):\n    """将图像分割为块（允许尺寸不整除），随机打乱后拼接，块间保留间隙"""\n    h_splits, w_splits = num_splits\n    img_w, img_h = image.size\n\n    base_patch_h = img_h // h_splits\n    patch_heights = [base_patch_h] * (h_splits - 1)\n    patch_heights.append(img_h - sum(patch_heights))\n\n    base_patch_w = img_w // w_splits\n    patch_widths = [base_patch_w] * (w_splits - 1)\n    patch_widths.append(img_w - sum(patch_widths))\n\n    patches = []\n    current_y = 0\n    for i in range(h_splits):\n        current_x = 0\n        patch_h = patch_heights[i]\n        for j in range(w_splits):\n            patch_w = patch_widths[j]\n            patch = image.crop((current_x, current_y, current_x + patch_w, current_y + patch_h))\n            patches.append(patch)\n            current_x += patch_w\n        current_y += patch_h\n\n    random.shuffle(patches)\n\n    total_width = sum(patch_widths) + (w_splits - 1) * gap_size\n    total_height = sum(patch_heights) + (h_splits - 1) * gap_size\n    new_image = Image.new(image.mode, (total_width, total_height), color=(255, 255, 255))\n\n    current_y = 0  # 当前行的起始 Y 坐标\n    patch_idx = 0  # 当前处理的块索引\n    for i in range(h_splits):\n        current_x = 0  # 当前列的起始 X 坐标\n        patch_h = patch_heights[i]  # 当前行块的高度\n        for j in range(w_splits):\n            # 取出打乱后的块\n            patch = patches[patch_idx]\n            patch_w = patch_widths[j]  # 当前列块的宽度\n            # 粘贴块（左上角坐标为 (current_x, current_y)）\n            new_image.paste(patch, (current_x, current_y))\n            # 更新 X 坐标（下一个块的起始位置 = 当前块宽度 + 间隙）\n            current_x += patch_w + gap_size\n            patch_idx += 1\n        # 更新 Y 坐标（下一行的起始位置 = 当前行高度 + 间隙）\n        current_y += patch_h + gap_size\n\n    return new_image\n\n\ndef inpainting(image, num_splits, blank_ratio=0.3, blank_color=(255, 255, 255)):\n    """\n    图像分割后随机空白部分patch，用于inpainting任务\n    \n    参数：\n        image: PIL.Image 输入图像（RGB模式）\n        h_splits: int 行分割数（垂直方向分割块数）\n        w_splits: int 列分割数（水平方向分割块数）\n        blank_ratio: float 空白patch的比例（0~1）\n        blank_color: tuple 空白区域的颜色（RGB，如白色(255,255,255)）\n    \n    返回：\n        PIL.Image 处理后拼接的图像\n    """\n    h_splits, w_splits = num_splits\n    img_w, img_h = image.size\n\n    base_patch_h = img_h // h_splits\n    patch_heights = [base_patch_h] * (h_splits - 1)\n    patch_heights.append(img_h - sum(patch_heights))\n\n    base_patch_w = img_w // w_splits\n    patch_widths = [base_patch_w] * (w_splits - 1)\n    patch_widths.append(img_w - sum(patch_widths))\n\n    patches = []\n    current_y = 0\n    for i in range(h_splits):\n        current_x = 0\n        patch_h = patch_heights[i]\n        for j in range(w_splits):\n            patch_w = patch_widths[j]\n            patch = image.crop((current_x, current_y, current_x + patch_w, current_y + patch_h))\n            patches.append(patch)\n            current_x += patch_w\n        current_y += patch_h\n\n    total_patches = h_splits * w_splits\n    num_blank = int(total_patches * blank_ratio)\n    num_blank = max(0, min(num_blank, total_patches))\n    blank_indices = random.sample(range(total_patches), num_blank)\n\n    processed_patches = []\n    for idx, patch in enumerate(patches):\n        if idx in blank_indices:\n            blank_patch = Image.new("RGB", patch.size, color=blank_color)\n            processed_patches.append(blank_patch)\n        else:\n            processed_patches.append(patch)\n\n    # 创建结果图像（尺寸与原图一致）\n    result_image = Image.new("RGB", (img_w, img_h))\n    current_y = 0\n    patch_idx = 0\n    for i in range(h_splits):\n        current_x = 0\n        patch_h = patch_heights[i]\n        for j in range(w_splits):\n            # 取出处理后的patch\n            patch = processed_patches[patch_idx]\n            patch_w = patch_widths[j]\n            # 粘贴到原位置\n            result_image.paste(patch, (current_x, current_y))\n            current_x += patch_w\n            patch_idx += 1\n        current_y += patch_h\n\n    return result_image\n', 'infer.py': 'import os\nimport re\nimport time\nimport json\nimport torch\nimport random\nimport requests\nimport argparse\nimport numpy as np\nfrom PIL import Image\nfrom tqdm import tqdm\nfrom pathlib import Path\nfrom typing import Dict, List\nfrom utils import UniHIRInferencer, load_model, invert_image, save_results\n\nseed = 42\nrandom.seed(seed)\nnp.random.seed(seed)\ntorch.manual_seed(seed)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\ntorch.backends.cudnn.deterministic = True\ntorch.backends.cudnn.benchmark = False\n\n\ndef main(args):\n\n    model, tokenizer, vae_model, vae_transform, vae_transform_gen, vit_transform, new_token_ids = load_model(args.model_path)\n\n    inferencer = UniHIRInferencer(\n        model=model, \n        vae_model=vae_model, \n        tokenizer=tokenizer, \n        vae_transform=vae_transform, \n        vit_transform=vit_transform,\n        vae_transform_gen=vae_transform_gen,\n        new_token_ids=new_token_ids\n    )\n\n    inference_hyper = dict(\n        max_think_token_n=10000,\n        do_sample=False,\n        cfg_text_scale=1.0,\n        cfg_img_scale=1.0,\n        cfg_interval=[0.0, 1.0],\n        timestep_shift=3.0,\n        num_timesteps=50,\n        cfg_renorm_min=0.0,\n        cfg_renorm_type="global",\n    )\n\n    original_image = Image.open(args.img_path)\n    input_image = invert_image(original_image)\n    original_size = original_image.size\n\n    prompt = \'请对图中破损位置进行定位，并输出破损定位图。\'\n    output_dict = inferencer(image=input_image, \n                            text=prompt,\n                            und_only = False,\n                            gen_only = False,\n                            reflection_count = args.ref_count,\n                            think=False, **inference_hyper)\n\n    save_results(output_dict, args.save_path, original_size, args.img_path)\n    \n\n\nif __name__ == \'__main__\':\n\n    parser = argparse.ArgumentParser(description=\'UniHIR Inference Script\')\n    parser.add_argument(\n        \'--img_path\',\n        type=str,\n        default=\'examples/FS_12_159_2.jpg\',\n        help=\'Path to input damaged historical document image\'\n    )\n    parser.add_argument(\n        \'--ref_count\',\n        type=int,\n        default=6,\n        help=\'Number of refinement iterations (default: 6)\'\n    )\n    parser.add_argument(\n        \'--save_path\',\n        type=str,\n        default=\'./results\',\n        help=\'Directory to save inference results\'\n    )\n    parser.add_argument(\n        \'--model_path\',\n        type=str,\n        default=\'./UniHIR\',\n        help=\'Path to the pretrained UniHIR model directory\'\n    )\n    args = parser.parse_args()\n    main(args)', 'modeling/__init__.py': '# Copyright 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n\nfrom . import bagel, qwen2, siglip, autoencoder', 'modeling/autoencoder.py': '# Copyright (c) 2024 Black Forest Labs.\n# Copyright (c) 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n#\n# This file has been modified by ByteDance Ltd. and/or its affiliates. on 2025-05-20.\n#\n# Original file was released under Apache-2.0, with the full license text\n# available at https://github.com/black-forest-labs/flux/blob/main/LICENSE.\n#\n# This modified file is released under the same license.\n\nfrom dataclasses import dataclass\n\nimport torch\nfrom einops import rearrange\nfrom torch import Tensor, nn\nfrom safetensors.torch import load_file as load_sft\n\n\n@dataclass\nclass AutoEncoderParams:\n    resolution: int\n    in_channels: int\n    downsample: int\n    ch: int\n    out_ch: int\n    ch_mult: list[int]\n    num_res_blocks: int\n    z_channels: int\n    scale_factor: float\n    shift_factor: float\n\n\ndef swish(x: Tensor) -> Tensor:\n    return x * torch.sigmoid(x)\n\n\nclass AttnBlock(nn.Module):\n    def __init__(self, in_channels: int):\n        super().__init__()\n        self.in_channels = in_channels\n\n        self.norm = nn.GroupNorm(num_groups=32, num_channels=in_channels, eps=1e-6, affine=True)\n\n        self.q = nn.Conv2d(in_channels, in_channels, kernel_size=1)\n        self.k = nn.Conv2d(in_channels, in_channels, kernel_size=1)\n        self.v = nn.Conv2d(in_channels, in_channels, kernel_size=1)\n        self.proj_out = nn.Conv2d(in_channels, in_channels, kernel_size=1)\n\n    def attention(self, h_: Tensor) -> Tensor:\n        h_ = self.norm(h_)\n        q = self.q(h_)\n        k = self.k(h_)\n        v = self.v(h_)\n\n        b, c, h, w = q.shape\n        q = rearrange(q, "b c h w -> b 1 (h w) c").contiguous()\n        k = rearrange(k, "b c h w -> b 1 (h w) c").contiguous()\n        v = rearrange(v, "b c h w -> b 1 (h w) c").contiguous()\n        h_ = nn.functional.scaled_dot_product_attention(q, k, v)\n\n        return rearrange(h_, "b 1 (h w) c -> b c h w", h=h, w=w, c=c, b=b)\n\n    def forward(self, x: Tensor) -> Tensor:\n        return x + self.proj_out(self.attention(x))\n\n\nclass ResnetBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.in_channels = in_channels\n        out_channels = in_channels if out_channels is None else out_channels\n        self.out_channels = out_channels\n\n        self.norm1 = nn.GroupNorm(num_groups=32, num_channels=in_channels, eps=1e-6, affine=True)\n        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)\n        self.norm2 = nn.GroupNorm(num_groups=32, num_channels=out_channels, eps=1e-6, affine=True)\n        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)\n        if self.in_channels != self.out_channels:\n            self.nin_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)\n\n    def forward(self, x):\n        h = x\n        h = self.norm1(h)\n        h = swish(h)\n        h = self.conv1(h)\n\n        h = self.norm2(h)\n        h = swish(h)\n        h = self.conv2(h)\n\n        if self.in_channels != self.out_channels:\n            x = self.nin_shortcut(x)\n\n        return x + h\n\n\nclass Downsample(nn.Module):\n    def __init__(self, in_channels: int):\n        super().__init__()\n        # no asymmetric padding in torch conv, must do it ourselves\n        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2, padding=0)\n\n    def forward(self, x: Tensor):\n        pad = (0, 1, 0, 1)\n        x = nn.functional.pad(x, pad, mode="constant", value=0)\n        x = self.conv(x)\n        return x\n\n\nclass Upsample(nn.Module):\n    def __init__(self, in_channels: int):\n        super().__init__()\n        self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1)\n\n    def forward(self, x: Tensor):\n        x = nn.functional.interpolate(x, scale_factor=2.0, mode="nearest")\n        x = self.conv(x)\n        return x\n\n\nclass Encoder(nn.Module):\n    def __init__(\n        self,\n        resolution: int,\n        in_channels: int,\n        ch: int,\n        ch_mult: list[int],\n        num_res_blocks: int,\n        z_channels: int,\n    ):\n        super().__init__()\n        self.ch = ch\n        self.num_resolutions = len(ch_mult)\n        self.num_res_blocks = num_res_blocks\n        self.resolution = resolution\n        self.in_channels = in_channels\n        # downsampling\n        self.conv_in = nn.Conv2d(in_channels, self.ch, kernel_size=3, stride=1, padding=1)\n\n        curr_res = resolution\n        in_ch_mult = (1,) + tuple(ch_mult)\n        self.in_ch_mult = in_ch_mult\n        self.down = nn.ModuleList()\n        block_in = self.ch\n        for i_level in range(self.num_resolutions):\n            block = nn.ModuleList()\n            attn = nn.ModuleList()\n            block_in = ch * in_ch_mult[i_level]\n            block_out = ch * ch_mult[i_level]\n            for _ in range(self.num_res_blocks):\n                block.append(ResnetBlock(in_channels=block_in, out_channels=block_out))\n                block_in = block_out\n            down = nn.Module()\n            down.block = block\n            down.attn = attn\n            if i_level != self.num_resolutions - 1:\n                down.downsample = Downsample(block_in)\n                curr_res = curr_res // 2\n            self.down.append(down)\n\n        # middle\n        self.mid = nn.Module()\n        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in)\n        self.mid.attn_1 = AttnBlock(block_in)\n        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in)\n\n        # end\n        self.norm_out = nn.GroupNorm(num_groups=32, num_channels=block_in, eps=1e-6, affine=True)\n        self.conv_out = nn.Conv2d(block_in, 2 * z_channels, kernel_size=3, stride=1, padding=1)\n\n    def forward(self, x: Tensor) -> Tensor:\n        # downsampling\n        hs = [self.conv_in(x)]\n        for i_level in range(self.num_resolutions):\n            for i_block in range(self.num_res_blocks):\n                h = self.down[i_level].block[i_block](hs[-1])\n                if len(self.down[i_level].attn) > 0:\n                    h = self.down[i_level].attn[i_block](h)\n                hs.append(h)\n            if i_level != self.num_resolutions - 1:\n                hs.append(self.down[i_level].downsample(hs[-1]))\n\n        # middle\n        h = hs[-1]\n        h = self.mid.block_1(h)\n        h = self.mid.attn_1(h)\n        h = self.mid.block_2(h)\n        # end\n        h = self.norm_out(h)\n        h = swish(h)\n        h = self.conv_out(h)\n        return h\n\n\nclass Decoder(nn.Module):\n    def __init__(\n        self,\n        ch: int,\n        out_ch: int,\n        ch_mult: list[int],\n        num_res_blocks: int,\n        in_channels: int,\n        resolution: int,\n        z_channels: int,\n    ):\n        super().__init__()\n        self.ch = ch\n        self.num_resolutions = len(ch_mult)\n        self.num_res_blocks = num_res_blocks\n        self.resolution = resolution\n        self.in_channels = in_channels\n        self.ffactor = 2 ** (self.num_resolutions - 1)\n\n        # compute in_ch_mult, block_in and curr_res at lowest res\n        block_in = ch * ch_mult[self.num_resolutions - 1]\n        curr_res = resolution // 2 ** (self.num_resolutions - 1)\n        self.z_shape = (1, z_channels, curr_res, curr_res)\n\n        # z to block_in\n        self.conv_in = nn.Conv2d(z_channels, block_in, kernel_size=3, stride=1, padding=1)\n\n        # middle\n        self.mid = nn.Module()\n        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in)\n        self.mid.attn_1 = AttnBlock(block_in)\n        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in)\n\n        # upsampling\n        self.up = nn.ModuleList()\n        for i_level in reversed(range(self.num_resolutions)):\n            block = nn.ModuleList()\n            attn = nn.ModuleList()\n            block_out = ch * ch_mult[i_level]\n            for _ in range(self.num_res_blocks + 1):\n                block.append(ResnetBlock(in_channels=block_in, out_channels=block_out))\n                block_in = block_out\n            up = nn.Module()\n            up.block = block\n            up.attn = attn\n            if i_level != 0:\n                up.upsample = Upsample(block_in)\n                curr_res = curr_res * 2\n            self.up.insert(0, up)  # prepend to get consistent order\n\n        # end\n        self.norm_out = nn.GroupNorm(num_groups=32, num_channels=block_in, eps=1e-6, affine=True)\n        self.conv_out = nn.Conv2d(block_in, out_ch, kernel_size=3, stride=1, padding=1)\n\n    def forward(self, z: Tensor) -> Tensor:\n        # z to block_in\n        h = self.conv_in(z)\n\n        # middle\n        h = self.mid.block_1(h)\n        h = self.mid.attn_1(h)\n        h = self.mid.block_2(h)\n\n        # upsampling\n        for i_level in reversed(range(self.num_resolutions)):\n            for i_block in range(self.num_res_blocks + 1):\n                h = self.up[i_level].block[i_block](h)\n                if len(self.up[i_level].attn) > 0:\n                    h = self.up[i_level].attn[i_block](h)\n            if i_level != 0:\n                h = self.up[i_level].upsample(h)\n\n        # end\n        h = self.norm_out(h)\n        h = swish(h)\n        h = self.conv_out(h)\n        return h\n\n\nclass DiagonalGaussian(nn.Module):\n    def __init__(self, sample: bool = True, chunk_dim: int = 1):\n        super().__init__()\n        self.sample = sample\n        self.chunk_dim = chunk_dim\n\n    def forward(self, z: Tensor) -> Tensor:\n        mean, logvar = torch.chunk(z, 2, dim=self.chunk_dim)\n        if self.sample:\n            std = torch.exp(0.5 * logvar)\n            return mean + std * torch.randn_like(mean)\n        else:\n            return mean\n\n\nclass AutoEncoder(nn.Module):\n    def __init__(self, params: AutoEncoderParams):\n        super().__init__()\n        self.encoder = Encoder(\n            resolution=params.resolution,\n            in_channels=params.in_channels,\n            ch=params.ch,\n            ch_mult=params.ch_mult,\n            num_res_blocks=params.num_res_blocks,\n            z_channels=params.z_channels,\n        )\n        self.decoder = Decoder(\n            resolution=params.resolution,\n            in_channels=params.in_channels,\n            ch=params.ch,\n            out_ch=params.out_ch,\n            ch_mult=params.ch_mult,\n            num_res_blocks=params.num_res_blocks,\n            z_channels=params.z_channels,\n        )\n        self.reg = DiagonalGaussian()\n\n        self.scale_factor = params.scale_factor\n        self.shift_factor = params.shift_factor\n\n    def encode(self, x: Tensor) -> Tensor:\n        z = self.reg(self.encoder(x))\n        z = self.scale_factor * (z - self.shift_factor)\n        return z\n\n    def decode(self, z: Tensor) -> Tensor:\n        z = z / self.scale_factor + self.shift_factor\n        return self.decoder(z)\n\n    def forward(self, x: Tensor) -> Tensor:\n        return self.decode(self.encode(x))\n\n\ndef print_load_warning(missing: list[str], unexpected: list[str]) -> None:\n    if len(missing) > 0 and len(unexpected) > 0:\n        print(f"Got {len(missing)} missing keys:\\n\\t" + "\\n\\t".join(missing))\n        print("\\n" + "-" * 79 + "\\n")\n        print(f"Got {len(unexpected)} unexpected keys:\\n\\t" + "\\n\\t".join(unexpected))\n    elif len(missing) > 0:\n        print(f"Got {len(missing)} missing keys:\\n\\t" + "\\n\\t".join(missing))\n    elif len(unexpected) > 0:\n        print(f"Got {len(unexpected)} unexpected keys:\\n\\t" + "\\n\\t".join(unexpected))\n\n\ndef load_ae(local_path: str) -> AutoEncoder:\n    ae_params = AutoEncoderParams(\n            resolution=256,\n            in_channels=3,\n            downsample=8,\n            ch=128,\n            out_ch=3,\n            ch_mult=[1, 2, 4, 4],\n            num_res_blocks=2,\n            z_channels=16,\n            scale_factor=0.3611,\n            shift_factor=0.1159,\n    )\n\n    # Loading the autoencoder\n    ae = AutoEncoder(ae_params)\n\n    if local_path is not None:\n        sd = load_sft(local_path)\n        missing, unexpected = ae.load_state_dict(sd, strict=False, assign=True)\n        print_load_warning(missing, unexpected)\n    return ae, ae_params\n', 'modeling/bagel/__init__.py': "# Copyright 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n\n\nfrom .bagel import BagelConfig, Bagel\nfrom .qwen2_navit import Qwen2Config, Qwen2Model, Qwen2ForCausalLM\nfrom .siglip_navit import SiglipVisionConfig, SiglipVisionModel\n\n\n__all__ = [\n    'BagelConfig',\n    'Bagel',\n    'Qwen2Config',\n    'Qwen2Model', \n    'Qwen2ForCausalLM',\n    'SiglipVisionConfig',\n    'SiglipVisionModel',\n]\n", 'modeling/bagel/bagel.py': 'import copy\nimport torch\nfrom torch import nn\nfrom tqdm import tqdm\nfrom PIL import Image\nimport matplotlib.pyplot as plt\nimport torch.nn.functional as F\nfrom typing import List, Tuple, Optional, Union\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.configuration_utils import PretrainedConfig\nfrom torch.nn.attention.flex_attention import create_block_mask\nfrom data.data_utils import (\n    create_sparse_mask, \n    get_flattened_position_ids_extrapolate, \n    get_flattened_position_ids_interpolate,\n    patchify, \n)\nfrom .qwen2_navit import NaiveCache\nfrom .modeling_utils import MLPconnector, TimestepEmbedder, PositionEmbedding, PositionEmbedding_infer\n\n\nclass BagelConfig(PretrainedConfig):\n    def __init__(\n        self,\n        visual_gen=True,\n        visual_und=True,\n        llm_config=None,\n        vit_config=None,\n        vae_config=None,\n        latent_patch_size=2,\n        max_latent_size=32,\n        vit_max_num_patch_per_side=70,\n        connector_act="gelu_pytorch_tanh",\n        interpolate_pos=False,\n        timestep_shift=1.0,\n        **kwargs\n    ):\n        super().__init__(**kwargs)\n        self.visual_gen = visual_gen\n        self.visual_und = visual_und\n        self.llm_config = llm_config\n        self.vit_config = vit_config\n        self.vae_config = vae_config\n        self.latent_patch_size = latent_patch_size\n        self.max_latent_size = max_latent_size\n        self.vit_max_num_patch_per_side = vit_max_num_patch_per_side\n        self.connector_act = connector_act\n        self.interpolate_pos = interpolate_pos\n        self.timestep_shift = timestep_shift\n\n\nclass Bagel(PreTrainedModel):\n    config_class = BagelConfig\n    base_model_prefix = \'bagel\'\n\n    def __init__(self, language_model, vit_model, config: BagelConfig):\n        super().__init__(config)    \n        self.language_model = language_model\n        self.hidden_size = config.llm_config.hidden_size\n        self.use_moe = "Mo" in config.llm_config.layer_module\n        self.num_heads = config.llm_config.num_attention_heads\n        if config.visual_gen:\n            self.latent_patch_size = config.latent_patch_size\n            self.timestep_shift = config.timestep_shift\n            self.latent_downsample = config.vae_config.downsample * config.latent_patch_size\n            self.max_latent_size = config.max_latent_size\n            self.latent_channel = config.vae_config.z_channels\n            self.patch_latent_dim = self.latent_patch_size ** 2 * self.latent_channel\n            self.time_embedder = TimestepEmbedder(self.hidden_size)\n            self.vae2llm = nn.Linear(self.patch_latent_dim, self.hidden_size)\n            self.llm2vae = nn.Linear(self.hidden_size, self.patch_latent_dim)\n            self.latent_pos_embed = PositionEmbedding(self.max_latent_size, self.hidden_size)\n        if config.visual_und:\n            self.vit_model = vit_model\n            self.vit_patch_size = config.vit_config.patch_size\n            self.vit_max_num_patch_per_side = config.vit_max_num_patch_per_side\n            self.vit_hidden_size = config.vit_config.hidden_size\n            self.connector = MLPconnector(self.vit_hidden_size, self.hidden_size, config.connector_act)\n            self.vit_pos_embed = PositionEmbedding(self.vit_max_num_patch_per_side, self.hidden_size)\n        if config.interpolate_pos:\n            self.get_flattened_position_ids = get_flattened_position_ids_interpolate\n        else:\n            self.get_flattened_position_ids = get_flattened_position_ids_extrapolate\n        self.config = config\n\n    def _init_weights(self):\n        if self.config.visual_gen:\n            nn.init.constant_(self.llm2vae.weight, 0)\n            nn.init.constant_(self.llm2vae.bias, 0)\n\n    def forward(\n        self,\n        sequence_length: Union[int, torch.Tensor],\n        packed_text_ids: torch.LongTensor,\n        packed_text_indexes: torch.LongTensor,\n        sample_lens: List[int],\n        packed_position_ids: torch.LongTensor,\n        nested_attention_masks: List[torch.Tensor] = None,\n        split_lens: List[int] = None,\n        attn_modes: List[str] = None,\n        # for visual understanding\n        ce_loss_indexes: Optional[torch.BoolTensor] = None,\n        packed_label_ids: Optional[torch.LongTensor] = None,\n        packed_vit_tokens: Optional[torch.Tensor] = None,\n        packed_vit_token_indexes: Optional[torch.LongTensor] = None,\n        packed_vit_position_ids: Optional[torch.LongTensor] = None,\n        vit_token_seqlens: Optional[torch.IntTensor] = None,\n        # for visual generation\n        padded_latent: Optional[torch.Tensor] = None,\n        patchified_vae_latent_shapes: Optional[List[Tuple[int, int]]] = None,\n        packed_latent_position_ids: Optional[torch.LongTensor] = None,\n        packed_vae_token_indexes: Optional[torch.LongTensor] = None,\n        packed_timesteps: Optional[torch.LongTensor] = None,\n        mse_loss_indexes: Optional[torch.BoolTensor] = None,\n    ) -> torch.Tensor:\n        """\n        Args:\n            sequence_length: length of sequence.\n            packed_text_ids: 1-D int tensor, packed text token ids.\n            packed_text_indexes: 1-D int tensor, packed text token indexes in sequence.\n            sample_lens: A list of N ints, length of each sample in packed_sequence.\n            nested_attention_masks: A list of N 2-D float tensor,  where 0.0 means attention and \n                -inf means ignore.\n            packed_position_ids: packed 1-D positions, an image has only one global position shared\n                by all latent tokens.\n\n            packed_vit_tokens: packed patchified image tokens for vit model.\n            packed_vit_position_ids: 1-D int tensor, the position of each token for vit model.\n            packed_vit_token_indexes: 1-D int tensor, packed vit token indexes in sequence.\n            vit_token_seqlens: 1-D int tensor, the length of each image tokens for vit model.\n            packed_label_ids: 1-D int tensor, packed label token ids.\n            ce_loss_indexes: 1-D bool tensor, where to compute ce loss.\n\n            padded_latent: padded latent from VAE encoder.\n            patchified_vae_latent_shapes: A list of (h, w) tuples, patchfied latent shapes of each image.\n            packed_latent_position_ids: 1-D int tensor, the position of each token for latent.\n            packed_vae_token_indexes: 1-D int tensor, padded image token indexes in sequence.\n            packed_timesteps: 1-D float tensor, flow timesteps. 0 indicates use clean image.\n            mse_loss_indexes: 1-D bool tensor, where to compute mse loss.\n        """\n        packed_text_embedding = self.language_model.model.embed_tokens(packed_text_ids)\n        \n        packed_sequence = packed_text_embedding.new_zeros(size=(sequence_length, self.hidden_size))\n        packed_sequence[packed_text_indexes] = packed_text_embedding\n\n        if nested_attention_masks is None:\n            \n            sparse_mask = create_sparse_mask(sample_lens, split_lens, attn_modes, packed_text_embedding.device)\n            seqlen = sum(sample_lens)\n            block_mask = create_block_mask(\n                sparse_mask, B=1, H=self.num_heads, Q_LEN=seqlen, KV_LEN=seqlen, \n                device=packed_text_embedding.device, BLOCK_SIZE=128, _compile=True\n            )\n            attention_mask = block_mask\n        else:\n            attention_mask = nested_attention_masks\n        \n        \n        if self.config.visual_und:\n\n            cu_seqlens = torch.nn.functional.pad(torch.cumsum(vit_token_seqlens, dim=0), (1, 0))\n            cu_seqlens = cu_seqlens.to(torch.int32)\n            max_seqlen = torch.max(vit_token_seqlens).item()\n            packed_vit_token_embed = self.vit_model(\n                packed_pixel_values=packed_vit_tokens, \n                packed_flattened_position_ids=packed_vit_position_ids,\n                cu_seqlens=cu_seqlens,\n                max_seqlen=max_seqlen,\n            )\n            packed_vit_token_embed = self.connector(packed_vit_token_embed)\n            vit_token_pos_emb = self.vit_pos_embed(packed_vit_position_ids)\n            packed_vit_token_embed = packed_vit_token_embed + vit_token_pos_emb\n            packed_sequence[packed_vit_token_indexes] = packed_vit_token_embed\n        \n        # 输入图像的处理\n\n        if self.config.visual_gen:\n            p = self.latent_patch_size\n            packed_latent = []\n            for latent, (h, w) in zip(padded_latent, patchified_vae_latent_shapes):\n                latent = latent[:, :h * p, :w * p].reshape(self.latent_channel, h, p, w, p)\n                latent = torch.einsum("chpwq->hwpqc", latent).reshape(-1, p * p * self.latent_channel)\n                packed_latent.append(latent)\n            packed_latent_clean = torch.cat(packed_latent, dim=0)\n            noise = torch.randn_like(packed_latent_clean)\n            packed_timesteps = torch.sigmoid(packed_timesteps)\n            packed_timesteps = self.timestep_shift * packed_timesteps / (1 + (self.timestep_shift - 1) * packed_timesteps)\n            packed_latent = (1 - packed_timesteps[:, None]) * packed_latent_clean + packed_timesteps[:, None] * noise\n            packed_timestep_embeds = self.time_embedder(packed_timesteps)\n            latent_token_pos_emb = self.latent_pos_embed(packed_latent_position_ids)\n            packed_latent = self.vae2llm(packed_latent) + packed_timestep_embeds + latent_token_pos_emb\n            packed_sequence[packed_vae_token_indexes] = packed_latent\n\n        extra_inputs = {}\n        if self.use_moe:\n            packed_und_token_indexes = packed_text_indexes\n            if packed_vit_token_indexes is not None:\n                packed_und_token_indexes=torch.cat([packed_text_indexes, packed_vit_token_indexes], dim=0)\n            extra_inputs.update(\n                packed_und_token_indexes=packed_und_token_indexes,\n                packed_gen_token_indexes=packed_vae_token_indexes,\n            )\n\n        \n        last_hidden_state = self.language_model(\n            packed_sequence=packed_sequence,\n            sample_lens=sample_lens,\n            attention_mask=attention_mask,\n            packed_position_ids=packed_position_ids,\n            **extra_inputs,\n        )\n\n        mse = None\n        if self.config.visual_gen:\n            packed_mse_preds = self.llm2vae(last_hidden_state[mse_loss_indexes])\n            target = noise - packed_latent_clean # NOTE: v_t=dx_t/dt=x_1-x_0, pointing from data to noise\n            has_mse = packed_timesteps > 0\n            mse = (packed_mse_preds - target[has_mse]) ** 2\n\n            timesteps_expanded = packed_timesteps[has_mse][:, None]\n            edited_latent_clean = packed_latent_clean[has_mse] - timesteps_expanded * packed_mse_preds\n            output_images = self.extract_image_predictions(edited_latent_clean, has_mse, patchified_vae_latent_shapes)\n\n        ce = None\n        if ce_loss_indexes is not None:\n            packed_ce_preds = self.language_model.lm_head(last_hidden_state[ce_loss_indexes])\n            ce = F.cross_entropy(packed_ce_preds, packed_label_ids, reduction="none")\n\n        return dict(mse=mse, ce=ce, output_images=output_images)\n\n    \n    def extract_image_predictions(self, packed_mse_preds, has_mse, patchified_vae_latent_shapes):\n        training_images = []\n        pred_idx = 0\n        \n        i = 0\n        while i < len(has_mse):\n            if has_mse[i]:\n                start = i\n                while i < len(has_mse) and has_mse[i]:\n                    i += 1\n                end = i\n                \n                segment_length = end - start\n                \n                for img_idx, (h, w) in enumerate(patchified_vae_latent_shapes):\n                    if h * w == segment_length:\n                        img_preds = packed_mse_preds[pred_idx:pred_idx + segment_length]\n                        training_images.append({\n                            \'img_idx\': img_idx,\n                            \'shape\': (h, w),\n                            \'predictions\': img_preds.reshape(h, w, -1)\n                        })\n                        pred_idx += segment_length\n                        break\n            else:\n                i += 1\n        \n        return training_images\n\n    def prepare_prompts(self, curr_kvlens, curr_rope, prompts, tokenizer, new_token_ids):\n        packed_text_ids = list()\n        packed_text_position_ids = list()\n        text_token_lens = list()\n        packed_text_indexes = list()\n        packed_key_value_indexes = list()\n\n        curr = 0\n        newlens, new_rope = list(), list()\n        for prompt, curr_kvlen, curr_position_id in zip(prompts, curr_kvlens, curr_rope):\n            packed_key_value_indexes.extend(range(curr, curr + curr_kvlen))\n            curr += curr_kvlen\n\n            text_ids = tokenizer.encode(prompt)\n            text_ids = [new_token_ids[\'bos_token_id\']] + text_ids + [new_token_ids[\'eos_token_id\']]\n            text_token_lens.append(len(text_ids))\n            packed_text_ids.extend(text_ids)\n            packed_text_position_ids.extend(range(curr_position_id, curr_position_id + len(text_ids)))\n            packed_text_indexes.extend(range(curr, curr + len(text_ids)))\n            newlens.append(curr_kvlen + len(text_ids))\n            new_rope.append(curr_position_id + len(text_ids))\n            curr += len(text_ids)\n\n        generation_input = {\n            "text_token_lens": torch.tensor(text_token_lens, dtype=torch.int),\n            "packed_text_ids": torch.tensor(packed_text_ids, dtype=torch.long),\n            "packed_text_position_ids": torch.tensor(packed_text_position_ids, dtype=torch.long),\n            "packed_text_indexes": torch.tensor(packed_text_indexes, dtype=torch.long),\n            "packed_key_value_indexes": torch.tensor(packed_key_value_indexes, dtype=torch.long),\n            "key_values_lens": torch.tensor(curr_kvlens, dtype=torch.int),\n        }\n\n        return generation_input, newlens, new_rope\n\n    @torch.no_grad\n    def forward_cache_update_text(\n        self,\n        past_key_values: NaiveCache,\n        packed_text_ids: torch.IntTensor,\n        packed_text_position_ids: torch.LongTensor,\n        text_token_lens: torch.LongTensor,\n        packed_text_indexes: torch.LongTensor,\n        packed_key_value_indexes: torch.LongTensor,\n        key_values_lens: torch.IntTensor,\n    ):\n        packed_text_embedding = self.language_model.model.embed_tokens(packed_text_ids)\n\n        extra_inputs = {}\n        if self.use_moe:\n            extra_inputs = {"mode": "und"}\n\n        output = self.language_model.forward_inference(\n            packed_query_sequence=packed_text_embedding,\n            query_lens=text_token_lens,\n            packed_query_position_ids=packed_text_position_ids,\n            packed_query_indexes=packed_text_indexes,\n            past_key_values=past_key_values,\n            packed_key_value_indexes=packed_key_value_indexes,\n            key_values_lens=key_values_lens,\n            update_past_key_values=True,\n            is_causal=True,\n            **extra_inputs,\n        )\n        past_key_values = output.past_key_values\n\n        return past_key_values\n\n    def prepare_vit_images(self, curr_kvlens, curr_rope, images, transforms, new_token_ids):\n        packed_vit_token_indexes = list()\n        vit_token_seqlens, packed_vit_tokens, packed_vit_position_ids = list(), list(), list()\n        packed_text_ids, packed_text_indexes = list(), list()\n        packed_seqlens, packed_position_ids, packed_indexes = list(), list(), list()\n        packed_key_value_indexes = list()\n\n        _curr = curr = 0\n        newlens, new_rope = list(), list()\n        for image, curr_kvlen, curr_position_id in zip(images, curr_kvlens, curr_rope):\n            packed_key_value_indexes.extend(range(curr, curr + curr_kvlen))\n            curr += curr_kvlen\n\n            packed_text_ids.append(new_token_ids[\'start_of_image\'])\n            packed_text_indexes.append(_curr)\n            packed_indexes.append(curr)\n            curr += 1\n            _curr += 1\n\n            image_tensor = transforms(image)\n            vit_position_ids = self.get_flattened_position_ids(\n                image_tensor.size(1), image_tensor.size(2), \n                self.vit_patch_size, \n                max_num_patches_per_side=self.vit_max_num_patch_per_side\n            )\n            vit_tokens = patchify(image_tensor, self.vit_patch_size)\n            packed_vit_tokens.append(vit_tokens)\n            num_img_tokens = vit_tokens.shape[0]\n            packed_vit_position_ids.append(vit_position_ids)\n            vit_token_seqlens.append(num_img_tokens)\n            packed_vit_token_indexes.extend(range(_curr, _curr + num_img_tokens))\n            packed_indexes.extend(range(curr, curr + num_img_tokens))\n            curr += num_img_tokens\n            _curr += num_img_tokens\n\n            packed_text_ids.append(new_token_ids[\'end_of_image\'])\n            packed_text_indexes.append(_curr)\n            packed_indexes.append(curr)\n            curr += 1\n            _curr += 1\n\n            packed_position_ids.extend([curr_position_id] * (num_img_tokens + 2))\n            packed_seqlens.append(num_img_tokens + 2)\n            newlens.append(curr_kvlen + num_img_tokens + 2)\n            new_rope.append(curr_position_id + 1)\n\n        generation_input = {\n            "packed_text_ids": torch.tensor(packed_text_ids, dtype=torch.long),\n            "packed_text_indexes": torch.tensor(packed_text_indexes, dtype=torch.long),\n            "vit_token_seqlens": torch.tensor(vit_token_seqlens, dtype=torch.int),\n            "packed_vit_tokens": torch.cat(packed_vit_tokens, dim=0),\n            "packed_vit_position_ids": torch.cat(packed_vit_position_ids, dim=0),\n            "packed_vit_token_indexes": torch.tensor(packed_vit_token_indexes, dtype=torch.long),\n            "packed_position_ids": torch.tensor(packed_position_ids, dtype=torch.long),\n            "packed_seqlens": torch.tensor(packed_seqlens, dtype=torch.int),\n            "packed_indexes": torch.tensor(packed_indexes, dtype=torch.long),\n            "packed_key_value_indexes": torch.tensor(packed_key_value_indexes, dtype=torch.long),\n            "key_values_lens": torch.tensor(curr_kvlens, dtype=torch.int),\n        }\n\n        return generation_input, newlens, new_rope\n\n    @torch.no_grad\n    def forward_cache_update_vit(\n        self,\n        past_key_values: NaiveCache,\n        packed_text_ids: torch.LongTensor,\n        packed_text_indexes: torch.LongTensor,\n        packed_vit_tokens: torch.Tensor,\n        packed_vit_token_indexes: torch.LongTensor,\n        packed_vit_position_ids: torch.LongTensor,\n        vit_token_seqlens: torch.IntTensor,\n        packed_position_ids: torch.LongTensor,\n        packed_seqlens: torch.IntTensor,\n        packed_indexes: torch.LongTensor,\n        packed_key_value_indexes: torch.LongTensor,\n        key_values_lens: torch.IntTensor,\n    ):\n        packed_text_embedding = self.language_model.model.embed_tokens(packed_text_ids)\n        packed_sequence = packed_text_embedding.new_zeros((sum(packed_seqlens), self.hidden_size))\n        packed_sequence[packed_text_indexes] = packed_text_embedding\n\n        cu_seqlens = torch.nn.functional.pad(torch.cumsum(vit_token_seqlens, dim=0), (1, 0))\n        cu_seqlens = cu_seqlens.to(torch.int32)\n        max_seqlen = torch.max(vit_token_seqlens).item()\n        packed_vit_token_embed = self.vit_model(\n            packed_pixel_values=packed_vit_tokens, \n            packed_flattened_position_ids=packed_vit_position_ids,\n            cu_seqlens=cu_seqlens,\n            max_seqlen=max_seqlen,\n        )\n        packed_vit_token_embed = self.connector(packed_vit_token_embed)\n        pos_emb = self.vit_pos_embed(packed_vit_position_ids)\n        packed_vit_token_embed = packed_vit_token_embed + pos_emb\n        if packed_vit_token_embed.dtype != packed_sequence.dtype:\n            packed_vit_token_embed = packed_vit_token_embed.to(packed_sequence.dtype)\n        packed_sequence[packed_vit_token_indexes] = packed_vit_token_embed\n\n        extra_inputs = {}\n        if self.use_moe:\n            extra_inputs = {"mode": "und"}\n\n        output = self.language_model.forward_inference(\n            packed_query_sequence=packed_sequence,\n            query_lens=packed_seqlens,\n            packed_query_position_ids=packed_position_ids,\n            packed_query_indexes=packed_indexes,\n            past_key_values=past_key_values,\n            packed_key_value_indexes=packed_key_value_indexes,\n            key_values_lens=key_values_lens,\n            update_past_key_values=True,\n            is_causal=False,\n            **extra_inputs,\n        )\n        past_key_values = output.past_key_values\n\n        return past_key_values\n\n    def prepare_vae_images(self, curr_kvlens, curr_rope, images, transforms, new_token_ids, timestep=0):\n        patchified_vae_latent_shapes, packed_vae_position_ids = list(), list()\n        packed_vae_token_indexes = list()\n        packed_text_ids, packed_text_indexes = list(), list()\n        packed_seqlens, packed_position_ids, packed_indexes = list(), list(), list()\n        packed_key_value_indexes = list()\n\n        _curr = curr = 0\n        vae_image_tensors = list()\n        newlens, new_rope = list(), list()\n        for image, curr_kvlen, curr_position_id in zip(images, curr_kvlens, curr_rope):\n            packed_key_value_indexes.extend(range(curr, curr + curr_kvlen))\n            curr += curr_kvlen\n\n            packed_text_ids.append(new_token_ids[\'start_of_image\'])\n            packed_text_indexes.append(_curr)\n            packed_indexes.append(curr)\n            curr += 1\n            _curr += 1\n            image_tensor = transforms(image)\n            vae_image_tensors.append(image_tensor)\n            vae_posiiton_ids = self.get_flattened_position_ids(\n                image_tensor.size(1), image_tensor.size(2),\n                self.latent_downsample, \n                max_num_patches_per_side=self.max_latent_size\n            )\n            packed_vae_position_ids.append(vae_posiiton_ids)\n            H, W = image_tensor.shape[1:]\n            h = H // self.latent_downsample\n            w = W // self.latent_downsample\n            patchified_vae_latent_shapes.append((h, w))\n\n            num_img_tokens = w * h\n            packed_vae_token_indexes.extend(range(_curr, _curr + num_img_tokens))\n            packed_indexes.extend(range(curr, curr + num_img_tokens))\n            curr += num_img_tokens\n            _curr += num_img_tokens\n\n            packed_text_ids.append(new_token_ids[\'end_of_image\'])\n            packed_text_indexes.append(_curr)\n            packed_indexes.append(curr)\n            curr += 1\n            _curr += 1\n\n            packed_position_ids.extend([curr_position_id] * (num_img_tokens + 2))\n            packed_seqlens.append(num_img_tokens + 2)\n            newlens.append(curr_kvlen + num_img_tokens + 2)\n            new_rope.append(curr_position_id + 1)\n\n        image_sizes = [item.shape for item in vae_image_tensors]\n        max_image_size = [max(item) for item in list(zip(*image_sizes))]\n        padded_images = torch.zeros(size=(len(vae_image_tensors), *max_image_size))\n        for i, image_tensor in enumerate(vae_image_tensors):\n            padded_images[i, :, :image_tensor.shape[1], :image_tensor.shape[2]] = image_tensor\n\n        generation_input = {\n            "padded_images": padded_images,\n            "patchified_vae_latent_shapes": patchified_vae_latent_shapes,\n            "packed_vae_position_ids": torch.cat(packed_vae_position_ids, dim=0),\n            "packed_timesteps": torch.tensor([timestep]),\n            "packed_vae_token_indexes": torch.tensor(packed_vae_token_indexes, dtype=torch.long),\n            "packed_text_ids": torch.tensor(packed_text_ids, dtype=torch.long),\n            "packed_text_indexes": torch.tensor(packed_text_indexes, dtype=torch.long),\n            "packed_position_ids": torch.tensor(packed_position_ids, dtype=torch.long),\n            "packed_seqlens": torch.tensor(packed_seqlens, dtype=torch.int),\n            "packed_indexes": torch.tensor(packed_indexes, dtype=torch.long),\n            "packed_key_value_indexes": torch.tensor(packed_key_value_indexes, dtype=torch.long),\n            "key_values_lens": torch.tensor(curr_kvlens, dtype=torch.int),\n        }\n\n        return generation_input, newlens, new_rope\n\n    @torch.no_grad\n    def forward_cache_update_vae(\n        self,\n        vae_model,\n        past_key_values: NaiveCache,\n        padded_images: torch.Tensor,\n        patchified_vae_latent_shapes: List,\n        packed_vae_position_ids: torch.LongTensor,\n        packed_timesteps: torch.Tensor,\n        packed_vae_token_indexes: torch.LongTensor,\n        packed_text_ids: torch.LongTensor,\n        packed_text_indexes: torch.LongTensor,\n        packed_position_ids: torch.LongTensor,\n        packed_seqlens: torch.IntTensor,\n        packed_indexes: torch.LongTensor,\n        key_values_lens: torch.IntTensor,\n        packed_key_value_indexes: torch.Tensor,\n    ):\n        packed_text_embedding = self.language_model.model.embed_tokens(packed_text_ids)\n        packed_sequence = packed_text_embedding.new_zeros((sum(packed_seqlens), self.hidden_size))\n        packed_sequence[packed_text_indexes] = packed_text_embedding\n\n        vae_dtype = next(vae_model.parameters()).dtype\n        vae_device = next(vae_model.parameters()).device\n        padded_images = padded_images.to(device=vae_device, dtype=vae_dtype)\n        padded_latent = vae_model.encode(padded_images)\n        padded_latent = vae_model.encode(padded_images)\n\n        p = self.latent_patch_size\n        packed_latent = list()\n        for latent, (h, w) in zip(padded_latent, patchified_vae_latent_shapes):\n            latent = latent[:, :h * p, :w * p].reshape(self.latent_channel, h, p, w, p)\n            latent = torch.einsum("chpwq->hwpqc", latent).reshape(-1, p * p * self.latent_channel)\n            packed_latent.append(latent)\n        packed_latent = torch.cat(packed_latent, dim=0)\n        packed_pos_embed = self.latent_pos_embed(packed_vae_position_ids)\n        packed_timestep_embeds = self.time_embedder(packed_timesteps)\n        packed_latent = self.vae2llm(packed_latent) + packed_timestep_embeds + packed_pos_embed # 变成llm输入的维度\n        if packed_latent.dtype != packed_sequence.dtype:\n            packed_latent = packed_latent.to(packed_sequence.dtype)\n        packed_sequence[packed_vae_token_indexes] = packed_latent\n\n        extra_inputs = {}\n        if self.use_moe:\n            extra_inputs = {\n                "mode": "gen",\n                "packed_vae_token_indexes": packed_vae_token_indexes,\n                "packed_text_indexes": packed_text_indexes\n            }\n\n        # 在这里就进模型推理了\n        output = self.language_model.forward_inference(\n            packed_query_sequence=packed_sequence,\n            query_lens=packed_seqlens,\n            packed_query_position_ids=packed_position_ids,\n            packed_query_indexes=packed_indexes,\n            past_key_values=past_key_values,\n            key_values_lens=key_values_lens,\n            packed_key_value_indexes=packed_key_value_indexes,\n            update_past_key_values=True,\n            is_causal=False,\n            **extra_inputs,\n        )\n        past_key_values = output.past_key_values\n\n        return past_key_values\n\n    def prepare_vae_latent(self, curr_kvlens, curr_rope, image_sizes, new_token_ids):\n        packed_text_ids, packed_text_indexes = list(), list()\n        packed_vae_position_ids, packed_vae_token_indexes, packed_init_noises = list(), list(), list()\n        packed_position_ids, packed_seqlens, packed_indexes = list(), list(), list()\n        packed_key_value_indexes = list()\n\n        query_curr = curr = 0\n        for (H, W), curr_kvlen, curr_position_id in zip(image_sizes, curr_kvlens, curr_rope):\n            packed_key_value_indexes.extend(range(curr, curr + curr_kvlen))\n            curr += curr_kvlen\n\n            packed_text_ids.append(new_token_ids[\'start_of_image\'])\n            packed_text_indexes.append(query_curr)\n            packed_indexes.append(curr)\n            curr += 1\n            query_curr += 1\n\n            vae_posiiton_ids = self.get_flattened_position_ids(\n                H, W,\n                self.latent_downsample, \n                max_num_patches_per_side=self.max_latent_size\n            )\n            packed_vae_position_ids.append(vae_posiiton_ids)\n\n            h, w = H // self.latent_downsample, W // self.latent_downsample\n            num_image_tokens = h * w\n            packed_init_noises.append(\n                torch.randn(num_image_tokens, self.latent_channel * self.latent_patch_size ** 2)\n            )\n            packed_vae_token_indexes.extend(range(query_curr, query_curr + num_image_tokens))\n            packed_indexes.extend(range(curr, curr + num_image_tokens))\n            curr += num_image_tokens\n            query_curr += num_image_tokens\n\n            packed_text_ids.append(new_token_ids[\'end_of_image\'])\n            packed_text_indexes.append(query_curr)\n            packed_indexes.append(curr)\n            curr += 1\n            query_curr += 1\n\n            packed_position_ids.extend([curr_position_id] * (num_image_tokens + 2))\n            packed_seqlens.append(num_image_tokens + 2)\n\n        generation_input = {\n            "packed_text_ids": torch.tensor(packed_text_ids, dtype=torch.long),\n            "packed_text_indexes": torch.tensor(packed_text_indexes, dtype=torch.long),\n            "packed_init_noises": torch.cat(packed_init_noises, dim=0),\n            "packed_vae_position_ids": torch.cat(packed_vae_position_ids, dim=0),\n            "packed_vae_token_indexes": torch.tensor(packed_vae_token_indexes, dtype=torch.long),\n            "packed_seqlens": torch.tensor(packed_seqlens, dtype=torch.int),\n            "packed_position_ids": torch.tensor(packed_position_ids, dtype=torch.long),\n            "key_values_lens": torch.tensor(curr_kvlens, dtype=torch.int),\n            "packed_indexes": torch.tensor(packed_indexes, dtype=torch.long),\n            "packed_key_value_indexes": torch.tensor(packed_key_value_indexes, dtype=torch.long),\n        }\n\n        return generation_input\n\n    def prepare_vae_latent_cfg(self, curr_kvlens, curr_rope, image_sizes):\n        packed_position_ids, packed_indexes, packed_key_value_indexes = list(), list(), list()\n\n        query_curr = curr = 0\n        for (H, W), curr_kvlen, curr_position_id in zip(image_sizes, curr_kvlens, curr_rope):\n            packed_key_value_indexes.extend(range(curr, curr + curr_kvlen))\n            curr += curr_kvlen\n\n            packed_indexes.append(curr)\n            curr += 1\n            query_curr += 1\n\n            h, w = H // self.latent_downsample, W // self.latent_downsample\n            num_image_tokens = h * w\n            packed_indexes.extend(range(curr, curr + num_image_tokens))\n            curr += num_image_tokens\n            query_curr += num_image_tokens\n\n            packed_indexes.append(curr)\n            curr += 1\n            query_curr += 1\n\n            packed_position_ids.extend([curr_position_id] * (num_image_tokens + 2))\n\n        generation_input = {\n            "cfg_packed_position_ids": torch.tensor(packed_position_ids, dtype=torch.long),\n            "cfg_key_values_lens": torch.tensor(curr_kvlens, dtype=torch.int),\n            "cfg_packed_query_indexes": torch.tensor(packed_indexes, dtype=torch.long),\n            "cfg_packed_key_value_indexes": torch.tensor(packed_key_value_indexes, dtype=torch.long),\n        }\n\n        return generation_input\n\n    @torch.no_grad\n    def generate_image(\n        self,\n        packed_text_ids: torch.LongTensor,\n        packed_text_indexes: torch.LongTensor,\n        packed_init_noises: torch.Tensor,\n        packed_vae_position_ids: torch.LongTensor,\n        packed_vae_token_indexes: torch.LongTensor,\n        packed_seqlens: torch.IntTensor,\n        packed_position_ids: torch.LongTensor,\n        packed_indexes: torch.LongTensor,\n        past_key_values: NaiveCache,\n        key_values_lens: torch.IntTensor,\n        packed_key_value_indexes: torch.LongTensor,\n        num_timesteps: int = 24,\n        timestep_shift: float = 1.0,\n        cfg_renorm_min: float = 0.0,\n        cfg_renorm_type: str = "global",\n        cfg_interval: Optional[Tuple[float, float]] = [0, 1],\n        # cfg_text\n        cfg_text_scale: float = 1.0,\n        cfg_text_packed_query_indexes: Optional[torch.LongTensor] = None,\n        cfg_text_packed_position_ids: Optional[torch.LongTensor] = None,\n        cfg_text_past_key_values: Optional[NaiveCache] = None,\n        cfg_text_key_values_lens: Optional[torch.IntTensor] = None,\n        cfg_text_packed_key_value_indexes: Optional[torch.LongTensor] = None,\n        # cfg_img\n        cfg_img_scale: float = 1.0,\n        cfg_img_packed_query_indexes: Optional[torch.LongTensor] = None,\n        cfg_img_packed_position_ids: Optional[torch.LongTensor] = None,\n        cfg_img_past_key_values: Optional[NaiveCache] = None,\n        cfg_img_key_values_lens: Optional[torch.IntTensor] = None,\n        cfg_img_packed_key_value_indexes: Optional[torch.LongTensor] = None,\n        cfg_type: str = "parallel",\n    ):\n        x_t = packed_init_noises # 就是一个噪声\n        timesteps = torch.linspace(1, 0, num_timesteps, device=x_t.device)\n        timesteps = timestep_shift * timesteps / (1 + (timestep_shift - 1) * timesteps)\n        dts =  timesteps[:-1] - timesteps[1:]\n        # import pdb; pdb.set_trace()\n        timesteps = timesteps[:-1]\n\n        for i, t in tqdm(enumerate(timesteps), total=len(timesteps)):\n            \n            timestep = torch.tensor([t] * x_t.shape[0], device=x_t.device)\n            if t > cfg_interval[0] and t <= cfg_interval[1]:\n                cfg_text_scale_ = cfg_text_scale\n                cfg_img_scale_ = cfg_img_scale\n            else:\n                cfg_text_scale_ = 1.0\n                cfg_img_scale_ = 1.0\n            v_t = self._forward_flow(\n                x_t=x_t,\n                timestep=timestep, \n                packed_vae_token_indexes=packed_vae_token_indexes,\n                packed_vae_position_ids=packed_vae_position_ids,\n                packed_text_ids=packed_text_ids,\n                packed_text_indexes=packed_text_indexes,\n                packed_position_ids=packed_position_ids,\n                packed_indexes=packed_indexes,\n                packed_seqlens=packed_seqlens,\n                key_values_lens=key_values_lens,\n                past_key_values=past_key_values,\n                packed_key_value_indexes=packed_key_value_indexes,\n                cfg_renorm_min=cfg_renorm_min,\n                cfg_renorm_type=cfg_renorm_type,\n                # cfg_text\n                cfg_text_scale=cfg_text_scale_,\n                cfg_text_packed_position_ids=cfg_text_packed_position_ids,\n                cfg_text_packed_query_indexes=cfg_text_packed_query_indexes,\n                cfg_text_key_values_lens=cfg_text_key_values_lens,\n                cfg_text_past_key_values=cfg_text_past_key_values,\n                cfg_text_packed_key_value_indexes=cfg_text_packed_key_value_indexes,\n                # cfg_img\n                cfg_img_scale=cfg_img_scale_,\n                cfg_img_packed_position_ids=cfg_img_packed_position_ids,\n                cfg_img_packed_query_indexes=cfg_img_packed_query_indexes,\n                cfg_img_key_values_lens=cfg_img_key_values_lens,\n                cfg_img_past_key_values=cfg_img_past_key_values,\n                cfg_img_packed_key_value_indexes=cfg_img_packed_key_value_indexes,\n                cfg_type=cfg_type,\n            )\n\n            x_t = x_t - v_t.to(x_t.device) * dts[i] # velocity pointing from data to noise\n\n        unpacked_latent = x_t.split((packed_seqlens - 2).tolist())\n        return unpacked_latent\n\n    @torch.no_grad\n    def _forward_flow(\n        self,\n        x_t: torch.Tensor,\n        timestep: torch.LongTensor,\n        packed_vae_token_indexes: torch.LongTensor,\n        packed_vae_position_ids: torch.LongTensor,\n        packed_text_ids: torch.LongTensor,\n        packed_text_indexes: torch.LongTensor,\n        packed_indexes: torch.LongTensor,\n        packed_position_ids: torch.LongTensor,\n        packed_seqlens: torch.IntTensor,\n        key_values_lens: torch.IntTensor,\n        past_key_values: NaiveCache,\n        packed_key_value_indexes: torch.LongTensor,\n        cfg_renorm_min: float = 0.0,\n        cfg_renorm_type: str = "global",\n        # cfg_text\n        cfg_text_scale: float = 1.0,\n        cfg_text_packed_position_ids: Optional[torch.LongTensor] = None,\n        cfg_text_packed_query_indexes: Optional[torch.LongTensor] = None,\n        cfg_text_key_values_lens: Optional[torch.Tensor] = None,\n        cfg_text_past_key_values: Optional[NaiveCache] = None,\n        cfg_text_packed_key_value_indexes: Optional[torch.LongTensor] = None,\n        # cfg_img\n        cfg_img_scale: float = 1.0,\n        cfg_img_packed_position_ids: Optional[torch.LongTensor] = None,\n        cfg_img_packed_query_indexes: Optional[torch.LongTensor] = None,\n        cfg_img_key_values_lens: Optional[torch.Tensor] = None,\n        cfg_img_past_key_values: Optional[NaiveCache] = None,\n        cfg_img_packed_key_value_indexes: Optional[torch.LongTensor] = None,\n        cfg_type: str = "parallel",\n    ):\n        packed_text_embedding = self.language_model.model.embed_tokens(packed_text_ids)\n        packed_sequence = packed_text_embedding.new_zeros((sum(packed_seqlens), self.hidden_size))\n        packed_sequence[packed_text_indexes] = packed_text_embedding\n\n        assert timestep.unique().shape[0] == 1\n        packed_pos_embed = self.latent_pos_embed(packed_vae_position_ids)\n        packed_timestep_embeds = self.time_embedder(timestep)\n        x_t = self.vae2llm(x_t) + packed_timestep_embeds + packed_pos_embed\n        if x_t.dtype != packed_sequence.dtype:\n            x_t = x_t.to(packed_sequence.dtype)\n        packed_sequence[packed_vae_token_indexes] = x_t \n\n        extra_inputs = {}\n        if self.use_moe:\n            extra_inputs = {\n                "mode": "gen",\n                "packed_vae_token_indexes": packed_vae_token_indexes,\n                "packed_text_indexes": packed_text_indexes\n            }\n\n        output = self.language_model.forward_inference(\n            packed_query_sequence=packed_sequence,\n            query_lens=packed_seqlens,\n            packed_query_position_ids=packed_position_ids,\n            packed_query_indexes=packed_indexes,\n            past_key_values=past_key_values,\n            key_values_lens=key_values_lens,\n            packed_key_value_indexes=packed_key_value_indexes,\n            update_past_key_values=False,\n            is_causal=False,\n            **extra_inputs,\n        )\n        v_t = self.llm2vae(output.packed_query_sequence)\n        v_t = v_t[packed_vae_token_indexes]\n\n        if cfg_text_scale > 1.0:\n            cfg_text_output = self.language_model.forward_inference(\n                packed_query_sequence=packed_sequence,\n                query_lens=packed_seqlens,\n                packed_query_position_ids=cfg_text_packed_position_ids,\n                packed_query_indexes=cfg_text_packed_query_indexes,\n                past_key_values=cfg_text_past_key_values,\n                key_values_lens=cfg_text_key_values_lens,\n                packed_key_value_indexes=cfg_text_packed_key_value_indexes,\n                update_past_key_values=False,\n                is_causal=False,\n                **extra_inputs,\n            )\n            cfg_text_v_t = self.llm2vae(cfg_text_output.packed_query_sequence)\n            cfg_text_v_t = cfg_text_v_t[packed_vae_token_indexes]\n\n        if cfg_img_scale > 1.0:\n            cfg_img_output = self.language_model.forward_inference(\n                packed_query_sequence=packed_sequence,\n                query_lens=packed_seqlens,\n                packed_query_position_ids=cfg_img_packed_position_ids,\n                packed_query_indexes=cfg_img_packed_query_indexes,\n                past_key_values=cfg_img_past_key_values,\n                key_values_lens=cfg_img_key_values_lens,\n                packed_key_value_indexes=cfg_img_packed_key_value_indexes,\n                update_past_key_values=False,\n                is_causal=False,\n                **extra_inputs,\n            )\n            cfg_img_v_t = self.llm2vae(cfg_img_output.packed_query_sequence)\n            cfg_img_v_t = cfg_img_v_t[packed_vae_token_indexes]\n\n        if cfg_text_scale > 1.0:\n            if cfg_renorm_type == "text_channel":\n                v_t_text_ = cfg_text_v_t + cfg_text_scale * (v_t - cfg_text_v_t)\n                norm_v_t = torch.norm(v_t, dim=-1, keepdim=True)\n                norm_v_t_text_ = torch.norm(v_t_text_, dim=-1, keepdim=True)\n                scale = (norm_v_t / (norm_v_t_text_ + 1e-8)).clamp(min=cfg_renorm_min, max=1.0)\n                v_t_text = v_t_text_ * scale\n                if cfg_img_scale > 1.0:\n                    v_t = cfg_img_v_t + cfg_img_scale * (v_t_text - cfg_img_v_t)\n                else:\n                    v_t = v_t_text\n            else:\n                v_t_text_ = cfg_text_v_t + cfg_text_scale * (v_t - cfg_text_v_t)\n                \n                if cfg_img_scale > 1.0:\n                    v_t_ = cfg_img_v_t + cfg_img_scale * (v_t_text_ - cfg_img_v_t)\n                else:\n                    v_t_ = v_t_text_\n\n                # NOTE norm is computed over all dimensions, thus currently only supports batch_size = 1 with navit\n                if cfg_renorm_type == "global":\n                    norm_v_t = torch.norm(v_t)\n                    norm_v_t_ = torch.norm(v_t_)\n                elif cfg_renorm_type == "channel":\n                    norm_v_t = torch.norm(v_t, dim=-1, keepdim=True)\n                    norm_v_t_ = torch.norm(v_t_, dim=-1, keepdim=True)\n                else:\n                    raise NotImplementedError(f"{cfg_renorm_type} is not suppoprted")\n                scale = (norm_v_t / (norm_v_t_ + 1e-8)).clamp(min=cfg_renorm_min, max=1.0)\n                v_t = v_t_ * scale\n        else:\n            # No CFG\n            pass\n\n        return v_t\n\n    def prepare_start_tokens(self, curr_kvlens, curr_rope, new_token_ids):\n        packed_start_tokens, packed_key_value_indexes = list(), list()\n        packed_query_position_ids = list()\n\n        curr = 0\n        for curr_kvlen, curr_position_id in zip(curr_kvlens, curr_rope):\n            packed_key_value_indexes.extend(range(curr, curr + curr_kvlen))\n            packed_start_tokens.append(new_token_ids[\'bos_token_id\'])\n            packed_query_position_ids.append(curr_position_id)\n            curr += curr_kvlen\n\n        generation_input = {\n            "packed_start_tokens": torch.tensor(packed_start_tokens, dtype=torch.long),\n            "packed_query_position_ids": torch.tensor(packed_query_position_ids, dtype=torch.long),\n            "key_values_lens": torch.tensor(curr_kvlens, dtype=torch.int),\n            "packed_key_value_indexes": torch.tensor(packed_key_value_indexes, dtype=torch.long),\n        }\n\n        return generation_input\n\n    @torch.no_grad\n    def generate_text(\n        self,\n        past_key_values: NaiveCache,\n        packed_key_value_indexes: torch.LongTensor,\n        key_values_lens: torch.IntTensor,\n        packed_start_tokens: torch.LongTensor,\n        packed_query_position_ids: torch.LongTensor,\n        max_length: int,\n        do_sample: bool = False,\n        temperature: float = 1.0,\n        end_token_id: int = None,\n    ):\n        step = 0\n        generated_sequence = []\n        curr_tokens = packed_start_tokens\n        while step < max_length:\n            generated_sequence.append(curr_tokens)\n            packed_text_embedding = self.language_model.model.embed_tokens(curr_tokens)\n            query_lens = torch.ones_like(curr_tokens)\n            packed_query_indexes = torch.cumsum(key_values_lens, dim=0) + torch.arange(\n                0, len(key_values_lens), \n                device=key_values_lens.device, \n                dtype=key_values_lens.dtype\n            )\n\n            uppacked = list(packed_key_value_indexes.split(key_values_lens.tolist(), dim=0))\n            for i in range(len(uppacked)):\n                uppacked[i] += i\n            packed_key_value_indexes = torch.cat(uppacked, dim=0)\n\n            extra_inputs = {}\n            if self.use_moe:\n                extra_inputs = {"mode": "und"}\n\n            output = self.language_model.forward_inference(\n                packed_query_sequence=packed_text_embedding,\n                query_lens=query_lens,\n                packed_query_position_ids=packed_query_position_ids,\n                packed_query_indexes=packed_query_indexes,\n                past_key_values=past_key_values,\n                key_values_lens=key_values_lens,\n                packed_key_value_indexes=packed_key_value_indexes,\n                update_past_key_values=True,\n                is_causal=True,\n                **extra_inputs,\n            )\n            past_key_values = output.past_key_values\n            packed_query_sequence = output.packed_query_sequence\n            pred_logits = self.language_model.lm_head(packed_query_sequence)\n\n            if do_sample:\n                probs = nn.functional.softmax(pred_logits / temperature, dim=-1)\n                curr_tokens = torch.multinomial(probs, num_samples=1).squeeze(1)\n            else:\n                curr_tokens = torch.argmax(pred_logits, dim=-1)\n\n            uppacked = list(packed_key_value_indexes.split(key_values_lens.tolist(), dim=0))\n            for i in range(len(uppacked)):\n                uppacked[i] = torch.cat(\n                    [uppacked[i], torch.tensor([uppacked[i][-1] + 1], device=uppacked[i].device)], dim=0\n                )\n            packed_key_value_indexes = torch.cat(uppacked, dim=0)\n            key_values_lens = key_values_lens + 1\n            packed_query_position_ids = packed_query_position_ids + 1\n            step += 1\n\n            if end_token_id is not None and curr_tokens[0] == end_token_id: # only support batch=1\n                break\n\n        output_device = generated_sequence[0].device\n        return torch.stack([i.to(output_device) for i in generated_sequence], dim=0)\n\n    # for evaluation\n    @torch.no_grad()\n    def chat(\n        self,\n        tokenizer,\n        new_token_ids,\n        image_transform,\n        images,\n        prompt,\n        max_length: int,\n        do_sample: bool = False,\n        temperature: float = 1.0,\n    ):\n        device = next(self.parameters()).device\n\n        if isinstance(new_token_ids, dict):\n            for k, v in new_token_ids.items():\n                if torch.is_tensor(v):\n                    new_token_ids[k] = v.to(device)\n        elif torch.is_tensor(new_token_ids):\n            new_token_ids = new_token_ids.to(device)\n\n        # prefill\n        past_key_values = NaiveCache(self.config.llm_config.num_hidden_layers)\n        newlens = [0]\n        new_rope = [0]\n\n        # add images\n        for image in images:\n            generation_input, newlens, new_rope = self.prepare_vit_images(\n                curr_kvlens=newlens,\n                curr_rope=new_rope, \n                images=[image], \n                transforms=image_transform,\n                new_token_ids=new_token_ids,\n            )\n            for k, v in generation_input.items():\n                if torch.is_tensor(v):\n                    generation_input[k] = v.to(device)\n            with torch.amp.autocast("cuda", enabled=True, dtype=torch.bfloat16):\n                past_key_values = self.forward_cache_update_vit(past_key_values, **generation_input)\n\n        # add text\n        generation_input, newlens, new_rope = self.prepare_prompts(\n            curr_kvlens=newlens,\n            curr_rope=new_rope, \n            prompts=[prompt],\n            tokenizer=tokenizer, \n            new_token_ids=new_token_ids,\n        )\n        for k, v in generation_input.items():\n            if torch.is_tensor(v):\n                generation_input[k] = v.to(device)\n        with torch.amp.autocast("cuda", enabled=True, dtype=torch.bfloat16):\n            past_key_values = self.forward_cache_update_text(past_key_values, **generation_input)\n\n        # decode\n        generation_input = self.prepare_start_tokens(newlens, new_rope, new_token_ids)\n        for k, v in generation_input.items():\n            if torch.is_tensor(v):\n                generation_input[k] = v.to(device)\n        with torch.amp.autocast("cuda", enabled=True, dtype=torch.bfloat16):\n            unpacked_latent = self.generate_text(\n                past_key_values=past_key_values,\n                max_length=max_length,\n                do_sample=do_sample,\n                temperature=temperature,\n                end_token_id=new_token_ids[\'eos_token_id\'],\n                **generation_input,\n            )\n        output = tokenizer.decode(unpacked_latent[:,0])\n        output = output.split(\'<|im_end|>\')[0].split(\'<|im_start|>\')[1]\n\n        return output\n', 'modeling/bagel/modeling_utils.py': '# Copyright (c) 2022 Facebook, Inc. and its affiliates.\n# Copyright (c) 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: CC BY-NC 4.0\n#\n# This file has been modified by ByteDance Ltd. and/or its affiliates. on 2025-05-20.\n#\n# Original file was released under CC BY-NC 4.0, with the full license text\n# available at https://github.com/facebookresearch/DiT/blob/main/LICENSE.txt.\n#\n# This modified file is released under the same license.\n\nimport math\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom transformers.activations import ACT2FN\n\n# --------------------------------------------------------\n# 2D sine-cosine position embedding\n# References:\n# DiT: https://github.com/facebookresearch/DiT/blob/main/models.py\n# --------------------------------------------------------\ndef get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False, extra_tokens=0):\n    grid_h = np.arange(grid_size, dtype=np.float32)\n    grid_w = np.arange(grid_size, dtype=np.float32)\n    grid = np.meshgrid(grid_w, grid_h)  # here w goes first\n    grid = np.stack(grid, axis=0)\n\n    grid = grid.reshape([2, 1, grid_size, grid_size])\n    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)\n    if cls_token and extra_tokens > 0:\n        pos_embed = np.concatenate([np.zeros([extra_tokens, embed_dim]), pos_embed], axis=0)\n    return pos_embed\n\n\ndef get_2d_sincos_pos_embed_from_grid(embed_dim, grid):\n    assert embed_dim % 2 == 0\n\n    # use half of dimensions to encode grid_h\n    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)\n    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)\n\n    emb = np.concatenate([emb_h, emb_w], axis=1) # (H*W, D)\n    return emb\n\n\ndef get_1d_sincos_pos_embed_from_grid(embed_dim, pos):\n    """\n    embed_dim: output dimension for each position\n    pos: a list of positions to be encoded: size (M,)\n    out: (M, D)\n    """\n    assert embed_dim % 2 == 0\n    omega = np.arange(embed_dim // 2, dtype=np.float64)\n    omega /= embed_dim / 2.\n    omega = 1. / 10000**omega  # (D/2,)\n\n    pos = pos.reshape(-1)  # (M,)\n    out = np.einsum(\'m,d->md\', pos, omega)  # (M, D/2), outer product\n\n    emb_sin = np.sin(out) # (M, D/2)\n    emb_cos = np.cos(out) # (M, D/2)\n\n    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)\n    return emb\n\n\n# --------------------------------------------------------\n# TimestepEmbedder\n# Reference:\n# DiT: https://github.com/facebookresearch/DiT/blob/main/models.py\n# --------------------------------------------------------\nclass TimestepEmbedder(nn.Module):\n    """\n    Embeds scalar timesteps into vector representations.\n    """\n    def __init__(self, hidden_size, frequency_embedding_size=256):\n        super().__init__()\n        self.mlp = nn.Sequential(\n            nn.Linear(frequency_embedding_size, hidden_size, bias=True),\n            nn.SiLU(),\n            nn.Linear(hidden_size, hidden_size, bias=True),\n        )\n        self.frequency_embedding_size = frequency_embedding_size\n\n    @staticmethod\n    def timestep_embedding(t, dim, max_period=10000):\n        """\n        Create sinusoidal timestep embeddings.\n        :param t: a 1-D Tensor of N indices, one per batch element.\n                          These may be fractional.\n        :param dim: the dimension of the output.\n        :param max_period: controls the minimum frequency of the embeddings.\n        :return: an (N, D) Tensor of positional embeddings.\n        """\n        half = dim // 2\n        freqs = torch.exp(\n            -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half\n        ).to(device=t.device)\n\n        args = t[:, None].float() * freqs[None]\n        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)\n        if dim % 2:\n            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)\n        return embedding\n\n    def forward(self, t):\n        t_freq = self.timestep_embedding(t, self.frequency_embedding_size)\n        if hasattr(self.mlp[0], \'weight\'):\n            target_dtype = self.mlp[0].weight.dtype\n            if t_freq.dtype != target_dtype:\n                t_freq = t_freq.to(target_dtype)\n        t_emb = self.mlp(t_freq)\n        return t_emb\n\n\nclass MLPconnector(nn.Module):\n    def __init__(self, in_dim: int, out_dim: int, hidden_act: str):\n        super().__init__()\n        self.activation_fn = ACT2FN[hidden_act]\n        self.fc1 = nn.Linear(in_dim, out_dim)\n        self.fc2 = nn.Linear(out_dim, out_dim)\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        hidden_states = self.fc1(hidden_states)\n        hidden_states = self.activation_fn(hidden_states)\n        hidden_states = self.fc2(hidden_states)\n        return hidden_states\n\n\nclass PositionEmbedding(nn.Module):\n    def __init__(self, max_num_patch_per_side, hidden_size):\n        super().__init__()\n        self.max_num_patch_per_side = max_num_patch_per_side\n        self.hidden_size = hidden_size\n        self.pos_embed = nn.Parameter(\n            torch.zeros(max_num_patch_per_side ** 2, hidden_size), \n            requires_grad=False\n        )\n        self._init_weights()\n\n    def _init_weights(self):\n        # Initialize (and freeze) pos_embed by sin-cos embedding:\n        pos_embed = get_2d_sincos_pos_embed(self.hidden_size, self.max_num_patch_per_side)\n        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float())\n\n    def forward(self, position_ids):\n        return self.pos_embed[position_ids]\n    \nclass PositionEmbedding_infer(nn.Module):\n    def __init__(self, max_num_patch_per_side, hidden_size):\n        super().__init__()\n        self.max_num_patch_per_side = max_num_patch_per_side\n        self.hidden_size = hidden_size\n        self.pos_embed = nn.Parameter(\n            torch.zeros(max_num_patch_per_side ** 2, hidden_size), \n            requires_grad=False\n        )\n        self._init_weights()\n    \n    def _init_weights(self):\n        # Initialize (and freeze) pos_embed by sin-cos embedding:\n        pos_embed = get_2d_sincos_pos_embed(self.hidden_size, self.max_num_patch_per_side)\n        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float())\n\n    def forward(self, position_ids):\n        return self.pos_embed[position_ids]', 'modeling/bagel/qwen2_navit.py': '# Copyright (c) 2024 The Qwen Team and The HuggingFace Inc. team.\n# Copyright (c) 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n#\n# This file has been modified by ByteDance Ltd. and/or its affiliates. on 2025-05-20.\n#\n# Original file was released under Apache-2.0, with the full license text\n# available at https://github.com/huggingface/transformers/blob/main/LICENSE.\n#\n# This modified file is released under the same license.\n\n\nfrom dataclasses import dataclass\nfrom functools import partial\nfrom typing import List, Optional, Tuple\n\nimport torch\nfrom torch import nn\nfrom torch.nn.attention import SDPBackend, sdpa_kernel\nfrom torch.nn.attention.flex_attention import flex_attention\nfrom torch.nn.functional import scaled_dot_product_attention\nfrom transformers.utils import ModelOutput\n\n\nfrom flash_attn import flash_attn_varlen_func\n\nfrom modeling.qwen2.modeling_qwen2 import (\n    Qwen2Attention, \n    Qwen2MLP, \n    Qwen2PreTrainedModel, \n    Qwen2RMSNorm, \n    Qwen2RotaryEmbedding,\n    apply_rotary_pos_emb,\n)\n\nfrom modeling.qwen2.configuration_qwen2 import Qwen2Config as _Qwen2Config\n\n\ntorch._dynamo.config.cache_size_limit = 512\ntorch._dynamo.config.accumulated_cache_size_limit = 4096\n# flex_attention = torch.compile(flex_attention) # , dynamic=True, mode=\'max-autotune\'\nflex_attention = torch.compile(flex_attention)\n\n\nclass Qwen2Config(_Qwen2Config):\n    r"""\n    This is the configuration class to store the configuration of a [`Qwen2Model`]. It is used to instantiate a\n    Qwen2 model according to the specified arguments, defining the model architecture. Instantiating a configuration\n    with the defaults will yield a similar configuration to that of\n    Qwen2-7B-beta [Qwen/Qwen2-7B-beta](https://huggingface.co/Qwen/Qwen2-7B-beta).\n\n    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the\n    documentation from [`PretrainedConfig`] for more information.\n\n    Args:\n        vocab_size (`int`, *optional*, defaults to 151936):\n            Vocabulary size of the Qwen2 model. Defines the number of different tokens that can be represented by the\n            `inputs_ids` passed when calling [`Qwen2Model`]\n        hidden_size (`int`, *optional*, defaults to 4096):\n            Dimension of the hidden representations.\n        intermediate_size (`int`, *optional*, defaults to 22016):\n            Dimension of the MLP representations.\n        num_hidden_layers (`int`, *optional*, defaults to 32):\n            Number of hidden layers in the Transformer encoder.\n        num_attention_heads (`int`, *optional*, defaults to 32):\n            Number of attention heads for each attention layer in the Transformer encoder.\n        num_key_value_heads (`int`, *optional*, defaults to 32):\n            This is the number of key_value heads that should be used to implement Grouped Query Attention. If\n            `num_key_value_heads=num_attention_heads`, the model will use Multi Head Attention (MHA), if\n            `num_key_value_heads=1` the model will use Multi Query Attention (MQA) otherwise GQA is used. When\n            converting a multi-head checkpoint to a GQA checkpoint, each group key and value head should be constructed\n            by meanpooling all the original heads within that group. For more details checkout [this\n            paper](https://arxiv.org/pdf/2305.13245.pdf). If it is not specified, will default to `32`.\n        hidden_act (`str` or `function`, *optional*, defaults to `"silu"`):\n            The non-linear activation function (function or string) in the decoder.\n        max_position_embeddings (`int`, *optional*, defaults to 32768):\n            The maximum sequence length that this model might ever be used with.\n        initializer_range (`float`, *optional*, defaults to 0.02):\n            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.\n        rms_norm_eps (`float`, *optional*, defaults to 1e-06):\n            The epsilon used by the rms normalization layers.\n        use_cache (`bool`, *optional*, defaults to `True`):\n            Whether or not the model should return the last key/values attentions (not used by all models). Only\n            relevant if `config.is_decoder=True`.\n        tie_word_embeddings (`bool`, *optional*, defaults to `False`):\n            Whether the model\'s input and output word embeddings should be tied.\n        rope_theta (`float`, *optional*, defaults to 10000.0):\n            The base period of the RoPE embeddings.\n        rope_scaling (`Dict`, *optional*):\n            Dictionary containing the scaling configuration for the RoPE embeddings. NOTE: if you apply new rope type\n            and you expect the model to work on longer `max_position_embeddings`, we recommend you to update this value\n            accordingly.\n            Expected contents:\n                `rope_type` (`str`):\n                    The sub-variant of RoPE to use. Can be one of [\'default\', \'linear\', \'dynamic\', \'yarn\', \'longrope\',\n                    \'llama3\'], with \'default\' being the original RoPE implementation.\n                `factor` (`float`, *optional*):\n                    Used with all rope types except \'default\'. The scaling factor to apply to the RoPE embeddings. In\n                    most scaling types, a `factor` of x will enable the model to handle sequences of length x *\n                    original maximum pre-trained length.\n                `original_max_position_embeddings` (`int`, *optional*):\n                    Used with \'dynamic\', \'longrope\' and \'llama3\'. The original max position embeddings used during\n                    pretraining.\n                `attention_factor` (`float`, *optional*):\n                    Used with \'yarn\' and \'longrope\'. The scaling factor to be applied on the attention\n                    computation. If unspecified, it defaults to value recommended by the implementation, using the\n                    `factor` field to infer the suggested value.\n                `beta_fast` (`float`, *optional*):\n                    Only used with \'yarn\'. Parameter to set the boundary for extrapolation (only) in the linear\n                    ramp function. If unspecified, it defaults to 32.\n                `beta_slow` (`float`, *optional*):\n                    Only used with \'yarn\'. Parameter to set the boundary for interpolation (only) in the linear\n                    ramp function. If unspecified, it defaults to 1.\n                `short_factor` (`List[float]`, *optional*):\n                    Only used with \'longrope\'. The scaling factor to be applied to short contexts (<\n                    `original_max_position_embeddings`). Must be a list of numbers with the same length as the hidden\n                    size divided by the number of attention heads divided by 2\n                `long_factor` (`List[float]`, *optional*):\n                    Only used with \'longrope\'. The scaling factor to be applied to long contexts (<\n                    `original_max_position_embeddings`). Must be a list of numbers with the same length as the hidden\n                    size divided by the number of attention heads divided by 2\n                `low_freq_factor` (`float`, *optional*):\n                    Only used with \'llama3\'. Scaling factor applied to low frequency components of the RoPE\n                `high_freq_factor` (`float`, *optional*):\n                    Only used with \'llama3\'. Scaling factor applied to high frequency components of the RoPE\n        use_sliding_window (`bool`, *optional*, defaults to `False`):\n            Whether to use sliding window attention.\n        sliding_window (`int`, *optional*, defaults to 4096):\n            Sliding window attention (SWA) window size. If not specified, will default to `4096`.\n        max_window_layers (`int`, *optional*, defaults to 28):\n            The number of layers that use SWA (Sliding Window Attention). The bottom layers use SWA while the top use full attention.\n        attention_dropout (`float`, *optional*, defaults to 0.0):\n            The dropout ratio for the attention probabilities.\n\n    ```python\n    >>> from transformers import Qwen2Model, Qwen2Config\n\n    >>> # Initializing a Qwen2 style configuration\n    >>> configuration = Qwen2Config()\n\n    >>> # Initializing a model from the Qwen2-7B style configuration\n    >>> model = Qwen2Model(configuration)\n\n    >>> # Accessing the model configuration\n    >>> configuration = model.config\n    ```"""\n\n    model_type = "qwen2"\n    keys_to_ignore_at_inference = ["past_key_values"]\n\n    def __init__(\n        self,\n        vocab_size=151936,\n        hidden_size=4096,\n        intermediate_size=22016,\n        num_hidden_layers=32,\n        num_attention_heads=32,\n        num_key_value_heads=32,\n        hidden_act="silu",\n        max_position_embeddings=32768,\n        initializer_range=0.02,\n        rms_norm_eps=1e-6,\n        use_cache=True,\n        tie_word_embeddings=False,\n        rope_theta=10000.0,\n        rope_scaling=None,\n        use_sliding_window=False,\n        sliding_window=4096,\n        max_window_layers=28,\n        attention_dropout=0.0,\n        is_causal=True,\n        _attn_implementation="flash_attention_2",\n        qk_norm=True,\n        layer_module="Qwen2DecoderLayer",\n        freeze_und=False,\n        **kwargs,\n    ):\n        super().__init__(\n            vocab_size=vocab_size,\n            hidden_size=hidden_size,\n            intermediate_size=intermediate_size,\n            num_hidden_layers=num_hidden_layers,\n            num_attention_heads=num_attention_heads,\n            num_key_value_heads=num_key_value_heads,\n            hidden_act=hidden_act,\n            max_position_embeddings=max_position_embeddings,\n            initializer_range=initializer_range,\n            rms_norm_eps=rms_norm_eps,\n            use_cache=use_cache,\n            tie_word_embeddings=tie_word_embeddings,\n            rope_theta=rope_theta,\n            rope_scaling=rope_scaling,\n            use_sliding_window=use_sliding_window,\n            sliding_window=sliding_window,\n            max_window_layers=max_window_layers,\n            attention_dropout=attention_dropout,\n            is_causal=is_causal,\n            _attn_implementation=_attn_implementation,\n            **kwargs,\n        )\n        self.qk_norm = qk_norm\n        self.layer_module = layer_module\n        self.freeze_und = freeze_und\n\n\nclass NaiveCache:\n    def __init__(self, num_layers):\n        self.key_cache = {k: None for k in range(num_layers)}\n        self.value_cache = {k: None for k in range(num_layers)}\n\n    @property\n    def num_layers(self):\n        return len(self.key_cache)\n\n    @property\n    def seq_lens(self):\n        if self.key_cache[0] is not None:\n            return self.key_cache[0].shape[0]\n        else:\n            return 0\n\n\n@dataclass\nclass BaseNavitOutputWithPast(ModelOutput):\n    packed_query_sequence: torch.FloatTensor = None\n    past_key_values: Optional[NaiveCache] = None\n\n\ndef pad_sequence(tensor, pad_size):\n    H, L, D = tensor.shape\n    pad_tensor = tensor.new_zeros((H, pad_size, D))\n    return torch.cat([tensor, pad_tensor], dim=1)\n\n\nclass PackedAttention(Qwen2Attention):\n    def __init__(self, config, layer_idx: Optional[int] = None):\n        super().__init__(config, layer_idx)\n        if self.config.qk_norm:\n            self.q_norm = Qwen2RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n            self.k_norm = Qwen2RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n        else:\n            # f(x) = x\n            self.q_norm = nn.Identity()\n            self.k_norm = nn.Identity()\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask: List[torch.Tensor],\n        packed_position_embeddings: Tuple[torch.Tensor, torch.Tensor],\n    ):\n        packed_query_states = self.q_proj(packed_sequence).view(-1, self.num_heads, self.head_dim)\n        packed_key_states = self.k_proj(packed_sequence).view(-1, self.num_key_value_heads, self.head_dim)\n        packed_value_states = self.v_proj(packed_sequence).view(-1, self.num_key_value_heads, self.head_dim)\n\n        packed_query_states = self.q_norm(packed_query_states)\n        packed_key_states = self.k_norm(packed_key_states)\n\n        packed_cos, packed_sin = packed_position_embeddings\n        packed_query_states, packed_key_states = apply_rotary_pos_emb(\n            packed_query_states, packed_key_states, packed_cos, packed_sin, unsqueeze_dim=1\n        )\n\n        if isinstance(attention_mask, List):\n            packed_key_states = packed_key_states[:, :, None, :].repeat(1, 1, self.num_key_value_groups, 1)\n            packed_key_states = packed_key_states.reshape(-1, self.num_heads, self.head_dim)\n            packed_value_states = packed_value_states[:, :, None, :].repeat(1, 1, self.num_key_value_groups, 1)\n            packed_value_states = packed_value_states.reshape(-1, self.num_heads, self.head_dim)\n\n            unpacked_query_states = packed_query_states.transpose(0, 1).split(sample_lens, dim=1)\n            unpacked_key_states = packed_key_states.transpose(0, 1).split(sample_lens, dim=1)\n            unpacked_value_states = packed_value_states.transpose(0, 1).split(sample_lens, dim=1)\n            upacked_attn_output = []\n            for query_states, key_states, value_states, attention_mask_per_sample in zip(\n                unpacked_query_states, unpacked_key_states, unpacked_value_states, attention_mask\n            ):\n                with sdpa_kernel(backends=[SDPBackend.EFFICIENT_ATTENTION]):\n                    attn_output = scaled_dot_product_attention(\n                        query_states.to(torch.bfloat16).unsqueeze(0), \n                        key_states.to(torch.bfloat16).unsqueeze(0), \n                        value_states.to(torch.bfloat16).unsqueeze(0),\n                        attention_mask_per_sample.to(torch.bfloat16).unsqueeze(0),\n                    )\n                upacked_attn_output.append(attn_output.squeeze(0))\n            packed_attn_output = torch.cat(upacked_attn_output, dim=1)\n        else:\n            pad_size = sum(sample_lens) - packed_query_states.shape[0]\n            packed_query_states = pad_sequence(packed_query_states.permute(1, 0, 2), pad_size)\n            packed_key_states = pad_sequence(packed_key_states.permute(1, 0, 2), pad_size)\n            packed_value_states = pad_sequence(packed_value_states.permute(1, 0, 2), pad_size)\n            packed_attn_output = flex_attention(\n                packed_query_states.unsqueeze(0), \n                packed_key_states.unsqueeze(0), \n                packed_value_states.unsqueeze(0), \n                enable_gqa=True,\n                block_mask=attention_mask,\n            )\n            end_index = packed_attn_output.shape[2] - pad_size\n            packed_attn_output = packed_attn_output[0, :, :end_index, :]\n\n        packed_attn_output = packed_attn_output.transpose(0, 1).reshape(-1, self.hidden_size)\n        packed_attn_output = self.o_proj(packed_attn_output)\n\n        return packed_attn_output\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_embeddings: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n    ):\n        packed_query_states = self.q_proj(packed_query_sequence).view(-1, self.num_heads, self.head_dim)\n        packed_key_states = self.k_proj(packed_query_sequence).view(-1, self.num_key_value_heads, self.head_dim)\n        packed_value_states = self.v_proj(packed_query_sequence).view(-1, self.num_key_value_heads, self.head_dim)\n\n        packed_query_states = self.q_norm(packed_query_states)\n        packed_key_states = self.k_norm(packed_key_states)\n\n        packed_cos, packed_sin = packed_query_position_embeddings\n        packed_query_states, packed_key_states = apply_rotary_pos_emb(\n            packed_query_states, packed_key_states, packed_cos, packed_sin, unsqueeze_dim=1\n        )\n\n        packed_query_states = packed_query_states.to(torch.bfloat16)\n        packed_key_states = packed_key_states.to(torch.bfloat16)\n        packed_value_states = packed_value_states.to(torch.bfloat16)\n\n        if past_key_values is not None and past_key_values.key_cache[self.layer_idx] is not None:\n            past_key_states = past_key_values.key_cache[self.layer_idx]\n            past_value_states = past_key_values.value_cache[self.layer_idx]\n\n            seqlens = sum(query_lens) + sum(key_values_lens)\n            merged_key_states = past_key_states.new_zeros((seqlens, self.num_key_value_heads, self.head_dim))\n            merged_value_states = past_key_states.new_zeros((seqlens, self.num_key_value_heads, self.head_dim))\n            merged_key_states[packed_query_indexes] = packed_key_states\n            merged_key_states[packed_key_value_indexes] = past_key_states\n            merged_value_states[packed_query_indexes] = packed_value_states\n            merged_value_states[packed_key_value_indexes] = past_value_states\n            key_values_lens = key_values_lens + query_lens\n        else:\n            merged_key_states = packed_key_states\n            merged_value_states = packed_value_states\n            key_values_lens = query_lens\n\n        cu_seqlens_q = torch.nn.functional.pad(torch.cumsum(query_lens, dim=0), (1, 0))\n        cu_seqlens_k = torch.nn.functional.pad(torch.cumsum(key_values_lens, dim=0), (1, 0))\n\n        packed_attn_output = flash_attn_varlen_func(\n            q=packed_query_states,\n            k=merged_key_states,\n            v=merged_value_states,\n            cu_seqlens_q=cu_seqlens_q.to(torch.int32),\n            cu_seqlens_k=cu_seqlens_k.to(torch.int32),\n            max_seqlen_q=max(query_lens).item(),\n            max_seqlen_k=max(key_values_lens).item(),\n            causal=is_causal,\n        )\n        packed_attn_output = packed_attn_output.reshape(-1, self.hidden_size)\n        packed_attn_output = self.o_proj(packed_attn_output)\n\n        if update_past_key_values:\n            past_key_values.key_cache[self.layer_idx] = merged_key_states\n            past_key_values.value_cache[self.layer_idx] = merged_value_states\n\n        return packed_attn_output, past_key_values\n\n\nclass PackedAttentionMoT(Qwen2Attention):\n    def __init__(self, config, layer_idx: Optional[int] = None):\n        super().__init__(config, layer_idx)\n        if self.config.qk_norm:\n            self.q_norm = Qwen2RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n            self.k_norm = Qwen2RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n            self.q_norm_moe_gen = Qwen2RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n            self.k_norm_moe_gen = Qwen2RMSNorm(self.head_dim, eps=config.rms_norm_eps)\n        else:\n            self.q_norm = nn.Identity()\n            self.k_norm = nn.Identity()\n            self.q_norm_moe_gen = nn.Identity()\n            self.k_norm_moe_gen = nn.Identity()\n\n        self.q_proj_moe_gen = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=True)\n        self.k_proj_moe_gen = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)\n        self.v_proj_moe_gen = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)\n        self.o_proj_moe_gen = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask,\n        packed_position_embeddings: Tuple[torch.Tensor, torch.Tensor],\n        packed_und_token_indexes: torch.LongTensor,\n        packed_gen_token_indexes: torch.LongTensor,\n    ):\n        packed_query_states = packed_sequence.new_zeros((packed_sequence.shape[0], self.num_heads * self.head_dim))\n        packed_key_states = packed_sequence.new_zeros((packed_sequence.shape[0], self.num_key_value_heads * self.head_dim))\n        packed_value_states = packed_sequence.new_zeros((packed_sequence.shape[0], self.num_key_value_heads * self.head_dim))\n\n        # 理解和生成分开\n        packed_sequence_und = packed_sequence[packed_und_token_indexes]\n        packed_sequence_gen = packed_sequence[packed_gen_token_indexes]\n\n        # 分开过不同的qkv\n        packed_query_states[packed_und_token_indexes] = self.q_proj(packed_sequence_und)\n        packed_query_states[packed_gen_token_indexes] = self.q_proj_moe_gen(packed_sequence_gen)\n\n        packed_key_states[packed_und_token_indexes] = self.k_proj(packed_sequence_und)\n        packed_key_states[packed_gen_token_indexes] = self.k_proj_moe_gen(packed_sequence_gen)\n\n        packed_value_states[packed_und_token_indexes] = self.v_proj(packed_sequence_und)\n        packed_value_states[packed_gen_token_indexes] = self.v_proj_moe_gen(packed_sequence_gen)\n\n        packed_query_states = packed_query_states.view(-1, self.num_heads, self.head_dim)\n        packed_key_states = packed_key_states.view(-1, self.num_key_value_heads, self.head_dim)\n        packed_value_states = packed_value_states.view(-1, self.num_key_value_heads, self.head_dim)\n        if self.config.freeze_und:\n            packed_value_states[packed_und_token_indexes] = packed_value_states[packed_und_token_indexes].detach()\n\n        packed_query_states_ = packed_query_states.new_zeros(packed_query_states.shape)\n        packed_key_states_ = packed_key_states.new_zeros(packed_key_states.shape)\n\n        packed_query_states_[packed_und_token_indexes] = self.q_norm(packed_query_states[packed_und_token_indexes])\n        if self.config.freeze_und:\n            packed_query_states_[packed_und_token_indexes] = packed_query_states_[packed_und_token_indexes].detach()\n        packed_query_states_[packed_gen_token_indexes] = self.q_norm_moe_gen(packed_query_states[packed_gen_token_indexes])\n\n        packed_key_states_[packed_und_token_indexes] = self.k_norm(packed_key_states[packed_und_token_indexes])\n        if self.config.freeze_und:\n            packed_key_states_[packed_und_token_indexes] = packed_key_states_[packed_und_token_indexes].detach()\n        packed_key_states_[packed_gen_token_indexes] = self.k_norm_moe_gen(packed_key_states[packed_gen_token_indexes])\n\n        packed_cos, packed_sin = packed_position_embeddings\n        packed_query_states_, packed_key_states_ = apply_rotary_pos_emb(\n            packed_query_states_, packed_key_states_, packed_cos, packed_sin, unsqueeze_dim=1\n        )\n\n        # 实际上是在这个位置做了交互\n        if isinstance(attention_mask, List):\n            packed_key_states_ = packed_key_states_[:, :, None, :].repeat(1, 1, self.num_key_value_groups, 1)\n            packed_key_states_ = packed_key_states_.reshape(-1, self.num_heads, self.head_dim)\n            packed_value_states = packed_value_states[:, :, None, :].repeat(1, 1, self.num_key_value_groups, 1)\n            packed_value_states = packed_value_states.reshape(-1, self.num_heads, self.head_dim)\n\n            unpacked_query_states = packed_query_states_.transpose(0, 1).split(sample_lens, dim=1)\n            unpacked_key_states = packed_key_states_.transpose(0, 1).split(sample_lens, dim=1)\n            unpacked_value_states = packed_value_states.transpose(0, 1).split(sample_lens, dim=1)\n            upacked_attn_output = []\n            for query_states, key_states, value_states, attention_mask_per_sample in zip(\n                unpacked_query_states, unpacked_key_states, unpacked_value_states, attention_mask\n            ):\n                with sdpa_kernel(backends=[SDPBackend.EFFICIENT_ATTENTION]):\n                    attn_output = scaled_dot_product_attention(\n                        query_states.to(torch.bfloat16).unsqueeze(0), \n                        key_states.to(torch.bfloat16).unsqueeze(0), \n                        value_states.to(torch.bfloat16).unsqueeze(0),\n                        attention_mask_per_sample.to(torch.bfloat16).unsqueeze(0),\n                    )\n                upacked_attn_output.append(attn_output.squeeze(0))\n            packed_attn_output = torch.cat(upacked_attn_output, dim=1)\n        else:\n            pad_size = sum(sample_lens) - packed_query_states.shape[0]\n            packed_query_states_ = pad_sequence(packed_query_states_.permute(1, 0, 2), pad_size)\n            packed_key_states_ = pad_sequence(packed_key_states_.permute(1, 0, 2), pad_size)\n            packed_value_states = pad_sequence(packed_value_states.permute(1, 0, 2), pad_size)\n            packed_attn_output = flex_attention(\n                packed_query_states_.unsqueeze(0), # 1, num_head, L, head_dim\n                packed_key_states_.unsqueeze(0), \n                packed_value_states.unsqueeze(0), \n                enable_gqa=True,\n                block_mask=attention_mask,\n            )\n            end_index = packed_attn_output.shape[2] - pad_size\n            packed_attn_output = packed_attn_output[0, :, :end_index, :]\n\n        packed_attn_output = packed_attn_output.transpose(0, 1).reshape(-1, self.num_heads * self.head_dim)\n        packed_attn_output_ = packed_attn_output.new_zeros(packed_attn_output.shape)\n        packed_attn_output_[packed_und_token_indexes] = self.o_proj(packed_attn_output[packed_und_token_indexes])\n        packed_attn_output_[packed_gen_token_indexes] = self.o_proj_moe_gen(packed_attn_output[packed_gen_token_indexes])\n\n        return packed_attn_output_\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_embeddings: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n        mode="und",\n        packed_vae_token_indexes=None,\n        packed_text_indexes=None,\n    ):\n        if mode == \'und\':\n            packed_query_states = self.q_proj(packed_query_sequence).view(-1, self.num_heads, self.head_dim)\n            packed_key_states = self.k_proj(packed_query_sequence).view(-1, self.num_key_value_heads, self.head_dim)\n            packed_value_states = self.v_proj(packed_query_sequence).view(-1, self.num_key_value_heads, self.head_dim)\n            packed_query_states = self.q_norm(packed_query_states)\n            packed_key_states = self.k_norm(packed_key_states)\n        elif mode == \'gen\':\n            packed_query_sequence = packed_query_sequence.to(torch.bfloat16) # 查询进来 整个vae的latent是一个q\n            packed_query_states = packed_query_sequence.new_zeros((packed_query_sequence.shape[0], self.num_heads * self.head_dim))\n            packed_key_states = packed_query_sequence.new_zeros((packed_query_sequence.shape[0], self.num_key_value_heads * self.head_dim))\n            packed_value_states = packed_query_sequence.new_zeros((packed_query_sequence.shape[0], self.num_key_value_heads * self.head_dim))\n\n            # 这里其实是要分开 过不同的 QKV\n            packed_text_query_sequence = packed_query_sequence[packed_text_indexes]\n            packed_vae_query_sequence = packed_query_sequence[packed_vae_token_indexes]\n\n            # 过不同的QKV\n            packed_query_states[packed_text_indexes] = self.q_proj(packed_text_query_sequence)\n            packed_query_states[packed_vae_token_indexes] = self.q_proj_moe_gen(packed_vae_query_sequence)\n\n            packed_key_states[packed_text_indexes] = self.k_proj(packed_text_query_sequence)\n            packed_key_states[packed_vae_token_indexes] = self.k_proj_moe_gen(packed_vae_query_sequence)\n\n            packed_value_states[packed_text_indexes] = self.v_proj(packed_text_query_sequence)\n            packed_value_states[packed_vae_token_indexes] = self.v_proj_moe_gen(packed_vae_query_sequence)\n\n            packed_query_states = packed_query_states.view(-1, self.num_heads, self.head_dim)\n            packed_key_states = packed_key_states.view(-1, self.num_key_value_heads, self.head_dim)\n            packed_value_states = packed_value_states.view(-1, self.num_key_value_heads, self.head_dim)\n            \n            # Q K 归一化 V 不需要\n            packed_query_states = packed_query_states.to(torch.float32)\n            packed_query_states[packed_text_indexes] = self.q_norm(packed_query_states[packed_text_indexes])\n            packed_query_states[packed_vae_token_indexes] = self.q_norm_moe_gen(packed_query_states[packed_vae_token_indexes])\n\n            packed_key_states = packed_key_states.to(torch.float32)\n            packed_key_states[packed_text_indexes] = self.k_norm(packed_key_states[packed_text_indexes])\n            packed_key_states[packed_vae_token_indexes] = self.k_norm_moe_gen(packed_key_states[packed_vae_token_indexes])\n\n        packed_cos, packed_sin = packed_query_position_embeddings\n        packed_query_states, packed_key_states = apply_rotary_pos_emb(\n            packed_query_states, packed_key_states, packed_cos, packed_sin, unsqueeze_dim=1\n        )\n\n        packed_query_states = packed_query_states.to(torch.bfloat16)\n        packed_key_states = packed_key_states.to(torch.bfloat16)\n        packed_value_states = packed_value_states.to(torch.bfloat16)\n\n        if past_key_values is not None and past_key_values.key_cache[self.layer_idx] is not None:\n            past_key_states = past_key_values.key_cache[self.layer_idx]\n            past_value_states = past_key_values.value_cache[self.layer_idx]\n\n            seqlens = sum(query_lens) + sum(key_values_lens)\n            merged_key_states = past_key_states.new_zeros(size=[seqlens, self.num_key_value_heads, self.head_dim])\n            merged_value_states = past_key_states.new_zeros(size=[seqlens, self.num_key_value_heads, self.head_dim])\n            merged_key_states[packed_query_indexes] = packed_key_states # q 是当前的， kv是之前的\n            merged_key_states[packed_key_value_indexes] = past_key_states\n            merged_value_states[packed_query_indexes] = packed_value_states\n            merged_value_states[packed_key_value_indexes] = past_value_states\n            key_values_lens = key_values_lens + query_lens\n        else:\n            merged_key_states = packed_key_states\n            merged_value_states = packed_value_states\n            key_values_lens = query_lens\n\n        cu_seqlens_q = torch.nn.functional.pad(torch.cumsum(query_lens, dim=0), (1, 0))\n        cu_seqlens_k = torch.nn.functional.pad(torch.cumsum(key_values_lens, dim=0), (1, 0))\n\n        # print(f"Q: {packed_query_states.shape}, K: {merged_key_states.shape}, V: {merged_value_states.shape}")\n        packed_attn_output = flash_attn_varlen_func(\n            q=packed_query_states,\n            k=merged_key_states,\n            v=merged_value_states,\n            cu_seqlens_q=cu_seqlens_q.to(torch.int32),\n            cu_seqlens_k=cu_seqlens_k.to(torch.int32),\n            max_seqlen_q=max(query_lens).item(),\n            max_seqlen_k=max(key_values_lens).item(),\n            causal=is_causal,\n        )\n        packed_attn_output = packed_attn_output.reshape(-1, self.hidden_size)\n        if mode == \'und\':\n            packed_attn_output = self.o_proj(packed_attn_output)\n        elif mode == \'gen\':\n            packed_attn_output[packed_text_indexes] = self.o_proj(packed_attn_output[packed_text_indexes])\n            packed_attn_output[packed_vae_token_indexes] = self.o_proj_moe_gen(packed_attn_output[packed_vae_token_indexes])\n\n        if update_past_key_values:\n            past_key_values.key_cache[self.layer_idx] = merged_key_states\n            past_key_values.value_cache[self.layer_idx] = merged_value_states\n\n        return packed_attn_output, past_key_values\n\n\nclass Qwen2DecoderLayer(nn.Module):\n    def __init__(self, config, layer_idx: Optional[int] = None):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n\n        self.self_attn = PackedAttention(config, layer_idx)\n\n        self.mlp = Qwen2MLP(config)\n        self.input_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask,\n        packed_position_embeddings: Tuple[torch.Tensor, torch.Tensor],\n    ) -> torch.Tensor:\n\n        residual = packed_sequence\n        packed_sequence = self.input_layernorm(packed_sequence)\n\n        # Self Attention\n        packed_sequence = self.self_attn(\n            packed_sequence=packed_sequence,\n            sample_lens=sample_lens,\n            attention_mask=attention_mask,\n            packed_position_embeddings=packed_position_embeddings,\n        )\n        packed_sequence = residual + packed_sequence\n\n        # Fully Connected\n        residual = packed_sequence\n        packed_sequence = self.post_attention_layernorm(packed_sequence)\n        packed_sequence = self.mlp(packed_sequence)\n        packed_sequence = residual + packed_sequence\n\n        return packed_sequence\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_embeddings: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n    ) -> BaseNavitOutputWithPast:\n\n        residual = packed_query_sequence\n        packed_query_sequence = self.input_layernorm(packed_query_sequence)\n\n        # Self Attention\n        packed_query_sequence, past_key_values = self.self_attn(\n            packed_query_sequence=packed_query_sequence,\n            query_lens=query_lens,\n            packed_query_position_embeddings=packed_query_position_embeddings,\n            packed_query_indexes=packed_query_indexes,\n            past_key_values=past_key_values,\n            key_values_lens=key_values_lens,\n            packed_key_value_indexes=packed_key_value_indexes,\n            update_past_key_values=update_past_key_values,\n            is_causal=is_causal,\n        )\n        packed_query_sequence = residual + packed_query_sequence\n\n        # Fully Connected\n        residual = packed_query_sequence\n        packed_query_sequence = self.post_attention_layernorm(packed_query_sequence)\n        packed_query_sequence = self.mlp(packed_query_sequence)\n        packed_query_sequence = residual + packed_query_sequence\n\n        return packed_query_sequence, past_key_values\n\n\nclass Qwen2MoTDecoderLayer(nn.Module):\n    def __init__(\n        self, \n        config, \n        layer_idx: Optional[int] = None, \n        attn_module: Optional[Qwen2Attention] = PackedAttentionMoT,\n    ):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n        self.freeze_und = config.freeze_und\n\n        self.self_attn = attn_module(config, layer_idx)\n\n        self.mlp = Qwen2MLP(config)\n        self.mlp_moe_gen = Qwen2MLP(config)\n        self.input_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.input_layernorm_moe_gen = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm_moe_gen = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask,\n        packed_position_embeddings: Tuple[torch.Tensor, torch.Tensor],\n        packed_und_token_indexes: torch.LongTensor,\n        packed_gen_token_indexes: torch.LongTensor,\n    ) -> torch.Tensor:\n\n        residual = packed_sequence\n        packed_sequence_ = packed_sequence.new_zeros(packed_sequence.shape)\n\n        # 理解和生成分开 要用不同的归一化\n        packed_sequence_[packed_und_token_indexes] = self.input_layernorm(packed_sequence[packed_und_token_indexes])\n        packed_sequence_[packed_gen_token_indexes] = self.input_layernorm_moe_gen(packed_sequence[packed_gen_token_indexes])\n\n        # Self Attention\n        packed_sequence_ = self.self_attn(\n            packed_sequence=packed_sequence_,\n            sample_lens=sample_lens,\n            attention_mask=attention_mask,\n            packed_position_embeddings=packed_position_embeddings,\n            packed_und_token_indexes=packed_und_token_indexes,\n            packed_gen_token_indexes=packed_gen_token_indexes,\n        )\n        if self.freeze_und:\n            packed_sequence_[packed_und_token_indexes] = packed_sequence_[packed_und_token_indexes].detach()\n        packed_sequence = residual + packed_sequence_\n\n        # Fully Connected\n        residual = packed_sequence\n        packed_sequence_ = packed_sequence.new_zeros(packed_sequence.shape)\n        packed_sequence_[packed_und_token_indexes] = self.mlp(\n            self.post_attention_layernorm(packed_sequence[packed_und_token_indexes])\n        )\n        if self.freeze_und:\n            packed_sequence_[packed_und_token_indexes] = packed_sequence_[packed_und_token_indexes].detach()\n    \n        packed_sequence_[packed_gen_token_indexes] = self.mlp_moe_gen(\n            self.post_attention_layernorm_moe_gen(packed_sequence[packed_gen_token_indexes])\n        )\n        packed_sequence = residual + packed_sequence_\n\n        return packed_sequence\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_embeddings: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n        mode="und",\n        packed_vae_token_indexes=None,\n        packed_text_indexes=None,\n    ) -> BaseNavitOutputWithPast:\n\n        residual = packed_query_sequence\n        if mode == "und":\n            packed_query_sequence = self.input_layernorm(packed_query_sequence)\n        elif mode == "gen":\n            packed_query_sequence_ = torch.zeros_like(packed_query_sequence)\n            packed_query_sequence_[packed_text_indexes] = self.input_layernorm(packed_query_sequence[packed_text_indexes])\n            packed_query_sequence_[packed_vae_token_indexes] = self.input_layernorm_moe_gen(packed_query_sequence[packed_vae_token_indexes])\n            packed_query_sequence = packed_query_sequence_\n\n        # Self Attention\n        packed_query_sequence, past_key_values = self.self_attn(\n            packed_query_sequence=packed_query_sequence,\n            query_lens=query_lens,\n            packed_query_position_embeddings=packed_query_position_embeddings,\n            packed_query_indexes=packed_query_indexes,\n            past_key_values=past_key_values,\n            key_values_lens=key_values_lens,\n            packed_key_value_indexes=packed_key_value_indexes,\n            update_past_key_values=update_past_key_values,\n            is_causal=is_causal,\n            mode=mode,\n            packed_vae_token_indexes=packed_vae_token_indexes,\n            packed_text_indexes=packed_text_indexes,\n        )\n        packed_query_sequence = residual + packed_query_sequence\n\n        # print(f"Q: {packed_query_sequence.shape}")\n\n        # Fully Connected\n        residual = packed_query_sequence\n        if mode == "und":\n            packed_query_sequence = self.post_attention_layernorm(packed_query_sequence)\n            packed_query_sequence = self.mlp(packed_query_sequence)\n        elif mode == "gen":\n            packed_text_query_sequence = packed_query_sequence[packed_text_indexes]\n            packed_vae_query_sequence = packed_query_sequence[packed_vae_token_indexes]\n            packed_text_query_sequence = self.post_attention_layernorm(packed_text_query_sequence).to(torch.bfloat16)\n            packed_vae_query_sequence = self.post_attention_layernorm_moe_gen(packed_vae_query_sequence).to(torch.bfloat16)\n\n            packed_query_sequence_ = torch.zeros_like(packed_query_sequence).to(torch.bfloat16)\n            packed_query_sequence_[packed_text_indexes] = self.mlp(packed_text_query_sequence)\n            packed_query_sequence_[packed_vae_token_indexes] = self.mlp_moe_gen(packed_vae_query_sequence)\n            packed_query_sequence = packed_query_sequence_\n\n        packed_query_sequence = residual + packed_query_sequence\n        return packed_query_sequence, past_key_values\n\n\nclass Qwen2MoEDecoderLayer(nn.Module):\n    def __init__(self, config, layer_idx: Optional[int] = None):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n\n        self.self_attn = PackedAttention(config, layer_idx)\n\n        self.mlp = Qwen2MLP(config)\n        self.mlp_moe_gen = Qwen2MLP(config)\n        self.input_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask,\n        packed_position_embeddings: Tuple[torch.Tensor, torch.Tensor],\n        packed_und_token_indexes: torch.LongTensor,\n        packed_gen_token_indexes: torch.LongTensor,\n    ) -> torch.Tensor:\n\n        residual = packed_sequence\n        packed_sequence = self.input_layernorm(packed_sequence)\n\n        # Self Attention\n        packed_sequence = self.self_attn(\n            packed_sequence=packed_sequence,\n            sample_lens=sample_lens,\n            attention_mask=attention_mask,\n            packed_position_embeddings=packed_position_embeddings,\n        )\n        packed_sequence = residual + packed_sequence\n\n        # Fully Connected\n        residual = packed_sequence\n        packed_sequence = self.post_attention_layernorm(packed_sequence)\n\n        packed_sequence_new = packed_sequence.new_zeros(packed_sequence.shape)\n        packed_sequence_und = self.mlp(packed_sequence[packed_und_token_indexes])\n        packed_sequence_gen = self.mlp_moe_gen(packed_sequence[packed_gen_token_indexes])\n        packed_sequence_new[packed_und_token_indexes] = packed_sequence_und\n        packed_sequence_new[packed_gen_token_indexes] = packed_sequence_gen\n\n        packed_sequence = residual + packed_sequence_new\n\n        return packed_sequence\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_embeddings: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n        mode="und",\n        packed_vae_token_indexes=None,\n        packed_text_indexes=None,\n    ) -> BaseNavitOutputWithPast:\n\n        residual = packed_query_sequence\n        packed_query_sequence = self.input_layernorm(packed_query_sequence)\n\n        # Self Attention\n        packed_query_sequence, past_key_values = self.self_attn(\n            packed_query_sequence=packed_query_sequence,\n            query_lens=query_lens,\n            packed_query_position_embeddings=packed_query_position_embeddings,\n            packed_query_indexes=packed_query_indexes,\n            past_key_values=past_key_values,\n            key_values_lens=key_values_lens,\n            packed_key_value_indexes=packed_key_value_indexes,\n            update_past_key_values=update_past_key_values,\n            is_causal=is_causal,\n        )\n        packed_query_sequence = residual + packed_query_sequence\n\n        # Fully Connected\n        residual = packed_query_sequence\n        packed_query_sequence = self.post_attention_layernorm(packed_query_sequence)\n        if mode == "und":\n            packed_query_sequence = self.mlp(packed_query_sequence)\n        elif mode == "gen":\n            packed_query_sequence_ = torch.zeros_like(packed_query_sequence).to(torch.bfloat16)\n            packed_query_sequence_[packed_text_indexes] = self.mlp(packed_query_sequence[packed_text_indexes])\n            packed_query_sequence_[packed_vae_token_indexes] = self.mlp_moe_gen(packed_query_sequence[packed_vae_token_indexes])\n            packed_query_sequence = packed_query_sequence_\n        packed_query_sequence = residual + packed_query_sequence\n\n        return packed_query_sequence, past_key_values\n\n\nDecoder_layer_dict = {\n    "Qwen2DecoderLayer": Qwen2DecoderLayer,\n    "Qwen2MoEDecoderLayer": Qwen2MoEDecoderLayer,\n    "Qwen2MoTDecoderLayer": partial(Qwen2MoTDecoderLayer, attn_module=PackedAttentionMoT),\n}\n\n\nclass Qwen2Model(Qwen2PreTrainedModel):\n    def __init__(self, config):\n        super().__init__(config)\n        self.padding_idx = config.pad_token_id\n        self.vocab_size = config.vocab_size\n        self.use_moe = \'Mo\' in config.layer_module\n\n        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)\n        layer_module = Decoder_layer_dict[config.layer_module]\n\n        self.layers = nn.ModuleList(\n            [layer_module(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]\n        )\n\n        self.norm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        if self.use_moe:\n            self.norm_moe_gen = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.rotary_emb = Qwen2RotaryEmbedding(config=config)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask,\n        packed_position_ids: torch.Tensor,\n        packed_und_token_indexes: Optional[torch.LongTensor] = None,\n        packed_gen_token_indexes: Optional[torch.LongTensor] = None,\n    ) -> torch.Tensor:\n\n        if self.config.freeze_und:\n            packed_sequence[packed_und_token_indexes] = packed_sequence[packed_und_token_indexes].detach()\n\n        # create position embeddings to be shared across the decoder layers\n        cos, sin = self.rotary_emb(packed_sequence, packed_position_ids.unsqueeze(0))\n        cos = cos.squeeze(0)\n        sin = sin.squeeze(0)\n        packed_position_embeddings = (cos, sin)\n\n        extra_inputs = {}\n        if self.use_moe:\n            assert packed_und_token_indexes is not None\n            if packed_gen_token_indexes is None:\n                packed_gen_token_indexes = packed_und_token_indexes.new_ones(size=[0])\n            extra_inputs.update(\n                packed_und_token_indexes=packed_und_token_indexes,\n                packed_gen_token_indexes=packed_gen_token_indexes,\n            )\n\n        for decoder_layer in self.layers:\n            packed_sequence = decoder_layer(\n                packed_sequence=packed_sequence,\n                sample_lens=sample_lens,\n                attention_mask=attention_mask,\n                packed_position_embeddings=packed_position_embeddings,\n                **extra_inputs\n            )\n\n        if self.use_moe:\n            packed_sequence_ = torch.zeros_like(packed_sequence)\n            packed_sequence_[packed_und_token_indexes] = self.norm(packed_sequence[packed_und_token_indexes])\n            if self.config.freeze_und:\n                packed_sequence_[packed_und_token_indexes] = packed_sequence_[packed_und_token_indexes].detach()\n            packed_sequence_[packed_gen_token_indexes] = self.norm_moe_gen(packed_sequence[packed_gen_token_indexes])\n            return packed_sequence_\n        else:\n            return self.norm(packed_sequence)\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_ids: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n        mode="und",\n        packed_vae_token_indexes=None,\n        packed_text_indexes=None,\n    ) -> BaseNavitOutputWithPast:\n\n        # create position embeddings to be shared across the decoder layers\n        cos, sin = self.rotary_emb(packed_query_sequence, packed_query_position_ids.unsqueeze(0))\n        cos = cos.squeeze(0)\n        sin = sin.squeeze(0)\n        packed_query_position_embeddings = (cos, sin)\n\n        extra_inputs = {}\n        if self.use_moe:\n            extra_inputs.update(mode=mode)\n            if mode == \'gen\':\n                assert packed_vae_token_indexes is not None\n                assert packed_text_indexes is not None\n                extra_inputs.update(\n                    packed_vae_token_indexes=packed_vae_token_indexes,\n                    packed_text_indexes=packed_text_indexes,\n                )\n\n        for decoder_layer in self.layers:\n            packed_query_sequence, past_key_values = decoder_layer(\n                packed_query_sequence=packed_query_sequence,\n                query_lens=query_lens,\n                packed_query_position_embeddings=packed_query_position_embeddings,\n                packed_query_indexes=packed_query_indexes,\n                past_key_values=past_key_values,\n                key_values_lens=key_values_lens,\n                packed_key_value_indexes=packed_key_value_indexes,\n                update_past_key_values=update_past_key_values,\n                is_causal=is_causal,\n                **extra_inputs,\n            )\n\n        if self.use_moe:\n            if mode == "und":\n                packed_query_sequence = self.norm(packed_query_sequence)\n            elif mode == "gen":\n                packed_query_sequence_ = torch.zeros_like(packed_query_sequence)\n                packed_query_sequence_[packed_text_indexes] = self.norm(packed_query_sequence[packed_text_indexes])\n                packed_query_sequence_[packed_vae_token_indexes] = self.norm_moe_gen(packed_query_sequence[packed_vae_token_indexes])\n                packed_query_sequence = packed_query_sequence_\n        else:\n            packed_query_sequence = self.norm(packed_query_sequence)\n\n        return BaseNavitOutputWithPast(\n            packed_query_sequence=packed_query_sequence,\n            past_key_values=past_key_values,\n        )\n\n\nclass Qwen2ForCausalLM(Qwen2PreTrainedModel):\n    _tied_weights_keys = ["lm_head.weight"]\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.model = Qwen2Model(config)\n        self.vocab_size = config.vocab_size\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def init_moe(self):\n        for name, param in self.named_parameters():\n            if "moe_gen" in name:\n                original_name = name.replace("_moe_gen", "")\n                param.data.copy_(self.state_dict()[original_name].data)\n\n    def get_input_embeddings(self):\n        return self.model.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.model.embed_tokens = value\n\n    def get_output_embeddings(self):\n        return self.lm_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.lm_head = new_embeddings\n\n    def set_decoder(self, decoder):\n        self.model = decoder\n\n    def get_decoder(self):\n        return self.model\n\n    def forward(self, *args, **kwargs):\n        if self.training:\n            return self.forward_train(*args, **kwargs)\n        else:\n            return self.forward_inference(*args, **kwargs)\n\n    def forward_train(\n        self,\n        packed_sequence: torch.Tensor,\n        sample_lens: List[int],\n        attention_mask,\n        packed_position_ids: torch.Tensor,\n        packed_und_token_indexes: Optional[torch.LongTensor] = None,\n        packed_gen_token_indexes: Optional[torch.LongTensor] = None,\n    ) -> torch.Tensor:\n\n        outputs = self.model(\n            packed_sequence=packed_sequence,\n            sample_lens=sample_lens,\n            packed_position_ids=packed_position_ids,\n            attention_mask=attention_mask,\n            packed_und_token_indexes=packed_und_token_indexes,\n            packed_gen_token_indexes=packed_gen_token_indexes,\n        )\n        return outputs\n\n    def forward_inference(\n        self,\n        packed_query_sequence: torch.Tensor,\n        query_lens: torch.Tensor,\n        packed_query_position_ids: torch.Tensor,\n        packed_query_indexes: torch.Tensor,\n        past_key_values: Optional[NaiveCache] = None,\n        key_values_lens: Optional[torch.Tensor] = None,\n        packed_key_value_indexes: Optional[torch.Tensor] = None,\n        update_past_key_values=True,\n        is_causal=True,\n        mode="und",\n        packed_vae_token_indexes=None,\n        packed_text_indexes=None,\n    ) -> BaseNavitOutputWithPast:\n\n        outputs = self.model(\n            packed_query_sequence=packed_query_sequence,\n            query_lens=query_lens,\n            packed_query_position_ids=packed_query_position_ids,\n            packed_query_indexes=packed_query_indexes,\n            past_key_values=past_key_values,\n            key_values_lens=key_values_lens,\n            packed_key_value_indexes=packed_key_value_indexes,\n            update_past_key_values=update_past_key_values,\n            is_causal=is_causal,\n            mode=mode,\n            packed_vae_token_indexes=packed_vae_token_indexes,\n            packed_text_indexes=packed_text_indexes,\n        )\n\n        return outputs\n', 'modeling/bagel/siglip_navit.py': '# Copyright (c) 2024 The HuggingFace Inc. team.\n# Copyright (c) 2025 Bytedance Ltd. and/or its affiliates.\n# SPDX-License-Identifier: Apache-2.0\n#\n# This file has been modified by ByteDance Ltd. and/or its affiliates. on 2025-05-20.\n#\n# Original file was released under Apache-2.0, with the full license text\n# available at https://github.com/huggingface/transformers/blob/main/LICENSE.\n#\n# This modified file is released under the same license.\n\nimport torch\nfrom torch import nn\n\nfrom transformers.activations import ACT2FN\nfrom modeling.siglip.configuration_siglip import SiglipVisionConfig as _SiglipVisionConfig\nfrom modeling.siglip.modeling_siglip import SiglipAttention, SiglipPreTrainedModel\nfrom flash_attn import flash_attn_varlen_func\n\n\n\nclass SiglipVisionConfig(_SiglipVisionConfig):\n    r"""\n    This is the configuration class to store the configuration of a [`SiglipVisionModel`]. It is used to instantiate a\n    Siglip vision encoder according to the specified arguments, defining the model architecture. Instantiating a\n    configuration with the defaults will yield a similar configuration to that of the vision encoder of the Siglip\n    [google/siglip-base-patch16-224](https://huggingface.co/google/siglip-base-patch16-224) architecture.\n\n    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the\n    documentation from [`PretrainedConfig`] for more information.\n\n    Args:\n        hidden_size (`int`, *optional*, defaults to 768):\n            Dimensionality of the encoder layers and the pooler layer.\n        intermediate_size (`int`, *optional*, defaults to 3072):\n            Dimensionality of the "intermediate" (i.e., feed-forward) layer in the Transformer encoder.\n        num_hidden_layers (`int`, *optional*, defaults to 12):\n            Number of hidden layers in the Transformer encoder.\n        num_attention_heads (`int`, *optional*, defaults to 12):\n            Number of attention heads for each attention layer in the Transformer encoder.\n        num_channels (`int`, *optional*, defaults to 3):\n            Number of channels in the input images.\n        image_size (`int`, *optional*, defaults to 224):\n            The size (resolution) of each image.\n        patch_size (`int`, *optional*, defaults to 16):\n            The size (resolution) of each patch.\n        hidden_act (`str` or `function`, *optional*, defaults to `"gelu_pytorch_tanh"`):\n            The non-linear activation function (function or string) in the encoder and pooler. If string, `"gelu"`,\n            `"relu"`, `"selu"` and `"gelu_new"` `"quick_gelu"` are supported.\n        layer_norm_eps (`float`, *optional*, defaults to 1e-06):\n            The epsilon used by the layer normalization layers.\n        attention_dropout (`float`, *optional*, defaults to 0.0):\n            The dropout ratio for the attention probabilities.\n\n    Example:\n\n    ```python\n    >>> from transformers import SiglipVisionConfig, SiglipVisionModel\n\n    >>> # Initializing a SiglipVisionConfig with google/siglip-base-patch16-224 style configuration\n    >>> configuration = SiglipVisionConfig()\n\n    >>> # Initializing a SiglipVisionModel (with random weights) from the google/siglip-base-patch16-224 style configuration\n    >>> model = SiglipVisionModel(configuration)\n\n    >>> # Accessing the model configuration\n    >>> configuration = model.config\n    ```"""\n\n    model_type = "siglip_vision_model"\n\n    def __init__(\n        self,\n        hidden_size=768,\n        intermediate_size=3072,\n        num_hidden_layers=12,\n        num_attention_heads=12,\n        num_channels=3,\n        image_size=224,\n        patch_size=16,\n        hidden_act="gelu_pytorch_tanh",\n        layer_norm_eps=1e-6,\n        attention_dropout=0.0,\n        rope=True,\n        **kwargs,\n    ):\n        super().__init__(\n            hidden_size=hidden_size,\n            intermediate_size=intermediate_size,\n            num_hidden_layers=num_hidden_layers,\n            num_attention_heads=num_attention_heads,\n            num_channels=num_channels,\n            image_size=image_size,\n            patch_size=patch_size,\n            hidden_act=hidden_act,\n            layer_norm_eps=layer_norm_eps,\n            attention_dropout=attention_dropout,\n            **kwargs)\n        \n        self.rope = rope\n\n\nclass RotaryEmbedding2D(torch.nn.Module):\n    def __init__(self, dim, max_h, max_w, base=10000):\n        super().__init__()\n        freq = torch.arange(0, dim, 2, dtype=torch.int64).float() / dim\n        inv_freq = 1.0 / (base ** freq)\n\n        grid_h = torch.arange(0, max_h)\n        grid_h = grid_h.to(inv_freq.dtype)\n        grid_h = grid_h[:, None].repeat(1, max_w)\n\n        grid_w = torch.arange(0, max_w)\n        grid_w = grid_w.to(inv_freq.dtype)\n        grid_w = grid_w[None, :].repeat(max_h, 1)\n\n        cos_h, sin_h = self._forward_one_side(grid_h, inv_freq)\n        cos_w, sin_w = self._forward_one_side(grid_w, inv_freq)\n\n        self.register_buffer("cos_h", cos_h)\n        self.register_buffer("sin_h", sin_h)\n        self.register_buffer("cos_w", cos_w)\n        self.register_buffer("sin_w", sin_w)\n\n    def _forward_one_side(self, grid, inv_freq):\n        freqs = grid[..., None] * inv_freq[None, None, :]\n        emb = torch.cat((freqs, freqs), dim=-1).flatten(0, 1)\n        return emb.cos(), emb.sin()\n\n\ndef rotate_half(x):\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat((-x2, x1), dim=-1)\n\n\ndef apply_rotary_pos_emb(q, k, cos, sin):\n    # unsqueeze due to the head dimension\n    cos = cos.unsqueeze(1)\n    sin = sin.unsqueeze(1)\n    q_embed = (q * cos) + (rotate_half(q) * sin)\n    k_embed = (k * cos) + (rotate_half(k) * sin)\n    return q_embed, k_embed\n\n\nclass SiglipVisionEmbeddings(nn.Module):\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n        self.config = config\n        self.embed_dim = config.hidden_size\n        self.image_size = config.image_size\n        self.patch_size = config.patch_size\n\n        self.patch_embedding = nn.Conv2d(\n            in_channels=config.num_channels,\n            out_channels=self.embed_dim,\n            kernel_size=self.patch_size,\n            stride=self.patch_size,\n            padding="valid",\n        )\n\n        self.num_patches_per_side = self.image_size // self.patch_size\n        self.num_patches = self.num_patches_per_side**2\n        self.num_positions = self.num_patches\n        if not config.rope:\n            self.position_embedding = nn.Embedding(self.num_positions, self.embed_dim)\n\n    def convert_conv2d_to_linear(self, config, meta=False):\n        if meta:\n            linear_patch_embedding = nn.Linear(\n                config.num_channels * self.patch_size ** 2, self.embed_dim, bias=True, device=\'meta\'\n            )\n        else:\n            linear_patch_embedding = nn.Linear(\n                config.num_channels * self.patch_size ** 2, self.embed_dim, bias=True\n            )\n        W = self.patch_embedding.weight.permute(0, 2, 3, 1).reshape(\n            self.embed_dim, config.num_channels * self.patch_size ** 2\n        )\n        linear_patch_embedding.weight.data = W\n        linear_patch_embedding.bias.data = self.patch_embedding.bias.data\n        del self.patch_embedding\n        self.patch_embedding = linear_patch_embedding\n\n    def forward(\n        self, \n        packed_pixel_values: torch.FloatTensor, \n        packed_flattened_position_ids: torch.LongTensor\n    ) -> torch.Tensor:\n\n        patch_embeds = self.patch_embedding(packed_pixel_values)\n        if not self.config.rope:\n            embeddings = patch_embeds + self.position_embedding(packed_flattened_position_ids)\n        else:\n            embeddings = patch_embeds\n        return embeddings\n\n\nclass SiglipFlashAttention2(SiglipAttention):\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        cu_seqlens: torch.IntTensor,\n        max_seqlen: int,\n        cos_h: torch.Tensor = None,\n        sin_h: torch.Tensor = None,\n        cos_w: torch.Tensor = None,\n        sin_w: torch.Tensor = None,\n        **kwargs,\n    ) -> torch.Tensor:\n\n        total_q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(total_q_len, self.num_heads, self.head_dim)\n        key_states = key_states.view(total_q_len, self.num_heads, self.head_dim)\n        value_states = value_states.view(total_q_len, self.num_heads, self.head_dim)\n\n        if self.config.rope:\n            qh, qw = query_states[:, :, :self.head_dim // 2], query_states[:, :, self.head_dim // 2:] \n            kh, kw = key_states[:, :, :self.head_dim // 2], key_states[:, :, self.head_dim // 2:]\n            qh, kh = apply_rotary_pos_emb(qh, kh, cos_h, sin_h)\n            qw, kw = apply_rotary_pos_emb(qw, kw, cos_w, sin_w)\n            query_states = torch.cat([qh, qw], dim=-1)\n            key_states = torch.cat([kh, kw], dim=-1)\n\n        attn_output = flash_attn_varlen_func(\n            query_states.to(torch.bfloat16),\n            key_states.to(torch.bfloat16),\n            value_states.to(torch.bfloat16),\n            cu_seqlens_q=cu_seqlens,\n            cu_seqlens_k=cu_seqlens,\n            max_seqlen_q=max_seqlen,\n            max_seqlen_k=max_seqlen,\n            causal=False,\n        )\n\n        attn_output = self.out_proj(attn_output.reshape(total_q_len, -1))\n        return attn_output\n\n\nclass SiglipMLP(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        self.activation_fn = ACT2FN[config.hidden_act]\n        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)\n        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        hidden_states = self.fc1(hidden_states)\n        hidden_states = self.activation_fn(hidden_states)\n        hidden_states = self.fc2(hidden_states)\n        return hidden_states\n\n\nclass SiglipEncoderLayer(nn.Module):\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n        self.embed_dim = config.hidden_size\n        self.self_attn = SiglipFlashAttention2(config)\n        self.layer_norm1 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)\n        self.mlp = SiglipMLP(config)\n        self.layer_norm2 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        cu_seqlens: torch.IntTensor,\n        max_seqlen: int,\n        cos_h: torch.Tensor = None,\n        sin_h: torch.Tensor = None,\n        cos_w: torch.Tensor = None,\n        sin_w: torch.Tensor = None\n    ) -> torch.Tensor:\n        residual = hidden_states\n\n        hidden_states = self.layer_norm1(hidden_states)\n        hidden_states = self.self_attn(\n            hidden_states=hidden_states,\n            cu_seqlens=cu_seqlens,\n            max_seqlen=max_seqlen,\n            cos_h=cos_h,\n            sin_h=sin_h,\n            cos_w=cos_w,\n            sin_w=sin_w\n        )\n        hidden_states = residual + hidden_states\n\n        residual = hidden_states\n        hidden_states = self.layer_norm2(hidden_states)\n        hidden_states = self.mlp(hidden_states)\n        hidden_states = residual + hidden_states\n\n        return hidden_states\n\n\nclass SiglipEncoder(nn.Module):\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n        self.config = config\n        self.layers = nn.ModuleList(\n            [SiglipEncoderLayer(config) for _ in range(config.num_hidden_layers)]\n        )\n\n    def forward(\n        self,\n        inputs_embeds: torch.Tensor,\n        cu_seqlens: torch.IntTensor,\n        max_seqlen: int,\n        cos_h: torch.Tensor = None,\n        sin_h: torch.Tensor = None,\n        cos_w: torch.Tensor = None,\n        sin_w: torch.Tensor = None,\n    ) -> torch.Tensor:\n\n        hidden_states = inputs_embeds\n        for encoder_layer in self.layers:\n            hidden_states = encoder_layer(hidden_states, cu_seqlens, max_seqlen,\n                                          cos_h=cos_h, sin_h=sin_h, cos_w=cos_w, sin_w=sin_w)\n\n        return hidden_states\n\n\nclass SiglipVisionTransformer(nn.Module):\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n        self.config = config\n        embed_dim = config.hidden_size\n\n        self.embeddings = SiglipVisionEmbeddings(config)\n        if config.rope:\n            max_size = config.image_size // config.patch_size\n            dim_head = config.hidden_size // config.num_attention_heads\n            self.rope = RotaryEmbedding2D(dim_head // 2, max_size, max_size)\n\n        self.encoder = SiglipEncoder(config)\n        self.post_layernorm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)\n\n    def forward(\n        self,\n        packed_pixel_values: torch.Tensor,\n        packed_flattened_position_ids: torch.LongTensor,\n        cu_seqlens: torch.IntTensor,\n        max_seqlen: int,\n    ) -> torch.Tensor:\n        hidden_states = self.embeddings(\n            packed_pixel_values=packed_pixel_values, \n            packed_flattened_position_ids=packed_flattened_position_ids\n        )\n\n        extra_inputs = {}\n        if self.config.rope:\n            extra_inputs.update(\n                cos_h = self.rope.cos_h[packed_flattened_position_ids],\n                sin_h = self.rope.sin_h[packed_flattened_position_ids],\n                cos_w = self.rope.cos_w[packed_flattened_position_ids],\n                sin_w = self.rope.sin_w[packed_flattened_position_ids]\n            )\n\n        last_hidden_state = self.encoder(\n            inputs_embeds=hidden_states, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen, \n            **extra_inputs\n        )\n        last_hidden_state = self.post_layernorm(last_hidden_state)\n        return last_hidden_state\n\n\nclass SiglipVisionModel(SiglipPreTrainedModel):\n    config_class = SiglipVisionConfig\n    main_input_name = "packed_pixel_values"\n\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__(config)\n\n        self.vision_model = SiglipVisionTransformer(config)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self) -> nn.Module:\n        return self.vision_model.embeddings.patch_embedding\n\n    def forward(\n        self,\n        packed_pixel_values: torch.Tensor,\n        packed_flattened_position_ids: torch.LongTensor,\n        cu_seqlens: torch.IntTensor,\n        max_seqlen: int,\n    ) -> torch.Tensor:\n\n        return self.vision_model(\n            packed_pixel_values=packed_pixel_values,\n            packed_flattened_position_ids=packed_flattened_position_ids,\n            cu_seqlens=cu_seqlens,\n            max_seqlen=max_seqlen,\n        )\n', 'modeling/infer.py': 'import os\nimport re\nimport json\nimport glob\nfrom tqdm import tqdm\nfrom fontTools.ttLib import TTFont\nfrom PIL import Image, ImageDraw, ImageFont\n\nclass FusionCharAnnotator:\n    def __init__(self, \n                 opacity=128, \n                 mask_color=(255, 0, 0),  # mask颜色 RGB\n                 mask_shape=\'square\'):  # \'circle\' 或 \'square\'\n        """\n        Args:\n            opacity: mask透明度 (0-255, 0完全透明，255完全不透明)\n            mask_color: mask颜色，RGB元组，如 (255, 0, 0) 为红色\n            mask_shape: mask形状，\'circle\' 或 \'square\'\n        """\n        self.opacity = opacity\n        self.mask_color = mask_color\n        self.mask_shape = mask_shape\n\n    def parse_ref_annotations(self, output_text):\n        annotation_pattern = r\'<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),(.)<\\|box_end\\|>\'\n        matches = re.findall(annotation_pattern, output_text)\n        \n        annotations = []\n        for match in matches:\n            x1, y1, x2, y2 = int(match[0]), int(match[1]), int(match[2]), int(match[3])\n            annotations.append((x1, y1, x2, y2, \'\'))\n        \n        return annotations\n    \n    def create_comparison_image(self, original_img, annotations):\n\n        image = original_img.copy()\n        image = image.convert(\'RGBA\')\n        width, height = image.size\n        \n        overlay = Image.new(\'RGBA\', (width, height), (0, 0, 0, 0))\n        draw = ImageDraw.Draw(overlay)\n        \n        for i, (x1, y1, x2, y2, char) in enumerate(annotations):\n            actual_x1 = int(x1 * width / 1000)\n            actual_y1 = int(y1 * height / 1000)\n            actual_x2 = int(x2 * width / 1000)\n            actual_y2 = int(y2 * height / 1000)\n\n            actual_x1 -= 2\n            actual_y1 -= 2\n            actual_x2 += 2\n            actual_y2 += 2\n\n            mask_color_with_alpha = (*self.mask_color, self.opacity)\n            outline_color = (255, 255, 255, self.opacity)\n            \n            if self.mask_shape == \'circle\':\n                draw.ellipse([\n                    actual_x1, actual_y1,\n                    actual_x2, actual_y2\n                ], fill=mask_color_with_alpha, outline=outline_color, width=2)\n            elif self.mask_shape == \'square\':\n                draw.rectangle([\n                    actual_x1, actual_y1,\n                    actual_x2, actual_y2\n                ], fill=mask_color_with_alpha, outline=outline_color, width=2)\n\n        result = Image.alpha_composite(image, overlay)\n        \n        return result.convert(\'RGB\')\n\nclass AnnotationRenderer:\n    def __init__(self, \n                 opacity=128, \n                 font_size=20,\n                 mask_color=(255, 0, 0),  # mask颜色 RGB\n                 text_color=(255, 255, 255),  # 文字颜色 RGB\n                 mask_shape=\'square\',  # \'circle\' 或 \'square\'\n                 text_bold=False,  # 文字是否加粗\n                 bold_strength=1,  # 加粗强度 (1-3)\n                 text_stroke_width=0,  # 文字描边宽度\n                 text_stroke_color=(0, 0, 0)):  # 文字描边颜色\n        """\n        初始化渲染器\n        Args:\n            opacity: 透明度 (0-255, 0完全透明，255完全不透明)\n            font_size: 字体大小\n            mask_color: mask颜色，RGB元组，如 (255, 0, 0) 为红色\n            text_color: 文字颜色，RGB元组\n            mask_shape: mask形状，\'circle\' 或 \'square\'\n            text_bold: 文字是否加粗\n            bold_strength: 加粗强度，1-3（1最轻，3最重）\n            text_stroke_width: 文字描边宽度\n            text_stroke_color: 文字描边颜色\n        """\n        self.opacity = opacity\n        self.font_size = font_size\n        self.mask_color = mask_color\n        self.text_color = text_color\n        self.mask_shape = mask_shape\n        self.text_bold = text_bold\n        self.bold_strength = max(1, min(3, bold_strength))\n        self.text_stroke_width = text_stroke_width\n        self.text_stroke_color = text_stroke_color\n        \n        font_list = [\n            "./font/KaiXinSongA.ttf",\n            "./font/KaiXinSongB.ttf",\n            "./font/TH-Tshyn-P0.ttf", \n            "./font/TH-Tshyn-P1.ttf", \n            "./font/TH-Tshyn-P16.ttf", \n            "./font/TH-Tshyn-P2.ttf", \n        ]\n        \n        self.ImageFont_fonts = []\n        self.TTFont_fonts = []\n        self.font_char_cache = {}\n        \n        try:\n            for font_path in font_list:\n                if os.path.exists(font_path):\n                    try:\n                        pil_font = ImageFont.truetype(font_path, self.font_size)\n                        self.ImageFont_fonts.append(pil_font)\n                        tt_font = TTFont(font_path)\n                        self.TTFont_fonts.append(tt_font)\n                        print(f"成功加载字体: {os.path.basename(font_path)}")\n                    except Exception as e:\n                        print(f"加载字体失败 {font_path}: {e}")\n            \n            if not self.ImageFont_fonts:\n                default_font = ImageFont.load_default()\n                self.ImageFont_fonts.append(default_font)\n                self.TTFont_fonts.append(None)\n                print("警告: 未找到任何字体文件，使用默认字体")\n            else:\n                print(f"总共加载了 {len(self.ImageFont_fonts)} 个字体文件")\n                \n        except Exception as e:\n            print(f"字体初始化失败: {e}")\n            default_font = ImageFont.load_default()\n            self.ImageFont_fonts.append(default_font)\n            self.TTFont_fonts.append(None)\n    \n    def is_char_in_font(self, TTFont_font, char):\n        if TTFont_font is None:  # 默认字体的情况\n            return True\n        try:\n            cmap = TTFont_font[\'cmap\']\n            for subtable in cmap.tables:\n                if ord(char) in subtable.cmap:\n                    return True\n            return False\n        except Exception as e:\n            print(f"检查字符 \'{char}\' 时出错: {e}")\n            return False\n\n    def find_font_for_char(self, char):\n        if char in self.font_char_cache:\n            return self.font_char_cache[char]\n        \n        for i, (ImageFont_font, TTFont_font) in enumerate(zip(self.ImageFont_fonts, self.TTFont_fonts)):\n            if self.is_char_in_font(TTFont_font, char):\n                self.font_char_cache[char] = ImageFont_font\n                return ImageFont_font\n            if i == len(self.ImageFont_fonts) - 1:\n                print(f"No font can render the character {char}")\n                fallback_font = self.ImageFont_fonts[0]\n                self.font_char_cache[char] = fallback_font\n                return fallback_font\n        \n        fallback_font = self.ImageFont_fonts[0]\n        self.font_char_cache[char] = fallback_font\n        return fallback_font\n\n    def adjust_overlapping_boxes(self, annotations):\n        if len(annotations) <= 1:\n            return annotations\n        \n        def calculate_iou(box1, box2):\n            x1_1, y1_1, x2_1, y2_1 = box1[:4]\n            x1_2, y1_2, x2_2, y2_2 = box2[:4]\n            \n            x_left = max(x1_1, x1_2)\n            y_top = max(y1_1, y1_2)\n            x_right = min(x2_1, x2_2)\n            y_bottom = min(y2_1, y2_2)\n            \n            if x_right < x_left or y_bottom < y_top:\n                return 0.0\n            \n            intersection = (x_right - x_left) * (y_bottom - y_top)\n            \n            area1 = (x2_1 - x1_1) * (y2_1 - y1_1)\n            area2 = (x2_2 - x1_2) * (y2_2 - y1_2)\n            union = area1 + area2 - intersection\n            \n            return intersection / union if union > 0 else 0\n        \n        sorted_annotations = sorted(annotations, key=lambda x: (x[1], x[0]))\n        adjusted = list(sorted_annotations)\n        \n        for i in range(len(adjusted)):\n            for j in range(i + 1, len(adjusted)):\n                box_i = adjusted[i]\n                box_j = adjusted[j]\n                \n                iou = calculate_iou(box_i, box_j)\n                \n                if iou > 0:\n                    x1_i, y1_i, x2_i, y2_i, char_i = box_i\n                    x1_j, y1_j, x2_j, y2_j, char_j = box_j\n\n                    overlap_left = max(x1_i, x1_j)\n                    overlap_right = min(x2_i, x2_j)\n                    overlap_top = max(y1_i, y1_j)\n                    overlap_bottom = min(y2_i, y2_j)\n                    \n                    overlap_width = overlap_right - overlap_left\n                    overlap_height = overlap_bottom - overlap_top\n                    \n                    if overlap_width < overlap_height:\n                        if (x1_i + x2_i) / 2 < (x1_j + x2_j) / 2:\n                            mid = (overlap_left + overlap_right) // 2\n                            adjusted[i] = (x1_i, y1_i, mid, y2_i, char_i)\n                            adjusted[j] = (mid, y1_j, x2_j, y2_j, char_j)\n                        else:\n                            mid = (overlap_left + overlap_right) // 2\n                            adjusted[j] = (x1_j, y1_j, mid, y2_j, char_j)\n                            adjusted[i] = (mid, y1_i, x2_i, y2_i, char_i)\n                    else:\n                        if (y1_i + y2_i) / 2 < (y1_j + y2_j) / 2:\n                            mid = (overlap_top + overlap_bottom) // 2\n                            adjusted[i] = (x1_i, y1_i, x2_i, mid, char_i)\n                            adjusted[j] = (x1_j, mid, x2_j, y2_j, char_j)\n                        else:\n                            mid = (overlap_top + overlap_bottom) // 2\n                            adjusted[j] = (x1_j, y1_j, x2_j, mid, char_j)\n                            adjusted[i] = (x1_i, mid, x2_i, y2_i, char_i)\n        \n        return adjusted\n\n    def parse_stepref_annotations(self, output_text, image_size):\n        width, height = image_size\n        \n        annotation_pattern = r\'<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),(.)<\\|box_end\\|>\'\n        matches = re.findall(annotation_pattern, output_text)\n        \n        annotations = []\n        for match in matches:\n            x1, y1, x2, y2, char = int(match[0]), int(match[1]), int(match[2]), int(match[3]), match[4]\n            \n            actual_x1 = int(x1 * width / 1000)\n            actual_y1 = int(y1 * height / 1000)\n            actual_x2 = int(x2 * width / 1000)\n            actual_y2 = int(y2 * height / 1000)\n            \n            actual_x1 -= 2\n            actual_y1 -= 2\n            actual_x2 += 2\n            actual_y2 += 2\n            \n            actual_x1 = max(0, actual_x1)\n            actual_y1 = max(0, actual_y1)\n            actual_x2 = min(width, actual_x2)\n            actual_y2 = min(height, actual_y2)\n            \n            annotations.append((actual_x1, actual_y1, actual_x2, actual_y2, char))\n        annotations = self.adjust_overlapping_boxes(annotations.copy())\n        \n        return annotations\n    \n    \n    def render_char_with_size(self, char, width, height, font_base):\n\n        try:\n            if width <= 0 or height <= 0:\n                print(f"无效的框尺寸: width={width}, height={height}")\n                return font_base\n            \n            initial_size = max(min(width, height) * 0.1, 10)\n            font_size = int(initial_size)\n            \n            if font_size <= 0:\n                font_size = 12 \n            \n            font = ImageFont.truetype(font_base.path, font_size) if hasattr(font_base, \'path\') else font_base\n\n            temp_img = Image.new(\'RGBA\', (width*2, height*2), (255, 255, 255, 0))\n            temp_draw = ImageDraw.Draw(temp_img)\n            \n            bbox = temp_draw.textbbox((0, 0), char, font=font)\n            text_width = bbox[2] - bbox[0]\n            text_height = bbox[3] - bbox[1]\n            \n            if text_width > 0 and text_height > 0:\n                scale_w = width * 0.95 / text_width\n                scale_h = height * 0.95 / text_height\n                scale = min(scale_w, scale_h)\n                \n                adjusted_font_size = max(8, int(font_size * scale))\n                if adjusted_font_size <= 0:\n                    adjusted_font_size = 8\n                \n                font = ImageFont.truetype(font_base.path, adjusted_font_size) if hasattr(font_base, \'path\') else font_base\n            \n            return font\n            \n        except Exception as e:\n            print(f"调整字体大小失败: {e}, 使用默认字体")\n            return font_base\n    \n    def draw_text_with_effects(self, draw, position, text, font, fill_color, stroke_width, stroke_color):\n        x, y = position\n        \n        if stroke_width > 0:\n            for dx in range(-stroke_width, stroke_width + 1):\n                for dy in range(-stroke_width, stroke_width + 1):\n                    if dx != 0 or dy != 0:\n                        draw.text((x + dx, y + dy), text, font=font, fill=stroke_color)\n\n        if self.text_bold:\n            offsets = []\n            \n            if self.bold_strength == 1:\n                offsets = [(1, 0)]\n            elif self.bold_strength == 2:\n                offsets = [(1, 0), (0, 1)]\n            else:\n                offsets = [(1, 0), (-1, 0), (0, 1), (0, -1)]\n\n            for dx, dy in offsets:\n                draw.text((x + dx, y + dy), text, font=font, fill=fill_color)\n\n        draw.text(position, text, font=font, fill=fill_color)\n    \n    def create_comparison_image(self, original_img, annotations):\n\n        image = original_img.copy()\n        image = image.convert(\'RGBA\')\n        img_width, img_height = image.size\n        \n        overlay = Image.new(\'RGBA\', (img_width, img_height), (0, 0, 0, 0))\n        draw = ImageDraw.Draw(overlay)\n        \n        for i, (x1, y1, x2, y2, char) in enumerate(annotations):\n\n            box_width = x2 - x1\n            box_height = y2 - y1\n            \n            if box_width <= 0 or box_height <= 0:\n                print(f"  跳过无效边界框: {x1},{y1},{x2},{y2}")\n                continue\n            \n            mask_color_with_alpha = (*self.mask_color, self.opacity)\n            outline_color = (255, 255, 255, self.opacity)\n            \n            if self.mask_shape == \'circle\':\n                center_x = (x1 + x2) // 2\n                center_y = (y1 + y2) // 2\n                radius = min(box_width, box_height) // 2\n                draw.ellipse([\n                    center_x - radius, center_y - radius,\n                    center_x + radius, center_y + radius\n                ], fill=mask_color_with_alpha, outline=outline_color, width=2)\n            elif self.mask_shape == \'square\':\n                draw.rectangle([x1, y1, x2, y2], \n                             fill=mask_color_with_alpha, outline=outline_color, width=2)\n            \n            base_font = self.find_font_for_char(char)\n            adjusted_font = self.render_char_with_size(char, box_width, box_height, base_font)\n            \n            try:\n                bbox = draw.textbbox((0, 0), char, font=adjusted_font)\n                text_width = bbox[2] - bbox[0]\n                text_height = bbox[3] - bbox[1]\n            except:\n                try:\n                    text_width, text_height = draw.textsize(char, font=adjusted_font)\n                except:\n                    text_width, text_height = box_width // 2, box_height // 2\n            \n            text_x = x1 + (box_width - text_width) // 2\n            text_y = y1 + (box_height - text_height) // 2\n            \n            text_alpha = 200\n            text_color_with_alpha = (*self.text_color, text_alpha)\n            stroke_color_with_alpha = (*self.text_stroke_color, text_alpha)\n            \n            self.draw_text_with_effects(\n                draw, (text_x, text_y), char, adjusted_font,\n                text_color_with_alpha, self.text_stroke_width, stroke_color_with_alpha\n            )\n        \n        result = Image.alpha_composite(image, overlay)\n\n        return result.convert(\'RGB\')\n', 'modeling/qwen2/__init__.py': '# Copyright 2024 The Qwen Team and The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\nfrom typing import TYPE_CHECKING\n\nfrom transformers.utils import (\n    OptionalDependencyNotAvailable,\n    _LazyModule,\n    is_tokenizers_available,\n    is_torch_available,\n)\n\n\n_import_structure = {\n    "configuration_qwen2": ["Qwen2Config"],\n    "tokenization_qwen2": ["Qwen2Tokenizer"],\n}\n\ntry:\n    if not is_tokenizers_available():\n        raise OptionalDependencyNotAvailable()\nexcept OptionalDependencyNotAvailable:\n    pass\nelse:\n    _import_structure["tokenization_qwen2_fast"] = ["Qwen2TokenizerFast"]\n\ntry:\n    if not is_torch_available():\n        raise OptionalDependencyNotAvailable()\nexcept OptionalDependencyNotAvailable:\n    pass\nelse:\n    _import_structure["modeling_qwen2"] = [\n        "Qwen2ForCausalLM",\n        "Qwen2Model",\n        "Qwen2PreTrainedModel",\n    ]\n\n\nif TYPE_CHECKING:\n    from .configuration_qwen2 import Qwen2Config\n    from .tokenization_qwen2 import Qwen2Tokenizer\n\n    try:\n        if not is_tokenizers_available():\n            raise OptionalDependencyNotAvailable()\n    except OptionalDependencyNotAvailable:\n        pass\n    else:\n        from .tokenization_qwen2_fast import Qwen2TokenizerFast\n\n    try:\n        if not is_torch_available():\n            raise OptionalDependencyNotAvailable()\n    except OptionalDependencyNotAvailable:\n        pass\n    else:\n        from .modeling_qwen2 import (\n            Qwen2ForCausalLM,\n            Qwen2Model,\n            Qwen2PreTrainedModel,\n        )\n\n\nelse:\n    import sys\n\n    sys.modules[__name__] = _LazyModule(__name__, globals()["__file__"], _import_structure, module_spec=__spec__)\n', 'modeling/qwen2/configuration_qwen2.py': '# Copyright 2024 The Qwen Team and The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Qwen2 model configuration"""\n\nfrom transformers.configuration_utils import PretrainedConfig\nfrom transformers.modeling_rope_utils import rope_config_validation\nfrom transformers.utils import logging\n\n\nlogger = logging.get_logger(__name__)\n\n\nclass Qwen2Config(PretrainedConfig):\n    r"""\n    This is the configuration class to store the configuration of a [`Qwen2Model`]. It is used to instantiate a\n    Qwen2 model according to the specified arguments, defining the model architecture. Instantiating a configuration\n    with the defaults will yield a similar configuration to that of\n    Qwen2-7B-beta [Qwen/Qwen2-7B-beta](https://huggingface.co/Qwen/Qwen2-7B-beta).\n\n    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the\n    documentation from [`PretrainedConfig`] for more information.\n\n\n    Args:\n        vocab_size (`int`, *optional*, defaults to 151936):\n            Vocabulary size of the Qwen2 model. Defines the number of different tokens that can be represented by the\n            `inputs_ids` passed when calling [`Qwen2Model`]\n        hidden_size (`int`, *optional*, defaults to 4096):\n            Dimension of the hidden representations.\n        intermediate_size (`int`, *optional*, defaults to 22016):\n            Dimension of the MLP representations.\n        num_hidden_layers (`int`, *optional*, defaults to 32):\n            Number of hidden layers in the Transformer encoder.\n        num_attention_heads (`int`, *optional*, defaults to 32):\n            Number of attention heads for each attention layer in the Transformer encoder.\n        num_key_value_heads (`int`, *optional*, defaults to 32):\n            This is the number of key_value heads that should be used to implement Grouped Query Attention. If\n            `num_key_value_heads=num_attention_heads`, the model will use Multi Head Attention (MHA), if\n            `num_key_value_heads=1` the model will use Multi Query Attention (MQA) otherwise GQA is used. When\n            converting a multi-head checkpoint to a GQA checkpoint, each group key and value head should be constructed\n            by meanpooling all the original heads within that group. For more details checkout [this\n            paper](https://arxiv.org/pdf/2305.13245.pdf). If it is not specified, will default to `32`.\n        hidden_act (`str` or `function`, *optional*, defaults to `"silu"`):\n            The non-linear activation function (function or string) in the decoder.\n        max_position_embeddings (`int`, *optional*, defaults to 32768):\n            The maximum sequence length that this model might ever be used with.\n        initializer_range (`float`, *optional*, defaults to 0.02):\n            The standard deviation of the truncated_normal_initializer for initializing all weight matrices.\n        rms_norm_eps (`float`, *optional*, defaults to 1e-06):\n            The epsilon used by the rms normalization layers.\n        use_cache (`bool`, *optional*, defaults to `True`):\n            Whether or not the model should return the last key/values attentions (not used by all models). Only\n            relevant if `config.is_decoder=True`.\n        tie_word_embeddings (`bool`, *optional*, defaults to `False`):\n            Whether the model\'s input and output word embeddings should be tied.\n        rope_theta (`float`, *optional*, defaults to 10000.0):\n            The base period of the RoPE embeddings.\n        rope_scaling (`Dict`, *optional*):\n            Dictionary containing the scaling configuration for the RoPE embeddings. NOTE: if you apply new rope type\n            and you expect the model to work on longer `max_position_embeddings`, we recommend you to update this value\n            accordingly.\n            Expected contents:\n                `rope_type` (`str`):\n                    The sub-variant of RoPE to use. Can be one of [\'default\', \'linear\', \'dynamic\', \'yarn\', \'longrope\',\n                    \'llama3\'], with \'default\' being the original RoPE implementation.\n                `factor` (`float`, *optional*):\n                    Used with all rope types except \'default\'. The scaling factor to apply to the RoPE embeddings. In\n                    most scaling types, a `factor` of x will enable the model to handle sequences of length x *\n                    original maximum pre-trained length.\n                `original_max_position_embeddings` (`int`, *optional*):\n                    Used with \'dynamic\', \'longrope\' and \'llama3\'. The original max position embeddings used during\n                    pretraining.\n                `attention_factor` (`float`, *optional*):\n                    Used with \'yarn\' and \'longrope\'. The scaling factor to be applied on the attention\n                    computation. If unspecified, it defaults to value recommended by the implementation, using the\n                    `factor` field to infer the suggested value.\n                `beta_fast` (`float`, *optional*):\n                    Only used with \'yarn\'. Parameter to set the boundary for extrapolation (only) in the linear\n                    ramp function. If unspecified, it defaults to 32.\n                `beta_slow` (`float`, *optional*):\n                    Only used with \'yarn\'. Parameter to set the boundary for interpolation (only) in the linear\n                    ramp function. If unspecified, it defaults to 1.\n                `short_factor` (`List[float]`, *optional*):\n                    Only used with \'longrope\'. The scaling factor to be applied to short contexts (<\n                    `original_max_position_embeddings`). Must be a list of numbers with the same length as the hidden\n                    size divided by the number of attention heads divided by 2\n                `long_factor` (`List[float]`, *optional*):\n                    Only used with \'longrope\'. The scaling factor to be applied to long contexts (<\n                    `original_max_position_embeddings`). Must be a list of numbers with the same length as the hidden\n                    size divided by the number of attention heads divided by 2\n                `low_freq_factor` (`float`, *optional*):\n                    Only used with \'llama3\'. Scaling factor applied to low frequency components of the RoPE\n                `high_freq_factor` (`float`, *optional*):\n                    Only used with \'llama3\'. Scaling factor applied to high frequency components of the RoPE\n        use_sliding_window (`bool`, *optional*, defaults to `False`):\n            Whether to use sliding window attention.\n        sliding_window (`int`, *optional*, defaults to 4096):\n            Sliding window attention (SWA) window size. If not specified, will default to `4096`.\n        max_window_layers (`int`, *optional*, defaults to 28):\n            The number of layers that use SWA (Sliding Window Attention). The bottom layers use SWA while the top use full attention.\n        attention_dropout (`float`, *optional*, defaults to 0.0):\n            The dropout ratio for the attention probabilities.\n\n    ```python\n    >>> from transformers import Qwen2Model, Qwen2Config\n\n    >>> # Initializing a Qwen2 style configuration\n    >>> configuration = Qwen2Config()\n\n    >>> # Initializing a model from the Qwen2-7B style configuration\n    >>> model = Qwen2Model(configuration)\n\n    >>> # Accessing the model configuration\n    >>> configuration = model.config\n    ```"""\n\n    model_type = "qwen2"\n    keys_to_ignore_at_inference = ["past_key_values"]\n\n    def __init__(\n        self,\n        vocab_size=151936,\n        hidden_size=4096,\n        intermediate_size=22016,\n        num_hidden_layers=32,\n        num_attention_heads=32,\n        num_key_value_heads=32,\n        hidden_act="silu",\n        max_position_embeddings=32768,\n        initializer_range=0.02,\n        rms_norm_eps=1e-6,\n        use_cache=True,\n        tie_word_embeddings=False,\n        rope_theta=10000.0,\n        rope_scaling=None,\n        use_sliding_window=False,\n        sliding_window=4096,\n        max_window_layers=28,\n        attention_dropout=0.0,\n        is_causal=True,\n        _attn_implementation="flash_attention_2",\n        **kwargs,\n    ):\n        self.vocab_size = vocab_size\n        self.max_position_embeddings = max_position_embeddings\n        self.hidden_size = hidden_size\n        self.intermediate_size = intermediate_size\n        self.num_hidden_layers = num_hidden_layers\n        self.num_attention_heads = num_attention_heads\n        self.use_sliding_window = use_sliding_window\n        self.sliding_window = sliding_window if use_sliding_window else None\n        self.max_window_layers = max_window_layers\n\n        # for backward compatibility\n        if num_key_value_heads is None:\n            num_key_value_heads = num_attention_heads\n\n        self.num_key_value_heads = num_key_value_heads\n        self.hidden_act = hidden_act\n        self.initializer_range = initializer_range\n        self.rms_norm_eps = rms_norm_eps\n        self.use_cache = use_cache\n        self.rope_theta = rope_theta\n        self.rope_scaling = rope_scaling\n        self.attention_dropout = attention_dropout\n        self.is_causal = is_causal\n        self._attn_implementation = _attn_implementation\n\n        # Validate the correctness of rotary position embeddings parameters\n        # BC: if there is a \'type\' field, move it to \'rope_type\'.\n        if self.rope_scaling is not None and "type" in self.rope_scaling:\n            self.rope_scaling["rope_type"] = self.rope_scaling["type"]\n        rope_config_validation(self)\n\n        super().__init__(\n            tie_word_embeddings=tie_word_embeddings,\n            **kwargs,\n        )\n', 'modeling/qwen2/modeling_qwen2.py': '# Copyright 2024 The Qwen Team and The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""PyTorch Qwen2 model."""\n\nimport math\nfrom typing import List, Optional, Tuple, Union\n\nimport torch\nimport torch.utils.checkpoint\nfrom torch import nn\n\nfrom transformers.activations import ACT2FN\nfrom transformers.cache_utils import Cache, DynamicCache\nfrom transformers.generation import GenerationMixin\nfrom transformers.modeling_outputs import (\n    BaseModelOutputWithPast,\n    CausalLMOutputWithPast,\n)\nfrom transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.utils import (\n    add_start_docstrings,\n    add_start_docstrings_to_model_forward,\n    is_flash_attn_2_available,\n    is_flash_attn_greater_or_equal_2_10,\n    logging,\n    replace_return_docstrings,\n)\nfrom .configuration_qwen2 import Qwen2Config\n\n\nif is_flash_attn_2_available():\n    from transformers.modeling_flash_attention_utils import _flash_attention_forward\n\n\nlogger = logging.get_logger(__name__)\n\n\n_CHECKPOINT_FOR_DOC = "Qwen/Qwen2-7B"\n_CONFIG_FOR_DOC = "Qwen2Config"\n\n\n# Copied from transformers.models.llama.modeling_llama.LlamaRMSNorm with Llama->Qwen2\nclass Qwen2RMSNorm(nn.Module):\n    def __init__(self, hidden_size, eps=1e-6):\n        """\n        Qwen2RMSNorm is equivalent to T5LayerNorm\n        """\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(hidden_size))\n        self.variance_epsilon = eps\n\n    def forward(self, hidden_states):\n        input_dtype = hidden_states.dtype\n        hidden_states = hidden_states.to(torch.float32)\n        variance = hidden_states.pow(2).mean(-1, keepdim=True)\n        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)\n        return self.weight * hidden_states.to(input_dtype)\n\n    def extra_repr(self):\n        return f"{tuple(self.weight.shape)}, eps={self.variance_epsilon}"\n\n\n# Copied from transformers.models.llama.modeling_llama.LlamaRotaryEmbedding with Llama->Qwen2\nclass Qwen2RotaryEmbedding(nn.Module):\n    def __init__(\n        self,\n        dim=None,\n        max_position_embeddings=2048,\n        base=10000,\n        device=None,\n        scaling_factor=1.0,\n        rope_type="default",\n        config: Optional[Qwen2Config] = None,\n    ):\n        super().__init__()\n        # TODO (joao): remove the `if` below, only used for BC\n        self.rope_kwargs = {}\n        if config is None:\n            logger.warning_once(\n                "`Qwen2RotaryEmbedding` can now be fully parameterized by passing the model config through the "\n                "`config` argument. All other arguments will be removed in v4.46"\n            )\n            self.rope_kwargs = {\n                "rope_type": rope_type,\n                "factor": scaling_factor,\n                "dim": dim,\n                "base": base,\n                "max_position_embeddings": max_position_embeddings,\n            }\n            self.rope_type = rope_type\n            self.max_seq_len_cached = max_position_embeddings\n            self.original_max_seq_len = max_position_embeddings\n        else:\n            # BC: "rope_type" was originally "type"\n            if config.rope_scaling is not None:\n                self.rope_type = config.rope_scaling.get("rope_type", config.rope_scaling.get("type"))\n            else:\n                self.rope_type = "default"\n            self.max_seq_len_cached = config.max_position_embeddings\n            self.original_max_seq_len = config.max_position_embeddings\n\n        self.config = config\n        self.rope_init_fn = ROPE_INIT_FUNCTIONS[self.rope_type]\n\n        inv_freq, self.attention_scaling = self.rope_init_fn(self.config, device, **self.rope_kwargs)\n        self.register_buffer("inv_freq", inv_freq, persistent=False)\n        self.original_inv_freq = self.inv_freq\n\n    def _dynamic_frequency_update(self, position_ids, device):\n        """\n        dynamic RoPE layers should recompute `inv_freq` in the following situations:\n        1 - growing beyond the cached sequence length (allow scaling)\n        2 - the current sequence length is in the original scale (avoid losing precision with small sequences)\n        """\n        seq_len = torch.max(position_ids) + 1\n        if seq_len > self.max_seq_len_cached:  # growth\n            inv_freq, self.attention_scaling = self.rope_init_fn(\n                self.config, device, seq_len=seq_len, **self.rope_kwargs\n            )\n            self.register_buffer("inv_freq", inv_freq, persistent=False)  # TODO joao: may break with compilation\n            self.max_seq_len_cached = seq_len\n\n        if seq_len < self.original_max_seq_len and self.max_seq_len_cached > self.original_max_seq_len:  # reset\n            self.register_buffer("inv_freq", self.original_inv_freq, persistent=False)\n            self.max_seq_len_cached = self.original_max_seq_len\n\n    @torch.no_grad()\n    def forward(self, x, position_ids):\n        if "dynamic" in self.rope_type:\n            self._dynamic_frequency_update(position_ids, device=x.device)\n\n        # Core RoPE block\n        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)\n        position_ids_expanded = position_ids[:, None, :].float()\n        # Force float32 (see https://github.com/huggingface/transformers/pull/29285)\n        device_type = x.device.type\n        device_type = device_type if isinstance(device_type, str) and device_type != "mps" else "cpu"\n        with torch.autocast(device_type=device_type, enabled=False):\n            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)\n            emb = torch.cat((freqs, freqs), dim=-1)\n            cos = emb.cos()\n            sin = emb.sin()\n\n        # Advanced RoPE types (e.g. yarn) apply a post-processing scaling factor, equivalent to scaling attention\n        cos = cos * self.attention_scaling\n        sin = sin * self.attention_scaling\n\n        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)\n\n\n# Copied from transformers.models.llama.modeling_llama.rotate_half\ndef rotate_half(x):\n    """Rotates half the hidden dims of the input."""\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat((-x2, x1), dim=-1)\n\n\n# Copied from transformers.models.llama.modeling_llama.apply_rotary_pos_emb\ndef apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):\n    """Applies Rotary Position Embedding to the query and key tensors.\n\n    Args:\n        q (`torch.Tensor`): The query tensor.\n        k (`torch.Tensor`): The key tensor.\n        cos (`torch.Tensor`): The cosine part of the rotary embedding.\n        sin (`torch.Tensor`): The sine part of the rotary embedding.\n        position_ids (`torch.Tensor`, *optional*):\n            Deprecated and unused.\n        unsqueeze_dim (`int`, *optional*, defaults to 1):\n            The \'unsqueeze_dim\' argument specifies the dimension along which to unsqueeze cos[position_ids] and\n            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note\n            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and\n            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes\n            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have\n            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.\n    Returns:\n        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.\n    """\n    cos = cos.unsqueeze(unsqueeze_dim)\n    sin = sin.unsqueeze(unsqueeze_dim)\n    q_embed = (q * cos) + (rotate_half(q) * sin)\n    k_embed = (k * cos) + (rotate_half(k) * sin)\n    return q_embed, k_embed\n\n\n# Copied from transformers.models.mistral.modeling_mistral.MistralMLP with Mistral->Qwen2\nclass Qwen2MLP(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n        self.intermediate_size = config.intermediate_size\n        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)\n        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)\n        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)\n        self.act_fn = ACT2FN[config.hidden_act]\n\n    def forward(self, hidden_state):\n        return self.down_proj(self.act_fn(self.gate_proj(hidden_state)) * self.up_proj(hidden_state))\n\n\n# Copied from transformers.models.llama.modeling_llama.repeat_kv\ndef repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:\n    """\n    This is the equivalent of torch.repeat_interleave(x, dim=1, repeats=n_rep). The hidden states go from (batch,\n    num_key_value_heads, seqlen, head_dim) to (batch, num_attention_heads, seqlen, head_dim)\n    """\n    batch, num_key_value_heads, slen, head_dim = hidden_states.shape\n    if n_rep == 1:\n        return hidden_states\n    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)\n    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)\n\n\nclass Qwen2Attention(nn.Module):\n    """\n    Multi-headed attention from \'Attention Is All You Need\' paper. Modified to use sliding window attention: Longformer\n    and "Generating Long Sequences with Sparse Transformers".\n    """\n\n    def __init__(self, config: Qwen2Config, layer_idx: Optional[int] = None):\n        super().__init__()\n        self.config = config\n        self.layer_idx = layer_idx\n        if layer_idx is None:\n            logger.warning_once(\n                f"Instantiating {self.__class__.__name__} without passing `layer_idx` is not recommended and will "\n                "to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` "\n                "when creating this class."\n            )\n\n        self.hidden_size = config.hidden_size\n        self.num_heads = config.num_attention_heads\n        self.head_dim = self.hidden_size // self.num_heads\n        self.num_key_value_heads = config.num_key_value_heads\n        self.num_key_value_groups = self.num_heads // self.num_key_value_heads\n        self.max_position_embeddings = config.max_position_embeddings\n        self.rope_theta = config.rope_theta\n        self.is_causal = config.is_causal\n        self.attention_dropout = config.attention_dropout\n\n        if (self.head_dim * self.num_heads) != self.hidden_size:\n            raise ValueError(\n                f"hidden_size must be divisible by num_heads (got `hidden_size`: {self.hidden_size}"\n                f" and `num_heads`: {self.num_heads})."\n            )\n        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=True)\n        self.k_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)\n        self.v_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True)\n        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Cache] = None,\n        output_attentions: bool = False,\n        use_cache: bool = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.46\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:\n        bsz, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        if position_embeddings is None:\n            logger.warning_once(\n                "The attention layers in this model are transitioning from computing the RoPE embeddings internally "\n                "through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed "\n                "`position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be "\n                "removed and `position_embeddings` will be mandatory."\n            )\n            cos, sin = self.rotary_emb(value_states, position_ids)\n        else:\n            cos, sin = position_embeddings\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n\n        if past_key_value is not None:\n            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}  # Specific to RoPE models\n            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)\n\n        # repeat k/v heads if n_kv_heads < n_heads\n        key_states = repeat_kv(key_states, self.num_key_value_groups)\n        value_states = repeat_kv(value_states, self.num_key_value_groups)\n\n        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)\n        if attention_mask is not None:  # no matter the length, we just slice it\n            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]\n            attn_weights = attn_weights + causal_mask\n\n        # upcast attention to fp32\n        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n        attn_weights = nn.functional.dropout(attn_weights, p=self.attention_dropout, training=self.training)\n        attn_output = torch.matmul(attn_weights, value_states)\n\n        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):\n            raise ValueError(\n                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"\n                f" {attn_output.size()}"\n            )\n\n        attn_output = attn_output.transpose(1, 2).contiguous()\n        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size)\n\n        attn_output = self.o_proj(attn_output)\n\n        if not output_attentions:\n            attn_weights = None\n\n        return attn_output, attn_weights, past_key_value\n\n\nclass Qwen2FlashAttention2(Qwen2Attention):\n    """\n    Qwen2 flash attention module, following Qwen2 attention module. This module inherits from `Qwen2Attention`\n    as the weights of the module stays untouched. The only required change would be on the forward pass\n    where it needs to correctly call the public API of flash attention and deal with padding tokens\n    in case the input contains any of them. Additionally, for sliding window attention, we apply SWA only to the bottom\n    config.max_window_layers layers.\n    """\n\n    # Copied from transformers.models.llama.modeling_llama.LlamaFlashAttention2.__init__\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n\n        # TODO: Should be removed once Flash Attention for RoCm is bumped to 2.1.\n        # flash_attn<2.1 generates top-left aligned causal mask, while what is needed here is bottom-right alignement, that was made default for flash_attn>=2.1. This attribute is used to handle this difference. Reference: https://github.com/Dao-AILab/flash-attention/releases/tag/v2.1.0.\n        # Beware that with flash_attn<2.1, using q_seqlen != k_seqlen (except for the case q_seqlen == 1) produces a wrong mask (top-left).\n        self._flash_attn_uses_top_left_mask = not is_flash_attn_greater_or_equal_2_10()\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Cache] = None,\n        output_attentions: bool = False,\n        use_cache: bool = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.46\n    ):\n        bsz, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        if position_embeddings is None:\n            logger.warning_once(\n                "The attention layers in this model are transitioning from computing the RoPE embeddings internally "\n                "through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed "\n                "`position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be "\n                "removed and `position_embeddings` will be mandatory."\n            )\n            cos, sin = self.rotary_emb(value_states, position_ids)\n        else:\n            cos, sin = position_embeddings\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n\n        if past_key_value is not None:\n            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}  # Specific to RoPE models\n            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)\n\n        # repeat k/v heads if n_kv_heads < n_heads\n        key_states = repeat_kv(key_states, self.num_key_value_groups)\n        value_states = repeat_kv(value_states, self.num_key_value_groups)\n        dropout_rate = 0.0 if not self.training else self.attention_dropout\n\n        # In PEFT, usually we cast the layer norms in float32 for training stability reasons\n        # therefore the input hidden states gets silently casted in float32. Hence, we need\n        # cast them back in float16 just to be sure everything works as expected.\n        input_dtype = query_states.dtype\n        if input_dtype == torch.float32:\n            if torch.is_autocast_enabled():\n                target_dtype = torch.get_autocast_gpu_dtype()\n            # Handle the case where the model is quantized\n            elif hasattr(self.config, "_pre_quantization_dtype"):\n                target_dtype = self.config._pre_quantization_dtype\n            else:\n                target_dtype = self.q_proj.weight.dtype\n\n            logger.warning_once(\n                f"The input hidden states seems to be silently casted in float32, this might be related to"\n                f" the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in"\n                f" {target_dtype}."\n            )\n\n            query_states = query_states.to(target_dtype)\n            key_states = key_states.to(target_dtype)\n            value_states = value_states.to(target_dtype)\n\n        # Reashape to the expected shape for Flash Attention\n        query_states = query_states.transpose(1, 2)\n        key_states = key_states.transpose(1, 2)\n        value_states = value_states.transpose(1, 2)\n\n        if (\n            self.config.use_sliding_window\n            and getattr(self.config, "sliding_window", None) is not None\n            and self.layer_idx >= self.config.max_window_layers\n        ):\n            sliding_window = self.config.sliding_window\n        else:\n            sliding_window = None\n\n        attn_output = _flash_attention_forward(\n            query_states,\n            key_states,\n            value_states,\n            attention_mask,\n            q_len,\n            position_ids=position_ids,\n            dropout=dropout_rate,\n            sliding_window=sliding_window,\n            is_causal=self.is_causal,\n            use_top_left_mask=self._flash_attn_uses_top_left_mask,\n        )\n\n        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size).contiguous()\n        attn_output = self.o_proj(attn_output)\n\n        if not output_attentions:\n            attn_weights = None\n\n        return attn_output, attn_weights, past_key_value\n\n\nQWEN2_ATTENTION_CLASSES = {\n    "eager": Qwen2Attention,\n    "flash_attention_2": Qwen2FlashAttention2,\n}\n\n\nclass Qwen2DecoderLayer(nn.Module):\n    def __init__(self, config: Qwen2Config, layer_idx: int):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n\n        if config.sliding_window and config._attn_implementation != "flash_attention_2":\n            logger.warning_once(\n                f"Sliding Window Attention is enabled but not implemented for `{config._attn_implementation}`; "\n                "unexpected results may be encountered."\n            )\n        self.self_attn = QWEN2_ATTENTION_CLASSES[config._attn_implementation](config, layer_idx)\n\n        self.mlp = Qwen2MLP(config)\n        self.input_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Tuple[torch.Tensor]] = None,\n        output_attentions: Optional[bool] = False,\n        use_cache: Optional[bool] = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.46\n        **kwargs,\n    ) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]:\n        """\n        Args:\n            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`\n            attention_mask (`torch.FloatTensor`, *optional*): attention mask of size\n                `(batch, sequence_length)` where padding elements are indicated by 0.\n            output_attentions (`bool`, *optional*):\n                Whether or not to return the attentions tensors of all attention layers. See `attentions` under\n                returned tensors for more detail.\n            use_cache (`bool`, *optional*):\n                If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding\n                (see `past_key_values`).\n            past_key_value (`Tuple(torch.FloatTensor)`, *optional*): cached past key and value projection states\n            cache_position (`torch.LongTensor` of shape `(sequence_length)`, *optional*):\n                Indices depicting the position of the input sequence tokens in the sequence.\n            position_embeddings (`Tuple[torch.FloatTensor, torch.FloatTensor]`, *optional*):\n                Tuple containing the cosine and sine positional embeddings of shape `(batch_size, seq_len, head_dim)`,\n                with `head_dim` being the embedding dimension of each attention head.\n            kwargs (`dict`, *optional*):\n                Arbitrary kwargs to be ignored, used for FSDP and other methods that injects code\n                into the model\n        """\n\n        residual = hidden_states\n\n        hidden_states = self.input_layernorm(hidden_states)\n\n        # Self Attention\n        hidden_states, self_attn_weights, present_key_value = self.self_attn(\n            hidden_states=hidden_states,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_value=past_key_value,\n            output_attentions=output_attentions,\n            use_cache=use_cache,\n            cache_position=cache_position,\n            position_embeddings=position_embeddings,\n        )\n        hidden_states = residual + hidden_states\n\n        # Fully Connected\n        residual = hidden_states\n        hidden_states = self.post_attention_layernorm(hidden_states)\n        hidden_states = self.mlp(hidden_states)\n        hidden_states = residual + hidden_states\n\n        outputs = (hidden_states,)\n\n        if output_attentions:\n            outputs += (self_attn_weights,)\n\n        if use_cache:\n            outputs += (present_key_value,)\n\n        return outputs\n\n\nQWEN2_START_DOCSTRING = r"""\n    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the\n    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads\n    etc.)\n\n    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.\n    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage\n    and behavior.\n\n    Parameters:\n        config ([`Qwen2Config`]):\n            Model configuration class with all the parameters of the model. Initializing with a config file does not\n            load the weights associated with the model, only the configuration. Check out the\n            [`~PreTrainedModel.from_pretrained`] method to load the model weights.\n"""\n\n\n@add_start_docstrings(\n    "The bare Qwen2 Model outputting raw hidden-states without any specific head on top.",\n    QWEN2_START_DOCSTRING,\n)\nclass Qwen2PreTrainedModel(PreTrainedModel):\n    config_class = Qwen2Config\n    base_model_prefix = "model"\n    supports_gradient_checkpointing = True\n    _no_split_modules = ["Qwen2DecoderLayer"]\n    _skip_keys_device_placement = "past_key_values"\n    _supports_flash_attn_2 = True\n    _supports_cache_class = True\n    _supports_quantized_cache = True\n    _supports_static_cache = True\n\n    def _init_weights(self, module):\n        std = self.config.initializer_range\n        if isinstance(module, nn.Linear):\n            module.weight.data.normal_(mean=0.0, std=std)\n            if module.bias is not None:\n                module.bias.data.zero_()\n        elif isinstance(module, nn.Embedding):\n            module.weight.data.normal_(mean=0.0, std=std)\n            if module.padding_idx is not None:\n                module.weight.data[module.padding_idx].zero_()\n\n\nQWEN2_INPUTS_DOCSTRING = r"""\n    Args:\n        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):\n            Indices of input sequence tokens in the vocabulary. Padding will be ignored by default should you provide\n            it.\n\n            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and\n            [`PreTrainedTokenizer.__call__`] for details.\n\n            [What are input IDs?](../glossary#input-ids)\n        attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:\n\n            - 1 for tokens that are **not masked**,\n            - 0 for tokens that are **masked**.\n\n            [What are attention masks?](../glossary#attention-mask)\n\n            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and\n            [`PreTrainedTokenizer.__call__`] for details.\n\n            If `past_key_values` is used, optionally only the last `decoder_input_ids` have to be input (see\n            `past_key_values`).\n\n            If you want to change padding behavior, you should read [`modeling_opt._prepare_decoder_attention_mask`]\n            and modify to your needs. See diagram 1 in [the paper](https://arxiv.org/abs/1910.13461) for more\n            information on the default strategy.\n\n            - 1 indicates the head is **not masked**,\n            - 0 indicates the head is **masked**.\n        position_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Indices of positions of each input sequence tokens in the position embeddings. Selected in the range `[0,\n            config.n_positions - 1]`.\n\n            [What are position IDs?](../glossary#position-ids)\n        past_key_values (`Cache` or `tuple(tuple(torch.FloatTensor))`, *optional*):\n            Pre-computed hidden-states (key and values in the self-attention blocks and in the cross-attention\n            blocks) that can be used to speed up sequential decoding. This typically consists in the `past_key_values`\n            returned by the model at a previous stage of decoding, when `use_cache=True` or `config.use_cache=True`.\n\n            Two formats are allowed:\n            - a [`~cache_utils.Cache`] instance, see our\n            [kv cache guide](https://huggingface.co/docs/transformers/en/kv_cache);\n            - Tuple of `tuple(torch.FloatTensor)` of length `config.n_layers`, with each tuple having 2 tensors of\n            shape `(batch_size, num_heads, sequence_length, embed_size_per_head)`). This is also known as the legacy\n            cache format.\n\n            The model will output the same cache format that is fed as input. If no `past_key_values` are passed, the\n            legacy cache format will be returned.\n\n            If `past_key_values` are used, the user can optionally input only the last `input_ids` (those that don\'t\n            have their past key value states given to this model) of shape `(batch_size, 1)` instead of all `input_ids`\n            of shape `(batch_size, sequence_length)`.\n        inputs_embeds (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`, *optional*):\n            Optionally, instead of passing `input_ids` you can choose to directly pass an embedded representation. This\n            is useful if you want more control over how to convert `input_ids` indices into associated vectors than the\n            model\'s internal embedding lookup matrix.\n        use_cache (`bool`, *optional*):\n            If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding (see\n            `past_key_values`).\n        output_attentions (`bool`, *optional*):\n            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned\n            tensors for more detail.\n        output_hidden_states (`bool`, *optional*):\n            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for\n            more detail.\n        return_dict (`bool`, *optional*):\n            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.\n        cache_position (`torch.LongTensor` of shape `(sequence_length)`, *optional*):\n            Indices depicting the position of the input sequence tokens in the sequence. Contrarily to `position_ids`,\n            this tensor is not affected by padding. It is used to update the cache in the correct position and to infer\n            the complete sequence length.\n"""\n\n\n@add_start_docstrings(\n    "The bare Qwen2 Model outputting raw hidden-states without any specific head on top.",\n    QWEN2_START_DOCSTRING,\n)\nclass Qwen2Model(Qwen2PreTrainedModel):\n    """\n    Transformer decoder consisting of *config.num_hidden_layers* layers. Each layer is a [`Qwen2DecoderLayer`]\n\n    Args:\n        config: Qwen2Config\n    """\n\n    def __init__(self, config: Qwen2Config):\n        super().__init__(config)\n        self.padding_idx = config.pad_token_id\n        self.vocab_size = config.vocab_size\n\n        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)\n        self.layers = nn.ModuleList(\n            [Qwen2DecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]\n        )\n        self._attn_implementation = config._attn_implementation\n        self.norm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.rotary_emb = Qwen2RotaryEmbedding(config=config)\n\n        self.gradient_checkpointing = False\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self):\n        return self.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.embed_tokens = value\n\n    @add_start_docstrings_to_model_forward(QWEN2_INPUTS_DOCSTRING)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[List[torch.FloatTensor]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n    ) -> Union[Tuple, BaseModelOutputWithPast]:\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        use_cache = use_cache if use_cache is not None else self.config.use_cache\n\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        if (input_ids is None) ^ (inputs_embeds is not None):\n            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")\n\n        if self.gradient_checkpointing and self.training:\n            if use_cache:\n                logger.warning_once(\n                    "`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`..."\n                )\n                use_cache = False\n\n        # kept for BC (non `Cache` `past_key_values` inputs)\n        return_legacy_cache = False\n        if use_cache and not isinstance(past_key_values, Cache):\n            return_legacy_cache = True\n            if past_key_values is None:\n                past_key_values = DynamicCache()\n            else:\n                past_key_values = DynamicCache.from_legacy_cache(past_key_values)\n                logger.warning_once(\n                    "We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and "\n                    "will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class "\n                    "(https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)"\n                )\n\n        if inputs_embeds is None:\n            inputs_embeds = self.embed_tokens(input_ids)\n\n        if cache_position is None:\n            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0\n            cache_position = torch.arange(\n                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device\n            )\n        if position_ids is None:\n            position_ids = cache_position.unsqueeze(0)\n\n        if attention_mask is not None and 0.0 in attention_mask:\n            causal_mask = attention_mask\n        else:\n            causal_mask = None\n\n        hidden_states = inputs_embeds\n        # create position embeddings to be shared across the decoder layers\n        position_embeddings = self.rotary_emb(hidden_states, position_ids)\n\n        # decoder layers\n        all_hidden_states = () if output_hidden_states else None\n        all_self_attns = () if output_attentions else None\n        next_decoder_cache = None\n\n        for decoder_layer in self.layers:\n            if output_hidden_states:\n                all_hidden_states += (hidden_states,)\n\n            if self.gradient_checkpointing and self.training:\n                layer_outputs = self._gradient_checkpointing_func(\n                    decoder_layer.__call__,\n                    hidden_states,\n                    causal_mask,\n                    position_ids,\n                    past_key_values,\n                    output_attentions,\n                    use_cache,\n                    cache_position,\n                    position_embeddings,\n                )\n            else:\n                layer_outputs = decoder_layer(\n                    hidden_states,\n                    attention_mask=causal_mask,\n                    position_ids=position_ids,\n                    past_key_value=past_key_values,\n                    output_attentions=output_attentions,\n                    use_cache=use_cache,\n                    cache_position=cache_position,\n                    position_embeddings=position_embeddings,\n                )\n\n            hidden_states = layer_outputs[0]\n\n            if use_cache:\n                next_decoder_cache = layer_outputs[2 if output_attentions else 1]\n\n            if output_attentions:\n                all_self_attns += (layer_outputs[1],)\n\n        hidden_states = self.norm(hidden_states)\n\n        # add hidden states from the last decoder layer\n        if output_hidden_states:\n            all_hidden_states += (hidden_states,)\n\n        next_cache = next_decoder_cache if use_cache else None\n        if return_legacy_cache:\n            next_cache = next_cache.to_legacy_cache()\n\n        if not return_dict:\n            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)\n        return BaseModelOutputWithPast(\n            last_hidden_state=hidden_states,\n            past_key_values=next_cache,\n            hidden_states=all_hidden_states,\n            attentions=all_self_attns,\n        )\n\n\nclass Qwen2ForCausalLM(Qwen2PreTrainedModel, GenerationMixin):\n    _tied_weights_keys = ["lm_head.weight"]\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.model = Qwen2Model(config)\n        self.vocab_size = config.vocab_size\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self):\n        return self.model.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.model.embed_tokens = value\n\n    def get_output_embeddings(self):\n        return self.lm_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.lm_head = new_embeddings\n\n    def set_decoder(self, decoder):\n        self.model = decoder\n\n    def get_decoder(self):\n        return self.model\n\n    @add_start_docstrings_to_model_forward(QWEN2_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=CausalLMOutputWithPast, config_class=_CONFIG_FOR_DOC)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[List[torch.FloatTensor]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        labels: Optional[torch.LongTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n        num_logits_to_keep: int = 0,\n        **loss_kwargs,\n    ) -> Union[Tuple, CausalLMOutputWithPast]:\n        r"""\n        Args:\n            labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n                Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,\n                config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored\n                (masked), the loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`.\n\n            num_logits_to_keep (`int`, *optional*):\n                Calculate logits for the last `num_logits_to_keep` tokens. If `0`, calculate logits for all\n                `input_ids` (special case). Only last token logits are needed for generation, and calculating them only for that\n                token can save memory, which becomes pretty significant for long sequences or large vocabulary size.\n\n        Returns:\n\n        Example:\n\n        ```python\n        >>> from transformers import AutoTokenizer, Qwen2ForCausalLM\n\n        >>> model = Qwen2ForCausalLM.from_pretrained(PATH_TO_CONVERTED_WEIGHTS)\n        >>> tokenizer = AutoTokenizer.from_pretrained(PATH_TO_CONVERTED_TOKENIZER)\n\n        >>> prompt = "Hey, are you conscious? Can you talk to me?"\n        >>> inputs = tokenizer(prompt, return_tensors="pt")\n\n        >>> # Generate\n        >>> generate_ids = model.generate(inputs.input_ids, max_length=30)\n        >>> tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]\n        "Hey, are you conscious? Can you talk to me?\\nI\'m not conscious, but I can talk to you."\n        ```"""\n\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        # decoder outputs consists of (dec_features, layer_state, dec_hidden, dec_attn)\n        outputs = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_values=past_key_values,\n            inputs_embeds=inputs_embeds,\n            use_cache=use_cache,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            cache_position=cache_position,\n        )\n\n        hidden_states = outputs[0]\n        # Only compute necessary logits, and do not upcast them to float if we are not computing the loss\n        logits = self.lm_head(hidden_states[:, -num_logits_to_keep:, :])\n\n        loss = None\n        if labels is not None:\n            loss = self.loss_function(logits, labels, self.vocab_size, **loss_kwargs)\n\n        if not return_dict:\n            output = (logits,) + outputs[1:]\n            return (loss,) + output if loss is not None else output\n\n        return CausalLMOutputWithPast(\n            loss=loss,\n            logits=logits,\n            past_key_values=outputs.past_key_values,\n            hidden_states=outputs.hidden_states,\n            attentions=outputs.attentions,\n        )\n', 'modeling/qwen2/tokenization_qwen2.py': '# Copyright 2024 The Qwen Team and The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Tokenization classes for Qwen2."""\n\nimport json\nimport os\nimport unicodedata\nfrom functools import lru_cache\nfrom typing import Optional, Tuple\n\nimport regex as re\n\nfrom transformers.tokenization_utils import AddedToken, PreTrainedTokenizer\nfrom transformers.utils import logging\n\n\nlogger = logging.get_logger(__name__)\n\nVOCAB_FILES_NAMES = {\n    "vocab_file": "vocab.json",\n    "merges_file": "merges.txt",\n}\n\n\nMAX_MODEL_INPUT_SIZES = {"qwen/qwen-tokenizer": 32768}\n\nPRETOKENIZE_REGEX = r"""(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+"""\n\n\n@lru_cache()\n# Copied from transformers.models.gpt2.tokenization_gpt2.bytes_to_unicode\ndef bytes_to_unicode():\n    """\n    Returns list of utf-8 byte and a mapping to unicode strings. We specifically avoids mapping to whitespace/control\n    characters the bpe code barfs on.\n\n    The reversible bpe codes work on unicode strings. This means you need a large # of unicode characters in your vocab\n    if you want to avoid UNKs. When you\'re at something like a 10B token dataset you end up needing around 5K for\n    decent coverage. This is a significant percentage of your normal, say, 32K bpe vocab. To avoid that, we want lookup\n    tables between utf-8 bytes and unicode strings.\n    """\n    bs = (\n        list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))\n    )\n    cs = bs[:]\n    n = 0\n    for b in range(2**8):\n        if b not in bs:\n            bs.append(b)\n            cs.append(2**8 + n)\n            n += 1\n    cs = [chr(n) for n in cs]\n    return dict(zip(bs, cs))\n\n\n# Copied from transformers.models.gpt2.tokenization_gpt2.get_pairs\ndef get_pairs(word):\n    """\n    Return set of symbol pairs in a word.\n\n    Word is represented as tuple of symbols (symbols being variable-length strings).\n    """\n    pairs = set()\n    prev_char = word[0]\n    for char in word[1:]:\n        pairs.add((prev_char, char))\n        prev_char = char\n    return pairs\n\n\nclass Qwen2Tokenizer(PreTrainedTokenizer):\n    """\n    Construct a Qwen2 tokenizer. Based on byte-level Byte-Pair-Encoding.\n\n    Same with GPT2Tokenizer, this tokenizer has been trained to treat spaces like parts of the tokens so a word will\n    be encoded differently whether it is at the beginning of the sentence (without space) or not:\n\n    ```python\n    >>> from transformers import Qwen2Tokenizer\n\n    >>> tokenizer = Qwen2Tokenizer.from_pretrained("Qwen/Qwen-tokenizer")\n    >>> tokenizer("Hello world")["input_ids"]\n    [9707, 1879]\n\n    >>> tokenizer(" Hello world")["input_ids"]\n    [21927, 1879]\n    ```\n    This is expected.\n\n    You should not use GPT2Tokenizer instead, because of the different pretokenization rules.\n\n    This tokenizer inherits from [`PreTrainedTokenizer`] which contains most of the main methods. Users should refer to\n    this superclass for more information regarding those methods.\n\n    Args:\n        vocab_file (`str`):\n            Path to the vocabulary file.\n        merges_file (`str`):\n            Path to the merges file.\n        errors (`str`, *optional*, defaults to `"replace"`):\n            Paradigm to follow when decoding bytes to UTF-8. See\n            [bytes.decode](https://docs.python.org/3/library/stdtypes.html#bytes.decode) for more information.\n        unk_token (`str`, *optional*, defaults to `"<|endoftext|>"`):\n            The unknown token. A token that is not in the vocabulary cannot be converted to an ID and is set to be this\n            token instead.\n        bos_token (`str`, *optional*):\n            The beginning of sequence token. Not applicable for this tokenizer.\n        eos_token (`str`, *optional*, defaults to `"<|endoftext|>"`):\n            The end of sequence token.\n        pad_token (`str`, *optional*, defaults to `"<|endoftext|>"`):\n            The token used for padding, for example when batching sequences of different lengths.\n        clean_up_tokenization_spaces (`bool`, *optional*, defaults to `False`):\n            Whether or not the model should cleanup the spaces that were added when splitting the input text during the\n            tokenization process. Not applicable to this tokenizer, since tokenization does not add spaces.\n        split_special_tokens (`bool`, *optional*, defaults to `False`):\n            Whether or not the special tokens should be split during the tokenization process. The default behavior is\n            to not split special tokens. This means that if `<|endoftext|>` is the `eos_token`, then `tokenizer.tokenize("<|endoftext|>") =\n            [\'<|endoftext|>`]. Otherwise, if `split_special_tokens=True`, then `tokenizer.tokenize("<|endoftext|>")` will be give `[\'<\',\n            \'|\', \'endo\', \'ft\', \'ext\', \'|\', \'>\']`. This argument is only supported for `slow` tokenizers for the moment.\n    """\n\n    vocab_files_names = VOCAB_FILES_NAMES\n    model_input_names = ["input_ids", "attention_mask"]\n\n    def __init__(\n        self,\n        vocab_file,\n        merges_file,\n        errors="replace",\n        unk_token="<|endoftext|>",\n        bos_token=None,\n        eos_token="<|endoftext|>",\n        pad_token="<|endoftext|>",\n        clean_up_tokenization_spaces=False,\n        split_special_tokens=False,\n        **kwargs,\n    ):\n        # Qwen vocab does not contain control tokens; added tokens need to be special\n        bos_token = (\n            AddedToken(bos_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(bos_token, str)\n            else bos_token\n        )\n        eos_token = (\n            AddedToken(eos_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(eos_token, str)\n            else eos_token\n        )\n        unk_token = (\n            AddedToken(unk_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(unk_token, str)\n            else unk_token\n        )\n        pad_token = (\n            AddedToken(pad_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(pad_token, str)\n            else pad_token\n        )\n\n        with open(vocab_file, encoding="utf-8") as vocab_handle:\n            self.encoder = json.load(vocab_handle)\n        self.decoder = {v: k for k, v in self.encoder.items()}\n        self.errors = errors  # how to handle errors in decoding\n        self.byte_encoder = bytes_to_unicode()\n        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}\n        bpe_merges = []\n        with open(merges_file, encoding="utf-8") as merges_handle:\n            for i, line in enumerate(merges_handle):\n                line = line.strip()\n                if (i == 0 and line.startswith("#version:")) or not line:\n                    continue\n                bpe_merges.append(tuple(line.split()))\n        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))\n        # NOTE: the cache can grow without bound and will get really large for long running processes\n        # (esp. for texts of language that do not use space between word, e.g. Chinese); technically\n        # not a memory leak but appears as one.\n        # GPT2Tokenizer has the same problem, so let\'s be consistent.\n        self.cache = {}\n\n        self.pat = re.compile(PRETOKENIZE_REGEX)\n\n        if kwargs.get("add_prefix_space", False):\n            logger.warning_once(\n                f"{self.__class__.__name} does not support `add_prefix_space`, setting it to True has no effect."\n            )\n\n        super().__init__(\n            errors=errors,\n            bos_token=bos_token,\n            eos_token=eos_token,\n            pad_token=pad_token,\n            unk_token=unk_token,\n            clean_up_tokenization_spaces=clean_up_tokenization_spaces,\n            split_special_tokens=split_special_tokens,\n            **kwargs,\n        )\n\n    @property\n    def vocab_size(self) -> int:\n        return len(self.encoder)\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer.get_vocab\n    def get_vocab(self):\n        return dict(self.encoder, **self.added_tokens_encoder)\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer.bpe\n    def bpe(self, token):\n        if token in self.cache:\n            return self.cache[token]\n        word = tuple(token)\n        pairs = get_pairs(word)\n\n        if not pairs:\n            return token\n\n        while True:\n            bigram = min(pairs, key=lambda pair: self.bpe_ranks.get(pair, float("inf")))\n            if bigram not in self.bpe_ranks:\n                break\n            first, second = bigram\n            new_word = []\n            i = 0\n            while i < len(word):\n                try:\n                    j = word.index(first, i)\n                except ValueError:\n                    new_word.extend(word[i:])\n                    break\n                else:\n                    new_word.extend(word[i:j])\n                    i = j\n\n                if word[i] == first and i < len(word) - 1 and word[i + 1] == second:\n                    new_word.append(first + second)\n                    i += 2\n                else:\n                    new_word.append(word[i])\n                    i += 1\n            new_word = tuple(new_word)\n            word = new_word\n            if len(word) == 1:\n                break\n            else:\n                pairs = get_pairs(word)\n        word = " ".join(word)\n        self.cache[token] = word\n        return word\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer._tokenize\n    def _tokenize(self, text):\n        """Tokenize a string."""\n        bpe_tokens = []\n        for token in re.findall(self.pat, text):\n            token = "".join(\n                self.byte_encoder[b] for b in token.encode("utf-8")\n            )  # Maps all our bytes to unicode strings, avoiding control tokens of the BPE (spaces in our case)\n            bpe_tokens.extend(bpe_token for bpe_token in self.bpe(token).split(" "))\n        return bpe_tokens\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer._convert_token_to_id\n    def _convert_token_to_id(self, token):\n        """Converts a token (str) in an id using the vocab."""\n        return self.encoder.get(token, self.encoder.get(self.unk_token))\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer._convert_id_to_token\n    def _convert_id_to_token(self, index):\n        """Converts an index (integer) in a token (str) using the vocab."""\n        return self.decoder.get(index)\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer.convert_tokens_to_string\n    def convert_tokens_to_string(self, tokens):\n        """Converts a sequence of tokens (string) in a single string."""\n        text = "".join(tokens)\n        text = bytearray([self.byte_decoder[c] for c in text]).decode("utf-8", errors=self.errors)\n        return text\n\n    def decode(\n        self,\n        token_ids,\n        skip_special_tokens: bool = False,\n        clean_up_tokenization_spaces: Optional[bool] = False,\n        spaces_between_special_tokens: bool = False,\n        **kwargs,\n    ) -> str:\n        # `spaces_between_special_tokens` defaults to True for _decode in slow tokenizers\n        # and cannot be configured elsewhere, but it should default to False for Qwen2Tokenizer\n        return super().decode(\n            token_ids,\n            skip_special_tokens=skip_special_tokens,\n            clean_up_tokenization_spaces=clean_up_tokenization_spaces,\n            spaces_between_special_tokens=spaces_between_special_tokens,\n            **kwargs,\n        )\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer.save_vocabulary\n    def save_vocabulary(self, save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str]:\n        if not os.path.isdir(save_directory):\n            logger.error(f"Vocabulary path ({save_directory}) should be a directory")\n            return\n        vocab_file = os.path.join(\n            save_directory, (filename_prefix + "-" if filename_prefix else "") + VOCAB_FILES_NAMES["vocab_file"]\n        )\n        merge_file = os.path.join(\n            save_directory, (filename_prefix + "-" if filename_prefix else "") + VOCAB_FILES_NAMES["merges_file"]\n        )\n\n        with open(vocab_file, "w", encoding="utf-8") as f:\n            f.write(json.dumps(self.encoder, indent=2, sort_keys=True, ensure_ascii=False) + "\\n")\n\n        index = 0\n        with open(merge_file, "w", encoding="utf-8") as writer:\n            writer.write("#version: 0.2\\n")\n            for bpe_tokens, token_index in sorted(self.bpe_ranks.items(), key=lambda kv: kv[1]):\n                if index != token_index:\n                    logger.warning(\n                        f"Saving vocabulary to {merge_file}: BPE merge indices are not consecutive."\n                        " Please check that the tokenizer is not corrupted!"\n                    )\n                    index = token_index\n                writer.write(" ".join(bpe_tokens) + "\\n")\n                index += 1\n\n        return vocab_file, merge_file\n\n    def prepare_for_tokenization(self, text, **kwargs):\n        text = unicodedata.normalize("NFC", text)\n        return (text, kwargs)\n', 'modeling/qwen2/tokenization_qwen2_fast.py': '# Copyright 2024 The Qwen Team and The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Tokenization classes for Qwen2."""\n\nfrom typing import Optional, Tuple\n\nfrom transformers.tokenization_utils import AddedToken\nfrom transformers.tokenization_utils_fast import PreTrainedTokenizerFast\nfrom transformers.utils import logging\nfrom .tokenization_qwen2 import Qwen2Tokenizer\n\n\nlogger = logging.get_logger(__name__)\n\nVOCAB_FILES_NAMES = {\n    "vocab_file": "vocab.json",\n    "merges_file": "merges.txt",\n    "tokenizer_file": "tokenizer.json",\n}\n\n\nMAX_MODEL_INPUT_SIZES = {"qwen/qwen-tokenizer": 32768}\n\n\nclass Qwen2TokenizerFast(PreTrainedTokenizerFast):\n    """\n    Construct a "fast" Qwen2 tokenizer (backed by HuggingFace\'s *tokenizers* library). Based on byte-level\n    Byte-Pair-Encoding.\n\n    Same with GPT2Tokenizer, this tokenizer has been trained to treat spaces like parts of the tokens so a word will\n    be encoded differently whether it is at the beginning of the sentence (without space) or not:\n\n    ```python\n    >>> from transformers import Qwen2TokenizerFast\n\n    >>> tokenizer = Qwen2TokenizerFast.from_pretrained("Qwen/Qwen-tokenizer")\n    >>> tokenizer("Hello world")["input_ids"]\n    [9707, 1879]\n\n    >>> tokenizer(" Hello world")["input_ids"]\n    [21927, 1879]\n    ```\n    This is expected.\n\n    This tokenizer inherits from [`PreTrainedTokenizerFast`] which contains most of the main methods. Users should\n    refer to this superclass for more information regarding those methods.\n\n    Args:\n        vocab_file (`str`, *optional*):\n            Path to the vocabulary file.\n        merges_file (`str`, *optional*):\n            Path to the merges file.\n        tokenizer_file (`str`, *optional*):\n            Path to [tokenizers](https://github.com/huggingface/tokenizers) file (generally has a .json extension) that\n            contains everything needed to load the tokenizer.\n        unk_token (`str`, *optional*, defaults to `"<|endoftext|>"`):\n            The unknown token. A token that is not in the vocabulary cannot be converted to an ID and is set to be this\n            token instead. Not applicable to this tokenizer.\n        bos_token (`str`, *optional*):\n            The beginning of sequence token. Not applicable for this tokenizer.\n        eos_token (`str`, *optional*, defaults to `"<|endoftext|>"`):\n            The end of sequence token.\n        pad_token (`str`, *optional*, defaults to `"<|endoftext|>"`):\n            The token used for padding, for example when batching sequences of different lengths.\n    """\n\n    vocab_files_names = VOCAB_FILES_NAMES\n    model_input_names = ["input_ids", "attention_mask"]\n    slow_tokenizer_class = Qwen2Tokenizer\n\n    def __init__(\n        self,\n        vocab_file=None,\n        merges_file=None,\n        tokenizer_file=None,\n        unk_token="<|endoftext|>",\n        bos_token=None,\n        eos_token="<|endoftext|>",\n        pad_token="<|endoftext|>",\n        **kwargs,\n    ):\n        # We need to at least pass vocab_file and merges_file to base class\n        # in case a slow tokenizer needs to be initialized; other can be\n        # configured through files.\n        # following GPT2TokenizerFast, also adding unk_token, bos_token, and eos_token\n\n        bos_token = (\n            AddedToken(bos_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(bos_token, str)\n            else bos_token\n        )\n        eos_token = (\n            AddedToken(eos_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(eos_token, str)\n            else eos_token\n        )\n        unk_token = (\n            AddedToken(unk_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(unk_token, str)\n            else unk_token\n        )\n        pad_token = (\n            AddedToken(pad_token, lstrip=False, rstrip=False, special=True, normalized=False)\n            if isinstance(pad_token, str)\n            else pad_token\n        )\n\n        super().__init__(\n            vocab_file=vocab_file,\n            merges_file=merges_file,\n            tokenizer_file=tokenizer_file,\n            unk_token=unk_token,\n            bos_token=bos_token,\n            eos_token=eos_token,\n            pad_token=pad_token,\n            **kwargs,\n        )\n\n    # Copied from transformers.models.gpt2.tokenization_gpt2_fast.GPT2TokenizerFast.save_vocabulary\n    def save_vocabulary(self, save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str]:\n        files = self._tokenizer.model.save(save_directory, name=filename_prefix)\n        return tuple(files)\n', 'modeling/siglip/__init__.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\nfrom typing import TYPE_CHECKING\n\nfrom transformers.utils import (\n    OptionalDependencyNotAvailable,\n    _LazyModule,\n    is_sentencepiece_available,\n    is_torch_available,\n    is_vision_available,\n)\n\n\n_import_structure = {\n    "configuration_siglip": [\n        "SiglipConfig",\n        "SiglipTextConfig",\n        "SiglipVisionConfig",\n    ],\n    "processing_siglip": ["SiglipProcessor"],\n}\n\ntry:\n    if not is_sentencepiece_available():\n        raise OptionalDependencyNotAvailable()\nexcept OptionalDependencyNotAvailable:\n    pass\nelse:\n    _import_structure["tokenization_siglip"] = ["SiglipTokenizer"]\n\n\ntry:\n    if not is_vision_available():\n        raise OptionalDependencyNotAvailable()\nexcept OptionalDependencyNotAvailable:\n    pass\nelse:\n    _import_structure["image_processing_siglip"] = ["SiglipImageProcessor"]\n\ntry:\n    if not is_torch_available():\n        raise OptionalDependencyNotAvailable()\nexcept OptionalDependencyNotAvailable:\n    pass\nelse:\n    _import_structure["modeling_siglip"] = [\n        "SiglipModel",\n        "SiglipPreTrainedModel",\n        "SiglipTextModel",\n        "SiglipVisionModel",\n        "SiglipForImageClassification",\n    ]\n\n\nif TYPE_CHECKING:\n    from .configuration_siglip import (\n        SiglipConfig,\n        SiglipTextConfig,\n        SiglipVisionConfig,\n    )\n    from .processing_siglip import SiglipProcessor\n\n    try:\n        if not is_sentencepiece_available():\n            raise OptionalDependencyNotAvailable()\n    except OptionalDependencyNotAvailable:\n        pass\n    else:\n        from .tokenization_siglip import SiglipTokenizer\n\n    try:\n        if not is_vision_available():\n            raise OptionalDependencyNotAvailable()\n    except OptionalDependencyNotAvailable:\n        pass\n    else:\n        from .image_processing_siglip import SiglipImageProcessor\n\n    try:\n        if not is_torch_available():\n            raise OptionalDependencyNotAvailable()\n    except OptionalDependencyNotAvailable:\n        pass\n    else:\n        from .modeling_siglip import (\n            SiglipForImageClassification,\n            SiglipModel,\n            SiglipPreTrainedModel,\n            SiglipTextModel,\n            SiglipVisionModel,\n        )\n\n\nelse:\n    import sys\n\n    sys.modules[__name__] = _LazyModule(__name__, globals()["__file__"], _import_structure, module_spec=__spec__)\n', 'modeling/siglip/configuration_siglip.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Siglip model configuration"""\n\nimport os\nfrom typing import Union\n\nfrom transformers.configuration_utils import PretrainedConfig\nfrom transformers.utils import logging\n\n\nlogger = logging.get_logger(__name__)\n\n\nclass SiglipTextConfig(PretrainedConfig):\n    r"""\n    This is the configuration class to store the configuration of a [`SiglipTextModel`]. It is used to instantiate a\n    Siglip text encoder according to the specified arguments, defining the model architecture. Instantiating a\n    configuration with the defaults will yield a similar configuration to that of the text encoder of the Siglip\n    [google/siglip-base-patch16-224](https://huggingface.co/google/siglip-base-patch16-224) architecture.\n\n    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the\n    documentation from [`PretrainedConfig`] for more information.\n\n    Args:\n        vocab_size (`int`, *optional*, defaults to 32000):\n            Vocabulary size of the Siglip text model. Defines the number of different tokens that can be represented by\n            the `inputs_ids` passed when calling [`SiglipModel`].\n        hidden_size (`int`, *optional*, defaults to 768):\n            Dimensionality of the encoder layers and the pooler layer.\n        intermediate_size (`int`, *optional*, defaults to 3072):\n            Dimensionality of the "intermediate" (i.e., feed-forward) layer in the Transformer encoder.\n        num_hidden_layers (`int`, *optional*, defaults to 12):\n            Number of hidden layers in the Transformer encoder.\n        num_attention_heads (`int`, *optional*, defaults to 12):\n            Number of attention heads for each attention layer in the Transformer encoder.\n        max_position_embeddings (`int`, *optional*, defaults to 64):\n            The maximum sequence length that this model might ever be used with. Typically set this to something large\n            just in case (e.g., 512 or 1024 or 2048).\n        hidden_act (`str` or `function`, *optional*, defaults to `"gelu_pytorch_tanh"`):\n            The non-linear activation function (function or string) in the encoder and pooler. If string, `"gelu"`,\n            `"relu"`, `"selu"` and `"gelu_new"` `"quick_gelu"` are supported.\n        layer_norm_eps (`float`, *optional*, defaults to 1e-06):\n            The epsilon used by the layer normalization layers.\n        attention_dropout (`float`, *optional*, defaults to 0.0):\n            The dropout ratio for the attention probabilities.\n        pad_token_id (`int`, *optional*, defaults to 1):\n            The id of the padding token in the vocabulary.\n        bos_token_id (`int`, *optional*, defaults to 49406):\n            The id of the beginning-of-sequence token in the vocabulary.\n        eos_token_id (`int`, *optional*, defaults to 49407):\n            The id of the end-of-sequence token in the vocabulary.\n\n    Example:\n\n    ```python\n    >>> from transformers import SiglipTextConfig, SiglipTextModel\n\n    >>> # Initializing a SiglipTextConfig with google/siglip-base-patch16-224 style configuration\n    >>> configuration = SiglipTextConfig()\n\n    >>> # Initializing a SiglipTextModel (with random weights) from the google/siglip-base-patch16-224 style configuration\n    >>> model = SiglipTextModel(configuration)\n\n    >>> # Accessing the model configuration\n    >>> configuration = model.config\n    ```"""\n\n    model_type = "siglip_text_model"\n\n    def __init__(\n        self,\n        vocab_size=32000,\n        hidden_size=768,\n        intermediate_size=3072,\n        num_hidden_layers=12,\n        num_attention_heads=12,\n        max_position_embeddings=64,\n        hidden_act="gelu_pytorch_tanh",\n        layer_norm_eps=1e-6,\n        attention_dropout=0.0,\n        # This differs from `CLIPTokenizer`\'s default and from openai/siglip\n        # See https://github.com/huggingface/transformers/pull/24773#issuecomment-1632287538\n        pad_token_id=1,\n        bos_token_id=49406,\n        eos_token_id=49407,\n        **kwargs,\n    ):\n        super().__init__(pad_token_id=pad_token_id, bos_token_id=bos_token_id, eos_token_id=eos_token_id, **kwargs)\n\n        self.vocab_size = vocab_size\n        self.hidden_size = hidden_size\n        self.intermediate_size = intermediate_size\n        self.num_hidden_layers = num_hidden_layers\n        self.num_attention_heads = num_attention_heads\n        self.max_position_embeddings = max_position_embeddings\n        self.layer_norm_eps = layer_norm_eps\n        self.hidden_act = hidden_act\n        self.attention_dropout = attention_dropout\n\n    @classmethod\n    def from_pretrained(cls, pretrained_model_name_or_path: Union[str, os.PathLike], **kwargs) -> "PretrainedConfig":\n        cls._set_token_in_kwargs(kwargs)\n\n        config_dict, kwargs = cls.get_config_dict(pretrained_model_name_or_path, **kwargs)\n\n        # get the text config dict if we are loading from SiglipConfig\n        if config_dict.get("model_type") == "siglip":\n            config_dict = config_dict["text_config"]\n\n        if "model_type" in config_dict and hasattr(cls, "model_type") and config_dict["model_type"] != cls.model_type:\n            logger.warning(\n                f"You are using a model of type {config_dict[\'model_type\']} to instantiate a model of type "\n                f"{cls.model_type}. This is not supported for all configurations of models and can yield errors."\n            )\n\n        return cls.from_dict(config_dict, **kwargs)\n\n\nclass SiglipVisionConfig(PretrainedConfig):\n    r"""\n    This is the configuration class to store the configuration of a [`SiglipVisionModel`]. It is used to instantiate a\n    Siglip vision encoder according to the specified arguments, defining the model architecture. Instantiating a\n    configuration with the defaults will yield a similar configuration to that of the vision encoder of the Siglip\n    [google/siglip-base-patch16-224](https://huggingface.co/google/siglip-base-patch16-224) architecture.\n\n    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the\n    documentation from [`PretrainedConfig`] for more information.\n\n    Args:\n        hidden_size (`int`, *optional*, defaults to 768):\n            Dimensionality of the encoder layers and the pooler layer.\n        intermediate_size (`int`, *optional*, defaults to 3072):\n            Dimensionality of the "intermediate" (i.e., feed-forward) layer in the Transformer encoder.\n        num_hidden_layers (`int`, *optional*, defaults to 12):\n            Number of hidden layers in the Transformer encoder.\n        num_attention_heads (`int`, *optional*, defaults to 12):\n            Number of attention heads for each attention layer in the Transformer encoder.\n        num_channels (`int`, *optional*, defaults to 3):\n            Number of channels in the input images.\n        image_size (`int`, *optional*, defaults to 224):\n            The size (resolution) of each image.\n        patch_size (`int`, *optional*, defaults to 16):\n            The size (resolution) of each patch.\n        hidden_act (`str` or `function`, *optional*, defaults to `"gelu_pytorch_tanh"`):\n            The non-linear activation function (function or string) in the encoder and pooler. If string, `"gelu"`,\n            `"relu"`, `"selu"` and `"gelu_new"` `"quick_gelu"` are supported.\n        layer_norm_eps (`float`, *optional*, defaults to 1e-06):\n            The epsilon used by the layer normalization layers.\n        attention_dropout (`float`, *optional*, defaults to 0.0):\n            The dropout ratio for the attention probabilities.\n\n    Example:\n\n    ```python\n    >>> from transformers import SiglipVisionConfig, SiglipVisionModel\n\n    >>> # Initializing a SiglipVisionConfig with google/siglip-base-patch16-224 style configuration\n    >>> configuration = SiglipVisionConfig()\n\n    >>> # Initializing a SiglipVisionModel (with random weights) from the google/siglip-base-patch16-224 style configuration\n    >>> model = SiglipVisionModel(configuration)\n\n    >>> # Accessing the model configuration\n    >>> configuration = model.config\n    ```"""\n\n    model_type = "siglip_vision_model"\n\n    def __init__(\n        self,\n        hidden_size=768,\n        intermediate_size=3072,\n        num_hidden_layers=12,\n        num_attention_heads=12,\n        num_channels=3,\n        image_size=224,\n        patch_size=16,\n        hidden_act="gelu_pytorch_tanh",\n        layer_norm_eps=1e-6,\n        attention_dropout=0.0,\n        **kwargs,\n    ):\n        super().__init__(**kwargs)\n\n        self.hidden_size = hidden_size\n        self.intermediate_size = intermediate_size\n        self.num_hidden_layers = num_hidden_layers\n        self.num_attention_heads = num_attention_heads\n        self.num_channels = num_channels\n        self.patch_size = patch_size\n        self.image_size = image_size\n        self.attention_dropout = attention_dropout\n        self.layer_norm_eps = layer_norm_eps\n        self.hidden_act = hidden_act\n\n    @classmethod\n    def from_pretrained(cls, pretrained_model_name_or_path: Union[str, os.PathLike], **kwargs) -> "PretrainedConfig":\n        cls._set_token_in_kwargs(kwargs)\n\n        config_dict, kwargs = cls.get_config_dict(pretrained_model_name_or_path, **kwargs)\n\n        # get the vision config dict if we are loading from SiglipConfig\n        if config_dict.get("model_type") == "siglip":\n            config_dict = config_dict["vision_config"]\n\n        if "model_type" in config_dict and hasattr(cls, "model_type") and config_dict["model_type"] != cls.model_type:\n            logger.warning(\n                f"You are using a model of type {config_dict[\'model_type\']} to instantiate a model of type "\n                f"{cls.model_type}. This is not supported for all configurations of models and can yield errors."\n            )\n\n        return cls.from_dict(config_dict, **kwargs)\n\n\nclass SiglipConfig(PretrainedConfig):\n    r"""\n    [`SiglipConfig`] is the configuration class to store the configuration of a [`SiglipModel`]. It is used to\n    instantiate a Siglip model according to the specified arguments, defining the text model and vision model configs.\n    Instantiating a configuration with the defaults will yield a similar configuration to that of the Siglip\n    [google/siglip-base-patch16-224](https://huggingface.co/google/siglip-base-patch16-224) architecture.\n\n    Configuration objects inherit from [`PretrainedConfig`] and can be used to control the model outputs. Read the\n    documentation from [`PretrainedConfig`] for more information.\n\n    Args:\n        text_config (`dict`, *optional*):\n            Dictionary of configuration options used to initialize [`SiglipTextConfig`].\n        vision_config (`dict`, *optional*):\n            Dictionary of configuration options used to initialize [`SiglipVisionConfig`].\n        kwargs (*optional*):\n            Dictionary of keyword arguments.\n\n    Example:\n\n    ```python\n    >>> from transformers import SiglipConfig, SiglipModel\n\n    >>> # Initializing a SiglipConfig with google/siglip-base-patch16-224 style configuration\n    >>> configuration = SiglipConfig()\n\n    >>> # Initializing a SiglipModel (with random weights) from the google/siglip-base-patch16-224 style configuration\n    >>> model = SiglipModel(configuration)\n\n    >>> # Accessing the model configuration\n    >>> configuration = model.config\n\n    >>> # We can also initialize a SiglipConfig from a SiglipTextConfig and a SiglipVisionConfig\n    >>> from transformers import SiglipTextConfig, SiglipVisionConfig\n\n    >>> # Initializing a SiglipText and SiglipVision configuration\n    >>> config_text = SiglipTextConfig()\n    >>> config_vision = SiglipVisionConfig()\n\n    >>> config = SiglipConfig.from_text_vision_configs(config_text, config_vision)\n    ```"""\n\n    model_type = "siglip"\n\n    def __init__(self, text_config=None, vision_config=None, **kwargs):\n        super().__init__(**kwargs)\n\n        if text_config is None:\n            text_config = {}\n            logger.info("`text_config` is `None`. Initializing the `SiglipTextConfig` with default values.")\n\n        if vision_config is None:\n            vision_config = {}\n            logger.info("`vision_config` is `None`. initializing the `SiglipVisionConfig` with default values.")\n\n        self.text_config = SiglipTextConfig(**text_config)\n        self.vision_config = SiglipVisionConfig(**vision_config)\n\n        self.initializer_factor = 1.0\n\n    @classmethod\n    def from_text_vision_configs(cls, text_config: SiglipTextConfig, vision_config: SiglipVisionConfig, **kwargs):\n        r"""\n        Instantiate a [`SiglipConfig`] (or a derived class) from siglip text model configuration and siglip vision\n        model configuration.\n\n        Returns:\n            [`SiglipConfig`]: An instance of a configuration object\n        """\n\n        return cls(text_config=text_config.to_dict(), vision_config=vision_config.to_dict(), **kwargs)\n', 'modeling/siglip/convert_siglip_to_hf.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Convert SigLIP checkpoints from the original repository.\n\nURL: https://github.com/google-research/big_vision/tree/main\n"""\n\nimport argparse\nimport collections\nfrom pathlib import Path\n\nimport numpy as np\nimport requests\nimport torch\nfrom huggingface_hub import hf_hub_download\nfrom numpy import load\nfrom PIL import Image\n\nfrom transformers import SiglipConfig, SiglipImageProcessor, SiglipModel, SiglipProcessor, SiglipTokenizer\nfrom transformers.utils import logging\n\n\nlogging.set_verbosity_info()\nlogger = logging.get_logger(__name__)\n\n\nmodel_name_to_checkpoint = {\n    # base checkpoints\n    "siglip-base-patch16-224": "/Users/nielsrogge/Documents/SigLIP/webli_en_b16_224_63724782.npz",\n    "siglip-base-patch16-256": "/Users/nielsrogge/Documents/SigLIP/webli_en_b16_256_60500360.npz",\n    "siglip-base-patch16-384": "/Users/nielsrogge/Documents/SigLIP/webli_en_b16_384_68578854.npz",\n    "siglip-base-patch16-512": "/Users/nielsrogge/Documents/SigLIP/webli_en_b16_512_68580893.npz",\n    # large checkpoints\n    "siglip-large-patch16-256": "/Users/nielsrogge/Documents/SigLIP/webli_en_l16_256_60552751.npz",\n    "siglip-large-patch16-384": "/Users/nielsrogge/Documents/SigLIP/webli_en_l16_384_63634585.npz",\n    # multilingual checkpoint\n    "siglip-base-patch16-256-i18n": "/Users/nielsrogge/Documents/SigLIP/webli_i18n_b16_256_66117334.npz",\n    # so400m checkpoints\n    "siglip-so400m-patch14-384": "/Users/nielsrogge/Documents/SigLIP/webli_en_so400m_384_58765454.npz",\n}\n\nmodel_name_to_image_size = {\n    "siglip-base-patch16-224": 224,\n    "siglip-base-patch16-256": 256,\n    "siglip-base-patch16-384": 384,\n    "siglip-base-patch16-512": 512,\n    "siglip-large-patch16-256": 256,\n    "siglip-large-patch16-384": 384,\n    "siglip-base-patch16-256-i18n": 256,\n    "siglip-so400m-patch14-384": 384,\n}\n\n\ndef get_siglip_config(model_name):\n    config = SiglipConfig()\n\n    vocab_size = 250000 if "i18n" in model_name else 32000\n    image_size = model_name_to_image_size[model_name]\n    patch_size = 16 if "patch16" in model_name else 14\n\n    # size of the architecture\n    config.vision_config.image_size = image_size\n    config.vision_config.patch_size = patch_size\n    config.text_config.vocab_size = vocab_size\n\n    if "base" in model_name:\n        pass\n    elif "large" in model_name:\n        config.text_config.hidden_size = 1024\n        config.text_config.intermediate_size = 4096\n        config.text_config.num_hidden_layers = 24\n        config.text_config.num_attention_heads = 16\n        config.vision_config.hidden_size = 1024\n        config.vision_config.intermediate_size = 4096\n        config.vision_config.num_hidden_layers = 24\n        config.vision_config.num_attention_heads = 16\n    elif "so400m" in model_name:\n        config.text_config.hidden_size = 1152\n        config.text_config.intermediate_size = 4304\n        config.text_config.num_hidden_layers = 27\n        config.text_config.num_attention_heads = 16\n        config.vision_config.hidden_size = 1152\n        config.vision_config.intermediate_size = 4304\n        config.vision_config.num_hidden_layers = 27\n        config.vision_config.num_attention_heads = 16\n    else:\n        raise ValueError("Model not supported")\n\n    return config\n\n\ndef create_rename_keys(config):\n    rename_keys = []\n    # fmt: off\n\n    # vision encoder\n\n    rename_keys.append(("params/img/embedding/kernel", "vision_model.embeddings.patch_embedding.weight"))\n    rename_keys.append(("params/img/embedding/bias", "vision_model.embeddings.patch_embedding.bias"))\n    rename_keys.append(("params/img/pos_embedding", "vision_model.embeddings.position_embedding.weight"))\n\n    for i in range(config.vision_config.num_hidden_layers):\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/LayerNorm_0/scale", f"vision_model.encoder.layers.{i}.layer_norm1.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/LayerNorm_0/bias", f"vision_model.encoder.layers.{i}.layer_norm1.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/LayerNorm_1/scale", f"vision_model.encoder.layers.{i}.layer_norm2.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/LayerNorm_1/bias", f"vision_model.encoder.layers.{i}.layer_norm2.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MlpBlock_0/Dense_0/kernel", f"vision_model.encoder.layers.{i}.mlp.fc1.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MlpBlock_0/Dense_0/bias", f"vision_model.encoder.layers.{i}.mlp.fc1.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MlpBlock_0/Dense_1/kernel", f"vision_model.encoder.layers.{i}.mlp.fc2.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MlpBlock_0/Dense_1/bias", f"vision_model.encoder.layers.{i}.mlp.fc2.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/key/kernel", f"vision_model.encoder.layers.{i}.self_attn.k_proj.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/key/bias", f"vision_model.encoder.layers.{i}.self_attn.k_proj.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/value/kernel", f"vision_model.encoder.layers.{i}.self_attn.v_proj.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/value/bias", f"vision_model.encoder.layers.{i}.self_attn.v_proj.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/query/kernel", f"vision_model.encoder.layers.{i}.self_attn.q_proj.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/query/bias", f"vision_model.encoder.layers.{i}.self_attn.q_proj.bias"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/out/kernel", f"vision_model.encoder.layers.{i}.self_attn.out_proj.weight"))\n        rename_keys.append((f"params/img/Transformer/encoderblock_{i}/MultiHeadDotProductAttention_0/out/bias", f"vision_model.encoder.layers.{i}.self_attn.out_proj.bias"))\n\n    rename_keys.append(("params/img/Transformer/encoder_norm/scale", "vision_model.post_layernorm.weight"))\n    rename_keys.append(("params/img/Transformer/encoder_norm/bias", "vision_model.post_layernorm.bias"))\n\n    rename_keys.append(("params/img/MAPHead_0/probe", "vision_model.head.probe"))\n    rename_keys.append(("params/img/MAPHead_0/LayerNorm_0/scale", "vision_model.head.layernorm.weight"))\n    rename_keys.append(("params/img/MAPHead_0/LayerNorm_0/bias", "vision_model.head.layernorm.bias"))\n    rename_keys.append(("params/img/MAPHead_0/MlpBlock_0/Dense_0/kernel", "vision_model.head.mlp.fc1.weight"))\n    rename_keys.append(("params/img/MAPHead_0/MlpBlock_0/Dense_0/bias", "vision_model.head.mlp.fc1.bias"))\n    rename_keys.append(("params/img/MAPHead_0/MlpBlock_0/Dense_1/kernel", "vision_model.head.mlp.fc2.weight"))\n    rename_keys.append(("params/img/MAPHead_0/MlpBlock_0/Dense_1/bias", "vision_model.head.mlp.fc2.bias"))\n    rename_keys.append(("params/img/MAPHead_0/MultiHeadDotProductAttention_0/out/kernel", "vision_model.head.attention.out_proj.weight"))\n    rename_keys.append(("params/img/MAPHead_0/MultiHeadDotProductAttention_0/out/bias", "vision_model.head.attention.out_proj.bias"))\n\n    # text encoder\n\n    rename_keys.append(("params/txt/Embed_0/embedding", "text_model.embeddings.token_embedding.weight"))\n    rename_keys.append(("params/txt/pos_embedding", "text_model.embeddings.position_embedding.weight"))\n\n    for i in range(config.text_config.num_hidden_layers):\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/LayerNorm_0/scale", f"text_model.encoder.layers.{i}.layer_norm1.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/LayerNorm_0/bias", f"text_model.encoder.layers.{i}.layer_norm1.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/LayerNorm_1/scale", f"text_model.encoder.layers.{i}.layer_norm2.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/LayerNorm_1/bias", f"text_model.encoder.layers.{i}.layer_norm2.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MlpBlock_0/Dense_0/kernel", f"text_model.encoder.layers.{i}.mlp.fc1.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MlpBlock_0/Dense_0/bias", f"text_model.encoder.layers.{i}.mlp.fc1.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MlpBlock_0/Dense_1/kernel", f"text_model.encoder.layers.{i}.mlp.fc2.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MlpBlock_0/Dense_1/bias", f"text_model.encoder.layers.{i}.mlp.fc2.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/key/kernel", f"text_model.encoder.layers.{i}.self_attn.k_proj.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/key/bias", f"text_model.encoder.layers.{i}.self_attn.k_proj.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/value/kernel", f"text_model.encoder.layers.{i}.self_attn.v_proj.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/value/bias", f"text_model.encoder.layers.{i}.self_attn.v_proj.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/query/kernel", f"text_model.encoder.layers.{i}.self_attn.q_proj.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/query/bias", f"text_model.encoder.layers.{i}.self_attn.q_proj.bias"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/out/kernel", f"text_model.encoder.layers.{i}.self_attn.out_proj.weight"))\n        rename_keys.append((f"params/txt/Encoder_0/encoderblock_{i}/MultiHeadDotProductAttention_0/out/bias", f"text_model.encoder.layers.{i}.self_attn.out_proj.bias"))\n\n    rename_keys.append(("params/txt/Encoder_0/encoder_norm/scale", "text_model.final_layer_norm.weight"))\n    rename_keys.append(("params/txt/Encoder_0/encoder_norm/bias", "text_model.final_layer_norm.bias"))\n    rename_keys.append(("params/txt/head/kernel", "text_model.head.weight"))\n    rename_keys.append(("params/txt/head/bias", "text_model.head.bias"))\n\n    # learned temperature and bias\n    rename_keys.append(("params/t", "logit_scale"))\n    rename_keys.append(("params/b", "logit_bias"))\n\n    # fmt: on\n    return rename_keys\n\n\ndef rename_key(dct, old, new, config):\n    val = dct.pop(old)\n\n    if ("out_proj" in new or "v_proj" in new or "k_proj" in new or "q_proj" in new) and "vision" in new:\n        val = val.reshape(-1, config.vision_config.hidden_size)\n    if ("out_proj" in new or "v_proj" in new or "k_proj" in new or "q_proj" in new) and "text" in new:\n        val = val.reshape(-1, config.text_config.hidden_size)\n\n    if "patch_embedding.weight" in new:\n        val = val.transpose(3, 2, 0, 1)\n    elif new.endswith("weight") and "position_embedding" not in new and "token_embedding" not in new:\n        val = val.T\n\n    if "position_embedding" in new and "vision" in new:\n        val = val.reshape(-1, config.vision_config.hidden_size)\n    if "position_embedding" in new and "text" in new:\n        val = val.reshape(-1, config.text_config.hidden_size)\n\n    if new.endswith("bias"):\n        val = val.reshape(-1)\n\n    dct[new] = torch.from_numpy(val)\n\n\ndef read_in_q_k_v_head(state_dict, config):\n    # read in individual input projection layers\n    key_proj_weight = (\n        state_dict.pop("params/img/MAPHead_0/MultiHeadDotProductAttention_0/key/kernel")\n        .reshape(-1, config.vision_config.hidden_size)\n        .T\n    )\n    key_proj_bias = state_dict.pop("params/img/MAPHead_0/MultiHeadDotProductAttention_0/key/bias").reshape(-1)\n    value_proj_weight = (\n        state_dict.pop("params/img/MAPHead_0/MultiHeadDotProductAttention_0/value/kernel")\n        .reshape(-1, config.vision_config.hidden_size)\n        .T\n    )\n    value_proj_bias = state_dict.pop("params/img/MAPHead_0/MultiHeadDotProductAttention_0/value/bias").reshape(-1)\n    query_proj_weight = (\n        state_dict.pop("params/img/MAPHead_0/MultiHeadDotProductAttention_0/query/kernel")\n        .reshape(-1, config.vision_config.hidden_size)\n        .T\n    )\n    query_proj_bias = state_dict.pop("params/img/MAPHead_0/MultiHeadDotProductAttention_0/query/bias").reshape(-1)\n\n    # next, add them to the state dict as a single matrix + vector\n    state_dict["vision_model.head.attention.in_proj_weight"] = torch.from_numpy(\n        np.concatenate([query_proj_weight, key_proj_weight, value_proj_weight], axis=0)\n    )\n    state_dict["vision_model.head.attention.in_proj_bias"] = torch.from_numpy(\n        np.concatenate([query_proj_bias, key_proj_bias, value_proj_bias], axis=0)\n    )\n\n\n# We will verify our results on an image of cute cats\ndef prepare_img():\n    url = "http://images.cocodataset.org/val2017/000000039769.jpg"\n    image = Image.open(requests.get(url, stream=True).raw)\n    return image\n\n\ndef flatten_nested_dict(params, parent_key="", sep="/"):\n    items = []\n\n    for k, v in params.items():\n        new_key = parent_key + sep + k if parent_key else k\n\n        if isinstance(v, collections.abc.MutableMapping):\n            items.extend(flatten_nested_dict(v, new_key, sep=sep).items())\n        else:\n            items.append((new_key, v))\n    return dict(items)\n\n\n@torch.no_grad()\ndef convert_siglip_checkpoint(model_name, pytorch_dump_folder_path, verify_logits=True, push_to_hub=False):\n    """\n    Copy/paste/tweak model\'s weights to our SigLIP structure.\n    """\n\n    # define default SigLIP configuration\n    config = get_siglip_config(model_name)\n\n    # get checkpoint\n    checkpoint = model_name_to_checkpoint[model_name]\n\n    # get vocab file\n    if "i18n" in model_name:\n        vocab_file = "/Users/nielsrogge/Documents/SigLIP/multilingual_vocab/sentencepiece.model"\n    else:\n        vocab_file = "/Users/nielsrogge/Documents/SigLIP/english_vocab/sentencepiece.model"\n\n    # load original state dict\n    data = load(checkpoint)\n    state_dict = flatten_nested_dict(data)\n\n    # remove and rename some keys\n    rename_keys = create_rename_keys(config)\n    for src, dest in rename_keys:\n        rename_key(state_dict, src, dest, config)\n\n    # qkv matrices of attention pooling head need special treatment\n    read_in_q_k_v_head(state_dict, config)\n\n    # load HuggingFace model\n    model = SiglipModel(config).eval()\n    model.load_state_dict(state_dict)\n\n    # create processor\n    # important: make tokenizer not return attention_mask since original one doesn\'t require it\n    image_size = config.vision_config.image_size\n    size = {"height": image_size, "width": image_size}\n    image_processor = SiglipImageProcessor(size=size)\n    tokenizer = SiglipTokenizer(vocab_file=vocab_file, model_input_names=["input_ids"])\n    processor = SiglipProcessor(image_processor=image_processor, tokenizer=tokenizer)\n\n    # verify on dummy images and texts\n    url_1 = "https://cdn.openai.com/multimodal-neurons/assets/apple/apple-ipod.jpg"\n    image_1 = Image.open(requests.get(url_1, stream=True).raw).convert("RGB")\n    url_2 = "https://cdn.openai.com/multimodal-neurons/assets/apple/apple-blank.jpg"\n    image_2 = Image.open(requests.get(url_2, stream=True).raw).convert("RGB")\n    texts = ["an apple", "a picture of an apple"]\n\n    inputs = processor(images=[image_1, image_2], text=texts, return_tensors="pt", padding="max_length")\n\n    # verify input_ids against original ones\n    if image_size == 224:\n        filename = "siglip_pixel_values.pt"\n    elif image_size == 256:\n        filename = "siglip_pixel_values_256.pt"\n    elif image_size == 384:\n        filename = "siglip_pixel_values_384.pt"\n    elif image_size == 512:\n        filename = "siglip_pixel_values_512.pt"\n    else:\n        raise ValueError("Image size not supported")\n\n    filepath = hf_hub_download(repo_id="nielsr/test-image", filename=filename, repo_type="dataset")\n    original_pixel_values = torch.load(filepath)\n    filepath = hf_hub_download(repo_id="nielsr/test-image", filename="siglip_input_ids.pt", repo_type="dataset")\n    original_input_ids = torch.load(filepath)\n\n    if "i18n" not in model_name:\n        assert inputs.input_ids.tolist() == original_input_ids.tolist()\n\n    print("Mean of original pixel values:", original_pixel_values.mean())\n    print("Mean of new pixel values:", inputs.pixel_values.mean())\n\n    # note: we\'re testing with original pixel values here since we don\'t have exact pixel values\n    with torch.no_grad():\n        outputs = model(input_ids=inputs.input_ids, pixel_values=original_pixel_values)\n\n    # with torch.no_grad():\n    #     outputs = model(input_ids=inputs.input_ids, pixel_values=inputs.pixel_values)\n\n    print(outputs.logits_per_image[:3, :3])\n\n    probs = torch.sigmoid(outputs.logits_per_image)  # these are the probabilities\n    print(f"{probs[0][0]:.1%} that image 0 is \'{texts[0]}\'")\n    print(f"{probs[0][1]:.1%} that image 0 is \'{texts[1]}\'")\n\n    if verify_logits:\n        if model_name == "siglip-base-patch16-224":\n            expected_slice = torch.tensor(\n                [[-2.9621, -2.1672], [-0.2713, 0.2910]],\n            )\n        elif model_name == "siglip-base-patch16-256":\n            expected_slice = torch.tensor(\n                [[-3.1146, -1.9894], [-0.7312, 0.6387]],\n            )\n        elif model_name == "siglip-base-patch16-384":\n            expected_slice = torch.tensor(\n                [[-2.8098, -2.1891], [-0.4242, 0.4102]],\n            )\n        elif model_name == "siglip-base-patch16-512":\n            expected_slice = torch.tensor(\n                [[-2.7899, -2.2668], [-0.4295, -0.0735]],\n            )\n        elif model_name == "siglip-large-patch16-256":\n            expected_slice = torch.tensor(\n                [[-1.5827, -0.5801], [-0.9153, 0.1363]],\n            )\n        elif model_name == "siglip-large-patch16-384":\n            expected_slice = torch.tensor(\n                [[-2.1523, -0.2899], [-0.2959, 0.7884]],\n            )\n        elif model_name == "siglip-so400m-patch14-384":\n            expected_slice = torch.tensor([[-1.2441, -0.6649], [-0.7060, 0.7374]])\n        elif model_name == "siglip-base-patch16-256-i18n":\n            expected_slice = torch.tensor(\n                [[-0.9064, 0.1073], [-0.0299, 0.5304]],\n            )\n\n        assert torch.allclose(outputs.logits_per_image[:3, :3], expected_slice, atol=1e-4)\n        print("Looks ok!")\n\n    if pytorch_dump_folder_path is not None:\n        Path(pytorch_dump_folder_path).mkdir(exist_ok=True)\n        print(f"Saving model {model_name} to {pytorch_dump_folder_path}")\n        model.save_pretrained(pytorch_dump_folder_path)\n        print(f"Saving processor to {pytorch_dump_folder_path}")\n        processor.save_pretrained(pytorch_dump_folder_path)\n\n    if push_to_hub:\n        model.push_to_hub(f"nielsr/{model_name}")\n        processor.push_to_hub(f"nielsr/{model_name}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    # Required parameters\n    parser.add_argument(\n        "--model_name",\n        default="siglip-base-patch16-224",\n        type=str,\n        choices=model_name_to_checkpoint.keys(),\n        help="Name of the model you\'d like to convert.",\n    )\n    parser.add_argument(\n        "--pytorch_dump_folder_path", default=None, type=str, help="Path to the output PyTorch model directory."\n    )\n    parser.add_argument(\n        "--verify_logits",\n        action="store_false",\n        help="Whether to verify logits against the original implementation.",\n    )\n    parser.add_argument(\n        "--push_to_hub", action="store_true", help="Whether or not to push the converted model to the 🤗 hub."\n    )\n\n    args = parser.parse_args()\n    convert_siglip_checkpoint(args.model_name, args.pytorch_dump_folder_path, args.verify_logits, args.push_to_hub)\n', 'modeling/siglip/image_processing_siglip.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Image processor class for SigLIP."""\n\nfrom typing import Dict, List, Optional, Union\n\nfrom transformers.image_processing_utils import BaseImageProcessor, BatchFeature, get_size_dict\nfrom transformers.image_transforms import (\n    convert_to_rgb,\n    resize,\n    to_channel_dimension_format,\n)\nfrom transformers.image_utils import (\n    IMAGENET_STANDARD_MEAN,\n    IMAGENET_STANDARD_STD,\n    ChannelDimension,\n    ImageInput,\n    PILImageResampling,\n    infer_channel_dimension_format,\n    is_scaled_image,\n    make_list_of_images,\n    to_numpy_array,\n    valid_images,\n    validate_preprocess_arguments,\n)\nfrom transformers.utils import TensorType, filter_out_non_signature_kwargs, is_vision_available, logging\n\n\nlogger = logging.get_logger(__name__)\n\n\nif is_vision_available():\n    import PIL\n\n\nclass SiglipImageProcessor(BaseImageProcessor):\n    r"""\n    Constructs a SigLIP image processor.\n\n    Args:\n        do_resize (`bool`, *optional*, defaults to `True`):\n            Whether to resize the image\'s (height, width) dimensions to the specified `size`. Can be overridden by\n            `do_resize` in the `preprocess` method.\n        size (`Dict[str, int]` *optional*, defaults to `{"height": 224, "width": 224}`):\n            Size of the image after resizing. Can be overridden by `size` in the `preprocess` method.\n        resample (`PILImageResampling`, *optional*, defaults to `Resampling.BICUBIC`):\n            Resampling filter to use if resizing the image. Can be overridden by `resample` in the `preprocess` method.\n        do_rescale (`bool`, *optional*, defaults to `True`):\n            Whether to rescale the image by the specified scale `rescale_factor`. Can be overridden by `do_rescale` in\n            the `preprocess` method.\n        rescale_factor (`int` or `float`, *optional*, defaults to `1/255`):\n            Scale factor to use if rescaling the image. Can be overridden by `rescale_factor` in the `preprocess`\n            method.\n        do_normalize (`bool`, *optional*, defaults to `True`):\n            Whether to normalize the image by the specified mean and standard deviation. Can be overridden by\n            `do_normalize` in the `preprocess` method.\n        image_mean (`float` or `List[float]`, *optional*, defaults to `[0.5, 0.5, 0.5]`):\n            Mean to use if normalizing the image. This is a float or list of floats the length of the number of\n            channels in the image. Can be overridden by the `image_mean` parameter in the `preprocess` method.\n        image_std (`float` or `List[float]`, *optional*, defaults to `[0.5, 0.5, 0.5]`):\n            Standard deviation to use if normalizing the image. This is a float or list of floats the length of the\n            number of channels in the image. Can be overridden by the `image_std` parameter in the `preprocess` method.\n            Can be overridden by the `image_std` parameter in the `preprocess` method.\n        do_convert_rgb (`bool`, *optional*, defaults to `True`):\n            Whether to convert the image to RGB.\n    """\n\n    model_input_names = ["pixel_values"]\n\n    def __init__(\n        self,\n        do_resize: bool = True,\n        size: Dict[str, int] = None,\n        resample: PILImageResampling = PILImageResampling.BICUBIC,\n        do_rescale: bool = True,\n        rescale_factor: Union[int, float] = 1 / 255,\n        do_normalize: bool = True,\n        image_mean: Optional[Union[float, List[float]]] = None,\n        image_std: Optional[Union[float, List[float]]] = None,\n        do_convert_rgb: bool = None,\n        **kwargs,\n    ) -> None:\n        super().__init__(**kwargs)\n        size = size if size is not None else {"height": 224, "width": 224}\n        image_mean = image_mean if image_mean is not None else IMAGENET_STANDARD_MEAN\n        image_std = image_std if image_std is not None else IMAGENET_STANDARD_STD\n\n        self.do_resize = do_resize\n        self.size = size\n        self.resample = resample\n        self.do_rescale = do_rescale\n        self.rescale_factor = rescale_factor\n        self.do_normalize = do_normalize\n        self.image_mean = image_mean\n        self.image_std = image_std\n        self.do_convert_rgb = do_convert_rgb\n\n    @filter_out_non_signature_kwargs()\n    def preprocess(\n        self,\n        images: ImageInput,\n        do_resize: bool = None,\n        size: Dict[str, int] = None,\n        resample: PILImageResampling = None,\n        do_rescale: bool = None,\n        rescale_factor: float = None,\n        do_normalize: bool = None,\n        image_mean: Optional[Union[float, List[float]]] = None,\n        image_std: Optional[Union[float, List[float]]] = None,\n        return_tensors: Optional[Union[str, TensorType]] = None,\n        data_format: Optional[ChannelDimension] = ChannelDimension.FIRST,\n        input_data_format: Optional[Union[str, ChannelDimension]] = None,\n        do_convert_rgb: bool = None,\n    ) -> PIL.Image.Image:\n        """\n        Preprocess an image or batch of images.\n\n        Args:\n            images (`ImageInput`):\n                Image to preprocess. Expects a single or batch of images with pixel values ranging from 0 to 255. If\n                passing in images with pixel values between 0 and 1, set `do_rescale=False`.\n            do_resize (`bool`, *optional*, defaults to `self.do_resize`):\n                Whether to resize the image.\n            size (`Dict[str, int]`, *optional*, defaults to `self.size`):\n                Size of the image after resizing.\n            resample (`int`, *optional*, defaults to `self.resample`):\n                Resampling filter to use if resizing the image. This can be one of the enum `PILImageResampling`. Only\n                has an effect if `do_resize` is set to `True`.\n            do_rescale (`bool`, *optional*, defaults to `self.do_rescale`):\n                Whether to rescale the image.\n            rescale_factor (`float`, *optional*, defaults to `self.rescale_factor`):\n                Rescale factor to rescale the image by if `do_rescale` is set to `True`.\n            do_normalize (`bool`, *optional*, defaults to `self.do_normalize`):\n                Whether to normalize the image.\n            image_mean (`float` or `List[float]`, *optional*, defaults to `self.image_mean`):\n                Image mean to use for normalization. Only has an effect if `do_normalize` is set to `True`.\n            image_std (`float` or `List[float]`, *optional*, defaults to `self.image_std`):\n                Image standard deviation to use for normalization. Only has an effect if `do_normalize` is set to\n                `True`.\n            return_tensors (`str` or `TensorType`, *optional*):\n                The type of tensors to return. Can be one of:\n                - Unset: Return a list of `np.ndarray`.\n                - `TensorType.TENSORFLOW` or `\'tf\'`: Return a batch of type `tf.Tensor`.\n                - `TensorType.PYTORCH` or `\'pt\'`: Return a batch of type `torch.Tensor`.\n                - `TensorType.NUMPY` or `\'np\'`: Return a batch of type `np.ndarray`.\n                - `TensorType.JAX` or `\'jax\'`: Return a batch of type `jax.numpy.ndarray`.\n            data_format (`ChannelDimension` or `str`, *optional*, defaults to `ChannelDimension.FIRST`):\n                The channel dimension format for the output image. Can be one of:\n                - `"channels_first"` or `ChannelDimension.FIRST`: image in (num_channels, height, width) format.\n                - `"channels_last"` or `ChannelDimension.LAST`: image in (height, width, num_channels) format.\n                - Unset: Use the channel dimension format of the input image.\n            input_data_format (`ChannelDimension` or `str`, *optional*):\n                The channel dimension format for the input image. If unset, the channel dimension format is inferred\n                from the input image. Can be one of:\n                - `"channels_first"` or `ChannelDimension.FIRST`: image in (num_channels, height, width) format.\n                - `"channels_last"` or `ChannelDimension.LAST`: image in (height, width, num_channels) format.\n                - `"none"` or `ChannelDimension.NONE`: image in (height, width) format.\n            do_convert_rgb (`bool`, *optional*, defaults to `self.do_convert_rgb`):\n                Whether to convert the image to RGB.\n        """\n        do_resize = do_resize if do_resize is not None else self.do_resize\n        size = size if size is not None else self.size\n        size = get_size_dict(size, param_name="size", default_to_square=False)\n        resample = resample if resample is not None else self.resample\n        do_rescale = do_rescale if do_rescale is not None else self.do_rescale\n        rescale_factor = rescale_factor if rescale_factor is not None else self.rescale_factor\n        do_normalize = do_normalize if do_normalize is not None else self.do_normalize\n        image_mean = image_mean if image_mean is not None else self.image_mean\n        image_std = image_std if image_std is not None else self.image_std\n        do_convert_rgb = do_convert_rgb if do_convert_rgb is not None else self.do_convert_rgb\n\n        images = make_list_of_images(images)\n\n        if not valid_images(images):\n            raise ValueError(\n                "Invalid image type. Must be of type PIL.Image.Image, numpy.ndarray, "\n                "torch.Tensor, tf.Tensor or jax.ndarray."\n            )\n        validate_preprocess_arguments(\n            do_rescale=do_rescale,\n            rescale_factor=rescale_factor,\n            do_normalize=do_normalize,\n            image_mean=image_mean,\n            image_std=image_std,\n            do_resize=do_resize,\n            size=size,\n            resample=resample,\n        )\n        # All transformations expect numpy arrays.\n        images = [to_numpy_array(image) for image in images]\n\n        if do_convert_rgb:\n            images = [convert_to_rgb(image) for image in images]\n\n        if is_scaled_image(images[0]) and do_rescale:\n            logger.warning_once(\n                "It looks like you are trying to rescale already rescaled images. If the input"\n                " images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again."\n            )\n\n        if input_data_format is None:\n            # We assume that all images have the same channel dimension format.\n            input_data_format = infer_channel_dimension_format(images[0])\n\n        if do_resize:\n            height, width = size["height"], size["width"]\n            images = [\n                resize(image=image, size=(height, width), resample=resample, input_data_format=input_data_format)\n                for image in images\n            ]\n\n        if do_rescale:\n            images = [\n                self.rescale(image=image, scale=rescale_factor, input_data_format=input_data_format)\n                for image in images\n            ]\n\n        if do_normalize:\n            images = [\n                self.normalize(image=image, mean=image_mean, std=image_std, input_data_format=input_data_format)\n                for image in images\n            ]\n\n        images = [\n            to_channel_dimension_format(image, data_format, input_channel_dim=input_data_format) for image in images\n        ]\n\n        data = {"pixel_values": images}\n        return BatchFeature(data=data, tensor_type=return_tensors)\n', 'modeling/siglip/modeling_siglip.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""PyTorch Siglip model."""\n\nimport math\nimport warnings\nfrom dataclasses import dataclass\nfrom typing import Any, Optional, Tuple, Union\n\nimport numpy as np\nimport torch\nimport torch.utils.checkpoint\nfrom torch import nn\nfrom torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss\nfrom torch.nn.init import _calculate_fan_in_and_fan_out\n\nfrom transformers.activations import ACT2FN\nfrom transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask\nfrom transformers.modeling_outputs import BaseModelOutput, BaseModelOutputWithPooling, ImageClassifierOutput\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.utils import (\n    ModelOutput,\n    add_start_docstrings,\n    add_start_docstrings_to_model_forward,\n    is_flash_attn_2_available,\n    is_flash_attn_greater_or_equal_2_10,\n    logging,\n    replace_return_docstrings,\n    torch_int,\n)\nfrom .configuration_siglip import SiglipConfig, SiglipTextConfig, SiglipVisionConfig\n\n\nif is_flash_attn_2_available():\n    from transformers.modeling_flash_attention_utils import _flash_attention_forward\n\n\nlogger = logging.get_logger(__name__)\n\n# General docstring\n_CONFIG_FOR_DOC = "SiglipConfig"\n_CHECKPOINT_FOR_DOC = "google/siglip-base-patch16-224"\n\n\ndef _trunc_normal_(tensor, mean, std, a, b):\n    # Cut & paste from PyTorch official master until it\'s in a few official releases - RW\n    # Method based on https://people.sc.fsu.edu/~jburkardt/presentations/truncated_normal.pdf\n    def norm_cdf(x):\n        # Computes standard normal cumulative distribution function\n        return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0\n\n    if (mean < a - 2 * std) or (mean > b + 2 * std):\n        warnings.warn(\n            "mean is more than 2 std from [a, b] in nn.init.trunc_normal_. "\n            "The distribution of values may be incorrect.",\n            stacklevel=2,\n        )\n\n    # Values are generated by using a truncated uniform distribution and\n    # then using the inverse CDF for the normal distribution.\n    # Get upper and lower cdf values\n    l = norm_cdf((a - mean) / std)\n    u = norm_cdf((b - mean) / std)\n\n    # Uniformly fill tensor with values from [l, u], then translate to\n    # [2l-1, 2u-1].\n    tensor.uniform_(2 * l - 1, 2 * u - 1)\n\n    # Use inverse cdf transform for normal distribution to get truncated\n    # standard normal\n    tensor.erfinv_()\n\n    # Transform to proper mean, std\n    tensor.mul_(std * math.sqrt(2.0))\n    tensor.add_(mean)\n\n    # Clamp to ensure it\'s in the proper range\n    tensor.clamp_(min=a, max=b)\n\n\ndef trunc_normal_tf_(\n    tensor: torch.Tensor, mean: float = 0.0, std: float = 1.0, a: float = -2.0, b: float = 2.0\n) -> torch.Tensor:\n    """Fills the input Tensor with values drawn from a truncated\n    normal distribution. The values are effectively drawn from the\n    normal distribution :math:`\\\\mathcal{N}(\\text{mean}, \\text{std}^2)`\n    with values outside :math:`[a, b]` redrawn until they are within\n    the bounds. The method used for generating the random values works\n    best when :math:`a \\\\leq \\text{mean} \\\\leq b`.\n\n    NOTE: this \'tf\' variant behaves closer to Tensorflow / JAX impl where the\n    bounds [a, b] are applied when sampling the normal distribution with mean=0, std=1.0\n    and the result is subsequently scaled and shifted by the mean and std args.\n\n    Args:\n        tensor: an n-dimensional `torch.Tensor`\n        mean: the mean of the normal distribution\n        std: the standard deviation of the normal distribution\n        a: the minimum cutoff value\n        b: the maximum cutoff value\n    """\n    with torch.no_grad():\n        _trunc_normal_(tensor, 0, 1.0, a, b)\n        tensor.mul_(std).add_(mean)\n\n\ndef variance_scaling_(tensor, scale=1.0, mode="fan_in", distribution="normal"):\n    fan_in, fan_out = _calculate_fan_in_and_fan_out(tensor)\n    if mode == "fan_in":\n        denom = fan_in\n    elif mode == "fan_out":\n        denom = fan_out\n    elif mode == "fan_avg":\n        denom = (fan_in + fan_out) / 2\n\n    variance = scale / denom\n\n    if distribution == "truncated_normal":\n        # constant is stddev of standard normal truncated to (-2, 2)\n        trunc_normal_tf_(tensor, std=math.sqrt(variance) / 0.87962566103423978)\n    elif distribution == "normal":\n        with torch.no_grad():\n            tensor.normal_(std=math.sqrt(variance))\n    elif distribution == "uniform":\n        bound = math.sqrt(3 * variance)\n        with torch.no_grad():\n            tensor.uniform_(-bound, bound)\n    else:\n        raise ValueError(f"invalid distribution {distribution}")\n\n\ndef lecun_normal_(tensor):\n    variance_scaling_(tensor, mode="fan_in", distribution="truncated_normal")\n\n\ndef default_flax_embed_init(tensor):\n    variance_scaling_(tensor, mode="fan_in", distribution="normal")\n\n\n@dataclass\n# Copied from transformers.models.clip.modeling_clip.CLIPVisionModelOutput with CLIP->Siglip\nclass SiglipVisionModelOutput(ModelOutput):\n    """\n    Base class for vision model\'s outputs that also contains image embeddings of the pooling of the last hidden states.\n\n    Args:\n        image_embeds (`torch.FloatTensor` of shape `(batch_size, output_dim)` *optional* returned when model is initialized with `with_projection=True`):\n            The image embeddings obtained by applying the projection layer to the pooler_output.\n        last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):\n            Sequence of hidden-states at the output of the last layer of the model.\n        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):\n            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +\n            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.\n\n            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.\n        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):\n            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,\n            sequence_length)`.\n\n            Attentions weights after the attention softmax, used to compute the weighted average in the self-attention\n            heads.\n    """\n\n    image_embeds: Optional[torch.FloatTensor] = None\n    last_hidden_state: torch.FloatTensor = None\n    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None\n    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None\n\n\n@dataclass\n# Copied from transformers.models.clip.modeling_clip.CLIPTextModelOutput with CLIP->Siglip\nclass SiglipTextModelOutput(ModelOutput):\n    """\n    Base class for text model\'s outputs that also contains a pooling of the last hidden states.\n\n    Args:\n        text_embeds (`torch.FloatTensor` of shape `(batch_size, output_dim)` *optional* returned when model is initialized with `with_projection=True`):\n            The text embeddings obtained by applying the projection layer to the pooler_output.\n        last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):\n            Sequence of hidden-states at the output of the last layer of the model.\n        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):\n            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +\n            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.\n\n            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.\n        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):\n            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,\n            sequence_length)`.\n\n            Attentions weights after the attention softmax, used to compute the weighted average in the self-attention\n            heads.\n    """\n\n    text_embeds: Optional[torch.FloatTensor] = None\n    last_hidden_state: torch.FloatTensor = None\n    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None\n    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None\n\n\n@dataclass\n# Copied from transformers.models.clip.modeling_clip.CLIPOutput with CLIP->Siglip\nclass SiglipOutput(ModelOutput):\n    """\n    Args:\n        loss (`torch.FloatTensor` of shape `(1,)`, *optional*, returned when `return_loss` is `True`):\n            Contrastive loss for image-text similarity.\n        logits_per_image (`torch.FloatTensor` of shape `(image_batch_size, text_batch_size)`):\n            The scaled dot product scores between `image_embeds` and `text_embeds`. This represents the image-text\n            similarity scores.\n        logits_per_text (`torch.FloatTensor` of shape `(text_batch_size, image_batch_size)`):\n            The scaled dot product scores between `text_embeds` and `image_embeds`. This represents the text-image\n            similarity scores.\n        text_embeds (`torch.FloatTensor` of shape `(batch_size, output_dim`):\n            The text embeddings obtained by applying the projection layer to the pooled output of [`SiglipTextModel`].\n        image_embeds (`torch.FloatTensor` of shape `(batch_size, output_dim`):\n            The image embeddings obtained by applying the projection layer to the pooled output of [`SiglipVisionModel`].\n        text_model_output (`BaseModelOutputWithPooling`):\n            The output of the [`SiglipTextModel`].\n        vision_model_output (`BaseModelOutputWithPooling`):\n            The output of the [`SiglipVisionModel`].\n    """\n\n    loss: Optional[torch.FloatTensor] = None\n    logits_per_image: torch.FloatTensor = None\n    logits_per_text: torch.FloatTensor = None\n    text_embeds: torch.FloatTensor = None\n    image_embeds: torch.FloatTensor = None\n    text_model_output: BaseModelOutputWithPooling = None\n    vision_model_output: BaseModelOutputWithPooling = None\n\n    def to_tuple(self) -> Tuple[Any]:\n        return tuple(\n            self[k] if k not in ["text_model_output", "vision_model_output"] else getattr(self, k).to_tuple()\n            for k in self.keys()\n        )\n\n\nclass SiglipVisionEmbeddings(nn.Module):\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n        self.config = config\n        self.embed_dim = config.hidden_size\n        self.image_size = config.image_size\n        self.patch_size = config.patch_size\n\n        self.patch_embedding = nn.Conv2d(\n            in_channels=config.num_channels,\n            out_channels=self.embed_dim,\n            kernel_size=self.patch_size,\n            stride=self.patch_size,\n            padding="valid",\n        )\n\n        self.num_patches = (self.image_size // self.patch_size) ** 2\n        self.num_positions = self.num_patches\n        self.position_embedding = nn.Embedding(self.num_positions, self.embed_dim)\n        self.register_buffer("position_ids", torch.arange(self.num_positions).expand((1, -1)), persistent=False)\n\n    def interpolate_pos_encoding(self, embeddings: torch.Tensor, height: int, width: int) -> torch.Tensor:\n        """\n        This method allows to interpolate the pre-trained position encodings, to be able to use the model on higher resolution\n        images. This method is also adapted to support torch.jit tracing and no class embeddings.\n\n        Adapted from:\n        - https://github.com/facebookresearch/dino/blob/de9ee3df6cf39fac952ab558447af1fa1365362a/vision_transformer.py#L174-L194, and\n        - https://github.com/facebookresearch/dinov2/blob/e1277af2ba9496fbadf7aec6eba56e8d882d1e35/dinov2/models/vision_transformer.py#L179-L211\n        """\n\n        num_patches = embeddings.shape[1]\n        num_positions = self.position_embedding.weight.shape[0]\n\n        # always interpolate when tracing to ensure the exported model works for dynamic input shapes\n        if not torch.jit.is_tracing() and num_patches == num_positions and height == width:\n            return self.position_embedding(self.position_ids)\n\n        patch_pos_embed = self.position_embedding.weight.unsqueeze(0)\n\n        dim = embeddings.shape[-1]\n\n        new_height = height // self.patch_size\n        new_width = width // self.patch_size\n\n        sqrt_num_positions = torch_int(num_positions**0.5)\n        patch_pos_embed = patch_pos_embed.reshape(1, sqrt_num_positions, sqrt_num_positions, dim)\n        patch_pos_embed = patch_pos_embed.permute(0, 3, 1, 2)\n\n        patch_pos_embed = nn.functional.interpolate(\n            patch_pos_embed,\n            size=(new_height, new_width),\n            mode="bicubic",\n            align_corners=False,\n        )\n\n        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).view(1, -1, dim)\n        return patch_pos_embed\n\n    def forward(self, pixel_values: torch.FloatTensor, interpolate_pos_encoding=False) -> torch.Tensor:\n        _, _, height, width = pixel_values.shape\n        patch_embeds = self.patch_embedding(pixel_values)  # shape = [*, width, grid, grid]\n        embeddings = patch_embeds.flatten(2).transpose(1, 2)\n\n        if interpolate_pos_encoding:\n            embeddings = embeddings + self.interpolate_pos_encoding(embeddings, height, width)\n        else:\n            embeddings = embeddings + self.position_embedding(self.position_ids)\n        return embeddings\n\n\n# Copied from transformers.models.clip.modeling_clip.CLIPTextEmbeddings with CLIP->Siglip\nclass SiglipTextEmbeddings(nn.Module):\n    def __init__(self, config: SiglipTextConfig):\n        super().__init__()\n        embed_dim = config.hidden_size\n\n        self.token_embedding = nn.Embedding(config.vocab_size, embed_dim)\n        self.position_embedding = nn.Embedding(config.max_position_embeddings, embed_dim)\n\n        # position_ids (1, len position emb) is contiguous in memory and exported when serialized\n        self.register_buffer(\n            "position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)), persistent=False\n        )\n\n    def forward(\n        self,\n        input_ids: Optional[torch.LongTensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n    ) -> torch.Tensor:\n        seq_length = input_ids.shape[-1] if input_ids is not None else inputs_embeds.shape[-2]\n\n        if position_ids is None:\n            position_ids = self.position_ids[:, :seq_length]\n\n        if inputs_embeds is None:\n            inputs_embeds = self.token_embedding(input_ids)\n\n        position_embeddings = self.position_embedding(position_ids)\n        embeddings = inputs_embeds + position_embeddings\n\n        return embeddings\n\n\nclass SiglipAttention(nn.Module):\n    """Multi-headed attention from \'Attention Is All You Need\' paper"""\n\n    # Copied from transformers.models.clip.modeling_clip.CLIPAttention.__init__\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        self.embed_dim = config.hidden_size\n        self.num_heads = config.num_attention_heads\n        self.head_dim = self.embed_dim // self.num_heads\n        if self.head_dim * self.num_heads != self.embed_dim:\n            raise ValueError(\n                f"embed_dim must be divisible by num_heads (got `embed_dim`: {self.embed_dim} and `num_heads`:"\n                f" {self.num_heads})."\n            )\n        self.scale = self.head_dim**-0.5\n        self.dropout = config.attention_dropout\n\n        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim)\n        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim)\n        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim)\n        self.out_proj = nn.Linear(self.embed_dim, self.embed_dim)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = False,\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:\n        """Input shape: Batch x Time x Channel"""\n\n        batch_size, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n\n        k_v_seq_len = key_states.shape[-2]\n        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) * self.scale\n\n        if attn_weights.size() != (batch_size, self.num_heads, q_len, k_v_seq_len):\n            raise ValueError(\n                f"Attention weights should be of size {(batch_size, self.num_heads, q_len, k_v_seq_len)}, but is"\n                f" {attn_weights.size()}"\n            )\n\n        if attention_mask is not None:\n            if attention_mask.size() != (batch_size, 1, q_len, k_v_seq_len):\n                raise ValueError(\n                    f"Attention mask should be of size {(batch_size, 1, q_len, k_v_seq_len)}, but is {attention_mask.size()}"\n                )\n            attn_weights = attn_weights + attention_mask\n\n        # upcast attention to fp32\n        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n        attn_weights = nn.functional.dropout(attn_weights, p=self.dropout, training=self.training)\n        attn_output = torch.matmul(attn_weights, value_states)\n\n        if attn_output.size() != (batch_size, self.num_heads, q_len, self.head_dim):\n            raise ValueError(\n                f"`attn_output` should be of size {(batch_size, self.num_heads, q_len, self.head_dim)}, but is"\n                f" {attn_output.size()}"\n            )\n\n        attn_output = attn_output.transpose(1, 2).contiguous()\n        attn_output = attn_output.reshape(batch_size, q_len, self.embed_dim)\n\n        attn_output = self.out_proj(attn_output)\n\n        return attn_output, attn_weights\n\n\nclass SiglipFlashAttention2(SiglipAttention):\n    """\n    SiglipAttention flash attention module. This module inherits from `SiglipAttention` as the weights of the module stays\n    untouched. The only required change would be on the forward pass where it needs to correctly call the public API of\n    flash attention and deal with padding tokens in case the input contains any of them.\n    """\n\n    is_causal = False\n\n    # Copied from transformers.models.llama.modeling_llama.LlamaFlashAttention2.__init__\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n\n        # TODO: Should be removed once Flash Attention for RoCm is bumped to 2.1.\n        # flash_attn<2.1 generates top-left aligned causal mask, while what is needed here is bottom-right alignement, that was made default for flash_attn>=2.1. This attribute is used to handle this difference. Reference: https://github.com/Dao-AILab/flash-attention/releases/tag/v2.1.0.\n        # Beware that with flash_attn<2.1, using q_seqlen != k_seqlen (except for the case q_seqlen == 1) produces a wrong mask (top-left).\n        self._flash_attn_uses_top_left_mask = not is_flash_attn_greater_or_equal_2_10()\n\n    # Adapted from transformers.models.llama.modeling_llama.LlamaFlashAttention2.forward\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.LongTensor] = None,\n        output_attentions: bool = False,\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:\n        output_attentions = False\n\n        batch_size, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        # Flash attention requires the input to have the shape\n        # batch_size x seq_length x head_dim x hidden_dim\n        # therefore we just need to keep the original shape\n        query_states = query_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n\n        # TODO: These transpose are quite inefficient but Flash Attention requires the layout [batch_size, sequence_length, num_heads, head_dim]. We would need to refactor the KV cache\n        # to be able to avoid many of these transpose/reshape/view.\n        query_states = query_states.transpose(1, 2)\n        key_states = key_states.transpose(1, 2)\n        value_states = value_states.transpose(1, 2)\n\n        dropout_rate = self.dropout if self.training else 0.0\n\n        # In PEFT, usually we cast the layer norms in float32 for training stability reasons\n        # therefore the input hidden states gets silently casted in float32. Hence, we need\n        # cast them back in the correct dtype just to be sure everything works as expected.\n        # This might slowdown training & inference so it is recommended to not cast the LayerNorms\n        # in fp32.\n\n        input_dtype = query_states.dtype\n        if input_dtype == torch.float32:\n            if torch.is_autocast_enabled():\n                target_dtype = torch.get_autocast_gpu_dtype()\n            # Handle the case where the model is quantized\n            elif hasattr(self.config, "_pre_quantization_dtype"):\n                target_dtype = self.config._pre_quantization_dtype\n            else:\n                target_dtype = self.q_proj.weight.dtype\n\n            logger.warning_once(\n                f"The input hidden states seems to be silently casted in float32, this might be related to"\n                f" the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in"\n                f" {target_dtype}."\n            )\n\n            query_states = query_states.to(target_dtype)\n            key_states = key_states.to(target_dtype)\n            value_states = value_states.to(target_dtype)\n\n        attn_output = _flash_attention_forward(\n            query_states,\n            key_states,\n            value_states,\n            attention_mask,\n            q_len,\n            dropout=dropout_rate,\n            is_causal=self.is_causal,\n            use_top_left_mask=self._flash_attn_uses_top_left_mask,\n        )\n\n        attn_output = attn_output.reshape(batch_size, q_len, self.embed_dim).contiguous()\n        attn_output = self.out_proj(attn_output)\n\n        if not output_attentions:\n            attn_weights = None\n\n        return attn_output, attn_weights\n\n\nclass SiglipSdpaAttention(SiglipAttention):\n    """\n    Siglip attention module using torch.nn.functional.scaled_dot_product_attention. This module inherits from\n    `SiglipAttention` as the weights of the module stays untouched. The only changes are on the forward pass to adapt to\n    SDPA API.\n    """\n\n    is_causal = False\n\n    # Adapted from SiglipAttention.forward and transformers.models.llama.modeling_llama.LlamaSdpaAttention.forward\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = False,\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:\n        if output_attentions:\n            # TODO: Improve this warning with e.g. `model.config.attn_implementation = "manual"` once this is implemented.\n            logger.warning_once(\n                "SiglipModel is using SiglipSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, "\n                \'but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.\'\n            )\n            return super().forward(\n                hidden_states=hidden_states,\n                attention_mask=attention_mask,\n                output_attentions=output_attentions,\n            )\n\n        batch_size, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(batch_size, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n\n        # SDPA with memory-efficient backend is currently (torch==2.1.2) bugged with non-contiguous inputs with custom attn_mask,\n        # Reference: https://github.com/pytorch/pytorch/issues/112577.\n        if query_states.device.type == "cuda" and attention_mask is not None:\n            query_states = query_states.contiguous()\n            key_states = key_states.contiguous()\n            value_states = value_states.contiguous()\n\n        # We dispatch to SDPA\'s Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment\n        # in SDPA to support both torch.compile\'s dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.\n        is_causal = True if self.is_causal and q_len > 1 else False\n\n        attn_output = torch.nn.functional.scaled_dot_product_attention(\n            query_states,\n            key_states,\n            value_states,\n            attn_mask=attention_mask,\n            dropout_p=self.dropout if self.training else 0.0,\n            is_causal=is_causal,\n        )\n\n        attn_output = attn_output.transpose(1, 2).contiguous()\n        attn_output = attn_output.view(batch_size, q_len, self.embed_dim)\n\n        attn_output = self.out_proj(attn_output)\n\n        return attn_output, None\n\n\nSIGLIP_ATTENTION_CLASSES = {\n    "eager": SiglipAttention,\n    "flash_attention_2": SiglipFlashAttention2,\n    "sdpa": SiglipSdpaAttention,\n}\n\n\n# Copied from transformers.models.clip.modeling_clip.CLIPMLP with CLIP->Siglip\nclass SiglipMLP(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        self.activation_fn = ACT2FN[config.hidden_act]\n        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)\n        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        hidden_states = self.fc1(hidden_states)\n        hidden_states = self.activation_fn(hidden_states)\n        hidden_states = self.fc2(hidden_states)\n        return hidden_states\n\n\nclass SiglipEncoderLayer(nn.Module):\n    def __init__(self, config: SiglipConfig):\n        super().__init__()\n        self.embed_dim = config.hidden_size\n        self.self_attn = SIGLIP_ATTENTION_CLASSES[config._attn_implementation](config=config)\n        self.layer_norm1 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)\n        self.mlp = SiglipMLP(config)\n        self.layer_norm2 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)\n\n    # Ignore copy\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: torch.Tensor,\n        output_attentions: Optional[bool] = False,\n    ) -> Tuple[torch.FloatTensor]:\n        """\n        Args:\n            hidden_states (`torch.FloatTensor`):\n                Input to the layer of shape `(batch, seq_len, embed_dim)`.\n            attention_mask (`torch.FloatTensor`):\n                Attention mask of shape `(batch, 1, q_len, k_v_seq_len)` where padding elements are indicated by very large negative values.\n            output_attentions (`bool`, *optional*, defaults to `False`):\n                Whether or not to return the attentions tensors of all attention layers. See `attentions` under\n                returned tensors for more detail.\n        """\n        residual = hidden_states\n\n        hidden_states = self.layer_norm1(hidden_states)\n        hidden_states, attn_weights = self.self_attn(\n            hidden_states=hidden_states,\n            attention_mask=attention_mask,\n            output_attentions=output_attentions,\n        )\n        hidden_states = residual + hidden_states\n\n        residual = hidden_states\n        hidden_states = self.layer_norm2(hidden_states)\n        hidden_states = self.mlp(hidden_states)\n        hidden_states = residual + hidden_states\n\n        outputs = (hidden_states,)\n\n        if output_attentions:\n            outputs += (attn_weights,)\n\n        return outputs\n\n\nclass SiglipPreTrainedModel(PreTrainedModel):\n    """\n    An abstract class to handle weights initialization and a simple interface for downloading and loading pretrained\n    models.\n    """\n\n    config_class = SiglipConfig\n    base_model_prefix = "siglip"\n    supports_gradient_checkpointing = True\n\n    _no_split_modules = [\n        "SiglipTextEmbeddings",\n        "SiglipEncoderLayer",\n        "SiglipVisionEmbeddings",\n        "SiglipEncoderLayer",\n        "SiglipMultiheadAttentionPoolingHead",\n    ]\n    _supports_flash_attn_2 = True\n    _supports_sdpa = True\n\n    def _init_weights(self, module):\n        """Initialize the weights"""\n        if isinstance(module, SiglipVisionEmbeddings):\n            width = (\n                self.config.vision_config.hidden_size\n                if isinstance(self.config, SiglipConfig)\n                else self.config.hidden_size\n            )\n            nn.init.normal_(module.position_embedding.weight, std=1 / np.sqrt(width))\n        elif isinstance(module, nn.Embedding):\n            default_flax_embed_init(module.weight)\n        elif isinstance(module, SiglipAttention):\n            nn.init.xavier_uniform_(module.q_proj.weight)\n            nn.init.xavier_uniform_(module.k_proj.weight)\n            nn.init.xavier_uniform_(module.v_proj.weight)\n            nn.init.xavier_uniform_(module.out_proj.weight)\n            nn.init.zeros_(module.q_proj.bias)\n            nn.init.zeros_(module.k_proj.bias)\n            nn.init.zeros_(module.v_proj.bias)\n            nn.init.zeros_(module.out_proj.bias)\n        elif isinstance(module, SiglipMLP):\n            nn.init.xavier_uniform_(module.fc1.weight)\n            nn.init.xavier_uniform_(module.fc2.weight)\n            nn.init.normal_(module.fc1.bias, std=1e-6)\n            nn.init.normal_(module.fc2.bias, std=1e-6)\n        elif isinstance(module, SiglipMultiheadAttentionPoolingHead):\n            nn.init.xavier_uniform_(module.probe.data)\n            nn.init.xavier_uniform_(module.attention.in_proj_weight.data)\n            nn.init.zeros_(module.attention.in_proj_bias.data)\n        elif isinstance(module, SiglipModel):\n            logit_scale_init = torch.log(torch.tensor(1.0))\n            module.logit_scale.data.fill_(logit_scale_init)\n            module.logit_bias.data.zero_()\n        elif isinstance(module, SiglipForImageClassification):\n            nn.init.normal_(\n                module.classifier.weight,\n                std=self.config.vision_config.hidden_size**-0.5 * self.config.initializer_factor,\n            )\n        elif isinstance(module, (nn.Linear, nn.Conv2d)):\n            lecun_normal_(module.weight)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.LayerNorm):\n            module.bias.data.zero_()\n            module.weight.data.fill_(1.0)\n\n\nSIGLIP_START_DOCSTRING = r"""\n    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the\n    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads\n    etc.)\n\n    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.\n    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage\n    and behavior.\n\n    Parameters:\n        config ([`SiglipConfig`]): Model configuration class with all the parameters of the model.\n            Initializing with a config file does not load the weights associated with the model, only the\n            configuration. Check out the [`~PreTrainedModel.from_pretrained`] method to load the model weights.\n"""\n\nSIGLIP_TEXT_INPUTS_DOCSTRING = r"""\n    Args:\n        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):\n            Indices of input sequence tokens in the vocabulary. Padding will be ignored by default should you provide\n            it.\n\n            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and\n            [`PreTrainedTokenizer.__call__`] for details.\n\n            [What are input IDs?](../glossary#input-ids)\n        attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:\n\n            - 1 for tokens that are **not masked**,\n            - 0 for tokens that are **masked**.\n\n            [What are attention masks?](../glossary#attention-mask)\n        position_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Indices of positions of each input sequence tokens in the position embeddings. Selected in the range `[0,\n            config.max_position_embeddings - 1]`.\n\n            [What are position IDs?](../glossary#position-ids)\n        output_attentions (`bool`, *optional*):\n            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned\n            tensors for more detail.\n        output_hidden_states (`bool`, *optional*):\n            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for\n            more detail.\n        return_dict (`bool`, *optional*):\n            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.\n"""\n\nSIGLIP_VISION_INPUTS_DOCSTRING = r"""\n    Args:\n        pixel_values (`torch.FloatTensor` of shape `(batch_size, num_channels, height, width)`):\n            Pixel values. Padding will be ignored by default should you provide it. Pixel values can be obtained using\n            [`AutoImageProcessor`]. See [`CLIPImageProcessor.__call__`] for details.\n        output_attentions (`bool`, *optional*):\n            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned\n            tensors for more detail.\n        output_hidden_states (`bool`, *optional*):\n            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for\n            more detail.\n        interpolate_pos_encoding (`bool`, *optional*, defaults to `False`):\n            Whether to interpolate the pre-trained position encodings.\n        return_dict (`bool`, *optional*):\n            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.\n"""\n\nSIGLIP_INPUTS_DOCSTRING = r"""\n    Args:\n        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):\n            Indices of input sequence tokens in the vocabulary. Padding will be ignored by default should you provide\n            it.\n\n            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and\n            [`PreTrainedTokenizer.__call__`] for details.\n\n            [What are input IDs?](../glossary#input-ids)\n        attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:\n\n            - 1 for tokens that are **not masked**,\n            - 0 for tokens that are **masked**.\n\n            [What are attention masks?](../glossary#attention-mask)\n        position_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Indices of positions of each input sequence tokens in the position embeddings. Selected in the range `[0,\n            config.max_position_embeddings - 1]`.\n\n            [What are position IDs?](../glossary#position-ids)\n        pixel_values (`torch.FloatTensor` of shape `(batch_size, num_channels, height, width)`):\n            Pixel values. Padding will be ignored by default should you provide it. Pixel values can be obtained using\n            [`AutoImageProcessor`]. See [`CLIPImageProcessor.__call__`] for details.\n        return_loss (`bool`, *optional*):\n            Whether or not to return the contrastive loss.\n        output_attentions (`bool`, *optional*):\n            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned\n            tensors for more detail.\n        output_hidden_states (`bool`, *optional*):\n            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for\n            more detail.\n        interpolate_pos_encoding (`bool`, *optional*, defaults to `False`):\n            Whether to interpolate the pre-trained position encodings.\n        return_dict (`bool`, *optional*):\n            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.\n"""\n\n\n# Copied from transformers.models.altclip.modeling_altclip.AltCLIPEncoder with AltCLIP->Siglip\nclass SiglipEncoder(nn.Module):\n    """\n    Transformer encoder consisting of `config.num_hidden_layers` self attention layers. Each layer is a\n    [`SiglipEncoderLayer`].\n\n    Args:\n        config: SiglipConfig\n    """\n\n    def __init__(self, config: SiglipConfig):\n        super().__init__()\n        self.config = config\n        self.layers = nn.ModuleList([SiglipEncoderLayer(config) for _ in range(config.num_hidden_layers)])\n        self.gradient_checkpointing = False\n\n    # Ignore copy\n    def forward(\n        self,\n        inputs_embeds,\n        attention_mask: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n    ) -> Union[Tuple, BaseModelOutput]:\n        r"""\n        Args:\n            inputs_embeds (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):\n                Optionally, instead of passing `input_ids` you can choose to directly pass an embedded representation.\n                This is useful if you want more control over how to convert `input_ids` indices into associated vectors\n                than the model\'s internal embedding lookup matrix.\n            attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):\n                Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:\n\n                - 1 for tokens that are **not masked**,\n                - 0 for tokens that are **masked**.\n\n                [What are attention masks?](../glossary#attention-mask)\n            output_attentions (`bool`, *optional*):\n                Whether or not to return the attentions tensors of all attention layers. See `attentions` under\n                returned tensors for more detail.\n            output_hidden_states (`bool`, *optional*):\n                Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors\n                for more detail.\n            return_dict (`bool`, *optional*):\n                Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.\n        """\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        encoder_states = () if output_hidden_states else None\n        all_attentions = () if output_attentions else None\n\n        hidden_states = inputs_embeds\n        for encoder_layer in self.layers:\n            if output_hidden_states:\n                encoder_states = encoder_states + (hidden_states,)\n            if self.gradient_checkpointing and self.training:\n                layer_outputs = self._gradient_checkpointing_func(\n                    encoder_layer.__call__,\n                    hidden_states,\n                    attention_mask,\n                    output_attentions,\n                )\n            else:\n                layer_outputs = encoder_layer(\n                    hidden_states,\n                    attention_mask,\n                    output_attentions=output_attentions,\n                )\n\n            hidden_states = layer_outputs[0]\n\n            if output_attentions:\n                all_attentions = all_attentions + (layer_outputs[1],)\n\n        if output_hidden_states:\n            encoder_states = encoder_states + (hidden_states,)\n\n        if not return_dict:\n            return tuple(v for v in [hidden_states, encoder_states, all_attentions] if v is not None)\n        return BaseModelOutput(\n            last_hidden_state=hidden_states, hidden_states=encoder_states, attentions=all_attentions\n        )\n\n\nclass SiglipTextTransformer(nn.Module):\n    def __init__(self, config: SiglipTextConfig):\n        super().__init__()\n        self.config = config\n        embed_dim = config.hidden_size\n        self.embeddings = SiglipTextEmbeddings(config)\n        self.encoder = SiglipEncoder(config)\n        self.final_layer_norm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)\n\n        self.head = nn.Linear(embed_dim, embed_dim)\n        self._use_flash_attention_2 = config._attn_implementation == "flash_attention_2"\n\n    @add_start_docstrings_to_model_forward(SIGLIP_TEXT_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=BaseModelOutputWithPooling, config_class=SiglipTextConfig)\n    def forward(\n        self,\n        input_ids: Optional[torch.Tensor] = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n    ) -> Union[Tuple, BaseModelOutputWithPooling]:\n        r"""\n        Returns:\n\n        """\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        if input_ids is None:\n            raise ValueError("You have to specify input_ids")\n\n        input_shape = input_ids.size()\n        input_ids = input_ids.view(-1, input_shape[-1])\n\n        hidden_states = self.embeddings(input_ids=input_ids, position_ids=position_ids)\n\n        # note: SigLIP\'s text model does not use a causal mask, unlike the original CLIP model.\n        # expand attention_mask\n        if attention_mask is not None and not self._use_flash_attention_2:\n            # [batch_size, seq_len] -> [batch_size, 1, tgt_seq_len, src_seq_len]\n            attention_mask = _prepare_4d_attention_mask(attention_mask, hidden_states.dtype)\n\n        encoder_outputs = self.encoder(\n            inputs_embeds=hidden_states,\n            attention_mask=attention_mask,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n        )\n\n        last_hidden_state = encoder_outputs[0]\n        last_hidden_state = self.final_layer_norm(last_hidden_state)\n\n        # Assuming "sticky" EOS tokenization, last token is always EOS.\n        pooled_output = last_hidden_state[:, -1, :]\n        pooled_output = self.head(pooled_output)\n\n        if not return_dict:\n            return (last_hidden_state, pooled_output) + encoder_outputs[1:]\n\n        return BaseModelOutputWithPooling(\n            last_hidden_state=last_hidden_state,\n            pooler_output=pooled_output,\n            hidden_states=encoder_outputs.hidden_states,\n            attentions=encoder_outputs.attentions,\n        )\n\n\n@add_start_docstrings(\n    """The text model from SigLIP without any head or projection on top.""",\n    SIGLIP_START_DOCSTRING,\n)\nclass SiglipTextModel(SiglipPreTrainedModel):\n    config_class = SiglipTextConfig\n\n    def __init__(self, config: SiglipTextConfig):\n        super().__init__(config)\n        self.text_model = SiglipTextTransformer(config)\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self) -> nn.Module:\n        return self.text_model.embeddings.token_embedding\n\n    def set_input_embeddings(self, value):\n        self.text_model.embeddings.token_embedding = value\n\n    @add_start_docstrings_to_model_forward(SIGLIP_TEXT_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=BaseModelOutputWithPooling, config_class=SiglipTextConfig)\n    def forward(\n        self,\n        input_ids: Optional[torch.Tensor] = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n    ) -> Union[Tuple, BaseModelOutputWithPooling]:\n        r"""\n        Returns:\n\n        Examples:\n\n        ```python\n        >>> from transformers import AutoTokenizer, SiglipTextModel\n\n        >>> model = SiglipTextModel.from_pretrained("google/siglip-base-patch16-224")\n        >>> tokenizer = AutoTokenizer.from_pretrained("google/siglip-base-patch16-224")\n\n        >>> # important: make sure to set padding="max_length" as that\'s how the model was trained\n        >>> inputs = tokenizer(["a photo of a cat", "a photo of a dog"], padding="max_length", return_tensors="pt")\n\n        >>> outputs = model(**inputs)\n        >>> last_hidden_state = outputs.last_hidden_state\n        >>> pooled_output = outputs.pooler_output  # pooled (EOS token) states\n        ```"""\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        return self.text_model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n        )\n\n\nclass SiglipVisionTransformer(nn.Module):\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n        self.config = config\n        embed_dim = config.hidden_size\n\n        self.embeddings = SiglipVisionEmbeddings(config)\n        self.encoder = SiglipEncoder(config)\n        self.post_layernorm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)\n        self.use_head = True if not hasattr(config, "vision_use_head") else config.vision_use_head\n        if self.use_head:\n            self.head = SiglipMultiheadAttentionPoolingHead(config)\n\n    @add_start_docstrings_to_model_forward(SIGLIP_VISION_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=BaseModelOutputWithPooling, config_class=SiglipVisionConfig)\n    def forward(\n        self,\n        pixel_values,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        interpolate_pos_encoding: Optional[bool] = False,\n    ) -> Union[Tuple, BaseModelOutputWithPooling]:\n        r"""\n        Returns:\n\n        """\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        hidden_states = self.embeddings(pixel_values, interpolate_pos_encoding=interpolate_pos_encoding)\n\n        encoder_outputs = self.encoder(\n            inputs_embeds=hidden_states,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n        )\n\n        last_hidden_state = encoder_outputs[0]\n        last_hidden_state = self.post_layernorm(last_hidden_state)\n\n        pooler_output = self.head(last_hidden_state) if self.use_head else None\n        if not return_dict:\n            return (last_hidden_state, pooler_output) + encoder_outputs[1:]\n\n        return BaseModelOutputWithPooling(\n            last_hidden_state=last_hidden_state,\n            pooler_output=pooler_output,\n            hidden_states=encoder_outputs.hidden_states,\n            attentions=encoder_outputs.attentions,\n        )\n\n\nclass SiglipMultiheadAttentionPoolingHead(nn.Module):\n    """Multihead Attention Pooling."""\n\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__()\n\n        self.probe = nn.Parameter(torch.randn(1, 1, config.hidden_size))\n        self.attention = torch.nn.MultiheadAttention(config.hidden_size, config.num_attention_heads, batch_first=True)\n        self.layernorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)\n        self.mlp = SiglipMLP(config)\n\n    def forward(self, hidden_state):\n        batch_size = hidden_state.shape[0]\n        probe = self.probe.repeat(batch_size, 1, 1)\n\n        hidden_state = self.attention(probe, hidden_state, hidden_state)[0]\n\n        residual = hidden_state\n        hidden_state = self.layernorm(hidden_state)\n        hidden_state = residual + self.mlp(hidden_state)\n\n        return hidden_state[:, 0]\n\n\n@add_start_docstrings(\n    """The vision model from SigLIP without any head or projection on top.""",\n    SIGLIP_START_DOCSTRING,\n)\nclass SiglipVisionModel(SiglipPreTrainedModel):\n    config_class = SiglipVisionConfig\n    main_input_name = "pixel_values"\n\n    def __init__(self, config: SiglipVisionConfig):\n        super().__init__(config)\n\n        self.vision_model = SiglipVisionTransformer(config)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self) -> nn.Module:\n        return self.vision_model.embeddings.patch_embedding\n\n    @add_start_docstrings_to_model_forward(SIGLIP_VISION_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=BaseModelOutputWithPooling, config_class=SiglipVisionConfig)\n    def forward(\n        self,\n        pixel_values,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        interpolate_pos_encoding: bool = False,\n    ) -> Union[Tuple, BaseModelOutputWithPooling]:\n        r"""\n        Returns:\n\n        Examples:\n\n        ```python\n        >>> from PIL import Image\n        >>> import requests\n        >>> from transformers import AutoProcessor, SiglipVisionModel\n\n        >>> model = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224")\n        >>> processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")\n\n        >>> url = "http://images.cocodataset.org/val2017/000000039769.jpg"\n        >>> image = Image.open(requests.get(url, stream=True).raw)\n\n        >>> inputs = processor(images=image, return_tensors="pt")\n\n        >>> outputs = model(**inputs)\n        >>> last_hidden_state = outputs.last_hidden_state\n        >>> pooled_output = outputs.pooler_output  # pooled features\n        ```"""\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        return self.vision_model(\n            pixel_values=pixel_values,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            interpolate_pos_encoding=interpolate_pos_encoding,\n        )\n\n\n@add_start_docstrings(SIGLIP_START_DOCSTRING)\nclass SiglipModel(SiglipPreTrainedModel):\n    config_class = SiglipConfig\n\n    def __init__(self, config: SiglipConfig):\n        super().__init__(config)\n\n        if not isinstance(config.text_config, SiglipTextConfig):\n            raise TypeError(\n                "config.text_config is expected to be of type SiglipTextConfig but is of type"\n                f" {type(config.text_config)}."\n            )\n\n        if not isinstance(config.vision_config, SiglipVisionConfig):\n            raise TypeError(\n                "config.vision_config is expected to be of type SiglipVisionConfig but is of type"\n                f" {type(config.vision_config)}."\n            )\n\n        text_config = config.text_config\n        vision_config = config.vision_config\n\n        # First, initialize the text and vision models with proper attention implementation\n        text_model = SiglipTextModel._from_config(text_config)\n        vision_model = SiglipVisionModel._from_config(vision_config)\n\n        # Second, get the text and vision submodules (for backward compatibility)\n        self.text_model = text_model.text_model\n        self.vision_model = vision_model.vision_model\n\n        self.logit_scale = nn.Parameter(torch.randn(1))\n        self.logit_bias = nn.Parameter(torch.randn(1))\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    @add_start_docstrings_to_model_forward(SIGLIP_TEXT_INPUTS_DOCSTRING)\n    def get_text_features(\n        self,\n        input_ids: Optional[torch.Tensor] = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n    ) -> torch.FloatTensor:\n        r"""\n        Returns:\n            text_features (`torch.FloatTensor` of shape `(batch_size, output_dim`): The text embeddings obtained by\n            applying the projection layer to the pooled output of [`SiglipTextModel`].\n\n        Examples:\n\n        ```python\n        >>> from transformers import AutoTokenizer, AutoModel\n        >>> import torch\n\n        >>> model = AutoModel.from_pretrained("google/siglip-base-patch16-224")\n        >>> tokenizer = AutoTokenizer.from_pretrained("google/siglip-base-patch16-224")\n\n        >>> # important: make sure to set padding="max_length" as that\'s how the model was trained\n        >>> inputs = tokenizer(["a photo of a cat", "a photo of a dog"], padding="max_length", return_tensors="pt")\n        >>> with torch.no_grad():\n        ...     text_features = model.get_text_features(**inputs)\n        ```"""\n        # Use SigLIP model\'s config for some fields (if specified) instead of those of vision & text components.\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        text_outputs = self.text_model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n        )\n\n        pooled_output = text_outputs[1]\n\n        return pooled_output\n\n    @add_start_docstrings_to_model_forward(SIGLIP_VISION_INPUTS_DOCSTRING)\n    def get_image_features(\n        self,\n        pixel_values: Optional[torch.FloatTensor] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        interpolate_pos_encoding: bool = False,\n    ) -> torch.FloatTensor:\n        r"""\n        Returns:\n            image_features (`torch.FloatTensor` of shape `(batch_size, output_dim`): The image embeddings obtained by\n            applying the projection layer to the pooled output of [`SiglipVisionModel`].\n\n        Examples:\n\n        ```python\n        >>> from PIL import Image\n        >>> import requests\n        >>> from transformers import AutoProcessor, AutoModel\n        >>> import torch\n\n        >>> model = AutoModel.from_pretrained("google/siglip-base-patch16-224")\n        >>> processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")\n\n        >>> url = "http://images.cocodataset.org/val2017/000000039769.jpg"\n        >>> image = Image.open(requests.get(url, stream=True).raw)\n\n        >>> inputs = processor(images=image, return_tensors="pt")\n\n        >>> with torch.no_grad():\n        ...     image_features = model.get_image_features(**inputs)\n        ```"""\n        # Use SiglipModel\'s config for some fields (if specified) instead of those of vision & text components.\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        vision_outputs = self.vision_model(\n            pixel_values=pixel_values,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            interpolate_pos_encoding=interpolate_pos_encoding,\n        )\n\n        pooled_output = vision_outputs[1]\n\n        return pooled_output\n\n    @add_start_docstrings_to_model_forward(SIGLIP_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=SiglipOutput, config_class=SiglipConfig)\n    def forward(\n        self,\n        input_ids: Optional[torch.LongTensor] = None,\n        pixel_values: Optional[torch.FloatTensor] = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        return_loss: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        interpolate_pos_encoding: bool = False,\n    ) -> Union[Tuple, SiglipOutput]:\n        r"""\n        Returns:\n\n        Examples:\n\n        ```python\n        >>> from PIL import Image\n        >>> import requests\n        >>> from transformers import AutoProcessor, AutoModel\n        >>> import torch\n\n        >>> model = AutoModel.from_pretrained("google/siglip-base-patch16-224")\n        >>> processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")\n\n        >>> url = "http://images.cocodataset.org/val2017/000000039769.jpg"\n        >>> image = Image.open(requests.get(url, stream=True).raw)\n\n        >>> texts = ["a photo of 2 cats", "a photo of 2 dogs"]\n        >>> # important: we pass `padding=max_length` since the model was trained with this\n        >>> inputs = processor(text=texts, images=image, padding="max_length", return_tensors="pt")\n\n        >>> with torch.no_grad():\n        ...     outputs = model(**inputs)\n\n        >>> logits_per_image = outputs.logits_per_image\n        >>> probs = torch.sigmoid(logits_per_image) # these are the probabilities\n        >>> print(f"{probs[0][0]:.1%} that image 0 is \'{texts[0]}\'")\n        31.9% that image 0 is \'a photo of 2 cats\'\n        ```"""\n        # Use SigLIP model\'s config for some fields (if specified) instead of those of vision & text components.\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        vision_outputs = self.vision_model(\n            pixel_values=pixel_values,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            interpolate_pos_encoding=interpolate_pos_encoding,\n        )\n\n        text_outputs = self.text_model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n        )\n\n        image_embeds = vision_outputs[1]\n        text_embeds = text_outputs[1]\n\n        # normalized features\n        image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)\n        text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)\n\n        # cosine similarity as logits\n        logits_per_text = (\n            torch.matmul(text_embeds, image_embeds.t().to(text_embeds.device)) * self.logit_scale.exp()\n            + self.logit_bias\n        )\n        logits_per_image = logits_per_text.t()\n\n        loss = None\n        if return_loss:\n            # Adapted from https://github.com/google-research/big_vision/blob/01edb81a4716f93a48be43b3a4af14e29cdb3a7f/big_vision/trainers/proj/image_text/siglip.py#L287\n            eye = torch.eye(logits_per_text.size(0), device=logits_per_text.device)\n            m1_diag1 = -torch.ones_like(logits_per_text) + 2 * eye\n            loglik = torch.nn.functional.logsigmoid(m1_diag1 * logits_per_text)\n            nll = -torch.sum(loglik, dim=-1)\n            loss = nll.mean()\n\n        if not return_dict:\n            output = (logits_per_image, logits_per_text, text_embeds, image_embeds, text_outputs, vision_outputs)\n            return ((loss,) + output) if loss is not None else output\n\n        return SiglipOutput(\n            loss=loss,\n            logits_per_image=logits_per_image,\n            logits_per_text=logits_per_text,\n            text_embeds=text_embeds,\n            image_embeds=image_embeds,\n            text_model_output=text_outputs,\n            vision_model_output=vision_outputs,\n        )\n\n\n@add_start_docstrings(\n    """\n    SigLIP vision encoder with an image classification head on top (a linear layer on top of the pooled final hidden states of\n    the patch tokens) e.g. for ImageNet.\n    """,\n    SIGLIP_START_DOCSTRING,\n)\nclass SiglipForImageClassification(SiglipPreTrainedModel):\n    main_input_name = "pixel_values"\n\n    def __init__(self, config: SiglipConfig) -> None:\n        super().__init__(config)\n\n        self.num_labels = config.num_labels\n\n        # Create the vision model with proper attention\n        # and take only vision_model submodule (for backward compatibility)\n        vision_model = SiglipVisionModel._from_config(config.vision_config)\n        self.vision_model = vision_model.vision_model\n\n        # Classifier head\n        self.classifier = (\n            nn.Linear(config.vision_config.hidden_size, config.num_labels) if config.num_labels > 0 else nn.Identity()\n        )\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    @add_start_docstrings_to_model_forward(SIGLIP_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=ImageClassifierOutput, config_class=_CONFIG_FOR_DOC)\n    def forward(\n        self,\n        pixel_values: Optional[torch.Tensor] = None,\n        labels: Optional[torch.Tensor] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        interpolate_pos_encoding: bool = False,\n    ) -> Union[tuple, ImageClassifierOutput]:\n        r"""\n        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):\n            Labels for computing the image classification/regression loss. Indices should be in `[0, ...,\n            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If\n            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).\n\n        Returns:\n\n        Examples:\n\n        ```python\n        >>> from transformers import AutoImageProcessor, SiglipForImageClassification\n        >>> import torch\n        >>> from PIL import Image\n        >>> import requests\n\n        >>> torch.manual_seed(3)  # doctest: +IGNORE_RESULT\n        >>> url = "http://images.cocodataset.org/val2017/000000039769.jpg"\n        >>> image = Image.open(requests.get(url, stream=True).raw)\n\n        >>> # note: we are loading a `SiglipModel` from the hub here,\n        >>> # so the head will be randomly initialized, hence the predictions will be random if seed is not set above.\n        >>> image_processor = AutoImageProcessor.from_pretrained("google/siglip-base-patch16-224")\n        >>> model = SiglipForImageClassification.from_pretrained("google/siglip-base-patch16-224")\n\n        >>> inputs = image_processor(images=image, return_tensors="pt")\n        >>> outputs = model(**inputs)\n        >>> logits = outputs.logits\n        >>> # model predicts one of the two classes\n        >>> predicted_class_idx = logits.argmax(-1).item()\n        >>> print("Predicted class:", model.config.id2label[predicted_class_idx])\n        Predicted class: LABEL_1\n        ```"""\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        outputs = self.vision_model(\n            pixel_values,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            interpolate_pos_encoding=interpolate_pos_encoding,\n        )\n\n        sequence_output = outputs[0]\n\n        # average pool the patch tokens\n        sequence_output = torch.mean(sequence_output, dim=1)\n        # apply classifier\n        logits = self.classifier(sequence_output)\n\n        loss = None\n        if labels is not None:\n            # move labels to correct device to enable model parallelism\n            labels = labels.to(logits.device)\n            if self.config.problem_type is None:\n                if self.num_labels == 1:\n                    self.config.problem_type = "regression"\n                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):\n                    self.config.problem_type = "single_label_classification"\n                else:\n                    self.config.problem_type = "multi_label_classification"\n\n            if self.config.problem_type == "regression":\n                loss_fct = MSELoss()\n                if self.num_labels == 1:\n                    loss = loss_fct(logits.squeeze(), labels.squeeze())\n                else:\n                    loss = loss_fct(logits, labels)\n            elif self.config.problem_type == "single_label_classification":\n                loss_fct = CrossEntropyLoss()\n                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))\n            elif self.config.problem_type == "multi_label_classification":\n                loss_fct = BCEWithLogitsLoss()\n                loss = loss_fct(logits, labels)\n\n        if not return_dict:\n            output = (logits,) + outputs[2:]\n            return ((loss,) + output) if loss is not None else output\n\n        return ImageClassifierOutput(\n            loss=loss,\n            logits=logits,\n            hidden_states=outputs.hidden_states,\n            attentions=outputs.attentions,\n        )\n', 'modeling/siglip/processing_siglip.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""\nImage/Text processor class for SigLIP.\n"""\n\nfrom typing import List, Optional, Union\n\nfrom transformers.feature_extraction_utils import BatchFeature\nfrom transformers.image_utils import ImageInput\nfrom transformers.processing_utils import ProcessorMixin\nfrom transformers.tokenization_utils_base import PaddingStrategy, PreTokenizedInput, TextInput, TruncationStrategy\nfrom transformers.utils import TensorType\n\n\nclass SiglipProcessor(ProcessorMixin):\n    r"""\n    Constructs a Siglip processor which wraps a Siglip image processor and a Siglip tokenizer into a single processor.\n\n    [`SiglipProcessor`] offers all the functionalities of [`SiglipImageProcessor`] and [`SiglipTokenizer`]. See the\n    [`~SiglipProcessor.__call__`] and [`~SiglipProcessor.decode`] for more information.\n\n    Args:\n        image_processor ([`SiglipImageProcessor`]):\n            The image processor is a required input.\n        tokenizer ([`SiglipTokenizer`]):\n            The tokenizer is a required input.\n    """\n\n    attributes = ["image_processor", "tokenizer"]\n    image_processor_class = "SiglipImageProcessor"\n    tokenizer_class = "SiglipTokenizer"\n\n    def __init__(self, image_processor, tokenizer):\n        super().__init__(image_processor, tokenizer)\n\n    def __call__(\n        self,\n        text: Union[TextInput, PreTokenizedInput, List[TextInput], List[PreTokenizedInput]] = None,\n        images: ImageInput = None,\n        padding: Union[bool, str, PaddingStrategy] = False,\n        truncation: Union[bool, str, TruncationStrategy] = None,\n        max_length: int = None,\n        return_tensors: Optional[Union[str, TensorType]] = TensorType.PYTORCH,\n    ) -> BatchFeature:\n        """\n        Main method to prepare for the model one or several sequences(s) and image(s). This method forwards the `text`\n        and `kwargs` arguments to SiglipTokenizer\'s [`~SiglipTokenizer.__call__`] if `text` is not `None` to encode\n        the text. To prepare the image(s), this method forwards the `images` argument to\n        SiglipImageProcessor\'s [`~SiglipImageProcessor.__call__`] if `images` is not `None`. Please refer to the doctsring\n        of the above two methods for more information.\n\n        Args:\n            text (`str`, `List[str]`, `List[List[str]]`):\n                The sequence or batch of sequences to be encoded. Each sequence can be a string or a list of strings\n                (pretokenized string). If the sequences are provided as list of strings (pretokenized), you must set\n                `is_split_into_words=True` (to lift the ambiguity with a batch of sequences).\n            images (`PIL.Image.Image`, `np.ndarray`, `torch.Tensor`, `List[PIL.Image.Image]`, `List[np.ndarray]`, `List[torch.Tensor]`):\n                The image or batch of images to be prepared. Each image can be a PIL image, NumPy array or PyTorch\n                tensor. Both channels-first and channels-last formats are supported.\n            padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `False`):\n                Select a strategy to pad the returned sequences (according to the model\'s padding side and padding\n                index) among:\n                - `True` or `\'longest\'`: Pad to the longest sequence in the batch (or no padding if only a single\n                  sequence if provided).\n                - `\'max_length\'`: Pad to a maximum length specified with the argument `max_length` or to the maximum\n                  acceptable input length for the model if that argument is not provided.\n                - `False` or `\'do_not_pad\'` (default): No padding (i.e., can output a batch with sequences of different\n                  lengths).\n            max_length (`int`, *optional*):\n                Maximum length of the returned list and optionally padding length (see above).\n            truncation (`bool`, *optional*):\n                Activates truncation to cut input sequences longer than `max_length` to `max_length`.\n            return_tensors (`str` or [`~utils.TensorType`], *optional*):\n                If set, will return tensors of a particular framework. Acceptable values are:\n\n                - `\'tf\'`: Return TensorFlow `tf.constant` objects.\n                - `\'pt\'`: Return PyTorch `torch.Tensor` objects.\n                - `\'np\'`: Return NumPy `np.ndarray` objects.\n                - `\'jax\'`: Return JAX `jnp.ndarray` objects.\n\n        Returns:\n            [`BatchFeature`]: A [`BatchFeature`] with the following fields:\n\n            - **input_ids** -- List of token ids to be fed to a model. Returned when `text` is not `None`.\n            - **attention_mask** -- List of indices specifying which tokens should be attended to by the model (when\n              `return_attention_mask=True` or if *"attention_mask"* is in `self.model_input_names` and if `text` is not\n              `None`).\n            - **pixel_values** -- Pixel values to be fed to a model. Returned when `images` is not `None`.\n        """\n\n        if text is None and images is None:\n            raise ValueError("You have to specify either text or images. Both cannot be none.")\n\n        if text is not None:\n            encoding = self.tokenizer(\n                text, return_tensors=return_tensors, padding=padding, truncation=truncation, max_length=max_length\n            )\n\n        if images is not None:\n            image_features = self.image_processor(images, return_tensors=return_tensors)\n\n        if text is not None and images is not None:\n            encoding["pixel_values"] = image_features.pixel_values\n            return encoding\n        elif text is not None:\n            return encoding\n        else:\n            return BatchFeature(data=dict(**image_features), tensor_type=return_tensors)\n\n    def decode(self, *args, **kwargs):\n        """\n        This method forwards all its arguments to SiglipTokenizer\'s [`~PreTrainedTokenizer.decode`]. Please refer to\n        the docstring of this method for more information.\n        """\n        return self.tokenizer.decode(*args, **kwargs)\n\n    def batch_decode(self, *args, **kwargs):\n        """\n        This method forwards all its arguments to SiglipTokenizer\'s [`~PreTrainedTokenizer.batch_decode`]. Please\n        refer to the docstring of this method for more information.\n        """\n        return self.tokenizer.batch_decode(*args, **kwargs)\n\n    @property\n    # Copied from transformers.models.clip.processing_clip.CLIPProcessor.model_input_names with CLIP->Siglip, T5->Siglip\n    def model_input_names(self):\n        tokenizer_input_names = self.tokenizer.model_input_names\n        image_processor_input_names = self.image_processor.model_input_names\n        return list(dict.fromkeys(tokenizer_input_names + image_processor_input_names))\n', 'modeling/siglip/tokenization_siglip.py': '# Copyright 2024 The HuggingFace Inc. team.\n# SPDX-License-Identifier: Apache-2.0\n\n"""Tokenization class for SigLIP model."""\n\nimport os\nimport re\nimport string\nimport warnings\nfrom shutil import copyfile\nfrom typing import TYPE_CHECKING, Any, Dict, List, Optional, Tuple\n\nimport sentencepiece as spm\n\nfrom transformers.convert_slow_tokenizer import import_protobuf\nfrom transformers.tokenization_utils import PreTrainedTokenizer\nfrom transformers.tokenization_utils_base import AddedToken\n\n\nif TYPE_CHECKING:\n    from transformers.tokenization_utils_base import TextInput\nfrom transformers.utils import logging, requires_backends\n\n\nlogger = logging.get_logger(__name__)\n\nVOCAB_FILES_NAMES = {"vocab_file": "spiece.model"}\n\n\nSPIECE_UNDERLINE = "▁"\n\n\nclass SiglipTokenizer(PreTrainedTokenizer):\n    """\n    Construct a Siglip tokenizer. Based on [SentencePiece](https://github.com/google/sentencepiece).\n\n    This tokenizer inherits from [`PreTrainedTokenizer`] which contains most of the main methods. Users should refer to\n    this superclass for more information regarding those methods.\n\n    Args:\n        vocab_file (`str`):\n            [SentencePiece](https://github.com/google/sentencepiece) file (generally has a *.spm* extension) that\n            contains the vocabulary necessary to instantiate a tokenizer.\n        eos_token (`str`, *optional*, defaults to `"</s>"`):\n            The end of sequence token.\n        unk_token (`str`, *optional*, defaults to `"<unk>"`):\n            The unknown token. A token that is not in the vocabulary cannot be converted to an ID and is set to be this\n            token instead.\n        pad_token (`str`, *optional*, defaults to `"</s>"`):\n            The token used for padding, for example when batching sequences of different lengths.\n        additional_special_tokens (`List[str]`, *optional*):\n            Additional special tokens used by the tokenizer.\n        sp_model_kwargs (`dict`, *optional*):\n            Will be passed to the `SentencePieceProcessor.__init__()` method. The [Python wrapper for\n            SentencePiece](https://github.com/google/sentencepiece/tree/master/python) can be used, among other things,\n            to set:\n\n            - `enable_sampling`: Enable subword regularization.\n            - `nbest_size`: Sampling parameters for unigram. Invalid for BPE-Dropout.\n\n              - `nbest_size = {0,1}`: No sampling is performed.\n              - `nbest_size > 1`: samples from the nbest_size results.\n              - `nbest_size < 0`: assuming that nbest_size is infinite and samples from the all hypothesis (lattice)\n                using forward-filtering-and-backward-sampling algorithm.\n\n            - `alpha`: Smoothing parameter for unigram sampling, and dropout probability of merge operations for\n              BPE-dropout.\n        model_max_length (`int`, *optional*, defaults to 64):\n            The maximum length (in number of tokens) for model inputs.\n        do_lower_case (`bool`, *optional*, defaults to `True`):\n            Whether or not to lowercase the input when tokenizing.\n    """\n\n    vocab_files_names = VOCAB_FILES_NAMES\n    model_input_names = ["input_ids", "attention_mask"]\n\n    def __init__(\n        self,\n        vocab_file,\n        eos_token="</s>",\n        unk_token="<unk>",\n        pad_token="</s>",\n        additional_special_tokens=None,\n        sp_model_kwargs: Optional[Dict[str, Any]] = None,\n        model_max_length=64,\n        do_lower_case=True,\n        **kwargs,\n    ) -> None:\n        requires_backends(self, "protobuf")\n\n        pad_token = (\n            AddedToken(pad_token, rstrip=True, lstrip=True, normalized=False, special=True)\n            if isinstance(pad_token, str)\n            else pad_token\n        )\n        unk_token = (\n            AddedToken(unk_token, rstrip=True, lstrip=True, normalized=False, special=True)\n            if isinstance(unk_token, str)\n            else unk_token\n        )\n        eos_token = (\n            AddedToken(eos_token, rstrip=True, lstrip=True, normalized=False, special=True)\n            if isinstance(eos_token, str)\n            else eos_token\n        )\n\n        self.sp_model_kwargs = {} if sp_model_kwargs is None else sp_model_kwargs\n\n        self.do_lower_case = do_lower_case\n        self.vocab_file = vocab_file\n\n        self.sp_model = self.get_spm_processor()\n        self.vocab_file = vocab_file\n\n        super().__init__(\n            eos_token=eos_token,\n            unk_token=unk_token,\n            pad_token=pad_token,\n            additional_special_tokens=additional_special_tokens,\n            sp_model_kwargs=self.sp_model_kwargs,\n            model_max_length=model_max_length,\n            do_lower_case=do_lower_case,\n            **kwargs,\n        )\n\n    def get_spm_processor(self):\n        tokenizer = spm.SentencePieceProcessor(**self.sp_model_kwargs)\n        with open(self.vocab_file, "rb") as f:\n            sp_model = f.read()\n            model_pb2 = import_protobuf()\n            model = model_pb2.ModelProto.FromString(sp_model)\n            normalizer_spec = model_pb2.NormalizerSpec()\n            normalizer_spec.add_dummy_prefix = False\n            model.normalizer_spec.MergeFrom(normalizer_spec)\n            sp_model = model.SerializeToString()\n            tokenizer.LoadFromSerializedProto(sp_model)\n        return tokenizer\n\n    @property\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.vocab_size\n    def vocab_size(self):\n        return self.sp_model.get_piece_size()\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.get_vocab\n    def get_vocab(self):\n        vocab = {self.convert_ids_to_tokens(i): i for i in range(self.vocab_size)}\n        vocab.update(self.added_tokens_encoder)\n        return vocab\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.get_special_tokens_mask\n    def get_special_tokens_mask(\n        self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None, already_has_special_tokens: bool = False\n    ) -> List[int]:\n        """\n        Retrieve sequence ids from a token list that has no special tokens added. This method is called when adding\n        special tokens using the tokenizer `prepare_for_model` method.\n\n        Args:\n            token_ids_0 (`List[int]`):\n                List of IDs.\n            token_ids_1 (`List[int]`, *optional*):\n                Optional second list of IDs for sequence pairs.\n            already_has_special_tokens (`bool`, *optional*, defaults to `False`):\n                Whether or not the token list is already formatted with special tokens for the model.\n\n        Returns:\n            `List[int]`: A list of integers in the range [0, 1]: 1 for a special token, 0 for a sequence token.\n        """\n        if already_has_special_tokens:\n            return super().get_special_tokens_mask(\n                token_ids_0=token_ids_0, token_ids_1=token_ids_1, already_has_special_tokens=True\n            )\n\n        # normal case: some special tokens\n        if token_ids_1 is None:\n            return ([0] * len(token_ids_0)) + [1]\n        return ([0] * len(token_ids_0)) + [1] + ([0] * len(token_ids_1)) + [1]\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer._add_eos_if_not_present\n    def _add_eos_if_not_present(self, token_ids: List[int]) -> List[int]:\n        """Do not add eos again if user already added it."""\n        if len(token_ids) > 0 and token_ids[-1] == self.eos_token_id:\n            warnings.warn(\n                f"This sequence already has {self.eos_token}. In future versions this behavior may lead to duplicated"\n                " eos tokens being added."\n            )\n            return token_ids\n        else:\n            return token_ids + [self.eos_token_id]\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.create_token_type_ids_from_sequences\n    def create_token_type_ids_from_sequences(\n        self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None\n    ) -> List[int]:\n        """\n        Create a mask from the two sequences passed to be used in a sequence-pair classification task. T5 does not make\n        use of token type ids, therefore a list of zeros is returned.\n\n        Args:\n            token_ids_0 (`List[int]`):\n                List of IDs.\n            token_ids_1 (`List[int]`, *optional*):\n                Optional second list of IDs for sequence pairs.\n\n        Returns:\n            `List[int]`: List of zeros.\n        """\n        eos = [self.eos_token_id]\n\n        if token_ids_1 is None:\n            return len(token_ids_0 + eos) * [0]\n        return len(token_ids_0 + eos + token_ids_1 + eos) * [0]\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.build_inputs_with_special_tokens\n    def build_inputs_with_special_tokens(\n        self, token_ids_0: List[int], token_ids_1: Optional[List[int]] = None\n    ) -> List[int]:\n        """\n        Build model inputs from a sequence or a pair of sequence for sequence classification tasks by concatenating and\n        adding special tokens. A sequence has the following format:\n\n        - single sequence: `X </s>`\n        - pair of sequences: `A </s> B </s>`\n\n        Args:\n            token_ids_0 (`List[int]`):\n                List of IDs to which the special tokens will be added.\n            token_ids_1 (`List[int]`, *optional*):\n                Optional second list of IDs for sequence pairs.\n\n        Returns:\n            `List[int]`: List of [input IDs](../glossary#input-ids) with the appropriate special tokens.\n        """\n        token_ids_0 = self._add_eos_if_not_present(token_ids_0)\n        if token_ids_1 is None:\n            return token_ids_0\n        else:\n            token_ids_1 = self._add_eos_if_not_present(token_ids_1)\n            return token_ids_0 + token_ids_1\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.__getstate__\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state["sp_model"] = None\n        return state\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.__setstate__\n    def __setstate__(self, d):\n        self.__dict__ = d\n\n        # for backward compatibility\n        if not hasattr(self, "sp_model_kwargs"):\n            self.sp_model_kwargs = {}\n\n        self.sp_model = spm.SentencePieceProcessor(**self.sp_model_kwargs)\n        self.sp_model.Load(self.vocab_file)\n\n    def remove_punctuation(self, text: str) -> str:\n        return text.translate(str.maketrans("", "", string.punctuation))\n\n    # source: https://github.com/google-research/big_vision/blob/3b8e5ab6ad4f96e32b32826f9e1b8fd277914f9c/big_vision/evaluators/proj/image_text/prompt_engineering.py#L94\n    def canonicalize_text(self, text, *, keep_punctuation_exact_string=None):\n        """Returns canonicalized `text` (puncuation removed).\n\n        Args:\n            text (`str`):\n                String to be canonicalized.\n            keep_punctuation_exact_string (`str`, *optional*):\n                If provided, then this exact string is kept. For example providing \'{}\' will keep any occurrences of \'{}\'\n                (but will still remove \'{\' and \'}\' that appear separately).\n        """\n        if keep_punctuation_exact_string:\n            text = keep_punctuation_exact_string.join(\n                self.remove_punctuation(part) for part in text.split(keep_punctuation_exact_string)\n            )\n        else:\n            text = self.remove_punctuation(text)\n        text = re.sub(r"\\s+", " ", text)\n        text = text.strip()\n\n        return text\n\n    def tokenize(self, text: "TextInput", add_special_tokens=False, **kwargs) -> List[str]:\n        """\n        Converts a string to a list of tokens.\n        """\n        tokens = super().tokenize(SPIECE_UNDERLINE + text.replace(SPIECE_UNDERLINE, " "), **kwargs)\n\n        if len(tokens) > 1 and tokens[0] == SPIECE_UNDERLINE and tokens[1] in self.all_special_tokens:\n            tokens = tokens[1:]\n        return tokens\n\n    @property\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.unk_token_length\n    def unk_token_length(self):\n        return len(self.sp_model.encode(str(self.unk_token)))\n\n    def _tokenize(self, text, **kwargs):\n        """\n        Returns a tokenized string.\n\n        We de-activated the `add_dummy_prefix` option, thus the sentencepiece internals will always strip any\n        SPIECE_UNDERLINE.\n\n        For example: `self.sp_model.encode(f"{SPIECE_UNDERLINE}Hey", out_type = str)` will give `[\'H\', \'e\', \'y\']` instead of `[\'▁He\', \'y\']`.\n\n        Thus we always encode `f"{unk_token}text"` and strip the `unk_token`. Here is an example with `unk_token = "<unk>"` and `unk_token_length = 4`.\n        `self.tokenizer.sp_model.encode("<unk> Hey", out_type = str)[4:]`.\n        """\n        text = self.canonicalize_text(text, keep_punctuation_exact_string=None)\n        tokens = self.sp_model.encode(text, out_type=str)\n\n        # 1. Encode string + prefix ex: "<unk> Hey"\n        tokens = self.sp_model.encode(self.unk_token + text, out_type=str)\n        # 2. Remove self.unk_token from [\'<\',\'unk\',\'>\', \'▁Hey\']\n        return tokens[self.unk_token_length :] if len(tokens) >= self.unk_token_length else tokens\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer._convert_token_to_id\n    def _convert_token_to_id(self, token):\n        """Converts a token (str) in an id using the vocab."""\n        return self.sp_model.piece_to_id(token)\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer._convert_id_to_token\n    def _convert_id_to_token(self, index):\n        """Converts an index (integer) in a token (str) using the vocab."""\n        token = self.sp_model.IdToPiece(index)\n        return token\n\n    def convert_tokens_to_string(self, tokens):\n        """Converts a sequence of tokens (string) in a single string."""\n        current_sub_tokens = []\n        out_string = ""\n        prev_is_special = False\n        for token in tokens:\n            # make sure that special tokens are not decoded using sentencepiece model\n            if token in self.all_special_tokens:\n                if not prev_is_special:\n                    out_string += " "\n                out_string += self.sp_model.decode(current_sub_tokens) + token\n                prev_is_special = True\n                current_sub_tokens = []\n            else:\n                current_sub_tokens.append(token)\n                prev_is_special = False\n        out_string += self.sp_model.decode(current_sub_tokens)\n        return out_string.strip()\n\n    # Copied from transformers.models.t5.tokenization_t5.T5Tokenizer.save_vocabulary\n    def save_vocabulary(self, save_directory: str, filename_prefix: Optional[str] = None) -> Tuple[str]:\n        if not os.path.isdir(save_directory):\n            logger.error(f"Vocabulary path ({save_directory}) should be a directory")\n            return\n        out_vocab_file = os.path.join(\n            save_directory, (filename_prefix + "-" if filename_prefix else "") + VOCAB_FILES_NAMES["vocab_file"]\n        )\n\n        if os.path.abspath(self.vocab_file) != os.path.abspath(out_vocab_file) and os.path.isfile(self.vocab_file):\n            copyfile(self.vocab_file, out_vocab_file)\n        elif not os.path.isfile(self.vocab_file):\n            with open(out_vocab_file, "wb") as fi:\n                content_spiece_model = self.sp_model.serialized_model_proto()\n                fi.write(content_spiece_model)\n\n        return (out_vocab_file,)\n', 'requirements.txt': 'accelerate==1.10.0\naddict==2.4.0\naiofiles==24.1.0\naiohappyeyeballs==2.6.1\naiohttp==3.13.3\naiosignal==1.4.0\nannotated-types==0.7.0\nanyio==4.10.0\nasync-timeout==5.0.1\nattrs==25.4.0\nbitsandbytes==0.46.1\nboto3==1.40.16\nbotocore==1.40.16\nBrotli==1.1.0\ncertifi==2025.8.3\ncharset-normalizer==3.4.3\nclick==8.2.1\ncontourpy==1.3.2\ncycler==0.12.1\ndatasets==4.7.0\ndecord==0.6.0\ndill==0.4.0\neinops==0.8.1\nexceptiongroup==1.3.0\nfastapi==0.116.1\nffmpy==0.6.1\nfilelock==3.13.1\nflash-attn==2.5.8\nfonttools==4.59.0\nfrozenlist==1.8.0\nfsspec==2024.2.0\ngitdb==4.0.12\nGitPython==3.1.45\ngradio==6.12.0\ngradio_client==2.4.1\ngroovy==0.1.2\nh11==0.16.0\nhf-gradio==0.4.1\nhf-xet==1.4.2\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface_hub==0.36.2\nidna==3.10\nImageIO==2.37.3\nJinja2==3.1.3\njmespath==1.0.1\njoblib==1.5.3\nkiwisolver==1.4.8\nlazy-loader==0.5\nmarkdown-it-py==3.0.0\nMarkupSafe==2.1.5\nmatplotlib==3.7.0\nmdurl==0.1.2\nmmengine==0.10.7\nmpmath==1.3.0\nmultidict==6.7.1\nmultiprocess==0.70.18\nnetworkx==3.2.1\nninja==1.11.1.4\nnumpy==1.24.4\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-ml-py==12.575.51\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\nnvitop==1.5.2\nopencv-python==4.7.0.72\norjson==3.11.1\npackaging==25.0\npandas==2.3.1\npillow==10.2.0\nplatformdirs==4.3.8\nprettytable==3.16.0\npropcache==0.4.1\nprotobuf==6.31.1\npsutil==7.0.0\npyarrow==23.0.1\npydantic==2.11.7\npydantic_core==2.33.2\npydub==0.25.1\npyecharts==2.0.8\nPygments==2.19.2\npynvml==12.0.0\npyparsing==3.2.3\npython-dateutil==2.9.0.post0\npython-multipart==0.0.20\npytz==2025.2\nPyYAML==6.0.2\nregex==2025.7.34\nrequests==2.32.3\nrich==13.9.4\nruff==0.12.8\ns3transfer==0.13.1\nsafehttpx==0.1.7\nsafetensors==0.4.5\nscikit-image==0.25.2\nscikit-learn==1.7.2\nscipy==1.15.3\nsemantic-version==2.10.0\nsentencepiece==0.1.99\nsentry-sdk==2.34.1\nshellingham==1.5.4\nsimplejson==3.20.1\nsix==1.17.0\nsmmap==5.0.2\nsniffio==1.3.1\nstarlette==0.47.2\nswankit==0.2.4\nswanlab==0.6.8\nsympy==1.13.1\ntermcolor==3.1.0\nthreadpoolctl==3.6.0\ntifffile==2025.5.10\ntokenizers==0.21.4\ntomli==2.2.1\ntomlkit==0.13.3\ntorch==2.5.1\ntorchaudio==2.5.0+cu124\ntorchvision==0.20.1\ntqdm==4.67.1\ntransformers==4.49.0\ntriton==3.1.0\ntyper==0.16.0\ntyping-inspection==0.4.1\ntyping_extensions==4.14.1\ntzdata==2025.2\nunzip==1.0.0\nurllib3==2.5.0\nuvicorn==0.35.0\nwandb==0.21.1\nwcwidth==0.2.13\nwebsockets==15.0.1\nwrapt==1.17.3\nxlsxwriter==3.2.5\nxxhash==3.6.0\nyapf==0.43.0\nyarl==1.23.0\n', 'utils.py': 'import os\nimport re\nimport time\nimport torch\nimport numpy as np\nfrom PIL import Image\nfrom pathlib import Path\nfrom copy import deepcopy\nfrom safetensors.torch import load_file\nfrom data.data_utils import pil_img2rgb\nfrom modeling.autoencoder import load_ae\nfrom modeling.qwen2 import Qwen2Tokenizer\nfrom data.transforms import ImageTransform\nfrom modeling.bagel.qwen2_navit import NaiveCache\nfrom typing import List, Dict, Optional, Union, Any\nfrom data.data_utils import pil_img2rgb, add_special_tokens\nfrom modeling.infer import FusionCharAnnotator, AnnotationRenderer\nfrom accelerate import infer_auto_device_map, load_checkpoint_and_dispatch, init_empty_weights\nfrom modeling.bagel import (\n    BagelConfig, Bagel, Qwen2Config, Qwen2ForCausalLM, SiglipVisionConfig, SiglipVisionModel\n)\nVLM_THINK_SYSTEM_PROMPT = \'\'\'You should first think about the reasoning process in the mind and then provide the user with the answer. \nThe reasoning process is enclosed within <think> </think> tags, i.e. <think> reasoning process here </think> answer here\'\'\'\n\nGEN_THINK_SYSTEM_PROMPT = \'\'\'You should first think about the planning process in the mind and then generate the image. \nThe planning process is enclosed within <think> </think> tags, i.e. <think> planning process here </think> image here\'\'\'\n\n\nclass InterleaveInferencer:\n    def __init__(self, model, vae_model, tokenizer, vae_transform, vae_transform_gen, vit_transform, new_token_ids):\n        self.model = model\n        self.vae_model = vae_model\n        self.tokenizer = tokenizer\n        self.vae_transform = vae_transform\n        self.vit_transform = vit_transform\n        self.vae_transform_gen = vae_transform_gen\n        self.new_token_ids = new_token_ids\n        self.fusion_render = FusionCharAnnotator(\n                                        opacity=160,\n                                        mask_color=(100, 150, 255),\n                                        mask_shape=\'square\',\n                                    )\n        self.anno_render = AnnotationRenderer(opacity=220,\n                                        font_size=42,\n                                        mask_color=(100, 150, 255),\n                                        text_color=(0, 0, 0),\n                                        mask_shape=\'square\',\n                                        text_bold=True,\n                                        bold_strength=1,\n                                        text_stroke_width=0,\n                                        text_stroke_color=(0, 0, 0)\n                                        )\n        \n    def init_gen_context(self): \n        gen_context = {\n            \'kv_lens\': [0],\n            \'ropes\': [0],\n            \'past_key_values\': NaiveCache(self.model.config.llm_config.num_hidden_layers),\n        }\n        return gen_context\n\n    @torch.no_grad()\n    def update_context_text(self, text, gen_context):\n\n        past_key_values = gen_context[\'past_key_values\']\n        kv_lens = gen_context[\'kv_lens\']\n        ropes = gen_context[\'ropes\']\n        generation_input, kv_lens, ropes = self.model.prepare_prompts(\n            curr_kvlens=kv_lens,\n            curr_rope=ropes, \n            prompts=[text],\n            tokenizer=self.tokenizer, \n            new_token_ids=self.new_token_ids,\n        )\n        past_key_values = self.model.forward_cache_update_text(past_key_values, **generation_input)        \n        gen_context[\'kv_lens\'] = kv_lens\n        gen_context[\'ropes\'] = ropes\n        gen_context[\'past_key_values\'] = past_key_values\n        \n        return gen_context\n\n    @torch.no_grad()\n    def update_context_image(self, image, gen_context, vae=True, vit=True):\n\n        assert vae or vit\n        past_key_values = gen_context[\'past_key_values\']\n        kv_lens = gen_context[\'kv_lens\']\n        ropes =  gen_context[\'ropes\']\n        \n        if vae:\n            ## update vae\n            generation_input, kv_lens, ropes = self.model.prepare_vae_images(\n                curr_kvlens=kv_lens,\n                curr_rope=ropes, \n                images=[image],\n                transforms=self.vae_transform, \n                new_token_ids=self.new_token_ids,\n            )\n            past_key_values = self.model.forward_cache_update_vae(self.vae_model, past_key_values, **generation_input)\n        \n        if vit:\n            ## update vit \n            generation_input, kv_lens, ropes = self.model.prepare_vit_images(\n                curr_kvlens=kv_lens,\n                curr_rope=ropes, \n                images=[image],\n                transforms=self.vit_transform, \n                new_token_ids=self.new_token_ids,\n            )\n            past_key_values = self.model.forward_cache_update_vit(past_key_values, **generation_input)\n        \n        gen_context[\'kv_lens\'] = kv_lens\n        gen_context[\'ropes\'] = ropes\n        gen_context[\'past_key_values\'] = past_key_values\n        \n        return gen_context\n\n    @torch.no_grad()\n    def update_context_image_gen(self, image, gen_context, vae=True, vit=True):\n\n        assert vae or vit\n        past_key_values = gen_context[\'past_key_values\']\n        kv_lens = gen_context[\'kv_lens\']\n        ropes =  gen_context[\'ropes\']\n        \n        if vae:\n            ## update vae\n            generation_input, kv_lens, ropes = self.model.prepare_vae_images(\n                curr_kvlens=kv_lens,\n                curr_rope=ropes, \n                images=[image],\n                transforms=self.vae_transform_gen, \n                new_token_ids=self.new_token_ids,\n            )\n            past_key_values = self.model.forward_cache_update_vae(self.vae_model, past_key_values, **generation_input)\n        \n        if vit:\n            ## update vit \n            generation_input, kv_lens, ropes = self.model.prepare_vit_images(\n                curr_kvlens=kv_lens,\n                curr_rope=ropes, \n                images=[image],\n                transforms=self.vit_transform, \n                new_token_ids=self.new_token_ids,\n            )\n            past_key_values = self.model.forward_cache_update_vit(past_key_values, **generation_input)\n        \n        gen_context[\'kv_lens\'] = kv_lens\n        gen_context[\'ropes\'] = ropes\n        gen_context[\'past_key_values\'] = past_key_values\n        \n        return gen_context\n\n    @torch.no_grad()\n    def gen_image(\n        self, \n        image_shape, \n        gen_context, \n        cfg_text_scale=4.0,\n        cfg_img_scale=1.5,\n\n        cfg_text_precontext=None, \n        cfg_img_precontext=None, \n        cfg_interval=(0.4, 1.0),\n        cfg_renorm_min=0.0,\n        cfg_renorm_type="global",\n        \n        num_timesteps=50, \n        timestep_shift=3.0\n    ):\n        past_key_values = gen_context[\'past_key_values\']\n        kv_lens = gen_context[\'kv_lens\']\n        ropes = gen_context[\'ropes\']\n        generation_input = self.model.prepare_vae_latent(\n            curr_kvlens=kv_lens,\n            curr_rope=ropes, \n            image_sizes=[image_shape], \n            new_token_ids=self.new_token_ids,\n        ) \n        \n        # text cfg\n        cfg_text_past_key_values = cfg_text_precontext[\'past_key_values\']\n        kv_lens_cfg = cfg_text_precontext[\'kv_lens\']\n        ropes_cfg = cfg_text_precontext[\'ropes\']\n        generation_input_cfg_text = self.model.prepare_vae_latent_cfg(\n            curr_kvlens=kv_lens_cfg,\n            curr_rope=ropes_cfg, \n            image_sizes=[image_shape], \n        )\n\n        # img cfg\n        cfg_img_past_key_values = cfg_img_precontext[\'past_key_values\']\n        kv_lens_cfg = cfg_img_precontext[\'kv_lens\']\n        ropes_cfg = cfg_img_precontext[\'ropes\']\n        generation_input_cfg_img = self.model.prepare_vae_latent_cfg(\n            curr_kvlens=kv_lens_cfg,\n            curr_rope=ropes_cfg, \n            image_sizes=[image_shape], \n        )\n\n        unpacked_latent = self.model.generate_image(\n            past_key_values=past_key_values,\n            cfg_text_past_key_values=cfg_text_past_key_values,\n            cfg_img_past_key_values=cfg_img_past_key_values,\n            num_timesteps=num_timesteps,\n            cfg_text_scale=cfg_text_scale,\n            cfg_img_scale=cfg_img_scale,\n            cfg_interval=cfg_interval,\n            cfg_renorm_min=cfg_renorm_min,\n            cfg_renorm_type=cfg_renorm_type,\n            timestep_shift=timestep_shift,\n            **generation_input,\n            cfg_text_packed_position_ids=generation_input_cfg_text[\'cfg_packed_position_ids\'],\n            cfg_text_packed_query_indexes=generation_input_cfg_text[\'cfg_packed_query_indexes\'],\n            cfg_text_key_values_lens=generation_input_cfg_text[\'cfg_key_values_lens\'],\n            cfg_text_packed_key_value_indexes=generation_input_cfg_text[\'cfg_packed_key_value_indexes\'],\n            cfg_img_packed_position_ids=generation_input_cfg_img[\'cfg_packed_position_ids\'],\n            cfg_img_packed_query_indexes=generation_input_cfg_img[\'cfg_packed_query_indexes\'],\n            cfg_img_key_values_lens=generation_input_cfg_img[\'cfg_key_values_lens\'],\n            cfg_img_packed_key_value_indexes=generation_input_cfg_img[\'cfg_packed_key_value_indexes\'],\n        )\n\n        image = self.decode_image(unpacked_latent[0], image_shape)\n        return image\n\n    def decode_image(self, latent, image_shape):\n        H, W = image_shape\n        h, w = H // self.model.latent_downsample, W // self.model.latent_downsample\n\n        latent = latent.reshape(1, h, w, self.model.latent_patch_size, self.model.latent_patch_size, self.model.latent_channel)\n        latent = torch.einsum("nhwpqc->nchpwq", latent)\n        latent = latent.reshape(1, self.model.latent_channel, h * self.model.latent_patch_size, w * self.model.latent_patch_size)\n\n        vae_dtype = next(self.vae_model.parameters()).dtype\n        vae_device = next(self.vae_model.parameters()).device\n        latent = latent.to(device=vae_device, dtype=vae_dtype)\n        image = self.vae_model.decode(latent)\n\n        image = (image * 0.5 + 0.5).clamp(0, 1)[0].permute(1, 2, 0) * 255\n        image = Image.fromarray((image).to(torch.uint8).cpu().numpy())\n\n        return image\n\n    @torch.no_grad()\n    def gen_text(self, gen_context, max_length: int = 500, do_sample: bool = True, temperature: float = 1.0):\n        gen_context = deepcopy(gen_context)\n        past_key_values = gen_context[\'past_key_values\']\n        kv_lens = gen_context[\'kv_lens\']\n        ropes = gen_context[\'ropes\']\n\n        generation_input = self.model.prepare_start_tokens(kv_lens, ropes, self.new_token_ids)\n        unpacked_latent = self.model.generate_text(\n            past_key_values=past_key_values,\n            max_length=max_length,\n            do_sample=do_sample,\n            temperature=temperature,\n            end_token_id=self.new_token_ids[\'eos_token_id\'],\n            **generation_input,\n        )\n        output = self.tokenizer.decode(unpacked_latent[:,0])\n        output = output.split(\'<|im_end|>\')[0].split(\'<|im_start|>\')[1]\n        return output\n\n\nclass UniHIRInferencer(InterleaveInferencer):\n\n\n    def merge_ocr_texts(self, original_text, correction_text):\n        """\n        整合两段OCR文本，根据修正文本更新原始文本\n        \n        Args:\n            original_text: 原始文本（第一段）\n            correction_text: 修正文本（第二段）\n        \n        Returns:\n            整合后的新文本\n        """\n        metadata_match = re.search(r\'这张古籍图像出自《([^》]+)》，刻经年代为([^。]+)。\', correction_text)\n        if metadata_match:\n            metadata = f"这张古籍图像出自《{metadata_match.group(1)}》，刻经年代为{metadata_match.group(2)}。"\n        else:\n            metadata = ""\n        \n        if \'我没有发现需要修改的地方\' in correction_text:\n            return metadata + original_text if metadata else original_text\n        \n        box_pattern = r\'<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),([^<]+)<\\|box_end\\|>\'\n        original_boxes = {}\n        original_boxes_order = []\n        \n        for match in re.finditer(box_pattern, original_text):\n            x1, y1, x2, y2, char = match.groups()\n            coord_key = (int(x1), int(y1), int(x2), int(y2))\n            original_boxes[coord_key] = char\n            original_boxes_order.append(coord_key)\n        \n        def coord_distance(coord1, coord2):\n            x1_1, y1_1, x2_1, y2_1 = coord1\n            x1_2, y1_2, x2_2, y2_2 = coord2\n            center1_x = (x1_1 + x2_1) / 2\n            center1_y = (y1_1 + y2_1) / 2\n            center2_x = (x1_2 + x2_2) / 2\n            center2_y = (y1_2 + y2_2) / 2\n            return ((center1_x - center2_x) ** 2 + (center1_y - center2_y) ** 2) ** 0.5\n        \n        def sort_boxes(boxes_dict):\n            if not boxes_dict:\n                return []\n            \n            box_centers = {}\n            total_width = 0\n            for coord in boxes_dict.keys():\n                x1, y1, x2, y2 = coord\n                center_x = (x1 + x2) / 2\n                center_y = (y1 + y2) / 2\n                box_centers[coord] = (center_x, center_y)\n                total_width += (x2 - x1)\n            \n            avg_box_width = total_width / len(boxes_dict) if boxes_dict else 30\n            coords_list = list(boxes_dict.keys())\n            columns = [[coord] for coord in coords_list]\n            column_threshold = avg_box_width * 0.5\n            \n            merged = True\n            while merged:\n                merged = False\n                new_columns = []\n                used = set()\n                \n                for i, col1 in enumerate(columns):\n                    if i in used:\n                        continue\n\n                    avg_x1 = sum(box_centers[c][0] for c in col1) / len(col1)\n                    merged_col = list(col1)\n                    for j, col2 in enumerate(columns[i+1:], start=i+1):\n                        if j in used:\n                            continue\n                        \n                        avg_x2 = sum(box_centers[c][0] for c in col2) / len(col2)\n                        if abs(avg_x1 - avg_x2) < column_threshold:\n                            merged_col.extend(col2)\n                            used.add(j)\n                            merged = True\n                    \n                    new_columns.append(merged_col)\n                    used.add(i)\n                \n                columns = new_columns\n\n            for column in columns:\n                column.sort(key=lambda c: box_centers[c][1])\n            columns.sort(key=lambda col: -sum(box_centers[c][0] for c in col) / len(col))\n            \n            sorted_coords = []\n            for column in columns:\n                sorted_coords.extend(column)\n            \n            return sorted_coords\n\n        corrections = {}\n        if \'我没有发现破损字符预测错误\' not in correction_text and \'以下的破损字符预测错误\' in correction_text:\n            correction_section = re.search(r\'以下的破损字符预测错误.*?\\n(.*?)(?:\\n\\n+|$)\', correction_text, re.DOTALL)\n            if correction_section:\n                for match in re.finditer(box_pattern, correction_section.group(1)):\n                    x1, y1, x2, y2, new_char = match.groups()\n                    coord_key = (int(x1), int(y1), int(x2), int(y2))\n                    corrections[coord_key] = new_char\n        \n\n        new_boxes = {}\n        if \'我没有发现破损位置需要新增框\' not in correction_text and \'以下的破损位置需要新增框\' in correction_text:\n            new_section = re.search(r\'以下的破损位置需要新增框.*?\\n(.*?)(?:\\n\\n+|$)\', correction_text, re.DOTALL)\n            if new_section:\n                for match in re.finditer(box_pattern, new_section.group(1)):\n                    x1, y1, x2, y2, char = match.groups()\n                    coord_key = (int(x1), int(y1), int(x2), int(y2))\n                    new_boxes[coord_key] = char\n\n        delete_coords = set()\n        if \'我没有发现需要删除的框\' not in correction_text and \'以下的框需要删除\' in correction_text:\n            delete_section = re.search(\n                r\'以下的框需要删除[。\\s]*\\n(.*?)(?=\\n\\n+|\\n[0-9１-９]）|$)\', \n                correction_text, \n                re.DOTALL\n            )\n            \n            if delete_section:\n                delete_text = delete_section.group(1).strip()\n                \n                for match in re.finditer(box_pattern, delete_text):\n                    x1, y1, x2, y2, _ = match.groups()\n                    coord_key = (int(x1), int(y1), int(x2), int(y2))\n                    delete_coords.add(coord_key)\n        \n        position_corrections = {}\n        if \'以下框的位置有偏移\' in correction_text or \'矫正后的结果如下\' in correction_text:\n            position_section = re.search(\n                r\'(?:以下框的位置有偏移|矫正后的结果如下)[：:：]*\\s*\\n(.*?)(?:\\n\\n+|$)\', \n                correction_text, \n                re.DOTALL\n            )\n            if position_section:\n                position_text = position_section.group(1).strip()\n\n                for match in re.finditer(box_pattern, position_text):\n                    new_x1, new_y1, new_x2, new_y2, char = match.groups()\n                    new_coord = (int(new_x1), int(new_y1), int(new_x2), int(new_y2))\n\n                    best_match = None\n                    min_distance = float(\'inf\')\n                    \n                    for old_coord, old_char in original_boxes.items():\n                        if old_char == char and old_coord != new_coord:\n                            distance = coord_distance(old_coord, new_coord)\n                            if distance < min_distance:\n                                min_distance = distance\n                                best_match = old_coord\n                    \n                    if best_match and min_distance < 100:\n                        position_corrections[best_match] = new_coord\n        \n        corrected_boxes = {}\n        \n        for old_coord in original_boxes_order:\n            char = original_boxes[old_coord]\n            \n            if old_coord in position_corrections:\n                new_coord = position_corrections[old_coord]\n                corrected_boxes[new_coord] = char\n            else:\n                corrected_boxes[old_coord] = char\n        \n        for coord, new_char in corrections.items():\n            if coord in corrected_boxes:\n                corrected_boxes[coord] = new_char\n        \n        corrected_boxes.update(new_boxes)\n        for old_coord in delete_coords:\n            if old_coord in position_corrections:\n                new_coord = position_corrections[old_coord]\n                corrected_boxes.pop(new_coord, None)\n            else:\n                corrected_boxes.pop(old_coord, None)\n        \n        sorted_coords = sort_boxes(corrected_boxes)\n        final_text = metadata\n        for coord in sorted_coords:\n            x1, y1, x2, y2 = coord\n            char = corrected_boxes[coord]\n            final_text += f"<|box_start|>{x1},{y1},{x2},{y2},{char}<|box_end|>"\n        \n        return final_text\n\n    @torch.no_grad()\n    def interleave_inference(\n        self,\n        input_lists: List[Union[str, Image.Image]],\n        think=False,\n        understanding_output=False,\n        und_only = False,\n        gen_only = False,\n        gen_first = False,\n        max_think_token_n=1000,\n        do_sample=False,\n        text_temperature=0.3,\n        cfg_text_scale=3.0,\n        cfg_img_scale=1.5,\n        cfg_interval=[0.4, 1.0],\n        timestep_shift=3.0,\n        num_timesteps=50,\n        cfg_renorm_min=0.0,\n        cfg_renorm_type="global",\n        image_shapes=(1024, 1024),\n        reflection_count=1,\n    ) -> List[Union[str, Image.Image]]:\n\n        output_list = []\n        gen_context = self.init_gen_context()\n        cfg_text_context = deepcopy(gen_context)\n        cfg_img_context = deepcopy(gen_context)\n\n        output_dict = {}\n\n        with torch.autocast(device_type="cuda", enabled=True, dtype=torch.bfloat16):\n            if think:\n                if understanding_output:\n                    system_prompt = VLM_THINK_SYSTEM_PROMPT \n                else:\n                    system_prompt = GEN_THINK_SYSTEM_PROMPT\n                gen_context = self.update_context_text(system_prompt, gen_context)\n                cfg_img_context = self.update_context_text(system_prompt, cfg_img_context)\n            \n            for input_term in input_lists:\n                if isinstance(input_term, str):\n                    cfg_text_context = deepcopy(gen_context)\n                    gen_context = self.update_context_text(input_term, gen_context)\n                    cfg_img_context = self.update_context_text(input_term, cfg_img_context)\n\n                elif isinstance(input_term, Image.Image):\n                    original_img = pil_img2rgb(input_term)\n                    input_term = self.vae_transform.resize_transform(pil_img2rgb(input_term))\n                    input_img = input_term\n                    gen_context = self.update_context_image(input_term, gen_context, vae=not understanding_output)\n\n                    image_shapes = input_term.size[::-1]\n                    cfg_text_context = deepcopy(gen_context)\n\n                else:\n                    raise ValueError(f"Unsupported input type: {type(input_term)}")\n            \n            img = self.gen_image(\n                image_shapes, \n                gen_context, \n                cfg_text_precontext=cfg_text_context, \n                cfg_img_precontext=cfg_img_context,\n                cfg_text_scale=cfg_text_scale, \n                cfg_img_scale=cfg_img_scale, \n                cfg_interval=cfg_interval, \n                timestep_shift=timestep_shift, \n                num_timesteps=num_timesteps,\n                cfg_renorm_min=cfg_renorm_min,\n                cfg_renorm_type=cfg_renorm_type,\n            )\n            output_list.append(img)\n            output_dict[\'draft\'] = img\n\n            input_term = self.vae_transform.resize_transform(pil_img2rgb(img))\n            gen_context = self.update_context_image(input_term, gen_context, vae=False)\n\n            gen_text = self.gen_text(gen_context, do_sample=do_sample, temperature=text_temperature, max_length=max_think_token_n)\n            # output_list.append(gen_text)\n\n        gen_context = self.init_gen_context()\n        cfg_text_context = deepcopy(gen_context)\n        cfg_img_context = deepcopy(gen_context)\n\n        print(\'中间步：\' + gen_text.split("\\n")[-1])\n        output_dict[\'text_step1\'] = gen_text\n        output_dict[\'text_merge\'] = gen_text\n        print(\'\\n\')\n\n        output_dict[\'tmp_img\'] = []\n        output_dict[\'tmp_img_gen\'] = []\n        output_dict[\'text_step2\'] = \'\'\n        \n        with torch.autocast(device_type="cuda", enabled=True, dtype=torch.bfloat16):\n            \n            input_term = self.vae_transform.resize_transform(pil_img2rgb(img))\n            annotations = self.fusion_render.parse_ref_annotations(gen_text)\n            annotated_img = self.fusion_render.create_comparison_image(original_img, annotations)\n            output_dict[\'tmp_img\'].append(annotated_img)\n\n            input_term = self.vae_transform.resize_transform(annotated_img)\n\n            for i in range(reflection_count):\n\n                start_time = time.time()\n                gen_context = self.update_context_image(input_term, gen_context, vae=True)\n\n                pattern = r\'<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),(.)<\\|box_end\\|>\'\n                matches = re.findall(pattern, gen_text)\n\n                reflec_text = \'\'\n                for match in matches:\n                    x1, y1, x2, y2, char = match\n                    reflec_text += f\'<|box_start|>{x1},{y1},{x2},{y2},{char}<|box_end|>\'\n\n                prompt = """请对当前页的破损检测结果进行精修。\n已给定信息：\n1. 破损检测图：在原图上叠加的当前模型预测破损蓝框。\n2. 破损定位坐标和预测的字符：每个候选框的 <|box_start|>x1,y1,x2,y2,预测的破损字符<|box_end|>。\n\n你的任务：\n- 再次仔细"看图"，结合当前破损检测图、破损定位坐标和预测的字符，对检测结果进行全面精修。请重点完成以下工作：\n1）观察破损检测图以及破损定位坐标，分析是否真正破损，若非破损字符，则需要把检测错误的框删除。\n2）观察破损检测图以及破损定位坐标，在图像中存在明显破损但当前并没有其破损定位坐标，新增合适大小的框，并补充预测该处的破损字符。\n3）观察破损检测图和利用OCR能力得到上下文内容，然后可以先预测这个片段出自于哪里，然后修正当前预测错误的破损字符。\n4）观察破损检测图以及破损定位坐标，检查框的位置是否准确，如果框的位置有偏移，需要矫正框的位置。\n\n\n破损定位坐标和预测的字符："""\n\n                prompt += reflec_text\n                cfg_text_context = deepcopy(gen_context)\n                gen_context = self.update_context_text(prompt, gen_context)\n                cfg_img_context = self.update_context_text(prompt, cfg_img_context)\n                gen_text = self.gen_text(gen_context, do_sample=do_sample, temperature=text_temperature, max_length=max_think_token_n)\n                \n                if \'经过检查，我没有发现需要修改的地方。\' in gen_text:\n                    output_dict[\'text_merge\'] = reflec_text\n                    break\n\n                print(f\'第{i+1}次修正\\n\')\n                print(f\'{gen_text}\\n\')\n                output_dict[\'text_step2\'] += f\'第{i+1}次修正\\n\'\n                output_dict[\'text_step2\'] += gen_text\n                output_dict[\'text_step2\'] += \'\\n\\n\'\n\n                result = self.merge_ocr_texts(reflec_text, gen_text)\n                result = result.replace(\' 》\', \'》\')\n                output_dict[\'text_merge\'] = result\n                output_list.append(result)\n\n\n                gen_context = self.update_context_text(gen_text, gen_context)\n                img_tmp_gen = self.gen_image(\n                    image_shapes, \n                    gen_context, \n                    cfg_text_precontext=cfg_text_context, \n                    cfg_img_precontext=cfg_img_context,\n                    cfg_text_scale=cfg_text_scale, \n                    cfg_img_scale=cfg_img_scale, \n                    cfg_interval=cfg_interval, \n                    timestep_shift=timestep_shift, \n                    num_timesteps=num_timesteps,\n                    cfg_renorm_min=cfg_renorm_min,\n                    cfg_renorm_type=cfg_renorm_type,\n                )\n                output_dict[\'tmp_img_gen\'].append(img_tmp_gen)\n\n                print(f\'第{i+1}次修正用时：{time.time()-start_time}\\n\')\n\n                gen_text = result\n                annotations = self.fusion_render.parse_ref_annotations(result)\n                annotated_img = self.fusion_render.create_comparison_image(original_img, annotations)\n\n                output_dict[\'tmp_img\'].append(annotated_img)\n\n                input_term = self.vae_transform.resize_transform(annotated_img)\n\n                gen_context = self.init_gen_context()\n                cfg_text_context = deepcopy(gen_context)\n                cfg_img_context = deepcopy(gen_context)\n\n            \n            gen_prompt = \'请参考图2中的蓝色mask及其中对应的文字内容，对图1中破损区域的字体进行修复。修复时需确保字体风格与图1中其他完好字体保持一致，同时不得影响其他完好字体的内容和样式。\'\n            annotations = self.anno_render.parse_stepref_annotations(output_dict[\'text_merge\'], original_img.size)\n            character_img = self.anno_render.create_comparison_image(original_img, annotations)\n            gen_context = self.init_gen_context()\n            cfg_text_context = deepcopy(gen_context)\n            cfg_img_context = deepcopy(gen_context)\n\n            input_term = self.vae_transform_gen.resize_transform(character_img)\n            gen_context = self.update_context_image_gen(input_term, gen_context, vae=True, vit=True)\n            image_shapes = input_term.size[::-1]\n            cfg_text_context = deepcopy(gen_context)\n            gen_context = self.update_context_text(gen_prompt, gen_context)\n            cfg_img_context = self.update_context_text(gen_prompt, cfg_img_context)\n\n            gen_text = \'我将根据图中破损图片以及破损定位图，对破损图片的破损区域进行字体风格一致的修复，最终输出风格一致的完好图片\'\n            gen_context = self.update_context_text(gen_text, gen_context)\n\n            img = self.gen_image(\n                    image_shapes, \n                    gen_context, \n                    cfg_text_precontext=cfg_text_context, \n                    cfg_img_precontext=cfg_img_context,\n                    cfg_text_scale=cfg_text_scale, \n                    cfg_img_scale=cfg_img_scale, \n                    cfg_interval=cfg_interval, \n                    timestep_shift=timestep_shift, \n                    num_timesteps=num_timesteps,\n                    cfg_renorm_min=cfg_renorm_min,\n                    cfg_renorm_type=cfg_renorm_type,\n                )\n\n        output_dict[\'final_image\'] = img\n\n        return output_dict\n    \n    def __call__(\n        self, \n        image: Optional[Image.Image] = None, \n        text: Optional[str] = None, \n        **kargs\n    ) -> Dict[str, Any]:\n        output_dict = {\'image\': None, \'text\': None}\n\n        if image is None and text is None:\n            print(\'Please provide at least one input: either an image or text.\')\n            return output_dict\n\n        input_list = []\n        if image is not None:\n            input_list.append(image)\n        if text is not None:\n            input_list.append(text)\n        \n        output_dict = self.interleave_inference(input_list, **kargs)\n\n        return output_dict\n\ndef load_model(model_path, max_mem_per_gpu="80GiB"):\n\n    print(f"Train path: {model_path}")\n\n    # LLM config preparing\n    llm_config = Qwen2Config.from_json_file(os.path.join(model_path, "llm_config.json"))\n    llm_config.qk_norm = True\n    llm_config.tie_word_embeddings = False\n    llm_config.layer_module = "Qwen2MoTDecoderLayer"\n\n    # ViT config preparing\n    vit_config = SiglipVisionConfig.from_json_file(os.path.join(model_path, "vit_config.json"))\n    vit_config.rope = False\n    vit_config.num_hidden_layers = vit_config.num_hidden_layers - 1\n\n    # VAE loading\n    vae_model, vae_config = load_ae(local_path=os.path.join(model_path, "ae.safetensors"))\n    print(\'VAE model loaded\')\n\n    # Bagel config preparing\n    config = BagelConfig(\n        visual_gen=True,\n        visual_und=True,\n        llm_config=llm_config, \n        vit_config=vit_config,\n        vae_config=vae_config,\n        vit_max_num_patch_per_side=70,\n        connector_act=\'gelu_pytorch_tanh\',\n        latent_patch_size=2,\n        max_latent_size=128,\n    )\n    print(\'BagelConfig initialized\')\n\n    with init_empty_weights():\n        language_model = Qwen2ForCausalLM(llm_config)\n        print(\'Language model initialized\')\n        vit_model = SiglipVisionModel(vit_config)\n        print(\'ViT model initialized\')\n        model = Bagel(language_model, vit_model, config)\n        print(\'Bagel model initialized\')\n        model.vit_model.vision_model.embeddings.convert_conv2d_to_linear(vit_config, meta=True)\n    print(\'Model structure prepared\')\n\n    # Tokenizer Preparing\n    tokenizer = Qwen2Tokenizer.from_pretrained(model_path)\n    tokenizer, new_token_ids, _ = add_special_tokens(tokenizer)\n\n    # Image Transform Preparing\n    vae_transform = ImageTransform(1024, 512, 16, True)\n    vae_transform_gen = ImageTransform(2048, 512, 16, True)\n    vit_transform = ImageTransform(980, 224, 14)\n\n    print(f\'Max memory per GPU: {max_mem_per_gpu}\')\n\n    device_map = infer_auto_device_map(\n        model,\n        max_memory={i: max_mem_per_gpu for i in range(torch.cuda.device_count())},\n        no_split_module_classes=["Bagel", "Qwen2MoTDecoderLayer"],\n    )\n    print(f\'Device map: {device_map}\')\n\n    same_device_modules = [\n        \'language_model.model.embed_tokens\',\n        \'time_embedder\',\n        \'latent_pos_embed\',\n        \'vae2llm\',\n        \'llm2vae\',\n        \'connector\',\n        \'vit_pos_embed\'\n    ]\n\n    if torch.cuda.device_count() == 1:\n        first_device = device_map.get(same_device_modules[0], "cuda:0")\n        for k in same_device_modules:\n            if k in device_map:\n                device_map[k] = first_device\n            else:\n                device_map[k] = "cuda:0"\n    else:\n        first_device = device_map.get(same_device_modules[0])\n        for k in same_device_modules:\n            if k in device_map:\n                device_map[k] = first_device\n\n    model = load_checkpoint_and_dispatch(\n        model,\n        checkpoint=os.path.join(model_path, "model.safetensors"),\n        device_map=device_map,\n        dtype=torch.bfloat16,\n        force_hooks=True,\n    )\n\n    model_device = next(model.parameters()).device\n    vae_model = vae_model.to(model_device).to(torch.bfloat16) \n    model = model.eval()\n    print(\'Model loaded and set to evaluation mode\')\n    \n    return model, tokenizer, vae_model, vae_transform, vae_transform_gen, vit_transform, new_token_ids\n\ndef invert_image(image):\n    is_pil = isinstance(image, Image.Image)\n    if is_pil:\n        image_array = np.array(image)\n        inverted = 255 - image_array\n        return Image.fromarray(inverted)\n    else:\n        return 255 - image\n\ndef save_results(output_dict, save_path, original_size, img_path):\n    output_image = output_dict[\'final_image\'].resize(original_size, Image.LANCZOS)\n    text_all = \'\\n\\n\\n\'.join([\n        output_dict[\'text_step1\'],\n        output_dict[\'text_step2\'],\n        output_dict[\'text_merge\']\n    ])\n    output_image = invert_image(output_image)\n    if not os.path.exists(save_path):\n        os.mkdir(save_path)\n    output_image.save(f\'{save_path}/{img_path.split("/")[-1]}\')\n\n    with open(f\'{save_path}/{img_path.split("/")[-1].replace(".jpg", ".txt")}\', \'w\') as f:\n        f.write(text_all)'}, 'BasicSR': {'LICENSE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2018-2022 BasicSR Authors\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'rrdbnet_arch.py': 'import torch\nfrom torch import nn as nn\nfrom torch.nn import functional as F\n\nfrom basicsr.utils.registry import ARCH_REGISTRY\nfrom .arch_util import default_init_weights, make_layer, pixel_unshuffle\n\n\nclass ResidualDenseBlock(nn.Module):\n    """Residual Dense Block.\n\n    Used in RRDB block in ESRGAN.\n\n    Args:\n        num_feat (int): Channel number of intermediate features.\n        num_grow_ch (int): Channels for each growth.\n    """\n\n    def __init__(self, num_feat=64, num_grow_ch=32):\n        super(ResidualDenseBlock, self).__init__()\n        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)\n        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)\n        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)\n        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)\n        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)\n\n        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)\n\n        # initialization\n        default_init_weights([self.conv1, self.conv2, self.conv3, self.conv4, self.conv5], 0.1)\n\n    def forward(self, x):\n        x1 = self.lrelu(self.conv1(x))\n        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))\n        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))\n        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))\n        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))\n        # Empirically, we use 0.2 to scale the residual for better performance\n        return x5 * 0.2 + x\n\n\nclass RRDB(nn.Module):\n    """Residual in Residual Dense Block.\n\n    Used in RRDB-Net in ESRGAN.\n\n    Args:\n        num_feat (int): Channel number of intermediate features.\n        num_grow_ch (int): Channels for each growth.\n    """\n\n    def __init__(self, num_feat, num_grow_ch=32):\n        super(RRDB, self).__init__()\n        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)\n        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)\n        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)\n\n    def forward(self, x):\n        out = self.rdb1(x)\n        out = self.rdb2(out)\n        out = self.rdb3(out)\n        # Empirically, we use 0.2 to scale the residual for better performance\n        return out * 0.2 + x\n\n\n@ARCH_REGISTRY.register()\nclass RRDBNet(nn.Module):\n    """Networks consisting of Residual in Residual Dense Block, which is used\n    in ESRGAN.\n\n    ESRGAN: Enhanced Super-Resolution Generative Adversarial Networks.\n\n    We extend ESRGAN for scale x2 and scale x1.\n    Note: This is one option for scale 1, scale 2 in RRDBNet.\n    We first employ the pixel-unshuffle (an inverse operation of pixelshuffle to reduce the spatial size\n    and enlarge the channel size before feeding inputs into the main ESRGAN architecture.\n\n    Args:\n        num_in_ch (int): Channel number of inputs.\n        num_out_ch (int): Channel number of outputs.\n        num_feat (int): Channel number of intermediate features.\n            Default: 64\n        num_block (int): Block number in the trunk network. Defaults: 23\n        num_grow_ch (int): Channels for each growth. Default: 32.\n    """\n\n    def __init__(self, num_in_ch, num_out_ch, scale=4, num_feat=64, num_block=23, num_grow_ch=32):\n        super(RRDBNet, self).__init__()\n        self.scale = scale\n        if scale == 2:\n            num_in_ch = num_in_ch * 4\n        elif scale == 1:\n            num_in_ch = num_in_ch * 16\n        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)\n        self.body = make_layer(RRDB, num_block, num_feat=num_feat, num_grow_ch=num_grow_ch)\n        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        # upsample\n        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)\n\n        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)\n\n    def forward(self, x):\n        if self.scale == 2:\n            feat = pixel_unshuffle(x, scale=2)\n        elif self.scale == 1:\n            feat = pixel_unshuffle(x, scale=4)\n        else:\n            feat = x\n        feat = self.conv_first(feat)\n        body_feat = self.conv_body(self.body(feat))\n        feat = feat + body_feat\n        # upsample\n        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode=\'nearest\')))\n        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode=\'nearest\')))\n        out = self.conv_last(self.lrelu(self.conv_hr(feat)))\n        return out\n'}, 'Real-ESRGAN': {'LICENSE': 'BSD 3-Clause License\n\nCopyright (c) 2021, Xintao Wang\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\n3. Neither the name of the copyright holder nor the names of its\n   contributors may be used to endorse or promote products derived from\n   this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n'}}
WORKER_SOURCE = '"""Generated verbatim by the notebook. No project imports and no network."""\nfrom __future__ import annotations\nimport os, sys, json, time, traceback, pathlib, socket, hashlib\n\nfor _key in (\'HF_HUB_OFFLINE\',\'TRANSFORMERS_OFFLINE\',\'HF_DATASETS_OFFLINE\',\'DIFFUSERS_OFFLINE\'):\n    os.environ[_key] = \'1\'\nos.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n\ndef deny_network(event, args):\n    if event==\'socket.connect\' and isinstance(args[1],str):\n        return # Local Unix-domain IPC used by compiler workers; no IP network.\n    if event in (\'socket.connect\', \'socket.getaddrinfo\'):\n        raise RuntimeError(\'OFFLINE: network access prohibited in image worker\')\nsys.addaudithook(deny_network)\n\ndef atomic_json(path, obj):\n    path = pathlib.Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.tmp\'); tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding=\'utf8\'); tmp.replace(path)\n\ndef rrdb_model():\n    # Inference architecture from XPixelGroup/BasicSR rrdbnet_arch.py (Apache-2.0).\n    # Identical released x4plus layer names/shapes/forward; training init omitted.\n    import torch\n    from torch import nn\n    from torch.nn import functional as F\n    class ResidualDenseBlock(nn.Module):\n        def __init__(self):\n            super().__init__()\n            for i in range(1,6):\n                setattr(self, \'conv\'+str(i), nn.Conv2d(64+(i-1)*32, 32 if i<5 else 64, 3,1,1))\n            self.lrelu = nn.LeakyReLU(.2, inplace=True)\n        def forward(self,x):\n            values=[x]\n            for i in range(1,5): values.append(self.lrelu(getattr(self,\'conv\'+str(i))(torch.cat(values,1))))\n            return self.conv5(torch.cat(values,1))*.2+x\n    class RRDB(nn.Module):\n        def __init__(self):\n            super().__init__(); self.rdb1=ResidualDenseBlock(); self.rdb2=ResidualDenseBlock(); self.rdb3=ResidualDenseBlock()\n        def forward(self,x): return self.rdb3(self.rdb2(self.rdb1(x)))*.2+x\n    class RRDBNet(nn.Module):\n        def __init__(self):\n            super().__init__(); self.conv_first=nn.Conv2d(3,64,3,1,1)\n            self.body=nn.Sequential(*[RRDB() for _ in range(23)])\n            self.conv_body=nn.Conv2d(64,64,3,1,1); self.conv_up1=nn.Conv2d(64,64,3,1,1)\n            self.conv_up2=nn.Conv2d(64,64,3,1,1); self.conv_hr=nn.Conv2d(64,64,3,1,1); self.conv_last=nn.Conv2d(64,3,3,1,1)\n            self.lrelu=nn.LeakyReLU(.2,inplace=True)\n        def forward(self,x):\n            x=self.conv_first(x); x=x+self.conv_body(self.body(x))\n            x=self.lrelu(self.conv_up1(F.interpolate(x,scale_factor=2,mode=\'nearest\')))\n            x=self.lrelu(self.conv_up2(F.interpolate(x,scale_factor=2,mode=\'nearest\')))\n            return self.conv_last(self.lrelu(self.conv_hr(x)))\n    return RRDBNet()\n\ndef starts(length,tile,overlap):\n    if not 0 <= overlap < tile: raise ValueError(\'Require 0 <= overlap < tile\')\n    if length <= tile: return [0]\n    values=list(range(0,length-tile+1,tile-overlap))\n    if values[-1]!=length-tile: values.append(length-tile)\n    return values\n\ndef tiled_tadisr(image, predict, tile, overlap):\n    """CPU accumulation, independent axes; no uncovered pixels on thin crops."""\n    import torch\n    from torch.nn import functional as F\n    if tile % 32 or overlap % 32: raise ValueError(\'TADiSR tile/overlap must be multiples of 32\')\n    h,w=image.shape[-2:]\n    image=F.pad(image,(0,max(0,tile-w),0,max(0,tile-h)),mode=\'replicate\')\n    ph,pw=image.shape[-2:]\n    sums=torch.zeros(1,3,ph,pw); masks=torch.zeros(1,1,ph,pw); weights=torch.zeros(1,1,ph,pw)\n    blend=torch.hann_window(tile,periodic=False).clamp_min(1e-3)\n    blend=(blend[:,None]*blend[None,:])[None,None]\n    ys,xs=starts(ph,tile,overlap),starts(pw,tile,overlap)\n    seam_diffs=[]\n    # Compare independently predicted patches in their actual overlap before blending.\n    for top in ys:\n        for left in xs:\n            sr,mask=predict(image[:,:,top:top+tile,left:left+tile])\n            sr,mask=sr.detach().float().cpu(),mask.detach().float().cpu()\n            if tuple(sr.shape)!=(1,3,tile,tile) or tuple(mask.shape)!=(1,1,tile,tile):\n                raise ValueError(\'TADiSR output violates tile contract\')\n            old_weights=weights[:,:,top:top+tile,left:left+tile]\n            old=sums[:,:,top:top+tile,left:left+tile]\n            occupied=(old_weights>0).expand_as(old)\n            if occupied.any():\n                previous=old/old_weights.clamp_min(1e-12)\n                seam_diffs.append(float((previous-sr).abs()[occupied].mean()))\n            old += sr*blend\n            masks[:,:,top:top+tile,left:left+tile] += mask*blend\n            old_weights += blend\n    if not (weights>0).all(): raise ValueError(\'Tiled output has uncovered pixels\')\n    return (sums/weights)[:,:,:h,:w],(masks/weights)[:,:,:h,:w],{\n        \'tiles\':len(ys)*len(xs),\'overlap_patch_mae\':max(seam_diffs,default=0.),\n        \'covered_all_pixels\':True,\'visual_seam_review\':\'required\'}\n\ndef load_backend(model, assets, options):\n    import torch\n    device=options.get(\'device\',\'cuda\')\n    if model==\'realesrgan_x4\':\n        net=rrdb_model()\n        sd=torch.load(assets[\'checkpoint\'],map_location=\'cpu\',weights_only=True)\n        net.load_state_dict(sd.get(\'params_ema\',sd.get(\'params\',sd)),strict=True)\n        net=net.eval().to(device)\n        if options.get(\'half\',False): net.half()\n        return net\n    if model==\'tadisr_x4\':\n        from tadisr.inference import CogView4Config,load_cogview4\n        # Upstream inspect_checkpoint runs before GPU construction and validates decoder.\n        return load_cogview4(CogView4Config(pathlib.Path(assets[\'base_model\']),\n            pathlib.Path(assets[\'checkpoint\']),pathlib.Path(assets[\'prompt_embeddings\']),device))\n    if model==\'difftsr\':\n        from omegaconf import OmegaConf\n        from model.IDM.utils.util import instantiate_from_config\n        cfg=OmegaConf.load(assets[\'config\'])\n        cfg.model.params.Text_Prediction_config.params={\'ckpt_path\':assets[\'transocr\'],\n            \'alphabet_path\':str(pathlib.Path(assets[\'vendor\'])/\'model/TDM/utils/benchmark.txt\')}\n        net=instantiate_from_config(cfg.model); net.load_model(assets[\'checkpoint\'])\n        net.IDM_ddim_schedule_init(ddim_num_steps=options[\'steps\'],ddim_eta=.2,verbose=False)\n        return net\n    if model==\'unihir_restoration\':\n        from utils import load_model,UniHIRInferencer\n        m,t,v,vt,vg,it,nt=load_model(assets[\'base_model\'],max_mem_per_gpu=options.get(\'max_memory\',\'80GiB\'))\n        sizes={}\n        def capture(name,transform):\n            def observed(image,*a,**kw):\n                result=transform(image,*a,**kw)\n                sizes[name]={\'input_size\':list(image.size),\'output_shape\':list(result.shape) if hasattr(result,\'shape\') else None}\n                return result\n            return observed\n        net=UniHIRInferencer(model=m,tokenizer=t,vae_model=v,vae_transform=capture(\'vae\',vt),\n            vae_transform_gen=capture(\'vae_gen\',vg),vit_transform=capture(\'vit\',it),new_token_ids=nt)\n        net._observed_processor_sizes=sizes\n        return net\n    raise ValueError(model)\n\ndef run_backend(model,net,image,options):\n    import torch, numpy as np\n    from PIL import Image\n    import torch.nn.functional as F\n    device=options.get(\'device\',\'cuda\');diagnostics={};extras={}\n    if model==\'realesrgan_x4\':\n        # Same local tile padding/cropping as RealESRGANer; RGB tensors throughout.\n        tile=options.get(\'tile\',256);pad=options.get(\'overlap\',24)\n        h,w=image.height,image.width;output=np.empty((h*4,w*4,3),dtype=np.uint8);processor_sizes=set()\n        dtype=next(net.parameters()).dtype\n        for y in range(0,h,tile):\n            for x in range(0,w,tile):\n                x0,y0=max(0,x-pad),max(0,y-pad);x1,y1=min(w,x+tile+pad),min(h,y+tile+pad)\n                a=np.asarray(image.crop((x0,y0,x1,y1)),dtype=np.float32)/255\n                tensor=torch.from_numpy(a.copy()).permute(2,0,1)[None].to(device=device,dtype=dtype)\n                processor_sizes.add((x1-x0,y1-y0))\n                result=net(tensor).clamp(0,1)[0].float().cpu().permute(1,2,0).numpy()\n                actual_w,actual_h=min(tile,w-x),min(tile,h-y)\n                output[y*4:(y+actual_h)*4,x*4:(x+actual_w)*4]=(result[\n                    (y-y0)*4:(y-y0+actual_h)*4,(x-x0)*4:(x-x0+actual_w)*4]*255).round().astype(np.uint8)\n        return Image.fromarray(output),extras,{\'visual_seam_review\':\'required\',\'tile_padding\':pad,\n            \'processor_input_sizes\':sorted(processor_sizes),\'processor_output_scale\':4}\n    if model==\'tadisr_x4\':\n        a=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()/255).permute(2,0,1)[None]\n        a=F.interpolate(a,size=(image.height*4,image.width*4),mode=\'bicubic\',align_corners=False)\n        def predict(patch): return net(patch.to(device))\n        sr,mask,diagnostics=tiled_tadisr(a,predict,options[\'tile\'],options[\'overlap\'])\n        diagnostics.update(processor_tile_size=[options[\'tile\'],options[\'tile\']],pre_upsample_size=[image.width*4,image.height*4])\n        out=(sr[0].clamp(0,1).permute(1,2,0).numpy()*255).round().astype(np.uint8)\n        extras[\'tadisr_text_mask\']=Image.fromarray((mask.sigmoid()[0,0].numpy()*255).round().astype(np.uint8))\n        return Image.fromarray(out),extras,diagnostics\n    if model==\'difftsr\':\n        # Only reviewed, near-4:1 horizontal lines; isotropic letterbox, never rotate.\n        canvas=Image.new(\'RGB\',(512,128),\'white\');copy=image.copy();copy.thumbnail((512,128),Image.Resampling.BICUBIC)\n        offset=((512-copy.width)//2,(128-copy.height)//2);canvas.paste(copy,offset)\n        result=Image.fromarray(net.DiffTSR_sample(canvas),\'RGB\')\n        # Released output is 512x128, not an arbitrary x4 contract.\n        result=result.crop((offset[0],offset[1],offset[0]+copy.width,offset[1]+copy.height))\n        return result,extras,{\'processor_size\':[512,128],\'content_size\':list(copy.size),\n            \'processor_offset\':list(offset),\'steps\':options[\'steps\'],\n            \'warning\':\'upstream_text_prior_uses_zhconv_zh_cn; inspect_traditional_forms\'}\n    if model==\'unihir_restoration\':\n        from utils import invert_image\n        result=net(image=invert_image(image),text=\'请对图中破损位置进行定位，并输出破损定位图。\',\n            und_only=False,gen_only=False,reflection_count=options[\'ref_count\'],think=False,\n            max_think_token_n=10000,do_sample=False,cfg_text_scale=1.,cfg_img_scale=1.,\n            cfg_interval=[0.,1.],timestep_shift=3.,num_timesteps=options[\'steps\'],\n            cfg_renorm_min=0.,cfg_renorm_type=\'global\')\n        final=invert_image(result[\'final_image\'].resize(image.size,Image.Resampling.LANCZOS))\n        extras[\'predictions\']={k:result.get(k) for k in (\'text_step1\',\'text_step2\',\'text_merge\')}\n        extras[\'predictions\'].update(refinement_count=options[\'ref_count\'],confidence=None,\n            confidence_status=\'upstream_does_not_supply_calibrated_confidence\')\n        import re\n        extras[\'predictions\'][\'damage_predictions\']=[\n            {\'box_normalized_1000\':[int(x) for x in match[:4]],\'predicted_character\':match[4],\n             \'evidence\':\'model_generated; not_source_observation\'}\n            for match in re.findall(r\'<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),(.?)<\\|box_end\\|>\',result.get(\'text_merge\',\'\'))]\n        # Keep damage/draft and all available intermediate diagnostic images separately.\n        for key in (\'draft\',):\n            if isinstance(result.get(key),Image.Image): extras[key]=result[key]\n        for key in (\'tmp_img\',\'tmp_img_gen\'):\n            for i,v in enumerate(result.get(key,[])):\n                if isinstance(v,Image.Image):extras[f\'{key}_{i:02d}\']=v\n        return final,extras,{\'generative_reconstruction\':True,\'alignment\':\'nominal_resize_only; generated_glyphs_unverified\',\n            \'observed_processor_sizes\':net._observed_processor_sizes}\n    raise ValueError(model)\n\ndef probe(model, options):\n    import importlib,importlib.metadata as md,platform\n    imports={\'realesrgan_x4\':[\'torch\',\'numpy\',\'PIL\'],\n        \'tadisr_x4\':[\'torch\',\'numpy\',\'diffusers\',\'peft\',\'transformers\',\'sentencepiece\',\'tadisr.inference\',\'tadisr.pipelines\'],\n        \'difftsr\':[\'torch\',\'torchvision\',\'omegaconf\',\'pytorch_lightning\',\'zhconv\',\'einops\',\n            \'model.DiffTSR\',\'model.IDM.models.autoencoder\',\'model.TDM.models.text_prediction\',\'model.TDM.models.modules\'],\n        \'unihir_restoration\':[\'torch\',\'flash_attn\',\'transformers\',\'accelerate\',\'fontTools\',\'utils\']}[model]\n    for name in imports: importlib.import_module(name)\n    import torch,numpy as np\n    if torch.from_numpy(np.ones((2,2),dtype=np.float32)).numpy().shape!=(2,2):raise RuntimeError(\'NumPy bridge failed\')\n    from packaging.version import Version\n    from packaging.tags import sys_tags,parse_tag\n    import base64,csv,io\n    records_verified=[]\n    for module_name,distribution_name in [(\'torch\',\'torch\'),(\'numpy\',\'numpy\'),(\'PIL\',\'pillow\'),\n            *([(n,n) for n in (\'diffusers\',\'peft\',\'transformers\')] if model==\'tadisr_x4\' else [])]:\n        module=importlib.import_module(module_name);dist=md.distribution(distribution_name)\n        actual=getattr(module,\'__version__\',None)\n        if actual and Version(actual).public!=Version(dist.version).public:\n            raise RuntimeError(f\'Active module/metadata mismatch: {module_name} {actual} vs {dist.version}\')\n        wheel=dist.read_text(\'WHEEL\') or \'\'\n        wheel_tags=set()\n        for line in wheel.splitlines():\n            if line.startswith(\'Tag: \'):wheel_tags.update(parse_tag(line[5:]))\n        if wheel_tags and not wheel_tags.intersection(set(sys_tags())):\n            raise RuntimeError(f\'Active distribution has incompatible wheel ABI: {distribution_name}\')\n        record=dist.read_text(\'RECORD\')\n        if not record:raise RuntimeError(f\'Cannot verify active distribution RECORD: {distribution_name}\')\n        rows=list(csv.reader(io.StringIO(record)))\n        verify_paths=[pathlib.Path(module.__file__).resolve()]\n        if module_name==\'torch\':verify_paths.append(pathlib.Path(torch._C.__file__).resolve())\n        if module_name==\'numpy\':\n            try:native=importlib.import_module(\'numpy._core._multiarray_umath\')\n            except ImportError:native=importlib.import_module(\'numpy.core._multiarray_umath\')\n            verify_paths.append(pathlib.Path(native.__file__).resolve())\n        for path in verify_paths:\n            matching=[r for r in rows if pathlib.Path(dist.locate_file(r[0])).resolve()==path]\n            if len(matching)!=1 or not matching[0][1]:raise RuntimeError(f\'Active file not owned by metadata RECORD: {path}\')\n            algorithm,expected=matching[0][1].split(\'=\',1)\n            hasher=hashlib.new(algorithm)\n            with path.open(\'rb\') as stream:\n                for block in iter(lambda:stream.read(1024*1024),b\'\'):hasher.update(block)\n            digest=hasher.digest()\n            if base64.urlsafe_b64encode(digest).decode().rstrip(\'=\')!=expected:\n                raise RuntimeError(f\'Active package RECORD checksum differs: {path}\')\n            records_verified.append(str(path))\n    report={\'python\':sys.version,\'abi\':sys.implementation.cache_tag,\'platform\':platform.platform(),\n        \'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'numpy\':np.__version__,\n        \'cuda_available\':torch.cuda.is_available(),\'imports\':imports,\'pid\':os.getpid(),\n        \'active_import_records_verified\':records_verified}\n    if options.get(\'device\',\'cuda\')==\'cuda\':\n        if not torch.cuda.is_available():raise RuntimeError(\'CUDA required but unavailable\')\n        capability=torch.cuda.get_device_capability();arch=torch.cuda.get_arch_list()\n        # A real operation is essential: version strings alone do not validate GPU support.\n        (torch.ones(8,device=\'cuda\')*2).sum().item();torch.cuda.synchronize()\n        report.update(gpu=torch.cuda.get_device_name(),capability=list(capability),compiled_arches=arch,\n            total_vram=torch.cuda.get_device_properties(0).total_memory)\n        if model==\'tadisr_x4\' and report[\'total_vram\'] < 24*1024**3:\n            raise RuntimeError(\'TADiSR upstream float32 stack needs >=24 GiB practical minimum; local GPU insufficient\')\n        if model==\'unihir_restoration\' and report[\'total_vram\'] < 75*1024**3:\n            raise RuntimeError(\'UniHIR upstream unquantized inference requires approximately 80 GB class GPU\')\n    if model==\'tadisr_x4\':\n        from diffusers import CogView4Transformer2DModel\n        from transformers import GlmModel\n    if model==\'unihir_restoration\':\n        from flash_attn import flash_attn_func\n        sample=torch.randn(1,16,2,64,device=\'cuda\',dtype=torch.bfloat16)\n        flash_attn_func(sample,sample,sample);torch.cuda.synchronize()\n    # Check only the dependency closure actually imported, not unrelated host notebook packages.\n    from packaging.requirements import Requirement\n    pending=[{\'PIL\':\'pillow\',\'fontTools\':\'fonttools\'}.get(n,n) for n in imports if \'.\' not in n]\n    checked=set()\n    while pending:\n        name=pending.pop()\n        if name in checked:continue\n        checked.add(name)\n        try:dist=md.distribution(name)\n        except md.PackageNotFoundError:continue\n        for spec in dist.requires or []:\n            req=Requirement(spec)\n            if req.marker and not req.marker.evaluate({\'extra\':\'\'}):continue\n            version=md.version(req.name)\n            if req.specifier and not req.specifier.contains(version,prereleases=True):\n                raise RuntimeError(f\'Dependency conflict: {dist.metadata["Name"]} requires {req}, got {version}\')\n            pending.append(req.name)\n    report[\'dependency_closure_checked\']=sorted(checked)\n    return report\n\ndef main():\n    cfg=json.loads(pathlib.Path(sys.argv[1]).read_text(encoding=\'utf8\'))\n    if cfg.get(\'vendor\'):sys.path.insert(0,cfg[\'vendor\'])\n    result_path=cfg[\'ready\']\n    try:\n        if cfg.get(\'action\')==\'isolation_probe\':\n            try:socket.create_connection((\'127.0.0.1\',9),timeout=.1)\n            except RuntimeError as error:\n                blocked=\'OFFLINE\' in str(error)\n            else:blocked=False\n            if not blocked:raise RuntimeError(\'Worker offline guard did not reject a connection\')\n            atomic_json(result_path,{\'state\':\'ready\',\'pid\':os.getpid(),\'offline\':True,\'network_block_verified\':blocked});return\n        report=probe(cfg[\'model\'],cfg[\'options\'])\n        if cfg.get(\'action\')==\'probe\':atomic_json(result_path,{\'state\':\'ready\',\'report\':report});return\n        if cfg.get(\'action\')==\'prepare_prompt\':\n            import runpy,torch\n            target=pathlib.Path(cfg[\'assets\'][\'prompt_output\']);target.parent.mkdir(parents=True,exist_ok=True)\n            temporary=target.with_suffix(\'.part\')\n            sys.argv=[\'prepare_prompt_embeddings.py\',\'--base-model\',cfg[\'assets\'][\'base_model\'],\n                \'--output\',str(temporary),\'--device\',cfg[\'options\'].get(\'device\',\'cuda\')]\n            torch.manual_seed(cfg[\'options\'].get(\'seed\',42));torch.cuda.reset_peak_memory_stats()\n            started=time.monotonic()\n            runpy.run_path(str(pathlib.Path(cfg[\'vendor\'])/\'scripts/prepare_prompt_embeddings.py\'),run_name=\'__main__\')\n            temporary.replace(target);torch.cuda.synchronize()\n            atomic_json(result_path,{\'state\':\'ready\',\'report\':report,\'prompt_seconds\':time.monotonic()-started,\n                \'peak_vram_bytes\':torch.cuda.max_memory_allocated(),\'derived_offline\':True});return\n        import torch, numpy as np\n        torch.manual_seed(cfg[\'options\'].get(\'seed\',42));np.random.seed(cfg[\'options\'].get(\'seed\',42))\n        started=time.monotonic();net=load_backend(cfg[\'model\'],cfg[\'assets\'],cfg[\'options\'])\n        if torch.cuda.is_available():torch.cuda.synchronize()\n        atomic_json(result_path,{\'state\':\'ready\',\'report\':report,\'load_seconds\':time.monotonic()-started})\n    except Exception:\n        atomic_json(result_path,{\'state\':\'failed\',\'error\':traceback.format_exc()});return\n    for line in sys.stdin:\n        job=json.loads(line);output=pathlib.Path(job[\'output\']);output.mkdir(parents=True,exist_ok=True)\n        started=time.monotonic()\n        try:\n            from PIL import Image\n            if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()\n            with Image.open(job[\'input\']) as src:image=src.convert(\'RGB\')\n            # Per-region seed makes interrupted/resumed diffusion independent of job ordering.\n            torch.manual_seed(job[\'seed\']);np.random.seed(job[\'seed\']%(2**32-1))\n            with torch.inference_mode():restored,extras,diagnostics=run_backend(cfg[\'model\'],net,image,cfg[\'options\'])\n            restored.save(output/\'result.png\')\n            extra_refs={}\n            for key,value in extras.items():\n                if isinstance(value,Image.Image):value.save(output/(key+\'.png\'));extra_refs[key]=key+\'.png\'\n                else:atomic_json(output/(key+\'.json\'),value);extra_refs[key]=key+\'.json\'\n            if torch.cuda.is_available():torch.cuda.synchronize()\n            peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0\n            reserved=torch.cuda.max_memory_reserved() if torch.cuda.is_available() else 0\n            atomic_json(job[\'response\'],{\'state\':\'ok\',\'seconds\':time.monotonic()-started,\n                \'peak_vram_bytes\':peak,\'peak_reserved_vram_bytes\':reserved,\'diagnostics\':diagnostics,\'extras\':extra_refs})\n        except Exception:\n            atomic_json(job[\'response\'],{\'state\':\'failed\',\'seconds\':time.monotonic()-started,\'error\':traceback.format_exc()})\nif __name__==\'__main__\':main()\n'
RESTORATION_CODE_SIGNATURE = '8d54331564e71492e0321b3d4bf9163118752b10c6f1d2d356a5dde230fcbe78'


## 4. Bootstrap offline, preflight, cache, QC và output

Chế độ process chỉ dùng file local. Worker bật các cờ offline và audit hook chặn
kết nối mạng. PDFium thiếu được cài từ wheel tương thích vào thư mục riêng;
không sửa kernel. TADiSR có venv overlay riêng, tái dùng Torch/NumPy đã probe,
kiểm tra dependency closure thực tế. DiffTSR/UniHIR cần runtime riêng đã xác minh,
không tự cài stack cũ lên Python3.13/Blackwell. Thiếu asset báo đúng đường dẫn.

Preflight kiểm checksum toàn bộ model trước GPU. Kiểm import/NumPy bridge/CUDA và
phép tính GPU trong worker; đây chưa thay smoke inference. Không tuyên bố model
sẵn sàng chỉ vì có weights. `ALLOW_PARTIAL=False` dừng trước PDF khi thiếu model
phù hợp đầu vào; lỗi region giữ cache nhưng không thay `dataset.json` hoàn chỉnh.


In [ ]:
from __future__ import annotations
import os, sys, json, time, hashlib, io, shutil, subprocess, tempfile, uuid
from pathlib import Path
from dataclasses import asdict

CLASSICAL = ('original','clahe','bicubic_x2','bicubic_x4','denoised')
NEURAL = ('realesrgan_x4','tadisr_x4','difftsr','unihir_restoration')
SUPPORTED = (*CLASSICAL,*NEURAL,'tadisr_text_mask')
PROVENANCE = {**dict.fromkeys(CLASSICAL,'deterministic_transformation'),
    'original':'source_observation',**dict.fromkeys(NEURAL,'neural_restoration'),
    'tadisr_text_mask':'neural_restoration','unihir_restoration':'generative_reconstruction'}

_canonical_dataset_write=dataset_write
_storage_guard=None
def dataset_write(root,relative,data):
    global _storage_guard
    path=dataset_path(root,relative)
    if _storage_guard and Path(root).resolve()==_storage_guard['root']:
        growth=max(0,len(data)-(path.stat().st_size if path.exists() else 0))
        if _storage_guard['used']+growth>_storage_guard['limit']:
            raise RuntimeError('max_storage_bytes exceeded before atomic write')
        ref=_canonical_dataset_write(root,relative,data)
        _storage_guard['used']+=growth
        return ref
    return _canonical_dataset_write(root,relative,data)

def prep_normalize(config):
    cfg=json.loads(json.dumps(config));variants=cfg['variants']
    if 'unihir' in variants:variants[variants.index('unihir')]='unihir_restoration'
    if set(variants)-set(SUPPORTED) or len(set(variants))!=len(variants):raise ValueError('Unknown/duplicate variants')
    if 'original' not in variants:variants.insert(0,'original')
    if 'tadisr_x4' in variants and 'tadisr_text_mask' not in variants:variants.append('tadisr_text_mask')
    if 'tadisr_text_mask' in variants and 'tadisr_x4' not in variants:raise ValueError('Mask requires tadisr_x4')
    if not cfg['crop_types'] or set(cfg['crop_types'])-{'page','group','column'}:raise ValueError('Invalid crop_types')
    for key in ('max_source_pixels','max_output_pixels','max_storage_bytes','timeout_seconds','load_timeout_seconds'):
        if cfg[key]<=0:raise ValueError(f'{key} must be positive')
    for model in NEURAL:
        opts=cfg['model_options'][model]
        if opts.get('tile',1)<=0 or not 0<=opts.get('overlap',0)<opts.get('tile',1):raise ValueError(f'{model}: invalid tiling')
    tad=cfg['model_options']['tadisr_x4']
    if tad['tile']%32 or tad['overlap']%32:raise ValueError('TADiSR tile/overlap require multiples of 32')
    if cfg['batch_size']!=1:raise ValueError('Only batch_size=1 has a validated memory contract')
    if not 1<=cfg['model_options']['difftsr']['steps']<=200:raise ValueError('DiffTSR steps must be 1..200')
    if cfg['deskew'] or cfg['dewarp']:raise ValueError('Deskew/dewarp need a separately validated geometry profile; disabled')
    return cfg

def prep_worker_environment():
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE'):env.pop(key,None)
    env.update(PYTHONNOUSERSITE='1',PYTHONUTF8='1',HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',
        DIFFUSERS_OFFLINE='1',HF_DATASETS_OFFLINE='1',HF_HUB_DISABLE_TELEMETRY='1',
        PIP_NO_INDEX='1',PIP_DISABLE_PIP_VERSION_CHECK='1',TOKENIZERS_PARALLELISM='false')
    return env

def prep_bootstrap(packages_dir, runtime_dir):
    """Install missing PDFium into a private import directory; never into kernel."""
    if Path('/kaggle/input').exists() and Path(runtime_dir).resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('RUNTIME_DIR must be writable and outside /kaggle/input')
    from importlib.util import find_spec
    import PIL,numpy,cv2
    # Existing NumPy/OpenCV must operate together; no ABI replacement in a running kernel.
    cv2.cvtColor(numpy.ones((4,4,3),dtype=numpy.uint8),cv2.COLOR_RGB2GRAY)
    if find_spec('pypdfium2') is None:
        candidates=sorted(Path(packages_dir).rglob('pypdfium2-*.whl'))
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
        tags=set(sys_tags());candidates=[p for p in candidates if tags & parse_wheel_filename(p.name)[3]]
        if not candidates:raise FileNotFoundError(f'Missing compatible PDFium wheel below {packages_dir}; use prepare_assets locally')
        target=Path(runtime_dir)/'base_lib';target.mkdir(parents=True,exist_ok=True)
        subprocess.run([sys.executable,'-m','pip','--isolated','install','--no-index','--no-deps',
            '--target',str(target),str(candidates[-1])],check=True,env=prep_worker_environment())
        sys.path.insert(0,str(target))
    import pypdfium2
    return {'pillow':PIL.__version__,'numpy':numpy.__version__,'opencv':cv2.__version__,
        'pdfium':str(pypdfium2.PDFIUM_INFO),'python':sys.version}

def prep_vendor(runtime_dir, model):
    name={'tadisr_x4':'TADiSR','difftsr':'DiffTSR','unihir_restoration':'UniHIR'}.get(model)
    if not name:return None
    folder=Path(runtime_dir)/'vendor'/name
    for relative,source in VENDOR_SOURCES[name].items():
        p=dataset_path(folder,relative);p.parent.mkdir(parents=True,exist_ok=True)
        if not p.exists() or p.read_text(encoding='utf8')!=source:p.write_text(source,encoding='utf8')
    return str(folder.resolve())

def prep_wait(process, response, timeout, log):
    started=time.monotonic();response=Path(response)
    while not response.exists():
        if process.poll() is not None:raise RuntimeError(f'Worker exit {process.returncode}; see {log}')
        if time.monotonic()-started>timeout:
            process.kill();process.wait();raise TimeoutError(f'Worker exceeded {timeout}s; see {log}')
        time.sleep(.1)
    data=json.loads(response.read_text(encoding='utf8'))
    if data['state']=='failed':raise RuntimeError(data['error'])
    return data

class PrepWorker:
    def __init__(self, runtime_dir, python, model, assets, options, timeout, action='run'):
        self.folder=Path(runtime_dir)/'jobs'/uuid.uuid4().hex;self.folder.mkdir(parents=True)
        self.log=self.folder/'worker.log';self.stream=self.log.open('w',encoding='utf8')
        self.counter=0;vendor=prep_vendor(runtime_dir,model)
        if vendor:assets={**assets,'vendor':vendor}
        # UniHIR upstream fonts are relative to its unmodified source cwd.
        if model=='unihir_restoration' and 'font_dir' in assets:
            font=Path(vendor)/'font'
            font.mkdir(parents=True,exist_ok=True)
            for source in Path(assets['font_dir']).iterdir():
                if source.is_file():
                    target=font/source.name
                    if not target.is_file() or dataset_file_hash(target)!=dataset_file_hash(source):
                        temporary=target.with_suffix(target.suffix+'.tmp');shutil.copyfile(source,temporary);temporary.replace(target)
        self.config={'model':model,'assets':assets,'options':options,'action':action,
            'vendor':vendor,'ready':str((self.folder/'ready.json').resolve())}
        spec=self.folder/'worker_config.json';spec.write_text(json.dumps(self.config),encoding='utf8')
        worker=Path(runtime_dir)/'image_worker.py';worker.write_text(WORKER_SOURCE,encoding='utf8')
        self.process=subprocess.Popen([str(python),'-u',str(worker.resolve()),str(spec.resolve())],
            stdin=subprocess.PIPE,stdout=self.stream,stderr=subprocess.STDOUT,text=True,
            encoding='utf8',env=prep_worker_environment(),cwd=vendor or self.folder,
            creationflags=getattr(subprocess,'CREATE_NO_WINDOW',0))
        try:self.ready=prep_wait(self.process,self.config['ready'],timeout,self.log)
        except BaseException:self.close();raise
    def infer(self, input_path, seed, timeout):
        self.counter+=1;folder=self.folder/f'region_{self.counter:07d}';folder.mkdir()
        response=(folder/'response.json').resolve()
        job={'input':str(Path(input_path).resolve()),'output':str(folder.resolve()),'seed':seed,'response':str(response)}
        self.process.stdin.write(json.dumps(job)+'\n');self.process.stdin.flush()
        return folder,prep_wait(self.process,response,timeout,self.log)
    def close(self):
        if hasattr(self,'process'):
            if self.process.stdin:
                try:self.process.stdin.close()
                except OSError:pass
            try:self.process.wait(timeout=10)
            except subprocess.TimeoutExpired:self.process.kill();self.process.wait()
        self.stream.close()

def prep_overlay(packages_dir, runtime_dir, model, config):
    """A separate venv per model, local wheels only, kernel files untouched."""
    override=config['runtime_pythons'].get(model)
    if override:
        if not Path(override).is_file():raise FileNotFoundError(override)
        return str(Path(override).resolve())
    if model=='realesrgan_x4':return sys.executable
    if model in ('difftsr','unihir_restoration'):
        raise RuntimeError(f'{model}: configure runtime_pythons with an independently verified offline runtime. '
            'Legacy dependency/native-extension stack has not been validated on Python 3.13/Blackwell')
    manifest=Path(packages_dir)/'image_restoration/manifests/tadisr_runtime.json'
    if not manifest.is_file():raise FileNotFoundError(f'Missing {manifest}; run prepare_assets locally')
    plan=json.loads(manifest.read_text(encoding='utf8'))
    wheel_root=manifest.parent.parent
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    tags=set(sys_tags());wheels=[]
    for record in plan['wheels']:
        wheel=dataset_path(wheel_root,record['path'])
        if not wheel.is_file():raise FileNotFoundError(wheel)
        if dataset_file_hash(wheel)!=record['sha256']:raise ValueError(f'Wheel checksum mismatch: {wheel}')
        if not tags & parse_wheel_filename(wheel.name)[3]:raise RuntimeError(f'Incompatible wheel ABI: {wheel.name}')
        wheels.append(str(wheel.resolve()))
    key=dataset_hash({'plan':plan,'python':sys.version,'prefix':sys.prefix})
    folder=Path(runtime_dir)/'environments'/('tadisr_'+key[:16]);python=folder/('Scripts/python.exe' if os.name=='nt' else 'bin/python')
    done=folder/'install.json'
    if not done.exists():
        subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(folder)],check=True,env=prep_worker_environment())
        # Only overlay wheels change. Torch/NumPy/torchvision inherited from kernel and probed.
        subprocess.run([str(python),'-m','pip','--isolated','install','--ignore-installed','--no-index',
            '--no-deps',*wheels],check=True,env=prep_worker_environment(),stdout=subprocess.DEVNULL)
        dataset_write(folder,done.name,dataset_json_bytes({'key':key}))
    return str(python.resolve())

def prep_check_manifest(model_dir, required):
    """Verify complete file list and hashes before GPU loading (not only checkpoint)."""
    root=Path(model_dir);manifest=root/'assets.json'
    if not manifest.is_file():raise FileNotFoundError(f'Missing {manifest}; prepare/inventory assets first')
    data=json.loads(manifest.read_text(encoding='utf8'));indexed={e['path']:e for e in data['files']}
    missing=[]
    for relative in required:
        p=dataset_path(root,relative)
        if not p.is_file():missing.append(str(p));continue
        if relative not in indexed:raise ValueError(f'Asset not inventoried: {p}')
    if missing:raise FileNotFoundError('Missing assets:\n'+'\n'.join(missing))
    for relative,record in indexed.items():
        p=dataset_path(root,relative)
        if not p.is_file():raise FileNotFoundError(p)
        if p.stat().st_size!=record['bytes'] or dataset_file_hash(p)!=record['sha256']:
            raise ValueError(f'Asset checksum mismatch: {p}')
    return dataset_hash({'files':data['files']})

def prep_assets(model, models_dir, prompt_override=None):
    root=Path(models_dir)/'image_restoration'
    if model=='realesrgan_x4':
        folder=root/'realesrgan';required=['RealESRGAN_x4plus.pth']
        paths={'checkpoint':str((folder/required[0]).resolve())}
    elif model=='tadisr_x4':
        folder=root/'tadisr';base=folder/'CogView4'
        required=['tadisr_cogview4_realce_17500.pkl',
            'CogView4/model_index.json','CogView4/vae/config.json','CogView4/vae/diffusion_pytorch_model.safetensors',
            'CogView4/transformer/config.json','CogView4/transformer/diffusion_pytorch_model.safetensors.index.json',
            'CogView4/text_encoder/config.json','CogView4/text_encoder/model.safetensors.index.json',
            'CogView4/tokenizer/tokenizer.json','CogView4/tokenizer/tokenizer_config.json',
            'CogView4/tokenizer/special_tokens_map.json','CogView4/scheduler/scheduler_config.json']
        if prompt_override is None:required.append('CogView4/saved_prompt_tokens_nocfg.pt')
        for name in ('transformer/diffusion_pytorch_model.safetensors.index.json','text_encoder/model.safetensors.index.json'):
            p=base/name
            if p.is_file():required.extend('CogView4/'+str(Path(name).parent/part).replace('\\','/')
                for part in sorted(set(json.loads(p.read_text())['weight_map'].values())))
        paths={'checkpoint':str((folder/required[0]).resolve()),'base_model':str(base.resolve()),
            'prompt_embeddings':str(Path(prompt_override).resolve()) if prompt_override else str((base/'saved_prompt_tokens_nocfg.pt').resolve())}
    elif model=='difftsr':
        folder=root/'difftsr';required=['DiffTSR.ckpt','transocr.pth']
        paths={'checkpoint':str((folder/required[0]).resolve()),'transocr':str((folder/required[1]).resolve())}
    else:
        folder=root/'unihir';required=['model.safetensors','ae.safetensors','llm_config.json','vit_config.json',
            'tokenizer_config.json','tokenizer.json','vocab.json','merges.txt']
        paths={'base_model':str(folder.resolve()),'font_dir':str((folder/'font').resolve())}
        required.extend('font/'+name for name in ('KaiXinSongA.ttf','KaiXinSongB.ttf',
            'TH-Tshyn-P0.ttf','TH-Tshyn-P1.ttf','TH-Tshyn-P16.ttf','TH-Tshyn-P2.ttf'))
    identity=prep_check_manifest(folder,required)
    if model=='tadisr_x4' and dataset_file_hash(paths['checkpoint'])!='7cec8d0917305037cbafcb1b1bf81062add1613b561f5d7e2d1e599fe0baf5e0':
        raise ValueError('TADiSR adapter differs from official release checksum')
    if prompt_override:
        identity=dataset_hash({'base_assets':identity,'derived_prompt_sha256':dataset_file_hash(prompt_override)})
    return paths,identity

def prep_offline_prompt(config,runtime_dir):
    """Derive upstream's fixed prompt offline, before loading the restoration model."""
    folder=Path(config['models_dir'])/'image_restoration/tadisr'
    if (folder/'CogView4/saved_prompt_tokens_nocfg.pt').is_file():return None
    # Verify every non-derived component before loading the text encoder.
    base_required=['CogView4/'+name for name in ('model_index.json','scheduler/scheduler_config.json',
        'tokenizer/tokenizer.json','tokenizer/tokenizer_config.json','tokenizer/special_tokens_map.json',
        'text_encoder/config.json','text_encoder/model.safetensors.index.json',
        'transformer/config.json','transformer/diffusion_pytorch_model.safetensors.index.json',
        'vae/config.json','vae/diffusion_pytorch_model.safetensors')]
    for prefix,index in (('text_encoder','model.safetensors.index.json'),('transformer','diffusion_pytorch_model.safetensors.index.json')):
        p=folder/'CogView4'/prefix/index
        if not p.is_file():raise FileNotFoundError(p)
        base_required.extend(f'CogView4/{prefix}/{part}' for part in set(json.loads(p.read_text())['weight_map'].values()))
    identity=prep_check_manifest(folder,['tadisr_cogview4_realce_17500.pkl',*base_required])
    key=dataset_hash({'assets':identity,'upstream':UPSTREAM_REVISIONS['TADiSR'],'prompt_contract':'upstream_fixed_quality_nocfg_v1'})
    output=Path(runtime_dir)/'derived_prompts'/key/'saved_prompt_tokens_nocfg.pt'
    receipt=output.with_suffix('.json')
    if output.is_file() and receipt.is_file():
        cached=json.loads(receipt.read_text())
        if cached.get('sha256')==dataset_file_hash(output):return str(output)
        raise ValueError('Derived prompt cache corrupted; preserve it for review and choose a fresh runtime directory')
    python=prep_overlay(config['packages_dir'],runtime_dir,'tadisr_x4',config)
    worker=PrepWorker(runtime_dir,python,'tadisr_x4',{'base_model':str((folder/'CogView4').resolve()),
        'prompt_output':str(output.resolve())},config['model_options']['tadisr_x4'],
        config['load_timeout_seconds'],action='prepare_prompt')
    try:report=worker.ready
    finally:worker.close()
    dataset_write(output.parent,receipt.name,dataset_json_bytes({'key':key,'sha256':dataset_file_hash(output),'worker':report}))
    return str(output)

def prep_applicability(model, book, page, region, config, size=None):
    key=f'{book}:{page+1}:{region["id"]}'
    review=config['reviewed_regions'].get(key,{})
    if model=='difftsr':
        if not review.get('upright_horizontal_single_line') or not 1<=review.get('character_count',0)<=24:
            return 'not_applicable','Requires reviewed upright single horizontal line with 1..24 characters; no vertical rotation/reflow'
        if not review.get('allow_simplified_text_prior',False):
            return 'not_applicable','Upstream Text_Prediction uses zhconv zh-cn; traditional-form fidelity not established'
        box=region['source_box'];ratio=(box[2]-box[0])/(box[3]-box[1])
        if not 1<=ratio<=8:return 'not_applicable','DiffTSR requires 1 <= width/height <= 8'
        # Stricter than upstream: near-native processor ratio avoids heavy aspect warp.
        if abs(ratio/4-1)>.05:return 'not_applicable','No validated aspect-preserving 512x128 contract for this crop (require near 4:1)'
    if model=='unihir_restoration':
        if not review.get('damaged_inscription'):
            return 'not_applicable','Ordinary blurred scan is not a reviewed damaged historical inscription'
        if not config['accept_unihir_noncommercial']:
            return 'missing_assets','UniHIR CC BY-NC-ND 4.0 research-use scope not configured; do not redistribute as unrestricted assets'
    return 'ready',None

def prep_preflight(config, runtime_dir):
    cfg=prep_normalize(config);report={};failures=[]
    if Path('/kaggle/input').exists() and Path(runtime_dir).resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Runtime artifacts must be outside /kaggle/input')
    # Models with no reviewed compatible region need no download/runtime; all crops get NA receipts.
    for model in NEURAL:
        if model not in cfg['variants']:continue
        if model in ('difftsr','unihir_restoration'):
            flag='upright_horizontal_single_line' if model=='difftsr' else 'damaged_inscription'
            if not any(r.get(flag) for r in cfg['reviewed_regions'].values()):
                report[model]={'state':'not_applicable','reason':'No reviewed input matches released model contract'};continue
        try:
            if model=='unihir_restoration' and not cfg['accept_unihir_noncommercial']:
                raise RuntimeError('UniHIR research-use scope not configured')
            prompt_override=None
            if model=='tadisr_x4' and cfg.get('generate_tadisr_prompt_offline',False):
                prompt_override=prep_offline_prompt(cfg,runtime_dir)
            assets,identity=prep_assets(model,cfg['models_dir'],prompt_override=prompt_override)
            python=prep_overlay(cfg['packages_dir'],runtime_dir,model,cfg)
            if model=='difftsr':assets['config']=str(Path(prep_vendor(runtime_dir,model))/'model/DiffTSR_config.yaml')
            checkpoint=Path(assets.get('checkpoint',str(Path(assets.get('base_model',''))/'model.safetensors')))
            inventory=json.loads((checkpoint.parent/'assets.json').read_text(encoding='utf8'))
            checkpoint_record=next(item for item in inventory['files'] if item['path']==checkpoint.name)
            worker=PrepWorker(runtime_dir,python,model,assets,cfg['model_options'][model],cfg['load_timeout_seconds'],action='probe')
            try:probe=worker.ready['report']
            finally:worker.close()
            report[model]={'state':'ready','assets':assets,'checkpoint_identity':identity,'python':python,'probe':probe,
                'checkpoint':{'filename':checkpoint.name,'sha256':checkpoint_record['sha256'],'bytes':checkpoint_record['bytes']},
                'vendor_identity':dataset_hash(VENDOR_SOURCES.get({'tadisr_x4':'TADiSR','difftsr':'DiffTSR','unihir_restoration':'UniHIR'}.get(model,''),{}))}
        except Exception as exc:
            report[model]={'state':'missing_assets' if isinstance(exc,FileNotFoundError) else 'failed','reason':str(exc)}
            failures.append(f'{model}: {exc}')
    Path(runtime_dir).mkdir(parents=True,exist_ok=True)
    (Path(runtime_dir)/'preflight.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
    if failures and not cfg['allow_partial']:raise RuntimeError('Preflight stopped before PDF processing:\n'+'\n'.join(failures))
    return report

def prep_classical(image, name, options):
    from PIL import Image
    if name=='clahe' and (not 0<options.get('clip_limit',1.5)<=4 or not 2<=options.get('tile_grid',8)<=32):
        raise ValueError('Conservative CLAHE requires clip_limit (0,4] and tile_grid [2,32]')
    if name in ('original','clahe','bicubic_x2'):return dataset_variant(image,name,options)
    if name=='bicubic_x4':return image.resize((image.width*4,image.height*4),Image.Resampling.BICUBIC),{
        'method':'bicubic','parameters':{'scale':4},'warnings':['interpolation_does_not_recover_missing_strokes']}
    if name=='denoised':
        import cv2,numpy as np
        params={'h':2.,'template_window':7,'search_window':21,**options}
        if not 0<params['h']<=5 or params['template_window']%2!=1 or params['search_window']%2!=1 or not 3<=params['template_window']<=params['search_window']<=63:
            raise ValueError('Conservative NLM requires 0<h<=5 and odd windows')
        # Denoise luminance only: retain chromatic notes/stamps, independent from CLAHE.
        lab=cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2LAB)
        lab[:,:,0]=cv2.fastNlMeansDenoising(lab[:,:,0],None,float(params['h']),
            params['template_window'],params['search_window'])
        return Image.fromarray(cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)),{'method':'nlm_lab_l',
            'parameters':params,'warnings':['may_remove_weak_strokes; opt_in_visual_review_required']}
    raise ValueError(name)

def prep_image_qc(source, output, name):
    import cv2,numpy as np
    from PIL import Image
    expected_scale=2 if name=='bicubic_x2' else 4 if name in ('bicubic_x4','realesrgan_x4','tadisr_x4','tadisr_text_mask') else 1
    if name!='difftsr' and output.size!=(source.width*expected_scale,source.height*expected_scale):
        raise ValueError(f'{name}: invalid output dimensions {output.size}')
    if name=='difftsr' and abs(output.width/output.height-source.width/source.height)>.06:
        raise ValueError('DiffTSR geometry distortion exceeds rounding tolerance')
    # Luminance comparison after area reduction is diagnostic, not a glyph-fidelity score.
    reduced=output.convert('RGB').resize(source.size,Image.Resampling.BOX)
    a=cv2.cvtColor(np.asarray(source),cv2.COLOR_RGB2GRAY)
    b=cv2.cvtColor(np.asarray(reduced),cv2.COLOR_RGB2GRAY)
    def components(gray):
        _,binary=cv2.threshold(gray,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
        return int(cv2.connectedComponents(binary)[0]-1)
    before,after=components(a),components(b)
    return {'state':'needs_visual_review' if name!='original' else 'verified_pixel_identity',
        'expected_scale':expected_scale if name!='difftsr' else None,
        'luminance_mae_at_source':float(np.abs(a.astype(float)-b.astype(float)).mean()),
        'components_before':before,'components_after':after,
        'component_loss_warning':after<before*.9,'glyph_fidelity':'unverified',
        'ocr_accuracy':'not_measured'}

def prep_usage(root):return sum(p.stat().st_size for p in Path(root).rglob('*') if p.is_file())

def prep_store(root,image,budget):
    # Pre-encode to enforce storage bound before object write; content addressing deduplicates.
    image=image.convert('RGB');stream=io.BytesIO();image.save(stream,format='PNG',compress_level=6)
    data=stream.getvalue();sha=hashlib.sha256(data).hexdigest();relative=f'objects/{sha[:2]}/{sha}.png'
    p=dataset_path(root,relative)
    if not p.exists() or dataset_file_hash(p)!=sha:
        if budget['used']+len(data)>budget['limit']:raise RuntimeError('max_storage_bytes exceeded')
        dataset_write(root,relative,data);budget['used']+=len(data)
    return {'path':relative,'sha256':sha,'size':list(image.size),'pixel_sha256':hashlib.sha256(image.tobytes()).hexdigest()}

def prep_receipt_valid(root, receipt, key):
    data=json.loads(dataset_path(root,receipt).read_bytes())
    if data['key']!=key or data['state']!='ok':raise ValueError('Incomplete/mismatched cache')
    dataset_load_image(root,data['image'])
    for extra in data.get('extras',{}).values():
        if 'pixel_sha256' in extra:dataset_load_image(root,extra)
        else:dataset_read(root,extra)
    return data

def prep_run(config, runtime_dir):
    global _storage_guard
    cfg=prep_normalize(config);root=Path(cfg['output_dir']).resolve();root.mkdir(parents=True,exist_ok=True)
    if Path('/kaggle/input').exists() and root.is_relative_to(Path('/kaggle/input').resolve()):raise ValueError('Output must not be in /kaggle/input')
    for key in ('data_dir','models_dir','packages_dir'):
        if not cfg[key] or not Path(cfg[key]).is_dir():raise FileNotFoundError(f'Configure valid {key}: {cfg[key]}')
        if root==Path(cfg[key]).resolve() or root.is_relative_to(Path(cfg[key]).resolve()):raise ValueError('Output must be outside input assets')
    started=time.monotonic();preflight=prep_preflight(cfg,runtime_dir)
    _storage_guard={'root':root,'used':prep_usage(root),'limit':cfg['max_storage_bytes']}
    # Conservative pre-allocation estimate after bounded PDF selection; no downscale fallback.
    # Canonical receipts are staged, and dataset.json is published only after restoration QC.
    canonical_config=ImageDatasetConfig(variants=['original'],max_source_pixels=cfg['max_source_pixels'],**cfg['layout'])
    profile_dir=Path(runtime_dir)/'profiles';profile_dir.mkdir(parents=True,exist_ok=True)
    for entry in list_pdfs(cfg['data_dir']):
        profile=cfg.get('book_layouts',{}).get(entry['stem'],{})
        (profile_dir/(entry['stem']+'.json')).write_text(json.dumps(profile),encoding='utf8')
    stats=prepare_image_dataset(cfg['data_dir'],root,config=canonical_config,books=cfg['books'],
        pages=cfg['pages'],limit=cfg['limit'],max_books=cfg['max_books'],profile_dir=profile_dir)
    _,base=dataset_open(stats['dataset']);budget={'used':prep_usage(root),'limit':cfg['max_storage_bytes']}
    if budget['used']>budget['limit']:raise RuntimeError('Canonical dataset exceeds max_storage_bytes')
    stage=Path(runtime_dir)/('manifest_'+uuid.uuid4().hex+'.jsonl');stage.parent.mkdir(parents=True,exist_ok=True)
    counts={};measurements=[];reused=0;failures=[];rows=0
    def write_row(stream,row):
        nonlocal rows
        stream.write(dataset_json_bytes(row));stream.flush();rows+=1
        counts[row['state']]=counts.get(row['state'],0)+1
    # Lock covers variant generation/publication; canonical builder has its own lock.
    with dataset_lock(root),stage.open('wb') as manifest:
        for name in cfg['variants']:
            if name=='tadisr_text_mask':continue # emitted alongside each TADiSR image, never omitted
            worker=None
            try:
                for entry in base['pages']:
                    record=json.loads(dataset_read(root,entry['record']))
                    for region in record['regions']:
                        original=region['images']['original'];x1,y1,x2,y2=region['source_box']
                        common={'book':entry['book'],'page':entry['page']+1,'page_index':entry['page'],
                            'source_sha256':entry['source_sha256'],'source_pdf':entry['source_pdf'],
                            'source_image':record['source'],'source_image_hash':record['source']['sha256'],
                            'source_box':region['source_box'],'crop_type':region['crop_type'],
                            'region_id':region['id'],'reading_order':region['reading_order'],
                            'canonical_image':original,'variant':name,'provenance':PROVENANCE[name]}
                        state,reason='ready',None
                        if region['crop_type'] not in cfg['crop_types']:state,reason='skipped_by_configuration','crop_type deselected'
                        elif name in NEURAL:state,reason=prep_applicability(name,entry['book'],entry['page'],region,cfg)
                        model_info=preflight.get(name,{})
                        if state=='ready' and name in NEURAL and model_info.get('state')!='ready':
                            state,reason=model_info.get('state','failed'),model_info.get('reason','Model not ready')
                        if state!='ready':
                            row={**common,'state':state,'reason':reason};write_row(manifest,row)
                            if name=='tadisr_x4':write_row(manifest,{**row,'variant':'tadisr_text_mask'})
                            if state in ('missing_assets','failed'):failures.append(row)
                            continue
                        options=cfg['model_options'].get(name,cfg['variant_options'].get(name,{}))
                        identity={'model':model_info.get('checkpoint_identity'),'checkpoint':model_info.get('checkpoint'),
                            'architecture':{'realesrgan_x4':'RRDBNet(3,3,64,23,32), scale=4','tadisr_x4':'CogView4-RealCE with upstream joint segmentation decoder',
                                'difftsr':'DiffTSR released IDM/TDM/MoM + TransOCR prior','unihir_restoration':'UniHIR BAGEL/MLLM'} .get(name),
                            'vendor':model_info.get('vendor_identity'),
                            'runtime':model_info.get('probe')}
                        # PID is diagnostic, never an input identity; changing process must not invalidate cache.
                        if identity['runtime']:identity['runtime']={k:v for k,v in identity['runtime'].items() if k!='pid'}
                        key=dataset_hash({'source':original['sha256'],'box':region['source_box'],'name':name,'options':options,
                            'model':identity,'code':RESTORATION_CODE_SIGNATURE,'opencv':__import__('cv2').__version__})
                        receipt=f'restoration_cache/{key}.json';result=None
                        if cfg['resume']:
                            try:result=prep_receipt_valid(root,receipt,key);reused+=1
                            except (OSError,ValueError,KeyError):pass
                        if result is None:
                            tick=time.monotonic()
                            try:
                                source=dataset_load_image(root,original)
                                scale=4 if name in ('bicubic_x4','realesrgan_x4','tadisr_x4') else 2 if name=='bicubic_x2' else 1
                                if source.width*source.height*scale**2>cfg['max_output_pixels']:
                                    raise ValueError(f'{name} output exceeds max_output_pixels; split crop or explicitly raise limit')
                                extras={};info={};operation={'method':name,'parameters':options}
                                if name in CLASSICAL:output,operation=prep_classical(source,name,options)
                                else:
                                    if worker is None:
                                        worker=PrepWorker(runtime_dir,model_info['python'],name,model_info['assets'],options,cfg['load_timeout_seconds'])
                                    folder,info=worker.infer(dataset_path(root,original['path']),int(key[:8],16),cfg['timeout_seconds'])
                                    info['model_load_seconds']=worker.ready.get('load_seconds')
                                    from PIL import Image
                                    with Image.open(folder/'result.png') as loaded:output=loaded.convert('RGB')
                                    for extra,relative in info['extras'].items():
                                        p=dataset_path(folder,relative)
                                        if p.suffix=='.png':
                                            with Image.open(p) as im:
                                                if im.width*im.height>cfg['max_output_pixels']:raise ValueError('Extra output exceeds pixel budget')
                                                extras[extra]=prep_store(root,im,budget)
                                        else:
                                            data=p.read_bytes();h=hashlib.sha256(data).hexdigest()
                                            extras[extra]=dataset_write(root,f'diagnostics/{h}.json',data)
                                qc=prep_image_qc(source,output,name)
                                if name=='tadisr_x4' and 'tadisr_text_mask' not in extras:
                                    raise ValueError('TADiSR must emit both SR image and text mask')
                                if 'tadisr_text_mask' in extras:
                                    mask=dataset_load_image(root,extras['tadisr_text_mask'])
                                    if mask.size!=output.size:raise ValueError('TADiSR mask and SR dimensions differ')
                                image=prep_store(root,output,budget)
                                mapping=[(x2-x1)/output.width,0,0,(y2-y1)/output.height,x1,y1]
                                result={'key':key,'state':'ok','image':{**image,'image_to_source':mapping},'extras':extras,
                                    'operation':operation,'model_identity':identity,'qc':qc,'seconds':round(time.monotonic()-tick,6),
                                    'coordinate_mapping_evidence':'nominal_canvas_only; generated_content_unverified' if name=='unihir_restoration' else 'crop_and_scale_geometry',
                                    'worker':info,'warnings':['possible_added_removed_merged_strokes; inspect_against_original'] if name in NEURAL else operation.get('warnings',[])}
                                dataset_write(root,receipt,dataset_json_bytes(result))
                                # Worker scratch is bounded per region; completed artifacts are already hashed/atomic.
                                if name in NEURAL:
                                    for p in folder.iterdir():
                                        if p.is_file():p.unlink()
                                    folder.rmdir()
                            except Exception as exc:
                                result={'key':key,'state':'failed','reason':str(exc),'seconds':round(time.monotonic()-tick,6)}
                                dataset_write(root,receipt,dataset_json_bytes(result));failures.append({**common,**result})
                                # Restart a failed/timed-out worker for the next region.
                                if worker is not None:worker.close();worker=None
                        row={**common,**result};write_row(manifest,row)
                        if result['state']=='ok':measurements.append({k:result[k] for k in ('seconds','worker')}|{'variant':name})
                        if name=='tadisr_x4':
                            mask=result.get('extras',{}).get('tadisr_text_mask')
                            maskrow={**row,'variant':'tadisr_text_mask'}
                            if mask:
                                maskrow['image']={**mask,'image_to_source':row['image']['image_to_source']}
                                maskrow['provenance']='neural_restoration';maskrow['warnings']=['model_predicted_mask_is_not_source_character_evidence']
                            else:maskrow.update(state='failed',reason='Missing paired TADiSR text mask')
                            write_row(manifest,maskrow)
            finally:
                if worker is not None:worker.close()
        manifest.flush();os.fsync(manifest.fileno())
    data=stage.read_bytes();sha=hashlib.sha256(data).hexdigest()
    manifest_ref=dataset_write(root,f'restoration_manifests/{sha}.jsonl',data)
    snapshot_preflight=json.loads(json.dumps(preflight))
    for info in snapshot_preflight.values():
        if 'probe' in info:info['probe'].pop('pid',None)
    report={'format':'vtm-image-restoration','version':1,'canonical_snapshot':base['snapshot_id'],
        'manifest':manifest_ref,'selected_variants':cfg['variants'],'crop_types':cfg['crop_types'],
        'states':counts,'partial':bool(failures),'preflight':snapshot_preflight,'provenance_categories':sorted(set(PROVENANCE.values())),
        'configuration':cfg,'code_sha256':RESTORATION_CODE_SIGNATURE}
    report_ref=dataset_write(root,f'restoration_manifests/{dataset_hash(report)}.json',dataset_json_bytes(report))
    # Validate every row, hash and expected pair before publishing immutable snapshot/marker.
    prep_validate_extension(root,base,report)
    summary={'canonical':stats,'states':counts,'manifest_rows':rows,'variants_reused':reused,
        'seconds':round(time.monotonic()-started,3),'storage_bytes':prep_usage(root),'measurements':measurements,
        'report':report_ref,'failures':len(failures)}
    if failures and not cfg['allow_partial']:
        dataset_write(root,'last_attempt.json',dataset_json_bytes(summary))
        raise RuntimeError(f'{len(failures)} variants failed; completed cache retained; dataset.json not replaced. See last_attempt.json')
    final={k:v for k,v in base.items() if k!='snapshot_id'}
    final['restoration']={'format':'vtm-image-restoration','version':1,'report':report_ref,'partial':bool(failures)}
    final['snapshot_id']=dataset_hash(final)
    with dataset_lock(root):
        dataset_write(root,f'snapshots/{final["snapshot_id"]}.json',dataset_json_bytes(final))
        dataset_write(root,'dataset.json',dataset_json_bytes(final))
    summary['snapshot_id']=final['snapshot_id'];summary['dataset']=str(root/'dataset.json')
    dataset_write(root,'last_run.json',dataset_json_bytes(summary))
    return summary

def prep_validate_extension(root, base, report):
    expected={}
    for entry in base['pages']:
        record=json.loads(dataset_read(root,entry['record']));dataset_validate_page(root,record)
        for r in record['regions']:
            for name in report['selected_variants']:
                expected[(entry['book'],entry['page']+1,r['id'],name)]=r
    seen=set();states={}
    for line in dataset_read(root,report['manifest']).splitlines():
        row=json.loads(line);key=(row['book'],row['page'],row['region_id'],row['variant'])
        if key not in expected or key in seen:raise ValueError('Unexpected/duplicate restoration manifest entry')
        seen.add(key);region=expected[key]
        states[row['state']]=states.get(row['state'],0)+1
        if row['provenance']!=PROVENANCE[row['variant']]:raise ValueError('Invalid evidence provenance')
        if row['source_box']!=region['source_box'] or row['reading_order']!=region['reading_order']:
            raise ValueError('Restoration crop geometry changed')
        if row['state']=='ok':
            image=dataset_load_image(root,row['image']);x1,y1,x2,y2=region['source_box']
            scale=2 if row['variant']=='bicubic_x2' else 4 if row['variant'] in ('bicubic_x4','realesrgan_x4','tadisr_x4','tadisr_text_mask') else 1
            if row['variant']!='difftsr' and image.size!=((x2-x1)*scale,(y2-y1)*scale):
                raise ValueError('Restoration variant dimensions violate scale contract')
            if row['variant']=='original' and row['image']['pixel_sha256']!=row['canonical_image']['pixel_sha256']:
                raise ValueError('Original variant differs from canonical crop pixels')
            if row['image']['image_to_source']!=[(x2-x1)/image.width,0,0,(y2-y1)/image.height,x1,y1]:raise ValueError('Invalid source mapping')
            for ref in row.get('extras',{}).values():dataset_read(root,ref)
        elif row['state'] not in ('not_applicable','missing_assets','failed','skipped_by_configuration'):
            raise ValueError('Invalid variant state')
    if seen!=set(expected):raise ValueError('Selected variants silently omitted')
    if states!=report['states']:raise ValueError('Manifest state counts differ from report')
    if (states.get('failed',0) or states.get('missing_assets',0)) and not report['partial']:
        raise ValueError('Incomplete selected methods falsely labelled complete')
    return {'rows':len(seen),'validated':True,'glyph_fidelity':'unverified'}

def prep_open(directory):
    root,base=dataset_open(directory)
    extension=base.get('restoration')
    if not extension or extension['version']!=1:raise ValueError('Missing/unsupported restoration extension')
    report=json.loads(dataset_read(root,extension['report']))
    return root,base,report

def prep_validate(directory):
    root,base,report=prep_open(directory)
    core=validate_image_dataset(directory)
    return {'canonical':core,'restoration':prep_validate_extension(root,base,report),'partial':report['partial']}

def prep_previews(directory, maximum=3, detail_box=(.1,.15,.4,.45)):
    from PIL import Image,ImageDraw
    root,base,report=prep_open(directory);chosen={}
    # Bounded number of identical regions, include every selected method/status for comparison.
    for line in dataset_read(root,report['manifest']).splitlines():
        row=json.loads(line);key=(row['book'],row['page'],row['region_id'])
        if key not in chosen and len(chosen)<maximum and row['crop_type'] in ('group','page'):chosen[key]=[]
        if key in chosen:chosen[key].append(row)
    outputs=[]
    if detail_box and (len(detail_box)!=4 or not 0<=detail_box[0]<detail_box[2]<=1 or not 0<=detail_box[1]<detail_box[3]<=1):
        raise ValueError('Preview detail_box requires fractional x1,y1,x2,y2 in [0,1]')
    for key,rows in chosen.items():
        cell_w,cell_h=300,350;columns=min(4,len(rows));height=((len(rows)+columns-1)//columns)*cell_h
        sheet=Image.new('RGB',(columns*cell_w,height),'#eeeeee');draw=ImageDraw.Draw(sheet)
        for i,row in enumerate(rows):
            left,top=i%columns*cell_w,i//columns*cell_h
            draw.text((left+8,top+5),row['variant']+' / '+row['state'],fill='black')
            if row['state']=='ok':
                image=dataset_load_image(root,row['image']);image.thumbnail((cell_w-16,cell_h-50))
                sheet.paste(image,(left+8,top+30))
            else:
                reason=row.get('reason','');draw.multiline_text((left+8,top+35),'\n'.join(reason[j:j+38] for j in range(0,min(240,len(reason)),38)),fill='black')
        path=root/'previews'/f'{key[0]}_p{key[1]}_{key[2]}.png';path.parent.mkdir(exist_ok=True)
        tmp=path.with_suffix('.tmp');sheet.save(tmp,format='PNG');tmp.replace(path);outputs.append(str(path))
        if detail_box:
            # All panels show exactly the same subregion in canonical source coordinates.
            detail=Image.new('RGB',(columns*400,((len(rows)+columns-1)//columns)*420),'#eeeeee')
            draw_detail=ImageDraw.Draw(detail)
            for i,row in enumerate(rows):
                left,top=i%columns*400,i//columns*420
                draw_detail.text((left+8,top+5),row['variant']+' / source-aligned detail',fill='black')
                if row['state']=='ok':
                    image=dataset_load_image(root,row['image'])
                    box=[round(detail_box[j]*image.size[j%2]) for j in range(4)]
                    box[0]=min(box[0],image.width-1);box[1]=min(box[1],image.height-1)
                    box[2]=min(image.width,max(box[0]+1,box[2]));box[3]=min(image.height,max(box[1]+1,box[3]))
                    patch=image.crop(box)
                    factor=min(384/patch.width,380/patch.height)
                    patch=patch.resize((max(1,round(patch.width*factor)),max(1,round(patch.height*factor))),
                        Image.Resampling.NEAREST if row['variant']=='original' else Image.Resampling.LANCZOS)
                    detail.paste(patch,(left+8,top+30))
                else:draw_detail.text((left+8,top+30),row['state'],fill='black')
            path=path.with_name(path.stem+'_detail.png');tmp=path.with_suffix('.tmp')
            detail.save(tmp,format='PNG');tmp.replace(path);outputs.append(str(path))
    return outputs


In [ ]:
from __future__ import annotations
import sys,os,json,hashlib,subprocess,time,urllib.request,shutil
from pathlib import Path

TADISR_WHEELS = ['diffusers==0.35.2','peft==0.17.1','transformers==4.56.2',
    'tokenizers==0.22.1','accelerate==1.10.1','huggingface-hub==0.35.3',
    'safetensors==0.6.2','sentencepiece==0.2.1','importlib-metadata==8.7.0',
    'zipp==3.23.0','packaging==25.0','pyyaml==6.0.3','psutil==7.1.0']

def assets_fetch(url,path,expected=None):
    """Local staging only. No overwrite of a completed asset with different bytes."""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.is_file():
        digest=dataset_file_hash(path)
        if expected and digest!=expected:raise ValueError(f'Existing asset differs from expected checksum: {path}; preserved')
        return {'path':str(path),'url':url,'sha256':digest,'bytes':path.stat().st_size,'state':'reused_verified'}
    temporary=path.with_name(path.name+'.part')
    request=urllib.request.Request(url,headers={'User-Agent':'vtm-image-preparation/1'})
    with urllib.request.urlopen(request,timeout=180) as response,temporary.open('wb') as output:
        if path.suffix in ('.pth','.ckpt','.safetensors','.pkl') and 'text/html' in response.headers.get('Content-Type',''):
            raise RuntimeError(f'Expected weights, received an HTML confirmation/authentication page: {url}')
        shutil.copyfileobj(response,output,4*1024*1024)
    digest=dataset_file_hash(temporary)
    if expected and digest!=expected:raise ValueError(f'Download checksum mismatch: {temporary}')
    temporary.replace(path)
    return {'path':str(path),'url':url,'sha256':digest,'bytes':path.stat().st_size,'state':'downloaded_verified'}

def assets_api(url):
    with urllib.request.urlopen(url,timeout=60) as response:return json.load(response)

def assets_inventory(folder,license_info,source_info):
    folder=Path(folder);files=[]
    for path in sorted(folder.rglob('*')):
        if path.is_file() and path.name!='assets.json' and not path.name.endswith('.part'):
            files.append({'path':path.relative_to(folder).as_posix(),'sha256':dataset_file_hash(path),'bytes':path.stat().st_size})
    obj={'format':'vtm-local-model-assets','version':1,'license':license_info,'sources':source_info,'files':files,
        'inference_verified':False}
    folder.mkdir(parents=True,exist_ok=True);target=folder/'assets.json'
    data=dataset_json_bytes(obj)
    # Inventories are regenerated from actual files; weights themselves are never overwritten.
    temporary=target.with_suffix('.tmp');temporary.write_bytes(data);temporary.replace(target)
    return obj

def assets_prepare(staging_dir, allow_internet=False, download_large_tadisr=False, python_version='313',
                   download_difftsr=False,download_unihir=False,accept_unihir_noncommercial=False):
    if Path('/kaggle/working').is_dir():raise RuntimeError('prepare_assets is local-only; prohibited on Kaggle')
    if not allow_internet:raise RuntimeError('Set ALLOW_ASSET_INTERNET=True explicitly in local prepare_assets mode')
    stage=Path(staging_dir).resolve()
    if stage.name!='kaggle_upload':raise ValueError('All downloadable assets must be under the existing kaggle_upload root')
    models=stage/'models/image_restoration';packages=stage/'packages/image_restoration'
    for name in ('wheels','requirements','manifests','sources'):(packages/name).mkdir(parents=True,exist_ok=True)
    results=[];blockers=[]
    def fetch(url,path,expected=None):
        try:results.append(assets_fetch(url,path,expected));return True
        except Exception as e:blockers.append({'path':str(path),'url':url,'error':str(e)});return False
    # Code is also staged for provenance, but execution uses copies embedded in the notebook.
    for name,files in VENDOR_SOURCES.items():
        folder=packages/'sources'/name
        for relative,source in files.items():
            p=dataset_path(folder,relative);p.parent.mkdir(parents=True,exist_ok=True)
            if p.exists() and p.read_text(encoding='utf8')!=source:raise ValueError(f'Preserve existing source: {p}')
            p.write_text(source,encoding='utf8')
    fetch('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        models/'realesrgan/RealESRGAN_x4plus.pth')
    fetch('https://huggingface.co/huqiming513/TADiSR-Models/resolve/e3e9a65d1fbbb767d593f91d8677224cb3b10293/tadisr_cogview4_realce_17500.pkl',
        models/'tadisr/tadisr_cogview4_realce_17500.pkl',
        '7cec8d0917305037cbafcb1b1bf81062add1613b561f5d7e2d1e599fe0baf5e0')
    if download_large_tadisr:
        # Pin revision. API tree supplies LFS sha256 and file sizes for all actual components.
        repo='zai-org/CogView4-6B';revision='63a52b7f6dace7033380cd6da14d0915eab3e6b5'
        files=assets_api(f'https://huggingface.co/api/models/{repo}/tree/{revision}?recursive=true&limit=1000')
        for record in files:
            if record['type']!='file' or record['path']=='.gitattributes':continue
            fetch(f'https://huggingface.co/{repo}/resolve/{revision}/{record["path"]}',
                models/'tadisr/CogView4'/record['path'],record.get('lfs',{}).get('oid'))
    elif not (models/'tadisr/CogView4/vae/diffusion_pytorch_model.safetensors').is_file():
        blockers.append({'model':'tadisr','error':'Full CogView4 download not enabled; verify all base shards before use'})
    if download_difftsr:
        for name,ident in [('DiffTSR.ckpt','1UZWJ0PlOovrikqAQPAhnBIhuMXFYAxy6'),('transocr.pth','1wcCVZwRTUzww6OLQN7gmWzpg2fDy-N8Q')]:
            fetch(f'https://drive.usercontent.google.com/download?id={ident}&export=download&confirm=t',models/'difftsr'/name)
    if download_unihir:
        if not accept_unihir_noncommercial:
            raise ValueError('UniHIR assets retain CC BY-NC-ND 4.0 restrictions; configure noncommercial research scope first')
        listing=assets_api('https://www.modelscope.cn/api/v1/models/zzxfzyy/UniHIR/repo/files?Revision=master&Recursive=true')
        for file in listing['Data']['Files']:
            if file['Type']=='blob' and file['Path']!='.gitattributes':
                fetch(f'https://www.modelscope.cn/models/zzxfzyy/UniHIR/resolve/{file["Revision"]}/{file["Path"]}',
                    dataset_path(models/'unihir',file['Path']),file.get('Sha256'))
        revision=UPSTREAM_REVISIONS['UniHIR']
        tree=assets_api(f'https://api.github.com/repos/ZZXF11/UniHIR/git/trees/{revision}?recursive=1')
        required_fonts={'font/'+name for name in ('KaiXinSongA.ttf','KaiXinSongB.ttf','TH-Tshyn-P0.ttf',
            'TH-Tshyn-P1.ttf','TH-Tshyn-P16.ttf','TH-Tshyn-P2.ttf')}
        for file in tree['tree']:
            if file['type']=='blob' and file['path'] in required_fonts:
                fetch(f'https://raw.githubusercontent.com/ZZXF11/UniHIR/{revision}/{file["path"]}',
                    dataset_path(models/'unihir',file['path']))
    # PDFium is already in the OCR package dataset; reuse its Linux artifact when present.
    base_wheels=list((stage/'packages/wheels').glob('pypdfium2-*.whl'))
    roots=list(TADISR_WHEELS)
    if not base_wheels:roots.append('pypdfium2==5.15.0')
    requirements=packages/'requirements/tadisr.txt';requirements.write_text('\n'.join(roots)+'\n',encoding='utf8')
    # These are overlays, deliberately excluding Torch/torchvision/NumPy. Preflight checks
    # the actual inherited stack and full active dependency closure in the target runtime.
    import importlib.util
    pip_command=[sys.executable,'-m','pip']
    if importlib.util.find_spec('pip') is None:
        pip_wheels=sorted((stage/'packages/wheels').glob('pip-*.whl'))
        if not pip_wheels:raise RuntimeError('Local staging needs pip or the existing packages/wheels/pip-*.whl')
        pip_command=[sys.executable,'-c',
            'import sys; sys.path.insert(0,sys.argv.pop(1)); from pip._internal.cli.main import main; sys.exit(main())',
            str(pip_wheels[-1])]
    command=[*pip_command,'--isolated','download','--only-binary=:all:','--no-deps',
        '--platform','manylinux_2_28_x86_64','--platform','manylinux2014_x86_64',
        '--python-version',python_version,'--implementation','cp','--abi','cp'+python_version,
        '--abi','abi3','--abi','none','--dest',str(packages/'wheels'),*roots]
    log=packages/'manifests/pip_download.log'
    with log.open('w',encoding='utf8') as stream:
        result=subprocess.run(command,stdout=stream,stderr=subprocess.STDOUT)
    if result.returncode:blockers.append({'runtime':'tadisr','error':f'Linux wheel download failed; see {log}'})
    wheel_records=[]
    for p in sorted((packages/'wheels').glob('*.whl')):
        if 'win_' in p.name:raise ValueError(f'Windows wheel in Kaggle assets: {p}')
        from packaging.utils import parse_wheel_filename
        import zipfile
        name,version,_,tags=parse_wheel_filename(p.name)
        metadata=assets_api(f'https://pypi.org/pypi/{name}/{version}/json')
        candidates=[record for record in metadata['urls'] if record['filename']==p.name]
        if len(candidates)!=1:raise RuntimeError(f'Cannot verify exact PyPI wheel identity: {p.name}')
        digest=dataset_file_hash(p)
        if digest!=candidates[0]['digests']['sha256']:raise ValueError(f'PyPI checksum differs: {p}')
        with zipfile.ZipFile(p) as archive:
            if archive.testzip():raise ValueError(f'Wheel CRC failed: {p}')
        wheel_records.append({'path':p.relative_to(packages).as_posix(),'sha256':digest,'bytes':p.stat().st_size,
            'source_url':candidates[0]['url'],'pypi_sha256_verified':True,
            'wheel_tags':sorted(str(t) for t in tags),
            'license':metadata['info'].get('license_expression') or metadata['info'].get('license') or
                [c for c in metadata['info'].get('classifiers',[]) if c.startswith('License ::')]})
    runtime={'format':'vtm-restoration-runtime','version':1,'python':python_version,'wheels':wheel_records,
        'kernel_reuse':['torch','torchvision','numpy','pillow','requests','filelock','fsspec','tqdm','regex','typing_extensions'],
        'requirements':roots,'target':'Linux x86_64; validate current Python ABI, CUDA and GPU in worker',
        'kaggle_install_executed':False,'gpu_inference_verified':False}
    (packages/'manifests/tadisr_runtime.json').write_bytes(dataset_json_bytes(runtime))
    assets_inventory(models/'realesrgan','Real-ESRGAN BSD-3-Clause; BasicSR inference architecture Apache-2.0',
        ['https://github.com/xinntao/Real-ESRGAN'])
    assets_inventory(models/'tadisr','MIT code; adapter research use; CogView4 base Apache-2.0',
        ['https://github.com/mingcv/TADiSR','https://huggingface.co/zai-org/CogView4-6B',
         'https://huggingface.co/huqiming513/TADiSR-Models'])
    if not (models/'tadisr/CogView4/saved_prompt_tokens_nocfg.pt').exists():
        blockers.append({'model':'tadisr','error':'Missing deterministic saved_prompt_tokens_nocfg.pt. Generate locally with embedded prepare_tadisr_prompt() in a verified model runtime; update inventory afterwards.'})
    for model,license_info,url in [('difftsr','Apache-2.0 upstream','https://github.com/YuzheZhang-1999/DiffTSR'),
            ('unihir','CC BY-NC-ND 4.0 noncommercial research; upstream/font/base terms also apply','https://github.com/ZZXF11/UniHIR')]:
        if (models/model).is_dir():assets_inventory(models/model,license_info,[url])
    blockers.extend([
        {'model':'difftsr','error':'Verify DiffTSR.ckpt AND transocr.pth inventory; independently compatible legacy runtime still required. Input contracts apply',
         'url':'https://drive.google.com/drive/folders/1K6k5ZcvF3w-1MDN_gXQTdsLgFZ2SM8qy'},
        {'model':'unihir','error':'Verify ModelScope full BAGEL assets/6 fonts; separately validated Python3.10/flash-attn runtime still required. CC BY-NC-ND 4.0 noncommercial research scope',
         'url':'https://www.modelscope.cn/models/zzxfzyy/UniHIR'}])
    summary={'downloads':results,'wheels':wheel_records,'reused_pdfium':[str(p) for p in base_wheels],
        'blockers':blockers,'automatic_upload':False,'target_install_tested':False}
    (packages/'manifests/staging_report.json').write_bytes(dataset_json_bytes(summary))
    return summary

def prepare_tadisr_prompt(models_dir, python, runtime_dir, device='cuda'):
    """Explicit local preparation; deterministic upstream prompt, separate process."""
    if Path('/kaggle/working').is_dir():raise RuntimeError('Prepare prompt locally before uploading assets')
    vendor=prep_vendor(runtime_dir,'tadisr_x4');base=Path(models_dir)/'image_restoration/tadisr/CogView4'
    output=base/'saved_prompt_tokens_nocfg.pt'
    if output.exists():raise FileExistsError(f'Preserve completed embedding: {output}')
    command=[str(python),str(Path(vendor)/'scripts/prepare_prompt_embeddings.py'),
        '--base-model',str(base.resolve()),'--output',str(output.resolve()),'--device',device]
    subprocess.run(command,check=True,env=prep_worker_environment())
    return assets_inventory(base.parent,'MIT code; adapter research use; CogView4 Apache-2.0',
        ['https://github.com/mingcv/TADiSR'])


## 5. Tự kiểm thử nhúng

Fixture PDF/image là tổng hợp; kiểm tra native/fallback/rotation/overlay/mask,
tọa độ và RTL, cả năm phương pháp cổ điển, missing assets, process isolation,
resume sau ngắt, đổi tham số/hash trọng số, ảnh hỏng, manifest, pixel/storage budget.
Test tiling dùng tensor thật và hàm identity: **không phải inference model thật**.
Kiểm thử không cần file trong tests/scripts. Nó không đo CER hay fidelity từng glyph.


In [ ]:
import zlib
from PIL import Image, ImageDraw
def make_scan_pdf(path, *, size=(120,80), pages=2, rotation=0, clip=False,
                  overlay=False, multiple=False, annotation=False, reflected=False,
                  cropped=False, alpha=False, inverted=False):
    """Minimal explicit PDF fixture exercises transformations PDFium must honor."""
    w,h = size
    image = Image.new("RGB",size,"#e8ddc4")
    draw = ImageDraw.Draw(image)
    for x in range(12,w-10,12):
        for y in range(10,h-10,12):
            draw.rectangle((x,y,x+3,y+6),fill="black")
    image.putpixel((2,3),(255,0,0))
    objects = [b"<< /Type /Catalog /Pages 2 0 R >>",b""]
    def add(data):
        objects.append(data); return len(objects)
    def stream(options,data):
        return b"<< "+options+b" /Length "+str(len(data)).encode()+b" >>\nstream\n"+data+b"\nendstream"
    mask_id = add(stream(f"/Type /XObject /Subtype /Image /Width {w} /Height {h} /ColorSpace /DeviceGray /BitsPerComponent 8 /Filter /FlateDecode".encode(),zlib.compress(bytes([128])*(w*h)))) if alpha else None
    image_id = add(stream((f"/Type /XObject /Subtype /Image /Width {w} /Height {h} /ColorSpace /DeviceRGB /BitsPerComponent 8 /Filter /FlateDecode"+
        (f" /SMask {mask_id} 0 R" if alpha else "")+ (" /Decode [1 0 1 0 1 0]" if inverted else "")).encode(),zlib.compress(image.tobytes())))
    page_ids = []
    for index in range(pages):
        command = (f"q {'10 0 '+str(w-10)+' '+str(h)+' re W n ' if clip else ''}"
                   f"{-w if reflected else w} 0 0 {h} {w if reflected else 0} 0 cm /Scan Do Q\n")
        if overlay:
            command += "0 0 1 rg 5 5 10 10 re f\n"
        if multiple:
            command += "q 20 0 0 20 1 1 cm /Scan Do Q\n"
        content = add(stream(b"",command.encode()))
        annot = add(b"<< /Type /Annot /Subtype /Square /Rect [3 3 30 30] /C [1 0 0] /F 4 >>") if annotation else None
        page_ids.append(add((f"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 {w} {h}] /Rotate {rotation} "+
            (f"/CropBox [5 5 {w-5} {h-5}] " if cropped else "")+
            f"/Resources << /XObject << /Scan {image_id} 0 R >> >> /Contents {content} 0 R "+
            (f"/Annots [{annot} 0 R] " if annot else "")+">>").encode()))
    objects[1] = f"<< /Type /Pages /Count {pages} /Kids [{' '.join(str(i)+' 0 R' for i in page_ids)}] >>".encode()
    output = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for i,obj in enumerate(objects,1):
        offsets.append(len(output)); output.extend(f"{i} 0 obj\n".encode()+obj+b"\nendobj\n")
    start = len(output)
    output.extend(f"xref\n0 {len(objects)+1}\n0000000000 65535 f \n".encode())
    for offset in offsets[1:]:
        output.extend(f"{offset:010d} 00000 n \n".encode())
    output.extend(f"trailer\n<< /Size {len(objects)+1} /Root 1 0 R >>\nstartxref\n{start}\n%%EOF\n".encode())
    Path(path).write_bytes(output)
    return image


import unittest,tempfile,copy,types,importlib.util
from unittest.mock import patch

def prep_test_config(folder):
    folder=Path(folder);inputs=folder/'input';inputs.mkdir(exist_ok=True)
    models=folder/'models';models.mkdir(exist_ok=True);packages=folder/'packages';packages.mkdir(exist_ok=True)
    return prep_normalize({'data_dir':str(inputs),'models_dir':str(models),'packages_dir':str(packages),
        'output_dir':str(folder/'dataset'),'books':['scan'],'pages':'1-2','limit':2,'max_books':1,
        'variants':['original','clahe','bicubic_x2','bicubic_x4','denoised'],
        'crop_types':['page','group','column'],'resume':True,'allow_partial':False,
        'max_source_pixels':1000000,'max_output_pixels':16000000,'max_storage_bytes':100000000,
        'timeout_seconds':20,'load_timeout_seconds':90,'batch_size':1,'deskew':False,'dewarp':False,
        'accept_unihir_noncommercial':False,'runtime_pythons':{},'reviewed_regions':{},'book_layouts':{},
        'layout':{'dpi':72,'column_cuts':[.25,.5,.75]},'variant_options':{},
        'model_options':{'realesrgan_x4':{'tile':64,'overlap':8,'device':'cuda','half':False},
            'tadisr_x4':{'tile':64,'overlap':32,'device':'cuda','seed':42},
            'difftsr':{'steps':20,'device':'cuda'},'unihir_restoration':{'steps':50,'ref_count':6,'device':'cuda'}}})

class OfflineImageTests(unittest.TestCase):
    def test_pdf_native_and_composition_fallback(self):
        import pypdfium2 as pdfium
        from PIL import ImageChops
        with tempfile.TemporaryDirectory() as directory:
            for options in ({},{'rotation':90},{'cropped':True},{'overlay':True},{'alpha':True},{'annotation':True}):
                path=Path(directory)/'scan.pdf';source=make_scan_pdf(path,pages=1,**options)
                with pdfium.PdfDocument(path) as doc:
                    p=doc[0];image,metadata=acquire_pdf_image(p,dpi=72)
                    if not options:
                        self.assertEqual(metadata['method'],'native_validated');self.assertEqual(source.tobytes(),image.tobytes())
                    else:
                        self.assertEqual(metadata['method'],'pdf_render')
                        bitmap=p.render(scale=metadata['render_scale'],draw_annots=True)
                        expected=bitmap.to_pil().convert('RGB');bitmap.close()
                        self.assertIsNone(ImageChops.difference(expected,image).getbbox())
                    p.close()
    def test_analysis_mapping_partition_and_rtl(self):
        from PIL import Image
        image=Image.new('RGB',(601,503),'white')
        cfg=ImageDatasetConfig(analysis_max_side=123,column_cuts=[.17,.34,.51,.68,.85])
        layout,_,_=dataset_plan_layout(image,cfg)
        self.assertEqual(layout['analysis_size'],[123,103])
        self.assertEqual(layout['analysis_to_source'],[601/123,0,0,503/103,0,0])
        for kind in ('page','group','column'):
            regions=[r for r in layout['regions'] if r['crop_type']==kind]
            dataset_partition_qc(regions,image.size,'vertical_rtl')
            self.assertEqual(sorted(r['reading_order'] for r in regions),list(range(len(regions))))
            self.assertEqual(sum((r['source_box'][2]-r['source_box'][0])*503 for r in regions),601*503)
    def test_all_classical_dimensions_alignment(self):
        from PIL import Image,ImageDraw
        im=Image.new('RGB',(64,48),'white');ImageDraw.Draw(im).rectangle((13,10,25,37),fill='black')
        for method in CLASSICAL:
            output,_=prep_classical(im,method,{})
            qc=prep_image_qc(im,output,method)
            self.assertEqual(qc['glyph_fidelity'],'unverified')
            if method=='original':self.assertEqual(output.tobytes(),im.tobytes())
    def test_missing_model_fails_before_pdf(self):
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);cfg['variants']=['original','realesrgan_x4']
            with self.assertRaisesRegex(RuntimeError,'Preflight stopped'):
                prep_run(cfg,Path(d)/'runtime')
            self.assertFalse((Path(cfg['output_dir'])/'canonical.json').exists())
    def test_manifest_resume_invalidation_corruption(self):
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);make_scan_pdf(Path(cfg['data_dir'])/'scan.pdf')
            runtime=Path(d)/'runtime';first=prep_run(cfg,runtime)
            original_acquire=globals()['acquire_pdf_image']
            def no_acquire(*a,**kw):raise AssertionError('Unnecessary source acquisition')
            globals()['acquire_pdf_image']=no_acquire
            try:second=prep_run(cfg,runtime)
            finally:globals()['acquire_pdf_image']=original_acquire
            self.assertEqual(first['snapshot_id'],second['snapshot_id'])
            self.assertEqual(second['states']['ok'],second['variants_reused'])
            cfg['variant_options']={'clahe':{'clip_limit':2.}}
            third=prep_run(cfg,runtime)
            self.assertNotEqual(second['snapshot_id'],third['snapshot_id'])
            self.assertEqual(third['canonical']['source_reused'],2)
            root,base,report=prep_open(cfg['output_dir'])
            rows=[json.loads(l) for l in dataset_read(root,report['manifest']).splitlines()]
            victim=next(r for r in rows if r['variant']=='clahe')
            dataset_path(root,victim['image']['path']).write_bytes(b'corrupt')
            with self.assertRaises(ValueError):prep_validate(cfg['output_dir'])
            repaired=prep_run(cfg,runtime)
            # Repaired pixels match; fresh measured runtime legitimately changes snapshot metadata.
            dataset_load_image(root,victim['image'])
            self.assertTrue(prep_validate(cfg['output_dir'])['restoration']['validated'])
    def test_interruption_preserves_last_published_snapshot(self):
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);cfg['variants']=['original'];make_scan_pdf(Path(cfg['data_dir'])/'scan.pdf')
            runtime=Path(d)/'runtime';first=prep_run(cfg,runtime)
            cfg['variants']+=['clahe'];original=globals()['prep_classical'];calls=[0]
            def interrupt(im,name,opts):
                if name=='clahe':
                    calls[0]+=1
                    if calls[0]==2:raise KeyboardInterrupt('simulated interruption')
                return original(im,name,opts)
            globals()['prep_classical']=interrupt
            try:
                with self.assertRaises(KeyboardInterrupt):prep_run(cfg,runtime)
            finally:globals()['prep_classical']=original
            self.assertEqual(dataset_open(cfg['output_dir'])[1]['snapshot_id'],first['snapshot_id'])
            completed=prep_run(cfg,runtime)
            self.assertGreater(completed['variants_reused'],0)
            self.assertNotEqual(completed['snapshot_id'],first['snapshot_id'])
    def test_failure_does_not_publish_complete_marker(self):
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);make_scan_pdf(Path(cfg['data_dir'])/'scan.pdf');cfg['max_output_pixels']=10000
            with self.assertRaisesRegex(RuntimeError,'variants failed'):prep_run(cfg,Path(d)/'runtime')
            self.assertFalse((Path(cfg['output_dir'])/'dataset.json').exists())
    def test_not_applicable_and_partial_states_are_explicit(self):
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);cfg['variants']=['original','difftsr','unihir_restoration','realesrgan_x4'];cfg['allow_partial']=True
            make_scan_pdf(Path(cfg['data_dir'])/'scan.pdf')
            result=prep_run(cfg,Path(d)/'runtime')
            self.assertGreater(result['states']['not_applicable'],0);self.assertGreater(result['states']['missing_assets'],0)
            self.assertTrue(prep_validate(cfg['output_dir'])['partial'])
    def test_contract_does_not_rotate_or_reflow(self):
        cfg=prep_test_config(tempfile.mkdtemp());r={'id':'column_000','source_box':[0,0,20,200]}
        self.assertEqual(prep_applicability('difftsr','scan',0,r,cfg)[0],'not_applicable')
        cfg['reviewed_regions']['scan:1:column_000']={'upright_horizontal_single_line':True,'character_count':12,'allow_simplified_text_prior':True}
        self.assertEqual(prep_applicability('difftsr','scan',0,r,cfg)[0],'not_applicable')
    def test_worker_process_isolation(self):
        with tempfile.TemporaryDirectory() as d:
            worker=PrepWorker(d,sys.executable,'realesrgan_x4',{}, {},90,action='isolation_probe')
            try:
                self.assertNotEqual(worker.ready['pid'],os.getpid());self.assertTrue(worker.ready['offline'])
                self.assertTrue(worker.ready['network_block_verified'])
            finally:worker.close()
    def test_thin_tiled_coverage_real_tensor_identity(self):
        # Actual tensor blending; no mock model inference claim.
        import torch
        spec=importlib.util.spec_from_loader('embedded_image_worker',loader=None);mod=importlib.util.module_from_spec(spec)
        # Do not execute worker's audit hook in kernel. Extract just the two tiling functions.
        import ast
        tree=ast.parse(WORKER_SOURCE)
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name in ('starts','tiled_tadisr'):
                exec(compile(ast.Module(body=[node],type_ignores=[]),'<tiling_test>','exec'),mod.__dict__)
        for h,w in ((211,19),(17,199),(80,90),(17,19)):
            a=torch.rand(1,3,h,w)
            out,mask,info=mod.tiled_tadisr(a,lambda p:(p,p[:,:1]*0),64,32)
            self.assertTrue(torch.allclose(a,out,atol=2e-6));self.assertTrue(info['covered_all_pixels'])
    def test_storage_and_output_limits(self):
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);make_scan_pdf(Path(cfg['data_dir'])/'scan.pdf');cfg['max_storage_bytes']=1024
            with self.assertRaisesRegex(RuntimeError,'max_storage_bytes'):prep_run(cfg,Path(d)/'runtime')
            self.assertFalse((Path(cfg['output_dir'])/'dataset.json').exists())
    def test_path_traversal_rejected(self):
        with self.assertRaises(ValueError):dataset_path('.', '../escape.png')
    def test_checkpoint_identity_changes_cache_key(self):
        # Actual hash contract used by preflight, with real file mutation.
        with tempfile.TemporaryDirectory() as d:
            root=Path(d);p=root/'test.pth';p.write_bytes(b'first')
            assets_inventory(root,'synthetic fixture',[]);a=prep_check_manifest(root,['test.pth'])
            p.write_bytes(b'second');assets_inventory(root,'synthetic fixture',[]);b=prep_check_manifest(root,['test.pth'])
            self.assertNotEqual(a,b)
    def test_missing_paired_tadisr_mask_blocks_publication(self):
        from PIL import Image
        original_preflight=globals()['prep_preflight'];original_worker=globals()['PrepWorker']
        class IncompleteBackend:
            def __init__(self,*a,**kw):self.ready={};self.index=0
            def infer(self,input_path,seed,timeout):
                self.index+=1;folder=Path(runtime)/str(self.index);folder.mkdir(parents=True)
                with Image.open(input_path) as image:
                    image.resize((image.width*4,image.height*4)).save(folder/'result.png')
                return folder,{'extras':{},'seconds':0.,'state':'ok'}
            def close(self):pass
        with tempfile.TemporaryDirectory() as d:
            cfg=prep_test_config(d);cfg['variants']=['original','tadisr_x4']
            make_scan_pdf(Path(cfg['data_dir'])/'scan.pdf');runtime=Path(d)/'runtime'
            globals()['prep_preflight']=lambda *a:{'tadisr_x4':{'state':'ready','python':sys.executable,
                'assets':{},'checkpoint_identity':'synthetic_backend_for_missing_mask_failure_test'}}
            globals()['PrepWorker']=IncompleteBackend
            try:
                with self.assertRaisesRegex(RuntimeError,'variants failed'):prep_run(cfg,runtime)
            finally:
                globals()['prep_preflight']=original_preflight;globals()['PrepWorker']=original_worker
            self.assertFalse((Path(cfg['output_dir'])/'dataset.json').exists())

def run_image_selftests():
    suite=unittest.defaultTestLoader.loadTestsFromTestCase(OfflineImageTests)
    result=unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():raise RuntimeError('Embedded selftests failed; stop before processing real PDFs')
    return {'tests':result.testsRun,'failures':len(result.failures),'errors':len(result.errors),
        'real_neural_inference':False,'note':'Synthetic PDF/image/tensor tests plus a real isolated subprocess'}


## 6. Chuẩn bị asset local hoặc chạy offline

**Local:** chọn `RUN_MODE='prepare_assets'`, `ALLOW_ASSET_INTERNET=True`; bật
`DOWNLOAD_LARGE_TADISR=True` khi tải CogView4. Tất cả tải mới nằm trong
`kaggle_upload/models/image_restoration` và `kaggle_upload/packages/image_restoration`.
Không sửa bộ OCR cũ. Wheel target Linux, không tải wheel Windows cho Kaggle.
Prompt embeddings còn thiếu: gọi `prepare_tadisr_prompt(MODELS_DIR, python_cua_runtime,
RUNTIME_DIR)` trên máy đủ RAM/VRAM đã cài runtime tương thích. Hàm dùng đúng prompt
upstream, không gọi API và cập nhật inventory. Không tự chạy thao tác tốn bộ nhớ này.

Trong process trên Kaggle, `GENERATE_TADISR_PROMPT_OFFLINE=True` còn cho phép tạo
embedding từ base đã kiểm checksum, bằng script upstream trong worker riêng rồi
thoát. File dẫn xuất nằm trong RUNTIME_DIR/derived_prompts, không ghi vào Input;
model phục hồi chỉ nạp sau khi bước này xong. Không tải mạng hoặc đổi prompt.

**Cập nhật Kaggle thủ công:** ghép thư mục image_restoration vào hai Dataset
models/packages hiện có, giữ cây gốc; upload bằng giao diện Kaggle. PDF Dataset
giữ riêng. Gắn phiên bản mới, tắt Internet, đặt ba đường dẫn, chọn process và Run All.

DiffTSR cần `DiffTSR.ckpt` **và** `transocr.pth` từ bản phát hành Google Drive/Baidu.
Input phải là một dòng ngang, <=24 ký tự và aspect ratio hợp lệ; không xoay cột.
Model có text prior chuyển `zh-cn` nội bộ, nên chỉ opt in với cảnh báo phồn thể.
UniHIR cần bộ ModelScope đầy đủ, 6 font upstream, runtime flash-attn riêng và vùng
inscription hư hại đã xác nhận. Ordinary scan được ghi not_applicable.
Không tạo kết quả thay thế giả cho model bị chặn.


In [ ]:
if RUN_MODE not in ('process','prepare_assets'):
    raise ValueError('RUN_MODE must be process or prepare_assets')
if RUN_MODE == 'prepare_assets':
    STAGING_REPORT = assets_prepare(STAGING_DIR,ALLOW_ASSET_INTERNET,
        DOWNLOAD_LARGE_TADISR,TARGET_PYTHON_VERSION,DOWNLOAD_DIFFTSR,DOWNLOAD_UNIHIR,
        ACCEPT_UNIHIR_NONCOMMERCIAL)
    print(json.dumps(STAGING_REPORT,ensure_ascii=False,indent=2))
elif RUN_PREPARATION:
    for key in ('packages_dir','models_dir','data_dir'):
        if not CONFIG[key] or not Path(CONFIG[key]).is_dir():
            raise FileNotFoundError(f'Điền đường dẫn {key} hợp lệ trong ô cấu hình: {CONFIG[key]!r}')
    BASE_ENVIRONMENT = prep_bootstrap(PACKAGES_DIR,RUNTIME_DIR)
    print('Môi trường cơ sở:',BASE_ENVIRONMENT)
    SELFTEST_REPORT = run_image_selftests() if RUN_SELF_TESTS else {'skipped_by_configuration':True}
    SUMMARY = prep_run(CONFIG,RUNTIME_DIR)
    print(json.dumps({k:v for k,v in SUMMARY.items() if k!='measurements'},ensure_ascii=False,indent=2))


## 7. Xác minh và xem cùng vùng giữa các phương pháp

`source_observation`, `deterministic_transformation`, `neural_restoration` và
`generative_reconstruction` là các loại bằng chứng khác nhau. UniHIR có thể thêm
chữ không được pixel gốc xác nhận; intermediate predictions lưu JSON riêng,
không đưa vào canonical/OCR. Text mask TADiSR là dự đoán, không phải chứng cứ có chữ.

Contact sheet dùng cùng region, giữ original để so nét thêm/mất, bộ thủ gộp,
chữ gãy, nền sắc giả và seam. `overlap_patch_mae` chỉ cảnh báo bất đồng ở tile,
không thay kiểm tra bằng mắt. Hãy thử thêm vùng mờ/hiếm bằng `BOOKS=['4']; PAGES='50'; LIMIT=1`.
NLM giữ mặc định không chọn; component count giảm là cảnh báo, không phải kết luận mất chữ.


In [ ]:
if RUN_MODE == 'process' and RUN_PREPARATION:
    if VALIDATE_OUTPUT:
        VALIDATION = prep_validate(OUTPUT_DIR)
        print(json.dumps(VALIDATION,ensure_ascii=False,indent=2))
    PREVIEWS = prep_previews(OUTPUT_DIR,maximum=3) if EXPORT_PREVIEWS else []
    try:
        from IPython.display import display,Image as DisplayImage
        for path in PREVIEWS: display(DisplayImage(filename=path))
    except ImportError:
        print('Preview PNG:',PREVIEWS)


## 8. Đọc output, resume và báo cáo thực nghiệm

Output mặc định `/kaggle/working/image_dataset/`: `dataset.json`, immutable
`snapshots/`, `objects/` PNG theo nội dung, `pages/`, `layouts/`, `sources/`,
`restoration_cache/`, `restoration_manifests/`, `diagnostics/`, `previews/`,
`last_run.json`. `canonical.json` chỉ là stage bảo toàn nguồn; dataset hoàn chỉnh
chỉ được công bố sau QC. Mọi selected method có receipt ok/not_applicable/
missing_assets/failed/skipped_by_configuration. Partial output luôn có cờ.

Để resume, giữ nguyên output và Run All với `RESUME=True`. Nếu phiên mới mất
working, tải output cũ rồi giải nén/chép vào thư mục ghi được dưới working;
không trỏ output đến input chỉ đọc. Hash nguồn/config/checkpoint/code đổi sẽ
chỉ vô hiệu hóa stage liên quan. Thêm model không tái tạo nguồn và ảnh cổ điển.
Giới hạn storage gồm object/metadata; worker scratch theo từng region. Chọn toàn
bộ corpus bằng cấu hình đã ghi ở ô 1; chia từng sách khi dung lượng vượt giới hạn.

Notebook 3 hiện đọc **original** trực tiếp bằng PREPARED_IMAGES=OUTPUT_DIR và
PREPARED_VARIANT='original'. Core v1 không chứa ảnh x4 nhằm giữ reader cũ nguyên
trạng. Đọc biến thể mở rộng bằng ví dụ dưới. Không dùng hàm ZIP v1 cũ để đóng
gói extension; chuyển cả thư mục dataset hoặc tham chiếu đầy đủ trong manifest.

`last_run.json` ghi thời gian thật từng region và peak allocated/reserved VRAM
nếu worker GPU đã chạy. `preflight.json` ở runtime chỉ là probe; không gọi nó là
kết quả inference. Không có kết luận OCR tốt hơn hoặc glyph đúng từ image metrics.

Nguồn chính thức: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN),
[TADiSR](https://github.com/mingcv/TADiSR), [DiffTSR](https://github.com/YuzheZhang-1999/DiffTSR),
[UniHIR](https://github.com/ZZXF11/UniHIR), [BasicSR](https://github.com/XPixelGroup/BasicSR).


In [ ]:
if RUN_MODE == 'process' and RUN_PREPARATION:
    root,descriptor,restoration = prep_open(OUTPUT_DIR)
    print('Dataset:',root/'dataset.json')
    print('Snapshot:',descriptor['snapshot_id'])
    print('Trạng thái:',restoration['states'])
    print('Không tự upload. Chưa đo OCR accuracy/CER.')
    # Ví dụ đọc tuần tự: chỉ nạp image của region cần xem.
    # for line in dataset_read(root,restoration['manifest']).splitlines():
    #     row=json.loads(line)
    #     if row['state']=='ok' and row['variant']=='tadisr_x4':
    #         image=dataset_load_image(root,row['image'])
    #         print(row['source_box'],row['image']['image_to_source'])
    #         break
